# Amazon-BuyBox-Econometrics-Analysis-Notebook

## FBA fulfillment status and ranking outcomes on Amazon Italy

This notebook documents the empirical workflow underlying the master thesis on the association between Fulfillment by Amazon (FBA) status and seller-list ranking outcomes for the Xiaomi Mi Smart Band 6 on Amazon Italy. The workflow proceeds from the raw scraped CSV, audits the data-generating structure, constructs a seller-market panel, estimates static and dynamic econometric specifications, evaluates robustness and dependence diagnostics, and exports the full set of audited tables to a structured output tree.

Lower values of `rank_pct` denote better within-market seller-list position. The empirical claim is observational: the final panel contains no within-seller variation in FBA status, so no specification identifies the causal effect of FBA adoption. The notebook is the replication record for the thesis chapters and appendices; it is not a substitute for the manuscript's interpretation.


## Software environment and methodological dependencies

This block documents the complete computational and methodological stack used by the notebook. Every package, model, and diagnostic invoked downstream is listed here with a reference to its official documentation.

### Python packages

| Package | Role in the notebook | Documentation |
|---|---|---|
| `pandas` | Tabular data ingestion, reshaping, and CSV export | https://pandas.pydata.org/docs/ |
| `numpy` | Numerical arrays and linear-algebra primitives | https://numpy.org/doc/stable/ |
| `statsmodels` | Linear models, GLM, robust and clustered inference, specification tests | https://www.statsmodels.org/stable/ |
| `scipy` | Distributions, hypothesis tests, sparse linear algebra | https://docs.scipy.org/doc/scipy/ |
| `patsy` | Formula parsing and design-matrix construction | https://patsy.readthedocs.io/en/latest/ |
| `scikit-learn` | Logistic regression for propensity-score estimation, preprocessing pipelines | https://scikit-learn.org/stable/ |
| `matplotlib` | Lightweight visualization for diagnostics | https://matplotlib.org/stable/contents.html |
| `PyYAML` | YAML serialization of manifests and model-specification payloads | https://pyyaml.org/wiki/PyYAMLDocumentation |
| `numba` | Optional JIT acceleration of inner loops in bootstrap routines | https://numba.readthedocs.io/en/stable/ |
| `IPython.display` | Inline rendering of pandas tables | https://ipython.readthedocs.io/en/stable/ |

### Econometric models

| Model | Implementation | Reference |
|---|---|---|
| Ordinary least squares with high-dimensional fixed effects | `statsmodels.regression.linear_model.OLS` and `statsmodels.formula.api.ols` | https://www.statsmodels.org/stable/regression.html |
| Fractional logit for `rank_pct` as a bounded outcome | `statsmodels.genmod.generalized_linear_model.GLM` with `family=Binomial()` and logit link | https://www.statsmodels.org/stable/glm.html |
| Binary logit for propensity-score estimation | `sklearn.linear_model.LogisticRegression` (L2-regularized) | https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html |
| Linear probability model for binary top-k entry | `statsmodels.regression.linear_model.OLS` on a 0/1 outcome | https://www.statsmodels.org/stable/regression.html |

The fractional-logit functional form follows Papke and Wooldridge (1996); the propensity-score balancing follows Imbens and Rubin (2015) and Cameron and Trivedi (2005).

### Inference and specification diagnostics

| Diagnostic | Implementation | Reference |
|---|---|---|
| Ramsey RESET test for functional-form misspecification | `statsmodels.stats.diagnostic.linear_reset` | https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.linear_reset.html |
| Harvey-Collier test for linearity | `statsmodels.stats.diagnostic.linear_harvey_collier` | https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.linear_harvey_collier.html |
| Variance inflation factors | `statsmodels.stats.outliers_influence.variance_inflation_factor` | https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html |
| Cook's distance, DFFITS, leverage, studentized residuals | `statsmodels.stats.outliers_influence.OLSInfluence` | https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.OLSInfluence.html |
| One-way and two-way clustered standard errors | `statsmodels.stats.sandwich_covariance.cov_cluster` and `cov_cluster_2groups` | https://www.statsmodels.org/stable/sandwich_covariance.html |
| Multiple-testing correction (Benjamini-Hochberg FDR, Holm) | `statsmodels.stats.multitest.multipletests` | https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html |

### Custom inference procedures

| Procedure | Reference |
|---|---|
| Wild cluster bootstrap with Rademacher weights | Cameron, Gelbach, and Miller (2008); Cameron and Miller (2015); MacKinnon and Webb (2017) |
| CR2 finite-cluster correction | Bell and McCaffrey (2002); MacKinnon, Nielsen, and Webb (2022) |
| Permutation-based temporal placebo tests | Imbens and Rubin (2015), chapter on randomization inference |
| Oster (2019) coefficient-stability sensitivity to selection on unobservables | Oster (2019), *Journal of Business and Economic Statistics* |
| Mundlak correlated-effects diagnostic | Mundlak (1978), *Econometrica* |

All custom routines are defined explicitly in the notebook; no external implementation is assumed beyond the packages listed above. Bootstrap and permutation routines use `WILD_BOOTSTRAP_REPLICATIONS = 4999` by default to keep rank-based bootstrap tails free of ties at the median.

### Reproducibility discipline

The active kernel recomputes every analytical object from the raw CSV. Exported CSV files are downstream products, not inputs to estimation. A SHA256 fingerprint of the raw input is recorded by the audit registry so that any discrepancy between the input file and the audited sample is flagged before the econometric tables are interpreted.


## Execution environment and input files

The notebook executes in Google Colab or in a local Jupyter environment. The Colab workflow assumes the Drive directory `Amazon-BuyBox-Econometrics-Analysis/Datasets/` contains the raw CSV; the local workflow assumes the CSV is reachable from the working directory or a candidate fallback path.

Required raw input: `Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv`.

Output directories created on first execution if absent:
- `Datasets/EDA-Results/`: exploratory and audit CSV tables produced by Part I.
- `Datasets/Econometrics-Results/`: static and dynamic regression tables, robustness diagnostics, manifests, and YAML specifications produced by Part II.
- `Datasets/Machine-Readable-Results-Aggregated/`: consolidated structured export aggregating the raw CSV, the EDA results, and the econometric results in a compact JSONL form for machine-readable replication.


## Empirical scope and recorded evidence

The raw extraction contains 9,424 rows, 32 source columns, and 62 timestamped markets. The raw file records 62 Buy Box-positive rows, one per market. The `Nuovo` (new-condition) subset contains 5,786 rows; after seller-best collapsing and the third-party restriction, the final econometric panel contains 5,107 seller-market observations across 119 third-party seller identities and 62 markets.

The EDA audit documents the unconditional ranking gap by fulfillment status in the preferred third-party seller sample. Non-FBA offers display a mean raw-rank position of 47.99 and a mean normalized rank of 0.574; FBA offers display a mean raw-rank position of 16.84 and a mean normalized rank of 0.194. These raw differences motivate the econometric design without being interpreted as causal estimates, since FBA status is not randomly assigned and co-varies with observable commercial characteristics.

The static market fixed-effect estimates record the within-market FBA association before and after conditioning on commercial covariates. Specification S1 (FBA only with market fixed effects) yields a coefficient of -0.3804; specification S4 (fully adjusted split-price model with reputation, shipping, delivery, and market fixed effects) yields a coefficient of -0.0510. The dynamic transition design estimates an FBA-by-total-turnover interaction of 0.002467 on `rank_pct_improvement` with p-value 0.0020.

These quantities define the scope of the analysis: the evidence supports an observational FBA-associated ranking premium within the studied seller-list panel; the design does not identify a causal adoption effect, a Featured Offer allocation effect, or external validity beyond this product page.


In [ ]:
# -----------------------------------------------------------------------------
# Setup. Google Drive mount and core-package availability check
# -----------------------------------------------------------------------------
# Mount Drive when running on Colab, verify that the core scientific stack is importable, and surface any missing packages before the analysis starts.

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is unavailable; continuing with local file resolution.')
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')

# Core packages are available in standard Colab runtimes. This cell keeps the
# dependency check explicit without changing the empirical design.
_required_packages = ['numpy', 'pandas', 'statsmodels', 'scipy', 'patsy', 'yaml', 'matplotlib']
_missing_packages = []
for _pkg in _required_packages:
    try:
        __import__(_pkg)
    except ModuleNotFoundError:
        _missing_packages.append(_pkg)
if _missing_packages:
    print('Missing packages:', _missing_packages)
    print('Install them before running the full notebook, for example with %pip install pyyaml statsmodels scipy patsy matplotlib')
else:
    print('Required core packages are importable.')

Mounted at /content/drive
Required core packages are importable.


# Part I. Raw-data audit and exploratory diagnostics

Part I audits the raw CSV before any econometric restriction is imposed. It defines the validated raw primitives, the deterministic repairs of source-field anomalies, the unit-of-analysis constraints, the candidate sample definitions, and the rank-support facts inherited by the econometric sections in Part II.


## A. Raw-data audit: `Nuovo` third-party market

The audit fixes which fields are usable as primitives, which fields require deterministic reconstruction from the raw text, and which empirical objects are supported by the data after the restrictions inherited by the econometric sample.

The audit focuses on the `Nuovo` condition and, for the ranking object of interest, on third-party sellers. The output of this part is a typed and audited representation of the raw extraction, a column-level provenance classification, candidate sample definitions, and a set of audit checks recorded in a structured registry.


### 1. Load packages, define helper functions, and resolve the input file

The cell initializes the notebook environment, defines the reusable helper functions, and resolves the raw CSV path. The resolver supports Colab Drive mounts, local working directories, and standard candidate paths. The cell also initializes the audit-check registry and records the basic file provenance (path, size, SHA256 fingerprint), which uniquely identifies the audited raw input.


In [ ]:
# -----------------------------------------------------------------------------
# Section 1. Packages, configuration, helper functions, and input-file resolver
# -----------------------------------------------------------------------------
# Load the Part I dependencies, fix display options, define the audit-registry helpers, and resolve the raw CSV path across Colab Drive, local working directories, and standard fallback paths.

import os
import re
import hashlib
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime
from pathlib import Path
import io
import json
import shutil
import tempfile
import zipfile
import fnmatch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 180)

# Notebook configuration
NOTEBOOK_VERSION = '2026-04-25'
STRICT_ASSERTS = False
EXPORT_FILES = True
DATA_PATH_OVERRIDE = None
EXPORT_SUBDIR_NAME = 'EDA-Results'

# Optional: keep warnings visible by default for reproducibility.
# If a specific warning later becomes too noisy, suppress it locally in that cell only.

CANDIDATE_FILE_NAMES = [
    'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6 - Foglio1.csv',
    'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv',
]

CANDIDATE_DIRS = [
    Path.cwd(),
    Path('/mnt/data'),
    Path('.'),
    Path('/content'),
    # Legacy fallback from earlier Colab-based workflow; retained only as a last-resort search path.
    Path('/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets'),
]


def resolve_input_file(data_path_override=None, candidate_names=None, candidate_dirs=None):
    """
    Resolve the Xiaomi raw CSV path in a portable way.

    Priority:
    1. explicit DATA_PATH_OVERRIDE
    2. direct lookup in candidate directories
    3. recursive lookup from common roots
    """
    if data_path_override is not None:
        p = Path(data_path_override)
        if p.exists():
            return p.resolve()
        raise FileNotFoundError(f'Explicit DATA_PATH_OVERRIDE not found: {data_path_override}')

    candidate_names = candidate_names or CANDIDATE_FILE_NAMES
    candidate_dirs = candidate_dirs or CANDIDATE_DIRS

    for directory in candidate_dirs:
        for name in candidate_names:
            p = directory / name
            if p.exists():
                return p.resolve()

    search_roots = [Path.cwd(), Path('/mnt/data'), Path('/content')]
    for root in search_roots:
        if root.exists():
            for name in candidate_names:
                hits = list(root.rglob(name))
                if hits:
                    return hits[0].resolve()

    raise FileNotFoundError(
        'Xiaomi CSV not found in any expected location. '
        'Set DATA_PATH_OVERRIDE explicitly if needed.'
    )


# -----------------------------------------------------------------------------
# Audit registry
# -----------------------------------------------------------------------------

AUDIT_CHECKS = []


def register_check(section, check_name, passed, detail=''):
    """
    Register an audit check and optionally raise if STRICT_ASSERTS is enabled.
    """
    record = {
        'section': str(section),
        'check_name': str(check_name),
        'passed': bool(passed),
        'detail': str(detail),
    }
    AUDIT_CHECKS.append(record)

    if STRICT_ASSERTS and (not passed):
        raise AssertionError(f'[{section}] {check_name} FAILED: {detail}')


def audit_checks_frame():
    """
    Return the accumulated audit checks as a DataFrame.
    """
    if not AUDIT_CHECKS:
        return pd.DataFrame(columns=['section', 'check_name', 'passed', 'detail'])
    return pd.DataFrame(AUDIT_CHECKS)


# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------

def build_row_signature(df, columns):
    """
    Build a row-level signature over selected columns.
    Useful for exact-duplicate diagnostics.
    """
    return df[columns].astype('string').fillna('<NA>').agg(' || '.join, axis=1)


def clean_string_columns(df):
    """
    Normalize string-like columns without changing the economic sample.
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype('string').str.strip()
            s = s.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
            df[col] = s
    return df


def parse_numeric_italian(series):
    """
    Parse Italian-formatted numeric strings (e.g. '1.234,56') into numeric dtype.
    """
    s = series.astype('string')
    s = s.str.replace('.', '', regex=False)
    s = s.str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')


def parse_boolean_flag(series):
    """
    Parse common boolean encodings into pandas BooleanDtype.
    """
    s = series.astype('string').str.strip().str.lower()
    mapped = s.map({
        'true': True, 'false': False,
        '1': True, '0': False
    })
    return mapped.astype('boolean')


def strip_accents(text):
    """
    Remove accents to make raw-text parsing more robust.
    """
    return ''.join(
        ch for ch in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(ch)
    )


# -----------------------------------------------------------------------------
# Delivery and shipping parsing helpers
# These are defined here because later audit sections use them repeatedly.
# -----------------------------------------------------------------------------

MONTH_MAP = {
    'gen': 1, 'gennaio': 1,
    'feb': 2, 'febbraio': 2,
    'mar': 3, 'marzo': 3,
    'apr': 4, 'aprile': 4,
    'mag': 5, 'maggio': 5,
    'giu': 6, 'giugno': 6,
    'lug': 7, 'luglio': 7,
    'ago': 8, 'agosto': 8,
    'set': 9, 'sett': 9, 'settembre': 9,
    'ott': 10, 'ottobre': 10,
    'nov': 11, 'novembre': 11,
    'dic': 12, 'dicembre': 12,
}

range_two_months_re = re.compile(r'(\d{1,2})\s+([a-z]+)\s*-\s*(\d{1,2})\s+([a-z]+)')
range_same_month_re = re.compile(r'(\d{1,2})\s*-\s*(\d{1,2})\s+([a-z]+)')
single_date_re = re.compile(
    r'(?:lunedi|martedi|mercoledi|giovedi|venerdi|sabato|domenica)?\s*,?\s*(\d{1,2})\s+([a-z]+)'
)

old_fast_split_re = re.compile(r'consegna\s+piu\s+veloce\s*:?', re.I)
old_primary_prefix_re = re.compile(
    r'^(?:spedizione\s+gratuita\s*:|consegna\s+a\s*[0-9]+(?:[\.,][0-9]+)?\s*e\s*:?)',
    re.I
)

robust_fast_split_re = re.compile(r'(?:oppure\s+)?consegna\s+piu\s+(?:veloce|rapida)\s*:?', re.I)
robust_primary_prefix_re = re.compile(
    r'^(?:spedizione\s+gratuita\s*:|consegna\s+gratuita\s*|consegna\s+a\s*[0-9]+(?:[\.,][0-9]+)?\s*e\s*:?)',
    re.I
)

shipping_price_re = re.compile(r'consegna\s+a\s*([0-9]+(?:[\.,][0-9]+)?)\s*€', re.I)
shipping_free_re = re.compile(r'(spedizione|consegna)\s+gratuita', re.I)
contact_courier_re = re.compile(r'verrai\s+contattato\s+dal\s+corriere', re.I)


def normalize_delivery_text(text):
    """
    Normalize raw delivery/shipping text before parsing.
    """
    if text is None or pd.isna(text):
        return None

    s = str(text).lower()
    s = strip_accents(s)
    s = s.replace('maggiori informazioni', ' ')
    s = re.sub(r'ordina entro [^.]*', ' ', s)
    s = s.replace('sul tuo primo ordine idoneo', ' ')
    s = s.replace('.', ' ')
    s = s.replace(':', ' : ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def build_delivery_date(ts, day, month_token):
    """
    Build a delivery date from day/month tokens using the timestamp year,
    with year rollover handling for adjacent-year deliveries.
    """
    month_token = month_token.strip().lower()
    month = MONTH_MAP.get(month_token, MONTH_MAP.get(month_token[:3]))

    if month is None:
        return None

    year = ts.year
    if month < ts.month - 1:
        year += 1

    try:
        return datetime(year, month, int(day)).date()
    except ValueError:
        return None


def parse_delivery_window(segment, ts):
    """
    Parse a delivery window from a normalized text segment.
    Returns (min_days, max_days).
    """
    if segment is None or pd.isna(segment) or pd.isna(ts):
        return (np.nan, np.nan)

    s = segment.strip()

    m = range_two_months_re.search(s)
    if m:
        d1, m1, d2, m2 = m.groups()
        dt1 = build_delivery_date(ts, d1, m1)
        dt2 = build_delivery_date(ts, d2, m2)
        if dt1 is not None and dt2 is not None:
            return ((dt1 - ts.date()).days, (dt2 - ts.date()).days)

    m = range_same_month_re.search(s)
    if m:
        d1, d2, m1 = m.groups()
        dt1 = build_delivery_date(ts, d1, m1)
        dt2 = build_delivery_date(ts, d2, m1)
        if dt1 is not None and dt2 is not None:
            return ((dt1 - ts.date()).days, (dt2 - ts.date()).days)

    m = single_date_re.search(s)
    if m:
        d1, m1 = m.groups()
        dt = build_delivery_date(ts, d1, m1)
        if dt is not None:
            diff = (dt - ts.date()).days
            return (diff, diff)

    return (np.nan, np.nan)


def parse_delivery_original_style(text, ts):
    """
    Approximate the logic of the original stored-parser.
    Used for parser-failure diagnostics, not as source of truth.
    """
    s = normalize_delivery_text(text)
    if s is None or pd.isna(ts):
        return [np.nan, np.nan, np.nan, np.nan]

    parts = old_fast_split_re.split(s, maxsplit=1)
    primary = old_primary_prefix_re.sub('', parts[0]).strip()
    fast = parts[1].strip() if len(parts) > 1 else None

    # If the primary prefix is not removed, original-style parsing likely failed.
    if primary == parts[0]:
        return [np.nan, np.nan, np.nan, np.nan]

    g_min, g_max = parse_delivery_window(primary, ts)
    gv_min, gv_max = parse_delivery_window(fast, ts) if fast else (0, 0)
    return [g_min, g_max, gv_min, gv_max]


def parse_delivery_robust(text, ts):
    """
    More robust delivery parser used for audit/reconstruction checks.
    """
    s = normalize_delivery_text(text)
    if s is None or pd.isna(ts):
        return [np.nan, np.nan, np.nan, np.nan]

    parts = robust_fast_split_re.split(s, maxsplit=1)
    primary = robust_primary_prefix_re.sub('', parts[0]).strip()
    fast = parts[1].strip() if len(parts) > 1 else None

    g_min, g_max = parse_delivery_window(primary, ts)
    gv_min, gv_max = parse_delivery_window(fast, ts) if fast else (0, 0)

    # Observed stored-data rule:
    # if only the fast window is exposed in text, it is reused as the normal one,
    # while fast fields remain zero.
    if pd.isna(g_min) and not pd.isna(gv_min):
        g_min, g_max, gv_min, gv_max = gv_min, gv_max, 0, 0

    return [g_min, g_max, gv_min, gv_max]


def parse_shipping_from_text(text):
    """
    Parse shipping price and shipping mode directly from the raw text field.
    Used for source-text audit, especially where stored shipping fields look inconsistent.
    """
    s = normalize_delivery_text(text)

    if s is None:
        return {
            'parsed_shipping_price': np.nan,
            'parsed_shipping_mode': 'missing'
        }

    if shipping_free_re.search(s):
        return {
            'parsed_shipping_price': 0.0,
            'parsed_shipping_mode': 'free_explicit'
        }

    m = shipping_price_re.search(s)
    if m:
        return {
            'parsed_shipping_price': float(m.group(1).replace(',', '.')),
            'parsed_shipping_mode': 'paid_explicit'
        }

    if contact_courier_re.search(s):
        return {
            'parsed_shipping_price': np.nan,
            'parsed_shipping_mode': 'unknown_contact_courier'
        }

    return {
        'parsed_shipping_price': np.nan,
        'parsed_shipping_mode': 'unknown_text'
    }


# -----------------------------------------------------------------------------
# Ranking helpers
# -----------------------------------------------------------------------------

def add_ranks(sample):
    """
    Add reranked positions within timestamp market after a reversible transformation.
    """
    sample = sample.sort_values(
        ['timestamp', 'visibility_order', 'venduto_da', 'spedito_da']
    ).copy()

    sample['rank_new'] = sample.groupby('timestamp').cumcount()
    sample['rank_pos'] = sample['rank_new'] + 1

    market_size = sample.groupby('timestamp')['timestamp'].transform('size')
    sample['market_size'] = market_size

    sample['rank_pct'] = np.where(
        market_size.gt(1),
        (sample['rank_pos'] - 1) / (market_size - 1),
        0.0
    )
    return sample


def summarize_candidate_sample(sample, sample_name):
    """
    Summarize the structure of a candidate analysis sample.
    """
    sample = sample.copy()
    market_size = sample.groupby('timestamp').size()
    unique_sellers = sample.groupby('timestamp')['venduto_da'].nunique()

    return pd.DataFrame({
        'sample': [sample_name],
        'rows': [int(len(sample))],
        'markets': [int(sample['timestamp'].nunique())],
        'mean_market_size': [float(market_size.mean())],
        'min_market_size': [int(market_size.min())],
        'max_market_size': [int(market_size.max())],
        'mean_unique_sellers_per_market': [float(unique_sellers.mean())],
    })


# -----------------------------------------------------------------------------
# Resolve the raw input file and record provenance
# -----------------------------------------------------------------------------

file_path = resolve_input_file(DATA_PATH_OVERRIDE)

file_size_bytes = file_path.stat().st_size
file_sha256 = hashlib.sha256(file_path.read_bytes()).hexdigest()

register_check('1', 'input_file_exists', file_path.exists(), str(file_path))
register_check(
    '1',
    'input_filename_recognized',
    file_path.name in CANDIDATE_FILE_NAMES,
    file_path.name
)
register_check('1', 'input_file_nonempty', file_size_bytes > 0, f'{file_size_bytes} bytes')

print('Notebook version:', NOTEBOOK_VERSION)
print('Using input file:')
print(file_path)
print('\nRaw-file provenance')
print('  File name :', file_path.name)
print('  File size :', file_size_bytes, 'bytes')
print('  SHA256    :', file_sha256)


Notebook version: 2026-04-25
Using input file:
/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv

Raw-file provenance
  File name : Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv
  File size : 2967342 bytes
  SHA256    : 88c85f33e3e640d6261d2ac1bb0b095ebf1c65ac8fc4edab9d850cd3c8abe5b5


### 2. Load the raw CSV with minimal normalization

The raw CSV is loaded with every column preserved as string. This avoids premature type coercion and retains the original text fields, including the upstream stored convenience variables and the raw shipping-delivery text, before any substantive parsing or filtering. A minimal normalization step removes leading and trailing whitespace and converts trivial placeholder strings into missing values, leaving the economic sample untouched.


In [ ]:
# -----------------------------------------------------------------------------
# Section 2. Load the raw CSV with minimal string normalization
# -----------------------------------------------------------------------------
# Read the raw CSV with every column preserved as string, trim leading and trailing whitespace, and convert trivial placeholders to missing values without changing the economic sample.

raw = pd.read_csv(file_path, dtype='string')
raw = clean_string_columns(raw)

EXPECTED_RAW_COLUMNS = [
    'timestamp', 'buy_box', 'visibility_order', 'condizione', 'venduto_da', 'spedito_da',
    'num_valutazioni', 'valutazioni_positive', 'prezzo', 'qta_min',
    'prezzo_prod_venduto(€)', 'tipo_spedizione', 'prezzo_spedizione(€)', 'prezzo_totale(€)',
    'dif_prezzo', 'dif_prezzo_sped', 'dif_prezzo_tot',
    'g_cons_min', 'g_cons_max', 'g_spedizione',
    'g_cons_vel_min', 'g_cons_vel_max', 'g_spedizione_vel',
    'spedizione_consegna', 'condizione_usato', 'stelle',
    'delta_consegna', 'delta_num_val', 'delta_val_pos(%)',
    'rapp_piu_basso', 'fba', 'vend_amazon'
]

missing_raw_columns = sorted(set(EXPECTED_RAW_COLUMNS) - set(raw.columns))
duplicate_column_names = raw.columns[raw.columns.duplicated()].tolist()

register_check('2', 'raw_loaded_nonempty', len(raw) > 0, f'{raw.shape}')
register_check('2', 'raw_column_count_expected', raw.shape[1] == len(EXPECTED_RAW_COLUMNS), f'{raw.shape[1]} columns')
register_check('2', 'raw_required_columns_present', len(missing_raw_columns) == 0, missing_raw_columns)
register_check('2', 'raw_column_names_unique', len(duplicate_column_names) == 0, duplicate_column_names)

print('Raw dataset shape:', raw.shape)
print('\nMissing expected columns:', missing_raw_columns if missing_raw_columns else 'None')
print('Duplicate column names:', duplicate_column_names if duplicate_column_names else 'None')

print('\nRaw columns:')
display(pd.DataFrame({'column': raw.columns}))

print('First 5 raw rows:')
display(raw.head())

Raw dataset shape: (9424, 32)

Missing expected columns: None
Duplicate column names: None

Raw columns:


,column
0,timestamp
1,buy_box
2,visibility_order
3,condizione
4,venduto_da
5,spedito_da
6,num_valutazioni
7,valutazioni_positive
8,prezzo
9,qta_min


First 5 raw rows:


,timestamp,buy_box,visibility_order,condizione,venduto_da,spedito_da,num_valutazioni,valutazioni_positive,prezzo,qta_min,prezzo_prod_venduto(€),tipo_spedizione,prezzo_spedizione(€),prezzo_totale(€),dif_prezzo,dif_prezzo_sped,dif_prezzo_tot,g_cons_min,g_cons_max,g_spedizione,g_cons_vel_min,g_cons_vel_max,g_spedizione_vel,spedizione_consegna,condizione_usato,stelle,delta_consegna,delta_num_val,delta_val_pos(%),rapp_piu_basso,fba,vend_amazon
0,2022-02-07T12:05:05,1,0,Nuovo,Amazon,Amazon,7978,0,"31,13",1,"31,13",Gratuita,0,"31,13","2,17",0,"2,17",3,3,0,0,0,0,"Spedizione GRATUITA: giovedì, 10 feb Maggiori informazioni",<NA>,"4,5",1,6348,100,"1,07",TRUE,TRUE
1,2022-02-07T12:05:05,0,1,Usato - Ottime condizioni,Amazon Warehouse,Amazon,0,0,"28,95",1,"28,95",Gratuita,0,"28,95",0,0,0,8,8,0,0,0,0,"Spedizione GRATUITA: martedì, 15 feb sul tuo primo ordine idoneo.","Piccola (minore di 0,5 cm x 0,5 cm) imperfezione estetica sulla parte anteriore dell'oggetto. L'articol... Piccola (minore di 0,5 cm x 0,5 cm) imperfezione estetica sulla parte...",0,6,14326,100,1,TRUE,FALSE
2,2022-02-07T12:05:05,0,2,Nuovo,SKY LINE E-COMM,Amazon,17,94,"38,5",1,"38,5",Gratuita,0,"38,5","9,55",0,"9,55",8,8,0,0,0,0,"Spedizione GRATUITA: martedì, 15 feb Maggiori informazioni",<NA>,"4,5",6,14309,6,"1,32",TRUE,FALSE
3,2022-02-07T12:05:05,0,3,Nuovo,ZaarioGmbH,Amazon,36,97,"38,93",1,"38,93",Gratuita,0,"38,93","9,98",0,"9,98",7,10,3,2,7,5,Spedizione GRATUITA: 14 - 17 feb Maggiori informazioni Consegna più veloce: 9 - 14 feb Maggiori informazioni,<NA>,5,0,14290,3,"1,34",TRUE,FALSE
4,2022-02-07T12:05:05,0,4,Nuovo,Bacom,Bacom,14326,96,"39,9",1,"39,9",Gratuita,0,"39,9","10,95",0,"10,95",8,11,3,7,9,2,Spedizione GRATUITA: 15 - 18 feb Maggiori informazioni Consegna più veloce: 14 - 16 feb Maggiori informazioni,<NA>,"4,5",5,0,4,"1,37",FALSE,FALSE


### 3. Column dictionary and audit-oriented provenance

The column glossary classifies each original CSV column by its role in the raw file. The labels refer to **the status of each column in the CSV**, not to the source-preferred reconstructions built downstream.

Each column is marked as one of the following:
- a direct raw field scraped from the page,
- a stored flag or convenience variable generated upstream by the scraping pipeline,
- a stored parsed field whose correctness is validated against the raw text,
- a field that requires source-based reconstruction before econometric use.

The second code cell of this section validates the glossary against the actual raw dataframe so that no undocumented or misclassified column escapes the audit registry.


In [ ]:
# -----------------------------------------------------------------------------
# Section 3. Column glossary (provenance classification)
# -----------------------------------------------------------------------------
# Build the column-by-column glossary that records, for each raw CSV column, whether it is a primitive scraped field, a stored convenience variable, a stored parsed field, or a field requiring source-based reconstruction.

column_glossary = pd.DataFrame([
    ('timestamp', 'page snapshot timestamp', 'raw field', 'core source field'),
    ('buy_box', 'stored indicator for the winner row', 'stored flag', 'descriptive only; not a safe causal covariate'),
    ('visibility_order', 'stored rank in the raw page list', 'raw field', 'core source field'),
    ('condizione', 'condition string shown on page', 'raw field', 'core source field'),
    ('venduto_da', 'seller name', 'raw field', 'core source field'),
    ('spedito_da', 'shipper name', 'raw field', 'core source field'),
    ('num_valutazioni', 'seller review count', 'raw field', 'core source field'),
    ('valutazioni_positive', 'positive rating percentage', 'raw field', 'core source field'),
    ('prezzo', 'product price excluding shipping', 'raw field', 'core source field'),
    ('qta_min', 'minimum quantity', 'raw field', 'redundancy check required'),
    ('prezzo_prod_venduto(€)', 'product price times minimum quantity', 'stored convenience', 'redundancy check required'),
    ('tipo_spedizione', 'free versus paid shipping label', 'stored convenience', 'mode check required against raw text'),
    ('prezzo_spedizione(€)', 'stored shipping price', 'stored convenience', 'must be validated against raw text'),
    ('prezzo_totale(€)', 'stored total price', 'stored convenience', 'validate against source components'),
    ('dif_prezzo', 'difference from market minimum product price', 'stored convenience', 'sample-dependent; unsafe after filtering'),
    ('dif_prezzo_sped', 'difference from market minimum shipping price', 'stored convenience', 'sample-dependent; audit required'),
    ('dif_prezzo_tot', 'difference from market minimum total price', 'stored convenience', 'sample-dependent; unsafe after filtering'),
    ('g_cons_min', 'stored normal-delivery lower bound in days', 'stored parsed field', 'validate against raw text'),
    ('g_cons_max', 'stored normal-delivery upper bound in days', 'stored parsed field', 'validate against raw text'),
    ('g_spedizione', 'stored normal-delivery span', 'stored convenience', 'reconstruct only after delivery audit'),
    ('g_cons_vel_min', 'stored fast-delivery lower bound in days', 'stored parsed field', 'validate against raw text'),
    ('g_cons_vel_max', 'stored fast-delivery upper bound in days', 'stored parsed field', 'validate against raw text'),
    ('g_spedizione_vel', 'stored fast-delivery span', 'stored convenience', 'reconstruct only after delivery audit'),
    ('spedizione_consegna', 'raw shipping and delivery text block', 'raw field', 'source field for reconstruction'),
    ('condizione_usato', 'used-condition description', 'raw field', 'not relevant in the Nuovo-focused sample'),
    ('stelle', 'stored star rating', 'raw field', 'core source field'),
    ('delta_consegna', 'difference from fastest delivery', 'stored convenience', 'delivery-parser dependent; unsafe without audit'),
    ('delta_num_val', 'gap from max review count', 'stored convenience', 'market-relative; not a primitive covariate'),
    ('delta_val_pos(%)', 'gap from max positive-rating percentage', 'stored convenience', 'market-relative; not a primitive covariate'),
    ('rapp_piu_basso', 'ratio to market minimum total price', 'stored convenience', 'sample-dependent; unsafe after filtering'),
    ('fba', 'stored FBA flag', 'stored flag', 'must be validated against shipper name'),
    ('vend_amazon', 'stored Amazon-retail flag', 'stored flag', 'must be validated against seller name'),
], columns=['column', 'description', 'status_in_raw_csv', 'audit_note'])

display(column_glossary)

,column,description,status_in_raw_csv,audit_note
0,timestamp,page snapshot timestamp,raw field,core source field
1,buy_box,stored indicator for the winner row,stored flag,descriptive only; not a safe causal covariate
2,visibility_order,stored rank in the raw page list,raw field,core source field
3,condizione,condition string shown on page,raw field,core source field
4,venduto_da,seller name,raw field,core source field
5,spedito_da,shipper name,raw field,core source field
6,num_valutazioni,seller review count,raw field,core source field
7,valutazioni_positive,positive rating percentage,raw field,core source field
8,prezzo,product price excluding shipping,raw field,core source field
9,qta_min,minimum quantity,raw field,redundancy check required


In [ ]:
# -----------------------------------------------------------------------------
# Section 3. Schema validation against the observed raw dataframe
# -----------------------------------------------------------------------------
# Cross-check the glossary against the actual raw dataframe and register the diagnostic in the audit registry.

glossary_columns = set(column_glossary['column'])
raw_columns = set(raw.columns)

missing_in_glossary = sorted(raw_columns - glossary_columns)
extra_in_glossary = sorted(glossary_columns - raw_columns)
duplicate_glossary_entries = column_glossary.loc[
    column_glossary['column'].duplicated(), 'column'
].tolist()

allowed_status_labels = {
    'raw field',
    'stored flag',
    'stored convenience',
    'stored parsed field'
}
invalid_status_labels = sorted(
    set(column_glossary['status_in_raw_csv']) - allowed_status_labels
)

register_check('3', 'glossary_covers_all_raw_columns', len(missing_in_glossary) == 0, missing_in_glossary)
register_check('3', 'glossary_has_no_extra_columns', len(extra_in_glossary) == 0, extra_in_glossary)
register_check('3', 'glossary_column_entries_unique', len(duplicate_glossary_entries) == 0, duplicate_glossary_entries)
register_check('3', 'glossary_status_labels_valid', len(invalid_status_labels) == 0, invalid_status_labels)

print('Columns missing in glossary:', missing_in_glossary if missing_in_glossary else 'None')
print('Glossary entries not found in raw data:', extra_in_glossary if extra_in_glossary else 'None')
print('Duplicate glossary entries:', duplicate_glossary_entries if duplicate_glossary_entries else 'None')
print('Invalid status labels:', invalid_status_labels if invalid_status_labels else 'None')

Columns missing in glossary: None
Glossary entries not found in raw data: None
Duplicate glossary entries: None
Invalid status labels: None


### 4. Standardize the raw table without changing the economic sample

The cell performs only non-substantive preprocessing. The economic sample remains the full raw CSV; no row is removed and no market is filtered. Transformations are limited to numeric parsing, boolean parsing, timestamp parsing, and a small number of transparent diagnostic reconstructions used downstream for audit checks.


In [ ]:
# -----------------------------------------------------------------------------
# Section 4. Non-substantive standardization and audit-oriented helper fields
# -----------------------------------------------------------------------------
# Apply numeric, boolean, and timestamp parsing and add audit-oriented helper fields. No row is removed; the economic sample remains the full raw CSV.

df_raw = raw.copy()

numeric_cols = [
    'buy_box', 'visibility_order', 'num_valutazioni', 'valutazioni_positive', 'prezzo', 'qta_min',
    'prezzo_prod_venduto(€)', 'prezzo_spedizione(€)', 'prezzo_totale(€)', 'dif_prezzo',
    'dif_prezzo_sped', 'dif_prezzo_tot', 'g_cons_min', 'g_cons_max', 'g_spedizione',
    'g_cons_vel_min', 'g_cons_vel_max', 'g_spedizione_vel', 'stelle', 'delta_consegna',
    'delta_num_val', 'delta_val_pos(%)', 'rapp_piu_basso'
]

for col in numeric_cols:
    df_raw[col] = parse_numeric_italian(df_raw[col])

for col in ['fba', 'vend_amazon']:
    df_raw[col] = parse_boolean_flag(df_raw[col])

df_raw['timestamp_dt'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')

int_like_cols = [
    'buy_box', 'visibility_order', 'num_valutazioni', 'qta_min',
    'g_cons_min', 'g_cons_max', 'g_spedizione',
    'g_cons_vel_min', 'g_cons_vel_max', 'g_spedizione_vel',
    'delta_consegna', 'delta_num_val'
]

for col in int_like_cols:
    df_raw[col] = df_raw[col].round().astype('Int64')

df_raw['prezzo_totale_reconstructed'] = (
    df_raw['prezzo'].fillna(0) + df_raw['prezzo_spedizione(€)'].fillna(0)
)

seller_name_present = df_raw['venduto_da'].notna()
shipper_name_present = df_raw['spedito_da'].notna()

df_raw['amazon_retail_from_name'] = (
    seller_name_present &
    df_raw['venduto_da'].astype('string').str.fullmatch('Amazon', case=False, na=False)
)

df_raw['amazon_warehouse_from_name'] = (
    seller_name_present &
    df_raw['venduto_da'].astype('string').str.contains('Amazon Warehouse', case=False, na=False)
)

df_raw['third_party_from_name'] = (
    seller_name_present &
    ~(df_raw['amazon_retail_from_name'] | df_raw['amazon_warehouse_from_name'])
)

df_raw['fba_from_shipper'] = (
    shipper_name_present &
    df_raw['spedito_da'].astype('string').str.fullmatch('Amazon', case=False, na=False)
)

df_raw['seller_type_from_name'] = np.select(
    [
        df_raw['amazon_retail_from_name'],
        df_raw['amazon_warehouse_from_name'],
        df_raw['third_party_from_name'],
    ],
    [
        'amazon_retail',
        'amazon_warehouse',
        'third_party',
    ],
    default='unclassified'
)

df_raw['is_malformed_row'] = (
    df_raw['condizione'].isna() |
    df_raw['venduto_da'].isna() |
    df_raw['spedito_da'].isna()
)

df_raw['prezzo_totale_gap'] = df_raw['prezzo_totale(€)'] - df_raw['prezzo_totale_reconstructed']
df_raw['prezzo_totale_gap_abs'] = df_raw['prezzo_totale_gap'].abs()

price_check_mask = (
    df_raw['prezzo'].notna() &
    df_raw['prezzo_spedizione(€)'].notna() &
    df_raw['prezzo_totale(€)'].notna()
)

vend_amazon_mismatch = (
    df_raw['vend_amazon'].notna() &
    (df_raw['vend_amazon'] != df_raw['amazon_retail_from_name'])
)

fba_mismatch = (
    df_raw['fba'].notna() &
    (df_raw['fba'] != df_raw['fba_from_shipper'])
)

standardization_audit = pd.DataFrame({
    'metric': [
        'rows_in_parsed_dataset',
        'invalid_timestamp_rows',
        'rows_with_both_price_components_and_stored_total',
        'rows_with_nonzero_total_price_gap_above_0.01',
        'vend_amazon_name_mismatch_rows',
        'fba_shipper_mismatch_rows',
        'malformed_rows',
        'unclassified_seller_rows',
    ],
    'value': [
        int(df_raw.shape[0]),
        int(df_raw['timestamp_dt'].isna().sum()),
        int(price_check_mask.sum()),
        int((price_check_mask & df_raw['prezzo_totale_gap_abs'].gt(0.01)).sum()),
        int(vend_amazon_mismatch.sum()),
        int(fba_mismatch.sum()),
        int(df_raw['is_malformed_row'].sum()),
        int(df_raw['seller_type_from_name'].eq('unclassified').sum()),
    ]
})

display(standardization_audit)

register_check(
    '4',
    'timestamps parse without invalid rows',
    df_raw['timestamp_dt'].isna().sum() == 0,
    f"invalid rows = {int(df_raw['timestamp_dt'].isna().sum())}"
)

register_check(
    '4',
    'stored total price matches reconstructed total',
    int((price_check_mask & df_raw['prezzo_totale_gap_abs'].gt(0.01)).sum()) == 0,
    'tolerance = 0.01 euro'
)

register_check(
    '4',
    'stored vend_amazon matches seller name',
    int(vend_amazon_mismatch.sum()) == 0,
    f"mismatches = {int(vend_amazon_mismatch.sum())}"
)

register_check(
    '4',
    'stored fba matches shipper name',
    int(fba_mismatch.sum()) == 0,
    f"mismatches = {int(fba_mismatch.sum())}"
)

register_check(
    '4',
    'exactly one malformed row is present',
    int(df_raw['is_malformed_row'].sum()) == 1,
    f"malformed rows = {int(df_raw['is_malformed_row'].sum())}"
)

register_check(
    '4',
    'malformed row is not silently classified as third party',
    not bool(df_raw.loc[df_raw['is_malformed_row'], 'third_party_from_name'].fillna(False).any()),
    'malformed seller rows must remain unclassified'
)

,metric,value
0,rows_in_parsed_dataset,9424
1,invalid_timestamp_rows,0
2,rows_with_both_price_components_and_stored_total,9424
3,rows_with_nonzero_total_price_gap_above_0.01,0
4,vend_amazon_name_mismatch_rows,0
5,fba_shipper_mismatch_rows,0
6,malformed_rows,1
7,unclassified_seller_rows,1


### 5. Raw extraction integrity

The cell audits the basic structural integrity of the raw CSV before any economic restriction is applied. It verifies that the file is usable as a snapshot of a seller-list panel: row counts, distinct timestamped markets, presence of the expected key columns, and absence of malformed rows in the identity fields.


In [ ]:
# -----------------------------------------------------------------------------
# Section 5. Raw extraction structural integrity
# -----------------------------------------------------------------------------
# Audit the basic structural integrity of the raw CSV (row counts, distinct markets, presence of key columns) before any economic restriction is applied.

raw_market_count = int(df_raw['timestamp'].nunique())
exact_duplicate_raw_rows = int(df_raw.duplicated().sum())
malformed_row_count = int(df_raw['is_malformed_row'].sum())

raw_integrity_summary = pd.DataFrame({
    'metric': [
        'rows_in_raw_csv',
        'unique_timestamps',
        'exact_duplicate_raw_rows',
        'malformed_rows',
        'min_timestamp',
        'max_timestamp',
    ],
    'value': [
        int(df_raw.shape[0]),
        raw_market_count,
        exact_duplicate_raw_rows,
        malformed_row_count,
        df_raw['timestamp_dt'].min(),
        df_raw['timestamp_dt'].max(),
    ]
})

display(raw_integrity_summary)

print('Structurally malformed row:')
display(df_raw.loc[df_raw['is_malformed_row']].head(10))

register_check(
    '5',
    'raw CSV has no exact full-row duplicates',
    exact_duplicate_raw_rows == 0,
    f"duplicate rows = {exact_duplicate_raw_rows}"
)

register_check(
    '5',
    'raw CSV spans 62 timestamps',
    raw_market_count == 62,
    f"timestamps = {raw_market_count}"
)

register_check(
    '5',
    'exactly one malformed row is present',
    malformed_row_count == 1,
    f"malformed rows = {malformed_row_count}"
)

register_check(
    '5',
    'malformed row is driven by missing core identity fields',
    bool(
        df_raw.loc[df_raw['is_malformed_row'], ['condizione', 'venduto_da', 'spedito_da']]
        .isna()
        .all(axis=1)
        .all()
    ),
    'expected missing fields: condizione, venduto_da, spedito_da'
)

,metric,value
0,rows_in_raw_csv,9424
1,unique_timestamps,62
2,exact_duplicate_raw_rows,0
3,malformed_rows,1
4,min_timestamp,2022-02-07 12:05:05
5,max_timestamp,2022-03-09 18:02:01


Structurally malformed row:


,timestamp,buy_box,visibility_order,condizione,venduto_da,spedito_da,num_valutazioni,valutazioni_positive,prezzo,qta_min,prezzo_prod_venduto(€),tipo_spedizione,prezzo_spedizione(€),prezzo_totale(€),dif_prezzo,dif_prezzo_sped,dif_prezzo_tot,g_cons_min,g_cons_max,g_spedizione,g_cons_vel_min,g_cons_vel_max,g_spedizione_vel,spedizione_consegna,condizione_usato,stelle,delta_consegna,delta_num_val,delta_val_pos(%),rapp_piu_basso,fba,vend_amazon,timestamp_dt,prezzo_totale_reconstructed,amazon_retail_from_name,amazon_warehouse_from_name,third_party_from_name,fba_from_shipper,seller_type_from_name,is_malformed_row,prezzo_totale_gap,prezzo_totale_gap_abs
8351,2022-03-06T18:01:33,0,177,<NA>,<NA>,<NA>,0,0,34.99,1,34.99,Gratuita,0.0,34.99,8.75,0.0,8.75,0,0,0,0,0,0,"Consegna GRATUITA venerdì, 11 marzo. Ordina entro 7 ore 12 min. Maggiori informazioni",<NA>,0.0,-2147483647,14485,100,1.33,False,False,2022-03-06 18:01:33,34.99,False,False,False,False,unclassified,True,0.0,0.0


### 6. Raw market structure, temporal coverage, and ranking consistency

Each raw timestamp behaves like a separate market: contiguous rank values, exactly one stored winner per timestamp, and no overlapping rank positions within the same timestamp. The cell checks these properties on the raw extraction and records the resulting diagnostics in the audit registry.


In [ ]:
# -----------------------------------------------------------------------------
# Section 6. Market structure, temporal coverage, and ranking consistency
# -----------------------------------------------------------------------------
# Describe the market structure of the raw extraction and verify that each timestamped market has a contiguous rank, exactly one stored winner, and no overlapping rank positions.

condition_distribution_raw = (
    df_raw['condizione']
    .value_counts(dropna=False)
    .rename_axis('condizione')
    .reset_index(name='rows')
)

raw_market_rank_audit = (
    df_raw.groupby('timestamp')
    .agg(
        n_rows=('visibility_order', 'size'),
        min_visibility_order=('visibility_order', 'min'),
        max_visibility_order=('visibility_order', 'max'),
        unique_visibility_order=('visibility_order', 'nunique'),
        buy_box_rows=('buy_box', lambda s: int(s.fillna(0).sum())),
    )
    .reset_index()
)

raw_market_rank_audit['contiguous_visibility_order'] = (
    raw_market_rank_audit['min_visibility_order'].eq(0) &
    raw_market_rank_audit['max_visibility_order'].eq(raw_market_rank_audit['n_rows'] - 1) &
    raw_market_rank_audit['unique_visibility_order'].eq(raw_market_rank_audit['n_rows'])
)

raw_market_rank_audit['winner_at_visibility_zero'] = (
    df_raw.loc[df_raw['buy_box'].eq(1)].groupby('timestamp')['visibility_order'].min().reindex(raw_market_rank_audit['timestamp']).fillna(np.nan).values == 0
)

raw_market_size_summary = (
    df_raw.groupby('timestamp').size().describe().rename('market_size_raw').to_frame().T
)

print('Condition distribution in the raw file:')
display(condition_distribution_raw)
print('Raw market size summary:')
display(raw_market_size_summary)
print('Raw rank-consistency audit (first 10 markets):')
display(raw_market_rank_audit.head(10))

register_check('6', 'all raw markets have contiguous visibility_order', raw_market_rank_audit['contiguous_visibility_order'].all(), f"non-contiguous markets = {int((~raw_market_rank_audit['contiguous_visibility_order']).sum())}")
register_check('6', 'all raw markets have exactly one stored buy_box row', raw_market_rank_audit['buy_box_rows'].eq(1).all(), f"markets failing = {int((~raw_market_rank_audit['buy_box_rows'].eq(1)).sum())}")
register_check('6', 'stored winner is always at visibility_order zero', raw_market_rank_audit['winner_at_visibility_zero'].all(), f"markets failing = {int((~raw_market_rank_audit['winner_at_visibility_zero']).sum())}")

Condition distribution in the raw file:


,condizione,rows
0,Nuovo,5786
1,Usato - Ottime condizioni,2860
2,Usato - Come nuovo,565
3,Usato - Condizioni accettabili,132
4,Usato - Buone condizioni,80
5,<NA>,1


Raw market size summary:


,count,mean,std,min,25%,50%,75%,max
market_size_raw,62.0,152.0,20.439435,119.0,130.5,158.0,169.0,183.0


Raw rank-consistency audit (first 10 markets):


,timestamp,n_rows,min_visibility_order,max_visibility_order,unique_visibility_order,buy_box_rows,contiguous_visibility_order,winner_at_visibility_zero
0,2022-02-07T12:05:05,122,0,121,122,1,True,True
1,2022-02-07T18:09:10,124,0,123,124,1,True,True
2,2022-02-08T12:07:00,122,0,121,122,1,True,True
3,2022-02-08T18:05:35,121,0,120,121,1,True,True
4,2022-02-09T12:08:19,120,0,119,120,1,True,True
5,2022-02-09T18:02:46,119,0,118,119,1,True,True
6,2022-02-10T12:02:37,119,0,118,119,1,True,True
7,2022-02-10T18:03:10,122,0,121,122,1,True,True
8,2022-02-11T12:02:48,120,0,119,120,1,True,True
9,2022-02-11T18:02:26,127,0,126,127,1,True,True


### 7. Seller flags, review fields, price arithmetic, and stored shipping-type consistency

The cell audits four core blocks in the full raw file:
1. seller-type reconstruction from the seller-name fields `venduto_da` and `spedito_da`,
2. review-field consistency between `num_valutazioni`, `valutazioni_positive`, and `stelle`,
3. arithmetic consistency between the stored `prezzo`, `prezzo_spedizione`, and `prezzo_totale` columns,
4. stored shipping-type labels relative to the reconstructed delivery and shipping fields.

The audit identifies the rows for which any stored convenience variable disagrees with the underlying raw text. These rows are flagged but not removed; the downstream econometric sample inherits the audit annotations.


In [ ]:
# -----------------------------------------------------------------------------
# Section 7. Seller flags, review fields, price arithmetic, and stored shipping labels
# -----------------------------------------------------------------------------
# Audit seller-type reconstruction from the seller-name fields, review-field consistency, arithmetic consistency of stored price columns, and stored shipping-type labels.

review_field_profile = (
    df_raw.groupby('seller_type_from_name')
    .agg(
        rows=('seller_type_from_name', 'size'),

        missing_review_count_rows=('num_valutazioni', lambda s: int(s.isna().sum())),
        zero_review_count_rows=('num_valutazioni', lambda s: int(s.fillna(np.nan).eq(0).sum())),

        missing_positive_rating_rows=('valutazioni_positive', lambda s: int(s.isna().sum())),
        zero_positive_rating_rows=('valutazioni_positive', lambda s: int(s.fillna(np.nan).eq(0).sum())),

        missing_stars_rows=('stelle', lambda s: int(s.isna().sum())),
        zero_stars_rows=('stelle', lambda s: int(s.fillna(np.nan).eq(0).sum())),
    )
    .reset_index()
)

shipping_free_positive_mask = (
    df_raw['tipo_spedizione'].eq('Gratuita') &
    df_raw['prezzo_spedizione(€)'].gt(0)
)

shipping_paid_zero_mask = (
    df_raw['tipo_spedizione'].eq('A Pagamento') &
    df_raw['prezzo_spedizione(€)'].fillna(0).eq(0)
)

shipping_basic_consistency = pd.DataFrame({
    'metric': [
        'qta_min_not_equal_to_1_rows',
        'prezzo_prod_venduto_not_equal_to_prezzo_rows',
        'stored_free_shipping_with_positive_price_rows',
        'stored_paid_shipping_with_zero_price_rows',
    ],
    'value': [
        int(df_raw['qta_min'].fillna(1).ne(1).sum()),
        int((df_raw['prezzo_prod_venduto(€)'] - df_raw['prezzo']).abs().gt(0.01).sum()),
        int(shipping_free_positive_mask.sum()),
        int(shipping_paid_zero_mask.sum()),
    ]
})

seller_flag_audit_raw = pd.DataFrame({
    'metric': [
        'stored_vend_amazon_true_rows',
        'amazon_retail_from_name_rows',
        'stored_fba_true_rows',
        'fba_from_shipper_rows',
        'amazon_warehouse_rows',
        'third_party_rows',
        'unclassified_rows',
    ],
    'value': [
        int(df_raw['vend_amazon'].fillna(False).sum()),
        int(df_raw['amazon_retail_from_name'].sum()),
        int(df_raw['fba'].fillna(False).sum()),
        int(df_raw['fba_from_shipper'].sum()),
        int(df_raw['amazon_warehouse_from_name'].sum()),
        int(df_raw['third_party_from_name'].sum()),
        int(df_raw['seller_type_from_name'].eq('unclassified').sum()),
    ]
})

paid_zero_profile = (
    df_raw.loc[shipping_paid_zero_mask]
    .groupby(['seller_type_from_name', 'fba_from_shipper'], dropna=False)
    .size()
    .rename('rows')
    .reset_index()
    .sort_values(['rows', 'seller_type_from_name'], ascending=[False, True])
)

paid_zero_by_seller = (
    df_raw.loc[shipping_paid_zero_mask]
    .groupby('venduto_da')
    .size()
    .rename('rows')
    .reset_index()
    .sort_values('rows', ascending=False)
)

paid_zero_sample = df_raw.loc[
    shipping_paid_zero_mask,
    [
        'timestamp', 'condizione', 'venduto_da', 'spedito_da',
        'tipo_spedizione', 'prezzo_spedizione(€)', 'spedizione_consegna'
    ]
].head(10)

print('Raw seller-flag audit:')
display(seller_flag_audit_raw)

print('Review-field profile by seller type (missing kept separate from zero):')
display(review_field_profile)

print('Redundancy and basic shipping-type consistency checks:')
display(shipping_basic_consistency)

print('Rows with stored "A Pagamento" but zero stored shipping price, profiled by seller type and FBA:')
display(paid_zero_profile)

print('Rows with stored "A Pagamento" but zero stored shipping price, by seller:')
display(paid_zero_by_seller.head(10))

print('Sample of rows with stored "A Pagamento" but zero stored shipping price:')
display(paid_zero_sample)

register_check(
    '7',
    'stored vend_amazon count matches seller-name reconstruction',
    int(df_raw['vend_amazon'].fillna(False).sum()) == int(df_raw['amazon_retail_from_name'].sum()),
    f"stored = {int(df_raw['vend_amazon'].fillna(False).sum())}, reconstructed = {int(df_raw['amazon_retail_from_name'].sum())}"
)

register_check(
    '7',
    'stored fba count matches shipper-name reconstruction',
    int(df_raw['fba'].fillna(False).sum()) == int(df_raw['fba_from_shipper'].sum()),
    f"stored = {int(df_raw['fba'].fillna(False).sum())}, reconstructed = {int(df_raw['fba_from_shipper'].sum())}"
)

register_check(
    '7',
    'qta_min is always equal to 1',
    int(df_raw['qta_min'].fillna(1).ne(1).sum()) == 0,
    f"rows failing = {int(df_raw['qta_min'].fillna(1).ne(1).sum())}"
)

register_check(
    '7',
    'prezzo_prod_venduto(€) equals prezzo everywhere',
    int((df_raw['prezzo_prod_venduto(€)'] - df_raw['prezzo']).abs().gt(0.01).sum()) == 0,
    f"rows failing = {int((df_raw['prezzo_prod_venduto(€)'] - df_raw['prezzo']).abs().gt(0.01).sum())}"
)

register_check(
    '7',
    'stored free shipping never has positive stored shipping price',
    int(shipping_free_positive_mask.sum()) == 0,
    f"rows failing = {int(shipping_free_positive_mask.sum())}"
)

register_check(
    '7',
    'stored paid shipping with zero stored price rows are flagged for later raw-text audit',
    True,
    f"rows flagged = {int(shipping_paid_zero_mask.sum())}"
)

Raw seller-flag audit:


,metric,value
0,stored_vend_amazon_true_rows,99
1,amazon_retail_from_name_rows,99
2,stored_fba_true_rows,4707
3,fba_from_shipper_rows,4707
4,amazon_warehouse_rows,3606
5,third_party_rows,5718
6,unclassified_rows,1


Review-field profile by seller type (missing kept separate from zero):


,seller_type_from_name,rows,missing_review_count_rows,zero_review_count_rows,missing_positive_rating_rows,zero_positive_rating_rows,missing_stars_rows,zero_stars_rows
0,amazon_retail,99,0,37,0,99,0.0,37.0
1,amazon_warehouse,3606,0,3606,0,3606,0.0,3606.0
2,third_party,5718,0,98,0,98,0.0,92.0
3,unclassified,1,0,1,0,1,0.0,1.0


Redundancy and basic shipping-type consistency checks:


...

,seller_type_from_name,fba_from_shipper,rows
0,third_party,False,98


Rows with stored "A Pagamento" but zero stored shipping price, by seller:


,venduto_da,rows
23,Topocentras EU,53
3,CAREBSRL-IT,6
9,HousePc,6
13,PRIME ITALIA,5
1,B&Bit Informatica,3
25,Vipdomo,2
17,Scontolo,2
5,Dagimarket,1
2,BuysVip,1
4,Computer Milano,1


Sample of rows with stored "A Pagamento" but zero stored shipping price:


,timestamp,condizione,venduto_da,spedito_da,tipo_spedizione,prezzo_spedizione(€),spedizione_consegna
54,2022-02-07T12:05:05,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 14 - 16 feb Maggiori informazioni
177,2022-02-07T18:09:10,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 14 - 16 feb Maggiori informazioni
298,2022-02-08T12:07:00,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 17 feb Maggiori informazioni
334,2022-02-08T12:07:00,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 16 feb Maggiori informazioni
421,2022-02-08T18:05:35,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 17 feb Maggiori informazioni
456,2022-02-08T18:05:35,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 16 feb Maggiori informazioni
542,2022-02-09T12:08:19,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 16 - 18 feb Maggiori informazioni
663,2022-02-09T18:02:46,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 16 - 18 feb Maggiori informazioni
780,2022-02-10T12:02:37,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 17 - 21 feb Maggiori informazioni
900,2022-02-10T18:03:10,Nuovo,Topocentras EU,Topocentras EU,A Pagamento,0.0,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 17 - 21 feb Maggiori informazioni


### 8. Delivery-source audit and parser verification

The delivery fields stored in the CSV are audited against the raw `spedizione_consegna` text. Two parsers are compared: the original upstream parser that produced `g_cons_min` and `g_cons_max`, and a robust parser introduced in this notebook that produces `g_cons_min_robust` and `g_cons_max_robust`. The cell quantifies parser failures on the markets where the original parser produced missing or implausible values and verifies that the robust parser repairs them. The robust parser is the variable used by the econometric sections in Part II.


In [ ]:
# -----------------------------------------------------------------------------
# Section 8. Delivery-source audit and parser verification
# -----------------------------------------------------------------------------
# Compare the upstream-parsed delivery fields with the source-text parsing and identify parser failures, then verify that the robust parser repairs the failed markets.

section8_temp_cols = df_raw.apply(
    lambda row: parse_delivery_original_style(row['spedizione_consegna'], row['timestamp_dt']),
    axis=1,
    result_type='expand'
)
section8_temp_cols.columns = [
    'g_cons_min_original_parser',
    'g_cons_max_original_parser',
    'g_cons_vel_min_original_parser',
    'g_cons_vel_max_original_parser'
]

df_raw = pd.concat([df_raw, section8_temp_cols], axis=1)

section8_temp_cols = df_raw.apply(
    lambda row: parse_delivery_robust(row['spedizione_consegna'], row['timestamp_dt']),
    axis=1,
    result_type='expand'
)
section8_temp_cols.columns = [
    'g_cons_min_robust',
    'g_cons_max_robust',
    'g_cons_vel_min_robust',
    'g_cons_vel_max_robust'
]

df_raw = pd.concat([df_raw, section8_temp_cols], axis=1)

df_raw['has_raw_delivery_text'] = (
    df_raw['spedizione_consegna'].notna() &
    df_raw['spedizione_consegna'].astype('string').str.strip().ne('')
)

df_raw['has_stored_delivery_signal'] = (
    df_raw[['g_cons_min', 'g_cons_max', 'g_cons_vel_min', 'g_cons_vel_max']].notna().any(axis=1) &
    ~df_raw[['g_cons_min', 'g_cons_max', 'g_cons_vel_min', 'g_cons_vel_max']].fillna(0).eq(0).all(axis=1)
)

df_raw['original_parser_has_signal'] = (
    df_raw[['g_cons_min_original_parser', 'g_cons_max_original_parser',
            'g_cons_vel_min_original_parser', 'g_cons_vel_max_original_parser']].notna().any(axis=1) &
    ~df_raw[['g_cons_min_original_parser', 'g_cons_max_original_parser',
             'g_cons_vel_min_original_parser', 'g_cons_vel_max_original_parser']].fillna(0).eq(0).all(axis=1)
)

df_raw['robust_parser_has_signal'] = (
    df_raw[['g_cons_min_robust', 'g_cons_max_robust',
            'g_cons_vel_min_robust', 'g_cons_vel_max_robust']].notna().any(axis=1) &
    ~df_raw[['g_cons_min_robust', 'g_cons_max_robust',
             'g_cons_vel_min_robust', 'g_cons_vel_max_robust']].fillna(0).eq(0).all(axis=1)
)

market_delivery_audit = (
    df_raw.loc[~df_raw['is_malformed_row']]
    .groupby('timestamp')
    .agg(
        rows=('timestamp', 'size'),
        raw_delivery_text_rows=('has_raw_delivery_text', 'sum'),
        stored_delivery_signal_rows=('has_stored_delivery_signal', 'sum'),
        original_parser_signal_rows=('original_parser_has_signal', 'sum'),
        robust_parser_signal_rows=('robust_parser_has_signal', 'sum'),
    )
    .reset_index()
)

failed_delivery_markets = market_delivery_audit.loc[
    market_delivery_audit['raw_delivery_text_rows'].eq(market_delivery_audit['rows']) &
    market_delivery_audit['stored_delivery_signal_rows'].eq(0)
].copy()

nonfailed_delivery_rows = df_raw.loc[
    ~df_raw['is_malformed_row'] &
    df_raw['timestamp'].isin(
        market_delivery_audit.loc[
            market_delivery_audit['stored_delivery_signal_rows'].gt(0), 'timestamp'
        ]
    )
].copy()

robust_vs_stored_delivery = []
for stored_col, robust_col in [
    ('g_cons_min', 'g_cons_min_robust'),
    ('g_cons_max', 'g_cons_max_robust'),
    ('g_cons_vel_min', 'g_cons_vel_min_robust'),
    ('g_cons_vel_max', 'g_cons_vel_max_robust'),
]:
    comparable = nonfailed_delivery_rows[stored_col].notna() & nonfailed_delivery_rows[robust_col].notna()
    mismatch_gt_1_day = comparable & (
        nonfailed_delivery_rows[stored_col] - nonfailed_delivery_rows[robust_col]
    ).abs().gt(1)

    robust_vs_stored_delivery.append({
        'field': stored_col,
        'comparable_rows': int(comparable.sum()),
        'rows_with_abs_gap_gt_1_day': int(mismatch_gt_1_day.sum()),
    })

robust_vs_stored_delivery = pd.DataFrame(robust_vs_stored_delivery)

failed_market_examples = df_raw.loc[
    df_raw['timestamp'].isin(failed_delivery_markets['timestamp']),
    [
        'timestamp', 'venduto_da', 'spedito_da', 'spedizione_consegna',
        'g_cons_min', 'g_cons_max',
        'g_cons_min_original_parser', 'g_cons_max_original_parser',
        'g_cons_min_robust', 'g_cons_max_robust'
    ]
].head(20)

print('Market-level delivery audit:')
display(market_delivery_audit)

print('Markets where stored delivery fields fail despite raw delivery text being present in every row:')
display(failed_delivery_markets)

print('Robust-versus-stored comparison on non-failed markets:')
display(robust_vs_stored_delivery)

print('Example rows from failed markets:')
display(failed_market_examples)

register_check(
    '8',
    'five delivery-failure markets are detected',
    int(len(failed_delivery_markets)) == 5,
    f"failed markets = {int(len(failed_delivery_markets))}"
)

register_check(
    '8',
    'failed delivery markets still contain raw delivery text in every row',
    failed_delivery_markets['raw_delivery_text_rows'].eq(failed_delivery_markets['rows']).all(),
    'all failed markets have full raw-text coverage'
)

register_check(
    '8',
    'original-style delivery parser fails on the failed markets',
    failed_delivery_markets['original_parser_signal_rows'].eq(0).all(),
    'no signal recovered by original-style parser'
)

register_check(
    '8',
    'robust delivery parser recovers signal in all failed markets',
    failed_delivery_markets['robust_parser_signal_rows'].eq(failed_delivery_markets['rows']).all(),
    'robust parser recovers every row in failed markets'
)

register_check(
    '8',
    'robust delivery parser reproduces stored non-failed markets within one day everywhere',
    robust_vs_stored_delivery['rows_with_abs_gap_gt_1_day'].eq(0).all(),
    'all audited fields have zero >1-day mismatches'
)

Market-level delivery audit:


,timestamp,rows,raw_delivery_text_rows,stored_delivery_signal_rows,original_parser_signal_rows,robust_parser_signal_rows
0,2022-02-07T12:05:05,122,122,122,101,122
1,2022-02-07T18:09:10,124,124,124,105,124
2,2022-02-08T12:07:00,122,122,122,103,122
3,2022-02-08T18:05:35,121,121,121,103,121
4,2022-02-09T12:08:19,120,120,120,102,120
5,2022-02-09T18:02:46,119,119,119,101,119
6,2022-02-10T12:02:37,119,119,119,102,119
7,2022-02-10T18:03:10,122,122,122,105,122
8,2022-02-11T12:02:48,120,120,120,103,120
9,2022-02-11T18:02:26,127,127,127,110,127


Markets where stored delivery fields fail despite raw delivery text being present in every row:


,timestamp,rows,raw_delivery_text_rows,stored_delivery_signal_rows,original_parser_signal_rows,robust_parser_signal_rows
51,2022-03-04T18:01:41,175,175,0,0,175
52,2022-03-05T12:01:52,179,179,0,0,179
55,2022-03-06T18:01:33,177,177,0,0,177
57,2022-03-07T18:01:43,176,176,0,0,176
61,2022-03-09T18:02:01,183,183,0,0,183


Robust-versus-stored comparison on non-failed markets:


,field,comparable_rows,rows_with_abs_gap_gt_1_day
0,g_cons_min,8533,0
1,g_cons_max,8533,0
2,g_cons_vel_min,8533,0
3,g_cons_vel_max,8533,0


Example rows from failed markets:


,timestamp,venduto_da,spedito_da,spedizione_consegna,g_cons_min,g_cons_max,g_cons_min_original_parser,g_cons_max_original_parser,g_cons_min_robust,g_cons_max_robust
7465,2022-03-04T18:01:41,Amazon,Amazon,"Consegna GRATUITA martedì, 8 marzo. Ordina entro 5 ore 58 min. Maggiori informazioni",0,0,NaN,NaN,4,4
7466,2022-03-04T18:01:41,Amazon Warehouse,Amazon,"Consegna GRATUITA giovedì, 10 marzo sul tuo primo ordine idoneo. Maggiori informazioni",0,0,NaN,NaN,6,6
7467,2022-03-04T18:01:41,Amazon,Amazon,"Consegna GRATUITA martedì, 8 marzo. Ordina entro 5 ore 58 min. Maggiori informazioni",0,0,NaN,NaN,4,4
7468,2022-03-04T18:01:41,ZaarioGmbH,Amazon,"Consegna GRATUITA giovedì, 10 marzo. Maggiori informazioni oppure consegna più rapida mercoledì, 9 marzo. Ordina entro 1 ora 13 min. Maggiori informazioni",0,0,NaN,NaN,6,6
7469,2022-03-04T18:01:41,SKY LINE E-COMM,Amazon,"Consegna GRATUITA giovedì, 10 marzo. Maggiori informazioni oppure consegna più rapida mercoledì, 9 marzo. Ordina entro 1 ora 13 min. Maggiori informazioni",0,0,NaN,NaN,6,6
7470,2022-03-04T18:01:41,Yulus,Yulus,Consegna GRATUITA 14 - 16 marzo. Maggiori informazioni,0,0,NaN,NaN,10,12
7471,2022-03-04T18:01:41,Welcome to Mall,Amazon,"Consegna GRATUITA mercoledì, 9 marzo. Ordina entro 3 ore 28 min. Maggiori informazioni",0,0,NaN,NaN,5,5
7472,2022-03-04T18:01:41,Miranda Electronics,Amazon,"Consegna GRATUITA martedì, 8 marzo. Ordina entro 5 ore 58 min. Maggiori informazioni",0,0,NaN,NaN,4,4
7473,2022-03-04T18:01:41,Bacom,Bacom,Consegna GRATUITA 14 - 17 marzo. Maggiori informazioni oppure consegna più rapida 11 - 15 marzo. Maggiori informazioni,0,0,NaN,NaN,10,13
7474,2022-03-04T18:01:41,GoPrice,GoPrice,Consegna GRATUITA 10 - 15 marzo. Maggiori informazioni,0,0,NaN,NaN,6,11


### 9. Shipping-price source audit and parser verification

A subset of rows is labelled `A Pagamento` (paid shipping) while storing `prezzo_spedizione(€) = 0`. The cell tests whether this inconsistency is a true data contradiction or whether the displayed shipping price genuinely equals zero in the raw text. The parser distinguishes three regimes for the paid-zero anomaly: explicit free-shipping disclosures embedded in the raw text, contact-courier offers for which the shipping price is unobservable at extraction time, and unclassified rows. The audit produces a deterministic repair of the shipping-price field, used downstream as `prezzo_spedizione_repaired`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 9. Shipping-price source audit and parser verification
# -----------------------------------------------------------------------------
# Parse shipping information from raw text, classify the paid-zero shipping anomaly into explicit-free, contact-courier, and unclassified regimes, and produce the deterministic repair of the shipping price.

shipping_text_parsed = df_raw['spedizione_consegna'].apply(parse_shipping_from_text).apply(pd.Series)
df_raw = pd.concat([df_raw, shipping_text_parsed], axis=1)

df_raw['stored_shipping_mode'] = np.select(
    [
        df_raw['tipo_spedizione'].eq('Gratuita'),
        df_raw['tipo_spedizione'].eq('A Pagamento'),
    ],
    [
        'free',
        'paid',
    ],
    default='missing'
)

df_raw['shipping_mode_matches_raw_text'] = np.where(
    df_raw['parsed_shipping_mode'].eq('free_explicit'),
    df_raw['stored_shipping_mode'].eq('free'),
    np.where(
        df_raw['parsed_shipping_mode'].eq('paid_explicit'),
        df_raw['stored_shipping_mode'].eq('paid'),
        pd.NA,
    )
)

df_raw['shipping_price_matches_raw_text'] = np.isclose(
    df_raw['prezzo_spedizione(€)'],
    df_raw['parsed_shipping_price'],
    atol=0.01,
    equal_nan=True,
)

explicit_paid_mask = df_raw['parsed_shipping_mode'].eq('paid_explicit')
explicit_free_mask = df_raw['parsed_shipping_mode'].eq('free_explicit')
contact_courier_mask = df_raw['parsed_shipping_mode'].eq('unknown_contact_courier')
unknown_other_mask = df_raw['parsed_shipping_mode'].eq('unknown_text')

paid_zero_anomalies = df_raw.loc[
    df_raw['tipo_spedizione'].eq('A Pagamento') &
    df_raw['prezzo_spedizione(€)'].fillna(0).eq(0)
].copy()

paid_zero_explicit_mask = paid_zero_anomalies['parsed_shipping_mode'].eq('paid_explicit')
paid_zero_contact_mask = paid_zero_anomalies['parsed_shipping_mode'].eq('unknown_contact_courier')
paid_zero_unknown_other_mask = paid_zero_anomalies['parsed_shipping_mode'].eq('unknown_text')

shipping_audit_summary = pd.DataFrame({
    'metric': [
        'rows_with_explicit_free_shipping_in_raw_text',
        'rows_with_explicit_paid_shipping_in_raw_text',
        'rows_with_unknown_contact_courier_shipping_in_raw_text',
        'rows_with_unclassified_other_shipping_text',
        'stored_free_shipping_with_positive_price_rows',
        'stored_paid_shipping_with_zero_price_rows',
        'explicit_paid_rows_with_stored_price_mismatch',
        'explicit_paid_rows_with_stored_price_match',
        'explicit_paid_rows_with_mode_mismatch',
        'explicit_free_rows_with_mode_mismatch',
    ],
    'value': [
        int(explicit_free_mask.sum()),
        int(explicit_paid_mask.sum()),
        int(contact_courier_mask.sum()),
        int(unknown_other_mask.sum()),
        int((df_raw['tipo_spedizione'].eq('Gratuita') & df_raw['prezzo_spedizione(€)'].gt(0)).sum()),
        int((df_raw['tipo_spedizione'].eq('A Pagamento') & df_raw['prezzo_spedizione(€)'].fillna(0).eq(0)).sum()),
        int((explicit_paid_mask & ~df_raw['shipping_price_matches_raw_text']).sum()),
        int((explicit_paid_mask & df_raw['shipping_price_matches_raw_text']).sum()),
        int((explicit_paid_mask & (df_raw['shipping_mode_matches_raw_text'] == False)).sum()),
        int((explicit_free_mask & (df_raw['shipping_mode_matches_raw_text'] == False)).sum()),
    ]
})

paid_zero_anomaly_breakdown = (
    paid_zero_anomalies.groupby('parsed_shipping_mode')
    .agg(
        rows=('timestamp', 'size'),
        timestamps=('timestamp', 'nunique'),
        sellers=('venduto_da', 'nunique'),
        fba_share=('fba_from_shipper', 'mean'),
    )
    .reset_index()
)

paid_zero_explicit_examples = paid_zero_anomalies.loc[
    paid_zero_explicit_mask,
    [
        'timestamp', 'venduto_da', 'spedito_da', 'tipo_spedizione',
        'prezzo_spedizione(€)', 'parsed_shipping_price', 'spedizione_consegna'
    ]
].head(20)

paid_zero_contact_examples = paid_zero_anomalies.loc[
    paid_zero_contact_mask,
    [
        'timestamp', 'venduto_da', 'spedito_da', 'tipo_spedizione',
        'prezzo_spedizione(€)', 'parsed_shipping_mode', 'spedizione_consegna'
    ]
].head(20)

paid_zero_explicit_by_timestamp = (
    paid_zero_anomalies.loc[paid_zero_explicit_mask]
    .groupby('timestamp')
    .size()
    .sort_values(ascending=False)
    .rename('rows')
    .reset_index()
    .head(10)
)

paid_zero_explicit_by_seller = (
    paid_zero_anomalies.loc[paid_zero_explicit_mask]
    .groupby('venduto_da')
    .size()
    .sort_values(ascending=False)
    .rename('rows')
    .reset_index()
    .head(15)
)

print('Shipping audit summary:')
display(shipping_audit_summary)

print('Breakdown of rows labelled A Pagamento but storing zero shipping price:')
display(paid_zero_anomaly_breakdown)

print('Examples where the raw text explicitly contains a positive shipping amount but the stored shipping price is zero:')
display(paid_zero_explicit_examples)

print('Examples where the raw text says that the courier will contact the buyer, so the numeric shipping amount cannot be recovered from text alone:')
display(paid_zero_contact_examples)

print('Explicit paid anomalies by timestamp (top 10):')
display(paid_zero_explicit_by_timestamp)

print('Explicit paid anomalies by seller (top 15):')
display(paid_zero_explicit_by_seller)

register_check(
    '9',
    'stored free shipping never has a positive stored price',
    int((df_raw['tipo_spedizione'].eq('Gratuita') & df_raw['prezzo_spedizione(€)'].gt(0)).sum()) == 0,
    'no free-label rows carry positive stored shipping price'
)

register_check(
    '9',
    'raw explicit paid shipping never contradicts the stored paid/free mode',
    int((explicit_paid_mask & (df_raw['shipping_mode_matches_raw_text'] == False)).sum()) == 0,
    'mode agrees whenever raw text is explicit'
)

register_check(
    '9',
    'raw explicit free shipping never contradicts the stored paid/free mode',
    int((explicit_free_mask & (df_raw['shipping_mode_matches_raw_text'] == False)).sum()) == 0,
    'mode agrees whenever raw text is explicit'
)

register_check(
    '9',
    'shipping parser leaves no residual unknown-text cases in the raw file',
    int(unknown_other_mask.sum()) == 0,
    f"unknown_text rows = {int(unknown_other_mask.sum())}"
)

register_check(
    '9',
    'paid-zero anomalies are fully classified into explicit-price versus contact-courier cases',
    int(paid_zero_unknown_other_mask.sum()) == 0 and
    int(paid_zero_explicit_mask.sum() + paid_zero_contact_mask.sum()) == int(len(paid_zero_anomalies)),
    (
        f"explicit = {int(paid_zero_explicit_mask.sum())}, "
        f"contact_courier = {int(paid_zero_contact_mask.sum())}, "
        f"unknown_other = {int(paid_zero_unknown_other_mask.sum())}, "
        f"total = {int(len(paid_zero_anomalies))}"
    )
)

register_check(
    '9',
    'explicit paid shipping price mismatches are exactly the explicit paid zero-price anomalies',
    int((explicit_paid_mask & ~df_raw['shipping_price_matches_raw_text']).sum()) == int(paid_zero_explicit_mask.sum()),
    (
        f"all explicit mismatches = {int((explicit_paid_mask & ~df_raw['shipping_price_matches_raw_text']).sum())}, "
        f"paid-zero explicit anomalies = {int(paid_zero_explicit_mask.sum())}"
    )
)

Shipping audit summary:


,metric,value
0,rows_with_explicit_free_shipping_in_raw_text,7536
1,rows_with_explicit_paid_shipping_in_raw_text,1835
2,rows_with_unknown_contact_courier_shipping_in_raw_text,53
3,rows_with_unclassified_other_shipping_text,0
4,stored_free_shipping_with_positive_price_rows,0
5,stored_paid_shipping_with_zero_price_rows,98
6,explicit_paid_rows_with_stored_price_mismatch,45
7,explicit_paid_rows_with_stored_price_match,1790
8,explicit_paid_rows_with_mode_mismatch,0
9,explicit_free_rows_with_mode_mismatch,0


Breakdown of rows labelled A Pagamento but storing zero shipping price:


,parsed_shipping_mode,rows,timestamps,sellers,fba_share
0,paid_explicit,45,6,27,0.0
1,unknown_contact_courier,53,50,1,0.0


Examples where the raw text explicitly contains a positive shipping amount but the stored shipping price is zero:


...

,timestamp,venduto_da,spedito_da,tipo_spedizione,prezzo_spedizione(€),parsed_shipping_mode,spedizione_consegna
54,2022-02-07T12:05:05,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 14 - 16 feb Maggiori informazioni
177,2022-02-07T18:09:10,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 14 - 16 feb Maggiori informazioni
298,2022-02-08T12:07:00,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 17 feb Maggiori informazioni
334,2022-02-08T12:07:00,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 16 feb Maggiori informazioni
421,2022-02-08T18:05:35,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 17 feb Maggiori informazioni
456,2022-02-08T18:05:35,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 15 - 16 feb Maggiori informazioni
542,2022-02-09T12:08:19,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 16 - 18 feb Maggiori informazioni
663,2022-02-09T18:02:46,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 16 - 18 feb Maggiori informazioni
780,2022-02-10T12:02:37,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 17 - 21 feb Maggiori informazioni
900,2022-02-10T18:03:10,Topocentras EU,Topocentras EU,A Pagamento,0.0,unknown_contact_courier,Verrai contattato dal corriere per fissare una data di consegna. Consegna più veloce: 17 - 21 feb Maggiori informazioni


Explicit paid anomalies by timestamp (top 10):


,timestamp,rows
0,2022-02-17T18:08:52,28
1,2022-03-09T18:02:01,4
2,2022-03-07T18:01:43,4
3,2022-03-04T18:01:41,3
4,2022-03-06T18:01:33,3
5,2022-03-05T12:01:52,3


Explicit paid anomalies by seller (top 15):


,venduto_da,rows
0,CAREBSRL-IT,6
1,HousePc,6
2,PRIME ITALIA,5
3,B&Bit Informatica,3
4,Scontolo,2
5,Vipdomo,2
6,BuysVip,1
7,Flash Shopping,1
8,All In Commerce,1
9,Dagimarket,1


### 10. Unit-of-analysis audit: repeated sellers, duplicate offers, and candidate diagnostic samples

The raw `Nuovo` sample is ambiguous between an offer-level and a seller-level unit of analysis. A single seller can appear multiple times in the same timestamped market with different offer rows, and the same offer can appear repeated across timestamps with no economic change. The cell quantifies repeated-seller exposure, exact-duplicate offers, and the candidate analytical samples evaluated before the final panel is fixed:
- the raw offer sample at the `Nuovo` level,
- the strict-deduplicated offer sample,
- the relaxed-deduplicated offer sample,
- the seller-best sample (one row per seller-market cell).

The seller-best third-party version of this last sample is the panel adopted by the econometric analysis in Part II.


In [ ]:
# -----------------------------------------------------------------------------
# Section 10. Unit-of-analysis audit and candidate diagnostic samples
# -----------------------------------------------------------------------------
# Quantify repeated sellers, duplicate offers, and the candidate analytical samples evaluated before the final econometric panel is fixed.

df_new_offer_raw = df_raw.loc[
    ~df_raw['is_malformed_row'] & df_raw['condizione'].eq('Nuovo')
].copy()

df_new_offer_raw = add_ranks(df_new_offer_raw)

df_new_offer_raw['strict_offer_signature'] = build_row_signature(
    df_new_offer_raw,
    [
        'timestamp', 'condizione', 'venduto_da', 'spedito_da',
        'prezzo', 'prezzo_spedizione(€)', 'spedizione_consegna',
        'num_valutazioni', 'valutazioni_positive', 'stelle'
    ]
)

df_new_offer_raw['relaxed_offer_signature'] = build_row_signature(
    df_new_offer_raw,
    [
        'timestamp', 'condizione', 'venduto_da', 'spedito_da',
        'prezzo', 'prezzo_spedizione(€)', 'spedizione_consegna'
    ]
)

seller_dup_groups = (
    df_new_offer_raw.groupby(['timestamp', 'venduto_da'])
    .size()
    .rename('rows_in_group')
    .reset_index()
)

seller_dup_groups = seller_dup_groups.loc[seller_dup_groups['rows_in_group'].gt(1)].copy()

strict_dup_groups = (
    df_new_offer_raw.groupby('strict_offer_signature')
    .size()
    .rename('rows_in_group')
    .reset_index()
)
strict_dup_groups = strict_dup_groups.loc[strict_dup_groups['rows_in_group'].gt(1)].copy()

relaxed_dup_groups = (
    df_new_offer_raw.groupby('relaxed_offer_signature')
    .size()
    .rename('rows_in_group')
    .reset_index()
)
relaxed_dup_groups = relaxed_dup_groups.loc[relaxed_dup_groups['rows_in_group'].gt(1)].copy()

markets_with_repeated_seller = int(seller_dup_groups['timestamp'].nunique())
seller_excess_rows_raw = int(len(df_new_offer_raw) - df_new_offer_raw.groupby(['timestamp', 'venduto_da']).ngroups)

winner_relaxed_signature = df_new_offer_raw.loc[
    df_new_offer_raw['buy_box'].eq(1),
    ['timestamp', 'relaxed_offer_signature']
].rename(columns={'relaxed_offer_signature': 'winner_relaxed_signature'})

winner_relaxed_dup_count = (
    df_new_offer_raw[['timestamp', 'relaxed_offer_signature']]
    .merge(winner_relaxed_signature, on='timestamp', how='left')
    .assign(is_winner_duplicate=lambda d: d['relaxed_offer_signature'].eq(d['winner_relaxed_signature']))
    .groupby('timestamp')['is_winner_duplicate']
    .sum()
    .rename('winner_relaxed_offer_rows')
    .reset_index()
)

unit_of_analysis_summary = pd.DataFrame({
    'metric': [
        'rows_in_nuovo_offer_raw',
        'timestamps_in_nuovo_offer_raw',
        'markets_with_repeated_seller',
        'timestamp_seller_groups_with_repeated_seller',
        'seller_excess_rows_beyond_one_per_timestamp_seller',
        'strict_duplicate_offer_groups',
        'rows_in_strict_duplicate_offer_groups',
        'relaxed_duplicate_offer_groups',
        'rows_in_relaxed_duplicate_offer_groups',
        'timestamps_with_relaxed_duplicate_winning_offer',
    ],
    'value': [
        int(len(df_new_offer_raw)),
        int(df_new_offer_raw['timestamp'].nunique()),
        markets_with_repeated_seller,
        int(len(seller_dup_groups)),
        seller_excess_rows_raw,
        int(len(strict_dup_groups)),
        int(strict_dup_groups['rows_in_group'].sum()),
        int(len(relaxed_dup_groups)),
        int(relaxed_dup_groups['rows_in_group'].sum()),
        int((winner_relaxed_dup_count['winner_relaxed_offer_rows'] > 1).sum()),
    ]
})

df_new_offer_strict_dedup = df_new_offer_raw.drop_duplicates(subset=['strict_offer_signature']).copy()
df_new_offer_strict_dedup = add_ranks(df_new_offer_strict_dedup)

df_new_offer_relaxed_dedup = df_new_offer_raw.drop_duplicates(subset=['relaxed_offer_signature']).copy()
df_new_offer_relaxed_dedup = add_ranks(df_new_offer_relaxed_dedup)

df_new_seller_best = (
    df_new_offer_strict_dedup
    .sort_values(['timestamp', 'visibility_order', 'prezzo_totale_reconstructed', 'venduto_da', 'spedito_da'])
    .groupby(['timestamp', 'venduto_da'], as_index=False)
    .first()
    .copy()
)
df_new_seller_best = add_ranks(df_new_seller_best)

candidate_sample_summary = pd.concat([
    summarize_candidate_sample(df_new_offer_raw, 'nuovo_offer_raw'),
    summarize_candidate_sample(df_new_offer_strict_dedup, 'nuovo_offer_strict_dedup'),
    summarize_candidate_sample(df_new_offer_relaxed_dedup, 'nuovo_offer_relaxed_dedup'),
    summarize_candidate_sample(df_new_seller_best, 'nuovo_seller_best'),
], ignore_index=True)

candidate_sample_summary['seller_duplicate_rows'] = [
    int(len(df_new_offer_raw) - df_new_offer_raw.groupby(['timestamp', 'venduto_da']).ngroups),
    int(len(df_new_offer_strict_dedup) - df_new_offer_strict_dedup.groupby(['timestamp', 'venduto_da']).ngroups),
    int(len(df_new_offer_relaxed_dedup) - df_new_offer_relaxed_dedup.groupby(['timestamp', 'venduto_da']).ngroups),
    int(len(df_new_seller_best) - df_new_seller_best.groupby(['timestamp', 'venduto_da']).ngroups),
]

candidate_sample_summary['strict_duplicate_rows'] = [
    int(len(df_new_offer_raw) - df_new_offer_raw.drop_duplicates(subset=['strict_offer_signature']).shape[0]),
    0,
    0,
    0,
]

candidate_sample_summary['relaxed_duplicate_rows'] = [
    int(len(df_new_offer_raw) - df_new_offer_raw.drop_duplicates(subset=['relaxed_offer_signature']).shape[0]),
    int(len(df_new_offer_strict_dedup) - df_new_offer_strict_dedup.drop_duplicates(subset=['relaxed_offer_signature']).shape[0]),
    0,
    0,
]

example_dup_groups = (
    seller_dup_groups
    .sort_values(['timestamp', 'venduto_da'])
    .head(10)[['timestamp', 'venduto_da']]
)

repeated_seller_examples = (
    df_new_offer_raw.merge(example_dup_groups, on=['timestamp', 'venduto_da'], how='inner')
    .sort_values(['timestamp', 'venduto_da', 'rank_pos'])
    [
        [
            'timestamp', 'rank_pos', 'buy_box', 'venduto_da', 'spedito_da',
            'prezzo', 'prezzo_spedizione(€)', 'num_valutazioni',
            'valutazioni_positive', 'spedizione_consegna'
        ]
    ]
    .head(20)
)

print('Unit-of-analysis summary for the Nuovo sample:')
display(unit_of_analysis_summary)

print('Candidate diagnostic samples:')
display(candidate_sample_summary)

print('Examples of repeated sellers within the same timestamp market:')
display(repeated_seller_examples)

register_check(
    '10',
    'the Nuovo raw-offer sample spans 62 markets',
    int(df_new_offer_raw['timestamp'].nunique()) == 62,
    f"markets = {int(df_new_offer_raw['timestamp'].nunique())}"
)

register_check(
    '10',
    'every Nuovo market contains at least one repeated seller',
    markets_with_repeated_seller == 62,
    f"markets with repeated seller = {markets_with_repeated_seller}"
)

register_check(
    '10',
    'seller_best removes all repeated sellers within market',
    int(len(df_new_seller_best) - df_new_seller_best.groupby(['timestamp', 'venduto_da']).ngroups) == 0,
    'seller_best has one row per seller per timestamp'
)

Unit-of-analysis summary for the Nuovo sample:


,metric,value
0,rows_in_nuovo_offer_raw,5786
1,timestamps_in_nuovo_offer_raw,62
2,markets_with_repeated_seller,62
3,timestamp_seller_groups_with_repeated_seller,442
4,seller_excess_rows_beyond_one_per_timestamp_seller,617
5,strict_duplicate_offer_groups,24
6,rows_in_strict_duplicate_offer_groups,48
7,relaxed_duplicate_offer_groups,56
8,rows_in_relaxed_duplicate_offer_groups,112
9,timestamps_with_relaxed_duplicate_winning_offer,32


Candidate diagnostic samples:


,sample,rows,markets,mean_market_size,min_market_size,max_market_size,mean_unique_sellers_per_market,seller_duplicate_rows,strict_duplicate_rows,relaxed_duplicate_rows
0,nuovo_offer_raw,5786,62,93.322581,78,111,83.370968,617,24,56
1,nuovo_offer_strict_dedup,5762,62,92.935484,77,111,83.370968,593,0,32
2,nuovo_offer_relaxed_dedup,5730,62,92.419355,77,111,83.370968,561,0,0
3,nuovo_seller_best,5169,62,83.370968,73,94,83.370968,0,0,0


Examples of repeated sellers within the same timestamp market:


,timestamp,rank_pos,buy_box,venduto_da,spedito_da,prezzo,prezzo_spedizione(€),num_valutazioni,valutazioni_positive,spedizione_consegna
4,2022-02-07T12:05:05,56,0,Happy Home Srl,Happy Home Srl,59.0,0.0,932,95,Spedizione GRATUITA: 10 - 14 feb Maggiori informazioni
10,2022-02-07T12:05:05,83,0,Happy Home Srl,Happy Home Srl,68.0,0.0,932,95,Spedizione GRATUITA: 10 - 14 feb Maggiori informazioni
13,2022-02-07T12:05:05,87,0,Happy Home Srl,Happy Home Srl,72.0,0.0,932,95,Spedizione GRATUITA: 10 - 11 feb Maggiori informazioni
5,2022-02-07T12:05:05,63,0,INSTYLEANDJOY,INSTYLEANDJOY,60.99,0.0,10,50,Spedizione GRATUITA: 18 - 23 feb Maggiori informazioni
9,2022-02-07T12:05:05,82,0,INSTYLEANDJOY,INSTYLEANDJOY,61.99,0.0,10,50,Spedizione GRATUITA: 22 feb - 2 mar Maggiori informazioni
12,2022-02-07T12:05:05,85,0,INSTYLEANDJOY,INSTYLEANDJOY,70.99,0.0,10,50,Spedizione GRATUITA: 22 feb - 2 mar Maggiori informazioni
0,2022-02-07T12:05:05,2,0,SKY LINE E-COMM,Amazon,38.5,0.0,17,94,"Spedizione GRATUITA: martedì, 15 feb Maggiori informazioni"
6,2022-02-07T12:05:05,79,0,SKY LINE E-COMM,SKY LINE E-COMM,42.0,0.0,17,94,Spedizione GRATUITA: 15 - 18 feb Maggiori informazioni
3,2022-02-07T12:05:05,50,0,Scontolo,Scontolo,50.0,5.99,2420,83,"Consegna a 5,99 € : 15 - 17 feb Maggiori informazioni Consegna più veloce: 14 - 16 feb Maggiori informazioni"
8,2022-02-07T12:05:05,81,0,Scontolo,Scontolo,54.0,5.99,2420,83,"Consegna a 5,99 € : 10 - 14 feb Maggiori informazioni Consegna più veloce: 9 - 11 feb Maggiori informazioni"


### 11. Stored convenience variables under the raw reference and candidate samples

Several columns in the raw CSV are stored as market-relative convenience variables, computed upstream against the raw `Nuovo` reference set. The cell recomputes those convenience variables inside each candidate sample to verify whether their values remain interpretable once the sample changes. The output is the convenience-audit table that records, for each candidate sample, which stored convenience variable can be reused as-is and which one is no longer interpretable after sample restriction.


In [ ]:
# -----------------------------------------------------------------------------
# Section 11. Recomputation of stored convenience variables across candidate samples
# -----------------------------------------------------------------------------
# Recompute the stored market-relative convenience variables inside each candidate sample and check whether they remain interpretable after sample restriction.

def convenience_audit(sample, sample_name):
    reference = (
        sample.groupby('timestamp')
        .agg(
            min_prezzo=('prezzo', 'min'),
            min_ship=('prezzo_spedizione(€)', 'min'),
            min_total=('prezzo_totale_reconstructed', 'min'),
            max_num_val=('num_valutazioni', 'max'),
            max_val_pos=('valutazioni_positive', 'max'),
        )
        .reset_index()
    )

    d = sample.merge(reference, on='timestamp', how='left')

    specs = [
        ('dif_prezzo', d['prezzo'] - d['min_prezzo']),
        ('dif_prezzo_sped', d['prezzo_spedizione(€)'] - d['min_ship']),
        ('dif_prezzo_tot', d['prezzo_totale_reconstructed'] - d['min_total']),
        ('delta_num_val', d['max_num_val'] - d['num_valutazioni']),
        ('delta_val_pos(%)', d['max_val_pos'] - d['valutazioni_positive']),
        ('rapp_piu_basso', d['prezzo_totale_reconstructed'] / d['min_total']),
    ]

    rows = []
    for metric, recomputed in specs:
        match = np.isclose(d[metric], recomputed, atol=0.01, equal_nan=True)
        rows.append({
            'sample': sample_name,
            'metric': metric,
            'match_rows': int(match.sum()),
            'rows': int(len(d)),
            'mismatch_rows': int((~match).sum()),
            'share_match': float(match.mean()),
        })
    return pd.DataFrame(rows)

convenience_audit_table = pd.concat([
    convenience_audit(df_raw.loc[~df_raw['is_malformed_row']].copy(), 'raw_non_malformed'),
    convenience_audit(df_new_offer_raw.copy(), 'nuovo_offer_raw'),
    convenience_audit(df_new_offer_strict_dedup.copy(), 'nuovo_offer_strict_dedup'),
    convenience_audit(df_new_offer_relaxed_dedup.copy(), 'nuovo_offer_relaxed_dedup'),
    convenience_audit(df_new_seller_best.copy(), 'nuovo_seller_best'),
], ignore_index=True)

convenience_takeaway = (
    convenience_audit_table.pivot(index='metric', columns='sample', values='share_match')
    .reset_index()
)

candidate_samples = [
    'nuovo_offer_raw',
    'nuovo_offer_strict_dedup',
    'nuovo_offer_relaxed_dedup',
    'nuovo_seller_best'
]

def classify_metric(row):
    candidate_values = [row[s] for s in candidate_samples]
    if all(np.isclose(v, 1.0) for v in candidate_values):
        return 'stable across audited candidate samples'
    if all(np.isclose(v, 0.0) for v in candidate_values):
        return 'invalid after sample redefinition'
    return 'partially sample-sensitive'

convenience_takeaway['candidate_sample_status'] = convenience_takeaway.apply(classify_metric, axis=1)

print('Convenience-variable validity across the raw reference and candidate samples:')
display(convenience_audit_table)

print('Metric-level takeaway:')
display(convenience_takeaway)

register_check(
    '11',
    'dif_prezzo becomes invalid after the Nuovo restriction',
    convenience_audit_table.query("sample == 'nuovo_offer_raw' and metric == 'dif_prezzo'")['share_match'].iloc[0] == 0.0,
    'stored dif_prezzo is sample-dependent'
)

register_check(
    '11',
    'dif_prezzo_tot becomes invalid after the Nuovo restriction',
    convenience_audit_table.query("sample == 'nuovo_offer_raw' and metric == 'dif_prezzo_tot'")['share_match'].iloc[0] == 0.0,
    'stored dif_prezzo_tot is sample-dependent'
)

register_check(
    '11',
    'rapp_piu_basso becomes invalid after the Nuovo restriction',
    convenience_audit_table.query("sample == 'nuovo_offer_raw' and metric == 'rapp_piu_basso'")['share_match'].iloc[0] == 0.0,
    'stored rapp_piu_basso is sample-dependent'
)

register_check(
    '11',
    'dif_prezzo_sped remains valid across all audited candidate samples',
    convenience_takeaway.loc[convenience_takeaway['metric'].eq('dif_prezzo_sped'), 'candidate_sample_status'].iloc[0] == 'stable across audited candidate samples',
    'shipping-price minimum is unchanged across these candidate samples'
)

register_check(
    '11',
    'delta_num_val remains valid across all audited candidate samples',
    convenience_takeaway.loc[convenience_takeaway['metric'].eq('delta_num_val'), 'candidate_sample_status'].iloc[0] == 'stable across audited candidate samples',
    'max review count is unchanged across these candidate samples'
)

register_check(
    '11',
    'delta_val_pos(%) remains valid across all audited candidate samples',
    convenience_takeaway.loc[convenience_takeaway['metric'].eq('delta_val_pos(%)'), 'candidate_sample_status'].iloc[0] == 'stable across audited candidate samples',
    'max positive rating is unchanged across these candidate samples'
)

register_check(
    '11',
    'raw reference already shows non-exactness for some stored convenience variables',
    (
        convenience_audit_table.query("sample == 'raw_non_malformed' and metric == 'dif_prezzo_tot'")['share_match'].iloc[0] < 1.0 and
        convenience_audit_table.query("sample == 'raw_non_malformed' and metric == 'rapp_piu_basso'")['share_match'].iloc[0] < 1.0
    ),
    'not all stored convenience variables are exact even before sample redesign'
)

Convenience-variable validity across the raw reference and candidate samples:


,sample,metric,match_rows,rows,mismatch_rows,share_match
0,raw_non_malformed,dif_prezzo,9423,9423,0,1.000000
1,raw_non_malformed,dif_prezzo_sped,9423,9423,0,1.000000
2,raw_non_malformed,dif_prezzo_tot,9321,9423,102,0.989175
3,raw_non_malformed,delta_num_val,9417,9423,6,0.999363
4,raw_non_malformed,delta_val_pos(%),9417,9423,6,0.999363
5,raw_non_malformed,rapp_piu_basso,9390,9423,33,0.996498
6,nuovo_offer_raw,dif_prezzo,0,5786,5786,0.000000
7,nuovo_offer_raw,dif_prezzo_sped,5786,5786,0,1.000000
8,nuovo_offer_raw,dif_prezzo_tot,0,5786,5786,0.000000
9,nuovo_offer_raw,delta_num_val,5786,5786,0,1.000000


Metric-level takeaway:


sample,metric,nuovo_offer_raw,nuovo_offer_relaxed_dedup,nuovo_offer_strict_dedup,nuovo_seller_best,raw_non_malformed,candidate_sample_status
0,delta_num_val,1.0,1.0,1.0,1.0,0.999363,stable across audited candidate samples
1,delta_val_pos(%),1.0,1.0,1.0,1.0,0.999363,stable across audited candidate samples
2,dif_prezzo,0.0,0.0,0.0,0.0,1.000000,invalid after sample redefinition
3,dif_prezzo_sped,1.0,1.0,1.0,1.0,1.000000,stable across audited candidate samples
4,dif_prezzo_tot,0.0,0.0,0.0,0.0,0.989175,invalid after sample redefinition
5,rapp_piu_basso,0.0,0.0,0.0,0.0,0.996498,invalid after sample redefinition


### 12. Outcome support under third-party candidate ranking samples

The cell quantifies which ranking outcomes have empirical support after the market is restricted to `Nuovo` and the seller list is further restricted to third-party sellers. It reports market-size distributions, normalized-rank distributions, and the share of observations falling in the top-5, top-10, top-15, and top-20 positions. The diagnostic motivates using the full normalized rank as the primary outcome and treating top-k indicators as auxiliary prominence translations.


In [ ]:
# -----------------------------------------------------------------------------
# Section 12. Outcome support under third-party candidate ranking samples
# -----------------------------------------------------------------------------
# Measure market-size and outcome support after third-party restriction and rank recomputation.

def prepare_third_party_candidate(sample):
    candidate = sample.loc[sample['third_party_from_name']].copy()
    candidate = add_ranks(candidate)
    candidate['best_third_party'] = candidate['rank_pos'].eq(1)
    return candidate


df_new_offer_raw_thirdparty = prepare_third_party_candidate(df_new_offer_raw)
df_new_offer_strict_dedup_thirdparty = prepare_third_party_candidate(df_new_offer_strict_dedup)
df_new_offer_relaxed_dedup_thirdparty = prepare_third_party_candidate(df_new_offer_relaxed_dedup)
df_new_seller_best_thirdparty = prepare_third_party_candidate(df_new_seller_best)


def build_outcome_support(sample, sample_name):
    sample = sample.copy()

    if sample.empty:
        return pd.DataFrame()

    rows = []
    for outcome_name, indicator in [
        ('stored_winner_flag', sample['buy_box'].fillna(0).eq(1)),
        ('best_third_party', sample['best_third_party'].astype(bool)),
        ('top_3', sample['rank_pos'].le(3)),
        ('top_5', sample['rank_pos'].le(5)),
        ('top_10', sample['rank_pos'].le(10)),
    ]:
        indicator = indicator.astype(bool)
        positive_rows = int(indicator.sum())

        market_counts = (
            sample.assign(indicator=indicator)
            .groupby('timestamp')['indicator']
            .sum()
        )

        markets_total = int(sample['timestamp'].nunique())
        markets_with_positive = int(market_counts.gt(0).sum())
        max_positive_rows_in_market = int(market_counts.max()) if len(market_counts) > 0 else 0

        rows.append({
            'sample': sample_name,
            'outcome': outcome_name,
            'positive_rows': positive_rows,
            'total_rows': int(len(sample)),
            'positive_share': float(indicator.mean()),
            'markets_total': markets_total,
            'markets_with_positive': markets_with_positive,
            'market_support_share': float(markets_with_positive / markets_total),
            'max_positive_rows_in_market': max_positive_rows_in_market,
            'fba_share_among_positive': np.nan if positive_rows == 0 else float(sample.loc[indicator, 'fba_from_shipper'].mean()),
        })

    return pd.DataFrame(rows)


outcome_support_table = pd.concat([
    build_outcome_support(df_new_offer_raw_thirdparty, 'nuovo_offer_raw_thirdparty'),
    build_outcome_support(df_new_offer_strict_dedup_thirdparty, 'nuovo_offer_strict_dedup_thirdparty'),
    build_outcome_support(df_new_offer_relaxed_dedup_thirdparty, 'nuovo_offer_relaxed_dedup_thirdparty'),
    build_outcome_support(df_new_seller_best_thirdparty, 'nuovo_seller_best_thirdparty'),
], ignore_index=True)

print('Outcome support table in the reranked third-party segment:')
display(outcome_support_table)

register_check(
    '12',
    'stored auxiliary winner flag has zero support in every candidate sample',
    outcome_support_table.query("outcome == 'stored_winner_flag'")['positive_rows'].eq(0).all(),
    'no third-party row carries the stored auxiliary winner flag after final restriction'
)

register_check(
    '12',
    'best_third_party is supported in every market',
    (
        outcome_support_table.query("outcome == 'best_third_party'")['positive_rows'].eq(62).all() and
        outcome_support_table.query("outcome == 'best_third_party'")['markets_with_positive'].eq(62).all()
    ),
    'one rank-1 third-party seller exists in each market after reranking'
)

Outcome support table in the reranked third-party segment:


,sample,outcome,positive_rows,total_rows,positive_share,markets_total,markets_with_positive,market_support_share,max_positive_rows_in_market,fba_share_among_positive
0,nuovo_offer_raw_thirdparty,stored_winner_flag,0,5687,0.000000,62,0,0.0,0,NaN
1,nuovo_offer_raw_thirdparty,best_third_party,62,5687,0.010902,62,62,1.0,1,0.870968
2,nuovo_offer_raw_thirdparty,top_3,186,5687,0.032706,62,62,1.0,3,0.591398
3,nuovo_offer_raw_thirdparty,top_5,310,5687,0.054510,62,62,1.0,5,0.545161
4,nuovo_offer_raw_thirdparty,top_10,620,5687,0.109021,62,62,1.0,10,0.567742
5,nuovo_offer_strict_dedup_thirdparty,stored_winner_flag,0,5663,0.000000,62,0,0.0,0,NaN
6,nuovo_offer_strict_dedup_thirdparty,best_third_party,62,5663,0.010948,62,62,1.0,1,0.870968
7,nuovo_offer_strict_dedup_thirdparty,top_3,186,5663,0.032845,62,62,1.0,3,0.591398
8,nuovo_offer_strict_dedup_thirdparty,top_5,310,5663,0.054741,62,62,1.0,5,0.545161
9,nuovo_offer_strict_dedup_thirdparty,top_10,620,5663,0.109483,62,62,1.0,10,0.567742


### 13. Covariate overlap and rank distributions in the preferred third-party seller-level sample

In the third-party seller-level sample, the cell summarizes covariate distributions and rank distributions separately by FBA status. It reports the mean and median of each covariate, the standardized mean difference between FBA and non-FBA rows, and the rank-position quantiles by group. These descriptive numbers populate the balance and rank-summary tables of Chapter 3 of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 13. Covariate overlap and rank distributions in the preferred third-party seller sample
# -----------------------------------------------------------------------------
# Summarize covariate distributions, standardized mean differences, and rank distributions by FBA status in the third-party seller-level sample.

df_preferred = df_new_seller_best_thirdparty.copy()
third_party_preferred = df_preferred.copy()

third_party_preferred['log1p_num_valutazioni'] = np.log1p(third_party_preferred['num_valutazioni'])

covariate_vars = [
    'prezzo_totale_reconstructed',
    'prezzo_spedizione(€)',
    'g_cons_min_robust',
    'num_valutazioni',
    'log1p_num_valutazioni',
    'valutazioni_positive',
    'stelle',
]

rank_vars = [
    'rank_pos',
    'rank_pct',
]

group_size_table = (
    third_party_preferred.groupby('fba_from_shipper')
    .size()
    .rename('rows')
    .reset_index()
)

covariate_summary_by_fba = (
    third_party_preferred.groupby('fba_from_shipper')[covariate_vars]
    .agg(['count', 'mean', 'median', 'min', 'max'])
)

rank_summary_by_fba = (
    third_party_preferred.groupby('fba_from_shipper')[rank_vars]
    .agg(['count', 'mean', 'median', 'min', 'max'])
)

smd_rows = []
for variable in covariate_vars:
    fba_vals = third_party_preferred.loc[third_party_preferred['fba_from_shipper'], variable].dropna()
    nonfba_vals = third_party_preferred.loc[~third_party_preferred['fba_from_shipper'], variable].dropna()

    pooled_sd = np.sqrt((fba_vals.var(ddof=1) + nonfba_vals.var(ddof=1)) / 2)
    smd = np.nan if pd.isna(pooled_sd) or pooled_sd == 0 else (fba_vals.mean() - nonfba_vals.mean()) / pooled_sd

    smd_rows.append({
        'variable': variable,
        'mean_fba': float(fba_vals.mean()),
        'mean_nonfba': float(nonfba_vals.mean()),
        'standardized_mean_difference': float(smd),
    })

covariate_smd_table = pd.DataFrame(smd_rows)

print('Third-party sample sizes in the seller-level diagnostic sample:')
display(group_size_table)

print('Third-party covariate summary by FBA status:')
display(covariate_summary_by_fba)

print('Standardized mean differences for observed covariates (FBA minus non-FBA):')
display(covariate_smd_table)

print('Third-party rank distributions by FBA status:')
display(rank_summary_by_fba)

Third-party sample sizes in the seller-level diagnostic sample:


,fba_from_shipper,rows
0,False,4111
1,True,996


Third-party covariate summary by FBA status:


prezzo_totale_reconstructed                                prezzo_spedizione(€)                              g_cons_min_robust                          num_valutazioni                                 \
                                       count       mean median   min    max                count      mean median  min    max             count      mean median min max           count         mean median min    max   
fba_from_shipper                                                                                                                                                                                                          
False                                   4111  55.862953  55.39  35.0  91.18                 4111  3.528125    0.0  0.0  21.09              4111  6.939431    6.0   2  28            4111  1479.713452  241.0   0  21464   
True                                     996  45.103484  44.99  38.0  59.99                  996       0.0    0.0  0.0    0.0               996  6.665663    6.0   3  60             996   317.631526   74.0   1   2120   

                 log1p_num_valutazioni                                         valutazioni_positive                            stelle                             
                                 count      mean    median       min       max                count       mean median min  max  count      mean median  min  max  
fba_from_shipper                                                                                                                                                  
False                             4111  5.545441  5.488938       0.0  9.974179                 4111  82.391146   87.0   0  100   4111  4.225979    4.5  0.0  5.0  
True                               996  4.191823  4.317488  0.693147  7.659643                  996  94.993976   97.0  71  100    996  4.710843    4.5  4.0  5.0

Standardized mean differences for observed covariates (FBA minus non-FBA):


,variable,mean_fba,mean_nonfba,standardized_mean_difference
0,prezzo_totale_reconstructed,45.103484,55.862953,-1.535896
1,prezzo_spedizione(€),0.000000,3.528125,-0.945965
2,g_cons_min_robust,6.665663,6.939431,-0.074528
3,num_valutazioni,317.631526,1479.713452,-0.604011
4,log1p_num_valutazioni,4.191823,5.545441,-0.630378
5,valutazioni_positive,94.993976,82.391146,0.946977
6,stelle,4.710843,4.225979,0.795075


Third-party rank distributions by FBA status:


rank_pos                           rank_pct                                   
                    count       mean median min max    count      mean    median  min       max
fba_from_shipper                                                                               
False                4111  47.989784   49.0   1  93     4111  0.574029  0.586207  0.0  1.000000
True                  996  16.841365   14.0   1  67      996  0.194446  0.160707  0.0  0.776316

### 14. Persistence, within-seller FBA variation, and split-risk diagnostics

Three properties affect the empirical design and are quantified here:
1. seller persistence across markets, which makes the panel persistent rather than i.i.d.,
2. within-seller variation in FBA status, which is zero in the final panel,
3. split-risk diagnostics evaluating whether a small number of seller identities drives any candidate result.

The cell measures the offer-state signature stability over time, the seller-presence pattern across the 62 markets, and the cross-market FBA-switching count under the seller-best collapsing rule. The diagnostic justifies the no-within-seller-FBA-switching limitation of the empirical strategy.


In [ ]:
# -----------------------------------------------------------------------------
# Section 14. Persistence, within-seller FBA variation, and split-risk diagnostics
# -----------------------------------------------------------------------------
# Quantify seller persistence across markets, within-seller variation in FBA status (zero in the final panel), and split-risk diagnostics.

df_new_offer_raw['offer_state_signature_core'] = build_row_signature(
    df_new_offer_raw,
    ['venduto_da', 'spedito_da', 'prezzo_totale_reconstructed', 'prezzo_spedizione(€)', 'fba_from_shipper', 'seller_type_from_name']
)

offer_repeat_summary_new = (
    df_new_offer_raw.groupby('offer_state_signature_core')
    .agg(
        n_rows=('timestamp', 'size'),
        n_markets=('timestamp', 'nunique'),
        min_rank=('rank_pos', 'min'),
        max_rank=('rank_pos', 'max'),
    )
    .reset_index()
)

offer_repeat_summary_new['repeated_across_markets'] = offer_repeat_summary_new['n_markets'].gt(1)

repeated_offer_signatures_new = set(
    offer_repeat_summary_new.loc[
        offer_repeat_summary_new['repeated_across_markets'],
        'offer_state_signature_core'
    ]
)

rows_in_repeated_offer_states_new = int(
    df_new_offer_raw['offer_state_signature_core'].isin(repeated_offer_signatures_new).sum()
)

offer_repeat_headline_new = pd.DataFrame({
    'metric': [
        'unique_core_offer_states_in_nuovo_offer_raw',
        'core_offer_states_repeated_across_markets',
        'rows_belonging_to_repeated_core_offer_states',
        'share_of_rows_in_repeated_core_offer_states',
    ],
    'value': [
        int(offer_repeat_summary_new.shape[0]),
        int(offer_repeat_summary_new['repeated_across_markets'].sum()),
        rows_in_repeated_offer_states_new,
        float(rows_in_repeated_offer_states_new / len(df_new_offer_raw)),
    ]
})

market_fingerprint_rows = []
for top_k in [None, 50, 40, 30, 20, 10, 5]:
    fingerprints = []
    for _, market_df in df_new_offer_raw.sort_values(['timestamp', 'rank_pos']).groupby('timestamp'):
        subset = market_df if top_k is None else market_df.head(top_k)
        sig = build_row_signature(
            subset,
            ['rank_pos', 'venduto_da', 'spedito_da', 'prezzo_totale_reconstructed', 'fba_from_shipper', 'seller_type_from_name']
        )
        fp = hashlib.sha256(' ||| '.join(sig.tolist()).encode('utf-8')).hexdigest()
        fingerprints.append(fp)

    fp_series = pd.Series(fingerprints)
    market_fingerprint_rows.append({
        'top_k': 'all' if top_k is None else int(top_k),
        'duplicate_market_count': int(fp_series.duplicated().sum()),
        'unique_fingerprint_count': int(fp_series.nunique()),
    })

market_fingerprint_summary = pd.DataFrame(market_fingerprint_rows)

# 14B. Seller persistence and FBA variation

third_party_offer_raw = df_new_offer_raw.loc[df_new_offer_raw['third_party_from_name']].copy()

seller_presence_summary = (
    third_party_offer_raw.groupby('venduto_da')
    .agg(
        n_markets=('timestamp', 'nunique'),
        n_rows=('timestamp', 'size'),
        n_fba_rows=('fba_from_shipper', 'sum'),
        n_nonfba_rows=('fba_from_shipper', lambda s: int((~s).sum())),
        distinct_shippers=('spedito_da', 'nunique'),
    )
    .reset_index()
)

seller_presence_summary['observed_both_fba_statuses'] = (
    seller_presence_summary['n_fba_rows'].gt(0) &
    seller_presence_summary['n_nonfba_rows'].gt(0)
)

observed_both_status_sellers = (
    seller_presence_summary.loc[seller_presence_summary['observed_both_fba_statuses']]
    .sort_values(['n_rows', 'venduto_da'], ascending=[False, True])
    .copy()
)

seller_market_mix = (
    third_party_offer_raw.groupby(['venduto_da', 'timestamp'])
    .agg(
        n_rows=('timestamp', 'size'),
        n_fba=('fba_from_shipper', 'sum'),
    )
    .reset_index()
)

seller_market_mix['n_nonfba'] = seller_market_mix['n_rows'] - seller_market_mix['n_fba']
seller_market_mix['mixed_within_market'] = (
    seller_market_mix['n_fba'].gt(0) &
    seller_market_mix['n_nonfba'].gt(0)
)

mixed_seller_summary = (
    seller_market_mix.loc[seller_market_mix['mixed_within_market']]
    .groupby('venduto_da')
    .agg(
        mixed_markets=('timestamp', 'nunique'),
        mixed_rows=('n_rows', 'sum'),
        mean_rows_per_mixed_market=('n_rows', 'mean'),
    )
    .reset_index()
    .sort_values(['mixed_markets', 'venduto_da'], ascending=[False, True])
)

third_party_seller_best = df_new_seller_best_thirdparty.copy()

fba_switch_summary_seller_best = (
    third_party_seller_best.groupby('venduto_da')
    .agg(
        n_markets=('timestamp', 'nunique'),
        fba_nunique=('fba_from_shipper', 'nunique'),
    )
    .reset_index()
)

seller_switch_headline = pd.DataFrame({
    'metric': [
        'third_party_sellers_in_nuovo_offer_raw',
        'third_party_sellers_in_all_62_markets',
        'third_party_sellers_with_both_fba_statuses_in_raw_nuovo',
        'third_party_sellers_with_mixed_within_market_status',
        'third_party_sellers_with_across_market_fba_switch_in_seller_best',
    ],
    'value': [
        int(seller_presence_summary.shape[0]),
        int((seller_presence_summary['n_markets'] == 62).sum()),
        int(seller_presence_summary['observed_both_fba_statuses'].sum()),
        int(mixed_seller_summary.shape[0]),
        int(fba_switch_summary_seller_best['fba_nunique'].gt(1).sum()),
    ]
})

# 14C. Repeated split diagnostic with class-validity checks

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

ml_diag = third_party_seller_best.copy()
ml_diag['top_5'] = ml_diag['rank_pos'].le(5).astype(int)

ml_num_features = [
    'prezzo_totale_reconstructed',
    'prezzo_spedizione(€)',
    'g_cons_min_robust',
    'num_valutazioni',
    'valutazioni_positive',
    'stelle',
]
ml_cat_base = ['fba_from_shipper']
ml_cat_with_seller = ['fba_from_shipper', 'venduto_da']


def run_logit_split_diagnostic(df, target, train_idx, test_idx, categorical_features):
    train = df.iloc[train_idx].copy()
    test = df.iloc[test_idx].copy()

    train_positive_count = int(train[target].sum())
    test_positive_count = int(test[target].sum())
    train_negative_count = int((1 - train[target]).sum())
    test_negative_count = int((1 - test[target]).sum())

    valid_train = train_positive_count > 0 and train_negative_count > 0
    valid_test = test_positive_count > 0 and test_negative_count > 0

    result = {
        'train_rows': int(len(train)),
        'test_rows': int(len(test)),
        'train_positives': train_positive_count,
        'test_positives': test_positive_count,
        'train_prevalence': float(train[target].mean()),
        'test_prevalence': float(test[target].mean()),
        'valid_for_auc_ap': bool(valid_train and valid_test),
    }

    if not result['valid_for_auc_ap']:
        result.update({
            'average_precision': np.nan,
            'roc_auc': np.nan,
        })
        return result

    for col in categorical_features:
        train[col] = train[col].astype('string')
        test[col] = test[col].astype('string')

    preprocess = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), ml_num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])

    model = Pipeline([
        ('preprocess', preprocess),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ])

    model.fit(train[ml_num_features + categorical_features], train[target])
    pred = model.predict_proba(test[ml_num_features + categorical_features])[:, 1]

    result.update({
        'average_precision': float(average_precision_score(test[target], pred)),
        'roc_auc': float(roc_auc_score(test[target], pred)),
    })
    return result


ml_results = []
target = 'top_5'
idx_all = np.arange(len(ml_diag))

for seed in range(20):
    # random-row split
    train_idx, test_idx = train_test_split(
        idx_all,
        test_size=0.30,
        random_state=seed,
        stratify=ml_diag[target],
    )
    for feature_set_name, categorical_features in [
        ('no_seller_identity', ml_cat_base),
        ('with_seller_identity', ml_cat_with_seller),
    ]:
        metrics = run_logit_split_diagnostic(ml_diag, target, train_idx, test_idx, categorical_features)
        ml_results.append({'seed': seed, 'split': 'random_row', 'feature_set': feature_set_name, **metrics})

    # market-grouped split
    train_idx, test_idx = next(
        GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
        .split(ml_diag, ml_diag[target], groups=ml_diag['timestamp'])
    )
    for feature_set_name, categorical_features in [
        ('no_seller_identity', ml_cat_base),
        ('with_seller_identity', ml_cat_with_seller),
    ]:
        metrics = run_logit_split_diagnostic(ml_diag, target, train_idx, test_idx, categorical_features)
        ml_results.append({'seed': seed, 'split': 'market_grouped', 'feature_set': feature_set_name, **metrics})

    # seller-blocked split
    train_idx, test_idx = next(
        GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
        .split(ml_diag, ml_diag[target], groups=ml_diag['venduto_da'])
    )
    for feature_set_name, categorical_features in [
        ('no_seller_identity', ml_cat_base),
        ('with_seller_identity', ml_cat_with_seller),
    ]:
        metrics = run_logit_split_diagnostic(ml_diag, target, train_idx, test_idx, categorical_features)
        ml_results.append({'seed': seed, 'split': 'seller_blocked', 'feature_set': feature_set_name, **metrics})

ml_split_results = pd.DataFrame(ml_results)

ml_split_summary = (
    ml_split_results.groupby(['split', 'feature_set'])
    .agg(
        attempted_splits=('seed', 'count'),
        valid_splits=('valid_for_auc_ap', 'sum'),
        invalid_splits=('valid_for_auc_ap', lambda s: int((~s).sum())),
        mean_average_precision=('average_precision', 'mean'),
        sd_average_precision=('average_precision', 'std'),
        mean_roc_auc=('roc_auc', 'mean'),
        sd_roc_auc=('roc_auc', 'std'),
        mean_test_prevalence=('test_prevalence', 'mean'),
    )
    .reset_index()
)

print('Offer-state persistence headline:')
display(offer_repeat_headline_new)

print('Market-fingerprint duplication by top-k slice:')
display(market_fingerprint_summary)

print('Seller-level FBA-variation headline:')
display(seller_switch_headline)

print('Third-party sellers observed with both FBA and non-FBA rows in the raw Nuovo sample:')
display(observed_both_status_sellers)

print('Mixed FBA/non-FBA seller-market cells, summarized by seller:')
display(mixed_seller_summary)

print('Across-market FBA switching in the seller-level sample:')
display(fba_switch_summary_seller_best.loc[fba_switch_summary_seller_best['fba_nunique'].gt(1)])

print('Repeated split diagnostic on the top-5 outcome:')
display(ml_split_summary)

register_check(
    '14',
    'at least 98% of Nuovo rows belong to repeated core offer states',
    (rows_in_repeated_offer_states_new / len(df_new_offer_raw)) >= 0.98,
    f"share = {rows_in_repeated_offer_states_new / len(df_new_offer_raw):.6f}"
)

register_check(
    '14',
    'exactly two third-party sellers show both FBA and non-FBA rows in raw Nuovo',
    int(seller_presence_summary['observed_both_fba_statuses'].sum()) == 2,
    f"count = {int(seller_presence_summary['observed_both_fba_statuses'].sum())}"
)

register_check(
    '14',
    'the two mixed-status sellers are mixed within market rather than switching across markets',
    int(mixed_seller_summary.shape[0]) == 2 and int(fba_switch_summary_seller_best['fba_nunique'].gt(1).sum()) == 0,
    'SKY LINE E-COMM and TPC Mobile mix statuses within market; no across-market switching survives in seller_best'
)

Offer-state persistence headline:


,metric,value
0,unique_core_offer_states_in_nuovo_offer_raw,427.000000
1,core_offer_states_repeated_across_markets,348.000000
2,rows_belonging_to_repeated_core_offer_states,5706.000000
3,share_of_rows_in_repeated_core_offer_states,0.986174


Market-fingerprint duplication by top-k slice:


,top_k,duplicate_market_count,unique_fingerprint_count
0,all,0,62
1,50,0,62
2,40,0,62
3,30,0,62
4,20,1,61
5,10,15,47
6,5,30,32


Seller-level FBA-variation headline:


...

,venduto_da,mixed_markets,mixed_rows,mean_rows_per_mixed_market
0,SKY LINE E-COMM,62,124,2.0
1,TPC Mobile,62,124,2.0


Across-market FBA switching in the seller-level sample:


,venduto_da,n_markets,fba_nunique


Repeated split diagnostic on the top-5 outcome:


,split,feature_set,attempted_splits,valid_splits,invalid_splits,mean_average_precision,sd_average_precision,mean_roc_auc,sd_roc_auc,mean_test_prevalence
0,market_grouped,no_seller_identity,20,20,0,0.887567,0.032555,0.994968,0.001038,0.060651
1,market_grouped,with_seller_identity,20,20,0,0.936073,0.013192,0.996266,0.000700,0.060651
2,random_row,no_seller_identity,20,20,0,0.879366,0.032815,0.994837,0.001286,0.060665
3,random_row,with_seller_identity,20,20,0,0.926312,0.030523,0.995885,0.001400,0.060665
4,seller_blocked,no_seller_identity,20,19,1,0.742191,0.235348,0.966367,0.102726,0.053241
5,seller_blocked,with_seller_identity,20,19,1,0.661407,0.246403,0.956241,0.093935,0.053241


### 15. Variable provenance and allowed-use classification for econometric use

The cell records a variable-by-variable provenance classification and assigns each variable an admissible-use label for the econometric analysis. The classification separates raw primitives, deterministic reconstructions, audit-only diagnostics, and variables retained only as descriptive supplements. The audit registry treats this classification as a binding contract: variables flagged as audit-only cannot enter regression specifications in Part II.


In [ ]:
# -----------------------------------------------------------------------------
# Section 15. Variable provenance and admissible-use classification
# -----------------------------------------------------------------------------
# Record the variable-by-variable provenance classification and the admissible-use labels enforced by the econometric specifications in Part II.

table_variable_provenance = pd.DataFrame([
    ('timestamp', 'raw source field', 'timestamp_dt',
     'yes', 'yes', 'as market/time fixed effect only',
     'Core market identifier.'),

    ('is_malformed_row', 'audit flag', 'is_malformed_row',
     'sample-construction only', 'no', 'no',
     'Use only to exclude the single malformed row.'),

    ('condizione', 'raw source field', 'condizione',
     'yes', 'yes', 'sample-definition only',
     'Use only after defining the market of interest.'),

    ('visibility_order', 'raw source field', 'visibility_order',
     'yes', 'diagnostic only', 'no',
     'Raw page order. For rank-based outcomes it is near-outcome information and should not be a headline predictor.'),

    ('rank_pos / rank_pct', 'derived within-sample ranking variables', 'recompute with add_ranks inside the final sample',
     'yes as outcomes/descriptives', 'target only, not predictor', 'no',
     'They depend on filtering and deduplication, so they must be recomputed inside the final sample.'),

    ('buy_box', 'stored flag', 'buy_box',
     'descriptive only', 'descriptive only', 'no',
     'Stored auxiliary winner flag. It is not part of the thesis outcomes and has no third-party support.'),

    ('best_third_party / top_3 / top_5 / top_10', 'derived outcome block', 'recompute inside the final sample',
     'yes', 'yes as targets only', 'no',
     'These are the supported ranking outcomes after reranking inside the final third-party competitive set.'),

    ('venduto_da', 'raw source field', 'venduto_da',
     'yes', 'diagnostic only under seller-blocked evaluation', 'seller fixed effect only when the design supports it',
     'Seller identity materially raises apparent predictive performance under non-blocked splits and should not be a standard feature.'),

    ('seller_type_from_name / third_party_from_name / amazon_retail_from_name / amazon_warehouse_from_name',
     'reconstructed from seller name', 'same reconstructed fields',
     'yes', 'sample-construction only', 'no',
     'Useful for segmentation and audit logic, but mostly deterministic functions of seller identity.'),

    ('spedito_da', 'raw source field', 'spedito_da',
     'yes', 'yes', 'yes with caution',
     'Preferred source for FBA reconstruction and useful for distinguishing mixed-within-seller offers.'),

    ('fba', 'stored flag', 'fba_from_shipper',
     'yes after reconstruction', 'yes after reconstruction', 'no if FBA is the treatment of interest',
     'Stored flag matches the shipper-based reconstruction here, but the reconstructed version is preferred.'),

    ('vend_amazon', 'stored flag', 'amazon_retail_from_name',
     'yes after reconstruction', 'sample-construction only', 'no',
     'Stored flag matches the name-based reconstruction here, but the reconstructed version is preferred.'),

    ('prezzo', 'raw source field', 'prezzo',
     'yes', 'yes', 'yes',
     'Core primitive price component.'),

    ('prezzo_spedizione(€)', 'stored convenience',
     'parsed_shipping_price when explicit, else stored value flagged as uncertain',
     'yes with caution', 'yes with caution', 'yes with caution',
     'There are 45 explicit extraction failures and 53 contact-courier cases where no numeric amount is recoverable from raw text.'),

    ('tipo_spedizione', 'stored convenience', 'parsed_shipping_mode when explicit',
     'yes with caution', 'not recommended as core regressor', 'no',
     'Mode is mostly coherent, but it is not sufficient for recovering numeric shipping cost.'),

    ('prezzo_totale(€)', 'stored convenience', 'prezzo_totale_reconstructed',
     'yes after reconstruction', 'yes after reconstruction', 'yes after reconstruction',
     'Use the reconstructed total built from audited components.'),

    ('num_valutazioni', 'raw source field', 'num_valutazioni',
     'yes', 'yes', 'yes with caution',
     'Structural zeros for Amazon-type sellers should not be interpreted as quality zero.'),

    ('valutazioni_positive', 'raw source field', 'valutazioni_positive',
     'yes', 'yes', 'yes with caution',
     'Structural zeros for Amazon-type sellers should not be interpreted as quality zero.'),

    ('stelle', 'raw source field', 'stelle',
     'yes', 'yes', 'yes with caution',
     'Structural zeros for Amazon-type sellers should not be interpreted as quality zero.'),

    ('spedizione_consegna', 'raw source field', 'spedizione_consegna',
     'source for reconstruction only', 'source for reconstruction only', 'source for reconstruction only',
     'Preferred raw text source for delivery and shipping reconstruction.'),

    ('g_cons_min / g_cons_max / g_cons_vel_min / g_cons_vel_max', 'stored parsed fields',
     'g_cons_min_robust / g_cons_max_robust / g_cons_vel_min_robust / g_cons_vel_max_robust',
     'yes after reconstruction', 'yes after reconstruction', 'yes after reconstruction',
     'Stored delivery fields fail in five entire markets and must be reparsed from raw text.'),

    ('g_spedizione / g_spedizione_vel', 'stored convenience', 'recompute from reparsed delivery bounds',
     'yes after reconstruction', 'yes after reconstruction', 'yes after reconstruction',
     'Only use after the delivery parser is repaired.'),

    ('dif_prezzo', 'stored convenience', 'recompute only inside the final sample if needed',
     'no', 'no', 'no',
     'Invalid after the Nuovo restriction and invalid in every audited candidate sample.'),

    ('dif_prezzo_tot', 'stored convenience', 'recompute only inside the final sample if needed',
     'no', 'no', 'no',
     'Invalid after the Nuovo restriction and invalid in every audited candidate sample.'),

    ('rapp_piu_basso', 'stored convenience', 'recompute only inside the final sample if needed',
     'no', 'no', 'no',
     'Invalid after the Nuovo restriction and invalid in every audited candidate sample.'),

    ('dif_prezzo_sped', 'stored convenience', 'recompute only inside the final sample if needed',
     'yes with caution', 'yes with caution', 'no',
     'Stable numerically in the audited candidate samples, but still market-relative.'),

    ('delta_num_val', 'stored convenience', 'recompute only inside the final sample if needed',
     'yes with caution', 'yes with caution', 'no',
     'Market-relative, not a primitive covariate.'),

    ('delta_val_pos(%)', 'stored convenience', 'recompute only inside the final sample if needed',
     'yes with caution', 'yes with caution', 'no',
     'Market-relative, not a primitive covariate.'),

    ('delta_consegna', 'stored convenience', 'recompute from reparsed delivery fields inside the final sample if needed',
     'yes with caution', 'yes with caution', 'no',
     'Depends on repaired delivery parsing and on market minima.'),

    ('qta_min', 'raw source field', 'qta_min',
     'redundant', 'redundant', 'redundant',
     'Always equal to 1 in this file.'),

    ('prezzo_prod_venduto(€)', 'stored convenience', 'prezzo',
     'redundant', 'redundant', 'redundant',
     'Equal to prezzo everywhere in this file.')
], columns=[
    'variable_block',
    'status_in_raw_csv',
    'preferred_version_after_audit',
    'descriptive_use',
    'predictive_use',
    'causal_adjustment_use',
    'note'
])

display(table_variable_provenance)

,variable_block,status_in_raw_csv,preferred_version_after_audit,descriptive_use,predictive_use,causal_adjustment_use,note
0,timestamp,raw source field,timestamp_dt,yes,yes,as market/time fixed effect only,Core market identifier.
1,is_malformed_row,audit flag,is_malformed_row,sample-construction only,no,no,Use only to exclude the single malformed row.
2,condizione,raw source field,condizione,yes,yes,sample-definition only,Use only after defining the market of interest.
3,visibility_order,raw source field,visibility_order,yes,diagnostic only,no,Raw page order. For rank-based outcomes it is near-outcome information and should not be a headline predictor.
4,rank_pos / rank_pct,derived within-sample ranking variables,recompute with add_ranks inside the final sample,yes as outcomes/descriptives,"target only, not predictor",no,"They depend on filtering and deduplication, so they must be recomputed inside the final sample."
5,buy_box,stored flag,buy_box,descriptive only,descriptive only,no,Stored auxiliary winner flag. It is not part of the thesis outcomes and has no third-party support.
6,best_third_party / top_3 / top_5 / top_10,derived outcome block,recompute inside the final sample,yes,yes as targets only,no,These are the supported ranking outcomes after reranking inside the final third-party competitive set.
7,venduto_da,raw source field,venduto_da,yes,diagnostic only under seller-blocked evaluation,seller fixed effect only when the design supports it,Seller identity materially raises apparent predictive performance under non-blocked splits and should not be a standard feature.
8,seller_type_from_name / third_party_from_name / amazon_retail_from_name / amazon_warehouse_from_name,reconstructed from seller name,same reconstructed fields,yes,sample-construction only,no,"Useful for segmentation and audit logic, but mostly deterministic functions of seller identity."
9,spedito_da,raw source field,spedito_da,yes,yes,yes with caution,Preferred source for FBA reconstruction and useful for distinguishing mixed-within-seller offers.


### 16. Review-field usability, contact-courier sensitivity, rank variation, same-day market change, and offer-count support

The cell closes five practical questions raised by the preceding audit:
1. usability of the review fields when `num_valutazioni`, `valutazioni_positive`, and `stelle` disagree,
2. sensitivity of the panel to the 50 contact-courier rows concentrated in a single seller identity,
3. within-seller rank variation among the 52 sellers present in all 62 markets,
4. magnitude of same-day market change across consecutive snapshots,
5. dispersion of market sizes after the third-party restriction.

Each diagnostic feeds an explicit decision in the empirical design: review-problem rows enter the specifications as narrow control flags rather than as diffuse controls, contact-courier rows enter as a separate flag, and rank variation supports the dynamic-design feasibility check.


In [ ]:
# -----------------------------------------------------------------------------
# Section 16. Review usability, contact-courier sensitivity, rank variation, same-day change, and offer support
# -----------------------------------------------------------------------------
# Examine review-field usability, contact-courier exposure, within-seller rank variation, same-day market change, and the third-party market-size distribution.

df_preferred = df_new_seller_best_thirdparty.copy()
third_party_preferred = df_preferred.copy()

review_cols = ['num_valutazioni', 'valutazioni_positive', 'stelle']

review_field_status = []
for col in review_cols:
    s = third_party_preferred[col]
    review_field_status.append({
        'variable': col,
        'rows': int(len(s)),
        'missing_rows': int(s.isna().sum()),
        'zero_rows': int(s.eq(0).sum()),
        'positive_rows': int(s.fillna(0).gt(0).sum()),
        'share_missing': float(s.isna().mean()),
        'share_zero': float(s.eq(0).mean()),
        'share_positive': float(s.fillna(0).gt(0).mean()),
    })

review_field_status = pd.DataFrame(review_field_status)

review_strict_support_mask = (
    third_party_preferred['num_valutazioni'].fillna(0).gt(0) &
    third_party_preferred['valutazioni_positive'].fillna(0).gt(0) &
    third_party_preferred['stelle'].fillna(0).gt(0)
)

review_any_problem_mask = (
    third_party_preferred['num_valutazioni'].isna() |
    third_party_preferred['valutazioni_positive'].isna() |
    third_party_preferred['stelle'].isna() |
    third_party_preferred['num_valutazioni'].fillna(0).eq(0) |
    third_party_preferred['valutazioni_positive'].fillna(0).eq(0) |
    third_party_preferred['stelle'].fillna(0).eq(0)
)

review_usability_headline = pd.DataFrame({
    'metric': [
        'third_party_rows_in_preferred_sample',
        'third_party_sellers_in_preferred_sample',
        'rows_with_strict_positive_review_support',
        'share_rows_with_strict_positive_review_support',
        'rows_with_any_review_problem',
        'share_rows_with_any_review_problem',
        'sellers_with_any_review_problem',
    ],
    'value': [
        int(len(third_party_preferred)),
        int(third_party_preferred['venduto_da'].nunique()),
        int(review_strict_support_mask.sum()),
        float(review_strict_support_mask.mean()),
        int(review_any_problem_mask.sum()),
        float(review_any_problem_mask.mean()),
        int(third_party_preferred.loc[review_any_problem_mask, 'venduto_da'].nunique()),
    ]
})

review_problem_examples = third_party_preferred.loc[
    review_any_problem_mask,
    ['timestamp', 'venduto_da', 'spedito_da', 'num_valutazioni', 'valutazioni_positive', 'stelle', 'rank_pos', 'rank_pct']
].sort_values(['timestamp', 'rank_pos', 'venduto_da']).head(20)

contact_courier_preferred = third_party_preferred.loc[
    third_party_preferred['parsed_shipping_mode'].eq('unknown_contact_courier')
].copy()

contact_courier_preferred['shipping_price_contact_zero_sensitivity'] = 0.0

contact_courier_summary = pd.DataFrame({
    'metric': [
        'contact_courier_rows_in_preferred_sample',
        'share_of_preferred_sample',
        'contact_courier_markets',
        'contact_courier_sellers',
        'contact_courier_rows_with_stored_zero_shipping',
        'contact_courier_rows_labeled_paid_shipping',
    ],
    'value': [
        int(len(contact_courier_preferred)),
        float(len(contact_courier_preferred) / len(third_party_preferred)) if len(third_party_preferred) > 0 else np.nan,
        int(contact_courier_preferred['timestamp'].nunique()),
        int(contact_courier_preferred['venduto_da'].nunique()),
        int(contact_courier_preferred['prezzo_spedizione(€)'].fillna(0).eq(0).sum()),
        int(contact_courier_preferred['tipo_spedizione'].eq('A Pagamento').sum()),
    ]
})

contact_courier_policy = pd.DataFrame([
    {
        'policy_case': 'audited_baseline',
        'shipping_price_treatment': 'keep observed stored value and flag shipping uncertainty',
        'interpretation': 'the raw text does not reveal a numeric shipping amount',
    },
    {
        'policy_case': 'later_sensitivity_case',
        'shipping_price_treatment': 'set shipping price to 0 for contact-courier rows',
        'interpretation': 'practical interpretation when no extra customer-paid numeric amount is shown',
    },
])

contact_courier_examples = contact_courier_preferred.loc[
    :,
    ['timestamp', 'venduto_da', 'spedito_da', 'tipo_spedizione', 'prezzo_spedizione(€)', 'shipping_price_contact_zero_sensitivity', 'spedizione_consegna', 'rank_pos']
].sort_values(['timestamp', 'rank_pos', 'venduto_da']).head(15)

rank_variation_by_seller = (
    third_party_preferred.groupby('venduto_da')
    .agg(
        n_markets=('timestamp', 'nunique'),
        min_rank=('rank_pos', 'min'),
        max_rank=('rank_pos', 'max'),
        mean_rank=('rank_pos', 'mean'),
        sd_rank=('rank_pos', 'std'),
        min_rank_pct=('rank_pct', 'min'),
        max_rank_pct=('rank_pct', 'max'),
        mean_rank_pct=('rank_pct', 'mean'),
        sd_rank_pct=('rank_pct', 'std'),
    )
    .reset_index()
)

rank_variation_by_seller['rank_range'] = rank_variation_by_seller['max_rank'] - rank_variation_by_seller['min_rank']
rank_variation_by_seller['rank_pct_range'] = rank_variation_by_seller['max_rank_pct'] - rank_variation_by_seller['min_rank_pct']

rank_variation_all_62 = rank_variation_by_seller.loc[
    rank_variation_by_seller['n_markets'].eq(third_party_preferred['timestamp'].nunique())
].copy()

rank_variation_headline = pd.DataFrame({
    'metric': [
        'third_party_sellers_total',
        'third_party_sellers_in_all_62_markets',
        'all_62_sellers_with_any_rank_change',
        'share_all_62_sellers_with_any_rank_change',
        'median_rank_range_among_all_62_sellers',
        'median_rank_pct_range_among_all_62_sellers',
        'max_rank_range_among_all_62_sellers',
        'max_rank_pct_range_among_all_62_sellers',
    ],
    'value': [
        int(rank_variation_by_seller.shape[0]),
        int(rank_variation_all_62.shape[0]),
        int(rank_variation_all_62['rank_range'].gt(0).sum()),
        float(rank_variation_all_62['rank_range'].gt(0).mean()) if len(rank_variation_all_62) > 0 else np.nan,
        float(rank_variation_all_62['rank_range'].median()) if len(rank_variation_all_62) > 0 else np.nan,
        float(rank_variation_all_62['rank_pct_range'].median()) if len(rank_variation_all_62) > 0 else np.nan,
        float(rank_variation_all_62['rank_range'].max()) if len(rank_variation_all_62) > 0 else np.nan,
        float(rank_variation_all_62['rank_pct_range'].max()) if len(rank_variation_all_62) > 0 else np.nan,
    ]
})

timestamp_order = (
    third_party_preferred[['timestamp', 'timestamp_dt']]
    .drop_duplicates()
    .sort_values('timestamp_dt')['timestamp']
    .tolist()
)

top_turnover_rows = []
for prev_ts, curr_ts in zip(timestamp_order[:-1], timestamp_order[1:]):
    prev_df = third_party_preferred.loc[third_party_preferred['timestamp'].eq(prev_ts)]
    curr_df = third_party_preferred.loc[third_party_preferred['timestamp'].eq(curr_ts)]

    prev_top5 = set(prev_df.loc[prev_df['rank_pos'].le(5), 'venduto_da'])
    curr_top5 = set(curr_df.loc[curr_df['rank_pos'].le(5), 'venduto_da'])

    prev_top10 = set(prev_df.loc[prev_df['rank_pos'].le(10), 'venduto_da'])
    curr_top10 = set(curr_df.loc[curr_df['rank_pos'].le(10), 'venduto_da'])

    top5_union = len(prev_top5 | curr_top5)
    top10_union = len(prev_top10 | curr_top10)

    top_turnover_rows.append({
        'from_timestamp': prev_ts,
        'to_timestamp': curr_ts,
        'top_5_jaccard': np.nan if top5_union == 0 else len(prev_top5 & curr_top5) / top5_union,
        'top_10_jaccard': np.nan if top10_union == 0 else len(prev_top10 & curr_top10) / top10_union,
    })

top_turnover_table = pd.DataFrame(top_turnover_rows)

top_turnover_headline = pd.DataFrame({
    'metric': [
        'adjacent_market_pairs',
        'mean_top_5_jaccard',
        'median_top_5_jaccard',
        'mean_top_10_jaccard',
        'median_top_10_jaccard',
    ],
    'value': [
        int(len(top_turnover_table)),
        float(top_turnover_table['top_5_jaccard'].mean()) if len(top_turnover_table) > 0 else np.nan,
        float(top_turnover_table['top_5_jaccard'].median()) if len(top_turnover_table) > 0 else np.nan,
        float(top_turnover_table['top_10_jaccard'].mean()) if len(top_turnover_table) > 0 else np.nan,
        float(top_turnover_table['top_10_jaccard'].median()) if len(top_turnover_table) > 0 else np.nan,
    ]
})

same_day_calendar = (
    third_party_preferred[['timestamp', 'timestamp_dt']]
    .drop_duplicates()
    .assign(market_date=lambda d: d['timestamp_dt'].dt.date)
    .sort_values(['market_date', 'timestamp_dt'])
)

same_day_rows = []
same_day_seller_changes = []

for market_date, day_df in same_day_calendar.groupby('market_date'):
    day_df = day_df.sort_values('timestamp_dt').reset_index(drop=True)
    if len(day_df) != 2:
        continue

    first_ts = day_df.loc[0, 'timestamp']
    second_ts = day_df.loc[1, 'timestamp']

    first_df = third_party_preferred.loc[third_party_preferred['timestamp'].eq(first_ts)].copy()
    second_df = third_party_preferred.loc[third_party_preferred['timestamp'].eq(second_ts)].copy()

    first_top5 = set(first_df.loc[first_df['rank_pos'].le(5), 'venduto_da'])
    second_top5 = set(second_df.loc[second_df['rank_pos'].le(5), 'venduto_da'])
    first_top10 = set(first_df.loc[first_df['rank_pos'].le(10), 'venduto_da'])
    second_top10 = set(second_df.loc[second_df['rank_pos'].le(10), 'venduto_da'])

    top5_union = len(first_top5 | second_top5)
    top10_union = len(first_top10 | second_top10)

    first_best = first_df.sort_values(['rank_pos', 'venduto_da']).iloc[0]['venduto_da'] if len(first_df) > 0 else pd.NA
    second_best = second_df.sort_values(['rank_pos', 'venduto_da']).iloc[0]['venduto_da'] if len(second_df) > 0 else pd.NA

    rank_merge = (
        first_df[['venduto_da', 'rank_pos', 'rank_pct']]
        .rename(columns={'rank_pos': 'rank_pos_first', 'rank_pct': 'rank_pct_first'})
        .merge(
            second_df[['venduto_da', 'rank_pos', 'rank_pct']]
            .rename(columns={'rank_pos': 'rank_pos_second', 'rank_pct': 'rank_pct_second'}),
            on='venduto_da',
            how='inner'
        )
    )

    rank_merge['abs_rank_change'] = (rank_merge['rank_pos_second'] - rank_merge['rank_pos_first']).abs()
    rank_merge['abs_rank_pct_change'] = (rank_merge['rank_pct_second'] - rank_merge['rank_pct_first']).abs()

    if len(rank_merge) > 0:
        rank_merge = rank_merge.assign(
            market_date=market_date,
            first_timestamp=first_ts,
            second_timestamp=second_ts,
        )
        same_day_seller_changes.append(rank_merge)

    same_day_rows.append({
        'market_date': market_date,
        'first_timestamp': first_ts,
        'second_timestamp': second_ts,
        'common_sellers': int(len(rank_merge)),
        'sellers_with_rank_change': int(rank_merge['abs_rank_change'].gt(0).sum()) if len(rank_merge) > 0 else 0,
        'share_common_sellers_with_rank_change': float(rank_merge['abs_rank_change'].gt(0).mean()) if len(rank_merge) > 0 else np.nan,
        'median_abs_rank_change': float(rank_merge['abs_rank_change'].median()) if len(rank_merge) > 0 else np.nan,
        'max_abs_rank_change': float(rank_merge['abs_rank_change'].max()) if len(rank_merge) > 0 else np.nan,
        'best_third_party_changed': bool(first_best != second_best) if pd.notna(first_best) and pd.notna(second_best) else pd.NA,
        'top_5_jaccard': np.nan if top5_union == 0 else len(first_top5 & second_top5) / top5_union,
        'top_10_jaccard': np.nan if top10_union == 0 else len(first_top10 & second_top10) / top10_union,
    })

same_day_pair_table = pd.DataFrame(same_day_rows)

if len(same_day_seller_changes) > 0:
    same_day_seller_changes = pd.concat(same_day_seller_changes, ignore_index=True)
else:
    same_day_seller_changes = pd.DataFrame(
        columns=['venduto_da', 'rank_pos_first', 'rank_pct_first', 'rank_pos_second', 'rank_pct_second', 'abs_rank_change', 'abs_rank_pct_change', 'market_date', 'first_timestamp', 'second_timestamp']
    )

same_day_headline = pd.DataFrame({
    'metric': [
        'same_day_market_pairs',
        'pairs_with_any_rank_change_among_common_sellers',
        'share_pairs_with_any_rank_change_among_common_sellers',
        'mean_share_common_sellers_with_rank_change',
        'median_share_common_sellers_with_rank_change',
        'mean_same_day_top_5_jaccard',
        'median_same_day_top_5_jaccard',
        'mean_same_day_top_10_jaccard',
        'median_same_day_top_10_jaccard',
        'pairs_with_best_third_party_change',
        'share_pairs_with_best_third_party_change',
    ],
    'value': [
        int(len(same_day_pair_table)),
        int(same_day_pair_table['sellers_with_rank_change'].gt(0).sum()) if len(same_day_pair_table) > 0 else 0,
        float(same_day_pair_table['sellers_with_rank_change'].gt(0).mean()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['share_common_sellers_with_rank_change'].mean()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['share_common_sellers_with_rank_change'].median()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['top_5_jaccard'].mean()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['top_5_jaccard'].median()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['top_10_jaccard'].mean()) if len(same_day_pair_table) > 0 else np.nan,
        float(same_day_pair_table['top_10_jaccard'].median()) if len(same_day_pair_table) > 0 else np.nan,
        int(same_day_pair_table['best_third_party_changed'].fillna(False).sum()) if len(same_day_pair_table) > 0 else 0,
        float(same_day_pair_table['best_third_party_changed'].fillna(False).mean()) if len(same_day_pair_table) > 0 else np.nan,
    ]
})

same_day_change_examples = same_day_seller_changes.sort_values(
    ['abs_rank_change', 'abs_rank_pct_change', 'venduto_da'],
    ascending=False
).head(20)

third_party_market_size = third_party_preferred.groupby('timestamp').size()

third_party_market_size_summary = (
    third_party_market_size.describe()
    .rename('third_party_market_size')
    .to_frame()
    .T
)

offer_threshold_support_rows = []
for k in [3, 5, 10, 20, 30, 40, 50]:
    retained_rows_if_capped = int(np.minimum(third_party_market_size, k).sum())
    markets_with_at_least_k = int((third_party_market_size >= k).sum())

    offer_threshold_support_rows.append({
        'top_k_third_party_offers_per_market': int(k),
        'rows_retained_if_capped_at_k': retained_rows_if_capped,
        'share_rows_retained_if_capped_at_k': float(retained_rows_if_capped / len(third_party_preferred)),
        'markets_with_at_least_k_offers': markets_with_at_least_k,
        'share_markets_with_at_least_k_offers': float(markets_with_at_least_k / third_party_market_size.shape[0]),
    })

offer_threshold_support = pd.DataFrame(offer_threshold_support_rows)

print('Review-field usability in the preferred third-party seller-level sample:')
display(review_usability_headline)
display(review_field_status)

print('Example rows with zero or missing review support:')
display(review_problem_examples)

print('Contact-courier rows in the preferred sample:')
display(contact_courier_summary)
display(contact_courier_policy)
display(contact_courier_examples)

print('Rank variation across markets:')
display(rank_variation_headline)
display(top_turnover_headline)
display(rank_variation_all_62.sort_values(['rank_range', 'rank_pct_range'], ascending=False).head(20))

print('Same-day rank change across the two daily markets:')
display(same_day_headline)
display(same_day_pair_table)
display(same_day_change_examples)

print('Third-party offer-count support by market:')
display(third_party_market_size_summary)
display(offer_threshold_support)

register_check(
    '16',
    'review variables retain positive support in at least 98% of preferred third-party rows',
    float(review_strict_support_mask.mean()) >= 0.98,
    f"share = {float(review_strict_support_mask.mean()):.6f}"
)

register_check(
    '16',
    'contact-courier rows remain below 1% of the preferred third-party sample',
    float(len(contact_courier_preferred) / len(third_party_preferred)) <= 0.01 if len(third_party_preferred) > 0 else False,
    f"share = {float(len(contact_courier_preferred) / len(third_party_preferred)):.6f}" if len(third_party_preferred) > 0 else 'empty preferred sample'
)

register_check(
    '16',
    'all third-party sellers observed in all 62 markets change rank at least once',
    bool(rank_variation_all_62['rank_range'].gt(0).all()) if len(rank_variation_all_62) > 0 else False,
    f"count_with_change = {int(rank_variation_all_62['rank_range'].gt(0).sum())} out of {int(len(rank_variation_all_62))}"
)

register_check(
    '16',
    'the panel contains one same-day market pair for each observed calendar day',
    int(len(same_day_pair_table)) == int(same_day_calendar['market_date'].nunique()),
    f"same_day_pairs = {int(len(same_day_pair_table))}, observed_days = {int(same_day_calendar['market_date'].nunique())}"
)

Review-field usability in the preferred third-party seller-level sample:


,metric,value
0,third_party_rows_in_preferred_sample,5107.000000
1,third_party_sellers_in_preferred_sample,119.000000
2,rows_with_strict_positive_review_support,5015.000000
3,share_rows_with_strict_positive_review_support,0.981986
4,rows_with_any_review_problem,92.000000
5,share_rows_with_any_review_problem,0.018014
6,sellers_with_any_review_problem,3.000000


,variable,rows,missing_rows,zero_rows,positive_rows,share_missing,share_zero,share_positive
0,num_valutazioni,5107,0,92,5015,0.0,0.018014,0.981986
1,valutazioni_positive,5107,0,92,5015,0.0,0.018014,0.981986
2,stelle,5107,0,92,5015,0.0,0.018014,0.981986


Example rows with zero or missing review support:


,timestamp,venduto_da,spedito_da,num_valutazioni,valutazioni_positive,stelle,rank_pos,rank_pct
74,2022-02-07T12:05:05,i-Storeparts srl,i-Storeparts srl,0,0,0.0,13,0.155844
65,2022-02-07T12:05:05,Versão Portátil Não enviamos para ilhas Portuguesa,Versão Portátil Não enviamos para ilhas Portuguesa,0,0,0.0,69,0.883117
153,2022-02-07T18:09:10,i-Storeparts srl,i-Storeparts srl,0,0,0.0,13,0.155844
146,2022-02-07T18:09:10,Versão Portátil Não enviamos para ilhas Portuguesa,Versão Portátil Não enviamos para ilhas Portuguesa,0,0,0.0,70,0.896104
231,2022-02-08T12:07:00,i-Storeparts srl,i-Storeparts srl,0,0,0.0,13,0.157895
224,2022-02-08T12:07:00,Versão Portátil Não enviamos para ilhas Portuguesa,Versão Portátil Não enviamos para ilhas Portuguesa,0,0,0.0,69,0.894737
310,2022-02-08T18:05:35,i-Storeparts srl,i-Storeparts srl,0,0,0.0,14,0.168831
303,2022-02-08T18:05:35,Versão Portátil Não enviamos para ilhas Portuguesa,Versão Portátil Não enviamos para ilhas Portuguesa,0,0,0.0,70,0.896104
390,2022-02-09T12:08:19,i-Storeparts srl,i-Storeparts srl,0,0,0.0,15,0.181818
382,2022-02-09T12:08:19,Versão Portátil Não enviamos para ilhas Portuguesa,Versão Portátil Não enviamos para ilhas Portuguesa,0,0,0.0,71,0.909091


...

,market_date,first_timestamp,second_timestamp,common_sellers,sellers_with_rank_change,share_common_sellers_with_rank_change,median_abs_rank_change,max_abs_rank_change,best_third_party_changed,top_5_jaccard,top_10_jaccard
0,2022-02-07,2022-02-07T12:05:05,2022-02-07T18:09:10,76,38,0.500000,0.5,22.0,False,1.000000,1.000000
1,2022-02-08,2022-02-08T12:07:00,2022-02-08T18:05:35,75,54,0.720000,1.0,22.0,False,1.000000,0.818182
2,2022-02-09,2022-02-09T12:08:19,2022-02-09T18:02:46,78,36,0.461538,0.0,23.0,False,1.000000,1.000000
3,2022-02-10,2022-02-10T12:02:37,2022-02-10T18:03:10,77,35,0.454545,0.0,18.0,False,1.000000,1.000000
4,2022-02-11,2022-02-11T12:02:48,2022-02-11T18:02:26,74,48,0.648649,1.0,18.0,False,0.666667,0.818182
5,2022-02-12,2022-02-12T12:02:21,2022-02-12T18:03:58,73,17,0.232877,0.0,4.0,False,1.000000,0.818182
6,2022-02-13,2022-02-13T12:12:47,2022-02-13T18:02:08,72,67,0.930556,1.0,2.0,False,0.666667,0.818182
7,2022-02-14,2022-02-14T12:02:21,2022-02-14T18:02:15,73,36,0.493151,0.0,17.0,False,0.666667,0.818182
8,2022-02-15,2022-02-15T12:02:18,2022-02-15T18:02:22,75,58,0.773333,1.0,16.0,False,1.000000,0.818182
9,2022-02-16,2022-02-16T12:04:03,2022-02-16T18:02:12,75,46,0.613333,1.0,16.0,True,0.666667,0.818182


,venduto_da,rank_pos_first,rank_pct_first,rank_pos_second,rank_pct_second,abs_rank_change,abs_rank_pct_change,market_date,first_timestamp,second_timestamp
2470,Leelbox-EU,37,0.404494,6,0.056818,31,0.347676,2022-03-09,2022-03-09T12:01:41,2022-03-09T18:02:01
2306,Red Tech GmbH,55,0.586957,81,0.869565,26,0.282609,2022-03-07,2022-03-07T12:02:07,2022-03-07T18:01:43
1637,Monclick,8,0.080460,33,0.367816,25,0.287356,2022-02-28,2022-02-28T12:01:51,2022-02-28T18:01:44
1949,PARTY-GAMES,57,0.629213,33,0.359551,24,0.269663,2022-03-03,2022-03-03T12:01:31,2022-03-03T18:07:18
2310,PARTY-GAMES,59,0.630435,35,0.369565,24,0.260870,2022-03-07,2022-03-07T12:02:07,2022-03-07T18:01:43
208,PARTY-GAMES,58,0.740260,35,0.441558,23,0.298701,2022-02-09,2022-02-09T12:08:19,2022-02-09T18:02:46
2039,PARTY-GAMES,57,0.643678,34,0.370787,23,0.272892,2022-03-04,2022-03-04T12:01:39,2022-03-04T18:01:41
1860,PARTY-GAMES,55,0.613636,32,0.356322,23,0.257315,2022-03-02,2022-03-02T12:01:40,2022-03-02T18:01:41
132,PARTY-GAMES,57,0.736842,35,0.441558,22,0.295284,2022-02-08,2022-02-08T12:07:00,2022-02-08T18:05:35
56,PARTY-GAMES,57,0.727273,35,0.441558,22,0.285714,2022-02-07,2022-02-07T12:05:05,2022-02-07T18:09:10


Third-party offer-count support by market:


,count,mean,std,min,25%,50%,75%,max
third_party_market_size,62.0,82.370968,6.199508,72.0,77.0,80.0,88.75,93.0


,top_k_third_party_offers_per_market,rows_retained_if_capped_at_k,share_rows_retained_if_capped_at_k,markets_with_at_least_k_offers,share_markets_with_at_least_k_offers
0,3,186,0.036421,62,1.0
1,5,310,0.060701,62,1.0
2,10,620,0.121402,62,1.0
3,20,1240,0.242804,62,1.0
4,30,1860,0.364206,62,1.0
5,40,2480,0.485608,62,1.0
6,50,3100,0.607010,62,1.0


### 17. Audit conclusions and EDA export

The cell registers the final audit checks in the structured registry and exports the EDA tables to `Datasets/EDA-Results/`. The exported tables include the typed raw audit dataframe, the candidate sample definitions, the convenience-audit table, the variable-provenance table, the rank-summary table, the balance table, the seller-presence and FBA-switching tables, the review and contact-courier diagnostics, and the audit-check registry itself. These tables are the data input for Chapter 3 and the related Appendix of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 17. Audit-check registry and EDA export
# -----------------------------------------------------------------------------
# Register the final EDA audit checks and export the audited tables to the EDA-Results directory.

audit_check_table = pd.DataFrame(AUDIT_CHECKS)
print('Audit checks registered in this notebook:')
display(audit_check_table)

if EXPORT_FILES:
    file_dir = file_path.resolve().parent / EXPORT_SUBDIR_NAME
    file_dir.mkdir(parents=True, exist_ok=True)

    def _flatten_for_export(obj):
        if obj is None:
            return None
        if isinstance(obj, pd.Series):
            obj = obj.to_frame()

        df = obj.copy()

        if isinstance(df.index, pd.MultiIndex) or any(name is not None for name in df.index.names):
            df = df.reset_index()

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                '_'.join([str(part) for part in col if str(part) != '']).strip('_')
                for col in df.columns.to_flat_index()
            ]

        return df

    def export_csv(filename, obj):
        df = _flatten_for_export(obj)
        if df is None:
            return
        df.to_csv(file_dir / f'{filename}.csv', index=False)

    observed_both_status_sellers = seller_presence_summary.loc[
        seller_presence_summary['observed_both_fba_statuses']
    ].copy()

    across_market_fba_switchers = fba_switch_summary_seller_best.loc[
        fba_switch_summary_seller_best['fba_nunique'].gt(1)
    ].copy()

    manual_validation_delivery_shipping_parts = []

    if len(failed_delivery_markets) > 0:
        manual_validation_delivery_shipping_parts.append(
            df_raw.loc[
                df_raw['timestamp'].isin(failed_delivery_markets['timestamp']),
                [
                    'timestamp',
                    'venduto_da',
                    'spedito_da',
                    'spedizione_consegna',
                    'g_cons_min',
                    'g_cons_max',
                    'g_cons_min_robust',
                    'g_cons_max_robust',
                    'tipo_spedizione',
                    'prezzo_spedizione(€)',
                    'parsed_shipping_mode',
                    'parsed_shipping_price',
                ]
            ].head(6)
        )

    if len(paid_zero_explicit_examples) > 0:
        manual_validation_delivery_shipping_parts.append(
            paid_zero_explicit_examples.head(4)
        )

    if len(paid_zero_contact_examples) > 0:
        manual_validation_delivery_shipping_parts.append(
            paid_zero_contact_examples.head(4)
        )

    manual_validation_delivery_shipping_sample = (
        pd.concat(manual_validation_delivery_shipping_parts, ignore_index=True)
        if manual_validation_delivery_shipping_parts
        else pd.DataFrame()
    )

    files_to_export = {
        # Core audited data
        'df_raw_typed_audit': df_raw,
        'df_new_offer_raw': df_new_offer_raw,
        'df_new_offer_strict_dedup': df_new_offer_strict_dedup,
        'df_new_offer_relaxed_dedup': df_new_offer_relaxed_dedup,
        'df_new_seller_best': df_new_seller_best,
        'df_new_seller_best_thirdparty': df_new_seller_best_thirdparty,

        # Section 8
        'table_market_delivery_audit': market_delivery_audit,
        'table_failed_delivery_markets': failed_delivery_markets,
        'table_robust_vs_stored_delivery': robust_vs_stored_delivery,
        'table_failed_market_examples': failed_market_examples,

        # Section 9
        'table_shipping_audit_summary': shipping_audit_summary,
        'table_paid_zero_anomaly_breakdown': paid_zero_anomaly_breakdown,
        'table_paid_zero_explicit_examples': paid_zero_explicit_examples,
        'table_paid_zero_contact_examples': paid_zero_contact_examples,
        'table_paid_zero_explicit_by_timestamp': paid_zero_explicit_by_timestamp,
        'table_paid_zero_explicit_by_seller': paid_zero_explicit_by_seller,

        # Section 10
        'table_unit_of_analysis_summary': unit_of_analysis_summary,
        'table_candidate_sample_summary': candidate_sample_summary,
        'table_repeated_seller_examples': globals().get('repeated_seller_examples'),

        # Section 11
        'table_convenience_audit': convenience_audit_table,
        'table_convenience_takeaway': globals().get('convenience_takeaway'),

        # Section 12
        'table_outcome_support': outcome_support_table,

        # Section 13
        'table_group_size': globals().get('group_size_table'),
        'table_covariate_summary_by_fba': covariate_summary_by_fba,
        'table_covariate_smd': covariate_smd_table,
        'table_rank_summary_by_fba': globals().get('rank_summary_by_fba'),

        # Section 14
        'table_offer_repeat_headline': offer_repeat_headline_new,
        'table_market_fingerprint_summary': market_fingerprint_summary,
        'table_seller_presence_summary': seller_presence_summary,
        'table_seller_switch_headline': globals().get('seller_switch_headline'),
        'table_observed_both_status_sellers': observed_both_status_sellers,
        'table_mixed_seller_summary': globals().get('mixed_seller_summary'),
        'table_across_market_fba_switchers': across_market_fba_switchers,
        'table_ml_split_results': globals().get('ml_split_results'),
        'table_ml_split_summary': globals().get('ml_split_summary'),

        # Section 15
        'table_variable_provenance': table_variable_provenance,

# Section 16
'table_review_usability_headline': globals().get('review_usability_headline'),
'table_review_field_status': globals().get('review_field_status'),
'table_review_problem_examples': globals().get('review_problem_examples'),
'table_contact_courier_summary': globals().get('contact_courier_summary'),
'table_contact_courier_policy': globals().get('contact_courier_policy'),
'table_contact_courier_examples': globals().get('contact_courier_examples'),
'table_rank_variation_headline': globals().get('rank_variation_headline'),
'table_top_turnover_headline': globals().get('top_turnover_headline'),
'table_same_day_headline': globals().get('same_day_headline'),
'table_same_day_pair_table': globals().get('same_day_pair_table'),
'table_same_day_change_examples': globals().get('same_day_change_examples'),
'table_third_party_market_size_summary': globals().get('third_party_market_size_summary'),
'table_offer_threshold_support': globals().get('offer_threshold_support'),

        # Audit registry
        'table_audit_checks': audit_check_table,

        # Manual validation sample
        'manual_validation_delivery_shipping_sample': manual_validation_delivery_shipping_sample,
    }

    for filename, obj in files_to_export.items():
        export_csv(filename, obj)

    print('Files exported to:')
    print(file_dir)

Audit checks registered in this notebook:


,section,check_name,passed,detail
0,1,input_file_exists,True,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv
1,1,input_filename_recognized,True,Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv
2,1,input_file_nonempty,True,2967342 bytes
3,2,raw_loaded_nonempty,True,"(9424, 32)"
4,2,raw_column_count_expected,True,32 columns
5,2,raw_required_columns_present,True,[]
6,2,raw_column_names_unique,True,[]
7,3,glossary_covers_all_raw_columns,True,[]
8,3,glossary_has_no_extra_columns,True,[]
9,3,glossary_column_entries_unique,True,[]


Files exported to:
/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/EDA-Results


# Part II. Econometric analysis and output synthesis

Part II inherits the audited source-field logic of Part I, constructs the final seller-market panel, and estimates the static and dynamic ranking specifications. The empirical interpretation remains descriptive and associational; sections that record a narrower validation claim state that claim explicitly.


## B. FBA fulfillment status and ranking outcomes on Amazon Italy

## Single-product econometric audit: Xiaomi Mi Smart Band 6 seller-list panel

The empirical object is the association between Fulfillment by Amazon (FBA) status and observed seller-list ranking outcomes on Amazon.it for one product, the Xiaomi Mi Smart Band 6. The final estimation panel contains 5,107 seller-market observations, 62 timestamped offer-list snapshots, and 119 third-party seller identities. The primary static outcome is `rank_pct`, a normalized within-market rank where lower values denote better competitive position. The primary dynamic outcome is `rank_pct_improvement`, defined as lagged `rank_pct` minus lead `rank_pct`, so positive values denote upward movement between consecutive snapshots.

The empirical evidence is organized by identification strength. The notebook reports the raw within-market FBA ranking gap; estimates the controlled static residual association after conditioning on reputation, price, shipping, delivery, and market fixed effects; evaluates static support, overlap, bounded-outcome, and logistics-value-adjusted price diagnostics; estimates a dynamic FBA-by-turnover rank-movement association among continuing sellers with seller and transition fixed effects; validates the dynamic estimate using finite-cluster inference, starting-rank-position adjustments, temporal placebo checks, distance-banded mechanism diagnostics, and conservative support tests; and reports top-of-list binary prominence as auxiliary salience evidence.

The dynamic model hierarchy is:

- **D1 baseline turnover model**: seller fixed effects, transition fixed effects, lagged rank, lagged offer controls, total seller-list turnover, and directional-balance controls. The FBA-by-turnover coefficient is 0.002467 with p-value 0.0020.
- **D2 starting-rank-quartile adjustment**: D1 plus additive quartile indicators for lagged `rank_pct`; the omitted category is the top quartile. The coefficient is 0.002710 with p-value 0.0028.
- **D3 smooth starting-rank adjustment**: D1 plus a quadratic term in lagged `rank_pct`; the linear lagged-rank term is already included in the dynamic controls. The coefficient is 0.002278 with p-value 0.0013.
- **D4 saturated starting-rank stress test**: seller fixed effects plus `transition_id × lag_rank_tier` fixed effects. The coefficient is -0.000075 with p-value 0.9153 and condition number about 6.8e6, recorded as a severe support-sensitive stress test rather than the primary dynamic estimator.

All results are observational. The notebook does not identify the causal effect of adopting FBA, since the final panel contains no within-seller FBA switching; it does not identify Amazon's ranking algorithm, platform intent, sales, clicks, or external validity beyond this product page. The admissible claim is bounded: FBA status is associated with better observed rank position within markets and with marginally more favorable rank movement among continuing sellers during seller-list turnover, conditional on observed commercial and reputation characteristics.


### 1. Computational environment, provenance, and reproducibility discipline

The cell fixes the computational environment, resolves the raw CSV, and records the SHA256 fingerprint of the input file. The raw file used by the audited thesis run has SHA256 `88c85f33e3e640d6261d2ac1bb0b095ebf1c65ac8fc4edab9d850cd3c8abe5b5`. A mismatch between the file actually loaded and this fingerprint is logged in the audit registry before any econometric table is computed.

The reproducibility discipline has three layers. Every variable used by the regressions is reconstructed from the raw CSV in the active kernel. Every econometric table is recomputed before export; the saved CSV files are downstream products, not inputs to the results. The audit registry records sample counts, schema requirements, and numerical checks; critical checks halt execution when a required object is absent or inconsistent, while warnings remain visible when a diagnostic is numerically fragile, weakly powered, or intended only as a sensitivity analysis.

External context enters only as the institutional definition of the object under study: FBA refers to Amazon's fulfillment service, and the empirical setting is the Xiaomi Mi Smart Band 6 product page. External sources do not enter the estimated coefficients.


In [ ]:
# -----------------------------------------------------------------------------
# Section 1. Computational environment, provenance, and reproducibility discipline
# -----------------------------------------------------------------------------
# Load the Part II dependencies, fix display and configuration options, resolve the raw CSV path on Colab Drive or locally, and record the SHA256 fingerprint of the input.

from pathlib import Path
from datetime import datetime
import hashlib
import math
import re
import unicodedata
import warnings
import json
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
import gc

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import yaml
from IPython.display import display
from scipy import stats
from scipy import linalg
from scipy.special import expit
from patsy import build_design_matrices, dmatrices
from statsmodels.stats.diagnostic import linear_harvey_collier, linear_reset
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor
from statsmodels.stats.sandwich_covariance import cov_cluster, cov_cluster_2groups

try:
    from numba import njit, prange, set_num_threads
    NUMBA_AVAILABLE = True
    set_num_threads(min(8, max(1, os.cpu_count() or 1)))
except Exception:
    NUMBA_AVAILABLE = False
    njit = None
    prange = range

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 180)

# Notebook configuration
NOTEBOOK_VERSION = "final-empirical-documentation"
STRICT_ASSERTS = True
EXPORT_FILES = True
PRODUCT_LABEL = "Xiaomi Mi Smart Band 6"
TEMPORAL_HOLDOUT_FRAC = 0.20
FAST_VALIDATION_MODE = os.environ.get("NOTEBOOK_FAST_VALIDATION", "0").strip().lower() in {"1", "true", "yes"}
STANDARD_REPLICATIONS = 4999
FAST_VALIDATION_REPLICATIONS = int(os.environ.get("NOTEBOOK_FAST_REPLICATIONS", "49"))
WILD_BOOTSTRAP_REPLICATIONS = FAST_VALIDATION_REPLICATIONS if FAST_VALIDATION_MODE else STANDARD_REPLICATIONS
WILD_BOOTSTRAP_MIN_VALID_SHARE = 0.80
WILD_BOOTSTRAP_SEED = 20260504
DYNAMIC_PERMUTATION_REPLICATIONS = FAST_VALIDATION_REPLICATIONS if FAST_VALIDATION_MODE else STANDARD_REPLICATIONS
DYNAMIC_PERMUTATION_SEED = WILD_BOOTSTRAP_SEED + 202
RUN_STATIC_BOOTSTRAP_SENSITIVITY = False
WILD_BOOTSTRAP_REPLICATION_NOTE = (
    "The audited thesis analysis uses 4,999 bootstrap/permutation replications by default. "
    f"The optional NOTEBOOK_FAST_VALIDATION=1 mode uses {FAST_VALIDATION_REPLICATIONS} replications only as a smoke test and must not replace the complete-run inference."
)
PREFERRED_DYNAMIC_SPEC = "split_price_linear_stars"
PREFERRED_DYNAMIC_PREMIUM_TERM = "fba_x_dropouts_total_focal"
BINARY_OUTCOMES_PRIMARY_INFERENCE_INCLUDED = False
BINARY_OUTCOMES_AUXILIARY_SALIENCE_INCLUDED = True
BINARY_TOP10_COMPLEMENTARY_ESTIMAND_INCLUDED = True
BINARY_TOP10_PROMINENCE_THRESHOLD = 10
BINARY_PROMINENCE_THRESHOLDS = [5, 10, 15, 20]
BINARY_TOP10_PERMUTATION_REPLICATIONS = FAST_VALIDATION_REPLICATIONS if FAST_VALIDATION_MODE else STANDARD_REPLICATIONS
BINARY_TOP10_PERMUTATION_SEED = WILD_BOOTSTRAP_SEED + 404
BINARY_TOP10_WILD_BOOTSTRAP_REPLICATIONS = WILD_BOOTSTRAP_REPLICATIONS
MDE_MULTIPLIER_80_POWER_ALPHA_005 = 2.80
# Fast-delivery monetary imputation. The scraped CSV identifies the presence
# and timing of a faster-delivery option but does not expose a separate euro
# price for that option. The thesis therefore keeps EUR 4.99 as the baseline
# benchmark for the monetary value/cost of the fast-delivery component and
# treats lower/higher values only as sensitivity scenarios, never as raw
# variables estimated from the sample. The FBA-only logistics-value-adjusted
# specification applies this benchmark only to FBA offers with fast-delivery
# availability, before the static model is estimated.
FAST_DELIVERY_PREMIUM_EUR = 4.99
FAST_DELIVERY_PREMIUM_LABEL = "premium_delivery_baseline_eur_4_99"
FAST_DELIVERY_PREMIUM_SOURCE = (
    "External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; "
    "baseline EUR 4.99, not directly observed in the scraped CSV"
)
FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR = {
    "pickup_point_lower_sensitivity_eur_3_99": 3.99,
    "premium_delivery_baseline_eur_4_99": 4.99,
    "same_day_upper_sensitivity_eur_8_99": 8.99,
}

# Paths
DRIVE_DATASETS_DIR = Path("/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets")
LOCAL_OUTPUT_DIR = Path("econometrics_results")
OUTPUT_DIR = DRIVE_DATASETS_DIR / "Econometrics-Results" if DRIVE_DATASETS_DIR.exists() else LOCAL_OUTPUT_DIR
if EXPORT_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH_OVERRIDE = None
CANDIDATE_FILE_NAMES = [
    "Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv",
    "Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6 - Foglio1.csv",
]
CANDIDATE_DIRS = [
    Path.cwd(),
    Path("/mnt/data"),
    Path("."),
    Path("/content"),
    DRIVE_DATASETS_DIR,
]

EXPECTED_AUDIT_COUNTS = {
    "final_rows": 5107,
    "n_markets": 62,
    "fba_rows": 996,
    "non_fba_rows": 4111,
    "buy_box_third_party_support": 0,
    "review_problem_rows": 92,
    "review_problem_sellers": 3,
    "contact_courier_rows": 50,
    "contact_courier_sellers": 1,
    "mixed_fba_sellers_raw_new": 2,
    "mixed_fba_sellers_final": 0,
    "delivery_failure_markets": 5,
}

# -----------------------------------------------------------------------------
# Audit registry
# -----------------------------------------------------------------------------

AUDIT_CHECKS = []

def register_check(section, check_name, passed, detail=""):
    record = {
        "section": str(section),
        "check_name": str(check_name),
        "passed": bool(passed),
        "detail": str(detail),
    }
    AUDIT_CHECKS.append(record)
    if STRICT_ASSERTS and (not passed):
        raise AssertionError(f"[{section}] {check_name} UNSATISFIED: {detail}")

def audit_checks_frame():
    if not AUDIT_CHECKS:
        return pd.DataFrame(columns=["section", "check_name", "passed", "detail"])
    return pd.DataFrame(AUDIT_CHECKS)

# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------

def clean_string_columns(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or str(df[col].dtype) == "string":
            s = df[col].astype("string").str.strip()
            s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            df[col] = s
    return df

def parse_numeric_italian(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    s = series.astype("string")
    s = s.str.replace(".", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parse_boolean_flag(series):
    s = series.astype("string").str.strip().str.lower()
    mapped = s.map({"true": True, "false": False, "1": True, "0": False})
    return mapped.astype("boolean")

def strip_accents(text):
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )

def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    s = str(value).strip().lower()
    s = strip_accents(s)
    s = re.sub(r"\s+", " ", s)
    return s

def slugify_text(value):
    return re.sub(r"[^a-z0-9]+", "_", normalize_text(value)).strip("_")

def third_party_from_name(seller_name):
    return not bool(re.search(r"\bamazon\b", normalize_text(seller_name)))

def fba_from_shipper(shipper_name):
    return bool(re.search(r"\bamazon\b", normalize_text(shipper_name)))

def resolve_input_file(data_path_override=None, candidate_names=None, candidate_dirs=None):
    if data_path_override is not None:
        p = Path(data_path_override)
        if p.exists():
            return p.resolve()
        raise FileNotFoundError(f"Explicit DATA_PATH_OVERRIDE not found: {data_path_override}")

    candidate_names = candidate_names or CANDIDATE_FILE_NAMES
    candidate_dirs = candidate_dirs or CANDIDATE_DIRS

    for directory in candidate_dirs:
        for name in candidate_names:
            p = directory / name
            if p.exists():
                return p.resolve()

    search_roots = [Path.cwd(), Path("/mnt/data"), Path("/content")]
    for root in search_roots:
        if root.exists():
            for name in candidate_names:
                hits = list(root.rglob(name))
                if hits:
                    return hits[0].resolve()

    raise FileNotFoundError(
        "Xiaomi CSV not found in any expected location. Set DATA_PATH_OVERRIDE explicitly if needed."
    )

# -----------------------------------------------------------------------------
# Delivery and shipping parsing helpers
# -----------------------------------------------------------------------------

MONTH_MAP = {
    "gen": 1, "gennaio": 1,
    "feb": 2, "febbraio": 2,
    "mar": 3, "marzo": 3,
    "apr": 4, "aprile": 4,
    "mag": 5, "maggio": 5,
    "giu": 6, "giugno": 6,
    "lug": 7, "luglio": 7,
    "ago": 8, "agosto": 8,
    "set": 9, "sett": 9, "settembre": 9,
    "ott": 10, "ottobre": 10,
    "nov": 11, "novembre": 11,
    "dic": 12, "dicembre": 12,
}

range_two_months_re = re.compile(r"(\d{1,2})\s+([a-z]+)\s*-\s*(\d{1,2})\s+([a-z]+)")
range_same_month_re = re.compile(r"(\d{1,2})\s*-\s*(\d{1,2})\s+([a-z]+)")
single_date_re = re.compile(r"(?:lunedi|martedi|mercoledi|giovedi|venerdi|sabato|domenica)?\s*,?\s*(\d{1,2})\s+([a-z]+)")
robust_fast_split_re = re.compile(r"(?:oppure\s+)?consegna\s+piu\s+(?:veloce|rapida)\s*:?", re.I)
robust_primary_prefix_re = re.compile(
    r"^(?:spedizione\s+gratuita\s*:|consegna\s+gratuita\s*|consegna\s+a\s*[0-9]+(?:[\.,][0-9]+)?\s*€?\s*:?)",
    re.I,
)
shipping_price_re = re.compile(r"consegna\s+a\s*([0-9]+(?:[\.,][0-9]+)?)\s*€", re.I)
shipping_free_re = re.compile(r"(spedizione|consegna)\s+gratuita", re.I)
contact_courier_re = re.compile(r"verrai\s+contattato\s+dal\s+corriere", re.I)

def normalize_delivery_text(text):
    if text is None or pd.isna(text):
        return None
    s = str(text).lower()
    s = strip_accents(s)
    s = s.replace("maggiori informazioni", " ")
    s = re.sub(r"ordina entro [^.]*", " ", s)
    s = s.replace("sul tuo primo ordine idoneo", " ")
    s = s.replace(".", " ")
    s = s.replace(":", " : ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_delivery_date(ts, day, month_token):
    month_token = month_token.strip().lower()
    month = MONTH_MAP.get(month_token, MONTH_MAP.get(month_token[:3]))
    if month is None:
        return None
    year = ts.year
    if month < ts.month - 1:
        year += 1
    try:
        return datetime(year, month, int(day)).date()
    except ValueError:
        return None

def date_diff_days(ts, dt):
    if dt is None:
        return None
    delta = (dt - ts.date()).days
    return delta if delta >= 0 else None

def parse_delivery_segment(segment, ts):
    if segment is None:
        return {"g_cons_min": np.nan, "g_cons_max": np.nan}
    seg = normalize_delivery_text(segment)
    if not seg:
        return {"g_cons_min": np.nan, "g_cons_max": np.nan}

    m = range_two_months_re.search(seg)
    if m:
        start_dt = build_delivery_date(ts, m.group(1), m.group(2))
        end_dt = build_delivery_date(ts, m.group(3), m.group(4))
        return {"g_cons_min": date_diff_days(ts, start_dt), "g_cons_max": date_diff_days(ts, end_dt)}

    m = range_same_month_re.search(seg)
    if m:
        start_dt = build_delivery_date(ts, m.group(1), m.group(3))
        end_dt = build_delivery_date(ts, m.group(2), m.group(3))
        return {"g_cons_min": date_diff_days(ts, start_dt), "g_cons_max": date_diff_days(ts, end_dt)}

    m = single_date_re.search(seg)
    if m:
        dt = build_delivery_date(ts, m.group(1), m.group(2))
        dd = date_diff_days(ts, dt)
        return {"g_cons_min": dd, "g_cons_max": dd}

    return {"g_cons_min": np.nan, "g_cons_max": np.nan}

def parse_delivery_robust(text, ts):
    norm = normalize_delivery_text(text)
    if norm is None or pd.isna(ts):
        return (np.nan, np.nan, np.nan, np.nan)

    parts = robust_fast_split_re.split(norm, maxsplit=1)
    primary = robust_primary_prefix_re.sub("", parts[0]).strip()
    primary_parsed = parse_delivery_segment(primary, ts)

    if len(parts) > 1:
        fast_parsed = parse_delivery_segment(parts[1].strip(), ts)
    else:
        fast_parsed = {"g_cons_min": np.nan, "g_cons_max": np.nan}

    return (
        primary_parsed["g_cons_min"],
        primary_parsed["g_cons_max"],
        fast_parsed["g_cons_min"],
        fast_parsed["g_cons_max"],
    )

def parse_shipping_from_text(text):
    norm = normalize_delivery_text(text)
    if norm is None:
        return {"parsed_shipping_mode": pd.NA, "parsed_shipping_price": np.nan}
    m = shipping_price_re.search(norm)
    if m:
        price = float(m.group(1).replace(",", "."))
        return {"parsed_shipping_mode": "paid_explicit", "parsed_shipping_price": price}
    if contact_courier_re.search(norm):
        return {"parsed_shipping_mode": "unknown_contact_courier", "parsed_shipping_price": np.nan}
    if shipping_free_re.search(norm):
        return {"parsed_shipping_mode": "free_explicit", "parsed_shipping_price": 0.0}
    return {"parsed_shipping_mode": pd.NA, "parsed_shipping_price": np.nan}

# -----------------------------------------------------------------------------
# Resolve the audited input file and record provenance
# -----------------------------------------------------------------------------

file_path = resolve_input_file(DATA_PATH_OVERRIDE, CANDIDATE_FILE_NAMES, CANDIDATE_DIRS)
register_check("1", "input_file_exists", file_path.exists(), str(file_path))
register_check("1", "input_filename_recognized", file_path.name in CANDIDATE_FILE_NAMES, file_path.name)
register_check("1", "input_file_nonempty", file_path.stat().st_size > 0, f"{file_path.stat().st_size} bytes")

file_sha256 = hashlib.sha256(file_path.read_bytes()).hexdigest()

print(f"Notebook version: {NOTEBOOK_VERSION}")
print("Using input file:")
print(file_path)
print()
print("Raw-file provenance")
print(f"  File name : {file_path.name}")
print(f"  File size : {file_path.stat().st_size} bytes")
print(f"  SHA256    : {file_sha256}")

Notebook version: final-empirical-documentation
Using input file:
/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv

Raw-file provenance
  File name : Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv
  File size : 2967342 bytes
  SHA256    : 88c85f33e3e640d6261d2ac1bb0b095ebf1c65ac8fc4edab9d850cd3c8abe5b5


### 2. Raw extraction, seller identity, and final seller-market unit

The raw extraction contains 9,424 rows. The final econometric unit is a seller-market observation. A *market* is one timestamped offer-list snapshot, and a *seller* is a normalized seller-identity string after capitalization, accent stripping, and corporate-suffix unification. The cell reads the raw CSV, rebuilds the audited seller-level third-party sample inherited from Part I, and verifies its row counts against the audit registry: 9,424 raw rows, 5,786 new-condition rows, 5,169 seller-best rows, 5,107 third-party seller-best rows. The 5,107-row panel is the final static econometric panel.


In [ ]:
# -----------------------------------------------------------------------------
# Section 2. Raw extraction, seller identity, and final seller-market unit
# -----------------------------------------------------------------------------
# Read the raw CSV, rebuild the audited seller-best third-party sample, and verify its row counts against the audit registry.

raw_df = pd.read_csv(file_path).copy()
register_check("2", "raw_dataframe_nonempty", len(raw_df) > 0, f"rows={len(raw_df)}")

required_core_columns = [
    "timestamp",
    "condizione",
    "venduto_da",
    "spedito_da",
    "fba",
]
for col in required_core_columns:
    register_check("2", f"column_exists::{col}", col in raw_df.columns, col)

raw_df.insert(0, "raw_row_id", np.arange(len(raw_df), dtype=int))
raw_df["product_name"] = PRODUCT_LABEL
raw_df = clean_string_columns(raw_df)

raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")
raw_df["timestamp_date"] = raw_df["timestamp"].dt.normalize()

raw_df["seller_name"] = raw_df["venduto_da"].astype("string").str.strip()
raw_df["shipper_name"] = raw_df["spedito_da"].astype("string").str.strip()
raw_df["seller_name_norm"] = raw_df["seller_name"].map(normalize_text)
raw_df["seller_id"] = raw_df["seller_name"].map(slugify_text)
raw_df["shipper_name_norm"] = raw_df["shipper_name"].map(normalize_text)

raw_df["raw_identity_problem_flag"] = raw_df["seller_name"].isna() | raw_df["shipper_name"].isna()
register_check(
    "2",
    "raw_identity_problem_rows_count",
    int(raw_df["raw_identity_problem_flag"].sum()) == 1,
    f"raw_identity_problem_rows={int(raw_df['raw_identity_problem_flag'].sum())}",
)

raw_df["condizione"] = raw_df["condizione"].astype("string").str.strip()
raw_df["third_party_from_name"] = raw_df["seller_name"].map(third_party_from_name)
raw_df["fba_from_shipper"] = raw_df["shipper_name"].map(fba_from_shipper)
raw_df["fba_raw"] = parse_boolean_flag(raw_df["fba"])

numeric_map = {
    "prezzo": "prezzo",
    "prezzo_prod_venduto(€)": "prezzo_prodotto_venduto",
    "prezzo_spedizione(€)": "prezzo_spedizione_raw",
    "prezzo_totale(€)": "prezzo_totale_raw",
    "dif_prezzo": "dif_prezzo_raw",
    "dif_prezzo_sped": "dif_prezzo_sped_raw",
    "dif_prezzo_tot": "dif_prezzo_tot_raw",
    "rapp_piu_basso": "rapp_piu_basso_raw",
    "stelle": "stelle",
}
for old_col, new_col in numeric_map.items():
    raw_df[new_col] = parse_numeric_italian(raw_df[old_col])

integer_columns = [
    "buy_box",
    "visibility_order",
    "num_valutazioni",
    "valutazioni_positive",
    "qta_min",
    "g_cons_min",
    "g_cons_max",
    "g_spedizione",
    "g_cons_vel_min",
    "g_cons_vel_max",
    "g_spedizione_vel",
    "delta_consegna",
    "delta_num_val",
    "delta_val_pos(%)",
]
for col in integer_columns:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

preferred_offer_sample = (
    raw_df.loc[
        raw_df["condizione"].eq("Nuovo")
        & raw_df["third_party_from_name"]
        & (~raw_df["raw_identity_problem_flag"])
    ]
    .copy()
    .sort_values(["timestamp", "seller_name_norm", "visibility_order", "raw_row_id"], kind="mergesort")
)

df_final = (
    preferred_offer_sample.groupby(["timestamp", "seller_name_norm"], as_index=False, sort=False)
    .first()
    .copy()
)

df_final["entity_time_id"] = (
    df_final["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")
    + "__"
    + df_final["seller_id"]
)

shipping_text_parsed = df_final["spedizione_consegna"].apply(parse_shipping_from_text).apply(pd.Series)
df_final = pd.concat([df_final, shipping_text_parsed], axis=1)

df_final["shipping_amount_text"] = df_final["parsed_shipping_price"]
df_final["shipping_mode_text"] = df_final["parsed_shipping_mode"]
df_final["contact_courier_flag"] = df_final["shipping_mode_text"].eq("unknown_contact_courier")

df_final["explicit_shipping_repair_flag"] = (
    (~df_final["contact_courier_flag"])
    & df_final["shipping_amount_text"].notna()
    & df_final["prezzo_spedizione_raw"].fillna(0).eq(0)
    & df_final["shipping_amount_text"].gt(0)
)

df_final["prezzo_spedizione_repaired"] = np.where(
    df_final["shipping_amount_text"].notna(),
    df_final["shipping_amount_text"],
    df_final["prezzo_spedizione_raw"],
)

df_final["prezzo_spedizione_contact_zero"] = np.where(
    df_final["contact_courier_flag"],
    0.0,
    df_final["prezzo_spedizione_repaired"],
)

delivery_parsed = df_final.apply(
    lambda row: pd.Series(
        parse_delivery_robust(row["spedizione_consegna"], row["timestamp"]),
        index=["parsed_g_cons_min", "parsed_g_cons_max", "parsed_g_cons_vel_min", "parsed_g_cons_vel_max"],
    ),
    axis=1,
)
df_final = pd.concat([df_final, delivery_parsed], axis=1)

df_final["parsed_g_spedizione"] = df_final["parsed_g_cons_max"] - df_final["parsed_g_cons_min"]

market_delivery_flags = pd.concat(
    [
        df_final.groupby("timestamp")["g_cons_min"].apply(lambda s: s.fillna(0).eq(0).all()).rename("market_delivery_all_zero_flag"),
        df_final.groupby("timestamp")["parsed_g_cons_min"].apply(lambda s: s.fillna(0).gt(0).any()).rename("market_delivery_text_support_flag"),
    ],
    axis=1,
)
market_delivery_flags["market_delivery_failure_flag"] = (
    market_delivery_flags["market_delivery_all_zero_flag"]
    & market_delivery_flags["market_delivery_text_support_flag"]
)

df_final = df_final.merge(
    market_delivery_flags[["market_delivery_failure_flag"]],
    on="timestamp",
    how="left",
)

df_final["delivery_repaired_flag"] = (
    df_final["market_delivery_failure_flag"] & df_final["parsed_g_cons_min"].notna()
)

df_final["g_cons_min_robust"] = np.where(
    df_final["delivery_repaired_flag"],
    df_final["parsed_g_cons_min"],
    df_final["g_cons_min"],
)
df_final["g_cons_max_robust"] = np.where(
    df_final["delivery_repaired_flag"] & df_final["parsed_g_cons_max"].notna(),
    df_final["parsed_g_cons_max"],
    df_final["g_cons_max"],
)
df_final["g_spedizione_robust"] = np.where(
    df_final["delivery_repaired_flag"] & df_final["parsed_g_spedizione"].notna(),
    df_final["parsed_g_spedizione"],
    df_final["g_spedizione"],
)

# Additional delivery-structure variables used only in extended-control sensitivity checks.
# They are not promoted to the headline specification because delivery variables may be
# channels through which FBA is associated with ranking, not merely confounders.
df_final["delivery_window_robust"] = (df_final["g_cons_max_robust"] - df_final["g_cons_min_robust"]).clip(lower=0)
df_final["g_cons_vel_min_robust"] = np.where(
    df_final["parsed_g_cons_vel_min"].notna(),
    df_final["parsed_g_cons_vel_min"],
    df_final["g_cons_vel_min"],
)
df_final["g_cons_vel_max_robust"] = np.where(
    df_final["parsed_g_cons_vel_max"].notna(),
    df_final["parsed_g_cons_vel_max"],
    df_final["g_cons_vel_max"],
)
df_final["fast_delivery_available"] = (
    pd.Series(df_final["g_cons_vel_min_robust"]).fillna(0).gt(0)
    | pd.Series(df_final["g_cons_vel_max_robust"]).fillna(0).gt(0)
).astype(int)

# FBA-only fast-delivery value proxy. The raw extraction records the
# availability and timing of the faster delivery option but does not expose a
# separate price for that option. The thesis sensitivity specification therefore
# imputes EUR 4.99 as the baseline Premium fast-delivery benchmark only when
# an FBA offer has fast delivery available. This is a logistics-value adjustment,
# not a directly observed checkout price.
df_final["fast_delivery_cost_imputed"] = (
    FAST_DELIVERY_PREMIUM_EUR
    * df_final["fast_delivery_available"]
    * df_final["fba_from_shipper"]
)
df_final["prezzo_totale_fast_delivery_imputed"] = (
    df_final["prezzo"]
    + df_final["prezzo_spedizione_repaired"]
    + df_final["fast_delivery_cost_imputed"]
)

# One-way FBA-only monetary sensitivity translations. These columns are exported
# for audit transparency but are not included in the headline specification.
for _fast_delivery_scenario, _fast_delivery_premium in FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR.items():
    _scenario_slug = slugify_text(_fast_delivery_scenario)
    _cost_col = f"fast_delivery_cost_imputed_{_scenario_slug}"
    _total_col = f"prezzo_totale_fast_delivery_imputed_{_scenario_slug}"
    df_final[_cost_col] = (
        float(_fast_delivery_premium)
        * df_final["fast_delivery_available"]
        * df_final["fba_from_shipper"]
    )
    df_final[_total_col] = (
        df_final["prezzo"]
        + df_final["prezzo_spedizione_repaired"]
        + df_final[_cost_col]
    )

df_final["prezzo_totale_reconstructed"] = df_final["prezzo"] + df_final["prezzo_spedizione_repaired"]
df_final["prezzo_totale_contact_zero"] = df_final["prezzo"] + df_final["prezzo_spedizione_contact_zero"]

df_final["review_support_positive_flag"] = (
    df_final["num_valutazioni"].fillna(0).gt(0)
    & df_final["valutazioni_positive"].fillna(0).gt(0)
    & df_final["stelle"].fillna(0).gt(0)
)
df_final["review_problem_flag"] = ~df_final["review_support_positive_flag"]
df_final["log1p_num_valutazioni"] = np.log1p(df_final["num_valutazioni"].clip(lower=0))
df_final["fba_reconstruction_mismatch_flag"] = (
    (df_final["fba_raw"].astype("boolean") != df_final["fba_from_shipper"].astype("boolean")).fillna(False)
)

df_final = df_final.sort_values(
    ["timestamp", "visibility_order", "seller_name_norm", "raw_row_id"],
    kind="mergesort",
).copy()

df_final["market_id"] = df_final["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")
df_final["market_n_sellers"] = df_final.groupby("market_id")["entity_time_id"].transform("size")
df_final["rank_pos"] = df_final.groupby("market_id").cumcount() + 1
df_final["rank_pct"] = np.where(
    df_final["market_n_sellers"].gt(1),
    (df_final["rank_pos"] - 1) / (df_final["market_n_sellers"] - 1),
    0.0,
)

market_order = (
    pd.Series(sorted(df_final["market_id"].unique()), name="market_id")
    .to_frame()
    .assign(market_order=lambda d: np.arange(1, len(d) + 1, dtype=int))
)
n_markets = len(market_order)
temporal_holdout_markets = max(1, math.ceil(TEMPORAL_HOLDOUT_FRAC * n_markets))
market_order["temporal_holdout_flag"] = 0
market_order.loc[market_order["market_order"] > (n_markets - temporal_holdout_markets), "temporal_holdout_flag"] = 1
df_final = df_final.merge(market_order, on="market_id", how="left")

actual_audit_counts = {
    "final_rows": int(len(df_final)),
    "n_markets": int(df_final["market_id"].nunique()),
    "fba_rows": int(df_final["fba_from_shipper"].sum()),
    "non_fba_rows": int((~df_final["fba_from_shipper"]).sum()),
    "buy_box_third_party_support": int(df_final["buy_box"].sum()),
    "review_problem_rows": int(df_final["review_problem_flag"].sum()),
    "review_problem_sellers": int(df_final.loc[df_final["review_problem_flag"], "seller_name_norm"].nunique()),
    "contact_courier_rows": int(df_final["contact_courier_flag"].sum()),
    "contact_courier_sellers": int(df_final.loc[df_final["contact_courier_flag"], "seller_name_norm"].nunique()),
    "mixed_fba_sellers_raw_new": int(
        preferred_offer_sample.groupby("seller_name_norm")["fba_from_shipper"].nunique().gt(1).sum()
    ),
    "mixed_fba_sellers_final": int(
        df_final.groupby("seller_name_norm")["fba_from_shipper"].nunique().gt(1).sum()
    ),
    "delivery_failure_markets": int(
        df_final.loc[df_final["market_delivery_failure_flag"], "market_id"].nunique()
    ),
}

for metric, expected_value in EXPECTED_AUDIT_COUNTS.items():
    actual_value = actual_audit_counts[metric]
    register_check("2", f"audit_count::{metric}", actual_value == expected_value, f"expected={expected_value}, actual={actual_value}")

audit_reconciliation = pd.DataFrame(
    [{"metric": k, "expected": EXPECTED_AUDIT_COUNTS[k], "actual": actual_audit_counts[k], "matches_audit": EXPECTED_AUDIT_COUNTS[k] == actual_audit_counts[k]} for k in EXPECTED_AUDIT_COUNTS]
)
display(audit_reconciliation)

final_sample_preview = df_final[
    [
        "entity_time_id",
        "market_id",
        "seller_id",
        "seller_name",
        "timestamp",
        "market_order",
        "temporal_holdout_flag",
        "product_name",
        "rank_pos",
        "rank_pct",
        "fba_from_shipper",
        "log1p_num_valutazioni",
        "valutazioni_positive",
        "stelle",
        "prezzo_totale_reconstructed",
        "contact_courier_flag",
        "g_cons_min_robust",
        "g_cons_max_robust",
        "delivery_window_robust",
        "fast_delivery_available",
        "fast_delivery_cost_imputed",
        "prezzo_totale_fast_delivery_imputed",
        "prezzo",
        "prezzo_spedizione_repaired",
        "review_problem_flag",
        "explicit_shipping_repair_flag",
        "delivery_repaired_flag",
        "market_delivery_failure_flag",
    ]
].head(10)
display(final_sample_preview)

,metric,expected,actual,matches_audit
0,final_rows,5107,5107,True
1,n_markets,62,62,True
2,fba_rows,996,996,True
3,non_fba_rows,4111,4111,True
4,buy_box_third_party_support,0,0,True
5,review_problem_rows,92,92,True
6,review_problem_sellers,3,3,True
7,contact_courier_rows,50,50,True
8,contact_courier_sellers,1,1,True
9,mixed_fba_sellers_raw_new,2,2,True


,entity_time_id,market_id,seller_id,seller_name,timestamp,market_order,temporal_holdout_flag,product_name,rank_pos,rank_pct,fba_from_shipper,log1p_num_valutazioni,valutazioni_positive,stelle,prezzo_totale_reconstructed,contact_courier_flag,g_cons_min_robust,g_cons_max_robust,delivery_window_robust,fast_delivery_available,fast_delivery_cost_imputed,prezzo_totale_fast_delivery_imputed,prezzo,prezzo_spedizione_repaired,review_problem_flag,explicit_shipping_repair_flag,delivery_repaired_flag,market_delivery_failure_flag
0,2022-02-07T12:05:05__sky_line_e_comm,2022-02-07T12:05:05,sky_line_e_comm,SKY LINE E-COMM,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,1,0.000000,True,2.890372,94,4.5,38.5,False,8.0,8.0,0.0,0,0.00,38.5,38.5,0.0,False,False,False,False
1,2022-02-07T12:05:05__zaariogmbh,2022-02-07T12:05:05,zaariogmbh,ZaarioGmbH,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,2,0.012987,True,3.610918,97,5.0,38.93,False,7.0,10.0,3.0,1,4.99,43.92,38.93,0.0,False,False,False,False
2,2022-02-07T12:05:05__bacom,2022-02-07T12:05:05,bacom,Bacom,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,3,0.025974,False,9.569901,96,4.5,39.9,False,8.0,11.0,3.0,1,0.00,39.9,39.9,0.0,False,False,False,False
3,2022-02-07T12:05:05__goprice,2022-02-07T12:05:05,goprice,GoPrice,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,4,0.038961,False,8.823795,75,4.0,39.99,False,4.0,9.0,5.0,0,0.00,39.99,39.99,0.0,False,False,False,False
4,2022-02-07T12:05:05__dear_mi,2022-02-07T12:05:05,dear_mi,Dear Mi,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,5,0.051948,True,5.849325,95,4.5,40.0,False,4.0,4.0,0.0,0,0.00,40.0,40.0,0.0,False,False,False,False
5,2022-02-07T12:05:05__albaeleshop,2022-02-07T12:05:05,albaeleshop,Albaeleshop,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,6,0.064935,False,5.521461,95,4.5,40.7,False,8.0,11.0,3.0,0,0.00,40.7,40.7,0.0,False,False,False,False
6,2022-02-07T12:05:05__fortune_trade,2022-02-07T12:05:05,fortune_trade,FORTUNE TRADE,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,7,0.077922,True,5.164786,89,4.5,42.0,False,8.0,8.0,0.0,0,0.00,42.0,42.0,0.0,False,False,False,False
7,2022-02-07T12:05:05__excellent_mi_store,2022-02-07T12:05:05,excellent_mi_store,Excellent Mi Store,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,8,0.090909,True,6.469250,93,4.5,42.0,False,3.0,3.0,0.0,0,0.00,42.0,42.0,0.0,False,False,False,False
8,2022-02-07T12:05:05__niumi_technology,2022-02-07T12:05:05,niumi_technology,Niumi Technology,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,9,0.103896,True,6.821107,89,4.5,42.0,False,8.0,8.0,0.0,0,0.00,42.0,42.0,0.0,False,False,False,False
9,2022-02-07T12:05:05__gioiapura,2022-02-07T12:05:05,gioiapura,GIOIAPURA,2022-02-07 12:05:05,1,0,Xiaomi Mi Smart Band 6,10,0.116883,False,8.869961,89,4.5,43.64,False,3.0,7.0,4.0,0,0.00,43.64,43.64,0.0,False,False,False,False


### 3. Sample attrition and audit reconciliation

The attrition path is reported before estimation. The analysis starts from 9,424 raw rows. The new-condition restriction leaves 5,786 rows. Excluding malformed identity rows, repeated offer duplicates, and Amazon-as-seller rows produces the 5,169-row seller-best sample; the third-party restriction then yields the 5,107-row final panel. The cell registers each attrition step in the audit registry and writes the attrition table to `Datasets/Econometrics-Results/attrition_table.csv`. The table corresponds to Table 1 (Sample construction and attrition) of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 3. Sample attrition and audit reconciliation
# -----------------------------------------------------------------------------
# Reconstruct the attrition path from 9,424 raw rows to the 5,107-row final panel and write the attrition table to the Econometrics-Results directory.

_step_raw = len(raw_df)
_mask_new = raw_df["condizione"].astype(str).str.strip().str.lower().isin(
    ["nuovo", "new"]
) if "condizione" in raw_df.columns else pd.Series(True, index=raw_df.index)
_step_after_new = int(_mask_new.sum())
_seller_norm_raw = raw_df.get("seller_name", pd.Series([""] * len(raw_df))).astype(str).map(
    lambda s: unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii").strip().lower()
)
_mask_thirdparty = ~_seller_norm_raw.isin(["amazon", "amazon.it", "amazon italia"])
_step_after_thirdparty = int((_mask_new & _mask_thirdparty).sum())
_step_final = len(df_final)

attrition_table = pd.DataFrame([
    {"step": "raw_rows",                   "rows": _step_raw,
     "note": "All rows in the raw CSV"},
    {"step": "after_new_condition_filter", "rows": _step_after_new,
     "note": "Restrict to new-condition offers"},
    {"step": "after_third_party_filter",   "rows": _step_after_thirdparty,
     "note": "Exclude Amazon-as-seller and Amazon buy-box rows"},
    {"step": "final_econometric_panel",    "rows": _step_final,
     "note": "After malformed-identity exclusion and within-market re-ranking"},
])
attrition_table["rows_dropped_step"] = -attrition_table["rows"].diff().fillna(0).astype(int)
attrition_table = attrition_table[["step", "rows", "rows_dropped_step", "note"]]

print("Table B1 - Stepwise sample attrition")
display(attrition_table)
assert _step_final == len(df_final), "attrition table mismatch with df_final"

Table B1 - Stepwise sample attrition


,step,rows,rows_dropped_step,note
0,raw_rows,9424,0,All rows in the raw CSV
1,after_new_condition_filter,5786,3638,Restrict to new-condition offers
2,after_third_party_filter,5687,99,Exclude Amazon-as-seller and Amazon buy-box rows
3,final_econometric_panel,5107,580,After malformed-identity exclusion and within-market re-ranking


### 3a. Seller-list price-order audit

The seller list retained in each market is primarily ordered by reconstructed total price. The cell materializes the price-order audit reported in the data chapter of the thesis: it groups the final third-party panel by market, sorts sellers by recomputed rank, builds adjacent-pair comparisons, classifies each pair as price-consistent or price-inverted, and computes the market-level Spearman rank correlation between price and rank. The audit records 50 apparent inversions, all involving the same contact-courier offer whose checkout-relevant shipping price is unobserved. Once contact-courier offers are set aside, no confirmable departure from price sorting remains. The audit numerical record is exported to `price_order_audit_rankpct.csv` and `price_order_audit_trace_rankpct.csv`; the trace corresponds to Appendix Table B (`tab:app-price-order-trace`) of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 3a. Seller-list price-order audit
# -----------------------------------------------------------------------------
# Materialize the price-order audit on the final third-party panel: adjacent-pair classification, market-level Spearman correlation between price and rank, and replication trace.

_po = df_final[[
    "market_id", "rank_pos", "prezzo_totale_reconstructed",
    "fba_from_shipper", "contact_courier_flag", "seller_name_norm",
]].copy()
_po = _po.sort_values(["market_id", "rank_pos"], kind="mergesort").reset_index(drop=True)

# Adjacent-in-rank seller pairs within each retained market.
_po["next_market_id"] = _po["market_id"].shift(-1)
_po["next_price"] = _po["prezzo_totale_reconstructed"].shift(-1)
_po["next_fba"] = _po["fba_from_shipper"].shift(-1)
_po["next_contact_courier"] = _po["contact_courier_flag"].shift(-1)
_po["next_seller_name_norm"] = _po["seller_name_norm"].shift(-1)

_pairs = _po.loc[_po["market_id"].eq(_po["next_market_id"])].copy()

_pairs["price_non_decreasing"] = _pairs["next_price"] >= _pairs["prezzo_totale_reconstructed"]
_pairs["is_inversion"] = ~_pairs["price_non_decreasing"]
# A pair "involves a contact-courier row" when either adjacent seller is a
# contact-courier offer: the checkout-relevant shipping price is unobserved, so
# reconstructed total price is a lower bound and the apparent inversion cannot
# be confirmed as a genuine price-order violation.
_pairs["involves_contact_courier"] = (
    _pairs["contact_courier_flag"].astype(bool)
    | _pairs["next_contact_courier"].astype(bool)
)
_pairs["pair_two_non_fba"] = (
    (~_pairs["fba_from_shipper"].astype(bool)) & (~_pairs["next_fba"].astype(bool))
)
_pairs["pair_fba_above_non_fba"] = (
    _pairs["fba_from_shipper"].astype(bool) & (~_pairs["next_fba"].astype(bool))
)

_inv = _pairs.loc[_pairs["is_inversion"]].copy()
_inv_determinate = _inv.loc[~_inv["involves_contact_courier"]].copy()
_inv_contact = _inv.loc[_inv["involves_contact_courier"]].copy()

# Within-market Spearman correlation between rank position and reconstructed total price.
_spearman_by_market = (
    df_final.groupby("market_id")[["rank_pos", "prezzo_totale_reconstructed"]]
    .apply(lambda g: g["rank_pos"].corr(g["prezzo_totale_reconstructed"], method="spearman"))
)
_mean_within_market_spearman = float(_spearman_by_market.mean())

def _distinct_pair_identities(frame):
    pairs_iter = frame[["seller_name_norm", "next_seller_name_norm"]].astype(str)
    return {tuple(sorted(p)) for p in pairs_iter.itertuples(index=False, name=None)}

_n_markets_po = int(df_final["market_id"].nunique())
_n_pairs = int(len(_pairs))
_n_non_decreasing = int(_pairs["price_non_decreasing"].sum())
_n_inversions = int(_pairs["is_inversion"].sum())
_n_inversions_contact = int(_inv["involves_contact_courier"].sum())
_n_inversions_determinate = int(len(_inv_determinate))
_n_det_two_non_fba = int(_inv_determinate["pair_two_non_fba"].sum())
_n_det_fba_above = int(_inv_determinate["pair_fba_above_non_fba"].sum())
_n_inversion_markets = int(_inv["market_id"].nunique())
_n_inversion_pair_ids = int(len(_distinct_pair_identities(_inv)))
_inversion_share = _n_inversions / _n_pairs if _n_pairs else float("nan")

price_order_audit_table = pd.DataFrame([
    {"audit_quantity": "Retained markets",
     "value": str(_n_markets_po),
     "interpretation": "Timestamped third-party seller lists in the final panel."},
    {"audit_quantity": "Adjacent-in-rank seller pairs checked",
     "value": str(_n_pairs),
     "interpretation": "Adjacent comparisons after sorting sellers by recomputed rank position within each retained market."},
    {"audit_quantity": "Adjacent-in-rank pairs non-decreasing in total price",
     "value": str(_n_non_decreasing),
     "interpretation": "Seller-list order is overwhelmingly aligned with reconstructed total price."},
    {"audit_quantity": "Adjacent-in-rank price-order inversions",
     "value": str(_n_inversions),
     "interpretation": f"Apparent departures from exact price sorting, equal to {_inversion_share:.1%} of adjacent comparisons."},
    {"audit_quantity": "Inversions involving a contact-courier row",
     "value": str(_n_inversions_contact),
     "interpretation": "One adjacent seller is a contact-courier offer with an unobserved checkout shipping price; the apparent inversion cannot be confirmed."},
    {"audit_quantity": "Determinate inversions (both sellers fully price-observed)",
     "value": str(_n_inversions_determinate),
     "interpretation": "Inversions confirmable from observed prices."},
    {"audit_quantity": "Determinate inversions comparing two non-FBA sellers",
     "value": str(_n_det_two_non_fba),
     "interpretation": "Most confirmable inversions compare sellers with the same fulfillment classification."},
    {"audit_quantity": "Determinate inversions with FBA above a cheaper non-FBA seller",
     "value": str(_n_det_fba_above),
     "interpretation": "Only a small subset of confirmable inversions has the fulfillment pattern most relevant to the residual FBA interpretation."},
    {"audit_quantity": "Markets containing at least one inversion",
     "value": str(_n_inversion_markets),
     "interpretation": "Apparent inversions are spread across many markets rather than concentrated in a few."},
    {"audit_quantity": "Distinct adjacent seller-pair identities among inversions",
     "value": str(_n_inversion_pair_ids),
     "interpretation": "Apparent inversions recur across a limited set of seller-pair identities."},
    {"audit_quantity": "Mean within-market Spearman correlation",
     "value": f"{_mean_within_market_spearman:.3f}",
     "interpretation": "Rank position and reconstructed total price are nearly monotone in the retained panel."},
])

price_order_audit_trace_table = pd.DataFrame([
    {"step": "Market grouping",
     "operation": "Group the final third-party panel by retained seller-list market.",
     "resulting_quantity": f"{_n_markets_po} retained markets."},
    {"step": "Rank ordering",
     "operation": "Sort sellers inside each market by recomputed rank position.",
     "resulting_quantity": "Seller order used for adjacent-in-rank comparisons."},
    {"step": "Adjacent pairing",
     "operation": "Pair each seller with the immediately following seller in rank order.",
     "resulting_quantity": f"{_n_pairs} adjacent-in-rank seller pairs."},
    {"step": "Price-consistency check",
     "operation": "Code a pair as consistent when the following seller's reconstructed total price is weakly greater than the current seller's reconstructed total price.",
     "resulting_quantity": f"{_n_non_decreasing} price-consistent pairs and {_n_inversions} inversions."},
    {"step": "Correlation check",
     "operation": "Compute Spearman's rank correlation between recomputed rank position and reconstructed total price within each market, then average across retained markets.",
     "resulting_quantity": f"Mean within-market Spearman correlation of {_mean_within_market_spearman:.3f}."},
    {"step": "Inversion audit",
     "operation": "Split inverted adjacent pairs by contact-courier exposure, then classify the determinate inversions by FBA status.",
     "resulting_quantity": (
         f"{_n_inversions_contact} inversions involve a contact-courier row and cannot be confirmed; "
         f"of the {_n_inversions_determinate} determinate inversions, {_n_det_two_non_fba} compare two non-FBA sellers "
         f"and {_n_det_fba_above} place an FBA seller above a cheaper non-FBA seller."
     )},
])

print("Seller-list price-order audit")
display(price_order_audit_table)
print("Replication trace for the seller-list price-order audit")
display(price_order_audit_trace_table)

# Internal-consistency checks only (no hard-coded magic numbers), so a complete
# Colab run is never interrupted by the audit while still recording the invariants.
register_check(
    "3a", "price_order_pairs_equal_sellers_minus_markets",
    _n_pairs == int(len(df_final)) - _n_markets_po,
    f"pairs={_n_pairs}, sellers-markets={int(len(df_final)) - _n_markets_po}",
)
register_check(
    "3a", "price_order_consistent_plus_inversions_equal_pairs",
    _n_non_decreasing + _n_inversions == _n_pairs,
    f"{_n_non_decreasing} + {_n_inversions} = {_n_non_decreasing + _n_inversions}, pairs={_n_pairs}",
)
register_check(
    "3a", "price_order_determinate_plus_contact_equal_inversions",
    _n_inversions_determinate + _n_inversions_contact == _n_inversions,
    f"{_n_inversions_determinate} + {_n_inversions_contact} = {_n_inversions_determinate + _n_inversions_contact}, inversions={_n_inversions}",
)

if EXPORT_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    price_order_audit_table.to_csv(OUTPUT_DIR / "price_order_audit_rankpct.csv", index=False)
    price_order_audit_trace_table.to_csv(OUTPUT_DIR / "price_order_audit_trace_rankpct.csv", index=False)


Seller-list price-order audit


,audit_quantity,value,interpretation
0,Retained markets,62,Timestamped third-party seller lists in the final panel.
1,Adjacent-in-rank seller pairs checked,5045,Adjacent comparisons after sorting sellers by recomputed rank position within each retained market.
2,Adjacent-in-rank pairs non-decreasing in total price,4995,Seller-list order is overwhelmingly aligned with reconstructed total price.
3,Adjacent-in-rank price-order inversions,50,"Apparent departures from exact price sorting, equal to 1.0% of adjacent comparisons."
4,Inversions involving a contact-courier row,50,One adjacent seller is a contact-courier offer with an unobserved checkout shipping price; the apparent inversion cannot be confirmed.
5,Determinate inversions (both sellers fully price-observed),0,Inversions confirmable from observed prices.
6,Determinate inversions comparing two non-FBA sellers,0,Most confirmable inversions compare sellers with the same fulfillment classification.
7,Determinate inversions with FBA above a cheaper non-FBA seller,0,Only a small subset of confirmable inversions has the fulfillment pattern most relevant to the residual FBA interpretation.
8,Markets containing at least one inversion,50,Apparent inversions are spread across many markets rather than concentrated in a few.
9,Distinct adjacent seller-pair identities among inversions,2,Apparent inversions recur across a limited set of seller-pair identities.


Replication trace for the seller-list price-order audit


,step,operation,resulting_quantity
0,Market grouping,Group the final third-party panel by retained seller-list market.,62 retained markets.
1,Rank ordering,Sort sellers inside each market by recomputed rank position.,Seller order used for adjacent-in-rank comparisons.
2,Adjacent pairing,Pair each seller with the immediately following seller in rank order.,5045 adjacent-in-rank seller pairs.
3,Price-consistency check,Code a pair as consistent when the following seller's reconstructed total price is weakly greater than the current seller's reconstructed total price.,4995 price-consistent pairs and 50 inversions.
4,Correlation check,"Compute Spearman's rank correlation between recomputed rank position and reconstructed total price within each market, then average across retained markets.",Mean within-market Spearman correlation of 0.991.
5,Inversion audit,"Split inverted adjacent pairs by contact-courier exposure, then classify the determinate inversions by FBA status.","50 inversions involve a contact-courier row and cannot be confirmed; of the 0 determinate inversions, 0 compare two non-FBA sellers and 0 place an FBA seller above a cheaper no..."


### 3b. Seller-best collapse and turnover-construction audit

The seller-best collapse keeps one row per normalized seller per market and recomputes rank inside the retained competitive set. The cell compares the pre-collapse third-party offer sample with the final seller-best panel `df_final` and verifies that the collapse preserves market-level seller presence across consecutive snapshots. The audit confirms that duplicate-row removal and rank recomputation do not manufacture seller-presence turnover, although the cleaned rank scale changes because duplicate seller appearances are removed and the rank is recomputed inside the seller-level market. The diagnostic is exported to `turnover_construction_audit.csv` and corresponds to Appendix Table C of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 3b. Seller-best collapse and turnover-construction audit
# -----------------------------------------------------------------------------
# Compare the pre-collapse third-party offers with the final seller-best panel and verify that seller-presence turnover is preserved by the collapse.

_tc_raw = preferred_offer_sample[[
    "timestamp", "seller_name_norm", "seller_name",
    "visibility_order", "raw_row_id", "fba_from_shipper",
]].copy()
_tc_raw = _tc_raw.sort_values(
    ["timestamp", "visibility_order", "raw_row_id"], kind="mergesort"
).reset_index(drop=True)
_tc_raw["raw_third_party_rank"] = _tc_raw.groupby("timestamp").cumcount() + 1

# Seller-best row per (market, normalised seller): the minimum visibility-order offer.
_tc_best = (
    _tc_raw.sort_values(
        ["timestamp", "seller_name_norm", "visibility_order", "raw_row_id"], kind="mergesort"
    )
    .groupby(["timestamp", "seller_name_norm"], as_index=False)
    .first()
)
_market_order_map = df_final[["timestamp", "market_order"]].drop_duplicates()
_tc_best = _tc_best.merge(_market_order_map, on="timestamp", how="left")

# --- Check: raw-name aliases collapsed into one normalised seller identity ---
_alias_counts = (
    preferred_offer_sample.groupby(["timestamp", "seller_name_norm"])["seller_name"].nunique()
)
_alias_collapsed_cells = int((_alias_counts > 1).sum())

# --- Checks: integer-rank compression induced by the collapse ---
_comp = df_final[["market_id", "timestamp", "seller_name_norm", "rank_pos"]].merge(
    _tc_best[["timestamp", "seller_name_norm", "raw_third_party_rank"]],
    on=["timestamp", "seller_name_norm"], how="left",
)
_comp["rank_compression"] = _comp["raw_third_party_rank"] - _comp["rank_pos"]
_all_cells_matched = bool(_comp["raw_third_party_rank"].notna().all())
_compressed_cells = int((_comp["rank_compression"].fillna(0) != 0).sum())
_max_compression = int(_comp["rank_compression"].fillna(0).max())

# --- Check: seller set per market unchanged by the collapse ---
_final_sets = df_final.groupby("timestamp")["seller_name_norm"].apply(frozenset)
_raw_sets = _tc_raw.groupby("timestamp")["seller_name_norm"].apply(frozenset)
_changed_seller_set_markets = int(
    sum(_final_sets.get(ts) != _raw_sets.get(ts) for ts in _final_sets.index)
)
_n_markets_tc = int(_final_sets.shape[0])

# --- Checks: the collapse does not manufacture seller-presence turnover ---
# Rank-ordered seller sequence per market, before and after the collapse.
_final_order = (
    df_final.sort_values(["market_order", "rank_pos"], kind="mergesort")
    .groupby("market_order")["seller_name_norm"].apply(list)
)
_raw_best_order = (
    _tc_best.sort_values(
        ["market_order", "visibility_order", "raw_row_id"], kind="mergesort"
    )
    .groupby("market_order")["seller_name_norm"].apply(list)
)

def _turnover_counts(order_by_market):
    """Per-transition total and above-focal dropout counts, plus dropout/entrant sets."""
    keys = sorted(order_by_market.index)
    total_dropouts, above_focal = {}, {}
    dropout_sellers, entrant_sellers = {}, {}
    for prev_k, next_k in zip(keys[:-1], keys[1:]):
        prev_list = list(order_by_market[prev_k])
        next_list = list(order_by_market[next_k])
        next_set, prev_set = set(next_list), set(prev_list)
        dropped = [s for s in prev_list if s not in next_set]
        dropped_set = set(dropped)
        dropout_sellers[(prev_k, next_k)] = dropped_set
        entrant_sellers[(prev_k, next_k)] = set(next_list) - prev_set
        total_dropouts[(prev_k, next_k)] = len(dropped)
        running_above, af = 0, 0
        for s in prev_list:
            if s in dropped_set:
                running_above += 1
            elif s in next_set:
                af += running_above
        above_focal[(prev_k, next_k)] = af
    return total_dropouts, above_focal, dropout_sellers, entrant_sellers

_final_total, _final_above, _final_dropouts, _final_entrants = _turnover_counts(_final_order)
_raw_total, _raw_above, _raw_dropouts, _raw_entrants = _turnover_counts(_raw_best_order)

_total_dropout_mismatch = int(sum(_final_total[k] != _raw_total.get(k) for k in _final_total))
_above_focal_mismatch = int(sum(_final_above[k] != _raw_above.get(k) for k in _final_above))

# Final-panel dropouts/entrants must not still be present in the raw third-party
# data at the adjacent timestamp; otherwise the collapse would have created them.
_raw_set_by_order = _raw_best_order.apply(set)
_dropout_present_in_raw_next = 0
for (prev_k, next_k), sellers in _final_dropouts.items():
    raw_next = _raw_set_by_order.get(next_k, set())
    _dropout_present_in_raw_next += sum(1 for s in sellers if s in raw_next)
_entrant_present_in_raw_prev = 0
for (prev_k, next_k), sellers in _final_entrants.items():
    raw_prev = _raw_set_by_order.get(prev_k, set())
    _entrant_present_in_raw_prev += sum(1 for s in sellers if s in raw_prev)

_n_final_cells = int(len(df_final))

turnover_construction_audit_table = pd.DataFrame([
    {"check": "Markets with changed seller set after seller-best collapse",
     "value": f"{_changed_seller_set_markets} of {_n_markets_tc}",
     "interpretation": "Duplicate removal does not change seller presence within retained markets."},
    {"check": "Final-panel dropouts present in raw third-party data at the next timestamp",
     "value": str(_dropout_present_in_raw_next),
     "interpretation": "Observed dropouts are not generated by preprocessing."},
    {"check": "Final-panel entrants present in raw third-party data at the previous timestamp",
     "value": str(_entrant_present_in_raw_prev),
     "interpretation": "Observed entrants are not generated by preprocessing."},
    {"check": "Transition-level mismatch in total dropout counts",
     "value": str(_total_dropout_mismatch),
     "interpretation": "Total dropout exposure is unchanged by duplicate collapse."},
    {"check": "Transition-level mismatch in above-focal dropout counts",
     "value": str(_above_focal_mismatch),
     "interpretation": "Directional dropout exposure is unchanged by duplicate collapse."},
    {"check": "Raw-name aliases collapsed into the same normalized seller identity",
     "value": str(_alias_collapsed_cells),
     "interpretation": "Identity normalization does not merge distinct raw seller names."},
    {"check": "Seller-market cells with compressed integer rank after collapse",
     "value": f"{_compressed_cells} of {_n_final_cells}",
     "interpretation": "Re-ranking affects position for a minority of seller-market cells."},
    {"check": "Maximum integer-rank compression after collapse",
     "value": f"{_max_compression} positions",
     "interpretation": "Rank movement is interpreted in the cleaned seller-level panel."},
])

print("Seller-presence turnover construction audit")
display(turnover_construction_audit_table)

# Internal-consistency checks only, so a complete Colab run is never interrupted.
register_check(
    "3b", "turnover_audit_all_final_cells_matched_to_raw_rank",
    _all_cells_matched,
    "every final-panel seller-market cell is matched to a pre-collapse third-party rank",
)
register_check(
    "3b", "turnover_audit_compression_non_negative",
    _max_compression >= 0 and _compressed_cells >= 0,
    f"compressed_cells={_compressed_cells}, max_compression={_max_compression}",
)
register_check(
    "3b", "turnover_audit_compressed_cells_within_panel",
    _compressed_cells <= _n_final_cells,
    f"compressed_cells={_compressed_cells}, panel_rows={_n_final_cells}",
)

if EXPORT_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    turnover_construction_audit_table.to_csv(
        OUTPUT_DIR / "turnover_construction_audit.csv", index=False
    )


Seller-presence turnover construction audit


,check,value,interpretation
0,Markets with changed seller set after seller-best collapse,0 of 62,Duplicate removal does not change seller presence within retained markets.
1,Final-panel dropouts present in raw third-party data at the next timestamp,0,Observed dropouts are not generated by preprocessing.
2,Final-panel entrants present in raw third-party data at the previous timestamp,0,Observed entrants are not generated by preprocessing.
3,Transition-level mismatch in total dropout counts,0,Total dropout exposure is unchanged by duplicate collapse.
4,Transition-level mismatch in above-focal dropout counts,0,Directional dropout exposure is unchanged by duplicate collapse.
5,Raw-name aliases collapsed into the same normalized seller identity,0,Identity normalization does not merge distinct raw seller names.
6,Seller-market cells with compressed integer rank after collapse,207 of 5107,Re-ranking affects position for a minority of seller-market cells.
7,Maximum integer-rank compression after collapse,7 positions,Rank movement is interpreted in the cleaned seller-level panel.


### 4. Identification framework and admissible estimands

The cell states the identification boundaries before any result is interpreted. The empirical strategy is observational throughout. It does not identify a causal adoption effect because no seller is observed under both fulfillment statuses in the final panel.

**Static estimand.** The static specification estimates a within-market linear projection of `rank_pct` on FBA status and observed offer/seller controls. The FBA coefficient is identified by within-market variation in fulfillment status across third-party sellers in the same timestamped seller list.

**Dynamic estimand.** The dynamic specification estimates a fixed-effects linear projection of `rank_pct_improvement` on the interaction between FBA status and the count of disappearing sellers around the focal seller, conditional on seller and transition fixed effects and lagged offer controls. The interaction coefficient is identified by within-seller across-transition variation in turnover exposure.

The cell freezes the analysis-relevant identifier columns, control sets, headline inference choice (two-way clustering by seller and market for the static design; by seller and transition for the dynamic design), and the identification diagnostics carried through the rest of the notebook.


In [ ]:
# -----------------------------------------------------------------------------
# Section 4. Identification framework and admissible estimands
# -----------------------------------------------------------------------------
# Freeze the continuous-rank outcome, the control sets, the inference choice (two-way clustering), and the identification diagnostics carried through the rest of the notebook.

IDENTIFIER_COLUMNS = [
    "entity_time_id",
    "market_id",
    "seller_id",
    "seller_name",
    "timestamp",
    "market_order",
    "temporal_holdout_flag",
    "product_name",
]

CONTINUOUS_OUTCOME = "rank_pct"
CONTINUOUS_OUTCOME_LABEL = "Normalized within-market rank"
HEADLINE_SPEC = "spec_4_fba_reputation_price_shipping_delivery"
TOTAL_PRICE_PARALLEL_SPEC = "spec_3_fba_reputation_totalprice_delivery"
HEADLINE_INFERENCE = "two_way_seller_market"

SPECIFICATIONS = {
    "spec_1_fba_only": ["fba_from_shipper"],
    "spec_2_fba_reputation": [
        "fba_from_shipper",
        "log1p_num_valutazioni",
        "valutazioni_positive",
        "stelle",
    ],
    "spec_3_fba_reputation_totalprice_delivery": [
        "fba_from_shipper",
        "log1p_num_valutazioni",
        "valutazioni_positive",
        "stelle",
        "prezzo_totale_reconstructed",
        "contact_courier_flag",
        "g_cons_min_robust",
    ],
    "spec_4_fba_reputation_price_shipping_delivery": [
        "fba_from_shipper",
        "log1p_num_valutazioni",
        "valutazioni_positive",
        "stelle",
        "prezzo",
        "prezzo_spedizione_repaired",
        "contact_courier_flag",
        "g_cons_min_robust",
    ],
}

numeric_analysis_columns = [
    "rank_pct", "rank_pos", "fba_from_shipper", "log1p_num_valutazioni",
    "valutazioni_positive", "stelle", "prezzo_totale_reconstructed",
    "contact_courier_flag", "g_cons_min_robust", "g_cons_max_robust",
    "delivery_window_robust", "g_cons_vel_min_robust", "g_cons_vel_max_robust",
    "fast_delivery_available", "fast_delivery_cost_imputed",
    "prezzo_totale_fast_delivery_imputed", "qta_min", "prezzo",
    "prezzo_spedizione_repaired", "prezzo_spedizione_contact_zero",
    "prezzo_totale_contact_zero", "review_problem_flag",
    "review_support_positive_flag", "explicit_shipping_repair_flag",
    "delivery_repaired_flag", "market_delivery_failure_flag",
    "temporal_holdout_flag",
]
for col in numeric_analysis_columns:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

df_final["fba_from_shipper"] = df_final["fba_from_shipper"].astype(int)
df_final["contact_courier_flag"] = df_final["contact_courier_flag"].astype(int)
df_final["review_problem_flag"] = df_final["review_problem_flag"].astype(int)
df_final["market_delivery_failure_flag"] = df_final["market_delivery_failure_flag"].astype(int)
df_final["fast_delivery_available"] = df_final["fast_delivery_available"].fillna(0).astype(int)
df_final["fast_delivery_cost_imputed"] = (
    FAST_DELIVERY_PREMIUM_EUR
    * df_final["fast_delivery_available"]
    * df_final["fba_from_shipper"]
)
df_final["prezzo_totale_fast_delivery_imputed"] = (
    df_final["prezzo"]
    + df_final["prezzo_spedizione_repaired"]
    + df_final["fast_delivery_cost_imputed"]
)

# Star ratings are ordinal. The headline keeps the linear term for comparability
# with the nested specifications, while sensitivity checks also use category
# indicators with 4.5 stars as the reference category.
df_final["stelle_cat"] = df_final["stelle"].map(lambda x: f"{float(x):.1f}" if pd.notna(x) else pd.NA).astype("category")
STELLE_CATEGORICAL_TERM = 'C(stelle_cat, Treatment(reference="4.5"))'


PRICE_MEASURE_SPECIFICATIONS = {
    "split_price_headline": SPECIFICATIONS[HEADLINE_SPEC],
    "total_price_parallel": SPECIFICATIONS[TOTAL_PRICE_PARALLEL_SPEC],
}

STAR_FORM_TERMS = {
    "linear_stars": ["stelle"],
    "categorical_stars": [STELLE_CATEGORICAL_TERM],
}

POSITIVE_REVIEW_FORM_TERMS = {
    "positive_reviews_included": ["valutazioni_positive"],
    "positive_reviews_dropped": [],
}

PRICE_MEASURE_DEFINITIONS = {
    "split_price": "product price plus shipping price separately",
    "total_price": "total reconstructed buyer-facing price",
}

STAR_FORM_DEFINITIONS = {
    "linear_stars": "star rating entered linearly",
    "categorical_stars": "star rating entered as category indicators, 4.5-star reference",
}

POSITIVE_REVIEW_DEFINITIONS = {
    "positive_reviews_included": "positive-review percentage included",
    "positive_reviews_dropped": "positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy",
}


def build_rhs_for_price_star_and_positive_reviews(price_measure, star_form, positive_review_form="positive_reviews_included"):
    price_terms = ["prezzo", "prezzo_spedizione_repaired"] if price_measure == "split_price" else ["prezzo_totale_reconstructed"]
    return [
        "fba_from_shipper",
        "log1p_num_valutazioni",
        *POSITIVE_REVIEW_FORM_TERMS[positive_review_form],
        *STAR_FORM_TERMS[star_form],
        *price_terms,
        "contact_courier_flag",
        "g_cons_min_robust",
    ]


PRICE_STAR_METADATA = {}
for price_measure in ["split_price", "total_price"]:
    for star_form in ["linear_stars", "categorical_stars"]:
        for positive_review_form in ["positive_reviews_included", "positive_reviews_dropped"]:
            base_label = f"{price_measure}_{star_form}"
            model_label = base_label if positive_review_form == "positive_reviews_included" else f"{base_label}_no_positive_reviews"
            if price_measure == "split_price" and star_form == "linear_stars" and positive_review_form == "positive_reviews_included":
                role = "headline_price_decomposition"
            elif positive_review_form == "positive_reviews_dropped":
                role = "positive_review_exclusion_sensitivity"
            elif price_measure == "total_price":
                role = "parallel_total_price_sensitivity"
            else:
                role = "categorical_star_sensitivity"
            PRICE_STAR_METADATA[model_label] = {
                "price_measure": price_measure,
                "star_form": star_form,
                "positive_review_form": positive_review_form,
                "price_control_definition": PRICE_MEASURE_DEFINITIONS[price_measure],
                "star_control_definition": STAR_FORM_DEFINITIONS[star_form],
                "positive_review_control_definition": POSITIVE_REVIEW_DEFINITIONS[positive_review_form],
                "role": role,
            }


PRICE_STAR_SPECIFICATIONS = {
    label: build_rhs_for_price_star_and_positive_reviews(
        meta["price_measure"],
        meta["star_form"],
        meta["positive_review_form"],
    )
    for label, meta in PRICE_STAR_METADATA.items()
}

EXTENDED_CONTROL_SPECIFICATIONS = {
    "headline_baseline": SPECIFICATIONS[HEADLINE_SPEC],
    "delivery_max_substituted": [
        term if term != "g_cons_min_robust" else "g_cons_max_robust"
        for term in SPECIFICATIONS[HEADLINE_SPEC]
    ],
    "delivery_window_added": SPECIFICATIONS[HEADLINE_SPEC] + ["delivery_window_robust"],
    "fast_delivery_cost_imputed_added": SPECIFICATIONS[HEADLINE_SPEC] + ["fast_delivery_cost_imputed"],
    # qta_min is inspected in the support table but not estimated as an extended control
    # when it has no variation in the final sample. The extended logistics bundle uses
    # the FBA-only imputed monetary proxy rather than treating fast delivery as a
    # directly observed checkout-price component.
    "extended_logistics_bundle": SPECIFICATIONS[HEADLINE_SPEC] + [
        "delivery_window_robust",
        "fast_delivery_cost_imputed",
    ],
}

identification_diagnostics = pd.DataFrame(
    [
        {"metric": "final_rows", "value": int(len(df_final))},
        {"metric": "markets", "value": int(df_final["market_id"].nunique())},
        {"metric": "third_party_sellers", "value": int(df_final["seller_id"].nunique())},
        {"metric": "fba_rows", "value": int(df_final["fba_from_shipper"].sum())},
        {"metric": "non_fba_rows", "value": int((1 - df_final["fba_from_shipper"]).sum())},
        {"metric": "sellers_in_all_markets", "value": int(df_final.groupby("seller_id")["market_id"].nunique().eq(df_final["market_id"].nunique()).sum())},
        {"metric": "within_seller_fba_switches_final", "value": int(df_final.groupby("seller_id")["fba_from_shipper"].nunique().gt(1).sum())},
        {"metric": "within_seller_fba_switches_raw_new", "value": int(preferred_offer_sample.groupby("seller_name_norm")["fba_from_shipper"].nunique().gt(1).sum())},
        {"metric": "review_problem_rows", "value": int(df_final["review_problem_flag"].sum())},
        {"metric": "contact_courier_rows", "value": int(df_final["contact_courier_flag"].sum())},
        {"metric": "delivery_failure_markets", "value": int(df_final.loc[df_final["market_delivery_failure_flag"] == 1, "market_id"].nunique())},
    ]
)
display(identification_diagnostics)

specification_catalog = pd.DataFrame(
    [
        {
            "specification": k,
            "controls": v,
            "role": "headline" if k == HEADLINE_SPEC else "nested_sequence",
        }
        for k, v in SPECIFICATIONS.items()
    ]
)
display(specification_catalog)

price_measure_catalog = pd.DataFrame(
    [
        {
            "price_star_review_specification": k,
            "price_measure": PRICE_STAR_METADATA[k]["price_measure"],
            "star_form": PRICE_STAR_METADATA[k]["star_form"],
            "positive_review_form": PRICE_STAR_METADATA[k]["positive_review_form"],
            "price_control_definition": PRICE_STAR_METADATA[k]["price_control_definition"],
            "star_control_definition": PRICE_STAR_METADATA[k]["star_control_definition"],
            "positive_review_control_definition": PRICE_STAR_METADATA[k]["positive_review_control_definition"],
            "controls": v,
            "role": PRICE_STAR_METADATA[k]["role"],
        }
        for k, v in PRICE_STAR_SPECIFICATIONS.items()
    ]
)
display(price_measure_catalog)

extended_control_catalog = pd.DataFrame(
    [
        {
            "specification": k,
            "controls": v,
            "role": "canonical_extended_logistics_sensitivity" if k == "extended_logistics_bundle" else "extended_control_sensitivity",
            "fast_delivery_premium_eur": FAST_DELIVERY_PREMIUM_EUR if "fast_delivery_cost_imputed" in v else np.nan,
            "fast_delivery_cost_note": "fast_delivery_cost_imputed equals FAST_DELIVERY_PREMIUM_EUR times fast_delivery_available times fba_from_shipper; it is an FBA-only logistics-value proxy, not an observed checkout price" if "fast_delivery_cost_imputed" in v else "not applicable",
        }
        for k, v in EXTENDED_CONTROL_SPECIFICATIONS.items()
    ]
)
display(extended_control_catalog)

rank_outcome_summary = pd.DataFrame(
    [
        {
            "outcome": CONTINUOUS_OUTCOME,
            "rows": int(len(df_final)),
            "markets": int(df_final["market_id"].nunique()),
            "mean": float(df_final[CONTINUOUS_OUTCOME].mean()),
            "median": float(df_final[CONTINUOUS_OUTCOME].median()),
            "min": float(df_final[CONTINUOUS_OUTCOME].min()),
            "max": float(df_final[CONTINUOUS_OUTCOME].max()),
            "lower_values_mean": "better competitive position",
        }
    ]
)
display(rank_outcome_summary)

,metric,value
0,final_rows,5107
1,markets,62
2,third_party_sellers,119
3,fba_rows,996
4,non_fba_rows,4111
5,sellers_in_all_markets,52
6,within_seller_fba_switches_final,0
7,within_seller_fba_switches_raw_new,2
8,review_problem_rows,92
9,contact_courier_rows,50


,specification,controls,role
0,spec_1_fba_only,[fba_from_shipper],nested_sequence
1,spec_2_fba_reputation,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle]",nested_sequence
2,spec_3_fba_reputation_totalprice_delivery,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo_totale_reconstructed, contact_courier_flag, g_cons_min_robust]",nested_sequence
3,spec_4_fba_reputation_price_shipping_delivery,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust]",headline


,price_star_review_specification,price_measure,star_form,positive_review_form,price_control_definition,star_control_definition,positive_review_control_definition,controls,role
0,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,product price plus shipping price separately,star rating entered linearly,positive-review percentage included,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust]",headline_price_decomposition
1,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,product price plus shipping price separately,star rating entered linearly,positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,"[fba_from_shipper, log1p_num_valutazioni, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust]",positive_review_exclusion_sensitivity
2,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage included,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, C(stelle_cat, Treatment(reference=""4.5"")), prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_...",categorical_star_sensitivity
3,split_price_categorical_stars_no_positive_reviews,split_price,categorical_stars,positive_reviews_dropped,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,"[fba_from_shipper, log1p_num_valutazioni, C(stelle_cat, Treatment(reference=""4.5"")), prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust]",positive_review_exclusion_sensitivity
4,total_price_linear_stars,total_price,linear_stars,positive_reviews_included,total reconstructed buyer-facing price,star rating entered linearly,positive-review percentage included,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo_totale_reconstructed, contact_courier_flag, g_cons_min_robust]",parallel_total_price_sensitivity
5,total_price_linear_stars_no_positive_reviews,total_price,linear_stars,positive_reviews_dropped,total reconstructed buyer-facing price,star rating entered linearly,positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,"[fba_from_shipper, log1p_num_valutazioni, stelle, prezzo_totale_reconstructed, contact_courier_flag, g_cons_min_robust]",positive_review_exclusion_sensitivity
6,total_price_categorical_stars,total_price,categorical_stars,positive_reviews_included,total reconstructed buyer-facing price,"star rating entered as category indicators, 4.5-star reference",positive-review percentage included,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, C(stelle_cat, Treatment(reference=""4.5"")), prezzo_totale_reconstructed, contact_courier_flag, g_cons_min_robust]",parallel_total_price_sensitivity
7,total_price_categorical_stars_no_positive_reviews,total_price,categorical_stars,positive_reviews_dropped,total reconstructed buyer-facing price,"star rating entered as category indicators, 4.5-star reference",positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,"[fba_from_shipper, log1p_num_valutazioni, C(stelle_cat, Treatment(reference=""4.5"")), prezzo_totale_reconstructed, contact_courier_flag, g_cons_min_robust]",positive_review_exclusion_sensitivity


,specification,controls,role,fast_delivery_premium_eur,fast_delivery_cost_note
0,headline_baseline,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust]",extended_control_sensitivity,NaN,not applicable
1,delivery_max_substituted,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_max_robust]",extended_control_sensitivity,NaN,not applicable
2,delivery_window_added,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust, delivery_window_robust]",extended_control_sensitivity,NaN,not applicable
3,fast_delivery_cost_imputed_added,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust, fast_delivery_cost_imputed]",extended_control_sensitivity,4.99,"fast_delivery_cost_imputed equals FAST_DELIVERY_PREMIUM_EUR times fast_delivery_available times fba_from_shipper; it is an FBA-only logistics-value proxy, not an observed check..."
4,extended_logistics_bundle,"[fba_from_shipper, log1p_num_valutazioni, valutazioni_positive, stelle, prezzo, prezzo_spedizione_repaired, contact_courier_flag, g_cons_min_robust, delivery_window_robust, fas...",canonical_extended_logistics_sensitivity,4.99,"fast_delivery_cost_imputed equals FAST_DELIVERY_PREMIUM_EUR times fast_delivery_available times fba_from_shipper; it is an FBA-only logistics-value proxy, not an observed check..."


,outcome,rows,markets,mean,median,min,max,lower_values_mean
0,rank_pct,5107,62,0.5,0.5,0.0,1.0,better competitive position


### 5. Estimand map and interpretation rules

The analysis distinguishes empirical objects before any result is reported. The estimand map separates descriptive within-market associations, controlled residual associations, dynamic turnover-conditional associations, and auxiliary binary salience translations, and labels each one with the interpretive rule it admits. The cell stores this map in `estimand_map_table` and reuses it in the synthesis sections at the end of the notebook to keep the labelling consistent across exports.


In [ ]:
# -----------------------------------------------------------------------------
# Section 5. Estimand map and interpretation rules
# -----------------------------------------------------------------------------
# Build the explicit map from each empirical block to its admissible interpretation, keeping the labelling consistent across the synthesis exports.

estimand_map_table = pd.DataFrame([
    {
        "estimand": "1. Total FBA-bundle association",
        "empirical_object": "rank_pct on FBA + market FE (spec_1); +reputation (spec_2)",
        "main_table": "Table C1 spec_1 / spec_2",
        "identifying_variation": "Within-market between-seller variation in FBA status",
        "what_it_supports": "Descriptive within-market ranking advantage attached to FBA bundle",
        "what_it_outside_scope": "Causal effect of FBA; effect of switching INTO FBA",
    },
    {
        "estimand": "2. Residual FBA-label association",
        "empirical_object": "rank_pct on FBA + reputation + price + shipping + delivery + market FE (spec_3, spec_4)",
        "main_table": "Table C1 spec_3 / spec_4 (headline)",
        "identifying_variation": "Within-market variation in FBA status conditional on observed bundle",
        "what_it_supports": "Association of the FBA label after holding observed offer characteristics fixed",
        "what_it_outside_scope": "Total FBA effect if price/shipping/delivery are mechanisms (post-treatment-control concern)",
    },
    {
        "estimand": "2b. Comparable-support residual",
        "empirical_object": "Spec_4 estimated inside common-support / overlap-trimmed samples",
        "main_table": "Tables D1-D5",
        "identifying_variation": "Same as Estimand 2, on a restricted comparison population",
        "what_it_supports": "Residual association among offers with overlapping observed propensity",
        "what_it_outside_scope": "Population-representative effect; common-support sample is selected (see D4)",
    },
    {
        "estimand": "2c. FBA-only logistics-value-adjusted static association",
        "empirical_object": "rank_pct on FBA + reputation + FBA-only logistics-value-adjusted total price + delivery + market FE",
        "main_table": "Table C2c / logistics_value_adjusted_price_results_rankpct.csv",
        "identifying_variation": "Within-market cross-seller variation after monetizing the latent fast-delivery component only for FBA listings with fast-delivery availability",
        "what_it_supports": "Whether a residual FBA ranking premium remains after narrowing price differences by adding an imputed hidden logistics value/cost to FBA prices before estimation",
        "what_it_outside_scope": "A directly measured checkout-price effect; the adjustment is an imputed post-treatment sensitivity, not a causal decomposition",
    },
    {
        "estimand": "3. Dynamic FBA turnover-premium response",
        "empirical_object": "rank_pct_improvement on FBA x total seller turnover + directional churn control + lagged controls + seller FE + transition FE",
        "main_table": "Tables F5b, F5c, Table 7",
        "identifying_variation": "Within-seller variation in turnover exposure, compared within transition, with FBA status interacted with turnover",
        "what_it_supports": "FBA sellers gain more rank position during seller-turnover transitions, conditional on seller identity, transition shocks, lagged rank, and observed lagged offer characteristics",
        "what_it_outside_scope": "A pure seller-FE FBA main effect; no within-seller FBA switching is available in the final sample",
    },
])

print("Table B2 - Estimand map (the navigation key for the rest of the notebook)")
display(estimand_map_table)


estimand_map_addendum_table = pd.DataFrame([
    {
        "estimand": "5. Common-support feasibility diagnostic",
        "empirical_object": "Propensity-score support and post-restriction covariate balance",
        "main_table": "Table D2b / overlap_design_decision_rankpct.csv",
        "identifying_variation": "Observed covariate overlap between FBA and non-FBA offers",
        "what_it_supports": "Whether a static like-for-like comparison is empirically feasible",
        "what_it_outside_scope": "It does not refute the full-sample association when balance fails",
        "claim_language": "Overlap restrictions are diagnostics unless they achieve sufficient observed balance.",
    },
    {
        "estimand": "6. Dynamic turnover-conditional rank-movement premium",
        "empirical_object": "rank_pct_improvement on FBA x total seller turnover + seller FE + transition FE",
        "main_table": "Table F5c / dynamic_fba_premium_summary_rankpct.csv",
        "identifying_variation": "Between-FBA-status differential rank movement across transitions with different turnover intensity",
        "what_it_supports": "FBA sellers move differently during seller-list turnover episodes",
        "what_it_outside_scope": "It is not a direct estimate of FBA adoption and not direct proof of inventory stockout",
        "claim_language": "Dynamic evidence is a turnover-conditional rank-movement premium under a maintained stockout-consistent interpretation.",
    },
])

print("Estimand map addendum introduced by the methodological hardening revision")
display(estimand_map_addendum_table)

Table B2 - Estimand map (the navigation key for the rest of the notebook)


,estimand,empirical_object,main_table,identifying_variation,what_it_supports,what_it_outside_scope
0,1. Total FBA-bundle association,rank_pct on FBA + market FE (spec_1); +reputation (spec_2),Table C1 spec_1 / spec_2,Within-market between-seller variation in FBA status,Descriptive within-market ranking advantage attached to FBA bundle,Causal effect of FBA; effect of switching INTO FBA
1,2. Residual FBA-label association,"rank_pct on FBA + reputation + price + shipping + delivery + market FE (spec_3, spec_4)",Table C1 spec_3 / spec_4 (headline),Within-market variation in FBA status conditional on observed bundle,Association of the FBA label after holding observed offer characteristics fixed,Total FBA effect if price/shipping/delivery are mechanisms (post-treatment-control concern)
2,2b. Comparable-support residual,Spec_4 estimated inside common-support / overlap-trimmed samples,Tables D1-D5,"Same as Estimand 2, on a restricted comparison population",Residual association among offers with overlapping observed propensity,Population-representative effect; common-support sample is selected (see D4)
3,2c. FBA-only logistics-value-adjusted static association,rank_pct on FBA + reputation + FBA-only logistics-value-adjusted total price + delivery + market FE,Table C2c / logistics_value_adjusted_price_results_rankpct.csv,Within-market cross-seller variation after monetizing the latent fast-delivery component only for FBA listings with fast-delivery availability,Whether a residual FBA ranking premium remains after narrowing price differences by adding an imputed hidden logistics value/cost to FBA prices before estimation,"A directly measured checkout-price effect; the adjustment is an imputed post-treatment sensitivity, not a causal decomposition"
4,3. Dynamic FBA turnover-premium response,rank_pct_improvement on FBA x total seller turnover + directional churn control + lagged controls + seller FE + transition FE,"Tables F5b, F5c, Table 7","Within-seller variation in turnover exposure, compared within transition, with FBA status interacted with turnover","FBA sellers gain more rank position during seller-turnover transitions, conditional on seller identity, transition shocks, lagged rank, and observed lagged offer characteristics",A pure seller-FE FBA main effect; no within-seller FBA switching is available in the final sample


Estimand map addendum introduced by the methodological hardening revision


,estimand,empirical_object,main_table,identifying_variation,what_it_supports,what_it_outside_scope,claim_language
0,5. Common-support feasibility diagnostic,Propensity-score support and post-restriction covariate balance,Table D2b / overlap_design_decision_rankpct.csv,Observed covariate overlap between FBA and non-FBA offers,Whether a static like-for-like comparison is empirically feasible,It does not refute the full-sample association when balance fails,Overlap restrictions are diagnostics unless they achieve sufficient observed balance.
1,6. Dynamic turnover-conditional rank-movement premium,rank_pct_improvement on FBA x total seller turnover + seller FE + transition FE,Table F5c / dynamic_fba_premium_summary_rankpct.csv,Between-FBA-status differential rank movement across transitions with different turnover intensity,FBA sellers move differently during seller-list turnover episodes,It is not a direct estimate of FBA adoption and not direct proof of inventory stockout,Dynamic evidence is a turnover-conditional rank-movement premium under a maintained stockout-consistent interpretation.


### 6. Descriptive balance, price formation, and support

The descriptive block records why raw FBA comparisons cannot be read as causal. FBA offers are better ranked before any conditioning, but they also differ systematically from non-FBA offers on observable commercial characteristics: lower reconstructed total prices, zero displayed standard shipping prices by construction, fewer reviews, higher positive-review percentages, and higher star ratings. The cell builds the balance table by FBA status, the hypothesis-test row of the same table, the continuous-rank support diagnostics, and the market-overlap table. These objects populate Table 5 (`tab:balance-fba`) and Appendix Table B (`tab:final-panel-facts`) of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 6. Descriptive balance, price formation, and continuous-rank support
# -----------------------------------------------------------------------------
# Build the descriptive balance table by FBA status, the hypothesis-test row, the continuous-rank support diagnostics, and the market-overlap table.

def standardized_mean_difference(series, treat):
    x1 = series[treat == 1].dropna()
    x0 = series[treat == 0].dropna()
    if len(x1) == 0 or len(x0) == 0:
        return np.nan
    v = (x1.var(ddof=1) + x0.var(ddof=1)) / 2
    if pd.isna(v) or v <= 0:
        return np.nan
    return (x1.mean() - x0.mean()) / np.sqrt(v)


def balance_hypothesis_tests(data, variables):
    rows = []
    treat = data["fba_from_shipper"]
    for var in variables:
        x1 = pd.to_numeric(data.loc[treat == 1, var], errors="coerce").dropna()
        x0 = pd.to_numeric(data.loc[treat == 0, var], errors="coerce").dropna()
        try:
            welch = stats.ttest_ind(x1, x0, equal_var=False, nan_policy="omit")
            welch_stat, welch_p = float(welch.statistic), float(welch.pvalue)
        except Exception:
            welch_stat, welch_p = np.nan, np.nan
        try:
            mw = stats.mannwhitneyu(x1, x0, alternative="two-sided")
            mw_stat, mw_p = float(mw.statistic), float(mw.pvalue)
        except Exception:
            mw_stat, mw_p = np.nan, np.nan
        try:
            ks = stats.ks_2samp(x1, x0, alternative="two-sided", mode="auto")
            ks_stat, ks_p = float(ks.statistic), float(ks.pvalue)
        except Exception:
            ks_stat, ks_p = np.nan, np.nan

        by_market = (
            data.groupby(["market_id", "fba_from_shipper"])[var]
            .mean()
            .unstack()
            .dropna()
        )
        market_diff = by_market[1] - by_market[0] if {0, 1}.issubset(set(by_market.columns)) else pd.Series(dtype=float)
        if len(market_diff) > 1 and market_diff.std(ddof=1) > 0:
            mt = stats.ttest_1samp(market_diff, popmean=0.0)
            market_t_stat, market_t_p = float(mt.statistic), float(mt.pvalue)
        else:
            market_t_stat, market_t_p = np.nan, np.nan
        try:
            if len(market_diff) > 0 and not np.allclose(market_diff.to_numpy(), 0):
                wil = stats.wilcoxon(market_diff, alternative="two-sided", zero_method="wilcox")
                market_wilcoxon_stat, market_wilcoxon_p = float(wil.statistic), float(wil.pvalue)
            else:
                market_wilcoxon_stat, market_wilcoxon_p = np.nan, np.nan
        except Exception:
            market_wilcoxon_stat, market_wilcoxon_p = np.nan, np.nan

        rows.append({
            "variable": var,
            "n_fba": int(len(x1)),
            "n_nonfba": int(len(x0)),
            "mean_diff_fba_minus_nonfba": float(x1.mean() - x0.mean()) if len(x1) and len(x0) else np.nan,
            "median_diff_fba_minus_nonfba": float(x1.median() - x0.median()) if len(x1) and len(x0) else np.nan,
            "welch_t_stat": welch_stat,
            "welch_pvalue": welch_p,
            "mann_whitney_u_stat": mw_stat,
            "mann_whitney_pvalue": mw_p,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_p,
            "markets_with_both_groups": int(len(market_diff)),
            "market_mean_diff_fba_minus_nonfba": float(market_diff.mean()) if len(market_diff) else np.nan,
            "market_paired_t_stat": market_t_stat,
            "market_paired_t_pvalue": market_t_p,
            "market_wilcoxon_stat": market_wilcoxon_stat,
            "market_wilcoxon_pvalue": market_wilcoxon_p,
            "test_note": "Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.",
        })
    out = pd.DataFrame(rows)
    for pcol in ["welch_pvalue", "mann_whitney_pvalue", "ks_pvalue", "market_paired_t_pvalue", "market_wilcoxon_pvalue"]:
        valid = out[pcol].notna()
        adjusted = np.full(len(out), np.nan)
        if valid.any():
            adjusted[valid.to_numpy()] = multipletests(out.loc[valid, pcol], method="fdr_bh")[1]
        out[f"{pcol}_fdr_bh"] = adjusted
    return out

balance_variables = [
    "prezzo_totale_reconstructed",
    "prezzo",
    "prezzo_spedizione_repaired",
    "g_cons_min_robust",
    "fast_delivery_available",
    "fast_delivery_cost_imputed",
    "num_valutazioni",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
]

balance_table = []
treat = df_final["fba_from_shipper"]

for var in balance_variables:
    balance_table.append(
        {
            "variable": var,
            "mean_fba": float(df_final.loc[treat == 1, var].mean()),
            "mean_nonfba": float(df_final.loc[treat == 0, var].mean()),
            "median_fba": float(df_final.loc[treat == 1, var].median()),
            "median_nonfba": float(df_final.loc[treat == 0, var].median()),
            "smd_fba_minus_nonfba": float(standardized_mean_difference(df_final[var], treat)),
        }
    )

balance_table = pd.DataFrame(balance_table)
balance_hypothesis_tests_table = balance_hypothesis_tests(df_final, balance_variables)
display(balance_table)
display(balance_hypothesis_tests_table)

rank_summary_by_fba = (
    df_final.groupby("fba_from_shipper")[["rank_pos", "rank_pct"]]
    .agg(["mean", "median", "min", "max", "count"])
)
display(rank_summary_by_fba)

market_size_summary = (
    df_final.groupby("market_id").size().rename("third_party_market_size").describe().to_frame().T
)
display(market_size_summary)

market_overlap_table = (
    df_final.groupby("market_id")
    .agg(
        market_rows=("entity_time_id", "size"),
        fba_rows=("fba_from_shipper", "sum"),
        non_fba_rows=("fba_from_shipper", lambda s: int((1 - s).sum())),
        fba_share=("fba_from_shipper", "mean"),
    )
    .reset_index()
)
market_overlap_summary = pd.DataFrame(
    [
        {
            "markets": int(market_overlap_table["market_id"].nunique()),
            "markets_with_fba": int(market_overlap_table["fba_rows"].gt(0).sum()),
            "markets_with_non_fba": int(market_overlap_table["non_fba_rows"].gt(0).sum()),
            "min_fba_share": float(market_overlap_table["fba_share"].min()),
            "median_fba_share": float(market_overlap_table["fba_share"].median()),
            "max_fba_share": float(market_overlap_table["fba_share"].max()),
        }
    ]
)
display(market_overlap_summary)

# Shipping-price support by FBA status.
# This is used later to avoid an unsupported FBA-by-shipping interaction.
shipping_support_by_fba = (
    df_final.assign(positive_shipping=df_final["prezzo_spedizione_repaired"].gt(0).astype(int))
    .groupby("fba_from_shipper")
    .agg(
        rows=("rank_pct", "size"),
        positive_shipping_rows=("positive_shipping", "sum"),
        positive_shipping_share=("positive_shipping", "mean"),
        mean_shipping_price=("prezzo_spedizione_repaired", "mean"),
        median_shipping_price=("prezzo_spedizione_repaired", "median"),
        max_shipping_price=("prezzo_spedizione_repaired", "max"),
    )
    .reset_index()
)
display(shipping_support_by_fba)

# Delivery and support diagnostics used to motivate extended-control sensitivity.
delivery_support_table = df_final[[
    "g_cons_min_robust",
    "g_cons_max_robust",
    "delivery_window_robust",
    "fast_delivery_available",
    "qta_min",
]].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).T.reset_index().rename(columns={"index": "variable"})
display(delivery_support_table)

,variable,mean_fba,mean_nonfba,median_fba,median_nonfba,smd_fba_minus_nonfba
0,prezzo_totale_reconstructed,45.103484,55.945018,44.990000,55.390000,-1.552516
1,prezzo,45.103484,52.334829,44.990000,50.700000,-1.101420
2,prezzo_spedizione_repaired,0.000000,3.610190,0.000000,0.000000,-0.963208
3,g_cons_min_robust,6.662651,6.939188,6.000000,6.000000,-0.075647
4,fast_delivery_available,0.146586,0.205303,0.000000,0.000000,-0.154619
5,fast_delivery_cost_imputed,0.731466,0.000000,0.000000,0.000000,0.585820
6,num_valutazioni,317.631526,1479.713452,74.000000,241.000000,-0.604011
7,log1p_num_valutazioni,4.191823,5.545441,4.317488,5.488938,-0.630378
8,valutazioni_positive,94.993976,82.391146,97.000000,87.000000,0.946977
9,stelle,4.710843,4.225979,4.500000,4.500000,0.795075


,variable,n_fba,n_nonfba,mean_diff_fba_minus_nonfba,median_diff_fba_minus_nonfba,welch_t_stat,welch_pvalue,mann_whitney_u_stat,mann_whitney_pvalue,ks_statistic,ks_pvalue,markets_with_both_groups,market_mean_diff_fba_minus_nonfba,market_paired_t_stat,market_paired_t_pvalue,market_wilcoxon_stat,market_wilcoxon_pvalue,test_note,welch_pvalue_fdr_bh,mann_whitney_pvalue_fdr_bh,ks_pvalue_fdr_bh,market_paired_t_pvalue_fdr_bh,market_wilcoxon_pvalue_fdr_bh
0,prezzo_totale_reconstructed,996,4111,-10.841534,-10.40000,-53.347139,0.000000e+00,521064.0,1.219918e-292,0.681656,2.045432e-321,62,-10.748882,-110.845109,4.653584e-72,0.0,7.577765e-12,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,0.000000e+00,1.219918e-291,2.045432e-320,1.551195e-71,1.082538e-11
1,prezzo,996,4111,-7.231345,-5.71000,-36.831656,2.419816e-237,856083.0,4.113099e-179,0.545879,1.494581e-221,62,-7.158257,-94.844174,5.947098e-68,0.0,7.575538e-12,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,6.049540e-237,2.056550e-178,7.472904e-221,9.911831e-68,1.082538e-11
2,prezzo_spedizione_repaired,996,4111,-3.610190,0.00000,-43.669554,0.000000e+00,1218606.0,1.468301e-125,0.404768,2.040809e-118,62,-3.590626,-106.483891,5.318578e-71,0.0,7.537774e-12,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,0.000000e+00,2.447168e-125,5.102022e-118,1.063716e-70,1.082538e-11
3,g_cons_min_robust,996,4111,-0.276537,0.00000,-2.104460,3.550968e-02,1974689.0,8.001707e-02,0.151942,1.278014e-16,62,-0.288782,-1.587362,1.176011e-01,759.0,1.272819e-01,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,3.550968e-02,8.001707e-02,1.597517e-16,1.176011e-01,1.272819e-01
4,fast_delivery_available,996,4111,-0.058717,0.00000,-4.565209,5.352229e-06,1927069.0,2.607678e-05,0.058717,7.573314e-03,62,-0.059654,-3.568969,7.053880e-04,542.0,2.315958e-03,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,5.946921e-06,2.897420e-05,7.573314e-03,7.837645e-04,2.573286e-03
5,fast_delivery_cost_imputed,996,4111,0.731466,0.00000,13.073098,3.709089e-36,2347381.0,6.671783e-137,0.146586,1.700888e-15,62,0.733136,8.720070,2.563191e-12,0.0,3.442106e-10,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,4.636361e-36,1.334357e-136,1.889876e-15,3.203989e-12,4.302633e-10
6,num_valutazioni,996,4111,-1162.081926,-167.00000,-25.641156,1.888902e-136,1320508.5,7.023828e-68,0.338797,2.875365e-82,62,-1159.735395,-115.161594,4.575796e-73,0.0,7.577765e-12,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,3.148170e-136,8.779785e-68,4.107665e-82,4.575796e-72,1.082538e-11
7,log1p_num_valutazioni,996,4111,-1.353618,-1.17145,-18.704592,3.710816e-71,1320508.5,7.023828e-68,0.338797,2.875365e-82,62,-1.362655,-52.387375,1.998295e-52,0.0,7.577765e-12,Welch/Mann-Whitney/KS are row-level descriptive tests; market-paired tests compare FBA vs non-FBA means within timestamp markets and are the cleaner balance check for this design.,5.301166e-71,8.779785e-68,4.107665e-82,2.854707e-52,1.082538e-11
8,valutazioni_positive,996,4111,12.602830,10.00000,38.255804,1.479154e-280,3208191.5,1.346735e-170,0.439052,4.193427e-140,62,12.567763,112.086566,2.366724e-72,0.0,7.576652e-12

rank_pos                       rank_pct                               
                       mean median min max count      mean    median  min       max count
fba_from_shipper                                                                         
0                 47.989784   49.0   1  93  4111  0.574029  0.586207  0.0  1.000000  4111
1                 16.841365   14.0   1  67   996  0.194446  0.160707  0.0  0.776316   996

,count,mean,std,min,25%,50%,75%,max
third_party_market_size,62.0,82.370968,6.199508,72.0,77.0,80.0,88.75,93.0


,markets,markets_with_fba,markets_with_non_fba,min_fba_share,median_fba_share,max_fba_share
0,62,62,62,0.152941,0.2,0.226667


,fba_from_shipper,rows,positive_shipping_rows,positive_shipping_share,mean_shipping_price,median_shipping_price,max_shipping_price
0,0,4111,1664,0.404768,3.61019,0.0,21.09
1,1,996,0,0.000000,0.00000,0.0,0.00


,variable,count,mean,std,min,10%,25%,50%,75%,90%,99%,max
0,g_cons_min_robust,5107.0,6.885256,3.590603,2.0,3.0,5.0,6.0,8.0,11.0,19.0,59.0
1,g_cons_max_robust,5107.0,10.208146,5.817800,2.0,5.0,6.0,9.0,12.0,18.0,33.0,62.0
2,delivery_window_robust,5107.0,3.322890,3.093166,0.0,0.0,1.0,3.0,5.0,8.0,15.0,17.0
3,fast_delivery_available,5107.0,0.193852,0.395353,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,qta_min,5107.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### 7. Price, shipping, and fast-delivery support diagnostics

The support diagnostic separates raw-extraction support from final-panel support. Raw rows include repeated listings, observations outside the third-party seller set, and rows with missing fast-delivery information. The cell quantifies, by FBA status and across the static and dynamic samples, the share of rows with non-missing price, non-missing repaired shipping price, non-missing reconstructed total price, and non-missing fast-delivery availability. The diagnostic feeds the fast-delivery imputation audit (`fast_delivery_imputation_audit.csv`) used by the logistics-value-adjusted price sensitivity in Section 11.


In [ ]:
# -----------------------------------------------------------------------------
# Section 7. Price, shipping, and fast-delivery support diagnostics
# -----------------------------------------------------------------------------
# Distinguish raw extraction support from final-panel support; quantify non-missing shares of price, shipping, total price, and fast-delivery availability by FBA status across the static and dynamic samples.

def _bool_fba_from_series(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "vero"])

def _numeric_price_from_frame(frame, preferred_cols):
    for col in preferred_cols:
        if col in frame.columns:
            return pd.to_numeric(frame[col], errors="coerce") if pd.api.types.is_numeric_dtype(frame[col]) else parse_numeric_italian(frame[col])
    return pd.Series(np.nan, index=frame.index)

_raw_fba_flag = raw_df["fba_from_shipper"].astype(bool) if "fba_from_shipper" in raw_df.columns else _bool_fba_from_series(raw_df["fba"])
_raw_total_price = _numeric_price_from_frame(raw_df, ["prezzo_totale_reconstructed", "prezzo_totale(€)"])
_raw_product_price = _numeric_price_from_frame(raw_df, ["prezzo", "prezzo_prod_venduto(€)"])
_raw_shipping_price = _numeric_price_from_frame(raw_df, ["prezzo_spedizione_repaired", "prezzo_spedizione(€)"])

_final_fba_flag = df_final["fba_from_shipper"].astype(bool)
_final_total_price = pd.to_numeric(df_final["prezzo_totale_reconstructed"], errors="coerce")
_final_product_price = pd.to_numeric(df_final["prezzo"], errors="coerce")
_final_shipping_price = pd.to_numeric(df_final["prezzo_spedizione_repaired"], errors="coerce")

_price_rows = []
for sample_name, fba_flag, total_price, product_price, shipping_price in [
    ("raw_extraction", _raw_fba_flag, _raw_total_price, _raw_product_price, _raw_shipping_price),
    ("final_seller_market_panel", _final_fba_flag, _final_total_price, _final_product_price, _final_shipping_price),
]:
    for label, mask in [("FBA", fba_flag), ("non_FBA", ~fba_flag)]:
        _price_rows.append({
            "sample": sample_name,
            "group": label,
            "rows": int(mask.sum()),
            "min_total_price": float(total_price.loc[mask].min()) if mask.sum() else np.nan,
            "min_product_price": float(product_price.loc[mask].min()) if mask.sum() else np.nan,
            "rows_total_price_below_35": int((total_price.loc[mask] < 35).sum()),
            "positive_shipping_rows": int((shipping_price.loc[mask] > 0).sum()),
            "positive_shipping_share": float((shipping_price.loc[mask] > 0).mean()) if mask.sum() else np.nan,
        })
price_threshold_support_table = pd.DataFrame(_price_rows)
display(price_threshold_support_table)

def _fast_delivery_segment_has_price(text):
    """Return True when the text after the fast-delivery label contains an explicit monetary price."""
    if pd.isna(text):
        return False
    value = str(text)
    marker = "Consegna più veloce"
    if marker not in value:
        return False
    segment = value.split(marker, 1)[1]
    return bool(re.search(r"(€|\bEUR\b|\beuro\b)", segment, flags=re.IGNORECASE))

if "spedizione_consegna" in df_final.columns:
    df_final["fast_delivery_text_price_flag"] = df_final["spedizione_consegna"].map(_fast_delivery_segment_has_price).astype(int)
else:
    df_final["fast_delivery_text_price_flag"] = 0

fast_delivery_premium_benchmark_table = pd.DataFrame([
    {
        "scenario": "pickup_point_lower_sensitivity_eur_3_99",
        "premium_eur": FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR["pickup_point_lower_sensitivity_eur_3_99"],
        "role": "lower sensitivity",
        "notebook_use": "one-way sensitivity translation only; not the baseline fast-delivery premium",
    },
    {
        "scenario": FAST_DELIVERY_PREMIUM_LABEL,
        "premium_eur": FAST_DELIVERY_PREMIUM_EUR,
        "role": "baseline",
        "notebook_use": "main imputed Premium fast-delivery benchmark for fast_delivery_cost_imputed",
    },
    {
        "scenario": "same_day_upper_sensitivity_eur_8_99",
        "premium_eur": FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR["same_day_upper_sensitivity_eur_8_99"],
        "role": "upper sensitivity",
        "notebook_use": "one-way sensitivity translation only; appropriate only if faster delivery is interpreted as same-day delivery",
    },
])
fast_delivery_premium_benchmark_table["source_role"] = FAST_DELIVERY_PREMIUM_SOURCE
display(fast_delivery_premium_benchmark_table)

fast_delivery_imputation_audit_table = (
    df_final
    .assign(fba_group=lambda d: np.where(d["fba_from_shipper"].eq(1), "FBA", "non_FBA"))
    .groupby("fba_group", observed=True)
    .agg(
        observations=("rank_pct", "size"),
        fast_delivery_available_rows=("fast_delivery_available", "sum"),
        fast_delivery_available_share=("fast_delivery_available", "mean"),
        imputed_fast_delivery_cost_mean=("fast_delivery_cost_imputed", "mean"),
        imputed_fast_delivery_cost_max=("fast_delivery_cost_imputed", "max"),
        fast_delivery_text_price_rows=("fast_delivery_text_price_flag", "sum"),
    )
    .reset_index()
)
fast_delivery_imputation_audit_table["premium_eur"] = FAST_DELIVERY_PREMIUM_EUR
fast_delivery_imputation_audit_table["premium_label"] = FAST_DELIVERY_PREMIUM_LABEL
fast_delivery_imputation_audit_table["source_role"] = FAST_DELIVERY_PREMIUM_SOURCE
fast_delivery_imputation_audit_table["identification_note"] = (
    "The imputed monetary value is assigned only to FBA rows with fast-delivery availability; "
    "non-FBA rows receive zero in this FBA-only logistics-value adjustment."
)
display(fast_delivery_imputation_audit_table)

register_check(
    "6.1",
    "fast_delivery_baseline_premium_is_4_99",
    bool(np.isclose(float(FAST_DELIVERY_PREMIUM_EUR), 4.99)),
    f"premium_eur={FAST_DELIVERY_PREMIUM_EUR}",
)
register_check(
    "6.1",
    "fast_delivery_sensitivity_grid_contains_3_99_4_99_8_99",
    set(np.round(list(FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR.values()), 2)) == {3.99, 4.99, 8.99},
    str(FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR),
)
_expected_fast_delivery_cost_fba_only = (
    FAST_DELIVERY_PREMIUM_EUR
    * pd.to_numeric(df_final["fast_delivery_available"], errors="coerce")
    * pd.to_numeric(df_final["fba_from_shipper"], errors="coerce")
)
_actual_fast_delivery_cost = pd.to_numeric(df_final["fast_delivery_cost_imputed"], errors="coerce")
register_check(
    "6.1",
    "fast_delivery_cost_is_fba_only_indicator_rescale",
    bool(np.allclose(
        _actual_fast_delivery_cost,
        _expected_fast_delivery_cost_fba_only,
        equal_nan=True,
    )),
    f"premium_eur={FAST_DELIVERY_PREMIUM_EUR}; rule=premium*fast_delivery_available*fba_from_shipper",
)
register_check(
    "6.1",
    "non_fba_fast_delivery_cost_is_zero",
    bool(np.isclose(_actual_fast_delivery_cost.loc[df_final["fba_from_shipper"].eq(0)].fillna(0).max(), 0.0)),
    "non-FBA observations must receive zero imputed FBA logistics value",
)


,sample,group,rows,min_total_price,min_product_price,rows_total_price_below_35,positive_shipping_rows,positive_shipping_share
0,raw_extraction,FBA,4707,25.04,25.04,3528,0,0.000000
1,raw_extraction,non_FBA,4717,34.99,29.62,1,1790,0.379478
2,final_seller_market_panel,FBA,996,38.00,38.00,0,0,0.000000
3,final_seller_market_panel,non_FBA,4111,39.00,34.65,0,1664,0.404768


,scenario,premium_eur,role,notebook_use,source_role
0,pickup_point_lower_sensitivity_eur_3_99,3.99,lower sensitivity,one-way sensitivity translation only; not the baseline fast-delivery premium,"External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; baseline EUR 4.99, not directly observed in the scraped CSV"
1,premium_delivery_baseline_eur_4_99,4.99,baseline,main imputed Premium fast-delivery benchmark for fast_delivery_cost_imputed,"External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; baseline EUR 4.99, not directly observed in the scraped CSV"
2,same_day_upper_sensitivity_eur_8_99,8.99,upper sensitivity,one-way sensitivity translation only; appropriate only if faster delivery is interpreted as same-day delivery,"External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; baseline EUR 4.99, not directly observed in the scraped CSV"


,fba_group,observations,fast_delivery_available_rows,fast_delivery_available_share,imputed_fast_delivery_cost_mean,imputed_fast_delivery_cost_max,fast_delivery_text_price_rows,premium_eur,premium_label,source_role,identification_note
0,FBA,996,146,0.146586,0.731466,4.99,0,4.99,premium_delivery_baseline_eur_4_99,"External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; baseline EUR 4.99, not directly observed in the scraped CSV",The imputed monetary value is assigned only to FBA rows with fast-delivery availability; non-FBA rows receive zero in this FBA-only logistics-value adjustment.
1,non_FBA,4111,844,0.205303,0.000000,0.00,0,4.99,premium_delivery_baseline_eur_4_99,"External Amazon.it Premium fast-delivery benchmark supplied for the thesis analysis; baseline EUR 4.99, not directly observed in the scraped CSV",The imputed monetary value is assigned only to FBA rows with fast-delivery availability; non-FBA rows receive zero in this FBA-only logistics-value adjustment.


### 8. Estimation and inference methods

The static regressions are linear projections of `rank_pct` on FBA status and controls with market fixed effects. Market fixed effects compare sellers within the same timestamped seller list and absorb every market-level shock. Inference clusters by seller and market two-way (Cameron, Gelbach, and Miller 2011) because residual dependence operates along both dimensions in the panel. Finite-cluster supplements include the CR2 correction (Bell and McCaffrey 2002; MacKinnon, Nielsen, and Webb 2022) and the restricted-residual wild cluster bootstrap (Cameron and Miller 2015; MacKinnon and Webb 2017).

The cell defines the reusable estimation, inference, robustness, and interaction helpers for `rank_pct`, including the significance-stars formatter, the two-way clustering utility, the CR2 implementation, the wild cluster bootstrap with Rademacher weights, the Mundlak correlated-effects helper, the influence-trimming helper, and the multiple-testing correction wrapper. All routines run on the active kernel and produce the diagnostic tables exported at the end of Part II.


In [ ]:
# -----------------------------------------------------------------------------
# Section 8. Estimation, inference, and robustness helpers
# -----------------------------------------------------------------------------
# Define the reusable helpers used throughout Part II: significance-stars formatter, two-way clustering, CR2, wild cluster bootstrap, Mundlak, influence diagnostics, multiple-testing correction, interaction estimator.

def significance_stars(pvalue):
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "***"
    if pvalue < 0.05:
        return "**"
    if pvalue < 0.10:
        return "*"
    return ""

def model_formula(rhs, outcome=CONTINUOUS_OUTCOME, include_market_fe=True):
    terms = list(rhs)
    if include_market_fe:
        terms = terms + ["C(market_id)"]
    return outcome + " ~ " + " + ".join(terms)

def cluster_codes(series):
    return pd.Categorical(series).codes

def stable_symmetric_pinv(matrix, ridge_floor=1e-10, max_tries=8):
    """Stable inverse for symmetric cross-product matrices.

    The model matrices generated by patsy use reference categories, so the
    cross-products should be full rank after absorbed columns are dropped. To
    avoid occasional SVD stalls in repeated notebook execution, this helper uses
    Cholesky/linear solves with an adaptive ridge only when needed. The ridge is
    scaled to the matrix trace and is purely numerical stabilization; it is not a
    penalized estimator.
    """
    A = np.asarray(matrix, dtype=float)
    A = (A + A.T) / 2.0
    k = A.shape[0]
    if k == 0:
        return A.copy()
    eye = np.eye(k)
    scale = float(np.trace(A) / k) if np.isfinite(np.trace(A)) and k > 0 else 1.0
    scale = max(abs(scale), 1.0)
    # Try an unregularized solve first.
    try:
        return linalg.solve(A, eye, assume_a="sym", check_finite=False)
    except Exception:
        pass
    for j in range(max_tries):
        ridge = ridge_floor * (10 ** j) * scale
        try:
            return linalg.solve(A + ridge * eye, eye, assume_a="pos", check_finite=False)
        except Exception:
            continue
    # Last-resort pseudo-inverse. This branch should almost never be used; it is
    # kept to prevent notebook failure on pathological rank deficiencies.
    return linalg.pinv(A, rtol=1e-10, check_finite=False)

def resolve_param_name(res, base_name="fba_from_shipper"):
    names = list(res.model.exog_names)
    exact_candidates = [base_name, f"{base_name}[T.True]", f"{base_name}[True]", f"{base_name}[T.1]"]
    for candidate in exact_candidates:
        if candidate in names:
            return candidate
    for name in names:
        if name.startswith(base_name + "[") or name.startswith(base_name + "[T."):
            return name
    raise KeyError(f"Parameter {base_name} not found in model terms: {names}")

def covariance_for_inference(res_plain, data, inference):
    seller = cluster_codes(data["seller_id"])
    market = cluster_codes(data["market_id"])
    n_sellers = int(pd.Series(seller).nunique())
    n_markets = int(pd.Series(market).nunique())

    if inference == "seller_cluster":
        return cov_cluster(res_plain, seller), max(n_sellers - 1, 1), {"seller_clusters": n_sellers, "market_clusters": n_markets}
    if inference == "market_cluster":
        return cov_cluster(res_plain, market), max(n_markets - 1, 1), {"seller_clusters": n_sellers, "market_clusters": n_markets}
    if inference == "two_way_seller_market":
        cov_tw, _, _ = cov_cluster_2groups(res_plain, seller, market)
        return cov_tw, max(min(n_sellers, n_markets) - 1, 1), {"seller_clusters": n_sellers, "market_clusters": n_markets}
    raise ValueError(f"Unknown inference label: {inference}")

def extract_term_row_from_cov(res_plain, cov, specification, estimator, sample_label, inference_label, dof, cluster_info=None, base_name="fba_from_shipper"):
    names = list(res_plain.model.exog_names)
    params = pd.Series(np.asarray(res_plain.params), index=names, dtype=float)
    term_name = resolve_param_name(res_plain, base_name=base_name)
    idx = names.index(term_name)
    beta = float(params[term_name])
    variance = float(cov[idx, idx])
    se = float(np.sqrt(max(variance, 0))) if pd.notna(variance) else np.nan
    statistic = beta / se if se and se > 0 else np.nan
    p = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof and dof > 0 else 1.96
    out = {
        "sample": sample_label,
        "outcome": CONTINUOUS_OUTCOME,
        "specification": specification,
        "estimator": estimator,
        "inference": inference_label,
        "term_name": term_name,
        "coef_fba": beta,
        "se_fba": se,
        "test_statistic": statistic,
        "pvalue_fba": p,
        "ci_low": beta - crit * se if pd.notna(se) else np.nan,
        "ci_high": beta + crit * se if pd.notna(se) else np.nan,
        "stars": significance_stars(p),
        "dof_reference": int(dof),
        "nobs": int(res_plain.nobs),
        "r_squared": float(getattr(res_plain, "rsquared", np.nan)),
    }
    if cluster_info:
        out.update(cluster_info)
    return out

def fit_market_fe_ols_plain(data, rhs):
    return smf.ols(model_formula(rhs), data=data).fit()

def fit_ols_with_inference(data, rhs, inference=HEADLINE_INFERENCE, specification="", sample_label="full_sample"):
    res_plain = fit_market_fe_ols_plain(data=data, rhs=rhs)
    cov, dof, cluster_info = covariance_for_inference(res_plain, data, inference)
    row = extract_term_row_from_cov(
        res_plain,
        cov,
        specification=specification,
        estimator="market_fe_ols",
        sample_label=sample_label,
        inference_label=inference,
        dof=dof,
        cluster_info=cluster_info,
    )
    return row, res_plain, cov, dof

def run_market_fe_grid(data, specifications, inference=HEADLINE_INFERENCE, sample_label="full_sample"):
    rows = []
    result_store = {}
    covariance_store = {}
    dof_store = {}
    for spec_name, rhs in specifications.items():
        row, res_plain, cov, dof = fit_ols_with_inference(
            data=data,
            rhs=rhs,
            inference=inference,
            specification=spec_name,
            sample_label=sample_label,
        )
        result_store[spec_name] = res_plain
        covariance_store[spec_name] = cov
        dof_store[spec_name] = dof
        rows.append(row)
    return pd.DataFrame(rows), result_store, covariance_store, dof_store


def fit_ols_with_inference_fast(data, rhs, inference=HEADLINE_INFERENCE, specification="", sample_label="full_sample"):
    formula = model_formula(rhs)
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    ybar = float(np.mean(y))
    r2 = 1 - np.sum(resid ** 2) / np.sum((y - ybar) ** 2)
    design_data = data.loc[x_df.index]
    seller = cluster_codes(design_data["seller_id"])
    market = cluster_codes(design_data["market_id"])
    n_sellers = int(pd.Series(seller).nunique())
    n_markets = int(pd.Series(market).nunique())
    if inference == "seller_cluster":
        cov = one_way_cluster_cov_fast(X, resid, seller, bread=bread)
        dof = max(n_sellers - 1, 1)
    elif inference == "market_cluster":
        cov = one_way_cluster_cov_fast(X, resid, market, bread=bread)
        dof = max(n_markets - 1, 1)
    elif inference == "two_way_seller_market":
        cov = two_way_cluster_cov_fast(X, resid, seller, market, bread=bread)
        dof = max(min(n_sellers, n_markets) - 1, 1)
    else:
        raise ValueError(f"Unknown inference label: {inference}")
    row = row_from_matrix_estimate(
        beta=beta,
        cov=cov,
        names=names,
        specification=specification,
        estimator="market_fe_ols",
        sample_label=sample_label,
        inference_label=inference,
        dof=dof,
        nobs=len(y),
        r_squared=r2,
        cluster_info={"seller_clusters": n_sellers, "market_clusters": n_markets},
    )
    return row

def build_mundlak_means(data, vars_to_average, id_col="seller_id"):
    data = data.copy()
    for var in vars_to_average:
        data[f"{var}_seller_mean"] = data.groupby(id_col)[var].transform("mean")
    return data

def fit_mundlak_rank_model_plain(data, rhs_time_varying):
    mundlak_data = build_mundlak_means(data, rhs_time_varying, id_col="seller_id")
    mean_terms = [f"{v}_seller_mean" for v in rhs_time_varying]
    formula = CONTINUOUS_OUTCOME + " ~ fba_from_shipper + " + " + ".join(rhs_time_varying + mean_terms) + " + C(market_id)"
    res_plain = smf.ols(formula, data=mundlak_data).fit()
    return res_plain, mundlak_data

def translate_rank_effect(beta, data):
    mean_market_size = float(data.groupby("market_id").size().mean())
    median_market_size = float(data.groupby("market_id").size().median())
    return pd.DataFrame(
        [
            {
                "reference_market_size": mean_market_size,
                "implied_rank_shift": beta * (mean_market_size - 1),
                "note": "Mean third-party market size",
            },
            {
                "reference_market_size": median_market_size,
                "implied_rank_shift": beta * (median_market_size - 1),
                "note": "Median third-party market size",
            },
        ]
    )

def build_attenuation_decomposition(main_table, raw_spec="spec_1_fba_only"):
    table = main_table.set_index("specification")
    raw_beta = float(table.loc[raw_spec, "coef_fba"])
    raw_abs = abs(raw_beta)
    rows = []
    comparison_specs = [
        ("spec_2_fba_reputation", "conditioning_on_reputation"),
        ("spec_3_fba_reputation_totalprice_delivery", "conditioning_on_reputation_total_price_delivery"),
        ("spec_4_fba_reputation_price_shipping_delivery", "conditioning_on_reputation_price_shipping_delivery"),
    ]
    for spec, label in comparison_specs:
        beta = float(table.loc[spec, "coef_fba"])
        residual_share = abs(beta) / raw_abs if raw_abs > 0 else np.nan
        absorbed_share = 1 - residual_share if pd.notna(residual_share) else np.nan
        rows.append(
            {
                "comparison": f"{raw_spec} -> {spec}",
                "interpretation_label": label,
                "raw_fba_coef": raw_beta,
                "adjusted_fba_coef": beta,
                "absolute_raw_gap": raw_abs,
                "absolute_adjusted_gap": abs(beta),
                "share_absorbed_by_added_controls": absorbed_share,
                "share_remaining_as_residual_association": residual_share,
                "absorbed_percent": 100 * absorbed_share if pd.notna(absorbed_share) else np.nan,
                "remaining_percent": 100 * residual_share if pd.notna(residual_share) else np.nan,
                "note": "Attenuation accounting only; not a causal mediation design.",
            }
        )
    return pd.DataFrame(rows)

# Matrix helpers used for fast bootstrap, market-weighted OLS, and influence diagnostics.

def term_index_from_names(names, term="fba_from_shipper"):
    if term in names:
        return names.index(term)
    for candidate in [f"{term}[T.True]", f"{term}[True]", f"{term}[T.1]"]:
        if candidate in names:
            return names.index(candidate)
    raise KeyError(f"{term} not found in design columns")

def cluster_index_list(codes):
    codes = np.asarray(codes)
    return [np.where(codes == g)[0] for g in np.unique(codes)]

def one_way_cluster_cov_fast(X, resid, groups, bread=None):
    X = np.asarray(X, dtype=float)
    resid = np.asarray(resid, dtype=float)
    groups = np.asarray(groups)
    n, p = X.shape
    if bread is None:
        bread = stable_symmetric_pinv(X.T @ X)
    unique_groups, inverse = np.unique(groups, return_inverse=True)
    G = len(unique_groups)
    if G == n:
        # When every intersection cluster is a singleton, the meat is the HC0 meat.
        meat = X.T @ (X * (resid ** 2)[:, None])
    else:
        scores = np.zeros((G, p), dtype=float)
        np.add.at(scores, inverse, X * resid[:, None])
        meat = scores.T @ scores
    correction = (G / (G - 1)) * ((n - 1) / (n - p)) if G > 1 and n > p else 1.0
    return correction * (bread @ meat @ bread)

def two_way_cluster_cov_fast(X, resid, seller_groups, market_groups, bread=None):
    if bread is None:
        bread = stable_symmetric_pinv(X.T @ X)
    seller_cov = one_way_cluster_cov_fast(X, resid, seller_groups, bread=bread)
    market_cov = one_way_cluster_cov_fast(X, resid, market_groups, bread=bread)
    intersection = pd.Categorical(pd.Series(seller_groups).astype(str) + "__" + pd.Series(market_groups).astype(str)).codes
    intersection_cov = one_way_cluster_cov_fast(X, resid, intersection, bread=bread)
    return seller_cov + market_cov - intersection_cov

def row_from_matrix_estimate(beta, cov, names, specification, estimator, sample_label, inference_label, dof, nobs, r_squared, cluster_info=None, term="fba_from_shipper"):
    idx = term_index_from_names(names, term)
    beta_term = float(beta[idx])
    variance = float(cov[idx, idx])
    se = float(np.sqrt(max(variance, 0))) if pd.notna(variance) else np.nan
    statistic = beta_term / se if se and se > 0 else np.nan
    p = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof and dof > 0 else 1.96
    row = {
        "sample": sample_label,
        "outcome": CONTINUOUS_OUTCOME,
        "specification": specification,
        "estimator": estimator,
        "inference": inference_label,
        "term_name": names[idx],
        "coef_fba": beta_term,
        "se_fba": se,
        "test_statistic": statistic,
        "pvalue_fba": p,
        "ci_low": beta_term - crit * se if pd.notna(se) else np.nan,
        "ci_high": beta_term + crit * se if pd.notna(se) else np.nan,
        "stars": significance_stars(p),
        "dof_reference": int(dof),
        "nobs": int(nobs),
        "r_squared": float(r_squared),
    }
    if cluster_info:
        row.update(cluster_info)
    return row

def fit_market_equal_weighted_ols(data, rhs):
    formula = model_formula(rhs)
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    market_size = data.loc[x_df.index].groupby("market_id")["market_id"].transform("size")
    weights = 1.0 / np.asarray(market_size, dtype=float)
    sqrt_w = np.sqrt(weights)
    Xw = X * sqrt_w[:, None]
    yw = y * sqrt_w
    bread = stable_symmetric_pinv(Xw.T @ Xw)
    beta = bread @ Xw.T @ yw
    fitted = X @ beta
    resid_w = yw - Xw @ beta
    seller = cluster_codes(data.loc[x_df.index, "seller_id"])
    market = cluster_codes(data.loc[x_df.index, "market_id"])
    cov = two_way_cluster_cov_fast(Xw, resid_w, seller, market, bread=bread)
    ybar_w = np.average(y, weights=weights)
    r2 = 1 - np.sum(weights * (y - fitted) ** 2) / np.sum(weights * (y - ybar_w) ** 2)
    cluster_info = {"seller_clusters": int(pd.Series(seller).nunique()), "market_clusters": int(pd.Series(market).nunique())}
    dof = max(min(cluster_info["seller_clusters"], cluster_info["market_clusters"]) - 1, 1)
    return pd.DataFrame([
        row_from_matrix_estimate(
            beta=beta,
            cov=cov,
            names=names,
            specification="spec_4_market_equal_weighted",
            estimator="market_fe_wls_equal_market_weight",
            sample_label="full_sample",
            inference_label="two_way_seller_market",
            dof=dof,
            nobs=len(y),
            r_squared=r2,
            cluster_info=cluster_info,
        )
    ])

def fast_glm_cluster_covariance(res, X, y, data_for_design, inference):
    mu = np.asarray(res.fittedvalues, dtype=float)
    score_resid = np.asarray(y - mu, dtype=float)
    bread = np.asarray(res.normalized_cov_params, dtype=float)
    seller = cluster_codes(data_for_design["seller_id"])
    market = cluster_codes(data_for_design["market_id"])
    n_sellers = int(pd.Series(seller).nunique())
    n_markets = int(pd.Series(market).nunique())
    if inference == "seller_cluster":
        cov = one_way_cluster_cov_fast(X, score_resid, seller, bread=bread)
        dof = max(n_sellers - 1, 1)
    elif inference == "market_cluster":
        cov = one_way_cluster_cov_fast(X, score_resid, market, bread=bread)
        dof = max(n_markets - 1, 1)
    elif inference == "two_way_seller_market":
        cov = two_way_cluster_cov_fast(X, score_resid, seller, market, bread=bread)
        dof = max(min(n_sellers, n_markets) - 1, 1)
    else:
        raise ValueError(f"Unknown inference label: {inference}")
    return cov, dof, {"seller_clusters": n_sellers, "market_clusters": n_markets}

def fractional_logit_diagnostic(data, rhs, inference=HEADLINE_INFERENCE, specification=HEADLINE_SPEC):
    """Fast Papke-Wooldridge fractional-logit diagnostic via IRLS.

    This avoids slow generic GLM fitting in Colab while estimating the same quasi-likelihood
    score equations for a fractional response with corner values allowed.
    """
    formula = model_formula(rhs)
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    design_data = data.loc[x_df.index].copy()
    n, k = X.shape
    beta = np.zeros(k, dtype=float)
    ridge = 1e-8
    eye_k = np.eye(k)
    for _ in range(50):
        eta = np.clip(X @ beta, -35, 35)
        mu = np.clip(expit(eta), 1e-8, 1 - 1e-8)
        W = np.clip(mu * (1 - mu), 1e-8, None)
        z = eta + (y - mu) / W
        Xw = X * np.sqrt(W)[:, None]
        zw = z * np.sqrt(W)
        lhs = Xw.T @ Xw + ridge * eye_k
        rhs_vec = Xw.T @ zw
        try:
            beta_new = np.linalg.solve(lhs, rhs_vec)
        except np.linalg.LinAlgError:
            beta_new = stable_symmetric_pinv(lhs) @ rhs_vec
        if np.max(np.abs(beta_new - beta)) < 1e-7:
            beta = beta_new
            break
        beta = beta_new
    eta = np.clip(X @ beta, -35, 35)
    mu = np.clip(expit(eta), 1e-8, 1 - 1e-8)
    W = np.clip(mu * (1 - mu), 1e-8, None)
    bread = stable_symmetric_pinv((X * W[:, None]).T @ X + ridge * eye_k)
    score_resid = y - mu
    seller = cluster_codes(design_data["seller_id"])
    market = cluster_codes(design_data["market_id"])
    n_sellers = int(pd.Series(seller).nunique())
    n_markets = int(pd.Series(market).nunique())
    if inference == "seller_cluster":
        cov = one_way_cluster_cov_fast(X, score_resid, seller, bread=bread)
        dof = max(n_sellers - 1, 1)
    elif inference == "market_cluster":
        cov = one_way_cluster_cov_fast(X, score_resid, market, bread=bread)
        dof = max(n_markets - 1, 1)
    elif inference == "two_way_seller_market":
        cov = two_way_cluster_cov_fast(X, score_resid, seller, market, bread=bread)
        dof = max(min(n_sellers, n_markets) - 1, 1)
    else:
        raise ValueError(f"Unknown inference label: {inference}")
    coef_row = row_from_matrix_estimate(
        beta=beta,
        cov=cov,
        names=names,
        specification=specification,
        estimator="papke_wooldridge_fractional_logit_irls",
        sample_label="full_sample",
        inference_label=inference,
        dof=dof,
        nobs=len(y),
        r_squared=np.nan,
        cluster_info={"seller_clusters": n_sellers, "market_clusters": n_markets},
    )
    coef_row["scale_note"] = "Coefficient is on the fractional-logit index scale; estimated by IRLS quasi-likelihood."

    fba_idx = term_index_from_names(names, "fba_from_shipper")
    X_one = X.copy(); X_zero = X.copy()
    X_one[:, fba_idx] = 1.0
    X_zero[:, fba_idx] = 0.0

    def ape_at(p):
        return float(np.mean(expit(np.clip(X_one @ p, -35, 35)) - expit(np.clip(X_zero @ p, -35, 35))))

    ape = ape_at(beta)
    grad = np.zeros_like(beta)
    for j in range(len(beta)):
        step = 1e-5 * max(abs(beta[j]), 1.0)
        p_hi = beta.copy(); p_lo = beta.copy()
        p_hi[j] += step; p_lo[j] -= step
        grad[j] = (ape_at(p_hi) - ape_at(p_lo)) / (2 * step)
    variance = float(grad @ cov @ grad)
    se = float(np.sqrt(max(variance, 0))) if pd.notna(variance) else np.nan
    statistic = ape / se if se and se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof and dof > 0 else 1.96
    ape_row = {
        "sample": "full_sample",
        "outcome": CONTINUOUS_OUTCOME,
        "specification": specification,
        "estimator": "papke_wooldridge_fractional_logit_irls",
        "inference": inference,
        "effect_scale": "average_partial_effect_on_rank_pct",
        "term_name": "fba_from_shipper",
        "estimate": ape,
        "se": se,
        "test_statistic": statistic,
        "pvalue": pvalue,
        "ci_low": ape - crit * se if pd.notna(se) else np.nan,
        "ci_high": ape + crit * se if pd.notna(se) else np.nan,
        "stars": significance_stars(pvalue),
        "dof_reference": int(dof),
        "interpretation_note": "Conservative bounded-outcome diagnostic; not treated as confirmation or replacement of the linear rank-percent projection.",
    }
    return pd.DataFrame([coef_row]), pd.DataFrame([ape_row])

def ols_prediction_diagnostics(res_plain, data):
    predicted = pd.Series(np.asarray(res_plain.fittedvalues), index=data.index)
    outside = predicted.lt(0) | predicted.gt(1)
    return pd.DataFrame([
        {
            "outcome": CONTINUOUS_OUTCOME,
            "actual_min": float(data[CONTINUOUS_OUTCOME].min()),
            "actual_max": float(data[CONTINUOUS_OUTCOME].max()),
            "endpoint_zero_rows": int(data[CONTINUOUS_OUTCOME].eq(0).sum()),
            "endpoint_one_rows": int(data[CONTINUOUS_OUTCOME].eq(1).sum()),
            "fitted_min": float(predicted.min()),
            "fitted_max": float(predicted.max()),
            "fitted_outside_unit_interval_rows": int(outside.sum()),
            "fitted_outside_unit_interval_share": float(outside.mean()),
            "interpretation": "Diagnostic for bounded-outcome concerns; OLS remains the main linear rank-percent estimator.",
        }
    ])



def _target_one_way_variance_from_residuals(a, resid_matrix, groups, nobs, pcols):
    """Target-coefficient one-way cluster variance for many residual vectors.

    a is the projection vector X @ (X'X)^(-1)[:, target]. For any residual vector e,
    the target variance is sum_g (sum_{i in g} a_i e_i)^2 times the usual finite-sample
    correction. The function accepts residuals as an n x B matrix and returns B target
    variances. This computes the same target element of the cluster sandwich covariance
    without forming the full p x p covariance matrix in every bootstrap replication.
    """
    e = np.asarray(resid_matrix, dtype=float)
    if e.ndim == 1:
        e = e[:, None]
    groups = np.asarray(groups)
    unique_groups, inverse = np.unique(groups, return_inverse=True)
    G = len(unique_groups)
    ae = np.asarray(a, dtype=float)[:, None] * e
    if G == len(groups):
        meat_target = np.sum(ae ** 2, axis=0)
    else:
        scores = np.zeros((G, e.shape[1]), dtype=float)
        np.add.at(scores, inverse, ae)
        meat_target = np.sum(scores ** 2, axis=0)
    correction = (G / (G - 1)) * ((nobs - 1) / (nobs - pcols)) if G > 1 and nobs > pcols else 1.0
    return correction * meat_target


def _target_two_way_variance_components_from_residuals(a, resid_matrix, seller_groups, market_groups, nobs, pcols):
    """Target-parameter CGM variance components for vectorized bootstrap residuals.

    The two-way Cameron-Gelbach-Miller target variance is the inclusion-exclusion
    combination V_seller + V_market - V_intersection. With finite-sample corrections
    applied separately to the three components, the scalar target variance can be
    slightly negative in some bootstrap draws. Returning the components allows the
    studentized bootstrap to apply a documented non-negative safeguard instead of
    discarding those draws.
    """
    seller_var = _target_one_way_variance_from_residuals(a, resid_matrix, seller_groups, nobs, pcols)
    market_var = _target_one_way_variance_from_residuals(a, resid_matrix, market_groups, nobs, pcols)
    intersection = pd.Categorical(pd.Series(seller_groups).astype(str) + "__" + pd.Series(market_groups).astype(str)).codes
    intersection_var = _target_one_way_variance_from_residuals(a, resid_matrix, intersection, nobs, pcols)
    two_way_var = seller_var + market_var - intersection_var
    return two_way_var, seller_var, market_var, intersection_var


def _target_two_way_variance_from_residuals(a, resid_matrix, seller_groups, market_groups, nobs, pcols):
    two_way_var, _, _, _ = _target_two_way_variance_components_from_residuals(
        a, resid_matrix, seller_groups, market_groups, nobs, pcols
    )
    return two_way_var


def _apply_two_way_variance_floor(two_way_var, seller_var, market_var):
    """Non-negative safeguard for finite-sample corrected two-way target variances.

    In multiway clustered inference, finite-sample corrections can make the scalar
    inclusion-exclusion variance negative in a small subset of bootstrap draws. For
    those draws, the notebook uses the larger one-way target variance as a conservative
    non-negative fallback. The share of affected replications is exported as metadata.
    """
    two_way_var = np.asarray(two_way_var, dtype=float)
    seller_var = np.asarray(seller_var, dtype=float)
    market_var = np.asarray(market_var, dtype=float)
    fallback = np.maximum(seller_var, market_var)
    floor_mask = np.isfinite(two_way_var) & (two_way_var <= 0) & np.isfinite(fallback) & (fallback > 0)
    adjusted = np.where(floor_mask, fallback, two_way_var)
    return adjusted, floor_mask


def _target_group_design_for_fast_bootstrap(a, X, groups):
    """Precompute group-level sums of a_i x_i for target-variance bootstrap scoring."""
    unique_groups, inverse = np.unique(groups, return_inverse=True)
    group_ax = np.zeros((len(unique_groups), X.shape[1]), dtype=float)
    np.add.at(group_ax, inverse, np.asarray(a, dtype=float)[:, None] * X)
    return unique_groups, inverse, group_ax


def _target_one_way_variance_from_y_beta_fast(a, y_matrix, beta_matrix, inverse, group_ax, n_groups, nobs, pcols):
    """Target one-way variance using group scores without forming fitted values.

    For each bootstrap outcome y*, the target score in group g is
    sum_g a_i (y_i* - x_i' beta*) = sum_g a_i y_i* - (sum_g a_i x_i') beta*.
    This is algebraically equivalent to computing residuals first, but avoids the
    n-by-B fitted-value matrix in the static two-way bootstrap.
    """
    a = np.asarray(a, dtype=float)
    y_matrix = np.asarray(y_matrix, dtype=float)
    beta_matrix = np.asarray(beta_matrix, dtype=float)
    if n_groups == len(inverse):
        scores = a[:, None] * y_matrix - group_ax @ beta_matrix
    else:
        grouped_y = np.zeros((n_groups, y_matrix.shape[1]), dtype=float)
        np.add.at(grouped_y, inverse, a[:, None] * y_matrix)
        scores = grouped_y - group_ax @ beta_matrix
    meat_target = np.sum(scores ** 2, axis=0)
    correction = (n_groups / (n_groups - 1)) * ((nobs - 1) / (nobs - pcols)) if n_groups > 1 and nobs > pcols else 1.0
    return correction * meat_target


if NUMBA_AVAILABLE:
    @njit(parallel=True, fastmath=False)
    def _static_two_way_wcb_numba(X, bread, residual_null, target_projection, seller, market,
                                  seller_weights, market_weights, fba_idx, t_observed,
                                  seller_correction, market_correction, intersection_correction):
        nobs, pcols = X.shape
        B = seller_weights.shape[1]
        n_sellers = seller_weights.shape[0]
        n_markets = market_weights.shape[0]
        valid = np.zeros(B, np.int64)
        extreme = np.zeros(B, np.int64)
        negative_variance = np.zeros(B, np.int64)
        floor_applied = np.zeros(B, np.int64)
        for b in prange(B):
            xtz = np.zeros(pcols, np.float64)
            for i in range(nobs):
                w = seller_weights[seller[i], b] * market_weights[market[i], b]
                z = residual_null[i] * w
                for j in range(pcols):
                    xtz[j] += X[i, j] * z
            beta = np.zeros(pcols, np.float64)
            for j in range(pcols):
                acc = 0.0
                for l in range(pcols):
                    acc += bread[j, l] * xtz[l]
                beta[j] = acc
            seller_scores = np.zeros(n_sellers, np.float64)
            market_scores = np.zeros(n_markets, np.float64)
            intersection_meat = 0.0
            for i in range(nobs):
                w = seller_weights[seller[i], b] * market_weights[market[i], b]
                fitted_component = 0.0
                for j in range(pcols):
                    fitted_component += X[i, j] * beta[j]
                residual_component = residual_null[i] * w - fitted_component
                target_score = target_projection[i] * residual_component
                seller_scores[seller[i]] += target_score
                market_scores[market[i]] += target_score
                intersection_meat += target_score * target_score
            seller_meat = 0.0
            for g in range(n_sellers):
                seller_meat += seller_scores[g] * seller_scores[g]
            market_meat = 0.0
            for g in range(n_markets):
                market_meat += market_scores[g] * market_scores[g]
            seller_var = seller_correction * seller_meat
            market_var = market_correction * market_meat
            intersection_var = intersection_correction * intersection_meat
            variance = seller_var + market_var - intersection_var
            if variance <= 0.0:
                negative_variance[b] = 1
                fallback = seller_var if seller_var >= market_var else market_var
                if fallback > 0.0:
                    variance = fallback
                    floor_applied[b] = 1
            if variance > 0.0:
                t_star = beta[fba_idx] / np.sqrt(variance)
                valid[b] = 1
                if abs(t_star) >= abs(t_observed):
                    extreme[b] = 1
        return valid, extreme, negative_variance, floor_applied
else:
    _static_two_way_wcb_numba = None


def _bootstrap_quality_fields(valid_count, invalid_count, requested_count, pvalue):
    """Return transparent bootstrap quality metadata for exported inference tables."""
    requested_count = int(requested_count)
    valid_count = int(valid_count)
    invalid_count = int(invalid_count)
    valid_share = valid_count / requested_count if requested_count else np.nan
    invalid_share = invalid_count / requested_count if requested_count else np.nan
    mc_se = float(np.sqrt(pvalue * (1.0 - pvalue) / valid_count)) if (valid_count and pd.notna(pvalue)) else np.nan
    warning = bool(pd.notna(valid_share) and valid_share < WILD_BOOTSTRAP_MIN_VALID_SHARE)
    return {
        "valid_replication_share": float(valid_share) if pd.notna(valid_share) else np.nan,
        "invalid_replication_share": float(invalid_share) if pd.notna(invalid_share) else np.nan,
        "bootstrap_pvalue_monte_carlo_se": mc_se,
        "valid_replication_warning": warning,
    }


def wild_cluster_bootstrap_fast(data, rhs, cluster_var, replications=WILD_BOOTSTRAP_REPLICATIONS, seed=WILD_BOOTSTRAP_SEED):
    """Fully studentized one-way restricted wild cluster bootstrap.

    The bootstrap imposes the null by fitting the restricted model without the FBA
    term, draws Rademacher weights at the requested cluster level, refits the full
    model on each bootstrap outcome, and recomputes the target cluster-robust
    standard error inside each replication. The implementation vectorizes the target
    coefficient and target variance calculation, but it does not reuse the observed
    standard error as the bootstrap denominator.
    """
    unrestricted_formula = model_formula(rhs)
    restricted_rhs = [term for term in rhs if term != "fba_from_shipper"]
    restricted_formula = model_formula(restricted_rhs)

    y_unres, x_unres = dmatrices(unrestricted_formula, data=data, return_type="dataframe")
    _, x_res = dmatrices(restricted_formula, data=data, return_type="dataframe")
    y = np.asarray(y_unres).ravel()
    X = np.asarray(x_unres, dtype=float)
    X0 = np.asarray(x_res, dtype=float)
    names = list(x_unres.columns)
    design_data = data.loc[x_unres.index].copy()
    fba_idx = term_index_from_names(names, "fba_from_shipper")

    groups = cluster_codes(design_data[cluster_var])
    unique_groups, group_inv = np.unique(groups, return_inverse=True)
    n_clusters = int(len(unique_groups))
    nobs, pcols = X.shape

    bread_full = stable_symmetric_pinv(X.T @ X)
    beta_hat = bread_full @ X.T @ y
    resid_full = y - X @ beta_hat
    cov_obs = one_way_cluster_cov_fast(X, resid_full, groups, bread=bread_full)
    se_obs = float(np.sqrt(max(cov_obs[fba_idx, fba_idx], 0.0)))
    t_obs = float(beta_hat[fba_idx] / se_obs) if se_obs > 0 else np.nan

    bread_null = stable_symmetric_pinv(X0.T @ X0)
    beta_null = bread_null @ X0.T @ y
    fitted_null = X0 @ beta_null
    residual_null = y - fitted_null

    rng = np.random.default_rng(seed)
    B = int(replications)
    batch_size = B
    target_projection = X @ bread_full[:, fba_idx]
    seller_target_unique, seller_target_inv, seller_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, seller)
    market_target_unique, market_target_inv, market_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, market)
    intersection = pd.Categorical(pd.Series(seller).astype(str) + "__" + pd.Series(market).astype(str)).codes
    intersection_target_unique, intersection_target_inv, intersection_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, intersection)
    valid_count = 0
    extreme_count = 0
    invalid_count = 0
    negative_variance_count = 0
    floor_applied_count = 0
    for start_b in range(0, B, batch_size):
        b = min(batch_size, B - start_b)
        group_weights = rng.choice([-1.0, 1.0], size=(n_clusters, b))
        weights = group_weights[group_inv, :]
        y_star = fitted_null[:, None] + residual_null[:, None] * weights
        beta_star = target_projection @ y_star
        fitted_star = X @ (bread_full @ (X.T @ y_star))
        resid_star = y_star - fitted_star
        var_star = _target_one_way_variance_from_residuals(target_projection, resid_star, groups, nobs, pcols)
        se_star = np.sqrt(np.maximum(var_star, 0.0))
        valid = np.isfinite(se_star) & (se_star > 0) & np.isfinite(beta_star)
        t_star = np.empty_like(beta_star, dtype=float)
        t_star[:] = np.nan
        t_star[valid] = beta_star[valid] / se_star[valid]
        valid_t = t_star[np.isfinite(t_star)]
        negative_variance_count += int(np.sum(np.isfinite(raw_var_star) & (raw_var_star <= 0)))
        floor_applied_count += int(np.sum(floor_mask))
        valid_count += int(len(valid_t))
        invalid_count += int(b - len(valid_t))
        extreme_count += int(np.sum(np.abs(valid_t) >= abs(t_obs)))
    pvalue = float((1 + extreme_count) / (valid_count + 1)) if valid_count else np.nan
    return pd.DataFrame([
        {
            "sample": "full_sample",
            "outcome": CONTINUOUS_OUTCOME,
            "specification": HEADLINE_SPEC,
            "estimator": "market_fe_ols",
            "inference": f"wild_cluster_bootstrap_{cluster_var}_studentized",
            "cluster_var": cluster_var,
            "clusters": n_clusters,
            "requested_replications": B,
            "valid_replications": int(valid_count),
            "invalid_replications": int(invalid_count),
            **_bootstrap_quality_fields(valid_count, invalid_count, B, pvalue),
            "seed": int(seed),
            "coef_fba": float(beta_hat[fba_idx]),
            "cluster_se_fba": se_obs,
            "t_observed": t_obs,
            "bootstrap_pvalue_fba": pvalue,
            "stars": significance_stars(pvalue),
            "null": "fba_from_shipper coefficient equals zero",
            "studentized": True,
            "recomputed_se_each_replication": True,
            "bootstrap_design": "restricted residual wild cluster bootstrap with Rademacher weights; full-model coefficient and target one-way cluster SE recomputed in every bootstrap replication",
        }
    ])


def wild_cluster_bootstrap_twoway_fast(data, rhs, replications=WILD_BOOTSTRAP_REPLICATIONS, seed=WILD_BOOTSTRAP_SEED):
    """Fully studentized two-way restricted wild cluster bootstrap.

    The bootstrap imposes the null by fitting the restricted model without the FBA
    term, draws product Rademacher weights over seller and market clusters, refits
    the full model on every bootstrap outcome, and recomputes the target two-way
    cluster-robust standard error in every replication. When the finite-sample
    corrected two-way inclusion-exclusion variance is non-positive, the draw is
    retained using the larger one-way target variance as a documented non-negative
    fallback. This prevents the bootstrap distribution from being conditioned on
    successful variance draws while preserving a conservative studentized statistic.
    """
    unrestricted_formula = model_formula(rhs)
    restricted_rhs = [term for term in rhs if term != "fba_from_shipper"]
    restricted_formula = model_formula(restricted_rhs)

    y_unres, x_unres = dmatrices(unrestricted_formula, data=data, return_type="dataframe")
    _, x_res = dmatrices(restricted_formula, data=data, return_type="dataframe")
    y = np.asarray(y_unres).ravel()
    X = np.asarray(x_unres, dtype=float)
    X0 = np.asarray(x_res, dtype=float)
    names = list(x_unres.columns)
    design_data = data.loc[x_unres.index].copy()
    fba_idx = term_index_from_names(names, "fba_from_shipper")

    seller = cluster_codes(design_data["seller_id"]).astype(np.int64)
    market = cluster_codes(design_data["market_id"]).astype(np.int64)
    seller_unique, seller_inv = np.unique(seller, return_inverse=True)
    market_unique, market_inv = np.unique(market, return_inverse=True)
    nobs, pcols = X.shape

    bread_full = stable_symmetric_pinv(X.T @ X)
    beta_hat = bread_full @ X.T @ y
    resid_full = y - X @ beta_hat
    cov_obs = two_way_cluster_cov_fast(X, resid_full, seller, market, bread=bread_full)
    se_obs = float(np.sqrt(max(cov_obs[fba_idx, fba_idx], 0.0)))
    t_obs = float(beta_hat[fba_idx] / se_obs) if se_obs > 0 else np.nan

    bread_null = stable_symmetric_pinv(X0.T @ X0)
    beta_null = bread_null @ X0.T @ y
    fitted_null = X0 @ beta_null
    residual_null = y - fitted_null

    rng = np.random.default_rng(seed + 303)
    B = int(replications)
    target_projection = X @ bread_full[:, fba_idx]
    intersection = pd.Categorical(pd.Series(seller).astype(str) + "__" + pd.Series(market).astype(str)).codes.astype(np.int64)
    n_intersections = int(pd.Series(intersection).nunique())
    seller_correction = (len(seller_unique) / (len(seller_unique) - 1)) * ((nobs - 1) / (nobs - pcols)) if len(seller_unique) > 1 and nobs > pcols else 1.0
    market_correction = (len(market_unique) / (len(market_unique) - 1)) * ((nobs - 1) / (nobs - pcols)) if len(market_unique) > 1 and nobs > pcols else 1.0
    intersection_correction = (n_intersections / (n_intersections - 1)) * ((nobs - 1) / (nobs - pcols)) if n_intersections > 1 and nobs > pcols else 1.0

    # Generate weights in the same 500-draw batches used in earlier notebook runs.
    seller_weights_all = np.empty((len(seller_unique), B), dtype=float)
    market_weights_all = np.empty((len(market_unique), B), dtype=float)
    cursor = 0
    while cursor < B:
        batch = min(500, B - cursor)
        seller_weights_all[:, cursor:cursor + batch] = rng.choice([-1.0, 1.0], size=(len(seller_unique), batch))
        market_weights_all[:, cursor:cursor + batch] = rng.choice([-1.0, 1.0], size=(len(market_unique), batch))
        cursor += batch

    if NUMBA_AVAILABLE and _static_two_way_wcb_numba is not None:
        valid_arr, extreme_arr, negative_arr, floor_arr = _static_two_way_wcb_numba(
            np.asarray(X, dtype=np.float64),
            np.asarray(bread_full, dtype=np.float64),
            np.asarray(residual_null, dtype=np.float64),
            np.asarray(target_projection, dtype=np.float64),
            np.asarray(seller, dtype=np.int64),
            np.asarray(market, dtype=np.int64),
            np.asarray(seller_weights_all, dtype=np.float64),
            np.asarray(market_weights_all, dtype=np.float64),
            int(fba_idx),
            float(t_obs),
            float(seller_correction),
            float(market_correction),
            float(intersection_correction),
        )
        valid_count = int(valid_arr.sum())
        extreme_count = int(extreme_arr.sum())
        invalid_count = int(B - valid_count)
        negative_variance_count = int(negative_arr.sum())
        floor_applied_count = int(floor_arr.sum())
        implementation = "numba_parallel_static_two_way_wcb"
    else:
        # Portable fallback. It is algebraically identical but slower, and is kept so
        # the notebook remains executable in environments without numba.
        seller_target_unique, seller_target_inv, seller_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, seller)
        market_target_unique, market_target_inv, market_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, market)
        intersection_target_unique, intersection_target_inv, intersection_group_ax = _target_group_design_for_fast_bootstrap(target_projection, X, intersection)
        valid_count = 0
        extreme_count = 0
        invalid_count = 0
        negative_variance_count = 0
        floor_applied_count = 0
        batch_size = 500
        for start_b in range(0, B, batch_size):
            batch = min(batch_size, B - start_b)
            seller_weights = seller_weights_all[:, start_b:start_b + batch]
            market_weights = market_weights_all[:, start_b:start_b + batch]
            weights = seller_weights[seller_inv, :] * market_weights[market_inv, :]
            y_star = fitted_null[:, None] + residual_null[:, None] * weights
            beta_all_star = bread_full @ (X.T @ y_star)
            beta_star = beta_all_star[fba_idx, :]
            seller_var_star = _target_one_way_variance_from_y_beta_fast(
                target_projection, y_star, beta_all_star, seller_target_inv, seller_group_ax, len(seller_target_unique), nobs, pcols
            )
            market_var_star = _target_one_way_variance_from_y_beta_fast(
                target_projection, y_star, beta_all_star, market_target_inv, market_group_ax, len(market_target_unique), nobs, pcols
            )
            intersection_var_star = _target_one_way_variance_from_y_beta_fast(
                target_projection, y_star, beta_all_star, intersection_target_inv, intersection_group_ax, len(intersection_target_unique), nobs, pcols
            )
            raw_var_star = seller_var_star + market_var_star - intersection_var_star
            var_star, floor_mask = _apply_two_way_variance_floor(raw_var_star, seller_var_star, market_var_star)
            se_star = np.sqrt(np.maximum(var_star, 0.0))
            valid = np.isfinite(se_star) & (se_star > 0) & np.isfinite(beta_star)
            t_star = np.empty_like(beta_star, dtype=float)
            t_star[:] = np.nan
            t_star[valid] = beta_star[valid] / se_star[valid]
            valid_t = t_star[np.isfinite(t_star)]
            negative_variance_count += int(np.sum(np.isfinite(raw_var_star) & (raw_var_star <= 0)))
            floor_applied_count += int(np.sum(floor_mask))
            valid_count += int(len(valid_t))
            invalid_count += int(batch - len(valid_t))
            extreme_count += int(np.sum(np.abs(valid_t) >= abs(t_obs)))
        implementation = "portable_numpy_static_two_way_wcb"

    pvalue = float((1 + extreme_count) / (valid_count + 1)) if valid_count else np.nan
    return pd.DataFrame([
        {
            "sample": "full_sample",
            "outcome": CONTINUOUS_OUTCOME,
            "specification": HEADLINE_SPEC,
            "estimator": "market_fe_ols",
            "inference": "wild_cluster_bootstrap_two_way_seller_market_studentized",
            "seller_clusters": int(len(seller_unique)),
            "market_clusters": int(len(market_unique)),
            "requested_replications": B,
            "valid_replications": int(valid_count),
            "invalid_replications": int(invalid_count),
            "negative_two_way_variance_replications": int(negative_variance_count),
            "negative_two_way_variance_share": float(negative_variance_count / B) if B else np.nan,
            "variance_floor_applied_replications": int(floor_applied_count),
            "variance_floor_applied_share": float(floor_applied_count / B) if B else np.nan,
            "variance_floor_rule": "when the finite-sample corrected two-way target variance is non-positive, use max(seller one-way target variance, market one-way target variance)",
            "bootstrap_implementation": implementation,
            "bootstrap_extreme_replications": int(extreme_count),
            **_bootstrap_quality_fields(valid_count, invalid_count, B, pvalue),
            "seed": int(seed + 303),
            "coef_fba": float(beta_hat[fba_idx]),
            "two_way_cluster_se_fba": se_obs,
            "t_observed": t_obs,
            "bootstrap_pvalue_fba": pvalue,
            "stars": significance_stars(pvalue),
            "null": "fba_from_shipper coefficient equals zero",
            "studentized": True,
            "recomputed_se_each_replication": True,
            "bootstrap_design": "restricted residual wild cluster bootstrap with product Rademacher weights over seller and market clusters; full-model coefficient and target two-way cluster SE recomputed in every bootstrap replication; non-positive finite-sample corrected two-way target variances use the documented one-way fallback floor",
        }
    ])

def _market_within_beta(data, rhs, outcome=CONTINUOUS_OUTCOME, market_col="market_id"):
    """Estimate the market fixed-effect coefficient vector by within-market demeaning.

    The headline influence diagnostic uses only numeric controls. Recomputing the
    within transformation after each cluster deletion avoids the numerical file
    that arises when a deleted fixed-effect category leaves a zero or reference-only
    dummy in a precomputed design matrix.
    """
    rhs_terms = list(dict.fromkeys(rhs))
    unsupported = [term for term in rhs_terms if term.startswith("C(") or ":" in term or "I(" in term]
    if unsupported:
        raise ValueError(f"leave-one within diagnostic supports only linear RHS terms; unsupported={unsupported}")

    work_cols = [outcome, market_col] + rhs_terms
    work = data[work_cols].copy()
    for term in [outcome] + rhs_terms:
        work[term] = pd.to_numeric(work[term], errors="coerce")
    work = work.dropna(subset=[outcome, market_col] + rhs_terms)

    y = work[outcome].astype(float)
    X = work[rhs_terms].astype(float)
    y_within = y - y.groupby(work[market_col]).transform("mean")
    X_within = X - X.groupby(work[market_col]).transform("mean")

    nonzero_cols = X_within.var(axis=0).gt(1e-18)
    kept_terms = [term for term, keep in zip(rhs_terms, nonzero_cols) if keep]
    if not kept_terms:
        return {}, 0

    Xv = X_within.loc[:, kept_terms].to_numpy(dtype=float)
    yv = y_within.to_numpy(dtype=float)
    beta = stable_symmetric_pinv(Xv.T @ Xv) @ (Xv.T @ yv)
    return dict(zip(kept_terms, beta)), int(len(work))


def leave_one_cluster_influence_detail(data, rhs, target="fba_from_shipper"):
    """Exact leave-one-market and leave-one-seller detail table.

    The model matrix is effectively rebuilt after each deletion by recomputing the
    market-within transformation. This is equivalent to refitting the headline OLS
    with market fixed effects on each retained sample for the linear headline RHS.
    """
    baseline_beta, baseline_n = _market_within_beta(data, rhs)
    baseline_coef = float(baseline_beta[target])
    rows = []
    for dimension, col in [("market", "market_id"), ("seller", "seller_id")]:
        for value in pd.Series(data[col]).drop_duplicates():
            keep = data[col].ne(value)
            retained = data.loc[keep].copy()
            dropped = data.loc[~keep].copy()
            beta_i, n_i = _market_within_beta(retained, rhs)
            coef_i = float(beta_i[target]) if target in beta_i else np.nan
            rows.append({
                "leave_one_dimension": dimension,
                "dropped_cluster": value,
                "dropped_observations": int(len(dropped)),
                "dropped_fba_observations": int(dropped[target].sum()) if target in dropped.columns else np.nan,
                "dropped_nonfba_observations": int((1 - dropped[target]).sum()) if target in dropped.columns else np.nan,
                "retained_observations": int(n_i),
                "baseline_coef": baseline_coef,
                "deleted_cluster_coef": coef_i,
                "absolute_change": float(abs(coef_i - baseline_coef)) if np.isfinite(coef_i) else np.nan,
                "sign_flips_relative_to_baseline": bool(np.sign(coef_i) != np.sign(baseline_coef)) if np.isfinite(coef_i) else False,
                "method": "exact deleted-cluster OLS using within-market transformation recomputed after deletion",
            })
    out = pd.DataFrame(rows)
    return out.sort_values(["leave_one_dimension", "absolute_change"], ascending=[True, False]).reset_index(drop=True)


def leave_one_cluster_influence(data, rhs):
    """Summarize exact leave-one-market and leave-one-seller influence diagnostics."""
    detail = leave_one_cluster_influence_detail(data, rhs)
    baseline_coef = float(detail["baseline_coef"].dropna().iloc[0])
    rows = []
    for dimension, group in detail.groupby("leave_one_dimension", sort=True):
        coef_series = pd.to_numeric(group["deleted_cluster_coef"], errors="coerce").dropna()
        rows.append({
            "leave_one_dimension": dimension,
            "iterations": int(len(coef_series)),
            "baseline_coef": baseline_coef,
            "min_coef": float(coef_series.min()),
            "p10_coef": float(coef_series.quantile(0.10)),
            "median_coef": float(coef_series.median()),
            "p90_coef": float(coef_series.quantile(0.90)),
            "max_coef": float(coef_series.max()),
            "same_sign_share": float((np.sign(coef_series) == np.sign(baseline_coef)).mean()),
            "max_absolute_change": float(pd.to_numeric(group["absolute_change"], errors="coerce").max()),
            "method": "exact deleted-cluster OLS using within-market transformation recomputed after deletion",
        })
    return pd.DataFrame(rows)

def omitted_variable_sensitivity_from_row(row):
    t_value = abs(float(row["test_statistic"]))
    dof = float(row["dof_reference"])
    partial_r2 = t_value ** 2 / (t_value ** 2 + dof)
    f2 = t_value ** 2 / dof
    robustness_value_to_zero = (math.sqrt(f2 ** 2 + 4 * f2) - f2) / 2
    return pd.DataFrame([
        {
            "method": "Cinelli-Hazlett-style OLS sensitivity diagnostic",
            "coefficient": float(row["coef_fba"]),
            "t_statistic": float(row["test_statistic"]),
            "dof_reference": int(row["dof_reference"]),
            "partial_r2_fba_with_rank_given_controls": partial_r2,
            "robustness_value_to_reduce_estimate_to_zero": robustness_value_to_zero,
            "interpretation": "Approximate diagnostic only; it does not convert the associational design into a causal design.",
        }
    ])

def linear_combination_row(res_plain, cov, weights, dof, label, estimator, inference, sample="full_sample", specification=""):
    names = list(res_plain.model.exog_names)
    params = pd.Series(np.asarray(res_plain.params), index=names, dtype=float)
    w = np.zeros(len(names), dtype=float)
    resolved_weights = {}
    for term, value in weights.items():
        resolved = resolve_param_name(res_plain, term)
        w[names.index(resolved)] = value
        resolved_weights[resolved] = value
    estimate = float(np.dot(w, np.asarray(params)))
    variance = float(w @ cov @ w)
    se = float(np.sqrt(max(variance, 0))) if pd.notna(variance) else np.nan
    statistic = estimate / se if se and se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof and dof > 0 else 1.96
    return {
        "sample": sample,
        "outcome": CONTINUOUS_OUTCOME,
        "specification": specification,
        "estimator": estimator,
        "inference": inference,
        "contrast_label": label,
        "estimate": estimate,
        "se": se,
        "test_statistic": statistic,
        "pvalue": pvalue,
        "ci_low": estimate - crit * se if pd.notna(se) else np.nan,
        "ci_high": estimate + crit * se if pd.notna(se) else np.nan,
        "stars": significance_stars(pvalue),
        "dof_reference": int(dof),
        "resolved_weights": resolved_weights,
    }

def interaction_margins_for_moderator(res_plain, cov, dof, data, raw_var, centered_var, specification, inference=HEADLINE_INFERENCE):
    quantiles = [("p25", 0.25), ("median", 0.50), ("p75", 0.75)]
    rows = []
    mean_value = float(data[raw_var].mean())
    for label, q in quantiles:
        raw_value = float(data[raw_var].quantile(q))
        centered_value = raw_value - mean_value
        row = linear_combination_row(
            res_plain=res_plain,
            cov=cov,
            weights={"fba_from_shipper": 1.0, f"fba_from_shipper:{centered_var}": centered_value},
            dof=dof,
            label=f"FBA association at {raw_var} {label}",
            estimator="market_fe_ols_interaction",
            inference=inference,
            sample="full_sample",
            specification=specification,
        )
        row.update({"moderator": raw_var, "quantile": label, "raw_value": raw_value, "centered_value": centered_value})
        rows.append(row)
    return rows


# Additional audit-proof diagnostics and robustness helpers.

def matrix_design_for_formula(data, formula):
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    design_data = data.loc[x_df.index].copy()
    return y, X, names, design_data

def fit_custom_formula_with_inference(data, formula, specification, estimator="market_fe_ols", inference=HEADLINE_INFERENCE, sample_label="full_sample"):
    y, X, names, design_data = matrix_design_for_formula(data, formula)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    seller = cluster_codes(design_data["seller_id"])
    market = cluster_codes(design_data["market_id"])
    n_sellers = int(pd.Series(seller).nunique())
    n_markets = int(pd.Series(market).nunique())
    if inference == "seller_cluster":
        cov = one_way_cluster_cov_fast(X, resid, seller, bread=bread)
        dof = max(n_sellers - 1, 1)
    elif inference == "market_cluster":
        cov = one_way_cluster_cov_fast(X, resid, market, bread=bread)
        dof = max(n_markets - 1, 1)
    elif inference == "two_way_seller_market":
        cov = two_way_cluster_cov_fast(X, resid, seller, market, bread=bread)
        dof = max(min(n_sellers, n_markets) - 1, 1)
    else:
        raise ValueError(f"Unknown inference label: {inference}")
    ybar = float(np.mean(y))
    r2 = 1 - np.sum(resid ** 2) / np.sum((y - ybar) ** 2)
    row = row_from_matrix_estimate(
        beta=beta,
        cov=cov,
        names=names,
        specification=specification,
        estimator=estimator,
        sample_label=sample_label,
        inference_label=inference,
        dof=dof,
        nobs=len(y),
        r_squared=r2,
        cluster_info={"seller_clusters": n_sellers, "market_clusters": n_markets},
    )
    return row


def cr2_one_way_cluster_table(data, rhs, cluster_var="seller_id", specification=HEADLINE_SPEC, sample_label="full_sample"):
    """Bell-McCaffrey-style CR2 covariance for one clustering dimension.

    The implementation adjusts each cluster's residual vector by (I - H_g)^(-1/2).
    The degrees of freedom are a coefficient-specific Satterthwaite approximation.
    This is reported as a conservative one-way small-sample supplement, not as a
    replacement for the two-way cluster-robust headline covariance.
    """
    formula = model_formula(rhs)
    y, X, names, design_data = matrix_design_for_formula(data, formula)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    codes = cluster_codes(design_data[cluster_var])
    fba_idx = term_index_from_names(names, "fba_from_shipper")
    c = np.zeros(X.shape[1]); c[fba_idx] = 1.0
    meat = np.zeros((X.shape[1], X.shape[1]), dtype=float)
    q_values = []
    for idx in cluster_index_list(codes):
        Xg = X[idx, :]
        eg = resid[idx]
        Hg = Xg @ bread @ Xg.T
        Mg = np.eye(len(idx)) - Hg
        Mg = (Mg + Mg.T) / 2
        evals, evecs = np.linalg.eigh(Mg)
        evals = np.clip(evals, 1e-10, None)
        Ag = evecs @ np.diag(1.0 / np.sqrt(evals)) @ evecs.T
        sg = Xg.T @ (Ag @ eg)
        meat += np.outer(sg, sg)
        qg = float(c @ bread @ np.outer(sg, sg) @ bread @ c)
        q_values.append(max(qg, 0.0))
    cov = bread @ meat @ bread
    se = float(np.sqrt(max(cov[fba_idx, fba_idx], 0)))
    q_values = np.asarray(q_values, dtype=float)
    dof = float(2 * (q_values.sum() ** 2) / np.sum(q_values ** 2)) if np.sum(q_values ** 2) > 0 else max(len(q_values) - 1, 1)
    statistic = float(beta[fba_idx] / se) if se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof > 0 else 1.96
    return pd.DataFrame([
        {
            "sample": sample_label,
            "outcome": CONTINUOUS_OUTCOME,
            "specification": specification,
            "estimator": "market_fe_ols",
            "inference": f"cr2_{cluster_var}",
            "term_name": names[fba_idx],
            "coef_fba": float(beta[fba_idx]),
            "se_fba": se,
            "test_statistic": statistic,
            "pvalue_fba": pvalue,
            "ci_low": float(beta[fba_idx] - crit * se),
            "ci_high": float(beta[fba_idx] + crit * se),
            "stars": significance_stars(pvalue),
            "dof_reference": float(dof),
            "nobs": int(len(y)),
            "clusters": int(pd.Series(codes).nunique()),
            "r_squared": float(1 - np.sum(resid ** 2) / np.sum((y - np.mean(y)) ** 2)),
            "note": "One-way CR2-style small-sample correction; reported as a conservative supplement to two-way clustered headline inference.",
        }
    ])


def functional_form_diagnostics(res_plain):
    """Fast Ramsey RESET diagnostics for the high-dimensional market-FE OLS.

    statsmodels.linear_reset can be slow with many fixed-effect columns. This
    implementation computes the same nested-model F-test directly from the
    restricted RSS and augmented design matrix with powers of fitted values.
    """
    rows = []
    y = np.asarray(res_plain.model.endog, dtype=float).ravel()
    X = np.asarray(res_plain.model.exog, dtype=float)
    fitted = np.asarray(res_plain.fittedvalues, dtype=float).ravel()
    restricted_resid = np.asarray(res_plain.resid, dtype=float).ravel()
    rss_restricted = float(np.sum(restricted_resid ** 2))
    nobs = int(len(y))

    for power in [2, 3]:
        try:
            extra_terms = [fitted ** p for p in range(2, power + 1)]
            Z = np.column_stack([X] + extra_terms)
            bread = stable_symmetric_pinv(Z.T @ Z)
            beta_aug = bread @ Z.T @ y
            resid_aug = y - Z @ beta_aug
            rss_aug = float(np.sum(resid_aug ** 2))
            df_num = power - 1
            df_den = max(nobs - Z.shape[1], 1)
            f_stat = ((rss_restricted - rss_aug) / df_num) / (rss_aug / df_den) if rss_aug > 0 else np.nan
            pvalue = float(stats.f.sf(f_stat, df_num, df_den)) if pd.notna(f_stat) else np.nan
            rows.append({
                "diagnostic": f"Ramsey RESET power {power}",
                "status": "estimated",
                "statistic": float(f_stat) if pd.notna(f_stat) else np.nan,
                "pvalue": pvalue,
                "df_num": int(df_num),
                "df_den": int(df_den),
                "interpretation": "Direct nested-model RESET F-test using powers of fitted values; rejection flags possible functional-form misspecification.",
            })
        except Exception as exc:
            rows.append({
                "diagnostic": f"Ramsey RESET power {power}",
                "status": "not_computed",
                "statistic": np.nan,
                "pvalue": np.nan,
                "df_num": np.nan,
                "df_den": np.nan,
                "interpretation": f"not computed: {exc}",
            })
    rows.append({
        "diagnostic": "Harvey-Collier linearity test",
        "status": "not_reported",
        "statistic": np.nan,
        "pvalue": np.nan,
        "df_num": np.nan,
        "df_den": np.nan,
        "interpretation": "Not interpreted because the high-dimensional market fixed-effects design makes the recursive-residual setup ill-conditioned; RESET and nonlinear-control sensitivity are used instead.",
    })
    return pd.DataFrame(rows)

def spline_functional_form_sensitivity(data):
    """Parsimonious nonlinear-control sensitivity for the headline model.

    The function name is kept for audit-compatible exports, but the final
    notebook uses centered quadratic terms rather than a dense spline basis.
    This is faster, easier to explain in a thesis, and still tests whether the
    FBA coefficient depends on a strictly linear treatment of the key continuous
    controls.
    """
    d = data.copy()
    nonlinear_terms = []
    for var in ["prezzo", "log1p_num_valutazioni", "g_cons_min_robust"]:
        centered = f"{var}_centered_for_nonlinearity"
        squared = f"{var}_centered_sq"
        d[centered] = pd.to_numeric(d[var], errors="coerce") - pd.to_numeric(d[var], errors="coerce").mean()
        d[squared] = d[centered] ** 2
        nonlinear_terms.append(squared)
    row = fit_ols_with_inference_fast(
        d,
        SPECIFICATIONS[HEADLINE_SPEC] + nonlinear_terms,
        inference=HEADLINE_INFERENCE,
        specification="quadratic_price_reputation_delivery",
        sample_label="full_sample",
    )
    row["estimator"] = "market_fe_ols_quadratic_controls"
    row["nonlinear_terms_added"] = nonlinear_terms
    return pd.DataFrame([row])


def extended_control_sensitivity(data, specs):
    rows = []
    baseline_coef = None
    for spec_name, rhs in specs.items():
        row = fit_ols_with_inference_fast(
            data,
            rhs,
            inference=HEADLINE_INFERENCE,
            specification=spec_name,
            sample_label="full_sample",
        )
        if spec_name == "headline_baseline":
            baseline_coef = float(row["coef_fba"])
        rows.append(row)
    if baseline_coef is None:
        baseline_coef = float(rows[0]["coef_fba"])
    for row in rows:
        row["coef_change_from_headline"] = float(row["coef_fba"] - baseline_coef)
        row["percent_change_from_headline_abs"] = float(100 * (abs(row["coef_fba"]) - abs(baseline_coef)) / abs(baseline_coef)) if baseline_coef != 0 else np.nan
        row["within_30pct_stability_band"] = bool(abs(abs(row["coef_fba"]) - abs(baseline_coef)) <= 0.30 * abs(baseline_coef)) if baseline_coef != 0 else False
    return pd.DataFrame(rows)

def _fast_influence_frame(res_plain):
    """Compute leverage, studentized residuals, Cook's distance, and FBA DFBETA.

    This avoids statsmodels.OLSInfluence, which is unnecessarily slow for the
    fixed-effect design matrix used here. The formulas are standard OLS influence
    diagnostics based on the hat matrix diagonal and deletion-adjusted residuals.
    """
    X = np.asarray(res_plain.model.exog, dtype=float)
    y = np.asarray(res_plain.model.endog, dtype=float).ravel()
    names = list(res_plain.model.exog_names)
    beta = np.asarray(res_plain.params, dtype=float).ravel()
    resid = y - X @ beta
    n, k = X.shape
    bread = stable_symmetric_pinv(X.T @ X)
    leverage = np.einsum("ij,jk,ik->i", X, bread, X)
    leverage = np.clip(leverage, 0.0, 0.999999)
    rss = float(np.sum(resid ** 2))
    mse = rss / max(n - k, 1)
    denom = np.sqrt(np.maximum(mse * (1.0 - leverage), 1e-20))
    studentized = resid / denom
    cooks = (resid ** 2 / max(k * mse, 1e-20)) * (leverage / np.maximum((1.0 - leverage) ** 2, 1e-20))
    fba_name = resolve_param_name(res_plain, "fba_from_shipper")
    fba_idx = names.index(fba_name)
    x_bread_col = X @ bread[:, fba_idx]
    beta_se = np.sqrt(max(mse * bread[fba_idx, fba_idx], 1e-20))
    dfbeta_fba = (x_bread_col * resid / np.maximum(1.0 - leverage, 1e-20)) / beta_se
    return pd.DataFrame({
        "leverage": leverage,
        "studentized_residual": studentized,
        "cooks_distance": cooks,
        "dfbeta_fba": dfbeta_fba,
    }, index=res_plain.model.data.row_labels if hasattr(res_plain.model.data, "row_labels") else np.arange(n))


def observation_influence_diagnostics(res_plain, data, top_n=20):
    frame = _fast_influence_frame(res_plain)
    names = list(res_plain.model.exog_names)
    n = int(res_plain.nobs)
    k = int(len(names))
    frame["row_index"] = frame.index
    frame["high_leverage_flag"] = frame["leverage"].gt(2 * k / n)
    frame["high_cooks_flag"] = frame["cooks_distance"].gt(4 / n)
    frame["high_studentized_residual_flag"] = frame["studentized_residual"].abs().gt(3)
    summary = pd.DataFrame([
        {
            "nobs": n,
            "parameters": k,
            "leverage_threshold_2k_over_n": 2 * k / n,
            "cooks_threshold_4_over_n": 4 / n,
            "high_leverage_rows": int(frame["high_leverage_flag"].sum()),
            "high_cooks_rows": int(frame["high_cooks_flag"].sum()),
            "high_studentized_residual_rows": int(frame["high_studentized_residual_flag"].sum()),
            "max_abs_dfbeta_fba": float(frame["dfbeta_fba"].abs().max()),
            "diagnostic_method": "fast OLS influence formulas from hat-matrix diagonal",
        }
    ])
    top = (
        frame.assign(abs_dfbeta_fba=frame["dfbeta_fba"].abs())
        .sort_values(["cooks_distance", "abs_dfbeta_fba"], ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    cols_to_add = ["seller_id", "market_id", "rank_pct", "fba_from_shipper", "seller_name"]
    joinable = data[cols_to_add].copy()
    top = top.merge(joinable, left_on="row_index", right_index=True, how="left")
    return summary, top


def influence_trimmed_estimates(data, rhs, res_plain, trim_share=0.01):
    frame = _fast_influence_frame(res_plain)
    cutoff = frame["cooks_distance"].quantile(1 - trim_share)
    keep_index = frame.index[frame["cooks_distance"].le(cutoff)]
    trimmed = data.loc[keep_index].copy()
    row = fit_ols_with_inference_fast(trimmed, rhs, inference=HEADLINE_INFERENCE, specification=f"drop_top_{int(100*trim_share)}pct_cooks_distance", sample_label="influence_trimmed")
    row["rows_dropped"] = int(len(data) - len(trimmed))
    row["cook_cutoff"] = float(cutoff)
    row["diagnostic_method"] = "fast Cook's-distance approximation from hat-matrix diagonal"
    return pd.DataFrame([row])

def propensity_overlap_diagnostics(data):
    ps_formula = (
        "fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + stelle "
        "+ prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_cons_min_robust + C(market_id)"
    )
    ps_res = smf.glm(ps_formula, data=data, family=sm.families.Binomial()).fit(maxiter=100, disp=0)
    pscore = pd.Series(np.asarray(ps_res.fittedvalues), index=data.index, name="propensity_score")
    treated = pscore[data["fba_from_shipper"].eq(1)]
    control = pscore[data["fba_from_shipper"].eq(0)]
    common_low = max(float(treated.min()), float(control.min()))
    common_high = min(float(treated.max()), float(control.max()))
    keep_common = pscore.between(common_low, common_high)
    keep_crump = pscore.between(0.10, 0.90)
    quant = (
        pd.DataFrame({"fba_from_shipper": data["fba_from_shipper"], "propensity_score": pscore})
        .groupby("fba_from_shipper")["propensity_score"]
        .quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        .unstack()
        .reset_index()
    )
    summary = pd.DataFrame([
        {
            "method": "logit propensity score diagnostic with market fixed effects",
            "nobs": int(len(pscore)),
            "treated_rows": int(data["fba_from_shipper"].sum()),
            "control_rows": int((1 - data["fba_from_shipper"]).sum()),
            "treated_ps_min": float(treated.min()),
            "treated_ps_max": float(treated.max()),
            "control_ps_min": float(control.min()),
            "control_ps_max": float(control.max()),
            "common_support_low": common_low,
            "common_support_high": common_high,
            "common_support_rows": int(keep_common.sum()),
            "common_support_share": float(keep_common.mean()),
            "crump_0_10_to_0_90_rows": int(keep_crump.sum()),
            "crump_0_10_to_0_90_share": float(keep_crump.mean()),
            "note": "Diagnostic only; the thesis estimand remains associational, not causal.",
        }
    ])
    return summary, quant, pscore


def overlap_trimmed_estimates(data, rhs, pscore):
    treated = pscore[data["fba_from_shipper"].eq(1)]
    control = pscore[data["fba_from_shipper"].eq(0)]
    common_low = max(float(treated.min()), float(control.min()))
    common_high = min(float(treated.max()), float(control.max()))
    cases = {
        "full_sample": data.copy(),
        "propensity_common_support": data.loc[pscore.between(common_low, common_high)].copy(),
        "propensity_0_05_0_95": data.loc[pscore.between(0.05, 0.95)].copy(),
        "propensity_0_10_0_90": data.loc[pscore.between(0.10, 0.90)].copy(),
    }
    rows = []
    for label, case_df in cases.items():
        row = fit_ols_with_inference_fast(case_df, rhs, inference=HEADLINE_INFERENCE, specification=HEADLINE_SPEC, sample_label=label)
        row["rows"] = int(len(case_df))
        row["markets"] = int(case_df["market_id"].nunique())
        row["fba_rows"] = int(case_df["fba_from_shipper"].sum())
        rows.append(row)
    return pd.DataFrame(rows)



def delivery_tail_sensitivity(data, rhs):
    rows = []
    baseline = fit_ols_with_inference_fast(
        data,
        rhs,
        inference=HEADLINE_INFERENCE,
        specification=HEADLINE_SPEC,
        sample_label="full_sample",
    )
    baseline["restriction"] = "none"
    rows.append(baseline)
    for var in ["g_cons_min_robust", "g_cons_max_robust"]:
        cutoff = data[var].quantile(0.99)
        case = data.loc[data[var].le(cutoff)].copy()
        row = fit_ols_with_inference_fast(
            case,
            rhs,
            inference=HEADLINE_INFERENCE,
            specification=f"drop_top_1pct_{var}",
            sample_label="delivery_tail_sensitivity",
        )
        row["restriction"] = f"drop rows with {var} above p99"
        row["cutoff"] = float(cutoff)
        row["rows_dropped"] = int(len(data) - len(case))
        rows.append(row)
    return pd.DataFrame(rows)


def fit_weighted_market_fe_ols(data, rhs, weights, specification, estimator, sample_label="full_sample"):
    formula = model_formula(rhs)
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    w = pd.Series(weights, index=data.index).loc[x_df.index].astype(float).to_numpy()
    w = np.where(np.isfinite(w) & (w > 0), w, np.nan)
    ok = np.isfinite(w)
    X = X[ok, :]
    y = y[ok]
    w = w[ok]
    design_data = data.loc[x_df.index].iloc[np.where(ok)[0]].copy()
    sqrt_w = np.sqrt(w)
    Xw = X * sqrt_w[:, None]
    yw = y * sqrt_w
    bread = stable_symmetric_pinv(Xw.T @ Xw)
    beta = bread @ Xw.T @ yw
    resid_w = yw - Xw @ beta
    seller = cluster_codes(design_data["seller_id"])
    market = cluster_codes(design_data["market_id"])
    cov = two_way_cluster_cov_fast(Xw, resid_w, seller, market, bread=bread)
    ybar_w = np.average(y, weights=w)
    denom = np.sum(w * (y - ybar_w) ** 2)
    r2 = 1 - np.sum(w * (y - X @ beta) ** 2) / denom if denom > 0 else np.nan
    cluster_info = {"seller_clusters": int(pd.Series(seller).nunique()), "market_clusters": int(pd.Series(market).nunique())}
    dof = max(min(cluster_info["seller_clusters"], cluster_info["market_clusters"]) - 1, 1)
    row = row_from_matrix_estimate(
        beta=beta,
        cov=cov,
        names=names,
        specification=specification,
        estimator=estimator,
        sample_label=sample_label,
        inference_label=HEADLINE_INFERENCE,
        dof=dof,
        nobs=len(y),
        r_squared=r2,
        cluster_info=cluster_info,
    )
    row["weight_sum"] = float(np.sum(w))
    row["weight_min"] = float(np.min(w))
    row["weight_p50"] = float(np.quantile(w, 0.50))
    row["weight_p95"] = float(np.quantile(w, 0.95))
    row["weight_max"] = float(np.max(w))
    return row



# -----------------------------------------------------------------------------
# Final overlap helpers: split-price/total-price, linear/categorical stars,
# and positive-review inclusion/exclusion sensitivity.
# -----------------------------------------------------------------------------

def propensity_price_terms(price_measure):
    """Return the price variables used in the propensity-score model."""
    if price_measure == "split_price":
        return ["prezzo", "prezzo_spedizione_repaired"]
    if price_measure == "total_price":
        return ["prezzo_totale_reconstructed"]
    raise ValueError("price_measure must be 'split_price' or 'total_price'")


def propensity_star_terms(star_form):
    """Return the star-rating term used in the propensity-score model."""
    if star_form == "linear_stars":
        return ["stelle"]
    if star_form == "categorical_stars":
        return [STELLE_CATEGORICAL_TERM]
    raise ValueError("star_form must be 'linear_stars' or 'categorical_stars'")


def propensity_positive_review_terms(positive_review_form="positive_reviews_included"):
    """Return the positive-review term used in the propensity-score model."""
    if positive_review_form == "positive_reviews_included":
        return ["valutazioni_positive"]
    if positive_review_form == "positive_reviews_dropped":
        return []
    raise ValueError("positive_review_form must be 'positive_reviews_included' or 'positive_reviews_dropped'")


def propensity_rhs_terms(price_measure, star_form="linear_stars", positive_review_form="positive_reviews_included"):
    return [
        "log1p_num_valutazioni",
        *propensity_positive_review_terms(positive_review_form),
        *propensity_star_terms(star_form),
        *propensity_price_terms(price_measure),
        "contact_courier_flag",
        "g_cons_min_robust",
    ]


def propensity_formula(price_measure, star_form="linear_stars", positive_review_form="positive_reviews_included"):
    return "fba_from_shipper ~ " + " + ".join(propensity_rhs_terms(price_measure, star_form, positive_review_form)) + " + C(market_id)"


def fit_logit_propensity_fast(data, formula, ridge=1e-8, max_iter=80, tol=1e-7):
    y_df, x_df = dmatrices(formula, data=data, return_type="dataframe")
    y = np.asarray(y_df).ravel().astype(float)
    X = np.asarray(x_df, dtype=float)
    beta = np.zeros(X.shape[1], dtype=float)
    eye_k = np.eye(X.shape[1])
    for _ in range(max_iter):
        eta = np.clip(X @ beta, -35, 35)
        mu = np.clip(expit(eta), 1e-8, 1 - 1e-8)
        W = np.clip(mu * (1 - mu), 1e-8, None)
        z = eta + (y - mu) / W
        Xw = X * np.sqrt(W)[:, None]
        zw = z * np.sqrt(W)
        lhs = Xw.T @ Xw + ridge * eye_k
        rhs_vec = Xw.T @ zw
        try:
            beta_new = np.linalg.solve(lhs, rhs_vec)
        except np.linalg.LinAlgError:
            beta_new = stable_symmetric_pinv(lhs) @ rhs_vec
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break
        beta = beta_new
    pscore = pd.Series(expit(np.clip(X @ beta, -35, 35)), index=x_df.index)
    return pscore, beta, list(x_df.columns)


def propensity_overlap_diagnostics(
    data,
    price_measure="split_price",
    star_form="linear_stars",
    positive_review_form="positive_reviews_included",
    model_label=None,
):
    ps_formula = propensity_formula(price_measure, star_form, positive_review_form)
    pscore, _, _ = fit_logit_propensity_fast(data, ps_formula)
    if model_label is None:
        model_label = f"{price_measure}_{star_form}_{positive_review_form}"
    pscore = pd.Series(np.asarray(pscore), index=pscore.index, name=f"propensity_score_{model_label}").reindex(data.index)
    treated_mask = data["fba_from_shipper"].eq(1)
    treated = pscore[treated_mask]
    control = pscore[~treated_mask]
    common_low = max(float(treated.min()), float(control.min()))
    common_high = min(float(treated.max()), float(control.max()))
    keep_common = pscore.between(common_low, common_high)
    keep_005_095 = pscore.between(0.05, 0.95)
    keep_010_090 = pscore.between(0.10, 0.90)
    quant = (
        pd.DataFrame({"fba_from_shipper": data["fba_from_shipper"], "propensity_score": pscore})
        .groupby("fba_from_shipper")["propensity_score"]
        .quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        .unstack()
        .reset_index()
    )
    quant.insert(0, "model_label", model_label)
    quant.insert(1, "price_measure", price_measure)
    quant.insert(2, "star_form", star_form)
    quant.insert(3, "positive_review_form", positive_review_form)
    summary = pd.DataFrame([
        {
            "model_label": model_label,
            "price_measure": price_measure,
            "star_form": star_form,
            "positive_review_form": positive_review_form,
            "method": "logit propensity score diagnostic with market fixed effects",
            "propensity_formula": ps_formula,
            "nobs": int(len(pscore)),
            "treated_rows": int(treated_mask.sum()),
            "control_rows": int((~treated_mask).sum()),
            "treated_ps_min": float(treated.min()),
            "treated_ps_max": float(treated.max()),
            "control_ps_min": float(control.min()),
            "control_ps_max": float(control.max()),
            "common_support_low": common_low,
            "common_support_high": common_high,
            "common_support_rows": int(keep_common.sum()),
            "common_support_share": float(keep_common.mean()),
            "trim_0_05_0_95_rows": int(keep_005_095.sum()),
            "trim_0_05_0_95_share": float(keep_005_095.mean()),
            "trim_0_10_0_90_rows": int(keep_010_090.sum()),
            "trim_0_10_0_90_share": float(keep_010_090.mean()),
            "note": "Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.",
        }
    ])
    return summary, quant, pscore


def overlap_trimmed_estimates(
    data,
    rhs,
    pscore,
    price_measure="split_price",
    star_form="linear_stars",
    positive_review_form="positive_reviews_included",
    model_label=None,
):
    pscore = pd.Series(pscore, index=data.index).astype(float)
    treated = data["fba_from_shipper"].eq(1)
    common_low = max(float(pscore[treated].min()), float(pscore[~treated].min()))
    common_high = min(float(pscore[treated].max()), float(pscore[~treated].max()))
    cases = {
        "full_sample": pd.Series(True, index=data.index),
        "propensity_common_support": pscore.between(common_low, common_high),
        "propensity_0_05_0_95": pscore.between(0.05, 0.95),
        "propensity_0_10_0_90": pscore.between(0.10, 0.90),
    }
    rows = []
    if model_label is None:
        model_label = f"{price_measure}_{star_form}_{positive_review_form}"
    for label, mask in cases.items():
        case_df = data.loc[mask].copy()
        row = fit_ols_with_inference_fast(
            case_df,
            rhs,
            inference=HEADLINE_INFERENCE,
            specification=model_label,
            sample_label=label,
        )
        row["model_label"] = model_label
        row["price_measure"] = price_measure
        row["star_form"] = star_form
        row["positive_review_form"] = positive_review_form
        row["comparison_population"] = label
        row["rows"] = int(len(case_df))
        row["markets"] = int(case_df["market_id"].nunique())
        row["fba_rows"] = int(case_df["fba_from_shipper"].sum())
        row["nonfba_rows"] = int((1 - case_df["fba_from_shipper"]).sum())
        row["fba_share"] = float(case_df["fba_from_shipper"].mean()) if len(case_df) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)



def overlap_case_masks(data, pscore):
    """Return the full-sample and overlap-restricted masks used in the overlap-diagnostics block."""
    pscore = pd.Series(pscore, index=data.index).astype(float)
    treated = data["fba_from_shipper"].eq(1)
    common_low = max(float(pscore[treated].min()), float(pscore[~treated].min()))
    common_high = min(float(pscore[treated].max()), float(pscore[~treated].max()))
    return {
        "full_sample": pd.Series(True, index=data.index),
        "propensity_common_support": pscore.between(common_low, common_high),
        "propensity_0_05_0_95": pscore.between(0.05, 0.95),
        "propensity_0_10_0_90": pscore.between(0.10, 0.90),
    }


def overlap_balance_diagnostics(
    data,
    pscore,
    model_label,
    price_measure,
    star_form,
    positive_review_form,
    variables=None,
):
    """Compare FBA and non-FBA covariate balance inside each overlap sample.

    The function reports standardized mean differences and descriptive tests for
    the full sample and for each overlap restriction. It deliberately does not
    preserve the original FBA share. The goal is to show whether the overlap
    restriction improves comparability on observed variables.
    """
    if variables is None:
        variables = [
            "prezzo_totale_reconstructed",
            "prezzo",
            "prezzo_spedizione_repaired",
            "log1p_num_valutazioni",
            "valutazioni_positive",
            "stelle",
            "contact_courier_flag",
            "g_cons_min_robust",
        ]
    variables = [v for v in variables if v in data.columns]
    pscore = pd.Series(pscore, index=data.index).astype(float)
    masks = overlap_case_masks(data, pscore)
    propensity_terms = set(propensity_rhs_terms(price_measure, star_form, positive_review_form))

    detailed_rows = []
    for population_label, mask in masks.items():
        case = data.loc[mask].copy()
        if case.empty:
            continue
        treat = case["fba_from_shipper"].astype(int)
        for var in variables:
            x = pd.to_numeric(case[var], errors="coerce")
            x1 = x.loc[treat.eq(1)].dropna()
            x0 = x.loc[treat.eq(0)].dropna()
            if len(x1) == 0 or len(x0) == 0:
                smd = np.nan
                welch_stat = np.nan
                welch_p = np.nan
                mw_stat = np.nan
                mw_p = np.nan
                ks_stat = np.nan
                ks_p = np.nan
            else:
                smd = standardized_mean_difference(x, treat)
                try:
                    welch = stats.ttest_ind(x1, x0, equal_var=False, nan_policy="omit")
                    welch_stat = float(welch.statistic)
                    welch_p = float(welch.pvalue)
                except Exception:
                    welch_stat, welch_p = np.nan, np.nan
                try:
                    mw = stats.mannwhitneyu(x1, x0, alternative="two-sided", method="asymptotic")
                    mw_stat = float(mw.statistic)
                    mw_p = float(mw.pvalue)
                except Exception:
                    mw_stat, mw_p = np.nan, np.nan
                try:
                    ks = stats.ks_2samp(x1, x0, alternative="two-sided", mode="asymp")
                    ks_stat = float(ks.statistic)
                    ks_p = float(ks.pvalue)
                except Exception:
                    ks_stat, ks_p = np.nan, np.nan

            try:
                paired_means = (
                    pd.DataFrame({
                        "market_id": case["market_id"],
                        "fba_from_shipper": treat,
                        "value": x,
                    })
                    .groupby(["market_id", "fba_from_shipper"], observed=True)["value"]
                    .mean()
                    .unstack()
                )
                if 0 in paired_means.columns and 1 in paired_means.columns:
                    paired_diffs = (paired_means[1] - paired_means[0]).dropna().astype(float).to_numpy()
                else:
                    paired_diffs = np.asarray([], dtype=float)
            except Exception:
                paired_diffs = np.asarray([], dtype=float)
            if len(paired_diffs) >= 2:
                try:
                    paired_t = stats.ttest_1samp(paired_diffs, 0.0, nan_policy="omit")
                    paired_t_stat = float(paired_t.statistic)
                    paired_t_p = float(paired_t.pvalue)
                except Exception:
                    paired_t_stat, paired_t_p = np.nan, np.nan
                try:
                    wil = stats.wilcoxon(paired_diffs, zero_method="wilcox", alternative="two-sided", method="approx")
                    wil_stat = float(wil.statistic)
                    wil_p = float(wil.pvalue)
                except Exception:
                    wil_stat, wil_p = np.nan, np.nan
            else:
                paired_t_stat, paired_t_p, wil_stat, wil_p = np.nan, np.nan, np.nan, np.nan

            variable_used = (
                var in propensity_terms
                or (var == "stelle" and STELLE_CATEGORICAL_TERM in propensity_terms)
                or (var == "prezzo" and "prezzo" in propensity_terms)
                or (var == "prezzo_spedizione_repaired" and "prezzo_spedizione_repaired" in propensity_terms)
                or (var == "prezzo_totale_reconstructed" and "prezzo_totale_reconstructed" in propensity_terms)
            )
            detailed_rows.append({
                "model_label": model_label,
                "price_measure": price_measure,
                "star_form": star_form,
                "positive_review_form": positive_review_form,
                "comparison_population": population_label,
                "variable": var,
                "variable_used_in_propensity": bool(variable_used),
                "rows": int(len(case)),
                "fba_rows": int(treat.sum()),
                "nonfba_rows": int((1 - treat).sum()),
                "fba_share": float(treat.mean()) if len(case) else np.nan,
                "mean_fba": float(x1.mean()) if len(x1) else np.nan,
                "mean_nonfba": float(x0.mean()) if len(x0) else np.nan,
                "mean_diff_fba_minus_nonfba": float(x1.mean() - x0.mean()) if len(x1) and len(x0) else np.nan,
                "median_fba": float(x1.median()) if len(x1) else np.nan,
                "median_nonfba": float(x0.median()) if len(x0) else np.nan,
                "median_diff_fba_minus_nonfba": float(x1.median() - x0.median()) if len(x1) and len(x0) else np.nan,
                "smd_fba_minus_nonfba": smd,
                "abs_smd": abs(smd) if pd.notna(smd) else np.nan,
                "welch_t_stat": welch_stat,
                "welch_pvalue": welch_p,
                "mann_whitney_u_stat": mw_stat,
                "mann_whitney_pvalue": mw_p,
                "ks_statistic": ks_stat,
                "ks_pvalue": ks_p,
                "markets_with_both_groups": int(len(paired_diffs)),
                "market_mean_diff_fba_minus_nonfba": float(np.mean(paired_diffs)) if len(paired_diffs) else np.nan,
                "market_paired_t_stat": paired_t_stat,
                "market_paired_t_pvalue": paired_t_p,
                "market_wilcoxon_stat": wil_stat,
                "market_wilcoxon_pvalue": wil_p,
                "test_note": "Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.",
            })

    detailed = pd.DataFrame(detailed_rows)
    if detailed.empty:
        return detailed, pd.DataFrame()

    pvalue_columns = [
        "welch_pvalue",
        "mann_whitney_pvalue",
        "ks_pvalue",
        "market_paired_t_pvalue",
        "market_wilcoxon_pvalue",
    ]
    for col in pvalue_columns:
        adj_col = f"{col}_fdr_bh"
        detailed[adj_col] = np.nan
        for population_label, idx in detailed.groupby("comparison_population").groups.items():
            vals = detailed.loc[idx, col]
            ok = vals.notna()
            if ok.any():
                _, adj, _, _ = multipletests(vals.loc[ok], method="fdr_bh")
                detailed.loc[vals.loc[ok].index, adj_col] = adj

    summary = (
        detailed
        .groupby(["model_label", "price_measure", "star_form", "positive_review_form", "comparison_population"], dropna=False)
        .agg(
            rows=("rows", "max"),
            fba_rows=("fba_rows", "max"),
            nonfba_rows=("nonfba_rows", "max"),
            fba_share=("fba_share", "max"),
            mean_abs_smd=("abs_smd", "mean"),
            median_abs_smd=("abs_smd", "median"),
            max_abs_smd=("abs_smd", "max"),
            variables_abs_smd_above_0_10=("abs_smd", lambda s: int((s > 0.10).sum())),
            variables_abs_smd_above_0_25=("abs_smd", lambda s: int((s > 0.25).sum())),
        )
        .reset_index()
    )
    summary["balance_note"] = "Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large."
    return detailed, summary


def propensity_weighted_estimates(
    data,
    rhs,
    pscore,
    price_measure="split_price",
    star_form="linear_stars",
    positive_review_form="positive_reviews_included",
    model_label=None,
):
    pscore = pd.Series(pscore, index=data.index).astype(float).clip(1e-6, 1 - 1e-6)
    treated = data["fba_from_shipper"].eq(1)
    p_treat = float(data["fba_from_shipper"].mean())
    stabilized_ate_w = pd.Series(np.where(treated, p_treat / pscore, (1 - p_treat) / (1 - pscore)), index=data.index)
    cap = stabilized_ate_w.quantile(0.99)
    stabilized_ate_w_capped = stabilized_ate_w.clip(upper=cap)
    overlap_w = pd.Series(np.where(treated, 1 - pscore, pscore), index=data.index)
    if model_label is None:
        model_label = f"{price_measure}_{star_form}_{positive_review_form}"
    weighted_rows = []
    row = fit_weighted_market_fe_ols(
        data,
        rhs,
        stabilized_ate_w_capped,
        specification=f"{model_label}_stabilized_ipw_capped_99pct",
        estimator="market_fe_wls_stabilized_ipw_capped",
    )
    row["model_label"] = model_label
    row["price_measure"] = price_measure
    row["star_form"] = star_form
    row["positive_review_form"] = positive_review_form
    row["weighting_design"] = "stabilized_ipw_capped_99pct"
    weighted_rows.append(row)
    row = fit_weighted_market_fe_ols(
        data,
        rhs,
        overlap_w,
        specification=f"{model_label}_overlap_weighted_observable_support",
        estimator="market_fe_wls_overlap_weights",
    )
    row["model_label"] = model_label
    row["price_measure"] = price_measure
    row["star_form"] = star_form
    row["positive_review_form"] = positive_review_form
    row["weighting_design"] = "overlap_weights"
    weighted_rows.append(row)
    weighted_table = pd.DataFrame(weighted_rows)
    weighted_table["diagnostic_note"] = "Propensity-weighted diagnostics only; they do not change the thesis estimand into a causal effect."
    return weighted_table

# -------------------------------------------------------------------------
# Feature-analysis helpers used in Block 10.
# -------------------------------------------------------------------------

from statsmodels.stats.outliers_influence import variance_inflation_factor


def model_param_names(res):
    """Return parameter names from a statsmodels result object."""
    if hasattr(res.params, "index"):
        return list(res.params.index)
    return list(res.model.exog_names)


def covariance_to_dataframe(cov, res):
    """Convert a covariance matrix into a DataFrame with model term labels."""
    names = model_param_names(res)
    if isinstance(cov, pd.DataFrame):
        return cov
    return pd.DataFrame(cov, index=names, columns=names)


def resolve_feature_param_name(res, feature):
    """Resolve a model term name for continuous, binary, or categorical terms."""
    names = model_param_names(res)

    direct_candidates = [
        feature,
        f"{feature}[T.True]",
        f"{feature}[T.False]",
        f"{feature}[T.1]",
        f"{feature}[T.0]",
        f"C({feature})[T.True]",
        f"C({feature})[T.False]",
        f"C({feature})[T.1]",
        f"C({feature})[T.0]",
    ]

    for candidate in direct_candidates:
        if candidate in names:
            return candidate

    prefix_candidates = [
        f"{feature}[T.",
        f"C({feature})[T.",
    ]

    for prefix in prefix_candidates:
        matches = [name for name in names if name.startswith(prefix)]
        if len(matches) == 1:
            return matches[0]

    return None


def standardized_feature_association_table(res_plain, cov, data, feature_cols, dof_reference=None):
    """Summarize raw and standardized associations for selected headline features."""
    cov_df = covariance_to_dataframe(cov, res_plain)
    param_names = model_param_names(res_plain)

    y = pd.to_numeric(data[CONTINUOUS_OUTCOME], errors="coerce").astype(float)
    y_sd = float(y.std(ddof=1))

    rows = []

    for feature in feature_cols:
        if feature not in data.columns:
            rows.append(
                {
                    "feature": feature,
                    "model_term": None,
                    "status": "feature_not_found_in_dataframe",
                }
            )
            continue

        x = pd.to_numeric(data[feature], errors="coerce").astype(float)
        x_mean = float(x.mean())
        x_sd = float(x.std(ddof=1))

        term = resolve_feature_param_name(res_plain, feature)

        if term is None or term not in param_names:
            rows.append(
                {
                    "feature": feature,
                    "model_term": None,
                    "feature_mean": x_mean,
                    "feature_sd": x_sd,
                    "status": "term_not_found_in_model",
                }
            )
            continue

        beta = float(pd.Series(np.asarray(res_plain.params), index=param_names)[term])
        variance = float(cov_df.loc[term, term])
        se = float(np.sqrt(max(variance, 0))) if pd.notna(variance) else np.nan
        statistic = beta / se if pd.notna(se) and se > 0 else np.nan

        if dof_reference is not None and np.isfinite(dof_reference):
            pvalue = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof_reference))) if pd.notna(statistic) else np.nan
            crit = float(stats.t.ppf(0.975, df=dof_reference))
        else:
            pvalue = float(2 * (1 - stats.norm.cdf(abs(statistic)))) if pd.notna(statistic) else np.nan
            crit = float(stats.norm.ppf(0.975))

        standardized_beta = beta * x_sd / y_sd if x_sd > 0 and y_sd > 0 else np.nan
        standardized_se = se * x_sd / y_sd if pd.notna(se) and x_sd > 0 and y_sd > 0 else np.nan

        rows.append(
            {
                "feature": feature,
                "model_term": term,
                "raw_coefficient": beta,
                "cluster_robust_se": se,
                "test_statistic": statistic,
                "pvalue": pvalue,
                "ci_low": beta - crit * se if pd.notna(se) else np.nan,
                "ci_high": beta + crit * se if pd.notna(se) else np.nan,
                "stars": significance_stars(pvalue),
                "feature_mean": x_mean,
                "feature_sd": x_sd,
                "outcome_sd": y_sd,
                "standardized_coefficient": standardized_beta,
                "standardized_se": standardized_se,
                "standardization_note": "Raw coefficient scaled by feature SD and rank_pct SD.",
                "status": "estimated",
            }
        )

    return pd.DataFrame(rows)


def numeric_feature_frame(data, feature_cols):
    """Build a numeric feature matrix from selected columns."""
    out = pd.DataFrame(index=data.index)

    for col in feature_cols:
        if col not in data.columns:
            continue
        out[col] = pd.to_numeric(data[col], errors="coerce").astype(float)

    return out


def feature_correlation_and_vif_tables(data, feature_cols, market_col="market_id"):
    """Return market-demeaned feature correlations, strongest pairs, and VIFs.

    VIFs are computed from the inverse correlation matrix with a small ridge
    safeguard rather than repeated auxiliary regressions. This is algebraically
    equivalent for standardized regressors and materially faster/more stable in
    audit reruns.
    """
    X = numeric_feature_frame(data, feature_cols)

    if market_col in data.columns:
        X_dm = X - X.groupby(data[market_col]).transform("mean")
        correlation_note = "Market-demeaned correlations; market-level common components are removed."
    else:
        X_dm = X.copy()
        correlation_note = "Raw correlations; market_id was not available."

    valid_cols = [
        col for col in X_dm.columns
        if X_dm[col].notna().sum() > 2 and X_dm[col].std(ddof=1) > 1e-12
    ]
    X_dm = X_dm[valid_cols]

    if len(valid_cols) == 0:
        correlation_table = pd.DataFrame(columns=["feature", "correlation_note"])
        top_pairs_table = pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation", "correlation_note"])
        vif_table = pd.DataFrame(columns=["feature", "vif", "vif_note"])
        return correlation_table, top_pairs_table, vif_table

    correlation_table = X_dm.corr()
    correlation_table.insert(0, "feature", correlation_table.index)
    correlation_table = correlation_table.reset_index(drop=True)
    correlation_table["correlation_note"] = correlation_note

    pair_rows = []
    for i, first in enumerate(valid_cols):
        for second in valid_cols[i + 1:]:
            corr_value = float(X_dm[[first, second]].corr().iloc[0, 1])
            pair_rows.append({
                "feature_1": first,
                "feature_2": second,
                "correlation": corr_value,
                "abs_correlation": abs(corr_value),
                "correlation_note": correlation_note,
            })

    if pair_rows:
        top_pairs_table = pd.DataFrame(pair_rows).sort_values("abs_correlation", ascending=False).reset_index(drop=True).head(20)
    else:
        top_pairs_table = pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "abs_correlation", "correlation_note"])

    vif_rows = []
    X_vif = X_dm.dropna()
    if len(valid_cols) >= 2 and X_vif.shape[0] > len(valid_cols) + 1:
        Z = (X_vif[valid_cols] - X_vif[valid_cols].mean()) / X_vif[valid_cols].std(ddof=1).replace(0, np.nan)
        Z = Z.replace([np.inf, -np.inf], np.nan).dropna()
        if Z.shape[0] > len(valid_cols) + 1:
            R = np.corrcoef(np.asarray(Z, dtype=float), rowvar=False)
            R_inv = stable_symmetric_pinv(R, ridge_floor=1e-10, max_tries=6)
            vifs = np.diag(R_inv)
        else:
            vifs = [np.nan] * len(valid_cols)
        for col, vif_value in zip(valid_cols, vifs):
            vif_rows.append({
                "feature": col,
                "vif": float(vif_value) if pd.notna(vif_value) else np.nan,
                "vif_note": "VIF computed on market-demeaned standardized features from the inverse correlation matrix.",
            })
    else:
        for col in valid_cols:
            vif_rows.append({"feature": col, "vif": np.nan, "vif_note": "VIF not computed because too few valid features or observations remain."})

    vif_table = pd.DataFrame(vif_rows)
    return correlation_table, top_pairs_table, vif_table


def feature_block_wald_tests(res_plain, cov, feature_blocks, dof_reference=None):
    """Run Wald tests for groups of covariates in the headline specification."""
    cov_df = covariance_to_dataframe(cov, res_plain)
    param_names = model_param_names(res_plain)
    params = pd.Series(np.asarray(res_plain.params), index=param_names, dtype=float)

    rows = []

    for block_name, block_features in feature_blocks.items():
        terms = []
        missing_features = []

        for feature in block_features:
            term = resolve_feature_param_name(res_plain, feature)
            if term is None:
                missing_features.append(feature)
            elif term in params.index and term in cov_df.index and term in cov_df.columns:
                terms.append(term)
            else:
                missing_features.append(feature)

        if len(terms) == 0:
            rows.append(
                {
                    "feature_block": block_name,
                    "features": block_features,
                    "model_terms": [],
                    "df_num": 0,
                    "df_denom": np.nan,
                    "test_statistic": np.nan,
                    "wald_chi2": np.nan,
                    "pvalue": np.nan,
                    "stars": "",
                    "test_type": "not_computed",
                    "status": "no_estimable_terms",
                    "missing_features": missing_features,
                }
            )
            continue

        b = np.asarray(params[terms], dtype=float)
        V = np.asarray(cov_df.loc[terms, terms], dtype=float)

        try:
            wald_chi2 = float(b.T @ stable_symmetric_pinv(V) @ b)
        except Exception:
            wald_chi2 = np.nan

        df_num = int(len(terms))

        if pd.notna(wald_chi2) and df_num > 0:
            if dof_reference is not None and np.isfinite(dof_reference):
                test_statistic = wald_chi2 / df_num
                pvalue = float(stats.f.sf(test_statistic, df_num, dof_reference))
                test_type = "F test using supplied denominator degrees of freedom"
                df_denom = float(dof_reference)
            else:
                test_statistic = wald_chi2
                pvalue = float(stats.chi2.sf(wald_chi2, df_num))
                test_type = "Wald chi-square test"
                df_denom = np.nan
        else:
            test_statistic = np.nan
            pvalue = np.nan
            test_type = "not_computed"
            df_denom = np.nan

        rows.append(
            {
                "feature_block": block_name,
                "features": block_features,
                "model_terms": terms,
                "df_num": df_num,
                "df_denom": df_denom,
                "test_statistic": test_statistic,
                "wald_chi2": wald_chi2,
                "pvalue": pvalue,
                "stars": significance_stars(pvalue),
                "test_type": test_type,
                "status": "estimated",
                "missing_features": missing_features,
            }
        )

    return pd.DataFrame(rows)

### 9. Static results: total FBA advantage and residual FBA premium

The raw within-market FBA association is large. With market fixed effects and no additional controls (specification S1), the FBA coefficient on `rank_pct` is -0.3804. Adding reputation, price, shipping, and delivery controls (specification S4, the headline fully adjusted split-price model) attenuates the coefficient to -0.0510. The attenuation accounting decomposes the gap between the raw S1 coefficient and the residual S4 coefficient across the four control blocks.

The cell estimates the four nested static specifications (S1 FBA-only, S2 + reputation, S3 + reputation + price, S4 + reputation + price + shipping + delivery), computes the attenuation table, and writes the main results to `main_results_market_fe_rankpct.csv` and the attenuation decomposition to `attenuation_decomposition_rankpct.csv`. These tables correspond to Table 4 (`tab:ch5-static-models`) and Table 5 (`tab:ch5-attenuation`) of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 9. Static results: total FBA advantage and residual FBA premium
# -----------------------------------------------------------------------------
# Estimate the four nested static specifications (S1 to S4), compute the attenuation decomposition relative to S1, and export the main static results.

main_rows = []
main_result_store = {}
main_covariance_store = {}
main_dof_store = {}
for spec_name, rhs in SPECIFICATIONS.items():
    row = fit_ols_with_inference_fast(
        df_final,
        rhs,
        inference=HEADLINE_INFERENCE,
        specification=spec_name,
        sample_label="full_sample",
    )
    main_rows.append(row)
main_results_table = pd.DataFrame(main_rows)

headline_plain_res = fit_market_fe_ols_plain(df_final, SPECIFICATIONS[HEADLINE_SPEC])
headline_cov, headline_dof, headline_cluster_info = covariance_for_inference(
    headline_plain_res, df_final, HEADLINE_INFERENCE
)
main_result_store[HEADLINE_SPEC] = headline_plain_res
main_covariance_store[HEADLINE_SPEC] = headline_cov
main_dof_store[HEADLINE_SPEC] = headline_dof

# Internal equivalence check between the matrix headline estimate and the
# statsmodels headline estimate used for diagnostics.
_headline_term_name = resolve_param_name(headline_plain_res, base_name="fba_from_shipper")
_headline_matrix_beta = float(main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC), "coef_fba"].iloc[0])
_headline_statsmodels_beta = float(headline_plain_res.params[_headline_term_name])
register_check(
    "6",
    "matrix_and_statsmodels_headline_fba_match",
    bool(np.isclose(_headline_matrix_beta, _headline_statsmodels_beta, atol=1e-8)),
    f"matrix={_headline_matrix_beta:.12f}; statsmodels={_headline_statsmodels_beta:.12f}",
)

display(main_results_table)

headline_beta = _headline_matrix_beta

effect_translation_table = translate_rank_effect(headline_beta, df_final)
display(effect_translation_table)

attenuation_decomposition_table = build_attenuation_decomposition(main_results_table)
display(attenuation_decomposition_table)

price_measure_ols_rows = []
for model_label, rhs in PRICE_STAR_SPECIFICATIONS.items():
    meta = PRICE_STAR_METADATA[model_label]
    row = fit_ols_with_inference_fast(
        df_final,
        rhs,
        inference=HEADLINE_INFERENCE,
        specification=model_label,
        sample_label="full_sample",
    )
    row["model_label"] = model_label
    row["price_measure"] = meta["price_measure"]
    row["star_form"] = meta["star_form"]
    row["positive_review_form"] = meta["positive_review_form"]
    row["price_control_definition"] = meta["price_control_definition"]
    row["star_control_definition"] = meta["star_control_definition"]
    row["positive_review_control_definition"] = meta["positive_review_control_definition"]
    price_measure_ols_rows.append(row)
price_measure_ols_comparison_table = pd.DataFrame(price_measure_ols_rows)
display(price_measure_ols_comparison_table)

,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters
0,full_sample,rank_pct,spec_1_fba_only,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.380436,0.049852,-7.631346,1.897245e-10,-0.480120,-0.280751,***,61,5107,0.265520,119,62
1,full_sample,rank_pct,spec_2_fba_reputation,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.396409,0.055813,-7.102434,1.544898e-09,-0.508015,-0.284804,***,61,5107,0.304715,119,62
2,full_sample,rank_pct,spec_3_fba_reputation_totalprice_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.050726,0.015953,-3.179737,2.316896e-03,-0.082626,-0.018826,***,61,5107,0.959363,119,62
3,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,2.206536e-03,-0.082969,-0.019108,***,61,5107,0.959381,119,62


,reference_market_size,implied_rank_shift,note
0,82.370968,-4.153024,Mean third-party market size
1,80.000000,-4.032014,Median third-party market size


,comparison,interpretation_label,raw_fba_coef,adjusted_fba_coef,absolute_raw_gap,absolute_adjusted_gap,share_absorbed_by_added_controls,share_remaining_as_residual_association,absorbed_percent,remaining_percent,note
0,spec_1_fba_only -> spec_2_fba_reputation,conditioning_on_reputation,-0.380436,-0.396409,0.380436,0.396409,-0.041988,1.041988,-4.198844,104.198844,Attenuation accounting only; not a causal mediation design.
1,spec_1_fba_only -> spec_3_fba_reputation_totalprice_delivery,conditioning_on_reputation_total_price_delivery,-0.380436,-0.050726,0.380436,0.050726,0.866664,0.133336,86.666366,13.333634,Attenuation accounting only; not a causal mediation design.
2,spec_1_fba_only -> spec_4_fba_reputation_price_shipping_delivery,conditioning_on_reputation_price_shipping_delivery,-0.380436,-0.051038,0.380436,0.051038,0.865843,0.134157,86.584283,13.415717,Attenuation accounting only; not a causal mediation design.


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,model_label,price_measure,star_form,positive_review_form,price_control_definition,star_control_definition,positive_review_control_definition
0,full_sample,rank_pct,split_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,61,5107,0.959381,119,62,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,product price plus shipping price separately,star rating entered linearly,positive-review percentage included
1,full_sample,rank_pct,split_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.049066,0.014419,-3.402739,0.001183,-0.077899,-0.020232,***,61,5107,0.959292,119,62,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,product price plus shipping price separately,star rating entered linearly,positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy
2,full_sample,rank_pct,split_price_categorical_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.048094,0.016095,-2.988123,0.004040,-0.080278,-0.015910,***,61,5107,0.959973,119,62,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage included
3,full_sample,rank_pct,split_price_categorical_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.047433,0.014448,-3.283111,0.001702,-0.076323,-0.018543,***,61,5107,0.959963,119,62,split_price_categorical_stars_no_positive_reviews,split_price,categorical_stars,positive_reviews_dropped,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy
4,full_sample,rank_pct,total_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.050726,0.015953,-3.179737,0.002317,-0.082626,-0.018826,***,61,5107,0.959363,119,62,total_price_linear_stars,total_price,linear_stars,positive_reviews_included,total reconstructed buyer-facing price,star rating entered linearly,positive-review percentage included
5,full_sample,rank_pct,total_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.048892,0.014437,-3.386447,0.001244,-0.077761,-0.020022,***,61,5107,0.959281,119,62,total_price_linear_stars_no_positive_reviews,total_price,linear_stars,positive_reviews_dropped,total reconstructed buyer-facing price,star rating entered linearly,positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy
6,full_sample,rank_pct,total_price_categorical_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.047756,0.016040,-2.977319,0.004166,-0.079830,-0.015682,***,61,5107,0.959958,119,62,total_price_categorical_stars,total_price,categorical_stars,positive_reviews_included,total reconstructed buyer-facing price,"star rating entered as category indicators, 4.5-star reference",positive-review percentage included
7,full_sample,rank_pct,total_price_categorical_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.047075,0.014355,-3.279410,0.001721,-0.075779,-0.018371,***,61,5107,0.959948,119,62,total_price_categorical_stars_no_positive_reviews,total_price,categorical_stars,positive_reviews_dropped,total reconstructed buyer-facing price,"star rating entered as category indicators, 4.5-star reference",positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy


### 10. Post-treatment-control risk in the residual specification

The residual specification conditions on price, shipping, delivery, and reputation. The conditioning improves comparability but also changes the interpretation of the FBA coefficient, since some of those controls are part of the FBA commercial bundle. Repaired standard shipping price equals zero for FBA offers by construction; fast-delivery availability and Prime expectations are bundled with FBA enrollment. The residual coefficient is therefore read as ordering beyond the visible price-and-shipping structure rather than as the total effect of FBA on page position. This interpretation rule applies to every static specification that includes the shipping or delivery block.


### 11. Bounded-outcome, functional-form, and logistics-value-adjusted price diagnostics

`rank_pct` is bounded between 0 and 1. A linear fixed-effects model is a useful linear projection of the bounded outcome but cannot reproduce the boundedness. The cell computes three families of diagnostics:

1. **Bounded-outcome audit**: predicted-value distribution of the headline OLS, share of fitted values outside `[0, 1]`, and reference Papke-Wooldridge fractional logit re-estimation.
2. **Functional-form sensitivity**: quadratic price specification, spline-based price specification, equal-market-weighted specification, Mundlak correlated-effects specification.
3. **Logistics-value-adjusted price sensitivity**: re-estimation of S4 after replacing reconstructed total price with a total price that adds a hidden fast-delivery value `v` to FBA listings with fast-delivery availability. The baseline imputation is `v = 4.99` EUR; the upper-bound variant is `v = 8.99` EUR. By construction the adjustment widens the residual FBA coefficient, so the exercise is read as a parametric bounding sensitivity, not as a robustness confirmation.

The outputs populate `tab:ch5-static-validity`, `tab:app-ch6-bounded-outcome`, `tab:app-ch6-functional-form`, `tab:app-fast-delivery-imputation-audit`, and `tab:app-logistics-value-adjusted-price-results`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 11. Bounded-outcome, functional-form, and logistics-value-adjusted price diagnostics
# -----------------------------------------------------------------------------
# Bounded-outcome audit, fractional logit reference re-estimation, functional-form sensitivity (quadratic price, spline price, equal-market weighting, Mundlak), and logistics-value-adjusted price sensitivity at v in {4.99, 8.99}.

headline_rhs = SPECIFICATIONS[HEADLINE_SPEC]
headline_plain_res = main_result_store[HEADLINE_SPEC]

bounded_outcome_diagnostics = ols_prediction_diagnostics(headline_plain_res, df_final)
display(bounded_outcome_diagnostics)

# The fractional-response model is reported as a diagnostic, not as a replacement for the linear estimand.
fractional_logit_table, fractional_logit_ape_table = fractional_logit_diagnostic(
    df_final,
    headline_rhs,
    inference=HEADLINE_INFERENCE,
    specification=HEADLINE_SPEC,
)
display(fractional_logit_table)
display(fractional_logit_ape_table)

bounded_outcome_method_note = pd.DataFrame([
    {
        "method_considered": "bounded-outcome diagnostics",
        "decision_in_this_notebook": "report fitted-value diagnostics and fractional-response diagnostics, while keeping the linear rank-percent projection as headline",
        "reason": "The headline coefficient is directly interpretable in rank-percent units. Fractional response estimates a different nonlinear conditional-mean object and is therefore treated as a diagnostic rather than as confirmatory evidence.",
    }
])
display(bounded_outcome_method_note)

functional_form_diagnostics_table = functional_form_diagnostics(headline_plain_res)
display(functional_form_diagnostics_table)

spline_functional_form_table = spline_functional_form_sensitivity(df_final)
display(spline_functional_form_table)

# Thesis-facing functional-form synthesis. This table makes the RESET and
# nonlinear-estimator disagreement explicit instead of leaving it implicit in
# separate appendix files.
try:
    _reset_power2 = functional_form_diagnostics_table.loc[
        functional_form_diagnostics_table["diagnostic"].eq("Ramsey RESET power 2")
    ].iloc[0]
except Exception:
    _reset_power2 = pd.Series(dtype="object")
_functional_rows = []
_functional_rows.append({
    "diagnostic_family": "linear_projection_headline",
    "estimate": float(main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC), "coef_fba"].iloc[0]),
    "se": float(main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC), "se_fba"].iloc[0]),
    "pvalue": float(main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC), "pvalue_fba"].iloc[0]),
    "interpretation": "Headline within-market linear rank-percent projection; directly interpretable but not a proof of correct conditional-mean functional form.",
})
_functional_rows.append({
    "diagnostic_family": "quadratic_controls",
    "estimate": float(spline_functional_form_table["coef_fba"].iloc[0]),
    "se": float(spline_functional_form_table["se_fba"].iloc[0]),
    "pvalue": float(spline_functional_form_table["pvalue_fba"].iloc[0]),
    "interpretation": "Sensitivity with centered quadratic price, review-count, and delivery controls; magnitude is smaller but the sign remains negative.",
})
_functional_rows.append({
    "diagnostic_family": "fractional_logit_ape",
    "estimate": float(fractional_logit_ape_table["estimate"].iloc[0]),
    "se": float(fractional_logit_ape_table["se"].iloc[0]),
    "pvalue": float(fractional_logit_ape_table["pvalue"].iloc[0]),
    "interpretation": "Bounded-outcome diagnostic. It estimates a different nonlinear conditional-mean object and does not confirm the OLS magnitude.",
})
functional_form_fragility_table = pd.DataFrame(_functional_rows)
functional_form_fragility_table["ratio_to_headline_abs"] = (
    functional_form_fragility_table["estimate"].abs()
    / max(abs(functional_form_fragility_table.loc[0, "estimate"]), 1e-12)
)
functional_form_fragility_table["reset_power2_statistic"] = float(_reset_power2.get("statistic", np.nan))
functional_form_fragility_table["reset_power2_pvalue"] = float(_reset_power2.get("pvalue", np.nan))
functional_form_fragility_table["claim_implication"] = (
    "Report a model-dependent magnitude range; do not present the static coefficient as invariant to functional form."
)
display(functional_form_fragility_table)


extended_control_sensitivity_table = extended_control_sensitivity(df_final, EXTENDED_CONTROL_SPECIFICATIONS)
display(extended_control_sensitivity_table)


def fast_delivery_premium_sensitivity(data):
    """Document the algebraic invariance of the FBA coefficient to the
    euro level used to rescale the binary fast-delivery indicator.

    Construction. The variable cost = premium * fast_delivery_available * fba_from_shipper is a
    deterministic rescaling of the same FBA-only fast-delivery regressor. Multiplying a single
    regressor by a scalar leaves the OLS coefficients on all other regressors
    numerically invariant; only the coefficient on the rescaled variable itself
    moves, by the inverse multiplicative factor. This sensitivity table is
    therefore a transparency exercise that confirms an algebraic identity:
    changing the euro premium between 3.99, 4.99, and 8.99 does not change the
    underlying FBA-only fast-delivery support pattern. The substantive sensitivity
    question is not the level of the imputed premium but the entry form of the
    fast-delivery channel into the model and the aggregation of price and
    the fast-delivery monetary channel into the constructed price. This is addressed in the
    FBA-only logistics-value-adjusted price specification below, which is not algebraically equivalent
    to the headline split-price design.
    """
    rows = []
    baseline_coef = float(main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC), "coef_fba"].iloc[0])
    for scenario, premium in FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR.items():
        d = data.copy()
        slug = slugify_text(scenario) if "slugify_text" in globals() else re.sub(r"[^A-Za-z0-9_]+", "_", str(scenario)).strip("_")
        cost_col = f"fast_delivery_cost_imputed_for_sensitivity_{slug}"
        d[cost_col] = (
            float(premium)
            * d["fast_delivery_available"].fillna(0).astype(float)
            * d["fba_from_shipper"].fillna(0).astype(float)
        )
        rhs = SPECIFICATIONS[HEADLINE_SPEC] + [cost_col]
        row = fit_ols_with_inference_fast(
            d,
            rhs,
            inference=HEADLINE_INFERENCE,
            specification=f"fast_delivery_premium_{slug}",
            sample_label="full_sample",
        )
        row["scenario"] = scenario
        row["premium_eur"] = float(premium)
        row["premium_role"] = "baseline" if abs(float(premium) - float(FAST_DELIVERY_PREMIUM_EUR)) < 1e-12 else "sensitivity"
        row["fast_delivery_cost_variable"] = cost_col
        row["coef_change_from_static_headline"] = float(row["coef_fba"] - baseline_coef)
        row["absolute_change_from_static_headline"] = float(abs(row["coef_fba"] - baseline_coef))
        row["interpretation"] = "FBA-only monetary proxy check: cost = premium * fast_delivery_available * fba_from_shipper. The euro premium changes the monetary scale but not the underlying FBA-only fast-delivery support pattern. Substantive sensitivity to the price aggregator is reported in the logistics-value-adjusted price specification."
        rows.append(row)
    out = pd.DataFrame(rows)
    out = out.sort_values("premium_eur").reset_index(drop=True)
    baseline_grid_coef = out.loc[out["premium_role"].eq("baseline"), "coef_fba"]
    if not baseline_grid_coef.empty:
        b = float(baseline_grid_coef.iloc[0])
        out["coef_change_from_grid_baseline"] = out["coef_fba"] - b
        out["abs_coef_change_from_grid_baseline"] = (out["coef_fba"] - b).abs()
    return out

fast_delivery_premium_sensitivity_table = fast_delivery_premium_sensitivity(df_final)
display(fast_delivery_premium_sensitivity_table)


def build_oster_sensitivity_grid(results_table, multipliers=(1.005, 1.01, 1.02, 1.03, 1.04, 1.05, 1.10)):
    """Oster-style coefficient-stability grid across Rmax multipliers.

    Multiplier choice. The headline specification has fixed-effect R-squared
    around 0.96, so any multiplier above approximately 1.04 produces an Rmax
    that is mechanically capped at the implementation ceiling 0.999. Probing
    only multipliers above the cap collapses the grid to a single value and
    fails to illustrate Rmax sensitivity. The default grid here mixes
    sub-cap multipliers (1.005, 1.01, 1.02, 1.03, 1.04) that yield distinct
    Rmax values with one above-cap multiplier (1.05) and a far-above multiplier
    (1.10) that document the cap floor for the diagnostic is not a causal
    proof: the delta_to_zero magnitude can be very large at small multipliers
    when (Rmax - R_full) is small, so the grid is interpreted as a stability
    band, not as a single point estimate.
    """
    short = results_table.loc[results_table["specification"].eq("spec_1_fba_only")].iloc[0]
    full = results_table.loc[results_table["specification"].eq(HEADLINE_SPEC)].iloc[0]
    beta_short = float(short["coef_fba"])
    beta_full = float(full["coef_fba"])
    r_short = float(short["r_squared"])
    r_full = float(full["r_squared"])
    rows = []
    for mult in multipliers:
        rmax = min(float(mult) * r_full, 0.999)
        denom = (beta_short - beta_full) * (rmax - r_full)
        delta = np.nan if (rmax <= r_full or abs(denom) < 1e-12) else ((beta_full - 0.0) * (r_full - r_short)) / denom
        rows.append({
            "comparison": "spec_1_fba_only_to_spec_4_full_controls",
            "rmax_multiplier": float(mult),
            "rmax_used": float(rmax),
            "beta_short": beta_short,
            "r2_short": r_short,
            "beta_full": beta_full,
            "r2_full": r_full,
            "delta_to_zero": float(delta) if pd.notna(delta) else np.nan,
            "abs_delta_gt_one": bool(pd.notna(delta) and abs(delta) > 1),
            "interpretation": "Oster-style diagnostic; coefficient stability is evaluated jointly with R-squared movement and high FE R-squared makes exact delta magnitudes sensitive to Rmax.",
        })
    return pd.DataFrame(rows)

oster_sensitivity_grid_table = build_oster_sensitivity_grid(main_results_table)
display(oster_sensitivity_grid_table)


market_equal_weighted_table = fit_market_equal_weighted_ols(df_final, headline_rhs)
display(market_equal_weighted_table)

delivery_tail_sensitivity_table = delivery_tail_sensitivity(df_final, headline_rhs)
display(delivery_tail_sensitivity_table)


# -----------------------------------------------------------------------------
# FBA-only logistics-value-adjusted price specification.
#
# Rationale. The headline split-price specification controls for product price
# and displayed/repaired standard shipping price separately. A binary
# fast-delivery indicator, or a deterministic euro-rescaling of that indicator,
# does not change the price object entering the model. This block therefore
# constructs a substantive price sensitivity before estimation.
#
# The adjusted total price adds an imputed monetary value/cost of fast delivery
# only to FBA observations with fast-delivery availability:
#
#     prezzo_totale_logistics_value_adjusted =
#         prezzo + prezzo_spedizione_repaired
#         + surcharge_eur * fast_delivery_available * fba_from_shipper
#
# The purpose is to narrow price differences before the static model is fitted.
# The adjustment monetizes a logistics component embedded in the FBA bundle that
# is not separately observed in the scraped offer price. It must not be read as
# a directly observed checkout price. The coefficient is therefore a residual
# association after an FBA-only hidden-logistics-value imputation, not a causal
# estimate and not a robustness check that preserves the headline estimand.
#
# The EUR 4.99 benchmark is the baseline monetary translation. EUR 3.99 and
# EUR 8.99 are one-way sensitivity translations.

LOGISTICS_VALUE_SURCHARGE_BASELINE_EUR = float(FAST_DELIVERY_PREMIUM_EUR)
LOGISTICS_VALUE_SURCHARGE_LOWER_SENSITIVITY_EUR = 3.99
LOGISTICS_VALUE_SURCHARGE_UPPER_SENSITIVITY_EUR = 8.99

LOGISTICS_VALUE_PRICE_RHS = [
    "fba_from_shipper",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
    "prezzo_totale_logistics_value_adjusted",
    "contact_courier_flag",
    "g_cons_min_robust",
]


def build_logistics_value_adjusted_total_price(data, surcharge_eur, variant_label):
    """Construct the FBA-only logistics-value-adjusted total price.

    Adjusted total price =
        prezzo
        + prezzo_spedizione_repaired
        + surcharge_eur * fast_delivery_available * fba_from_shipper

    The `fba_from_shipper` factor is intentional. The specification imputes a
    hidden fast-delivery value/cost only to FBA listings that expose fast
    delivery availability. Non-FBA listings receive zero in this sensitivity.
    The constructed variable is not a directly observed checkout price.
    """
    out = data.copy()
    fd = out["fast_delivery_available"].fillna(0).astype(float)
    fba = out["fba_from_shipper"].astype(float)
    fba_only_imputation = float(surcharge_eur) * fd * fba
    out["fba_only_fast_delivery_value_imputed"] = fba_only_imputation
    out["prezzo_totale_logistics_value_adjusted"] = (
        out["prezzo"].astype(float)
        + out["prezzo_spedizione_repaired"].astype(float)
        + fba_only_imputation
    )
    out["logistics_value_variant"] = variant_label
    return out


# Construct the baseline and one-way sensitivity panels.
_lva_baseline_panel = build_logistics_value_adjusted_total_price(
    df_final,
    LOGISTICS_VALUE_SURCHARGE_BASELINE_EUR,
    "logistics_value_adjusted_fba_only_4_99_baseline",
)
_lva_lower_panel = build_logistics_value_adjusted_total_price(
    df_final,
    LOGISTICS_VALUE_SURCHARGE_LOWER_SENSITIVITY_EUR,
    "logistics_value_adjusted_fba_only_3_99_lower",
)
_lva_upper_panel = build_logistics_value_adjusted_total_price(
    df_final,
    LOGISTICS_VALUE_SURCHARGE_UPPER_SENSITIVITY_EUR,
    "logistics_value_adjusted_fba_only_8_99_upper",
)

logistics_value_adjusted_audit_rows = []
for label, panel in [
    ("logistics_value_adjusted_fba_only_4_99_baseline", _lva_baseline_panel),
    ("logistics_value_adjusted_fba_only_3_99_lower", _lva_lower_panel),
    ("logistics_value_adjusted_fba_only_8_99_upper", _lva_upper_panel),
]:
    for fba_label, fba_value in [("FBA", 1), ("non_FBA", 0)]:
        sub = panel.loc[panel["fba_from_shipper"] == fba_value]
        logistics_value_adjusted_audit_rows.append({
            "logistics_value_variant": label,
            "fba_group": fba_label,
            "n_obs": int(len(sub)),
            "n_with_fast_delivery": int(
                (sub["fast_delivery_available"].astype(float) == 1).sum()
            ),
            "share_with_fast_delivery": float(
                sub["fast_delivery_available"].astype(float).mean()
            ),
            "mean_prezzo_totale_reconstructed": float(
                sub["prezzo_totale_reconstructed"].mean()
            ),
            "mean_prezzo_totale_logistics_value_adjusted": float(
                sub["prezzo_totale_logistics_value_adjusted"].mean()
            ),
            "mean_fba_only_fast_delivery_value_imputed": float(
                sub["fba_only_fast_delivery_value_imputed"].mean()
            ),
            "max_fba_only_fast_delivery_value_imputed": float(
                sub["fba_only_fast_delivery_value_imputed"].max()
            ),
        })
logistics_value_adjusted_audit_table = pd.DataFrame(logistics_value_adjusted_audit_rows)
display(logistics_value_adjusted_audit_table)


# Diagnostic: rank of design with the adjusted-price specification.
def _logistics_value_design_rank_check(panel, label):
    cols = [c for c in LOGISTICS_VALUE_PRICE_RHS if c in panel.columns]
    sub = panel[cols].dropna()
    X = sub.values.astype(float)
    if X.shape[0] == 0 or X.shape[1] == 0:
        return {
            "logistics_value_variant": label,
            "n_after_dropna": int(X.shape[0]),
            "n_design_columns": int(X.shape[1]),
            "rank": 0,
            "condition_number": float("nan"),
            "full_rank": False,
        }
    rank = int(np.linalg.matrix_rank(X))
    cond = float(np.linalg.cond(X))
    return {
        "logistics_value_variant": label,
        "n_after_dropna": int(X.shape[0]),
        "n_design_columns": int(X.shape[1]),
        "rank": rank,
        "condition_number": cond,
        "full_rank": bool(rank == X.shape[1]),
    }


logistics_value_adjusted_design_rank_table = pd.DataFrame([
    _logistics_value_design_rank_check(
        _lva_baseline_panel, "logistics_value_adjusted_fba_only_4_99_baseline"
    ),
    _logistics_value_design_rank_check(
        _lva_lower_panel, "logistics_value_adjusted_fba_only_3_99_lower"
    ),
    _logistics_value_design_rank_check(
        _lva_upper_panel, "logistics_value_adjusted_fba_only_8_99_upper"
    ),
])
display(logistics_value_adjusted_design_rank_table)


# Estimation across the FBA-only adjusted-price panels.
_split_price_headline_coef = float(
    main_results_table.loc[
        main_results_table["specification"].eq(HEADLINE_SPEC), "coef_fba"
    ].iloc[0]
)


def _fit_logistics_value_adjusted_price(panel, label):
    row = fit_ols_with_inference_fast(
        panel,
        LOGISTICS_VALUE_PRICE_RHS,
        inference=HEADLINE_INFERENCE,
        specification=label,
        sample_label="full_sample",
    )
    row["logistics_value_variant"] = label
    row["coef_change_from_split_price_headline"] = float(
        row["coef_fba"] - _split_price_headline_coef
    )
    row["abs_pct_change_from_split_price_headline"] = float(
        100.0 * abs(row["coef_fba"] - _split_price_headline_coef)
        / max(abs(_split_price_headline_coef), 1e-12)
    )
    return row


logistics_value_adjusted_results_table = pd.DataFrame([
    _fit_logistics_value_adjusted_price(
        _lva_baseline_panel, "logistics_value_adjusted_fba_only_4_99_baseline"
    ),
    _fit_logistics_value_adjusted_price(
        _lva_lower_panel, "logistics_value_adjusted_fba_only_3_99_lower"
    ),
    _fit_logistics_value_adjusted_price(
        _lva_upper_panel, "logistics_value_adjusted_fba_only_8_99_upper"
    ),
])
logistics_value_adjusted_results_table["interpretation"] = (
    "FBA-only logistics-value-adjusted price specification. The split-price "
    "headline and this row estimate different empirical objects: the headline "
    "is the residual rank gap after conditioning on product price and displayed "
    "shipping price separately; this row is the residual rank gap after adding "
    "an imputed hidden fast-delivery value/cost only to FBA offers with fast "
    "delivery availability before estimation. The constructed price is not a "
    "directly observed checkout price. The adjustment is post-treatment with "
    "respect to the FBA bundle and is reported as a sensitivity estimand, not "
    "as a causal robustness check on the headline."
)
display(logistics_value_adjusted_results_table)

# Cross-variant summary: stability of the FBA coefficient across monetary translations.
logistics_value_adjusted_summary_table = logistics_value_adjusted_results_table[[
    "specification", "logistics_value_variant", "coef_fba", "se_fba", "pvalue_fba",
    "ci_low", "ci_high", "stars",
    "coef_change_from_split_price_headline",
    "abs_pct_change_from_split_price_headline",
    "nobs", "r_squared",
]].copy()
logistics_value_adjusted_summary_table["split_price_headline_coef_fba"] = _split_price_headline_coef
display(logistics_value_adjusted_summary_table)


,outcome,actual_min,actual_max,endpoint_zero_rows,endpoint_one_rows,fitted_min,fitted_max,fitted_outside_unit_interval_rows,fitted_outside_unit_interval_share,interpretation
0,rank_pct,0.0,1.0,62,62,-0.070025,1.568807,373,0.073037,Diagnostic for bounded-outcome concerns; OLS remains the main linear rank-percent estimator.


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,scale_note
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,papke_wooldridge_fractional_logit_irls,two_way_seller_market,fba_from_shipper,-0.085002,0.070025,-1.213867,0.229477,-0.225026,0.055023,,61,5107,NaN,119,62,Coefficient is on the fractional-logit index scale; estimated by IRLS quasi-likelihood.


,sample,outcome,specification,estimator,inference,effect_scale,term_name,estimate,se,test_statistic,pvalue,ci_low,ci_high,stars,dof_reference,interpretation_note
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,papke_wooldridge_fractional_logit_irls,two_way_seller_market,average_partial_effect_on_rank_pct,fba_from_shipper,-0.014065,0.011644,-1.207963,0.231724,-0.037348,0.009218,,61,Conservative bounded-outcome diagnostic; not treated as confirmation or replacement of the linear rank-percent projection.


,method_considered,decision_in_this_notebook,reason
0,bounded-outcome diagnostics,"report fitted-value diagnostics and fractional-response diagnostics, while keeping the linear rank-percent projection as headline",The headline coefficient is directly interpretable in rank-percent units. Fractional response estimates a different nonlinear conditional-mean object and is therefore treated a...


,diagnostic,status,statistic,pvalue,df_num,df_den,interpretation
0,Ramsey RESET power 2,estimated,5994.163008,0.0,1.0,5036.0,Direct nested-model RESET F-test using powers of fitted values; rejection flags possible functional-form misspecification.
1,Ramsey RESET power 3,estimated,8320.171193,0.0,2.0,5035.0,Direct nested-model RESET F-test using powers of fitted values; rejection flags possible functional-form misspecification.
2,Harvey-Collier linearity test,not_reported,NaN,NaN,NaN,NaN,Not interpreted because the high-dimensional market fixed-effects design makes the recursive-residual setup ill-conditioned; RESET and nonlinear-control sensitivity are used in...


...

,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,restriction,cutoff,rows_dropped
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,61,5107,0.959381,119,62,none,NaN,NaN
1,delivery_tail_sensitivity,rank_pct,drop_top_1pct_g_cons_min_robust,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.052336,0.016013,-3.268336,0.001780,-0.084357,-0.020316,***,61,5071,0.960872,117,62,drop rows with g_cons_min_robust above p99,19.0,36.0
2,delivery_tail_sensitivity,rank_pct,drop_top_1pct_g_cons_max_robust,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.052080,0.016136,-3.227558,0.002010,-0.084346,-0.019814,***,61,5060,0.960748,117,62,drop rows with g_cons_max_robust above p99,33.0,47.0


,logistics_value_variant,fba_group,n_obs,n_with_fast_delivery,share_with_fast_delivery,mean_prezzo_totale_reconstructed,mean_prezzo_totale_logistics_value_adjusted,mean_fba_only_fast_delivery_value_imputed,max_fba_only_fast_delivery_value_imputed
0,logistics_value_adjusted_fba_only_4_99_baseline,FBA,996,146,0.146586,45.103484,45.834950,0.731466,4.99
1,logistics_value_adjusted_fba_only_4_99_baseline,non_FBA,4111,844,0.205303,55.945018,55.945018,0.000000,0.00
2,logistics_value_adjusted_fba_only_3_99_lower,FBA,996,146,0.146586,45.103484,45.688363,0.584880,3.99
3,logistics_value_adjusted_fba_only_3_99_lower,non_FBA,4111,844,0.205303,55.945018,55.945018,0.000000,0.00
4,logistics_value_adjusted_fba_only_8_99_upper,FBA,996,146,0.146586,45.103484,46.421295,1.317811,8.99
5,logistics_value_adjusted_fba_only_8_99_upper,non_FBA,4111,844,0.205303,55.945018,55.945018,0.000000,0.00


,logistics_value_variant,n_after_dropna,n_design_columns,rank,condition_number,full_rank
0,logistics_value_adjusted_fba_only_4_99_baseline,5107,7,7,1040.391383,True
1,logistics_value_adjusted_fba_only_3_99_lower,5107,7,7,1040.231400,True
2,logistics_value_adjusted_fba_only_8_99_upper,5107,7,7,1041.039719,True


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,logistics_value_variant,coef_change_from_split_price_headline,abs_pct_change_from_split_price_headline,interpretation
0,full_sample,rank_pct,logistics_value_adjusted_fba_only_4_99_baseline,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.076010,0.015597,-4.873232,8.167038e-06,-0.107199,-0.044821,***,61,5107,0.954124,119,62,logistics_value_adjusted_fba_only_4_99_baseline,-0.024972,48.927807,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...
1,full_sample,rank_pct,logistics_value_adjusted_fba_only_3_99_lower,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.070470,0.015543,-4.533896,2.762769e-05,-0.101550,-0.039390,***,61,5107,0.956160,119,62,logistics_value_adjusted_fba_only_3_99_lower,-0.019432,38.072795,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...
2,full_sample,rank_pct,logistics_value_adjusted_fba_only_8_99_upper,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.100119,0.016594,-6.033453,1.026807e-07,-0.133301,-0.066937,***,61,5107,0.941277,119,62,logistics_value_adjusted_fba_only_8_99_upper,-0.049081,96.165531,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...


,specification,logistics_value_variant,coef_fba,se_fba,pvalue_fba,ci_low,ci_high,stars,coef_change_from_split_price_headline,abs_pct_change_from_split_price_headline,nobs,r_squared,split_price_headline_coef_fba
0,logistics_value_adjusted_fba_only_4_99_baseline,logistics_value_adjusted_fba_only_4_99_baseline,-0.076010,0.015597,8.167038e-06,-0.107199,-0.044821,***,-0.024972,48.927807,5107,0.954124,-0.051038
1,logistics_value_adjusted_fba_only_3_99_lower,logistics_value_adjusted_fba_only_3_99_lower,-0.070470,0.015543,2.762769e-05,-0.101550,-0.039390,***,-0.019432,38.072795,5107,0.956160,-0.051038
2,logistics_value_adjusted_fba_only_8_99_upper,logistics_value_adjusted_fba_only_8_99_upper,-0.100119,0.016594,1.026807e-07,-0.133301,-0.066937,***,-0.049081,96.165531,5107,0.941277,-0.051038


### 12. Static inference and robustness

The headline static coefficient is evaluated under several inference and specification checks. Two-way seller-market clustering is the headline inference choice because residual dependence operates along both dimensions of the panel. The cell computes:

- alternative clustering structures (one-way by seller, one-way by market, two-way),
- CR2 finite-cluster correction with Bell-McCaffrey degrees of freedom,
- restricted-residual wild cluster bootstrap with 4,999 Rademacher draws,
- sample-exclusion sensitivity (review-problem rows, contact-courier rows, failed-delivery markets, top and bottom rank percentiles),
- temporal-stability splits across consecutive market windows,
- Mundlak correlated-effects specification,
- influence-trimming on the highest-Cook'''s-distance observations,
- omitted-variable sensitivity following Oster (2019).

The outputs feed Tables 6 and 7 of the thesis and the corresponding appendix tables `tab:app-ch6-finite-cluster-inference`, `tab:app-ch6-sample-sensitivity`, `tab:app-ch6-extended-controls`, `tab:app-ch6-static-additional`, `tab:app-ch6-influence`, `tab:app-ch6-omitted-variable-sensitivity`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 12. Static inference and robustness
# -----------------------------------------------------------------------------
# Sample-exclusion sensitivity, temporal-stability splits, Mundlak correlated-effects diagnostic, influence trimming, CR2 correction, wild cluster bootstrap, and Oster (2019) sensitivity.

sensitivity_cases = {
    "full_sample": df_final.copy(),
    "drop_review_problem": df_final.loc[df_final["review_problem_flag"].eq(0)].copy(),
    "drop_contact_courier": df_final.loc[df_final["contact_courier_flag"].eq(0)].copy(),
    "drop_delivery_failure_markets": df_final.loc[df_final["market_delivery_failure_flag"].eq(0)].copy(),
}

sample_sensitivity_rows = []
for case_name, case_df in sensitivity_cases.items():
    row = fit_ols_with_inference_fast(
        case_df,
        headline_rhs,
        inference=HEADLINE_INFERENCE,
        specification=HEADLINE_SPEC,
        sample_label=case_name,
    )
    row["markets"] = int(case_df["market_id"].nunique())
    sample_sensitivity_rows.append(row)
sample_sensitivity_table = pd.DataFrame(sample_sensitivity_rows)
display(sample_sensitivity_table)

# Temporal stability using the preprocessing holdout flag.
temporal_rows = []
temporal_cases = {
    "estimation_sample": df_final.loc[df_final["temporal_holdout_flag"].eq(0)].copy(),
    "holdout_sample": df_final.loc[df_final["temporal_holdout_flag"].eq(1)].copy(),
}
for case_name, case_df in temporal_cases.items():
    row = fit_ols_with_inference_fast(
        case_df,
        headline_rhs,
        inference=HEADLINE_INFERENCE,
        specification=HEADLINE_SPEC,
        sample_label=case_name,
    )
    row["markets"] = int(case_df["market_id"].nunique())
    temporal_rows.append(row)
temporal_stability_table = pd.DataFrame(temporal_rows)
display(temporal_stability_table)

# Mundlak correlated-effects robustness.
mundlak_rhs = [term for term in headline_rhs if term != "fba_from_shipper"]
mundlak_data = build_mundlak_means(df_final, mundlak_rhs, id_col="seller_id")
mundlak_mean_terms = [f"{term}_seller_mean" for term in mundlak_rhs]
mundlak_row = fit_ols_with_inference_fast(
    mundlak_data,
    ["fba_from_shipper"] + mundlak_rhs + mundlak_mean_terms,
    inference=HEADLINE_INFERENCE,
    specification="mundlak_correlated_effects",
    sample_label="full_sample",
)
mundlak_row["estimator"] = "pooled_ols_with_seller_means_and_market_fe"
mundlak_rank_table = pd.DataFrame([mundlak_row])
display(mundlak_rank_table)

# Leave-one-cluster and observation-level influence checks.
leave_one_cluster_detail_table = leave_one_cluster_influence_detail(df_final, headline_rhs)
influence_diagnostics_table = leave_one_cluster_influence(df_final, headline_rhs)
observation_influence_summary_table, top_observation_influence_table = observation_influence_diagnostics(headline_plain_res, df_final, top_n=20)
influence_trimmed_table = influence_trimmed_estimates(df_final, headline_rhs, headline_plain_res, trim_share=0.01)
display(influence_diagnostics_table)
display(leave_one_cluster_detail_table.head(12))
display(observation_influence_summary_table)
display(top_observation_influence_table)
display(influence_trimmed_table)

# Omitted-variable sensitivity calibration.
headline_row = main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC)].iloc[0]
omitted_variable_sensitivity_table = omitted_variable_sensitivity_from_row(headline_row)
display(omitted_variable_sensitivity_table)

# Inference sensitivity for the primary specification.
cluster_sensitivity_rows = []
for inference_label in ["two_way_seller_market", "seller_cluster", "market_cluster"]:
    cov_i, dof_i, cluster_info_i = covariance_for_inference(headline_plain_res, df_final, inference_label)
    row_i = extract_term_row_from_cov(
        headline_plain_res,
        cov_i,
        specification=HEADLINE_SPEC,
        estimator="market_fe_ols",
        sample_label="full_sample",
        inference_label=inference_label,
        dof=dof_i,
        cluster_info=cluster_info_i,
    )
    cluster_sensitivity_rows.append(row_i)
cluster_sensitivity_table = pd.DataFrame(cluster_sensitivity_rows)
display(cluster_sensitivity_table)

# Small-sample cluster correction and wild-cluster bootstrap inference.
# The two-way studentized WCB remains the primary bootstrap check. One-way seller and
# market bootstrap rows are exported as sensitivity checks because the static two-way
# target variance can be numerically fragile in some bootstrap draws.
cr2_correction_table = cr2_one_way_cluster_table(df_final, headline_rhs, cluster_var="seller_id")
display(cr2_correction_table)

wild_cluster_bootstrap_table = wild_cluster_bootstrap_twoway_fast(df_final, headline_rhs)
if RUN_STATIC_BOOTSTRAP_SENSITIVITY:
    wild_cluster_bootstrap_sensitivity_table = pd.concat(
        [
            wild_cluster_bootstrap_table.assign(bootstrap_role="primary_two_way_seller_market"),
            wild_cluster_bootstrap_fast(df_final, headline_rhs, cluster_var="seller_id", seed=WILD_BOOTSTRAP_SEED + 101).assign(bootstrap_role="one_way_seller_sensitivity"),
            wild_cluster_bootstrap_fast(df_final, headline_rhs, cluster_var="market_id", seed=WILD_BOOTSTRAP_SEED + 202).assign(bootstrap_role="one_way_market_sensitivity"),
        ],
        ignore_index=True,
        sort=False,
    )
else:
    wild_cluster_bootstrap_sensitivity_table = wild_cluster_bootstrap_table.assign(
        bootstrap_role="primary_two_way_seller_market",
        sensitivity_note="One-way bootstrap sensitivity is disabled by default in the thesis notebook; enable RUN_STATIC_BOOTSTRAP_SENSITIVITY for an optional rerun.",
    )
display(wild_cluster_bootstrap_sensitivity_table)

,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,markets
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,61,5107,0.959381,119,62,62
1,drop_review_problem,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.053826,0.016113,-3.340599,0.001431,-0.086046,-0.021607,***,61,5015,0.959199,117,62,62
2,drop_contact_courier,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051031,0.015969,-3.195543,0.002211,-0.082964,-0.019098,***,61,5057,0.959297,119,62,62
3,drop_delivery_failure_markets,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.048727,0.015491,-3.145562,0.002654,-0.079758,-0.017695,***,56,4654,0.960511,119,57,57


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,markets
0,estimation_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.044176,0.014616,-3.022351,0.004016,-0.073565,-0.014788,***,48,3931,0.963436,113,49,49
1,holdout_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.064433,0.019088,-3.375506,0.005514,-0.106023,-0.022843,***,12,1176,0.958865,97,13,13


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters
0,full_sample,rank_pct,mundlak_correlated_effects,pooled_ols_with_seller_means_and_market_fe,two_way_seller_market,fba_from_shipper,-0.049724,0.016431,-3.026191,0.003624,-0.08258,-0.016868,***,61,5107,0.959853,119,62


,leave_one_dimension,iterations,baseline_coef,min_coef,p10_coef,median_coef,p90_coef,max_coef,same_sign_share,max_absolute_change,method
0,market,62,-0.051038,-0.051417,-0.051340,-0.051134,-0.050632,-0.050387,1.0,0.000651,exact deleted-cluster OLS using within-market transformation recomputed after deletion
1,seller,119,-0.051038,-0.058103,-0.052579,-0.051031,-0.049511,-0.042324,1.0,0.008714,exact deleted-cluster OLS using within-market transformation recomputed after deletion


,leave_one_dimension,dropped_cluster,dropped_observations,dropped_fba_observations,dropped_nonfba_observations,retained_observations,baseline_coef,deleted_cluster_coef,absolute_change,sign_flips_relative_to_baseline,method
0,market,2022-03-04T18:01:41,90,18,72,5017,-0.051038,-0.050387,0.000651,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
1,market,2022-03-05T12:01:52,90,18,72,5017,-0.051038,-0.050535,0.000503,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
2,market,2022-03-05T18:01:56,90,18,72,5017,-0.051038,-0.050555,0.000483,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
3,market,2022-03-06T12:01:35,91,19,72,5016,-0.051038,-0.050569,0.000469,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
4,market,2022-03-06T18:01:33,91,19,72,5016,-0.051038,-0.050570,0.000468,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
5,market,2022-03-08T12:01:55,90,17,73,5017,-0.051038,-0.050614,0.000425,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
6,market,2022-03-04T12:01:39,88,18,70,5019,-0.051038,-0.050623,0.000415,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
7,market,2022-02-13T12:12:47,72,14,58,5035,-0.051038,-0.051417,0.000379,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
8,market,2022-02-17T18:08:52,80,18,62,5027,-0.051038,-0.051403,0.000365,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion
9,market,2022-02-13T18:02:08,73,15,58,5034,-0.051038,-0.051400,0.000362,False,exact deleted-cluster OLS using within-market transformation recomputed after deletion


...

,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,rows_dropped,cook_cutoff,diagnostic_method
0,influence_trimmed,rank_pct,drop_top_1pct_cooks_distance,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.040756,0.013131,-3.103787,0.002895,-0.067014,-0.014499,***,61,5055,0.97109,116,62,52,0.00192,fast Cook's-distance approximation from hat-matrix diagonal


,method,coefficient,t_statistic,dof_reference,partial_r2_fba_with_rank_given_controls,robustness_value_to_reduce_estimate_to_zero,interpretation
0,Cinelli-Hazlett-style OLS sensitivity diagnostic,-0.051038,-3.196231,61,0.14345,0.333978,Approximate diagnostic only; it does not convert the associational design into a causal design.


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,61,5107,0.959381,119,62
1,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,seller_cluster,fba_from_shipper,-0.051038,0.016062,-3.177595,0.001896,-0.082845,-0.019231,***,118,5107,0.959381,119,62
2,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,market_cluster,fba_from_shipper,-0.051038,0.002092,-24.400062,0.000000,-0.055221,-0.046855,***,61,5107,0.959381,119,62


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,clusters,r_squared,note
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,cr2_seller_id,fba_from_shipper,-0.051038,0.016748,-3.047441,0.006922,-0.08622,-0.015856,***,18.026985,5107,119,0.959381,One-way CR2-style small-sample correction; reported as a conservative supplement to two-way clustered headline inference.


,sample,outcome,specification,estimator,inference,seller_clusters,market_clusters,requested_replications,valid_replications,invalid_replications,negative_two_way_variance_replications,negative_two_way_variance_share,variance_floor_applied_replications,variance_floor_applied_share,variance_floor_rule,bootstrap_implementation,bootstrap_extreme_replications,valid_replication_share,invalid_replication_share,bootstrap_pvalue_monte_carlo_se,valid_replication_warning,seed,coef_fba,two_way_cluster_se_fba,t_observed,bootstrap_pvalue_fba,stars,null,studentized,recomputed_se_each_replication,bootstrap_design,bootstrap_role,sensitivity_note
0,full_sample,rank_pct,spec_4_fba_reputation_price_shipping_delivery,market_fe_ols,wild_cluster_bootstrap_two_way_seller_market_studentized,119,62,4999,4999,0,1006,0.20124,1006,0.20124,"when the finite-sample corrected two-way target variance is non-positive, use max(seller one-way target variance, market one-way target variance)",numba_parallel_static_two_way_wcb,65,1.0,0.0,0.001614,False,20260807,-0.051038,0.015968,-3.196231,0.0132,**,fba_from_shipper coefficient equals zero,True,True,restricted residual wild cluster bootstrap with product Rademacher weights over seller and market clusters; full-model coefficient and target two-way cluster SE recomputed in e...,primary_two_way_seller_market,One-way bootstrap sensitivity is disabled by default in the thesis notebook; enable RUN_STATIC_BOOTSTRAP_SENSITIVITY for an optional rerun.


### 13. Propensity overlap and common-support diagnostics

Common-support restrictions are treated as a comparison-design question rather than as a mechanical robustness exercise. The cell estimates propensity-score models for the FBA indicator under several control configurations: total-price linear-stars, total-price categorical-stars, split-price linear-stars, split-price categorical-stars, total-price linear-stars without positive-review percentage. For each configuration the cell reports the overlap region, the trimming threshold, the trimmed sample size, and the balance of the trimmed sample. The headline overlap design is `total_price_linear_stars`. The exported tables are `propensity_overlap_summary_rankpct.csv`, `propensity_overlap_quantiles_rankpct.csv`, `propensity_overlap_interpretation_rankpct.csv`, and `overlap_design_decision_rankpct.csv`; they populate `tab:app-ch6-propensity-overlap`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 13. Propensity overlap and common-support diagnostics
# -----------------------------------------------------------------------------
# Estimate propensity-score models under several control configurations, compute the overlap region, trim the sample, and check the balance of the trimmed sample.

overlap_summary_frames = []
overlap_quantile_frames = []
overlap_estimate_frames = []
overlap_balance_detail_frames = []
overlap_balance_summary_frames = []
propensity_score_store = {}

for model_label, rhs in PRICE_STAR_SPECIFICATIONS.items():
    meta = PRICE_STAR_METADATA[model_label]
    summary, quantiles, pscore = propensity_overlap_diagnostics(
        df_final,
        price_measure=meta["price_measure"],
        star_form=meta["star_form"],
        positive_review_form=meta["positive_review_form"],
        model_label=model_label,
    )
    propensity_score_store[model_label] = pscore
    overlap_summary_frames.append(summary)
    overlap_quantile_frames.append(quantiles)
    overlap_estimate_frames.append(
        overlap_trimmed_estimates(
            df_final,
            rhs,
            pscore,
            price_measure=meta["price_measure"],
            star_form=meta["star_form"],
            positive_review_form=meta["positive_review_form"],
            model_label=model_label,
        )
    )
    balance_detail, balance_summary = overlap_balance_diagnostics(
        df_final,
        pscore,
        model_label=model_label,
        price_measure=meta["price_measure"],
        star_form=meta["star_form"],
        positive_review_form=meta["positive_review_form"],
    )
    overlap_balance_detail_frames.append(balance_detail)
    overlap_balance_summary_frames.append(balance_summary)

propensity_overlap_summary_table = pd.concat(overlap_summary_frames, ignore_index=True)
propensity_overlap_quantiles_table = pd.concat(overlap_quantile_frames, ignore_index=True)
overlap_trimmed_estimates_table = pd.concat(overlap_estimate_frames, ignore_index=True)
overlap_balance_tests_table = pd.concat(overlap_balance_detail_frames, ignore_index=True)
overlap_balance_summary_table = pd.concat(overlap_balance_summary_frames, ignore_index=True)

propensity_overlap_interpretation_table = pd.DataFrame(
    [
        {
            "object": "ratio_preservation",
            "interpretation": "The overlap diagnostics do not force the original FBA/non-FBA ratio. Common-support restrictions intentionally change the comparison population.",
        },
        {
            "object": "split_price_overlap",
            "interpretation": "The split-price propensity model uses product price and shipping price separately, mirroring the headline OLS specification.",
        },
        {
            "object": "total_price_overlap",
            "interpretation": "The total-price propensity model replaces product and shipping price with reconstructed total price, checking whether the overlap conclusion depends on decomposed price representation.",
        },
        {
            "object": "categorical_star_overlap",
            "interpretation": "The categorical-star variants replace the linear stelle term with star-rating indicators, using 4.5 stars as the reference category.",
        },
        {
            "object": "positive_review_exclusion",
            "interpretation": "Each price/star specification is also estimated after dropping valutazioni_positive, because positive-review percentage is highly correlated with stelle and may make the reputation controls unnecessarily collinear.",
        },
        {
            "object": "overlap_balance_tests",
            "interpretation": "For every overlap design, FBA and non-FBA covariate balance is tested again inside the full sample and overlap-restricted samples. SMDs are the preferred balance metric; p-values are descriptive.",
        },
    ]
)

display(propensity_overlap_summary_table)
display(propensity_overlap_quantiles_table)
display(overlap_trimmed_estimates_table)
display(overlap_balance_summary_table)
display(overlap_balance_tests_table)
display(propensity_overlap_interpretation_table)

,model_label,price_measure,star_form,positive_review_form,method,propensity_formula,nobs,treated_rows,control_rows,treated_ps_min,treated_ps_max,control_ps_min,control_ps_max,common_support_low,common_support_high,common_support_rows,common_support_share,trim_0_05_0_95_rows,trim_0_05_0_95_share,trim_0_10_0_90_rows,trim_0_10_0_90_share,note
0,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,logit propensity score diagnostic with market fixed effects,fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + stelle + prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_cons_min_robust + C(market_id),5107,996,4111,0.141614,0.990978,6.305117e-16,0.987565,0.141614,0.987565,1534,0.300372,1833,0.358919,1490,0.291756,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
1,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,logit propensity score diagnostic with market fixed effects,fba_from_shipper ~ log1p_num_valutazioni + stelle + prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_cons_min_robust + C(market_id),5107,996,4111,0.038704,0.979513,6.305117e-16,0.974237,0.038704,0.974237,2208,0.432348,1979,0.387507,1616,0.316428,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
2,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,logit propensity score diagnostic with market fixed effects,"fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + C(stelle_cat, Treatment(reference=""4.5"")) + prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_con...",5107,996,4111,0.138798,0.992735,6.305117e-16,0.989524,0.138798,0.989524,1547,0.302918,1828,0.357940,1503,0.294302,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
3,split_price_categorical_stars_no_positive_reviews,split_price,categorical_stars,positive_reviews_dropped,logit propensity score diagnostic with market fixed effects,"fba_from_shipper ~ log1p_num_valutazioni + C(stelle_cat, Treatment(reference=""4.5"")) + prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_cons_min_robust + C(market...",5107,996,4111,0.068112,0.979221,6.305117e-16,0.973673,0.068112,0.973673,1980,0.387703,1877,0.367535,1593,0.311925,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
4,total_price_linear_stars,total_price,linear_stars,positive_reviews_included,logit propensity score diagnostic with market fixed effects,fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + stelle + prezzo_totale_reconstructed + contact_courier_flag + g_cons_min_robust + C(market_id),5107,996,4111,0.055603,0.981764,6.305117e-16,0.974532,0.055603,0.974532,2205,0.431760,2129,0.416879,1564,0.306246,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
5,total_price_linear_stars_no_positive_reviews,total_price,linear_stars,positive_reviews_dropped,logit propensity score diagnostic with market fixed effects,fba_from_shipper ~ log1p_num_valutazioni + stelle + prezzo_totale_reconstructed + contact_courier_flag + g_cons_min_robust + C(market_id),5107,996,4111,0.014873,0.978738,6.305117e-16,0.966245,0.014873,0.966245,2948,0.577247,2265,0.443509,1784,0.349324,Diagnostic only; overlap restrictions change the comparison population and do not preserve the original treatment ratio by design.
6,total_price_categorical_stars,total_price,categorical_stars,positive_reviews_included,logit propensity score diagnostic with market fixed effects,"fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + C(stelle_cat, Treatment(reference=""4.5"")) + prezzo_totale_reconstructed + contact_co

,model_label,price_measure,star_form,positive_review_form,fba_from_shipper,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99
0,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,0,6.305117e-16,6.305117e-16,6.305117e-16,6.305117e-16,0.003130,0.058213,0.258903,0.383259,0.880707
1,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,1,1.852774e-01,2.069960e-01,2.413568e-01,5.101253e-01,0.805395,0.899667,0.980899,0.986511,0.989116
2,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,0,6.305117e-16,6.305117e-16,6.305117e-16,6.305117e-16,0.003752,0.081878,0.275194,0.440187,0.724566
3,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,1,6.676929e-02,7.747094e-02,2.788291e-01,4.599740e-01,0.737109,0.881927,0.964241,0.973791,0.978484
4,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,0,6.305117e-16,6.305117e-16,6.305117e-16,6.305117e-16,0.000254,0.053904,0.252894,0.394626,0.876296
5,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,1,1.890209e-01,2.146746e-01,2.480523e-01,5.046974e-01,0.804045,0.897637,0.983163,0.987997,0.990550
6,split_price_categorical_stars_no_positive_reviews,split_price,categorical_stars,positive_reviews_dropped,0,6.305117e-16,6.305117e-16,6.305117e-16,6.305117e-16,0.000112,0.068279,0.297722,0.455509,0.774587
7,split_price_categorical_stars_no_positive_reviews,split_price,categorical_stars,positive_reviews_dropped,1,7.866723e-02,1.145084e-01,2.400139e-01,4.747681e-01,0.752105,0.898634,0.964183,0.972433,0.978465
8,total_price_linear_stars,total_price,linear_stars,positive_reviews_included,0,6.305117e-16,1.159471e-05,2.395074e-04,1.077910e-03,0.012915,0.077586,0.270846,0.422414,0.838852
9,total_price_linear_stars,total_price,linear_stars,positive_reviews_included,1,6.088078e-02,7.237006e-02,1.854025e-01,4.375931e-01,0.738214,0.860550,0.964799,0.976458,0.980556


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,model_label,price_measure,star_form,positive_review_form,comparison_population,rows,markets,fba_rows,nonfba_rows,fba_share
0,full_sample,rank_pct,split_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,61,5107,0.959381,119,62,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,5107,62,996,4111,0.195026
1,propensity_common_support,rank_pct,split_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,0.000415,0.004960,0.083666,0.933693,-0.009575,0.010405,,45,1534,0.986156,46,62,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_common_support,1534,62,958,576,0.624511
2,propensity_0_05_0_95,rank_pct,split_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.004875,0.004679,-1.041871,0.302114,-0.014255,0.004506,,54,1833,0.986358,55,62,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_0_05_0_95,1833,62,796,1037,0.434261
3,propensity_0_10_0_90,rank_pct,split_price_linear_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.003879,0.004729,-0.820366,0.416332,-0.013404,0.005645,,45,1490,0.988395,46,62,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_0_10_0_90,1490,62,747,743,0.501342
4,full_sample,rank_pct,split_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.049066,0.014419,-3.402739,0.001183,-0.077899,-0.020232,***,61,5107,0.959292,119,62,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,full_sample,5107,62,996,4111,0.195026
5,propensity_common_support,rank_pct,split_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,0.001366,0.005611,0.243415,0.808542,-0.009865,0.012596,,58,2208,0.986704,59,62,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_common_support,2208,62,948,1260,0.429348
6,propensity_0_05_0_95,rank_pct,split_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.001943,0.004845,-0.401095,0.689933,-0.011658,0.007771,,54,1979,0.987245,55,62,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_0_05_0_95,1979,62,824,1155,0.416372
7,propensity_0_10_0_90,rank_pct,split_price_linear_stars_no_positive_reviews,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.003143,0.005737,-0.547801,0.586316,-0.014671,0.008386,,49,1616,0.981577,50,62,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_0_10_0_90,1616,62,692,924,0.428218
8,full_sample,rank_pct,split_price_categorical_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.048094,0.016095,-2.988123,0.004040,-0.080278,-0.015910,***,61,5107,0.959973,119,62,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,full_sample,5107,62,996,4111,0.195026
9,propensity_common_support,rank_pct,split_price_categorical_stars,market_fe_ols,two_way_seller_market,fba_from_shipper,0.001640,0.005026,0.326353,0.745670,-0.008482,0.011762,,45,1547,0.985547,46,62,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,propensity_common_support,1547,62,968,579,0.625727


,model_label,price_measure,star_form,positive_review_form,comparison_population,rows,fba_rows,nonfba_rows,fba_share,mean_abs_smd,median_abs_smd,max_abs_smd,variables_abs_smd_above_0_10,variables_abs_smd_above_0_25,balance_note
0,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
1,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_0_05_0_95,1833,796,1037,0.434261,0.438307,0.480978,0.676445,5,5,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
2,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_0_10_0_90,1490,747,743,0.501342,0.322356,0.335480,0.569298,5,3,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
3,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,propensity_common_support,1534,958,576,0.624511,0.294412,0.291711,0.663289,4,3,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
4,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
5,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_0_05_0_95,1979,824,1155,0.416372,0.470924,0.578727,0.616450,5,5,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
6,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_0_10_0_90,1616,692,924,0.428218,0.492702,0.570894,0.712210,5,5,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
7,split_price_linear_stars_no_positive_reviews,split_price,linear_stars,positive_reviews_dropped,propensity_common_support,2208,948,1260,0.429348,0.537620,0.658857,0.685197,5,5,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
8,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.
9,split_price_categorical_stars,split_price,categorical_stars,positive_reviews_included,propensity_0_05_0_95,1828,800,1028,0.437637,0.473203,0.489692,0.749802,6,5,Lower absolute SMDs indicate better observed balance. P-values are secondary because overlap samples remain moderately large.


,model_label,price_measure,star_form,positive_review_form,comparison_population,variable,variable_used_in_propensity,rows,fba_rows,nonfba_rows,fba_share,mean_fba,mean_nonfba,mean_diff_fba_minus_nonfba,median_fba,median_nonfba,median_diff_fba_minus_nonfba,smd_fba_minus_nonfba,abs_smd,welch_t_stat,welch_pvalue,mann_whitney_u_stat,mann_whitney_pvalue,ks_statistic,ks_pvalue,markets_with_both_groups,market_mean_diff_fba_minus_nonfba,market_paired_t_stat,market_paired_t_pvalue,market_wilcoxon_stat,market_wilcoxon_pvalue,test_note,welch_pvalue_fdr_bh,mann_whitney_pvalue_fdr_bh,ks_pvalue_fdr_bh,market_paired_t_pvalue_fdr_bh,market_wilcoxon_pvalue_fdr_bh
0,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,prezzo_totale_reconstructed,False,5107,996,4111,0.195026,45.103484,55.945018,-10.841534,44.990000,55.390000,-10.400000,-1.552516,1.552516,-53.347139,0.000000e+00,521064.0,1.219918e-292,0.681656,0.000000e+00,62,-10.748882,-110.845109,4.653584e-72,0.0,7.577765e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,0.000000e+00,9.759343e-292,0.000000e+00,1.861434e-71,1.010369e-11
1,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,prezzo,True,5107,996,4111,0.195026,45.103484,52.334829,-7.231345,44.990000,50.700000,-5.710000,-1.101420,1.101420,-36.831656,2.419816e-237,856083.0,4.113099e-179,0.545879,1.215790e-224,62,-7.158257,-94.844174,5.947098e-68,0.0,7.575538e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,4.839632e-237,1.645240e-178,4.863159e-224,9.515357e-68,1.010369e-11
2,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,prezzo_spedizione_repaired,True,5107,996,4111,0.195026,0.000000,3.610190,-3.610190,0.000000,0.000000,0.000000,-0.963208,0.963208,-43.669554,0.000000e+00,1218606.0,1.468301e-125,0.404768,3.281528e-119,62,-3.590626,-106.483891,5.318578e-71,0.0,7.537774e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,0.000000e+00,2.349281e-125,6.563057e-119,1.063716e-70,1.010369e-11
3,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,log1p_num_valutazioni,True,5107,996,4111,0.195026,4.191823,5.545441,-1.353618,4.317488,5.488938,-1.171450,-0.630378,0.630378,-18.704592,3.710816e-71,1320508.5,7.023828e-68,0.338797,1.226297e-82,62,-1.362655,-52.387375,1.998295e-52,0.0,7.577765e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,4.947755e-71,9.365104e-68,1.635062e-82,2.664393e-52,1.010369e-11
4,split_price_linear_stars,split_price,linear_stars,positive_reviews_included,full_sample,valutazioni_positive,True,5107,996,4111,0.195026,94.993976,82.391146,12.602830,97.000000,87.000000,10.000000,0.946977,0.946977,38.255804,1.479154e-280,3208191.5,1.346735e-170,0.439052,3.068953e-141,62,12.567763,112.086566,2.366724e-72,0.0,7.576652e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,3.944412e-280,3.591293e-170,8.183874e-141,1.861434e-71,1.010369e-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,total_price_categorical_stars_no_positive_reviews,total_price,categorical_stars,positive_reviews_dropped,propensity_0_10_0_90,log1p_num_valutazioni,True,1807,693,1114,0.383509,4.928094,5.669873,-0.741780,4.624973,5.484797,-0.859824,-0.382931,0.382931,-8.294909,2.104926e-16,320988.0,1.652245e-09,0.265464,4.723150e-27,62,-0.745018,-26.206394,6.728340e-35,0.0,7.577765e-12,Overlap balance diagnostics are descriptive. SMDs are emphasized because p-values can remain small in large samples.,2.946896e-16,2.202994e-09,9.446301e-27,9.419676e-35,8.840726e-12
252,total_

,object,interpretation
0,ratio_preservation,The overlap diagnostics do not force the original FBA/non-FBA ratio. Common-support restrictions intentionally change the comparison population.
1,split_price_overlap,"The split-price propensity model uses product price and shipping price separately, mirroring the headline OLS specification."
2,total_price_overlap,"The total-price propensity model replaces product and shipping price with reconstructed total price, checking whether the overlap conclusion depends on decomposed price represe..."
3,categorical_star_overlap,"The categorical-star variants replace the linear stelle term with star-rating indicators, using 4.5 stars as the reference category."
4,positive_review_exclusion,"Each price/star specification is also estimated after dropping valutazioni_positive, because positive-review percentage is highly correlated with stelle and may make the reputa..."
5,overlap_balance_tests,"For every overlap design, FBA and non-FBA covariate balance is tested again inside the full sample and overlap-restricted samples. SMDs are the preferred balance metric; p-valu..."


### 14. Balance inside the common-support sample

The common-support sample is evaluated as a comparison design. The cell tests whether the trimmed sample is balanced on the main commercial covariates (price, shipping, delivery, reputation, review count) and on the auxiliary covariates (positive-review percentage, star rating, fast-delivery availability). The diagnostic shows that the trimmed subsample remains imbalanced on the main commercial covariates: the surviving non-FBA rows are disproportionately drawn from the top rank quartile (selection ratio 2.6). The trimmed sample is therefore not a balanced like-for-like FBA/non-FBA comparison. The outcome `rank_pct` is reported on the trimmed sample as an outcome-gap diagnostic, not as a balanced ATE. The output is exported to `common_support_balance_total_price_rankpct.csv` and corresponds to `tab:app-ch6-common-support-balance`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 14. Balance inside the common-support sample
# -----------------------------------------------------------------------------
# Test the balance of the trimmed common-support sample on the main commercial covariates and compute the rank-quartile selection ratios into the trimmed set.

_cs_propensity_design = {
    "price_measure":        "total_price",
    "star_form":            "linear_stars",
    "positive_review_form": "positive_reviews_included",
}

_cs_formula = propensity_formula(
    price_measure=_cs_propensity_design["price_measure"],
    star_form=_cs_propensity_design["star_form"],
    positive_review_form=_cs_propensity_design["positive_review_form"],
)
_cs_pscore_raw, _, _ = fit_logit_propensity_fast(df_final, _cs_formula)

_cs_logit_pscore = pd.Series(
    np.asarray(_cs_pscore_raw),
    index=_cs_pscore_raw.index,
    name="propensity_score_total_price_linear_stars",
).reindex(df_final.index)

# Common-support range and mask
_treated_mask = df_final["fba_from_shipper"].eq(1)
_pscore_treated = _cs_logit_pscore[_treated_mask]
_pscore_control = _cs_logit_pscore[~_treated_mask]
_cs_lo = max(float(_pscore_treated.min()), float(_pscore_control.min()))
_cs_hi = min(float(_pscore_treated.max()), float(_pscore_control.max()))
_cs_mask = _cs_logit_pscore.between(_cs_lo, _cs_hi)

cs_sample = df_final.loc[_cs_mask].copy()
cs_sample["_pscore"] = _cs_logit_pscore.loc[_cs_mask].values

print(f"Common-support sample (total-price design): {len(cs_sample):,} rows out of {len(df_final):,}")
print(f"  - FBA rows in CS:    {int((cs_sample['fba_from_shipper'] == 1).sum()):,}")
print(f"  - non-FBA rows in CS: {int((cs_sample['fba_from_shipper'] == 0).sum()):,}")

# --- Balance computations -------------------------------------------------

def _smd(x_treat, x_control):
    """Standardized mean difference (treat - control) / pooled SD."""
    m_t = np.nanmean(x_treat); m_c = np.nanmean(x_control)
    v_t = np.nanvar(x_treat, ddof=1); v_c = np.nanvar(x_control, ddof=1)
    pooled_sd = np.sqrt(0.5 * (v_t + v_c))
    if pooled_sd == 0 or not np.isfinite(pooled_sd):
        return np.nan
    return (m_t - m_c) / pooled_sd

def _within_market_paired_pvalue(data, var):
    """Paired t-test on within-market FBA mean minus non-FBA mean."""
    diffs = []
    for mid, grp in data.groupby("market_id", observed=True):
        a = grp.loc[grp["fba_from_shipper"] == 1, var].dropna()
        b = grp.loc[grp["fba_from_shipper"] == 0, var].dropna()
        if len(a) == 0 or len(b) == 0:
            continue
        diffs.append(a.mean() - b.mean())
    diffs = np.asarray(diffs, dtype=float)
    if len(diffs) < 2 or diffs.std(ddof=1) == 0:
        return np.nan
    t_stat = diffs.mean() / (diffs.std(ddof=1) / np.sqrt(len(diffs)))
    return float(2 * (1 - stats.t.cdf(abs(t_stat), df=len(diffs) - 1)))

_balance_vars = [
    "rank_pct", "prezzo", "log1p_num_valutazioni",
    "stelle", "valutazioni_positive", "g_cons_min_robust",
]

_cs_balance_rows = []
for var in _balance_vars:
    if var not in cs_sample.columns:
        continue
    a = cs_sample.loc[cs_sample["fba_from_shipper"] == 1, var].dropna()
    b = cs_sample.loc[cs_sample["fba_from_shipper"] == 0, var].dropna()
    smd = _smd(a.values, b.values)
    abs_smd = abs(smd) if pd.notna(smd) else np.nan
    if pd.isna(abs_smd):
        flag = "n/a"
    elif abs_smd <= 0.10:
        flag = "good (|SMD|<=0.10)"
    elif abs_smd <= 0.25:
        flag = "mild (0.10<|SMD|<=0.25)"
    else:
        flag = "material (|SMD|>0.25)"
    _cs_balance_rows.append({
        "variable":             var,
        "fba_mean":             a.mean(),
        "nonfba_mean":          b.mean(),
        "mean_difference":      a.mean() - b.mean(),
        "smd":                  smd,
        "within_market_pvalue": _within_market_paired_pvalue(cs_sample, var),
        "balance_flag":         flag,
    })

cs_balance_table = pd.DataFrame(_cs_balance_rows)
print()
print("Table D3 - Balance inside total-price common-support sample (FBA vs non-FBA)")
display(cs_balance_table)

Common-support sample (total-price design): 2,205 rows out of 5,107
  - FBA rows in CS:    934
  - non-FBA rows in CS: 1,271

Table D3 - Balance inside total-price common-support sample (FBA vs non-FBA)


,variable,fba_mean,nonfba_mean,mean_difference,smd,within_market_pvalue,balance_flag
0,rank_pct,0.200773,0.323463,-0.122690,-0.692601,0.000000,material (|SMD|>0.25)
1,prezzo,45.274165,46.653926,-1.379761,-0.276567,0.000000,material (|SMD|>0.25)
2,log1p_num_valutazioni,4.381407,5.563282,-1.181875,-0.583618,0.000000,material (|SMD|>0.25)
3,stelle,4.700749,4.562943,0.137807,0.410524,0.000000,material (|SMD|>0.25)
4,valutazioni_positive,94.760171,91.602675,3.157496,0.432579,0.000000,material (|SMD|>0.25)
5,g_cons_min_robust,6.646681,6.324941,0.321740,0.082670,0.109751,good (|SMD|<=0.10)


### 15. Common-support adequacy criterion

Every overlap population is classified before its coefficient is interpreted. A restricted sample is classified as a credible like-for-like comparison only when the trimmed FBA and non-FBA subsamples are balanced on the main commercial covariates and the selection ratios into the trimmed set are close to one. The cell computes the standardized mean differences on the key covariates, the rank-quartile selection ratios, and the explicit adequacy label for each overlap design. The table is exported to `overlap_design_status_summary_rankpct.csv` and to `credible_like_for_like_overlap`. None of the available overlap designs satisfies the adequacy criterion, which is the basis for the validity-boundary statement in Chapter 6 of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 15. Common-support adequacy criterion
# -----------------------------------------------------------------------------
# Classify each overlap design as credible like-for-like comparison or as a re-selected subpopulation, based on covariate balance and selection ratios.

OVERLAP_KEY_COVARIATES = [
    "prezzo",
    "prezzo_spedizione_repaired",
    "prezzo_totale_reconstructed",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
    "g_cons_min_robust",
]
OVERLAP_KEY_COVARIATES = [c for c in OVERLAP_KEY_COVARIATES if c in df_final.columns]


def classify_overlap_design(
    balance_summary,
    balance_tests,
    min_group_rows=250,
    max_mean_abs_smd=0.10,
    max_median_abs_smd=0.10,
    max_abs_smd_allowed=0.25,
    max_key_covariates_above_025=0,
):
    """Classify whether an overlap sample is credible as a like-for-like comparison.

    Failure does not invalidate the full-sample descriptive estimate. It means
    that the restricted overlap sample should be interpreted as a support
    diagnostic, not as a successful robustness check.
    """
    rows = []
    for _, s in balance_summary.iterrows():
        model_label = s["model_label"]
        pop = s["comparison_population"]
        bt = balance_tests[
            balance_tests["model_label"].eq(model_label)
            & balance_tests["comparison_population"].eq(pop)
            & balance_tests["variable"].isin(OVERLAP_KEY_COVARIATES)
        ].copy()
        key_above_025 = int(bt["abs_smd"].gt(max_abs_smd_allowed).sum()) if len(bt) else np.nan
        key_max_smd = float(bt["abs_smd"].max()) if len(bt) else np.nan
        enough_rows = bool(
            int(s["fba_rows"]) >= min_group_rows
            and int(s["nonfba_rows"]) >= min_group_rows
        )
        balance_pass = bool(
            enough_rows
            and float(s["mean_abs_smd"]) <= max_mean_abs_smd
            and float(s["median_abs_smd"]) <= max_median_abs_smd
            and pd.notna(key_max_smd)
            and key_max_smd <= max_abs_smd_allowed
            and key_above_025 <= max_key_covariates_above_025
        )
        if pop == "full_sample":
            status = "baseline_not_an_overlap_design"
        elif balance_pass:
            status = "credible_like_for_like_overlap"
        else:
            status = "failed_overlap_diagnostic_not_robustness"
        rows.append({
            "model_label": model_label,
            "comparison_population": pop,
            "rows": int(s["rows"]),
            "fba_rows": int(s["fba_rows"]),
            "nonfba_rows": int(s["nonfba_rows"]),
            "fba_share": float(s["fba_share"]),
            "mean_abs_smd": float(s["mean_abs_smd"]),
            "median_abs_smd": float(s["median_abs_smd"]),
            "max_abs_smd": float(s["max_abs_smd"]),
            "variables_abs_smd_above_0_10": int(s.get("variables_abs_smd_above_0_10", -1)),
            "variables_abs_smd_above_0_25": int(s.get("variables_abs_smd_above_0_25", -1)),
            "key_covariates_abs_smd_above_025": key_above_025,
            "key_covariates_max_abs_smd": key_max_smd,
            "enough_rows_each_group": enough_rows,
            "overlap_status": status,
            "interpretation_rule": (
                "promote as like-for-like overlap" if status == "credible_like_for_like_overlap"
                else "diagnostic only; do not treat coefficient as robustness evidence"
            ),
        })
    return pd.DataFrame(rows)


overlap_design_decision_table = classify_overlap_design(
    overlap_balance_summary_table,
    overlap_balance_tests_table,
)

overlap_design_status_summary_table = (
    overlap_design_decision_table
    .groupby("overlap_status", observed=True)
    .agg(
        designs=("model_label", "count"),
        min_nonfba_rows=("nonfba_rows", "min"),
        median_mean_abs_smd=("mean_abs_smd", "median"),
        max_key_covariate_abs_smd=("key_covariates_max_abs_smd", "max"),
    )
    .reset_index()
)

if EXPORT_FILES:
    overlap_design_decision_table.to_csv(OUTPUT_DIR / "overlap_design_decision_rankpct.csv", index=False)
    overlap_design_status_summary_table.to_csv(OUTPUT_DIR / "overlap_design_status_summary_rankpct.csv", index=False)

print("Table D2b - Common-support adequacy criterion")
display(overlap_design_decision_table)
print("Overlap-design status summary")
display(overlap_design_status_summary_table)

Table D2b - Common-support adequacy criterion


,model_label,comparison_population,rows,fba_rows,nonfba_rows,fba_share,mean_abs_smd,median_abs_smd,max_abs_smd,variables_abs_smd_above_0_10,variables_abs_smd_above_0_25,key_covariates_abs_smd_above_025,key_covariates_max_abs_smd,enough_rows_each_group,overlap_status,interpretation_rule
0,split_price_linear_stars,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,6,1.552516,True,baseline_not_an_overlap_design,diagnostic only; do not treat coefficient as robustness evidence
1,split_price_linear_stars,propensity_0_05_0_95,1833,796,1037,0.434261,0.438307,0.480978,0.676445,5,5,5,0.676445,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
2,split_price_linear_stars,propensity_0_10_0_90,1490,747,743,0.501342,0.322356,0.335480,0.569298,5,3,3,0.569298,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
3,split_price_linear_stars,propensity_common_support,1534,958,576,0.624511,0.294412,0.291711,0.663289,4,3,3,0.663289,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
4,split_price_linear_stars_no_positive_reviews,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,6,1.552516,True,baseline_not_an_overlap_design,diagnostic only; do not treat coefficient as robustness evidence
5,split_price_linear_stars_no_positive_reviews,propensity_0_05_0_95,1979,824,1155,0.416372,0.470924,0.578727,0.616450,5,5,5,0.616450,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
6,split_price_linear_stars_no_positive_reviews,propensity_0_10_0_90,1616,692,924,0.428218,0.492702,0.570894,0.712210,5,5,5,0.712210,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
7,split_price_linear_stars_no_positive_reviews,propensity_common_support,2208,948,1260,0.429348,0.537620,0.658857,0.685197,5,5,5,0.685197,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence
8,split_price_categorical_stars,full_sample,5107,996,4111,0.195026,0.777765,0.871026,1.552516,7,6,6,1.552516,True,baseline_not_an_overlap_design,diagnostic only; do not treat coefficient as robustness evidence
9,split_price_categorical_stars,propensity_0_05_0_95,1828,800,1028,0.437637,0.473203,0.489692,0.749802,6,5,5,0.749802,True,failed_overlap_diagnostic_not_robustness,diagnostic only; do not treat coefficient as robustness evidence


Overlap-design status summary


,overlap_status,designs,min_nonfba_rows,median_mean_abs_smd,max_key_covariate_abs_smd
0,baseline_not_an_overlap_design,8,4111,0.777765,1.552516
1,failed_overlap_diagnostic_not_robustness,24,576,0.473949,1.120339


### 16. Non-FBA survivor selection under common support

The cell studies which non-FBA offers survive the common-support restriction. If the retained non-FBA group is concentrated in unusual within-market rank positions, the trimmed sample does not represent the original FBA-vs-non-FBA comparison but a re-selected subpopulation. The diagnostic computes the rank-quartile distribution of surviving non-FBA offers and the price-quartile distribution of surviving non-FBA offers, both before and after the trimming. The output `common_support_nonfba_survivor_profile_rankpct.csv` shows that surviving non-FBA offers are concentrated in the top rank quartile, which confirms the re-selection.


In [ ]:
# -----------------------------------------------------------------------------
# Section 16. Non-FBA survivor selection under common support
# -----------------------------------------------------------------------------
# Profile the non-FBA offers retained by the overlap restriction in rank quartiles and price quartiles and document the re-selection toward the top rank quartile.

_nonfba_full = df_final.loc[df_final["fba_from_shipper"] == 0].copy()

def _within_market_nonfba_quartile(g):
    if len(g) < 4:
        return pd.qcut(g["rank_pct"].rank(method="first"),
                       q=4, labels=[1, 2, 3, 4]).astype(int)
    return pd.qcut(g["rank_pct"].rank(method="first"),
                   q=4, labels=[1, 2, 3, 4]).astype(int)

_nonfba_full["nonfba_quartile_in_market"] = (
    _nonfba_full.groupby("market_id", observed=True, group_keys=False)
                 .apply(_within_market_nonfba_quartile)
)

_nonfba_quartile_lookup = _nonfba_full[["entity_time_id", "nonfba_quartile_in_market"]]
cs_sample_aug = cs_sample.merge(_nonfba_quartile_lookup,
                                 on="entity_time_id", how="left")

_cs_nonfba         = cs_sample_aug.loc[cs_sample_aug["fba_from_shipper"] == 0]
_cs_nonfba_in_top  = (_cs_nonfba["nonfba_quartile_in_market"] == 1).sum()
_cs_nonfba_total   = len(_cs_nonfba)
_cs_nonfba_top_share = float(_cs_nonfba_in_top) / max(_cs_nonfba_total, 1)

cs_selection_table = pd.DataFrame([{
    "nonfba_rows_full":                     int((df_final["fba_from_shipper"] == 0).sum()),
    "nonfba_rows_common_support":           int(_cs_nonfba_total),
    "share_nonfba_survivors_in_top_quartile": _cs_nonfba_top_share,
    "expected_share_if_representative":      0.25,
    "selection_ratio":                       _cs_nonfba_top_share / 0.25 if _cs_nonfba_top_share else np.nan,
}])

print("Table D4 - Non-FBA common-support survivor profile")
display(cs_selection_table)
print()
print(f"Retained non-FBA top-quartile share: {_cs_nonfba_top_share*100:.1f}% (expected 25.0% if representative).")

Table D4 - Non-FBA common-support survivor profile


/tmp/ipykernel_2808/2905105748.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_within_market_nonfba_quartile)


,nonfba_rows_full,nonfba_rows_common_support,share_nonfba_survivors_in_top_quartile,expected_share_if_representative,selection_ratio
0,4111,1271,0.649095,0.25,2.596381



Retained non-FBA top-quartile share: 64.9% (expected 25.0% if representative).


### 17. Residual-gap diagnostic before and after common support

The cell compares the mean residual rank difference between FBA and non-FBA offers before and after the common-support restriction. The residualization is performed against the same control set as the headline static specification (`HEADLINE_SPEC`) minus the FBA indicator. If the residual gap is smaller in the trimmed sample, the restriction is removing variation correlated with the unmodelled commercial differences between FBA and non-FBA offers. The output is `common_support_residual_gap_rankpct.csv` and corresponds to `tab:app-ch6-common-support-residual-gap`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 17. Residual-gap diagnostic before and after common support
# -----------------------------------------------------------------------------
# Compare the mean residual rank difference between FBA and non-FBA offers before and after the common-support restriction, holding the residualization controls fixed.

_spec4_rhs_no_fba = [t for t in SPECIFICATIONS[HEADLINE_SPEC] if t != "fba_from_shipper"]

def _residualize_rank_pct(data):
    """Return rank_pct minus its OLS prediction on spec_4 controls + market FE."""
    formula = "rank_pct ~ " + " + ".join(_spec4_rhs_no_fba) + " + C(market_id)"
    res = smf.ols(formula, data=data).fit()
    pred = res.fittedvalues.reindex(data.index)
    return data["rank_pct"] - pred

_resid_full = _residualize_rank_pct(df_final)
_resid_cs   = _residualize_rank_pct(cs_sample)

def _resid_gap_row(label, data, resid):
    a = resid[data["fba_from_shipper"] == 1].dropna()
    b = resid[data["fba_from_shipper"] == 0].dropna()
    diff = a.mean() - b.mean()
    pooled_se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)) if len(a) > 1 and len(b) > 1 else np.nan
    t_stat = diff / pooled_se if pooled_se and pooled_se > 0 else np.nan
    p = float(2 * (1 - stats.t.cdf(abs(t_stat), df=min(len(a), len(b)) - 1))) if pd.notna(t_stat) else np.nan
    return {
        "sample":               label,
        "rows":                 len(data),
        "fba_rows":             len(a),
        "nonfba_rows":          len(b),
        "residual_mean_fba":    a.mean(),
        "residual_mean_nonfba": b.mean(),
        "residual_difference_fba_minus_nonfba": diff,
        "t_stat":               t_stat,
        "pvalue":               p,
    }

residual_gap_table = pd.DataFrame([
    _resid_gap_row("full_sample",    df_final,  _resid_full),
    _resid_gap_row("common_support", cs_sample, _resid_cs),
])

print("Table D5 - Residual rank gap before and after common-support restriction")
display(residual_gap_table)

Table D5 - Residual rank gap before and after common-support restriction


,sample,rows,fba_rows,nonfba_rows,residual_mean_fba,residual_mean_nonfba,residual_difference_fba_minus_nonfba,t_stat,pvalue
0,full_sample,5107,996,4111,-0.024889,0.006030,-0.030919,-20.891156,0.0000
1,common_support,2205,934,1271,-0.000099,0.000073,-0.000172,-0.176933,0.8596


### 18. Relation to platform-ranking evidence in the literature

The empirical strategy follows standard econometric conventions for observational platform data. Fixed effects compare units within the same market; clustering acknowledges the dependence introduced by repeated seller observations. The cell builds an explicit mapping between the design choices of this notebook and the closest empirical references in the literature: Chen and Tsai (2021) on Amazon-presence variation from stockouts, Reimers and Waldfogel (2023) on platform ranking, Farronato, Fradkin, and MacKay (2023) on product search rank, and Decarolis and Rovigatti (2021) on advertising on platforms. The mapping is conservative and treats the references as institutional and methodological context rather than as comparable estimates.


In [ ]:
# -----------------------------------------------------------------------------
# Section 18. Mapping to platform-ranking evidence in the literature
# -----------------------------------------------------------------------------
# Map the design choices of this notebook to the closest empirical references (Chen and Tsai 2021; Reimers and Waldfogel 2023; Farronato, Fradkin, and MacKay 2023; Decarolis and Rovigatti 2021), conservatively.

chen_tsai_mapping_table = pd.DataFrame([
    {
        "design_dimension": "Platform context",
        "chen_tsai_2024": "Amazon FBT recommendations and Amazon's dual role as seller/platform",
        "this_thesis": "Amazon Italy third-party offer-list ranking for Xiaomi Mi Smart Band 6",
        "implication_for_interpretation": "Same broad self-preferencing setting; different platform mechanism and narrower scope.",
    },
    {
        "design_dimension": "Data source",
        "chen_tsai_2024": "high-frequency Keepa scrape data",
        "this_thesis": "high-frequency scraped offer-list data",
        "implication_for_interpretation": "Both use public scrape data rather than internal algorithm data.",
    },
    {
        "design_dimension": "Scope and sample size",
        "chen_tsai_2024": "about 6.7 million products",
        "this_thesis": "one product page, 5,107 final seller-market rows",
        "implication_for_interpretation": "This thesis is an audited case study, not marketplace-wide evidence.",
    },
    {
        "design_dimension": "Time structure",
        "chen_tsai_2024": "five rounds, median 10-day gap",
        "this_thesis": "62 markets, 61 consecutive transitions",
        "implication_for_interpretation": "This thesis has richer within-product transition detail but much narrower external validity.",
    },
    {
        "design_dimension": "Treatment variation",
        "chen_tsai_2024": "Amazon seller presence changes during directly identified Amazon stockout episodes; the product remains available from third-party sellers",
        "this_thesis": "total seller turnover in the observed offer ranking; disappearance is interpreted as stockout-consistent offer unavailability under a maintained assumption",
        "implication_for_interpretation": "The thesis exploits high-frequency seller-availability shocks but cannot equate observed offer-list disappearance with verified inventory stockout.",
    },
    {
        "design_dimension": "Outcome",
        "chen_tsai_2024": "FBT recommendation indicator",
        "this_thesis": "rank_pct improvement and auxiliary top-of-list binary salience outcomes",
        "implication_for_interpretation": "Different algorithmic object; effect sizes are not directly comparable.",
    },
    {
        "design_dimension": "Fixed effects",
        "chen_tsai_2024": "product-pair FE and category-day FE",
        "this_thesis": "seller FE and transition FE; stress test absorbs transition x starting-rank-tier FE",
        "implication_for_interpretation": "Both absorb time-invariant unit heterogeneity and common time shocks; the thesis adds a starting-rank stress test because FBA support is rank-dependent.",
    },
    {
        "design_dimension": "Main coefficient",
        "chen_tsai_2024": "Amazon presence effect on FBT probability, around 0.08 in the reported table",
        "this_thesis": "FBA x total seller turnover in rank_pct improvement",
        "implication_for_interpretation": "The thesis coefficient is a differential turnover-period rank-movement premium, not a direct analogue of an Amazon-presence recommendation effect.",
    },
    {
        "design_dimension": "Validation and placebo logic",
        "chen_tsai_2024": "pre-stockout smoothness; real-time price and sales controls; third-party stockout placebo; FBA-vs-FBM seller-service check",
        "this_thesis": "future-turnover placebo, pre-trend placebo, permutation within transition x starting-rank-tier cells, dropout predictability, pre-disappearance smoothness, return-pattern and price-outlier checks",
        "implication_for_interpretation": "The thesis does not use below-ranked turnover as a placebo for the preferred estimand; above/below turnover is a decomposition of seller-list turnover.",
    },
    {
        "design_dimension": "Stockout/disappearance validation",
        "chen_tsai_2024": "smooth pre-stockout prices and sales support plausibly exogenous Amazon stockouts",
        "this_thesis": "broad and strict stockout-consistent restrictions are estimated separately; disappearance remains observed offer-list absence, not verified inventory stockout",
        "implication_for_interpretation": "The stockout interpretation is strengthened by diagnostics but remains a maintained assumption.",
    },
    {
        "design_dimension": "Interpretation",
        "chen_tsai_2024": "causal evidence of steering in FBT recommendations",
        "this_thesis": "credible but rank-tier-sensitive observational evidence of an FBA turnover-conditional rank-movement premium",
        "implication_for_interpretation": "The thesis claim must remain narrower and avoid definitive causal language.",
    },
    {
        "design_dimension": "External-validity caveat",
        "chen_tsai_2024": "broad Amazon product universe",
        "this_thesis": "single-product Italian marketplace snapshot panel",
        "implication_for_interpretation": "Concede limited external validity and treat the study as a high-audit case analysis.",
    },
])
print("Table F0. Chen and Tsai benchmark mapping")
display(chen_tsai_mapping_table)

Table F0. Chen and Tsai benchmark mapping


,design_dimension,chen_tsai_2024,this_thesis,implication_for_interpretation
0,Platform context,Amazon FBT recommendations and Amazon's dual role as seller/platform,Amazon Italy third-party offer-list ranking for Xiaomi Mi Smart Band 6,Same broad self-preferencing setting; different platform mechanism and narrower scope.
1,Data source,high-frequency Keepa scrape data,high-frequency scraped offer-list data,Both use public scrape data rather than internal algorithm data.
2,Scope and sample size,about 6.7 million products,"one product page, 5,107 final seller-market rows","This thesis is an audited case study, not marketplace-wide evidence."
3,Time structure,"five rounds, median 10-day gap","62 markets, 61 consecutive transitions",This thesis has richer within-product transition detail but much narrower external validity.
4,Treatment variation,Amazon seller presence changes during directly identified Amazon stockout episodes; the product remains available from third-party sellers,total seller turnover in the observed offer ranking; disappearance is interpreted as stockout-consistent offer unavailability under a maintained assumption,The thesis exploits high-frequency seller-availability shocks but cannot equate observed offer-list disappearance with verified inventory stockout.
5,Outcome,FBT recommendation indicator,rank_pct improvement and auxiliary top-of-list binary salience outcomes,Different algorithmic object; effect sizes are not directly comparable.
6,Fixed effects,product-pair FE and category-day FE,seller FE and transition FE; stress test absorbs transition x starting-rank-tier FE,Both absorb time-invariant unit heterogeneity and common time shocks; the thesis adds a starting-rank stress test because FBA support is rank-dependent.
7,Main coefficient,"Amazon presence effect on FBT probability, around 0.08 in the reported table",FBA x total seller turnover in rank_pct improvement,"The thesis coefficient is a differential turnover-period rank-movement premium, not a direct analogue of an Amazon-presence recommendation effect."
8,Validation and placebo logic,pre-stockout smoothness; real-time price and sales controls; third-party stockout placebo; FBA-vs-FBM seller-service check,"future-turnover placebo, pre-trend placebo, permutation within transition x starting-rank-tier cells, dropout predictability, pre-disappearance smoothness, return-pattern and p...",The thesis does not use below-ranked turnover as a placebo for the preferred estimand; above/below turnover is a decomposition of seller-list turnover.
9,Stockout/disappearance validation,smooth pre-stockout prices and sales support plausibly exogenous Amazon stockouts,"broad and strict stockout-consistent restrictions are estimated separately; disappearance remains observed offer-list absence, not verified inventory stockout",The stockout interpretation is strengthened by diagnostics but remains a maintained assumption.


### 19. Dynamic design: FBA premium during seller-turnover rank movement

The dynamic design uses consecutive offer-list snapshots. A seller-transition observation exists when the same seller appears in two consecutive retained markets. Within that sample, rank movement is sufficiently frequent to support a transition analysis. The outcome `rank_pct_improvement` is defined as lagged `rank_pct` minus lead `rank_pct`, so positive values denote upward movement.

The cell builds the dynamic transition panel. It identifies continuing sellers across consecutive markets, computes the turnover exposure of each focal seller (the count of disappearing sellers from the previous market), decomposes that exposure into above-focal and below-focal components, constructs the lagged offer and reputation controls, and validates the panel against the audit registry. The second code cell of this section runs targeted assertion checks on the panel: monotonicity of the rank bounds, sign and definition of the outcome, dimensional consistency between exposure components and total turnover.


In [ ]:
# -----------------------------------------------------------------------------
# Section 19. Dynamic transition panel construction
# -----------------------------------------------------------------------------
# Build the dynamic seller-transition panel: continuing sellers across consecutive markets, turnover exposure, above-focal and below-focal decomposition, lagged offer and reputation controls.

DYNAMIC_OUTCOME = "rank_pct_improvement"
DYNAMIC_EXPOSURE = "dropouts_above"
DYNAMIC_INTERACTION = "fba_x_dropouts_above"


def build_dynamic_vacancy_panel(data):
    """Build seller-transition observations for the dynamic seller-turnover design.

    A seller-transition observation exists when the same seller is present in two
    consecutive timestamp markets. For each continuing seller, dropouts_above
    counts sellers that were above it in the previous ranking and disappeared in
    the next timestamp market.
    """
    data = (
        data.copy()
        .sort_values(["market_order", "rank_pos", "seller_id"], kind="mergesort")
        .reset_index(drop=False)
        .rename(columns={"index": "original_row_index"})
    )

    if "stelle_cat" not in data.columns:
        data["stelle_cat"] = data["stelle"].astype("string").fillna("missing")

    transition_columns = [
        "seller_id", "seller_name", "market_id", "market_order", "timestamp",
        "rank_pos", "rank_pct", "fba_from_shipper",
        "prezzo", "prezzo_spedizione_repaired", "prezzo_totale_reconstructed",
        "log1p_num_valutazioni", "valutazioni_positive", "stelle", "stelle_cat",
        "contact_courier_flag", "g_cons_min_robust", "g_cons_max_robust",
        "delivery_window_robust", "fast_delivery_available", "fast_delivery_cost_imputed",
        "prezzo_totale_fast_delivery_imputed",
    ]
    transition_columns = [col for col in transition_columns if col in data.columns]

    rows = []
    orders = sorted(data["market_order"].dropna().unique())

    for previous_order, next_order in zip(orders[:-1], orders[1:]):
        previous_market = (
            data.loc[data["market_order"].eq(previous_order), transition_columns]
            .copy()
            .sort_values("rank_pos", kind="mergesort")
        )
        next_market = (
            data.loc[data["market_order"].eq(next_order), transition_columns]
            .copy()
            .sort_values("rank_pos", kind="mergesort")
        )

        previous_sellers = set(previous_market["seller_id"])
        next_sellers = set(next_market["seller_id"])

        dropout_mask = ~previous_market["seller_id"].isin(next_sellers)
        entrant_mask = ~next_market["seller_id"].isin(previous_sellers)
        dropout_ranks = np.asarray(previous_market.loc[dropout_mask, "rank_pos"], dtype=float)

        merged = previous_market.merge(
            next_market,
            on="seller_id",
            how="inner",
            suffixes=("_lag", "_lead"),
        )
        if merged.empty:
            continue

        lag_rank = np.asarray(merged["rank_pos_lag"], dtype=float)
        if len(dropout_ranks):
            merged["dropouts_above"] = (dropout_ranks[:, None] < lag_rank[None, :]).sum(axis=0).astype(int)
        else:
            merged["dropouts_above"] = 0

        above_possible = np.maximum(merged["rank_pos_lag"].astype(float) - 1, 0)
        merged["dropout_share_above"] = np.where(
            above_possible > 0,
            merged["dropouts_above"] / above_possible,
            0.0,
        )
        merged["rank_pct_improvement"] = merged["rank_pct_lag"].astype(float) - merged["rank_pct_lead"].astype(float)
        merged["rank_pos_improvement"] = merged["rank_pos_lag"].astype(float) - merged["rank_pos_lead"].astype(float)
        merged["transition_id"] = f"{int(previous_order):02d}_to_{int(next_order):02d}"
        merged["transition_order"] = int(previous_order)
        merged["dropouts_total_transition"] = int(dropout_mask.sum())
        merged["entrants_total_transition"] = int(entrant_mask.sum())
        merged["continuing_sellers_transition"] = int(len(merged))
        rows.append(merged)

    if not rows:
        return pd.DataFrame()

    panel = pd.concat(rows, ignore_index=True)
    rename_map = {}
    for col in list(panel.columns):
        if col.endswith("_lag"):
            rename_map[col] = "lag_" + col[:-4]
        elif col.endswith("_lead"):
            rename_map[col] = "lead_" + col[:-5]
    panel = panel.rename(columns=rename_map)

    for bool_col in ["lag_fba_from_shipper", "lag_contact_courier_flag", "lag_fast_delivery_available"]:
        if bool_col in panel.columns:
            panel[bool_col] = panel[bool_col].astype(int)

    panel["fba_from_shipper"] = panel["lag_fba_from_shipper"].astype(int)
    panel["fba_x_dropouts_above"] = panel["fba_from_shipper"] * panel["dropouts_above"]
    panel["fba_x_dropout_share_above"] = panel["fba_from_shipper"] * panel["dropout_share_above"]
    panel["exposed_to_dropout_above"] = panel["dropouts_above"].gt(0).astype(int)
    panel["fba_x_exposed_to_dropout_above"] = panel["fba_from_shipper"] * panel["exposed_to_dropout_above"]

    return panel


def dynamic_lag_terms_from_static_rhs(rhs):
    """Map a static rank-level specification into lagged dynamic controls."""
    mapped_terms = []
    for term in rhs:
        if term == "fba_from_shipper":
            continue
        if "stelle_cat" in term:
            mapped_terms.append("lag_stelle_cat")
        else:
            mapped_terms.append("lag_" + term)
    return mapped_terms


def make_dynamic_regressor_frame(panel, rhs, exposure_term=DYNAMIC_EXPOSURE, interaction_term=DYNAMIC_INTERACTION):
    """Construct the non-FE regressor matrix for the dynamic model.

    Fixed effects are absorbed by two-way demeaning rather than inserted as large
    dummy matrices. Categorical star ratings are expanded manually with 4.5 stars
    as the reference category when that category is present.
    """
    base_terms = [exposure_term, interaction_term, "lag_rank_pct"] + dynamic_lag_terms_from_static_rhs(rhs)
    base_terms = list(dict.fromkeys(base_terms))
    X = pd.DataFrame(index=panel.index)

    for term in base_terms:
        if term == "lag_stelle_cat":
            dummies_all = pd.get_dummies(panel[term].astype("string"), prefix="lag_stelle_cat", dtype=float)
            reference_col = "lag_stelle_cat_4.5"
            if reference_col in dummies_all.columns:
                dummies = dummies_all.drop(columns=[reference_col])
            else:
                dummies = pd.get_dummies(panel[term].astype("string"), prefix="lag_stelle_cat", drop_first=True, dtype=float)
            X = pd.concat([X, dummies], axis=1)
        else:
            X[term] = pd.to_numeric(panel[term], errors="coerce").astype(float)

    X = X.loc[:, ~X.columns.duplicated()].copy()
    return X


def residualize_two_way(frame, first_fe, second_fe, tol=1e-10, max_iter=100):
    """Absorb two sets of fixed effects by iterative demeaning."""
    values = frame.astype(float).to_numpy().copy()
    values = values - np.nanmean(values, axis=0)
    first_codes = pd.Categorical(first_fe).codes
    second_codes = pd.Categorical(second_fe).codes

    for _ in range(max_iter):
        old_values = values.copy()
        for codes in [first_codes, second_codes]:
            group_count = codes.max() + 1
            sums = np.zeros((group_count, values.shape[1]), dtype=float)
            counts = np.zeros(group_count, dtype=float)
            np.add.at(sums, codes, values)
            np.add.at(counts, codes, 1.0)
            means = sums / np.maximum(counts[:, None], 1.0)
            values = values - means[codes]
        if np.max(np.abs(values - old_values)) < tol:
            break
    return values


def fit_dynamic_vacancy_within_model(
    panel,
    rhs,
    specification,
    metadata=None,
    outcome=DYNAMIC_OUTCOME,
    exposure_term=DYNAMIC_EXPOSURE,
    interaction_term=DYNAMIC_INTERACTION,
):
    """Estimate the dynamic vacancy model with seller and transition fixed effects."""
    X_raw = make_dynamic_regressor_frame(panel, rhs, exposure_term=exposure_term, interaction_term=interaction_term)
    needed = [outcome, "seller_id", "transition_id"]
    model_frame = pd.concat([panel[needed], X_raw], axis=1).dropna().copy()
    X_raw = model_frame[X_raw.columns]

    residualized = residualize_two_way(
        pd.concat([model_frame[[outcome]], X_raw], axis=1),
        first_fe=model_frame["seller_id"],
        second_fe=model_frame["transition_id"],
    )
    y = residualized[:, 0]
    X = residualized[:, 1:]
    names = list(X_raw.columns)

    nonzero_columns = np.nanstd(X, axis=0) > 1e-12
    X = X[:, nonzero_columns]
    names = list(np.asarray(names)[nonzero_columns])

    if interaction_term not in names:
        raise KeyError(f"{interaction_term} was absorbed or not found in the dynamic design matrix.")

    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    residuals = y - X @ beta

    seller_codes = cluster_codes(model_frame["seller_id"])
    transition_codes = cluster_codes(model_frame["transition_id"])
    cov = two_way_cluster_cov_fast(X, residuals, seller_codes, transition_codes, bread=bread)
    dof = max(min(pd.Series(seller_codes).nunique(), pd.Series(transition_codes).nunique()) - 1, 1)

    idx = names.index(interaction_term)
    estimate = float(beta[idx])
    se = float(np.sqrt(max(float(cov[idx, idx]), 0)))
    statistic = estimate / se if se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(statistic), df=dof))) if pd.notna(statistic) else np.nan
    crit = float(stats.t.ppf(0.975, df=dof)) if dof > 0 else 1.96
    r_squared = float(1 - np.sum(residuals ** 2) / np.sum((y - np.mean(y)) ** 2))

    mean_market_size = float(df_final.groupby("market_id").size().mean())
    median_market_size = float(df_final.groupby("market_id").size().median())

    row = {
        "dynamic_design": "seller_fe_transition_fe_reference_above_turnover_exposure",
        "sample": "continuing_seller_transitions",
        "outcome": outcome,
        "outcome_interpretation": "positive values mean upward movement in rank_pct",
        "specification": specification,
        "estimator": "within_ols_absorbing_seller_and_transition_fe",
        "inference": "two_way_seller_transition_clustered",
        "term_name": interaction_term,
        "estimate": estimate,
        "se": se,
        "test_statistic": statistic,
        "pvalue": pvalue,
        "ci_low": estimate - crit * se if pd.notna(se) else np.nan,
        "ci_high": estimate + crit * se if pd.notna(se) else np.nan,
        "stars": significance_stars(pvalue),
        "dof_reference": int(dof),
        "nobs": int(len(y)),
        "r_squared_within_absorbed_fe": r_squared,
        "seller_clusters": int(pd.Series(seller_codes).nunique()),
        "transition_clusters": int(pd.Series(transition_codes).nunique()),
        "fixed_effects": "seller_id and transition_id",
        "exposure_variable": exposure_term,
        "mean_market_size_reference": mean_market_size,
        "median_market_size_reference": median_market_size,
        "implied_rank_position_shift_mean_market": estimate * (mean_market_size - 1),
        "implied_rank_position_shift_median_market": estimate * (median_market_size - 1),
        "interpretation_note": "Effect of one additional dropout above for FBA sellers relative to non-FBA sellers, after seller and transition fixed effects.",
    }
    if metadata:
        row.update(metadata)
    return row


def run_dynamic_vacancy_grid(panel):
    rows = []
    for specification, rhs in PRICE_STAR_SPECIFICATIONS.items():
        metadata = PRICE_STAR_METADATA[specification].copy()
        rows.append(
            fit_dynamic_vacancy_within_model(
                panel=panel,
                rhs=rhs,
                specification=specification,
                metadata=metadata,
            )
        )
    return pd.DataFrame(rows)


# Build the transition-level mechanism sample.
dynamic_vacancy_panel = build_dynamic_vacancy_panel(df_final)

register_check("10", "dynamic_vacancy_panel_nonempty", len(dynamic_vacancy_panel) > 0, f"rows={len(dynamic_vacancy_panel)}")
register_check("10", "dynamic_vacancy_transitions_available", dynamic_vacancy_panel["transition_id"].nunique() >= 2, f"transitions={dynamic_vacancy_panel['transition_id'].nunique()}")

# Within-seller variation diagnostics. These tables document why the dynamic
# design is a ranking-movement design, not a within-seller FBA-switching design.
DYNAMIC_WITHIN_SELLER_VARIABLES = [
    "rank_pct",
    "rank_pos",
    "prezzo",
    "prezzo_totale_reconstructed",
    "prezzo_spedizione_repaired",
    "g_cons_min_robust",
    "g_cons_max_robust",
    "delivery_window_robust",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
    "fba_from_shipper",
]


def build_within_seller_variation_tables(static_data, transition_panel, variables):
    """Summarize within-seller variation in levels and adjacent transitions."""
    seller_observation_counts = static_data.groupby("seller_id").size()
    sellers_observed_twice = seller_observation_counts.loc[seller_observation_counts.ge(2)].index
    repeated_seller_data = static_data.loc[static_data["seller_id"].isin(sellers_observed_twice)].copy()

    level_rows = []
    for variable in variables:
        if variable not in repeated_seller_data.columns:
            continue
        seller_nunique = repeated_seller_data.groupby("seller_id")[variable].nunique(dropna=False)
        varying_sellers = int(seller_nunique.gt(1).sum())
        total_sellers = int(len(seller_nunique))
        level_rows.append(
            {
                "variable": variable,
                "sellers_observed_at_least_twice": total_sellers,
                "sellers_with_within_seller_variation": varying_sellers,
                "sellers_without_within_seller_variation": int(total_sellers - varying_sellers),
                "share_with_within_seller_variation": float(varying_sellers / total_sellers) if total_sellers else np.nan,
                "interpretation_note": "Variation is computed across all observed markets for sellers appearing at least twice.",
            }
        )

    transition_rows = []
    for variable in variables:
        lag_col = f"lag_{variable}"
        lead_col = f"lead_{variable}"
        if lag_col not in transition_panel.columns or lead_col not in transition_panel.columns:
            continue
        pair = transition_panel[[lag_col, lead_col]].copy()
        valid = pair[lag_col].notna() & pair[lead_col].notna()
        if pd.api.types.is_numeric_dtype(pair[lag_col]) and pd.api.types.is_numeric_dtype(pair[lead_col]):
            changed = (pd.to_numeric(pair.loc[valid, lag_col], errors="coerce") != pd.to_numeric(pair.loc[valid, lead_col], errors="coerce"))
        else:
            changed = (pair.loc[valid, lag_col].astype("string") != pair.loc[valid, lead_col].astype("string"))
        valid_transitions = int(valid.sum())
        changed_transitions = int(changed.sum())
        transition_rows.append(
            {
                "variable": variable,
                "valid_adjacent_seller_transitions": valid_transitions,
                "transitions_with_change": changed_transitions,
                "transitions_without_change": int(valid_transitions - changed_transitions),
                "share_of_transitions_with_change": float(changed_transitions / valid_transitions) if valid_transitions else np.nan,
                "interpretation_note": "Variation is computed across adjacent markets for sellers present in both snapshots.",
            }
        )

    sample_summary = pd.DataFrame(
        [
            {
                "total_sellers": int(static_data["seller_id"].nunique()),
                "sellers_observed_at_least_twice": int(len(sellers_observed_twice)),
                "continuing_seller_transition_observations": int(len(transition_panel)),
                "sellers_in_all_markets": int((seller_observation_counts == static_data["market_id"].nunique()).sum()),
                "markets": int(static_data["market_id"].nunique()),
                "interpretation": "The dynamic design uses continuing seller-transition observations. FBA does not switch within seller, but ranking exposure and rank outcomes vary within seller over time.",
            }
        ]
    )

    level_table = pd.DataFrame(level_rows)
    transition_table = pd.DataFrame(transition_rows)
    return sample_summary, level_table, transition_table


dynamic_within_seller_sample_summary, dynamic_within_seller_variation_table, dynamic_consecutive_transition_variation_table = build_within_seller_variation_tables(
    static_data=df_final,
    transition_panel=dynamic_vacancy_panel,
    variables=DYNAMIC_WITHIN_SELLER_VARIABLES,
)

# The FBA main effect is intentionally not identified in a seller-FE model, but
# the FBA-by-vacancy interaction is identified because dropouts_above changes over time.
dynamic_vacancy_identification_logic_table = pd.DataFrame(
    [
        {
            "object": "fba_from_shipper",
            "within_seller_variation": "none in the final sample",
            "role_in_dynamic_design": "main FBA level is absorbed by seller fixed effects",
        },
        {
            "object": "dropouts_above",
            "within_seller_variation": "varies across seller-transition observations",
            "role_in_dynamic_design": "time-varying directional seller-disappearance exposure; used for decomposition rather than the primary premium estimand",
        },
        {
            "object": "fba_from_shipper × dropouts_above",
            "within_seller_variation": "varies because dropouts_above varies within seller",
            "role_in_dynamic_design": "retained as a directional decomposition check; the main dynamic claim is the broader FBA-by-turnover premium rather than local above-turnover filling",
        },
        {
            "object": "rank_pct_improvement",
            "within_seller_variation": "varies across consecutive seller transitions",
            "role_in_dynamic_design": "outcome measuring upward or downward rank movement",
        },
    ]
)

# High-level design diagnostics.
dynamic_vacancy_design_summary = pd.DataFrame(
    [
        {
            "observations": int(len(dynamic_vacancy_panel)),
            "transitions": int(dynamic_vacancy_panel["transition_id"].nunique()),
            "sellers": int(dynamic_vacancy_panel["seller_id"].nunique()),
            "fba_observations": int(dynamic_vacancy_panel["fba_from_shipper"].sum()),
            "nonfba_observations": int((1 - dynamic_vacancy_panel["fba_from_shipper"]).sum()),
            "observations_with_dropout_above": int(dynamic_vacancy_panel["exposed_to_dropout_above"].sum()),
            "share_with_dropout_above": float(dynamic_vacancy_panel["exposed_to_dropout_above"].mean()),
            "max_dropouts_above": int(dynamic_vacancy_panel["dropouts_above"].max()),
            "mean_dropouts_above": float(dynamic_vacancy_panel["dropouts_above"].mean()),
            "mean_rank_pct_improvement": float(dynamic_vacancy_panel["rank_pct_improvement"].mean()),
            "median_rank_pct_improvement": float(dynamic_vacancy_panel["rank_pct_improvement"].median()),
            "design_note": "Continuing seller-transition panel. Positive rank_pct_improvement means movement toward a better normalized rank.",
        }
    ]
)

# Transition-level churn summary. This is displayed to show that the mechanism is not based only on rank-1 events.
dynamic_vacancy_transition_summary_table = (
    dynamic_vacancy_panel.groupby("transition_id")
    .agg(
        continuing_sellers=("seller_id", "size"),
        dropouts_total=("dropouts_total_transition", "first"),
        entrants_total=("entrants_total_transition", "first"),
        exposed_rows=("exposed_to_dropout_above", "sum"),
        mean_dropouts_above=("dropouts_above", "mean"),
        max_dropouts_above=("dropouts_above", "max"),
        mean_rank_pct_improvement=("rank_pct_improvement", "mean"),
    )
    .reset_index()
)

dynamic_vacancy_exposure_distribution_table = (
    dynamic_vacancy_panel["dropouts_above"]
    .value_counts()
    .rename_axis("dropouts_above")
    .reset_index(name="observations")
    .sort_values("dropouts_above")
)
dynamic_vacancy_exposure_distribution_table["share"] = dynamic_vacancy_exposure_distribution_table["observations"] / len(dynamic_vacancy_panel)

dynamic_vacancy_transition_diagnostics_table = pd.DataFrame(
    [
        {
            "transitions": int(len(dynamic_vacancy_transition_summary_table)),
            "transitions_with_any_dropout": int(dynamic_vacancy_transition_summary_table["dropouts_total"].gt(0).sum()),
            "mean_total_dropouts_per_transition": float(dynamic_vacancy_transition_summary_table["dropouts_total"].mean()),
            "max_total_dropouts_in_transition": int(dynamic_vacancy_transition_summary_table["dropouts_total"].max()),
            "mean_exposed_rows_per_transition": float(dynamic_vacancy_transition_summary_table["exposed_rows"].mean()),
            "max_exposed_rows_in_transition": int(dynamic_vacancy_transition_summary_table["exposed_rows"].max()),
            "mean_transition_rank_pct_improvement": float(dynamic_vacancy_transition_summary_table["mean_rank_pct_improvement"].mean()),
            "display_note": "Full transition-level table is exported; this compact diagnostic is displayed to keep the notebook readable.",
        }
    ]
)

# Descriptive movement by FBA and exposure status.
dynamic_vacancy_descriptive_table = (
    dynamic_vacancy_panel.groupby(["exposed_to_dropout_above", "fba_from_shipper"])
    .agg(
        observations=("seller_id", "size"),
        sellers=("seller_id", "nunique"),
        transitions=("transition_id", "nunique"),
        mean_dropouts_above=("dropouts_above", "mean"),
        mean_rank_pct_improvement=("rank_pct_improvement", "mean"),
        median_rank_pct_improvement=("rank_pct_improvement", "median"),
        mean_lag_rank_pct=("lag_rank_pct", "mean"),
        mean_lead_rank_pct=("lead_rank_pct", "mean"),
    )
    .reset_index()
)

# Main dynamic seller-FE and transition-FE model grid.
dynamic_vacancy_models_table = run_dynamic_vacancy_grid(dynamic_vacancy_panel)


def assign_dynamic_rank_tier(panel):
    """Classify seller-transition observations by the seller's lagged rank_pct."""
    conditions = [
        panel["lag_rank_pct"].le(0.25),
        panel["lag_rank_pct"].gt(0.25) & panel["lag_rank_pct"].le(0.75),
        panel["lag_rank_pct"].gt(0.75),
    ]
    choices = ["top_25pct", "middle_50pct", "bottom_25pct"]
    out = panel.copy()
    out["lag_rank_tier"] = np.select(conditions, choices, default="unclassified")
    return out


def run_dynamic_vacancy_rank_tier_grid(panel):
    """Estimate the dynamic vacancy model separately by baseline rank tier.

    The tier split checks whether the FBA-by-vacancy adjustment is concentrated
    near the top of the ranking, where rank movements are more likely to be
    economically meaningful. Non-estimated tier-specification combinations are kept as
    diagnostic rows instead of stopping the notebook.
    """
    tiered = assign_dynamic_rank_tier(panel)
    tier_definitions = [
        ("top_25pct", "lag_rank_pct <= 0.25"),
        ("middle_50pct", "0.25 < lag_rank_pct <= 0.75"),
        ("bottom_25pct", "lag_rank_pct > 0.75"),
    ]
    rows = []
    for tier_name, tier_rule in tier_definitions:
        tier_panel = tiered.loc[tiered["lag_rank_tier"].eq(tier_name)].copy()
        tier_summary = {
            "lag_rank_tier": tier_name,
            "tier_rule": tier_rule,
            "tier_observations": int(len(tier_panel)),
            "tier_sellers": int(tier_panel["seller_id"].nunique()) if len(tier_panel) else 0,
            "tier_transitions": int(tier_panel["transition_id"].nunique()) if len(tier_panel) else 0,
            "tier_fba_observations": int(tier_panel["fba_from_shipper"].sum()) if len(tier_panel) else 0,
            "tier_nonfba_observations": int((1 - tier_panel["fba_from_shipper"]).sum()) if len(tier_panel) else 0,
            "tier_fba_share": float(tier_panel["fba_from_shipper"].mean()) if len(tier_panel) else np.nan,
            "tier_thin_fba_support_flag": bool(tier_panel["fba_from_shipper"].sum() < 30) if len(tier_panel) else True,
            "tier_exposed_observations": int(tier_panel["exposed_to_dropout_above"].sum()) if len(tier_panel) else 0,
            "tier_share_exposed": float(tier_panel["exposed_to_dropout_above"].mean()) if len(tier_panel) else np.nan,
            "tier_mean_lag_rank_pct": float(tier_panel["lag_rank_pct"].mean()) if len(tier_panel) else np.nan,
        }
        for specification, rhs in PRICE_STAR_SPECIFICATIONS.items():
            metadata = PRICE_STAR_METADATA[specification].copy()
            try:
                row = fit_dynamic_vacancy_within_model(
                    panel=tier_panel,
                    rhs=rhs,
                    specification=specification,
                    metadata=metadata,
                )
                row.update(tier_summary)
                row["rank_tier_status"] = "estimated"
            except Exception as exc:
                row = {
                    "dynamic_design": "seller_fe_transition_fe_reference_above_turnover_by_lag_rank_tier",
                    "sample": "continuing_seller_transitions",
                    "outcome": DYNAMIC_OUTCOME,
                    "specification": specification,
                    "term_name": DYNAMIC_INTERACTION,
                    "estimate": np.nan,
                    "se": np.nan,
                    "test_statistic": np.nan,
                    "pvalue": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "stars": "",
                    "rank_tier_status": f"not_estimated: {exc}",
                }
                row.update(metadata)
                row.update(tier_summary)
            rows.append(row)
    return pd.DataFrame(rows)


# Rank-tier dynamic sensitivity. This directly addresses the concern that a
# normalized rank movement may be more valuable near the top than near the bottom.
dynamic_vacancy_rank_tier_models_table = run_dynamic_vacancy_rank_tier_grid(dynamic_vacancy_panel)

# Row-level numerical-pathology flags. These flags prevent degenerate appendix rows
# from being read as substantive estimates. A row is not interpreted if its
# standard error or p-value is non-finite, if the standard error is numerically
# zero, or if the absorbed-FE within R-squared is outside a defensible numerical
# range because of near-singularity.
_est = pd.to_numeric(dynamic_vacancy_rank_tier_models_table.get("estimate"), errors="coerce")
_se = pd.to_numeric(dynamic_vacancy_rank_tier_models_table.get("se"), errors="coerce")
_p = pd.to_numeric(dynamic_vacancy_rank_tier_models_table.get("pvalue"), errors="coerce")
_r2 = pd.to_numeric(dynamic_vacancy_rank_tier_models_table.get("r_squared_within_absorbed_fe"), errors="coerce")
dynamic_vacancy_rank_tier_models_table["row_numerical_pathology_flag"] = (
    (~np.isfinite(_est))
    | (~np.isfinite(_se))
    | (~np.isfinite(_p))
    | _se.le(1e-12)
    | _r2.abs().gt(1e6)
)
dynamic_vacancy_rank_tier_models_table["row_interpretation_status"] = np.select(
    [
        dynamic_vacancy_rank_tier_models_table["row_numerical_pathology_flag"],
        dynamic_vacancy_rank_tier_models_table.get("tier_thin_fba_support_flag", False).astype(bool),
    ],
    [
        "not_interpreted_numerical_pathology",
        "interpreted_with_thin_fba_support_caution",
    ],
    default="interpreted",
)

dynamic_vacancy_rank_tier_summary_table = (
    dynamic_vacancy_rank_tier_models_table
    .loc[dynamic_vacancy_rank_tier_models_table["specification"].eq("split_price_linear_stars")]
    [[
        "lag_rank_tier", "tier_rule", "tier_observations", "tier_sellers", "tier_transitions",
        "tier_fba_observations", "tier_nonfba_observations", "tier_fba_share", "tier_thin_fba_support_flag",
        "tier_exposed_observations", "tier_share_exposed", "estimate", "se", "pvalue", "ci_low", "ci_high", "stars",
        "implied_rank_position_shift_mean_market", "rank_tier_status", "row_numerical_pathology_flag", "row_interpretation_status",
    ]]
    .reset_index(drop=True)
)


# Compact translation table for the primary dynamic specification.
preferred_dynamic_row = dynamic_vacancy_models_table.loc[
    dynamic_vacancy_models_table["specification"].eq("split_price_linear_stars")
].iloc[0]
dynamic_vacancy_effect_translation_table = pd.DataFrame(
    [
        {
            "reference_market_size": float(preferred_dynamic_row["mean_market_size_reference"]),
            "implied_rank_position_shift_per_dropout": float(preferred_dynamic_row["implied_rank_position_shift_mean_market"]),
            "note": "Mean third-party market size equivalent. This is an approximate translation of rank_pct units.",
        },
        {
            "reference_market_size": float(preferred_dynamic_row["median_market_size_reference"]),
            "implied_rank_position_shift_per_dropout": float(preferred_dynamic_row["implied_rank_position_shift_median_market"]),
            "note": "Median third-party market size equivalent. This is an approximate translation of rank_pct units.",
        },
    ]
)

dynamic_vacancy_interpretation_table = pd.DataFrame(
    [
        {
            "object": "dynamic_estimand",
            "interpretation": "differential rank_pct adjustment for FBA sellers during seller-disappearance transitions; in the final specification this object is treated as a decomposition component of the broader turnover-premium design",
        },
        {
            "object": "identification_gain_over_static_ols",
            "interpretation": "seller fixed effects absorb time-invariant seller quality and transition fixed effects absorb shocks common to a pair of consecutive markets",
        },
        {
            "object": "remaining_limitation",
            "interpretation": "seller disappearance is observed disappearance from the offer list, not verified random stock-out, so the design is stronger than static OLS but still observational",
        },
        {
            "object": "rank_tier_sensitivity",
            "interpretation": "the dynamic vacancy model is re-estimated by baseline rank tier to check whether the FBA-vacancy response is concentrated near the economically more visible top of the ranking",
        },
        {
            "object": "preferred_reading",
            "interpretation": "the final dynamic mechanism tests whether FBA status is associated with stronger rank adjustment during seller turnover; the clean local above-turnover channel is not the primary claim",
        },
    ]
)

display(dynamic_within_seller_sample_summary)
display(dynamic_within_seller_variation_table)
display(dynamic_consecutive_transition_variation_table)
display(dynamic_vacancy_identification_logic_table)
display(dynamic_vacancy_design_summary)
display(dynamic_vacancy_transition_diagnostics_table)
display(dynamic_vacancy_exposure_distribution_table)
display(dynamic_vacancy_descriptive_table)
display(dynamic_vacancy_models_table)
display(dynamic_vacancy_rank_tier_summary_table)
display(dynamic_vacancy_effect_translation_table)
display(dynamic_vacancy_interpretation_table)

,total_sellers,sellers_observed_at_least_twice,continuing_seller_transition_observations,sellers_in_all_markets,markets,interpretation
0,119,114,4958,52,62,"The dynamic design uses continuing seller-transition observations. FBA does not switch within seller, but ranking exposure and rank outcomes vary within seller over time."


,variable,sellers_observed_at_least_twice,sellers_with_within_seller_variation,sellers_without_within_seller_variation,share_with_within_seller_variation,interpretation_note
0,rank_pct,114,111,3,0.973684,Variation is computed across all observed markets for sellers appearing at least twice.
1,rank_pos,114,110,4,0.964912,Variation is computed across all observed markets for sellers appearing at least twice.
2,prezzo,114,59,55,0.517544,Variation is computed across all observed markets for sellers appearing at least twice.
3,prezzo_totale_reconstructed,114,67,47,0.587719,Variation is computed across all observed markets for sellers appearing at least twice.
4,prezzo_spedizione_repaired,114,15,99,0.131579,Variation is computed across all observed markets for sellers appearing at least twice.
5,g_cons_min_robust,114,107,7,0.938596,Variation is computed across all observed markets for sellers appearing at least twice.
6,g_cons_max_robust,114,108,6,0.947368,Variation is computed across all observed markets for sellers appearing at least twice.
7,delivery_window_robust,114,88,26,0.771930,Variation is computed across all observed markets for sellers appearing at least twice.
8,log1p_num_valutazioni,114,81,33,0.710526,Variation is computed across all observed markets for sellers appearing at least twice.
9,valutazioni_positive,114,61,53,0.535088,Variation is computed across all observed markets for sellers appearing at least twice.


,variable,valid_adjacent_seller_transitions,transitions_with_change,transitions_without_change,share_of_transitions_with_change,interpretation_note
0,rank_pct,4958,3893,1065,0.785196,Variation is computed across adjacent markets for sellers present in both snapshots.
1,rank_pos,4958,2879,2079,0.580678,Variation is computed across adjacent markets for sellers present in both snapshots.
2,prezzo,4958,288,4670,0.058088,Variation is computed across adjacent markets for sellers present in both snapshots.
3,prezzo_totale_reconstructed,4958,336,4622,0.067769,Variation is computed across adjacent markets for sellers present in both snapshots.
4,prezzo_spedizione_repaired,4958,51,4907,0.010286,Variation is computed across adjacent markets for sellers present in both snapshots.
5,g_cons_min_robust,4958,1558,3400,0.314240,Variation is computed across adjacent markets for sellers present in both snapshots.
6,g_cons_max_robust,4958,1581,3377,0.318879,Variation is computed across adjacent markets for sellers present in both snapshots.
7,delivery_window_robust,4958,699,4259,0.140984,Variation is computed across adjacent markets for sellers present in both snapshots.
8,log1p_num_valutazioni,4958,898,4060,0.181121,Variation is computed across adjacent markets for sellers present in both snapshots.
9,valutazioni_positive,4958,212,4746,0.042759,Variation is computed across adjacent markets for sellers present in both snapshots.


,object,within_seller_variation,role_in_dynamic_design
0,fba_from_shipper,none in the final sample,main FBA level is absorbed by seller fixed effects
1,dropouts_above,varies across seller-transition observations,time-varying directional seller-disappearance exposure; used for decomposition rather than the primary premium estimand
2,fba_from_shipper × dropouts_above,varies because dropouts_above varies within seller,retained as a directional decomposition check; the main dynamic claim is the broader FBA-by-turnover premium rather than local above-turnover filling
3,rank_pct_improvement,varies across consecutive seller transitions,outcome measuring upward or downward rank movement


,observations,transitions,sellers,fba_observations,nonfba_observations,observations_with_dropout_above,share_with_dropout_above,max_dropouts_above,mean_dropouts_above,mean_rank_pct_improvement,median_rank_pct_improvement,design_note
0,4958,61,114,962,3996,1739,0.350746,5,0.482251,0.000671,0.0,Continuing seller-transition panel. Positive rank_pct_improvement means movement toward a better normalized rank.


...

,exposed_to_dropout_above,fba_from_shipper,observations,sellers,transitions,mean_dropouts_above,mean_rank_pct_improvement,median_rank_pct_improvement,mean_lag_rank_pct,mean_lead_rank_pct
0,0,0,2484,87,56,0.000000,-0.001027,0.000000,0.523630,0.524657
1,0,1,735,26,58,0.000000,-0.001547,0.000000,0.176847,0.178394
2,1,0,1512,83,35,1.400132,0.003474,0.002426,0.653941,0.650467
3,1,1,227,22,26,1.207048,0.007775,0.008601,0.262937,0.255162


,dynamic_design,sample,outcome,outcome_interpretation,specification,estimator,inference,term_name,estimate,se,test_statistic,pvalue,ci_low,ci_high,stars,dof_reference,nobs,r_squared_within_absorbed_fe,seller_clusters,transition_clusters,fixed_effects,exposure_variable,mean_market_size_reference,median_market_size_reference,implied_rank_position_shift_mean_market,implied_rank_position_shift_median_market,interpretation_note,price_measure,star_form,positive_review_form,price_control_definition,star_control_definition,positive_review_control_definition,role
0,seller_fe_transition_fe_reference_above_turnover_exposure,continuing_seller_transitions,rank_pct_improvement,positive values mean upward movement in rank_pct,split_price_linear_stars,within_ols_absorbing_seller_and_transition_fe,two_way_seller_transition_clustered,fba_x_dropouts_above,0.003606,0.001678,2.149639,0.035626,0.000251,0.006962,**,60,4958,0.155860,114,61,seller_id and transition_id,dropouts_above,82.370968,80.0,0.293454,0.284904,"Effect of one additional dropout above for FBA sellers relative to non-FBA sellers, after seller and transition fixed effects.",split_price,linear_stars,positive_reviews_included,product price plus shipping price separately,star rating entered linearly,positive-review percentage included,headline_price_decomposition
1,seller_fe_transition_fe_reference_above_turnover_exposure,continuing_seller_transitions,rank_pct_improvement,positive values mean upward movement in rank_pct,split_price_linear_stars_no_positive_reviews,within_ols_absorbing_seller_and_transition_fe,two_way_seller_transition_clustered,fba_x_dropouts_above,0.003583,0.001682,2.130492,0.037240,0.000219,0.006946,**,60,4958,0.155579,114,61,seller_id and transition_id,dropouts_above,82.370968,80.0,0.291517,0.283023,"Effect of one additional dropout above for FBA sellers relative to non-FBA sellers, after seller and transition fixed effects.",split_price,linear_stars,positive_reviews_dropped,product price plus shipping price separately,star rating entered linearly,positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,positive_review_exclusion_sensitivity
2,seller_fe_transition_fe_reference_above_turnover_exposure,continuing_seller_transitions,rank_pct_improvement,positive values mean upward movement in rank_pct,split_price_categorical_stars,within_ols_absorbing_seller_and_transition_fe,two_way_seller_transition_clustered,fba_x_dropouts_above,0.003553,0.001677,2.119168,0.038224,0.000199,0.006907,**,60,4958,0.157582,114,61,seller_id and transition_id,dropouts_above,82.370968,80.0,0.289138,0.280713,"Effect of one additional dropout above for FBA sellers relative to non-FBA sellers, after seller and transition fixed effects.",split_price,categorical_stars,positive_reviews_included,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage included,categorical_star_sensitivity
3,seller_fe_transition_fe_reference_above_turnover_exposure,continuing_seller_transitions,rank_pct_improvement,positive values mean upward movement in rank_pct,split_price_categorical_stars_no_positive_reviews,within_ols_absorbing_seller_and_transition_fe,two_way_seller_transition_clustered,fba_x_dropouts_above,0.003542,0.001678,2.110627,0.038981,0.000185,0.006898,**,60,4958,0.157533,114,61,seller_id and transition_id,dropouts_above,82.370968,80.0,0.288193,0.279796,"Effect of one additional dropout above for FBA sellers relative to non-FBA sellers, after seller and transition fixed effects.",split_price,categorical_stars,positive_reviews_dropped,product price plus shipping price separately,"star rating entered as category indicators, 4.5-star reference",positive-review percentage excluded to test sensitivity to the highly collinear reputation proxy,positive_review_exclusion_sensitivity
4,seller_fe_transition_fe_reference_above_turnover_exposure,continuing_seller_transitions,rank_pct_improvement,po

,lag_rank_tier,tier_rule,tier_observations,tier_sellers,tier_transitions,tier_fba_observations,tier_nonfba_observations,tier_fba_share,tier_thin_fba_support_flag,tier_exposed_observations,tier_share_exposed,estimate,se,pvalue,ci_low,ci_high,stars,implied_rank_position_shift_mean_market,rank_tier_status,row_numerical_pathology_flag,row_interpretation_status
0,top_25pct,lag_rank_pct <= 0.25,1258,36,61,722,536,0.573927,False,244,0.193959,-0.001261,0.001603,0.436950,-0.004515,0.001994,,-0.102576,estimated,False,interpreted
1,middle_50pct,0.25 < lag_rank_pct <= 0.75,2456,67,61,228,2228,0.092834,False,865,0.352199,0.001456,0.003078,0.637981,-0.004701,0.007613,,0.118450,estimated,False,interpreted
2,bottom_25pct,lag_rank_pct > 0.75,1244,41,61,12,1232,0.009646,True,630,0.506431,0.001629,0.001053,0.129975,-0.000500,0.003757,,0.132515,estimated,False,interpreted_with_thin_fba_support_caution


,reference_market_size,implied_rank_position_shift_per_dropout,note
0,82.370968,0.293454,Mean third-party market size equivalent. This is an approximate translation of rank_pct units.
1,80.000000,0.284904,Median third-party market size equivalent. This is an approximate translation of rank_pct units.


,object,interpretation
0,dynamic_estimand,differential rank_pct adjustment for FBA sellers during seller-disappearance transitions; in the final specification this object is treated as a decomposition component of the ...
1,identification_gain_over_static_ols,seller fixed effects absorb time-invariant seller quality and transition fixed effects absorb shocks common to a pair of consecutive markets
2,remaining_limitation,"seller disappearance is observed disappearance from the offer list, not verified random stock-out, so the design is stronger than static OLS but still observational"
3,rank_tier_sensitivity,the dynamic vacancy model is re-estimated by baseline rank tier to check whether the FBA-vacancy response is concentrated near the economically more visible top of the ranking
4,preferred_reading,the final dynamic mechanism tests whether FBA status is associated with stronger rank adjustment during seller turnover; the clean local above-turnover channel is not the prima...


In [ ]:
# -----------------------------------------------------------------------------
# Section 19. Dynamic transition panel: consistency assertions
# -----------------------------------------------------------------------------
# Run assertion checks on the dynamic panel: monotonicity of the rank bounds, sign and definition of the outcome, dimensional consistency of the exposure components.

assert df_final["rank_pct"].between(0, 1).all(), "rank_pct must lie in [0,1]"
if "rank_pct_improvement" in dynamic_vacancy_panel.columns:
    _check = (dynamic_vacancy_panel["lag_rank_pct"] - dynamic_vacancy_panel["lead_rank_pct"])
    assert np.allclose(
        dynamic_vacancy_panel["rank_pct_improvement"],
        _check, equal_nan=False,
    ), "rank_pct_improvement must equal lag_rank_pct - lead_rank_pct"
    print("Sign safeguard OK: lower rank_pct = better; positive rank_pct_improvement = upward movement.")

Sign safeguard OK: lower rank_pct = better; positive rank_pct_improvement = upward movement.


### 20. Dynamic estimation helper

The dynamic helper residualizes variables with respect to seller fixed effects and transition fixed effects. Seller fixed effects absorb time-invariant seller characteristics; transition fixed effects absorb common shocks between consecutive market snapshots. The remaining residual variation identifies the FBA-by-turnover interaction. The cell defines `fit_dynamic_general_within_model`, the unified within-transformation estimator used for D1, D2, D3, and D4 and for every sensitivity variant of the dynamic block.


In [ ]:
# -----------------------------------------------------------------------------
# Section 20. Dynamic estimation helper
# -----------------------------------------------------------------------------
# Define the unified within-transformation estimator used for D1, D2, D3, and D4 with seller fixed effects, transition fixed effects, and clustered inference.

def fit_dynamic_general_within_model(
    panel,
    extra_cols,
    static_rhs=None,
    outcome=DYNAMIC_OUTCOME,
    seller_col="seller_id",
    transition_col="transition_id",
):
    """Fit a dynamic model with seller-FE and transition-FE absorbed by demeaning,
    using two-way clustered standard errors (Cameron-Gelbach-Miller).

    Parameters
    ----------
    panel : pd.DataFrame
        The dynamic panel (must include outcome, seller_col, transition_col, and
        every column listed in `extra_cols`; lag_* terms are added by
        `make_dynamic_regressor_frame` from `static_rhs` if it is supplied).
    extra_cols : list[str]
        Columns to ADD to the regressor matrix in addition to the standard
        dynamic regressors. These are the terms specific to the variant
        (e.g. ["dropouts_below", "fba_x_dropouts_below"] for the placebo).
    static_rhs : list[str] or None
        If provided, build the standard dynamic regressors using
        `make_dynamic_regressor_frame(panel, static_rhs)`. If None, only
        `extra_cols` are used as regressors (caller is fully responsible).

    Returns
    -------
    dict with keys: beta (np.array), cov (np.array), names (list[str]),
    dof (int), nobs (int), seller_clusters (int), transition_clusters (int),
    panel_used (pd.DataFrame).
    """
    if static_rhs is not None:
        X_standard = make_dynamic_regressor_frame(panel, static_rhs)
    else:
        X_standard = pd.DataFrame(index=panel.index)

    # Extra columns from the panel itself
    for col in extra_cols:
        X_standard[col] = pd.to_numeric(panel[col], errors="coerce").astype(float)

    # Drop any duplicate columns
    X_standard = X_standard.loc[:, ~X_standard.columns.duplicated()].copy()

    needed = [outcome, seller_col, transition_col]
    model_frame = pd.concat([panel[needed], X_standard], axis=1).dropna().copy()
    X_raw = model_frame[X_standard.columns]

    # Two-way demean (residualize_two_way returns a numpy array)
    residualized = residualize_two_way(
        pd.concat([model_frame[[outcome]], X_raw], axis=1),
        first_fe=model_frame[seller_col],
        second_fe=model_frame[transition_col],
    )
    y = residualized[:, 0]
    X = residualized[:, 1:]
    names = list(X_raw.columns)

    # Drop columns absorbed (near-zero variance after demeaning)
    nonzero_columns = np.nanstd(X, axis=0) > 1e-12
    X = X[:, nonzero_columns]
    names = list(np.asarray(names)[nonzero_columns])

    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    residuals = y - X @ beta

    seller_codes = cluster_codes(model_frame[seller_col])
    transition_codes = cluster_codes(model_frame[transition_col])
    cov = two_way_cluster_cov_fast(X, residuals, seller_codes, transition_codes, bread=bread)
    n_seller = int(pd.Series(seller_codes).nunique())
    n_trans  = int(pd.Series(transition_codes).nunique())
    dof = max(min(n_seller, n_trans) - 1, 1)

    return {
        "beta":                beta,
        "cov":                 cov,
        "names":               names,
        "dof":                 dof,
        "nobs":                int(len(y)),
        "seller_clusters":     n_seller,
        "transition_clusters": n_trans,
        "y":                   y,
        "X":                   X,
        "residuals":           residuals,
        "model_frame":         model_frame,
    }

def _term_row(fit, term):
    """Extract a labeled row for a single coefficient from a fit_dynamic_general result."""
    if term not in fit["names"]:
        return {"term": term, "estimate": np.nan, "se": np.nan, "pvalue": np.nan,
                "ci_low": np.nan, "ci_high": np.nan, "absorbed_or_missing": True,
                "nobs": fit["nobs"]}
    idx = fit["names"].index(term)
    b = float(fit["beta"][idx])
    s = float(np.sqrt(max(fit["cov"][idx, idx], 0.0)))
    t = b / s if s > 0 else np.nan
    p = float(2 * (1 - stats.t.cdf(abs(t), df=fit["dof"]))) if pd.notna(t) else np.nan
    crit = float(stats.t.ppf(0.975, df=fit["dof"]))
    return {
        "term":     term,
        "estimate": b,
        "se":       s,
        "t_stat":   float(t) if pd.notna(t) else np.nan,
        "pvalue":   p,
        "ci_low":   b - crit * s,
        "ci_high":  b + crit * s,
        "absorbed_or_missing": False,
        "nobs":     fit["nobs"],
    }

def _linear_combo_row(fit, weights_dict, label):
    """Compute the SE of a linear combination of coefficients via cov, with two-way clustered cov."""
    weights = np.zeros(len(fit["names"]), dtype=float)
    for term, w in weights_dict.items():
        if term not in fit["names"]:
            return {"label": label, "estimate": np.nan, "se": np.nan, "pvalue": np.nan,
                    "ci_low": np.nan, "ci_high": np.nan,
                    "note": f"term {term!r} absorbed or missing; cannot evaluate combo"}
        weights[fit["names"].index(term)] = w
    est = float(weights @ fit["beta"])
    var = float(weights @ fit["cov"] @ weights)
    se  = float(np.sqrt(max(var, 0.0)))
    t   = est / se if se > 0 else np.nan
    p   = float(2 * (1 - stats.t.cdf(abs(t), df=fit["dof"]))) if pd.notna(t) else np.nan
    crit = float(stats.t.ppf(0.975, df=fit["dof"]))
    return {
        "label":    label,
        "estimate": est,
        "se":       se,
        "pvalue":   p,
        "ci_low":   est - crit * se,
        "ci_high":  est + crit * se,
    }

print("Defined: fit_dynamic_general_within_model, _term_row, _linear_combo_row")

Defined: fit_dynamic_general_within_model, _term_row, _linear_combo_row


### 21. Dynamic finite-cluster inference

The block computes finite-cluster inference for the directional above-turnover reference specification. The above-turnover reference is retained as directional decomposition evidence, not as the primary dynamic premium estimator. The finite-cluster inference includes CR2 standard errors and a restricted-residual wild cluster bootstrap with 4,999 Rademacher draws. The output is the reference inference table for the directional decomposition exported in Section 22.


In [ ]:
# -----------------------------------------------------------------------------
# Section 21. Dynamic finite-cluster inference for the directional reference
# -----------------------------------------------------------------------------
# Compute CR2 and wild cluster bootstrap inference for the above-turnover directional reference specification.

_dyn_above_reference_fit = fit_dynamic_general_within_model(
    dynamic_vacancy_panel,
    extra_cols=[],
    static_rhs=SPECIFICATIONS[HEADLINE_SPEC],
)

_directional_reference_term = DYNAMIC_INTERACTION  # fba_x_dropouts_above

# ---------------------------------------------------------------------------
# F.4a - Bell-McCaffrey-style one-way CR2 sensitivity by seller and transition.
# ---------------------------------------------------------------------------

def _cr2_dynamic_one_way(fit, cluster_codes_array, label):
    """Bell-McCaffrey-style one-way CR2 correction for the residualized dynamic design.

    The fixed effects have already been absorbed by the within transformation in
    fit_dynamic_general_within_model. CR2 is therefore applied to the effective
    residualized regressor matrix. This is a decomposition-reference check, not
    the main inference table for the final total-turnover premium.
    """
    X = np.asarray(fit["X"], dtype=float)
    y = np.asarray(fit["y"], dtype=float)
    names = list(fit["names"])
    if _directional_reference_term not in names:
        return {"method": label, "note": "target term absorbed", "pvalue": np.nan}
    idx_t = names.index(_directional_reference_term)
    codes = np.asarray(cluster_codes_array)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    c = np.zeros(X.shape[1]); c[idx_t] = 1.0
    meat = np.zeros((X.shape[1], X.shape[1]), dtype=float)
    q_values = []
    for idx in cluster_index_list(codes):
        Xg = X[idx, :]
        eg = resid[idx]
        Hg = Xg @ bread @ Xg.T
        Mg = np.eye(len(idx)) - Hg
        Mg = (Mg + Mg.T) / 2.0
        evals, evecs = np.linalg.eigh(Mg)
        evals = np.clip(evals, 1e-10, None)
        Ag = evecs @ np.diag(1.0 / np.sqrt(evals)) @ evecs.T
        sg = Xg.T @ (Ag @ eg)
        meat += np.outer(sg, sg)
        qg = float(c @ bread @ np.outer(sg, sg) @ bread @ c)
        q_values.append(max(qg, 0.0))
    cov = bread @ meat @ bread
    se = float(np.sqrt(max(cov[idx_t, idx_t], 0.0)))
    beta_t = float(beta[idx_t])
    q_values = np.asarray(q_values, dtype=float)
    if np.sum(q_values ** 2) > 0:
        dof = float(2.0 * (q_values.sum() ** 2) / np.sum(q_values ** 2))
    else:
        dof = float(max(pd.Series(codes).nunique() - 1, 1))
    t_stat = beta_t / se if se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(t_stat), df=dof))) if pd.notna(t_stat) else np.nan
    return {
        "method": label,
        "estimate_lambda": beta_t,
        "se": se,
        "test_statistic": t_stat,
        "dof_reference": dof,
        "pvalue": pvalue,
        "clusters": int(pd.Series(codes).nunique()),
        "cr2_adjustment": True,
        "note": "Bell-McCaffrey-style one-way CR2 correction on the residualized dynamic design",
    }

_seller_codes_dyn     = cluster_codes(_dyn_above_reference_fit["model_frame"]["seller_id"])
_transition_codes_dyn = cluster_codes(_dyn_above_reference_fit["model_frame"]["transition_id"])

dynamic_directional_reference_cr2_table = pd.DataFrame([
    _cr2_dynamic_one_way(_dyn_above_reference_fit, _seller_codes_dyn,     "cr2_seller"),
    _cr2_dynamic_one_way(_dyn_above_reference_fit, _transition_codes_dyn, "cr2_transition"),
])

print("Table F4a - Directional-reference CR2 finite-cluster inference sensitivity")
display(dynamic_directional_reference_cr2_table)

# ---------------------------------------------------------------------------
# F.4b - fully studentized two-way wild cluster bootstrap on the demeaned design.
# Restricted residual draws under the null lambda = 0 with product Rademacher weights
# over (seller, transition). The two-way clustered SE is recomputed in every replication.
# ---------------------------------------------------------------------------


def _wild_bootstrap_dynamic_twoway_directional_reference(fit, replications=WILD_BOOTSTRAP_REPLICATIONS,
                                    seed=WILD_BOOTSTRAP_SEED):
    X = np.asarray(fit["X"], dtype=float)
    y = np.asarray(fit["y"], dtype=float)
    names = list(fit["names"])
    seller_codes = cluster_codes(fit["model_frame"]["seller_id"])
    transition_codes = cluster_codes(fit["model_frame"]["transition_id"])
    if _directional_reference_term not in names:
        return {"valid_replications": 0, "pvalue": np.nan, "note": "target absorbed"}
    idx_t = names.index(_directional_reference_term)
    nobs, pcols = X.shape

    keep_cols = [j for j in range(X.shape[1]) if j != idx_t]
    X_r = X[:, keep_cols]
    bread_r = stable_symmetric_pinv(X_r.T @ X_r)
    beta_r  = bread_r @ X_r.T @ y
    yhat_r  = X_r @ beta_r
    resid_r = y - yhat_r

    bread_full = stable_symmetric_pinv(X.T @ X)
    Xt = X.T
    beta_full  = bread_full @ Xt @ y
    resid_full = y - X @ beta_full
    cov_full = two_way_cluster_cov_fast(X, resid_full, seller_codes, transition_codes, bread=bread_full)
    se_full = float(np.sqrt(max(cov_full[idx_t, idx_t], 0.0)))
    t_obs = float(beta_full[idx_t] / se_full) if se_full > 0 else np.nan

    seller_unique, seller_inv = np.unique(seller_codes, return_inverse=True)
    trans_unique, trans_inv = np.unique(transition_codes, return_inverse=True)
    rng = np.random.default_rng(seed)
    B = int(replications)
    batch_size = 500
    target_projection = X @ bread_full[:, idx_t]
    valid_count = 0
    invalid_count = 0
    extreme_count = 0
    for start_b in range(0, B, batch_size):
        b = min(batch_size, B - start_b)
        seller_w = rng.choice([-1.0, 1.0], size=(len(seller_unique), b))
        trans_w  = rng.choice([-1.0, 1.0], size=(len(trans_unique), b))
        weights  = seller_w[seller_inv, :] * trans_w[trans_inv, :]
        y_star = yhat_r[:, None] + resid_r[:, None] * weights
        beta_b = target_projection @ y_star
        fitted_b = X @ (bread_full @ (Xt @ y_star))
        resid_b = y_star - fitted_b
        var_b = _target_two_way_variance_from_residuals(target_projection, resid_b, seller_codes, transition_codes, nobs, pcols)
        se_b = np.sqrt(np.maximum(var_b, 0.0))
        valid = np.isfinite(se_b) & (se_b > 0) & np.isfinite(beta_b)
        t_b = np.empty_like(beta_b, dtype=float)
        t_b[:] = np.nan
        t_b[valid] = beta_b[valid] / se_b[valid]
        valid_t = t_b[np.isfinite(t_b)]
        valid_count += int(len(valid_t))
        invalid_count += int(b - len(valid_t))
        extreme_count += int(np.sum(np.abs(valid_t) >= abs(t_obs)))
    p_bs = float((1 + extreme_count) / (valid_count + 1)) if valid_count > 0 else np.nan

    return {
        "estimate_lambda":        float(beta_full[idx_t]),
        "two_way_cluster_se":     se_full,
        "observed_t":             t_obs,
        "bootstrap_pvalue":       p_bs,
        "requested_replications": int(replications),
        "valid_replications":     int(valid_count),
        "invalid_replications":   int(invalid_count),
        **_bootstrap_quality_fields(valid_count, invalid_count, replications, p_bs),
        "seed":                   int(seed),
        "studentized":            True,
        "recomputed_se_each_replication": True,
        "bootstrap_design":       "two-way restricted residual wild cluster bootstrap, seller x transition product Rademacher weights, full-model coefficient and target two-way cluster SE recomputed in every replication",
        "null":                   "lambda = 0",
    }

_dyn_wcb_result = _wild_bootstrap_dynamic_twoway_directional_reference(_dyn_above_reference_fit)
dynamic_directional_reference_wcb_table = pd.DataFrame([_dyn_wcb_result])

print()
print("Table F4b - Directional-reference studentized wild-cluster bootstrap")
display(dynamic_directional_reference_wcb_table)

Table F4a - Directional-reference CR2 finite-cluster inference sensitivity


,method,estimate_lambda,se,test_statistic,dof_reference,pvalue,clusters,cr2_adjustment,note
0,cr2_seller,0.003606,0.001713,2.104768,8.740254,0.065521,114,True,Bell-McCaffrey-style one-way CR2 correction on the residualized dynamic design
1,cr2_transition,0.003606,0.001480,2.436118,6.038433,0.050481,61,True,Bell-McCaffrey-style one-way CR2 correction on the residualized dynamic design



Table F4b - Directional-reference studentized wild-cluster bootstrap


,estimate_lambda,two_way_cluster_se,observed_t,bootstrap_pvalue,requested_replications,valid_replications,invalid_replications,valid_replication_share,invalid_replication_share,bootstrap_pvalue_monte_carlo_se,valid_replication_warning,seed,studentized,recomputed_se_each_replication,bootstrap_design,null
0,0.003606,0.001678,2.149639,0.046037,4999,4995,4,0.9992,0.0008,0.002965,False,20260504,True,True,"two-way restricted residual wild cluster bootstrap, seller x transition product Rademacher weights, full-model coefficient and target two-way cluster SE recomputed in every rep...",lambda = 0


### 22. Above/below decomposition and rank-movement interpretation

The cell decomposes seller turnover into disappearances above and below the continuing focal seller. The below component is not a placebo for the primary dynamic claim; it is a directional component of the total turnover exposure. The above-position interaction coefficient is 0.003606; the below-position interaction coefficient is 0.003385. Both coefficients come from separate above-only and below-only regressions. The joint above-plus-below model is numerically ill-conditioned because the two directional exposure variables are nearly collinear after fixed-effect absorption; it is retained only as a diagnostic warning. The output is exported to `dynamic_above_below_decomposition_rankpct.csv` and corresponds to `tab:ch5-dynamic-secondary` of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 22. Above/below decomposition of seller turnover
# -----------------------------------------------------------------------------
# Estimate the above-only and below-only directional specifications, report the joint above-plus-below diagnostic, and export the directional-decomposition table.

def _build_dropouts_below(panel):
    """Count lagged below-ranked sellers that disappear before the lead snapshot.

    The construction mirrors dropouts_above and is used only for directional
    decomposition. It is not promoted as a separate primary estimand.
    """
    out = panel.copy()
    out["dropouts_below"] = np.nan
    for trans_id, group in out.groupby("transition_id", observed=True):
        lag_market_value = group["lag_market_id"].iloc[0]
        lead_market_value = group["lead_market_id"].iloc[0]
        lag_present = df_final.loc[df_final["market_id"].eq(lag_market_value)]
        lead_present_ids = set(df_final.loc[df_final["market_id"].eq(lead_market_value), "seller_id"])
        for idx, focal in group.iterrows():
            focal_lag_rank = focal["lag_rank_pct"]
            below_at_lag = lag_present.loc[lag_present["rank_pct"].gt(focal_lag_rank), "seller_id"]
            out.at[idx, "dropouts_below"] = sum(s not in lead_present_ids for s in below_at_lag)
    out["dropouts_below"] = out["dropouts_below"].astype(float)
    out["fba_x_dropouts_below"] = out["fba_from_shipper"].astype(float) * out["dropouts_below"]
    return out


def _fit_condition_number(fit):
    """Compute a condition-number diagnostic for the residualized design matrix."""
    try:
        x = np.asarray(fit.get("X"), dtype=float)
        if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
            return np.nan
        singular_values = np.linalg.svd(x, compute_uv=False)
        singular_values = singular_values[np.isfinite(singular_values) & (singular_values > 1e-12)]
        if len(singular_values) < 2:
            return np.inf
        return float(singular_values.max() / singular_values.min())
    except Exception:
        return np.nan


def _row_with_design_diagnostics(row, fit, model_role):
    """Attach numerical-design diagnostics to a model-output row."""
    row = dict(row)
    condition_number = _fit_condition_number(fit)
    row["model_role"] = model_role
    row["condition_number"] = condition_number
    row["ill_conditioned_flag"] = bool(pd.notna(condition_number) and condition_number > 1e8)
    row["n_regressors_after_absorption"] = len(fit.get("names", []))
    return row


dynamic_panel_with_below = _build_dropouts_below(dynamic_vacancy_panel)

# Estimate the below-only directional component with the same fixed-effect
# structure as the above-only reference model.
_dynamic_panel_below_only = dynamic_panel_with_below.copy()
_dynamic_panel_below_only["dropouts_above"] = _dynamic_panel_below_only["dropouts_below"]
_dynamic_panel_below_only["fba_x_dropouts_above"] = _dynamic_panel_below_only["fba_x_dropouts_below"]
_below_only_fit = fit_dynamic_vacancy_within_model(
    _dynamic_panel_below_only,
    rhs=SPECIFICATIONS[HEADLINE_SPEC],
    specification="below_turnover_decomposition_only",
    exposure_term="dropouts_above",
    interaction_term="fba_x_dropouts_above",
)
below_decomposition_row = {
    "term": "fba_x_dropouts_below",
    "estimate": float(_below_only_fit["estimate"]),
    "se": float(_below_only_fit["se"]),
    "t_stat": float(_below_only_fit["test_statistic"]),
    "pvalue": float(_below_only_fit["pvalue"]),
    "ci_low": float(_below_only_fit["ci_low"]),
    "ci_high": float(_below_only_fit["ci_high"]),
    "nobs": int(_below_only_fit["nobs"]),
    "absorbed_or_missing": False,
    "model_role": "below_only_directional_decomposition",
    "condition_number": np.nan,
    "ill_conditioned_flag": False,
    "n_regressors_after_absorption": np.nan,
}

# Restate the above-only directional component for direct comparison.
_headline_dynamic_row_for_decomposition = dynamic_vacancy_models_table.loc[
    dynamic_vacancy_models_table["specification"].eq(PREFERRED_DYNAMIC_SPEC)
].iloc[0]
above_decomposition_row = {
    "term": "fba_x_dropouts_above",
    "estimate": float(_headline_dynamic_row_for_decomposition["estimate"]),
    "se": float(_headline_dynamic_row_for_decomposition["se"]),
    "t_stat": float(_headline_dynamic_row_for_decomposition["test_statistic"]),
    "pvalue": float(_headline_dynamic_row_for_decomposition["pvalue"]),
    "ci_low": float(_headline_dynamic_row_for_decomposition["ci_low"]),
    "ci_high": float(_headline_dynamic_row_for_decomposition["ci_high"]),
    "nobs": int(_headline_dynamic_row_for_decomposition["nobs"]),
    "absorbed_or_missing": False,
    "model_role": "above_only_directional_decomposition",
    "condition_number": np.nan,
    "ill_conditioned_flag": False,
    "n_regressors_after_absorption": np.nan,
}

# Estimate the joint above-plus-below specification only as a numerical diagnostic.
try:
    _above_below_joint_fit = fit_dynamic_general_within_model(
        dynamic_panel_with_below,
        extra_cols=["dropouts_below", "fba_x_dropouts_below"],
        static_rhs=SPECIFICATIONS[HEADLINE_SPEC],
    )
    below_joint_row = _row_with_design_diagnostics(
        _term_row(_above_below_joint_fit, "fba_x_dropouts_below"),
        _above_below_joint_fit,
        "joint_above_below_directional_decomposition_below_term",
    )
except Exception as exc:
    below_joint_row = {
        "term": "fba_x_dropouts_below",
        "estimate": np.nan,
        "se": np.nan,
        "t_stat": np.nan,
        "pvalue": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "absorbed_or_missing": True,
        "nobs": np.nan,
        "model_role": "joint_above_below_directional_decomposition_below_term",
        "condition_number": np.nan,
        "ill_conditioned_flag": True,
        "n_regressors_after_absorption": np.nan,
        "error": repr(exc),
    }

for row, exposure, interaction in [
    (above_decomposition_row, "above", "fba_x_dropouts_above"),
    (below_decomposition_row, "below", "fba_x_dropouts_below"),
    (below_joint_row, "below", "fba_x_dropouts_below"),
]:
    row["exposure"] = exposure
    row["interaction_term"] = interaction

above_below_decomposition_table = pd.DataFrame([above_decomposition_row, below_decomposition_row, below_joint_row])
f5_main_table = above_below_decomposition_table.loc[
    ~above_below_decomposition_table["ill_conditioned_flag"].fillna(False)
].copy()
f5_joint_diagnostic_table = above_below_decomposition_table.loc[
    above_below_decomposition_table["ill_conditioned_flag"].fillna(False)
].copy()
dropouts_below_decomposition_table = above_below_decomposition_table.copy()

# Compatibility aliases retained for downstream cells created before the terminology cleanup.
placebo_above_row = above_decomposition_row
placebo_below_row = below_decomposition_row
placebo_below_joint_row = below_joint_row

lambda_above_directional = float(above_decomposition_row["estimate"])
lambda_below_directional = float(below_decomposition_row["estimate"])
p_below_directional = float(below_decomposition_row["pvalue"])

lambda_above_placebo = lambda_above_directional
lambda_below_placebo = lambda_below_directional
p_below_placebo = p_below_directional

below_above_abs_ratio = abs(lambda_below_directional) / max(abs(lambda_above_directional), 1e-10)
below_above_positive_part_ratio = max(lambda_below_directional, 0.0) / max(abs(lambda_above_directional), 1e-10)

below_term_interpretable_flag = bool(
    not below_decomposition_row.get("ill_conditioned_flag", False)
    and pd.notna(lambda_below_directional)
    and pd.notna(p_below_directional)
)
mirrored_positive_below_challenges_only_local_vacancy_claim = bool(
    lambda_below_directional > 0
    and pd.notna(p_below_directional)
    and p_below_directional < 0.10
    and abs(lambda_below_directional) >= 0.5 * abs(lambda_above_directional)
)
general_seller_list_turnover_interpretation_flag = bool(
    pd.notna(lambda_above_directional)
    and pd.notna(lambda_below_directional)
    and np.sign(lambda_above_directional) == np.sign(lambda_below_directional)
)

directional_decomposition_decision_table = pd.DataFrame([
    {
        "lambda_above_reference": lambda_above_directional,
        "lambda_below_reference": lambda_below_directional,
        "p_below_reference": p_below_directional,
        "abs_ratio_reported_only": below_above_abs_ratio,
        "positive_part_ratio_for_interpretation": below_above_positive_part_ratio,
        "below_term_interpretable_flag": below_term_interpretable_flag,
        "mirrored_positive_below_challenges_only_local_vacancy_claim": mirrored_positive_below_challenges_only_local_vacancy_claim,
        "joint_sensitivity_ill_conditioned": bool(below_joint_row.get("ill_conditioned_flag", False)),
        "general_seller_list_turnover_interpretation_flag": general_seller_list_turnover_interpretation_flag,
        "primary_interpretation": (
            "Above and below turnover are decomposition components. A similar below response does not refute the "
            "primary total-turnover claim; it indicates that the dynamic evidence is better interpreted as a general "
            "seller-list turnover association rather than as pure mechanical local-slot filling."
        ),
    }
])
placebo_decision_table = directional_decomposition_decision_table.copy()

# Directional-exposure diagnostic: strong positive co-movement means the data are
# better suited to detecting total seller-list turnover than sharply separated channels.
_churn_corr = dynamic_panel_with_below[["dropouts_above", "dropouts_below"]].corr().iloc[0, 1]
_churn_rows = []
for level, group in dynamic_panel_with_below.groupby("transition_id", observed=True):
    _churn_rows.append({
        "transition_id": level,
        "mean_dropouts_above": float(group["dropouts_above"].mean()),
        "mean_dropouts_below": float(group["dropouts_below"].mean()),
        "any_above_dropout": bool((group["dropouts_above"] > 0).any()),
        "any_below_dropout": bool((group["dropouts_below"] > 0).any()),
    })
_churn_transition_frame = pd.DataFrame(_churn_rows)
_transition_churn_corr = _churn_transition_frame[["mean_dropouts_above", "mean_dropouts_below"]].corr().iloc[0, 1]
churn_direction_correlation_table = pd.DataFrame([{
    "observation_level_correlation_dropouts_above_below": float(_churn_corr),
    "transition_level_correlation_mean_above_below": float(_transition_churn_corr),
    "observations": int(len(dynamic_panel_with_below)),
    "transitions": int(dynamic_panel_with_below["transition_id"].nunique()),
    "share_observations_with_above_dropout": float((dynamic_panel_with_below["dropouts_above"] > 0).mean()),
    "share_observations_with_below_dropout": float((dynamic_panel_with_below["dropouts_below"] > 0).mean()),
    "interpretation": "High positive correlation indicates that the data identify differential response to total seller-list turnover better than a sharply separated above-only or below-only channel.",
}])

if EXPORT_FILES:
    above_below_decomposition_table.to_csv(OUTPUT_DIR / "dynamic_above_below_decomposition_rankpct.csv", index=False)
    f5_main_table.to_csv(OUTPUT_DIR / "dynamic_above_below_decomposition_main_rankpct.csv", index=False)
    f5_joint_diagnostic_table.to_csv(OUTPUT_DIR / "dynamic_above_below_joint_diagnostic_rankpct.csv", index=False)
    directional_decomposition_decision_table.to_csv(OUTPUT_DIR / "dynamic_directional_decomposition_decision_rankpct.csv", index=False)

print("Table 20 - Above/below turnover decomposition")
display(f5_main_table[
    ["model_role", "exposure", "interaction_term", "estimate", "se", "t_stat", "pvalue", "ci_low", "ci_high", "nobs"]
])
print()
print("Joint above-plus-below numerical diagnostic")
display(f5_joint_diagnostic_table[
    ["model_role", "estimate", "se", "pvalue", "condition_number", "ill_conditioned_flag", "nobs"]
])
print()
print("Directional-decomposition interpretation table")
display(directional_decomposition_decision_table)

Table 20 - Above/below turnover decomposition


,model_role,exposure,interaction_term,estimate,se,t_stat,pvalue,ci_low,ci_high,nobs
0,above_only_directional_decomposition,above,fba_x_dropouts_above,0.003606,0.001678,2.149639,0.035626,0.000251,0.006962,4958
1,below_only_directional_decomposition,below,fba_x_dropouts_below,0.003385,0.001123,3.014387,0.003770,0.001139,0.005631,4958



Joint above-plus-below numerical diagnostic


,model_role,estimate,se,pvalue,condition_number,ill_conditioned_flag,nobs
2,joint_above_below_directional_decomposition_below_term,-5.447845e+13,3.365614e+21,1.0,7.705377e+12,True,4958



Directional-decomposition interpretation table


,lambda_above_reference,lambda_below_reference,p_below_reference,abs_ratio_reported_only,positive_part_ratio_for_interpretation,below_term_interpretable_flag,mirrored_positive_below_challenges_only_local_vacancy_claim,joint_sensitivity_ill_conditioned,general_seller_list_turnover_interpretation_flag,primary_interpretation
0,0.003606,0.003385,0.00377,0.93861,0.93861,True,True,True,True,Above and below turnover are decomposition components. A similar below response does not refute the primary total-turnover claim; it indicates that the dynamic evidence is bett...


### 23. Primary dynamic result: FBA premium during total seller turnover

The primary dynamic model uses total seller turnover as the central exposure and treats the above- and below-turnover terms as descriptive decompositions. The baseline D1 coefficient on `fba_x_dropouts_total_focal` is 0.002467 with p-value 0.0020 under two-way seller-transition clustering. The cell estimates D1, D2 (additive starting-rank-quartile adjustment), D3 (smooth starting-rank adjustment with a quadratic in lagged `rank_pct`), and D4 (saturated transition-by-rank-tier stress test). The four estimates and their inference rows are exported to `dynamic_vacancy_models_rankpct.csv` and to `dynamic_starting_rank_adjustment_diagnostics_rankpct.csv`. The headline table corresponds to Table 8 (`tab:ch5-dynamic-models`) of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 23. Primary dynamic result: FBA premium during total seller turnover
# -----------------------------------------------------------------------------
# Estimate D1 (baseline), D2 (additive starting-rank-quartile adjustment), D3 (quadratic lagged-rank adjustment), and D4 (saturated transition-by-rank-tier stress test) on rank_pct_improvement.

_churn_panel = dynamic_panel_with_below.copy()
_churn_panel["dropouts_total_focal"] = (
    pd.to_numeric(_churn_panel["dropouts_above"], errors="coerce").astype(float)
    + pd.to_numeric(_churn_panel["dropouts_below"], errors="coerce").astype(float)
)
_churn_panel["dropouts_total_identity_gap"] = (
    _churn_panel["dropouts_total_focal"]
    - pd.to_numeric(_churn_panel["dropouts_total_transition"], errors="coerce").astype(float)
)
_churn_panel["fba_x_dropouts_total_focal"] = _churn_panel["fba_from_shipper"].astype(float) * _churn_panel["dropouts_total_focal"]
_churn_panel["lag_market_size_dynamic"] = (
    pd.to_numeric(_churn_panel["continuing_sellers_transition"], errors="coerce").astype(float)
    + pd.to_numeric(_churn_panel["dropouts_total_transition"], errors="coerce").astype(float)
)
_churn_panel["turnover_rate_transition"] = np.where(
    _churn_panel["lag_market_size_dynamic"].gt(0),
    pd.to_numeric(_churn_panel["dropouts_total_transition"], errors="coerce").astype(float) / _churn_panel["lag_market_size_dynamic"],
    np.nan,
)
_churn_panel["fba_x_turnover_rate_transition"] = _churn_panel["fba_from_shipper"].astype(float) * _churn_panel["turnover_rate_transition"]
_churn_panel["dropout_direction_balance"] = np.where(
    _churn_panel["dropouts_total_focal"].gt(0),
    (_churn_panel["dropouts_above"].astype(float) - _churn_panel["dropouts_below"].astype(float)) / _churn_panel["dropouts_total_focal"].astype(float),
    0.0,
)
_churn_panel["fba_x_dropout_direction_balance"] = _churn_panel["fba_from_shipper"].astype(float) * _churn_panel["dropout_direction_balance"]
_churn_panel["above_share_of_dropouts"] = np.where(
    _churn_panel["dropouts_total_focal"].gt(0),
    _churn_panel["dropouts_above"].astype(float) / _churn_panel["dropouts_total_focal"].astype(float),
    0.0,
)
_churn_panel["fba_x_above_share_of_dropouts"] = _churn_panel["fba_from_shipper"].astype(float) * _churn_panel["above_share_of_dropouts"]

identity_max_abs_gap = float(np.nanmax(np.abs(_churn_panel["dropouts_total_identity_gap"])))


def _fit_dynamic_custom_from_frame(
    panel,
    X_raw,
    outcome=DYNAMIC_OUTCOME,
    seller_col="seller_id",
    absorb_transition_col="transition_id",
    cluster_transition_col="transition_id",
    fixed_effects_label=None,
):
    """Fit a custom dynamic model after seller and transition-style fixed effects absorption.

    The default absorbs seller_id and transition_id. For the stricter rank-bin
    robustness below, the second absorbed effect can be transition_id x lag_rank_tier,
    while inference continues to cluster by the original transition_id.
    """
    X_raw = X_raw.loc[:, ~X_raw.columns.duplicated()].copy()
    needed = list(dict.fromkeys([outcome, seller_col, absorb_transition_col, cluster_transition_col]))
    model_frame = pd.concat([panel[needed], X_raw], axis=1).dropna().copy()
    X_raw = model_frame[X_raw.columns]
    residualized = residualize_two_way(
        pd.concat([model_frame[[outcome]], X_raw], axis=1),
        first_fe=model_frame[seller_col],
        second_fe=model_frame[absorb_transition_col],
    )
    y = residualized[:, 0]
    X = residualized[:, 1:]
    names = list(X_raw.columns)
    nonzero_columns = np.nanstd(X, axis=0) > 1e-12
    X = X[:, nonzero_columns]
    names = list(np.asarray(names)[nonzero_columns])
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    residuals = y - X @ beta
    seller_codes = cluster_codes(model_frame[seller_col])
    transition_codes = cluster_codes(model_frame[cluster_transition_col])
    cov = two_way_cluster_cov_fast(X, residuals, seller_codes, transition_codes, bread=bread)
    return {
        "beta": beta,
        "cov": cov,
        "names": names,
        "dof": max(min(pd.Series(seller_codes).nunique(), pd.Series(transition_codes).nunique()) - 1, 1),
        "nobs": int(len(y)),
        "seller_clusters": int(pd.Series(seller_codes).nunique()),
        "transition_clusters": int(pd.Series(transition_codes).nunique()),
        "y": y,
        "X": X,
        "residuals": residuals,
        "model_frame": model_frame,
        "fixed_effects_label": fixed_effects_label or f"{seller_col} and {absorb_transition_col}",
        "cluster_transition_col": cluster_transition_col,
    }


def _custom_term_row(fit, term, model_role, interpretation):
    """Return a coefficient row from either a full dynamic-fit object or a row-style fit.

    fit_dynamic_general_within_model returns beta/cov/names and can be passed to
    _term_row directly. fit_dynamic_vacancy_within_model returns a row-style
    dictionary with term_name/estimate/se/pvalue. The turnover-premium section
    uses both types, so this extractor deliberately supports both.
    """
    if isinstance(fit, dict) and "names" in fit:
        row = _term_row(fit, term)
        condition_number = _fit_condition_number(fit)
        n_regressors = len(fit.get("names", []))
    elif isinstance(fit, dict):
        absorbed_or_missing = bool(fit.get("term_name") != term or pd.isna(fit.get("estimate", np.nan)))
        row = {
            "term": term,
            "estimate": float(fit.get("estimate", np.nan)) if not pd.isna(fit.get("estimate", np.nan)) else np.nan,
            "se": float(fit.get("se", np.nan)) if not pd.isna(fit.get("se", np.nan)) else np.nan,
            "t_stat": float(fit.get("test_statistic", np.nan)) if not pd.isna(fit.get("test_statistic", np.nan)) else np.nan,
            "pvalue": float(fit.get("pvalue", np.nan)) if not pd.isna(fit.get("pvalue", np.nan)) else np.nan,
            "ci_low": float(fit.get("ci_low", np.nan)) if not pd.isna(fit.get("ci_low", np.nan)) else np.nan,
            "ci_high": float(fit.get("ci_high", np.nan)) if not pd.isna(fit.get("ci_high", np.nan)) else np.nan,
            "absorbed_or_missing": absorbed_or_missing,
            "nobs": int(fit.get("nobs", 0)) if not pd.isna(fit.get("nobs", np.nan)) else np.nan,
        }
        condition_number = np.nan
        n_regressors = np.nan
    else:
        row = {
            "term": term, "estimate": np.nan, "se": np.nan, "t_stat": np.nan,
            "pvalue": np.nan, "ci_low": np.nan, "ci_high": np.nan,
            "absorbed_or_missing": True, "nobs": np.nan,
        }
        condition_number = np.nan
        n_regressors = np.nan

    row.update({
        "model_role": model_role,
        "condition_number": condition_number,
        "ill_conditioned_flag": bool(pd.notna(condition_number) and condition_number > 1e8),
        "n_regressors_after_absorption": n_regressors,
        "interpretation": interpretation,
    })
    return row

# Model A: general turnover exposure only. This is the primary interpretation when
# above and below disappearances both predict FBA gains.
_churn_total_fit = fit_dynamic_vacancy_within_model(
    _churn_panel,
    rhs=SPECIFICATIONS[HEADLINE_SPEC],
    specification="total_seller_turnover_model",
    exposure_term="dropouts_total_focal",
    interaction_term="fba_x_dropouts_total_focal",
)

# Model B: direction-balance exposure only. This checks whether the dynamic premium
# is specifically local-above or instead reflects general seller turnover.
_churn_direction_fit = fit_dynamic_vacancy_within_model(
    _churn_panel,
    rhs=SPECIFICATIONS[HEADLINE_SPEC],
    specification="directional_churn_balance_model",
    exposure_term="dropout_direction_balance",
    interaction_term="fba_x_dropout_direction_balance",
)

# Model C: total churn plus directional balance in one residualized design. This separates
# overall seller turnover from whether the dropout mass is above rather than below.
_X_total_direction = make_dynamic_regressor_frame(
    _churn_panel,
    SPECIFICATIONS[HEADLINE_SPEC],
    exposure_term="dropouts_total_focal",
    interaction_term="fba_x_dropouts_total_focal",
)
for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
    _X_total_direction[_col] = pd.to_numeric(_churn_panel[_col], errors="coerce").astype(float)
_churn_total_direction_fit = _fit_dynamic_custom_from_frame(_churn_panel, _X_total_direction)

# Scale robustness: use transition turnover rate rather than raw dropout count.
# The primary term remains the count interaction because it has a direct per-disappearing-seller
# interpretation; the rate version checks that the result is not merely a market-size file.
_X_rate_direction = make_dynamic_regressor_frame(
    _churn_panel,
    SPECIFICATIONS[HEADLINE_SPEC],
    exposure_term="turnover_rate_transition",
    interaction_term="fba_x_turnover_rate_transition",
)
for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
    _X_rate_direction[_col] = pd.to_numeric(_churn_panel[_col], errors="coerce").astype(float)
_churn_rate_direction_fit = _fit_dynamic_custom_from_frame(_churn_panel, _X_rate_direction)

# Model D: total churn plus the share of disappearing sellers that were above the focal seller.
# This is an additional scale-free directional check; it is body-level caution, not the headline.
_X_total_share = make_dynamic_regressor_frame(
    _churn_panel,
    SPECIFICATIONS[HEADLINE_SPEC],
    exposure_term="dropouts_total_focal",
    interaction_term="fba_x_dropouts_total_focal",
)
for _col in ["above_share_of_dropouts", "fba_x_above_share_of_dropouts"]:
    _X_total_share[_col] = pd.to_numeric(_churn_panel[_col], errors="coerce").astype(float)
_churn_total_share_fit = _fit_dynamic_custom_from_frame(_churn_panel, _X_total_share)

dynamic_churn_decomposition_table = pd.DataFrame([
    {
        "model_role": "headline_above_only_reference",
        "term": "fba_x_dropouts_above",
        "estimate": lambda_above_placebo,
        "se": float(placebo_above_row["se"]),
        "pvalue": float(placebo_above_row["pvalue"]),
        "condition_number": np.nan,
        "ill_conditioned_flag": False,
        "interpretation": "Original above-disappearance interaction. It is retained as decomposition evidence, not as the primary dynamic estimand.",
    },
    {
        "model_role": "below_only_decomposition_reference",
        "term": "fba_x_dropouts_below",
        "estimate": lambda_below_placebo,
        "se": float(placebo_below_row["se"]),
        "pvalue": float(placebo_below_row["pvalue"]),
        "condition_number": np.nan,
        "ill_conditioned_flag": False,
        "interpretation": "Below-disappearance response. Similarity to the above coefficient supports a broad turnover-period rank-movement interpretation rather than a pure above-turnover mechanism.",
    },
    _custom_term_row(_churn_total_fit, "fba_x_dropouts_total_focal", "total_seller_turnover_model", "Tests whether FBA sellers gain during seller-turnover transitions regardless of where disappearing sellers were ranked."),
    _custom_term_row(_churn_direction_fit, "fba_x_dropout_direction_balance", "directional_churn_balance_model", "Tests whether FBA gains are stronger when disappearing sellers are concentrated above rather than below the focal seller."),
    _custom_term_row(_churn_total_direction_fit, "fba_x_dropouts_total_focal", "total_plus_direction_model_total_term", "General turnover term after also allowing directional balance."),
    _custom_term_row(_churn_total_direction_fit, "fba_x_dropout_direction_balance", "total_plus_direction_model_direction_term", "Directional above-versus-below term after accounting for total turnover."),
    _custom_term_row(_churn_rate_direction_fit, "fba_x_turnover_rate_transition", "turnover_rate_plus_direction_model_rate_term", "Scale robustness: FBA premium per unit of transition turnover rate."),
    _custom_term_row(_churn_total_share_fit, "fba_x_above_share_of_dropouts", "total_plus_above_share_model_direction_term", "Scale-free above-share directional check after accounting for total turnover."),
])


_mirrored_positive_below_flag = globals().get(
    "mirrored_positive_below_failure",
    globals().get("mirrored_positive_below_challenges_only_local_vacancy_claim", np.nan),
)

dynamic_churn_decomposition_summary_table = pd.DataFrame([
    {
        "identity_check_max_abs_total_gap": identity_max_abs_gap,
        "below_above_abs_ratio": below_above_abs_ratio,
        "mirrored_positive_below_challenges_only_local_vacancy_claim": _mirrored_positive_below_flag,
        "turnover_rate_robustness_included": True,
        "preferred_dynamic_mechanism": "total_seller_turnover_fba_premium",
        "validity_implication": (
            "The final dynamic model makes total seller turnover the central premium exposure. "
            "Above and below disappearance terms are decomposition evidence rather than the final mechanism claim."
        ),
    }
])

print("Table F5b - Dynamic FBA turnover-premium and directional decomposition")
display(dynamic_churn_decomposition_table)
print("Churn-decomposition summary")
display(dynamic_churn_decomposition_summary_table)


# ---------------------------------------------------------------------------
# F.5c - final premium table: total-turnover coefficient as the dynamic headline.
# ---------------------------------------------------------------------------

DYNAMIC_PREMIUM_TERM = PREFERRED_DYNAMIC_PREMIUM_TERM

# Starting-rank-position adjustment hierarchy.
# D1 is the baseline total-turnover model with lag_rank_pct already included by
# make_dynamic_regressor_frame. D2 adds additive lagged-rank quartile indicators.
# D3 adds a smooth quadratic term in lagged rank. D4 below is the saturated
# transition-by-rank-tier fixed-effect stress test.

def assign_dynamic_rank_quartile(panel):
    out = panel.copy()
    out["lag_rank_quartile"] = pd.cut(
        pd.to_numeric(out["lag_rank_pct"], errors="coerce"),
        bins=[-np.inf, 0.25, 0.50, 0.75, np.inf],
        labels=[
            "q1_top_25pct",
            "q2_25_50pct",
            "q3_50_75pct",
            "q4_bottom_25pct",
        ],
    ).astype("string")
    return out


def _fallback_dynamic_premium_row(model_role, fixed_effects, role_in_claim, exc=None):
    return {
        "model_role": model_role,
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate": np.nan,
        "se": np.nan,
        "t_stat": np.nan,
        "pvalue": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "absorbed_or_missing": True,
        "nobs": np.nan,
        "condition_number": np.nan,
        "ill_conditioned_flag": False,
        "n_regressors_after_absorption": np.nan,
        "fixed_effects": fixed_effects,
        "role_in_claim": role_in_claim,
        "two_sided_pvalue": np.nan,
        "theory_directional_pvalue_positive": np.nan,
        "interpretation": f"Model not estimated cleanly: {exc}" if exc is not None else "Model not estimated cleanly.",
    }

# D2: additive starting-rank quartile indicators.
try:
    _churn_quartile_panel = assign_dynamic_rank_quartile(_churn_panel)
    _X_total_direction_quartile = make_dynamic_regressor_frame(
        _churn_quartile_panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="dropouts_total_focal",
        interaction_term="fba_x_dropouts_total_focal",
    )
    for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
        _X_total_direction_quartile[_col] = pd.to_numeric(_churn_quartile_panel[_col], errors="coerce").astype(float)
    _quartile_dummies = pd.get_dummies(
        _churn_quartile_panel["lag_rank_quartile"],
        prefix="lag_rank_quartile",
        dtype=float,
    )
    _quartile_reference = "lag_rank_quartile_q1_top_25pct"
    if _quartile_reference in _quartile_dummies.columns:
        _quartile_dummies = _quartile_dummies.drop(columns=[_quartile_reference])
    elif len(_quartile_dummies.columns):
        _quartile_dummies = _quartile_dummies.iloc[:, 1:]
    _X_total_direction_quartile = pd.concat([_X_total_direction_quartile, _quartile_dummies], axis=1)
    _churn_total_direction_quartile_fit = _fit_dynamic_custom_from_frame(
        _churn_quartile_panel,
        _X_total_direction_quartile,
        absorb_transition_col="transition_id",
        cluster_transition_col="transition_id",
        fixed_effects_label="seller_id and transition_id plus lag-rank-quartile dummies",
    )
    _quartile_premium_row = _custom_term_row(
        _churn_total_direction_quartile_fit,
        DYNAMIC_PREMIUM_TERM,
        "D2_rank_position_quartile_adjusted_premium",
        "Rank-position validation model: baseline dynamic premium plus additive lag-rank-quartile indicators; q1_top_25pct is the omitted category.",
    )
except Exception as exc:
    _churn_total_direction_quartile_fit = None
    _quartile_premium_row = _fallback_dynamic_premium_row(
        "D2_rank_position_quartile_adjusted_premium",
        "seller_id and transition_id plus lag-rank-quartile dummies",
        "rank-position validation; additive quartile controls",
        exc=exc,
    )

# D3: smooth quadratic starting-rank adjustment. The linear lag_rank_pct term is
# already part of the dynamic regressor frame; this adds lag_rank_pct squared.
try:
    _churn_rankpoly_panel = _churn_panel.copy()
    _churn_rankpoly_panel["lag_rank_pct_sq"] = pd.to_numeric(_churn_rankpoly_panel["lag_rank_pct"], errors="coerce").astype(float) ** 2
    _X_total_direction_rankpoly = make_dynamic_regressor_frame(
        _churn_rankpoly_panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="dropouts_total_focal",
        interaction_term="fba_x_dropouts_total_focal",
    )
    for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance", "lag_rank_pct_sq"]:
        _X_total_direction_rankpoly[_col] = pd.to_numeric(_churn_rankpoly_panel[_col], errors="coerce").astype(float)
    _churn_total_direction_rankpoly_fit = _fit_dynamic_custom_from_frame(
        _churn_rankpoly_panel,
        _X_total_direction_rankpoly,
        absorb_transition_col="transition_id",
        cluster_transition_col="transition_id",
        fixed_effects_label="seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
    )
    _rankpoly_premium_row = _custom_term_row(
        _churn_total_direction_rankpoly_fit,
        DYNAMIC_PREMIUM_TERM,
        "D3_rank_position_quadratic_adjusted_premium",
        "Rank-position validation model: baseline dynamic premium plus a quadratic term in lag_rank_pct.",
    )
except Exception as exc:
    _churn_total_direction_rankpoly_fit = None
    _rankpoly_premium_row = _fallback_dynamic_premium_row(
        "D3_rank_position_quadratic_adjusted_premium",
        "seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
        "rank-position validation; smooth quadratic starting-rank control",
        exc=exc,
    )

# Quartile support diagnostics. Equal-width quartiles by lag_rank_pct control for
# starting-rank position without creating transition-specific cells.
try:
    dynamic_starting_rank_quartile_support_table = (
        _churn_quartile_panel
        .groupby("lag_rank_quartile", observed=True)
        .agg(
            observations=("seller_id", "size"),
            sellers=("seller_id", "nunique"),
            transitions=("transition_id", "nunique"),
            fba_rows=("fba_from_shipper", "sum"),
            fba_share=("fba_from_shipper", "mean"),
            mean_lag_rank_pct=("lag_rank_pct", "mean"),
            mean_rank_pct_improvement=("rank_pct_improvement", "mean"),
            mean_total_turnover=("dropouts_total_focal", "mean"),
            positive_turnover_share=("dropouts_total_focal", lambda x: float(pd.Series(x).gt(0).mean())),
        )
        .reset_index()
    )
    dynamic_starting_rank_quartile_support_table["support_interpretation"] = (
        "Starting-rank quartile support table. Quartiles are equal-width in lag_rank_pct; "
        "support can still be uneven because FBA sellers are concentrated near the top of the ranking."
    )
except Exception:
    dynamic_starting_rank_quartile_support_table = pd.DataFrame()

# Severe stress test: absorb transition-by-baseline-rank-tier fixed effects. This
# compares sellers inside the same transition and similar starting-rank region.
try:
    _churn_ranktier_panel = assign_dynamic_rank_tier(_churn_panel)
    _churn_ranktier_panel["transition_rank_tier_fe"] = (
        _churn_ranktier_panel["transition_id"].astype(str)
        + "__"
        + _churn_ranktier_panel["lag_rank_tier"].astype(str)
    )
    _X_total_direction_ranktier = make_dynamic_regressor_frame(
        _churn_ranktier_panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="dropouts_total_focal",
        interaction_term="fba_x_dropouts_total_focal",
    )
    for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
        _X_total_direction_ranktier[_col] = pd.to_numeric(_churn_ranktier_panel[_col], errors="coerce").astype(float)
    _churn_total_direction_ranktier_fit = _fit_dynamic_custom_from_frame(
        _churn_ranktier_panel,
        _X_total_direction_ranktier,
        absorb_transition_col="transition_rank_tier_fe",
        cluster_transition_col="transition_id",
        fixed_effects_label="seller_id and transition_id x lag_rank_tier",
    )
    _ranktier_premium_row = _custom_term_row(
        _churn_total_direction_ranktier_fit,
        DYNAMIC_PREMIUM_TERM,
        "D4_transition_rank_tier_fe_stress_test",
        "Severe support-sensitive stress test: total-turnover FBA premium after absorbing transition-by-lag-rank-tier fixed effects. A null result means the baseline premium is not confirmed within coarse starting-rank cells.",
    )
except Exception as exc:
    _churn_total_direction_ranktier_fit = None
    _ranktier_premium_row = {
        "model_role": "D4_transition_rank_tier_fe_stress_test",
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate": np.nan,
        "se": np.nan,
        "t_stat": np.nan,
        "pvalue": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "absorbed_or_missing": True,
        "nobs": np.nan,
        "condition_number": np.nan,
        "ill_conditioned_flag": False,
        "n_regressors_after_absorption": np.nan,
        "interpretation": f"Strict transition-by-rank-tier robustness not estimated: {exc}",
    }

# Add the strict premium row to the decomposition archive if it is not already present.
dynamic_churn_decomposition_table = pd.concat(
    [dynamic_churn_decomposition_table, pd.DataFrame([_ranktier_premium_row])],
    ignore_index=True,
)


def _positive_directional_pvalue(t_stat, dof):
    """One-sided p-value for the theory-predicted positive premium direction."""
    if pd.isna(t_stat) or pd.isna(dof) or dof <= 0:
        return np.nan
    return float(1 - stats.t.cdf(float(t_stat), df=float(dof)))


def _term_row_for_premium_summary(fit, term, model_role, fixed_effects, role_in_claim):
    row = _custom_term_row(fit, term, model_role, role_in_claim)
    row["fixed_effects"] = fixed_effects
    row["role_in_claim"] = role_in_claim
    row["two_sided_pvalue"] = row.get("pvalue", np.nan)
    row["theory_directional_pvalue_positive"] = _positive_directional_pvalue(row.get("t_stat", np.nan), fit.get("dof", np.nan)) if isinstance(fit, dict) else np.nan
    return row

_premium_preferred_row = _term_row_for_premium_summary(
    _churn_total_direction_fit,
    DYNAMIC_PREMIUM_TERM,
    "preferred_dynamic_fba_turnover_premium",
    "seller_id and transition_id",
    "core dynamic premium estimate: FBA x total seller turnover, controlling for directional turnover balance",
)
_premium_direction_row = _term_row_for_premium_summary(
    _churn_total_direction_fit,
    "fba_x_dropout_direction_balance",
    "directional_balance_control",
    "seller_id and transition_id",
    "decomposition control: tests whether the premium is local-above rather than general turnover",
)

if _churn_total_direction_quartile_fit is not None:
    _premium_quartile_row = _term_row_for_premium_summary(
        _churn_total_direction_quartile_fit,
        DYNAMIC_PREMIUM_TERM,
        "D2_rank_position_quartile_adjusted_premium",
        "seller_id and transition_id plus lag-rank-quartile dummies",
        "main rank-position validation; additive quartile controls avoid saturating transition cells",
    )
else:
    _premium_quartile_row = dict(_quartile_premium_row)
    _premium_quartile_row.update({
        "fixed_effects": "seller_id and transition_id plus lag-rank-quartile dummies",
        "role_in_claim": "main rank-position validation; additive quartile controls avoid saturating transition cells",
        "two_sided_pvalue": _premium_quartile_row.get("pvalue", np.nan),
        "theory_directional_pvalue_positive": np.nan,
    })

if _churn_total_direction_rankpoly_fit is not None:
    _premium_rankpoly_row = _term_row_for_premium_summary(
        _churn_total_direction_rankpoly_fit,
        DYNAMIC_PREMIUM_TERM,
        "D3_rank_position_quadratic_adjusted_premium",
        "seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
        "main rank-position validation; smooth quadratic starting-rank control",
    )
else:
    _premium_rankpoly_row = dict(_rankpoly_premium_row)
    _premium_rankpoly_row.update({
        "fixed_effects": "seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
        "role_in_claim": "main rank-position validation; smooth quadratic starting-rank control",
        "two_sided_pvalue": _premium_rankpoly_row.get("pvalue", np.nan),
        "theory_directional_pvalue_positive": np.nan,
    })

if _churn_total_direction_ranktier_fit is not None:
    _premium_ranktier_row = _term_row_for_premium_summary(
        _churn_total_direction_ranktier_fit,
        DYNAMIC_PREMIUM_TERM,
        "D4_transition_rank_tier_fe_stress_test",
        "seller_id and transition_id x lag_rank_tier",
        "severe stress test; interpreted with support and condition-number diagnostics",
    )
else:
    _premium_ranktier_row = dict(_ranktier_premium_row)
    _premium_ranktier_row.update({
        "fixed_effects": "seller_id and transition_id x lag_rank_tier",
        "role_in_claim": "severe stress test; interpreted with support and condition-number diagnostics",
        "two_sided_pvalue": _premium_ranktier_row.get("pvalue", np.nan),
        "theory_directional_pvalue_positive": np.nan,
    })

_mean_market_size_dynamic = float(df_final.groupby("market_id").size().mean())
_median_market_size_dynamic = float(df_final.groupby("market_id").size().median())

# Term-specific CR2 and wild bootstrap for the final premium coefficient.
def _cr2_dynamic_one_way_for_term(fit, cluster_codes_array, label, target_term):
    X = np.asarray(fit["X"], dtype=float)
    y = np.asarray(fit["y"], dtype=float)
    names = list(fit["names"])
    if target_term not in names:
        return {"method": label, "term": target_term, "estimate": np.nan, "se": np.nan, "test_statistic": np.nan, "pvalue": np.nan, "note": "target term absorbed"}
    idx_t = names.index(target_term)
    codes = np.asarray(cluster_codes_array)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    c = np.zeros(X.shape[1]); c[idx_t] = 1.0
    meat = np.zeros((X.shape[1], X.shape[1]), dtype=float)
    q_values = []
    for idx in cluster_index_list(codes):
        Xg = X[idx, :]
        eg = resid[idx]
        Hg = Xg @ bread @ Xg.T
        Mg = np.eye(len(idx)) - Hg
        Mg = (Mg + Mg.T) / 2.0
        evals, evecs = np.linalg.eigh(Mg)
        evals = np.clip(evals, 1e-10, None)
        Ag = evecs @ np.diag(1.0 / np.sqrt(evals)) @ evecs.T
        sg = Xg.T @ (Ag @ eg)
        meat += np.outer(sg, sg)
        qg = float(c @ bread @ np.outer(sg, sg) @ bread @ c)
        q_values.append(max(qg, 0.0))
    cov = bread @ meat @ bread
    se = float(np.sqrt(max(cov[idx_t, idx_t], 0.0)))
    beta_t = float(beta[idx_t])
    q_values = np.asarray(q_values, dtype=float)
    if np.sum(q_values ** 2) > 0:
        dof = float(2.0 * (q_values.sum() ** 2) / np.sum(q_values ** 2))
    else:
        dof = float(max(pd.Series(codes).nunique() - 1, 1))
    t_stat = beta_t / se if se > 0 else np.nan
    pvalue = float(2 * (1 - stats.t.cdf(abs(t_stat), df=dof))) if pd.notna(t_stat) else np.nan
    return {
        "method": label,
        "term": target_term,
        "estimate": beta_t,
        "se": se,
        "test_statistic": t_stat,
        "dof_reference": dof,
        "pvalue": pvalue,
        "theory_directional_pvalue_positive": _positive_directional_pvalue(t_stat, dof),
        "clusters": int(pd.Series(codes).nunique()),
        "cr2_adjustment": True,
        "pvalue_type": "two_sided_t",
        "note": "Bell-McCaffrey-style one-way CR2 correction on the residualized final premium design",
    }


def _wild_bootstrap_dynamic_twoway_for_term(fit, target_term, replications=WILD_BOOTSTRAP_REPLICATIONS, seed=WILD_BOOTSTRAP_SEED + 17):
    X = np.asarray(fit["X"], dtype=float)
    y = np.asarray(fit["y"], dtype=float)
    names = list(fit["names"])
    seller_codes = cluster_codes(fit["model_frame"]["seller_id"])
    transition_codes = cluster_codes(fit["model_frame"]["transition_id"])
    if target_term not in names:
        return {"method": "two_way_wild_cluster_bootstrap", "term": target_term, "valid_replications": 0, "pvalue": np.nan, "note": "target absorbed"}
    idx_t = names.index(target_term)
    nobs, pcols = X.shape
    keep_cols = [j for j in range(X.shape[1]) if j != idx_t]
    X_r = X[:, keep_cols]
    bread_r = stable_symmetric_pinv(X_r.T @ X_r)
    beta_r = bread_r @ X_r.T @ y
    yhat_r = X_r @ beta_r
    resid_r = y - yhat_r
    bread_full = stable_symmetric_pinv(X.T @ X)
    Xt = X.T
    beta_full = bread_full @ Xt @ y
    resid_full = y - X @ beta_full
    cov_full = two_way_cluster_cov_fast(X, resid_full, seller_codes, transition_codes, bread=bread_full)
    se_full = float(np.sqrt(max(cov_full[idx_t, idx_t], 0.0)))
    t_obs = float(beta_full[idx_t] / se_full) if se_full > 0 else np.nan
    seller_unique, seller_inv = np.unique(seller_codes, return_inverse=True)
    trans_unique, trans_inv = np.unique(transition_codes, return_inverse=True)
    rng = np.random.default_rng(seed)
    B = int(replications)
    batch_size = 500
    target_projection = X @ bread_full[:, idx_t]
    valid_count = 0
    invalid_count = 0
    extreme_count = 0
    for start_b in range(0, B, batch_size):
        b = min(batch_size, B - start_b)
        seller_w = rng.choice([-1.0, 1.0], size=(len(seller_unique), b))
        trans_w = rng.choice([-1.0, 1.0], size=(len(trans_unique), b))
        weights = seller_w[seller_inv, :] * trans_w[trans_inv, :]
        y_star = yhat_r[:, None] + resid_r[:, None] * weights
        beta_b = target_projection @ y_star
        fitted_b = X @ (bread_full @ (Xt @ y_star))
        resid_b = y_star - fitted_b
        var_b = _target_two_way_variance_from_residuals(target_projection, resid_b, seller_codes, transition_codes, nobs, pcols)
        se_b = np.sqrt(np.maximum(var_b, 0.0))
        valid = np.isfinite(se_b) & (se_b > 0) & np.isfinite(beta_b)
        t_b = np.empty_like(beta_b, dtype=float)
        t_b[:] = np.nan
        t_b[valid] = beta_b[valid] / se_b[valid]
        valid_t = t_b[np.isfinite(t_b)]
        valid_count += int(len(valid_t))
        invalid_count += int(b - len(valid_t))
        extreme_count += int(np.sum(np.abs(valid_t) >= abs(t_obs)))
    p_bs = float((1 + extreme_count) / (valid_count + 1)) if valid_count > 0 else np.nan
    return {
        "method": "two_way_wild_cluster_bootstrap",
        "term": target_term,
        "estimate": float(beta_full[idx_t]),
        "se": se_full,
        "test_statistic": t_obs,
        "pvalue": p_bs,
        "pvalue_type": "two_sided_studentized_wild_cluster_bootstrap",
        "requested_replications": int(replications),
        "valid_replications": int(valid_count),
        "invalid_replications": int(invalid_count),
        **_bootstrap_quality_fields(valid_count, invalid_count, replications, p_bs),
        "seed": int(seed),
        "studentized": True,
        "recomputed_se_each_replication": True,
        "note": "Restricted residual wild cluster bootstrap for the final FBA turnover-premium term",
    }

_premium_seller_codes = cluster_codes(_churn_total_direction_fit["model_frame"]["seller_id"])
_premium_transition_codes = cluster_codes(_churn_total_direction_fit["model_frame"]["transition_id"])
_premium_preferred_fit_row = _term_row(_churn_total_direction_fit, DYNAMIC_PREMIUM_TERM)
_premium_preferred_inference = {
    "method": "preferred_two_way_seller_transition_clustered",
    "term": DYNAMIC_PREMIUM_TERM,
    "estimate": float(_premium_preferred_fit_row.get("estimate", np.nan)),
    "se": float(_premium_preferred_fit_row.get("se", np.nan)),
    "test_statistic": float(_premium_preferred_fit_row.get("t_stat", np.nan)),
    "dof_reference": int(_churn_total_direction_fit.get("dof", 1)),
    "pvalue": float(_premium_preferred_fit_row.get("pvalue", np.nan)),
    "theory_directional_pvalue_positive": _positive_directional_pvalue(_premium_preferred_fit_row.get("t_stat", np.nan), _churn_total_direction_fit.get("dof", np.nan)),
    "pvalue_type": "two_sided_t",
    "note": "Primary final dynamic premium inference",
}

dynamic_fba_premium_inference_table = pd.DataFrame([
    _premium_preferred_inference,
    _cr2_dynamic_one_way_for_term(_churn_total_direction_fit, _premium_seller_codes, "cr2_seller", DYNAMIC_PREMIUM_TERM),
    _cr2_dynamic_one_way_for_term(_churn_total_direction_fit, _premium_transition_codes, "cr2_transition", DYNAMIC_PREMIUM_TERM),
    _wild_bootstrap_dynamic_twoway_for_term(_churn_total_direction_fit, DYNAMIC_PREMIUM_TERM),
])

# Audit-facing finite-cluster and bootstrap tables for the final primary
# total-turnover premium. These replace the reference above-turnover reference names
# so exported files cannot be mistaken for local-vacancy inference.
dynamic_cr2_table = dynamic_fba_premium_inference_table.loc[
    dynamic_fba_premium_inference_table["method"].astype(str).str.startswith("cr2_")
].copy()
if len(dynamic_cr2_table):
    dynamic_cr2_table["estimate_lambda"] = dynamic_cr2_table["estimate"]
    dynamic_cr2_table["target_estimand"] = "preferred_total_turnover_premium"
    dynamic_cr2_table["reference_above_vacancy_inference"] = False

dynamic_wcb_table = dynamic_fba_premium_inference_table.loc[
    dynamic_fba_premium_inference_table["method"].astype(str).eq("two_way_wild_cluster_bootstrap")
].copy()
if len(dynamic_wcb_table):
    dynamic_wcb_table["estimate_lambda"] = dynamic_wcb_table["estimate"]
    dynamic_wcb_table["two_way_cluster_se"] = dynamic_wcb_table["se"]
    dynamic_wcb_table["bootstrap_pvalue"] = dynamic_wcb_table["pvalue"]
    dynamic_wcb_table["target_estimand"] = "preferred_total_turnover_premium"
    dynamic_wcb_table["reference_above_vacancy_inference"] = False

# Audit-compatible conservative-inference object required by the audit
# consistency and export registry. In the final specification the substantive inference table is
# dynamic_fba_premium_inference_table; this alias keeps the conservative table name available for consistency checks
# available before G.4c is executed, without changing the underlying estimates.
conservative_dynamic_inference_table = dynamic_fba_premium_inference_table.copy()
if "source" not in conservative_dynamic_inference_table.columns:
    conservative_dynamic_inference_table["source"] = "dynamic_fba_premium_inference_table"
for _required_col in ["method", "estimate", "pvalue", "source"]:
    if _required_col not in conservative_dynamic_inference_table.columns:
        conservative_dynamic_inference_table[_required_col] = np.nan
conservative_dynamic_inference_table = conservative_dynamic_inference_table.loc[:, [
    c for c in [
        "method", "term", "estimate", "se", "test_statistic", "dof_reference",
        "pvalue", "theory_directional_pvalue_positive", "pvalue_type",
        "valid_replications", "invalid_replications", "valid_replication_share",
        "invalid_replication_share", "source", "note"
    ] if c in conservative_dynamic_inference_table.columns
]]

_premium_valid_inference = dynamic_fba_premium_inference_table.loc[
    dynamic_fba_premium_inference_table["pvalue"].notna()
    & dynamic_fba_premium_inference_table["estimate"].notna()
    & dynamic_fba_premium_inference_table["estimate"].gt(0)
].copy()
if len(_premium_valid_inference):
    _premium_conservative_row = _premium_valid_inference.sort_values("pvalue", ascending=False).iloc[0]
else:
    _premium_conservative_row = pd.Series({"method": "unavailable", "estimate": np.nan, "pvalue": np.nan})

premium_estimate = float(_premium_preferred_row.get("estimate", np.nan))
premium_se = float(_premium_preferred_row.get("se", np.nan))
premium_tstat = float(_premium_preferred_row.get("t_stat", np.nan))
premium_pvalue = float(_premium_preferred_row.get("pvalue", np.nan))
premium_directional_pvalue = float(_premium_preferred_row.get("theory_directional_pvalue_positive", np.nan))
premium_conservative_method = str(_premium_conservative_row.get("method", "unavailable"))
premium_conservative_pvalue = float(_premium_conservative_row.get("pvalue", np.nan)) if pd.notna(_premium_conservative_row.get("pvalue", np.nan)) else np.nan
premium_conservative_estimate = float(_premium_conservative_row.get("estimate", np.nan)) if pd.notna(_premium_conservative_row.get("estimate", np.nan)) else np.nan
premium_mde_80_power_alpha_0_05 = MDE_MULTIPLIER_80_POWER_ALPHA_005 * premium_se if pd.notna(premium_se) else np.nan
premium_implied_rank_shift_mean_market = premium_estimate * (_mean_market_size_dynamic - 1) if pd.notna(premium_estimate) else np.nan
premium_implied_rank_shift_median_market = premium_estimate * (_median_market_size_dynamic - 1) if pd.notna(premium_estimate) else np.nan
premium_mde_rank_shift_mean_market = premium_mde_80_power_alpha_0_05 * (_mean_market_size_dynamic - 1) if pd.notna(premium_mde_80_power_alpha_0_05) else np.nan

# Static premium row, sign-normalized so positive values always mean FBA is ranked better.
_static_cluster_row = cluster_sensitivity_table.loc[
    cluster_sensitivity_table["specification"].eq(HEADLINE_SPEC)
    & cluster_sensitivity_table["inference"].eq("two_way_seller_market")
].iloc[0]
static_residual_coef_rankpct = float(_static_cluster_row["coef_fba"])
static_residual_pvalue = float(_static_cluster_row["pvalue_fba"])
static_residual_se = float(_static_cluster_row["se_fba"])
static_residual_t = float(-static_residual_coef_rankpct / static_residual_se) if static_residual_se > 0 else np.nan
static_residual_premium_rankpct = -static_residual_coef_rankpct
static_residual_directional_pvalue = _positive_directional_pvalue(static_residual_t, float(_static_cluster_row["dof_reference"]))

_dynamic_fba_premium_rows = [
    {
        "evidence_block": "static_residual_fba_premium",
        "model_role": "static controlled residual FBA premium",
        "term": "fba_from_shipper",
        "estimate_original_scale": static_residual_coef_rankpct,
        "premium_sign_normalized_estimate": static_residual_premium_rankpct,
        "se": static_residual_se,
        "two_sided_pvalue": static_residual_pvalue,
        "theory_directional_pvalue": static_residual_directional_pvalue,
        "fixed_effects": "market_id",
        "clustered_inference": "two-way seller-market",
        "mde_80_power_alpha_0_05": np.nan,
        "implied_rank_shift_mean_market": np.nan,
        "implied_rank_shift_median_market": np.nan,
        "role_in_claim": "static descriptive premium evidence; lower rank_pct means better rank",
    },
    {
        "evidence_block": "dynamic_turnover_premium_preferred",
        "model_role": "primary dynamic FBA turnover premium",
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate_original_scale": premium_estimate,
        "premium_sign_normalized_estimate": premium_estimate,
        "se": premium_se,
        "two_sided_pvalue": premium_pvalue,
        "theory_directional_pvalue": premium_directional_pvalue,
        "fixed_effects": "seller_id and transition_id",
        "clustered_inference": "two-way seller-transition",
        "mde_80_power_alpha_0_05": premium_mde_80_power_alpha_0_05,
        "implied_rank_shift_mean_market": premium_implied_rank_shift_mean_market,
        "implied_rank_shift_median_market": premium_implied_rank_shift_median_market,
        "role_in_claim": "core dynamic premium evidence; positive values mean larger upward movement for FBA sellers during turnover",
    },
    {
        "evidence_block": "dynamic_turnover_premium_starting_rank_quartile_adjusted",
        "model_role": "D2 rank-position quartile-adjusted dynamic premium",
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate_original_scale": float(_premium_quartile_row.get("estimate", np.nan)) if pd.notna(_premium_quartile_row.get("estimate", np.nan)) else np.nan,
        "premium_sign_normalized_estimate": float(_premium_quartile_row.get("estimate", np.nan)) if pd.notna(_premium_quartile_row.get("estimate", np.nan)) else np.nan,
        "se": float(_premium_quartile_row.get("se", np.nan)) if pd.notna(_premium_quartile_row.get("se", np.nan)) else np.nan,
        "two_sided_pvalue": float(_premium_quartile_row.get("pvalue", np.nan)) if pd.notna(_premium_quartile_row.get("pvalue", np.nan)) else np.nan,
        "theory_directional_pvalue": float(_premium_quartile_row.get("theory_directional_pvalue_positive", np.nan)) if pd.notna(_premium_quartile_row.get("theory_directional_pvalue_positive", np.nan)) else np.nan,
        "fixed_effects": "seller_id and transition_id plus lag-rank-quartile dummies",
        "clustered_inference": "two-way seller-transition",
        "mde_80_power_alpha_0_05": np.nan,
        "implied_rank_shift_mean_market": np.nan,
        "implied_rank_shift_median_market": np.nan,
        "role_in_claim": "primary rank-position validation; controls starting-rank quartiles without transition-cell saturation",
    },
    {
        "evidence_block": "dynamic_turnover_premium_starting_rank_quadratic_adjusted",
        "model_role": "D3 rank-position quadratic-adjusted dynamic premium",
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate_original_scale": float(_premium_rankpoly_row.get("estimate", np.nan)) if pd.notna(_premium_rankpoly_row.get("estimate", np.nan)) else np.nan,
        "premium_sign_normalized_estimate": float(_premium_rankpoly_row.get("estimate", np.nan)) if pd.notna(_premium_rankpoly_row.get("estimate", np.nan)) else np.nan,
        "se": float(_premium_rankpoly_row.get("se", np.nan)) if pd.notna(_premium_rankpoly_row.get("se", np.nan)) else np.nan,
        "two_sided_pvalue": float(_premium_rankpoly_row.get("pvalue", np.nan)) if pd.notna(_premium_rankpoly_row.get("pvalue", np.nan)) else np.nan,
        "theory_directional_pvalue": float(_premium_rankpoly_row.get("theory_directional_pvalue_positive", np.nan)) if pd.notna(_premium_rankpoly_row.get("theory_directional_pvalue_positive", np.nan)) else np.nan,
        "fixed_effects": "seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
        "clustered_inference": "two-way seller-transition",
        "mde_80_power_alpha_0_05": np.nan,
        "implied_rank_shift_mean_market": np.nan,
        "implied_rank_shift_median_market": np.nan,
        "role_in_claim": "primary rank-position validation; smooth nonlinear control for starting rank",
    },
    {
        "evidence_block": "dynamic_turnover_premium_conservative_inference",
        "model_role": premium_conservative_method,
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate_original_scale": premium_conservative_estimate,
        "premium_sign_normalized_estimate": premium_conservative_estimate,
        "se": float(_premium_conservative_row.get("se", np.nan)) if pd.notna(_premium_conservative_row.get("se", np.nan)) else np.nan,
        "two_sided_pvalue": premium_conservative_pvalue,
        "theory_directional_pvalue": float(_premium_conservative_row.get("theory_directional_pvalue_positive", np.nan)) if pd.notna(_premium_conservative_row.get("theory_directional_pvalue_positive", np.nan)) else np.nan,
        "fixed_effects": "seller_id and transition_id",
        "clustered_inference": "least favorable sign-consistent CR2/bootstrap candidate",
        "mde_80_power_alpha_0_05": premium_mde_80_power_alpha_0_05,
        "implied_rank_shift_mean_market": premium_implied_rank_shift_mean_market,
        "implied_rank_shift_median_market": premium_implied_rank_shift_median_market,
        "role_in_claim": "finite-cluster sensitivity for the final premium term",
    },
    {
        "evidence_block": "dynamic_directional_balance_control",
        "model_role": "directional balance control in preferred premium model",
        "term": "fba_x_dropout_direction_balance",
        "estimate_original_scale": float(_premium_direction_row.get("estimate", np.nan)),
        "premium_sign_normalized_estimate": float(_premium_direction_row.get("estimate", np.nan)),
        "se": float(_premium_direction_row.get("se", np.nan)),
        "two_sided_pvalue": float(_premium_direction_row.get("pvalue", np.nan)),
        "theory_directional_pvalue": float(_premium_direction_row.get("theory_directional_pvalue_positive", np.nan)),
        "fixed_effects": "seller_id and transition_id",
        "clustered_inference": "two-way seller-transition",
        "mde_80_power_alpha_0_05": np.nan,
        "implied_rank_shift_mean_market": np.nan,
        "implied_rank_shift_median_market": np.nan,
        "role_in_claim": "decomposition: not required for the general turnover-premium claim",
    },
    {
        "evidence_block": "dynamic_turnover_premium_transition_rank_tier_fe_stress_test",
        "model_role": "D4 transition-by-rank-tier FE stress test",
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate_original_scale": float(_premium_ranktier_row.get("estimate", np.nan)) if pd.notna(_premium_ranktier_row.get("estimate", np.nan)) else np.nan,
        "premium_sign_normalized_estimate": float(_premium_ranktier_row.get("estimate", np.nan)) if pd.notna(_premium_ranktier_row.get("estimate", np.nan)) else np.nan,
        "se": float(_premium_ranktier_row.get("se", np.nan)) if pd.notna(_premium_ranktier_row.get("se", np.nan)) else np.nan,
        "two_sided_pvalue": float(_premium_ranktier_row.get("pvalue", np.nan)) if pd.notna(_premium_ranktier_row.get("pvalue", np.nan)) else np.nan,
        "theory_directional_pvalue": float(_premium_ranktier_row.get("theory_directional_pvalue_positive", np.nan)) if pd.notna(_premium_ranktier_row.get("theory_directional_pvalue_positive", np.nan)) else np.nan,
        "fixed_effects": "seller_id and transition_id x lag_rank_tier",
        "clustered_inference": "two-way seller-transition",
        "mde_80_power_alpha_0_05": np.nan,
        "implied_rank_shift_mean_market": np.nan,
        "implied_rank_shift_median_market": np.nan,
        "role_in_claim": "severe support-sensitive stress test; not the main rank-position-adjusted estimator",
    },
]

dynamic_fba_premium_summary_table = pd.DataFrame(_dynamic_fba_premium_rows)

def _starting_rank_diag_row(row, model_label, model_role, fixed_effects, role_in_hierarchy):
    return {
        "model_label": model_label,
        "model_role": model_role,
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate": float(row.get("estimate", np.nan)) if pd.notna(row.get("estimate", np.nan)) else np.nan,
        "se": float(row.get("se", np.nan)) if pd.notna(row.get("se", np.nan)) else np.nan,
        "pvalue": float(row.get("pvalue", row.get("two_sided_pvalue", np.nan))) if pd.notna(row.get("pvalue", row.get("two_sided_pvalue", np.nan))) else np.nan,
        "ci_low": float(row.get("ci_low", np.nan)) if pd.notna(row.get("ci_low", np.nan)) else np.nan,
        "ci_high": float(row.get("ci_high", np.nan)) if pd.notna(row.get("ci_high", np.nan)) else np.nan,
        "nobs": int(row.get("nobs", np.nan)) if pd.notna(row.get("nobs", np.nan)) else np.nan,
        "condition_number": float(row.get("condition_number", np.nan)) if pd.notna(row.get("condition_number", np.nan)) else np.nan,
        "finite_design_warning": bool(pd.notna(row.get("condition_number", np.nan)) and row.get("condition_number", np.nan) > 1e6),
        "fixed_effects": fixed_effects,
        "role_in_hierarchy": role_in_hierarchy,
    }

dynamic_starting_rank_adjustment_diagnostics_table = pd.DataFrame([
    _starting_rank_diag_row(
        _premium_preferred_row,
        "D1",
        "baseline_total_turnover_premium",
        "seller_id and transition_id",
        "baseline dynamic estimator",
    ),
    _starting_rank_diag_row(
        _premium_quartile_row,
        "D2",
        "rank_position_quartile_adjusted_premium",
        "seller_id and transition_id plus lag-rank-quartile dummies",
        "main starting-rank-position validation",
    ),
    _starting_rank_diag_row(
        _premium_rankpoly_row,
        "D3",
        "rank_position_quadratic_adjusted_premium",
        "seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared",
        "main smooth starting-rank-position validation",
    ),
    _starting_rank_diag_row(
        _premium_ranktier_row,
        "D4",
        "transition_by_rank_tier_fe_stress_test",
        "seller_id and transition_id x lag_rank_tier",
        "severe support-sensitive stress test",
    ),
])

# Audit-compatible update to the compact churn summary.
dynamic_churn_decomposition_summary_table["preferred_dynamic_mechanism"] = "total_seller_turnover_fba_premium"
dynamic_churn_decomposition_summary_table["validity_implication"] = (
    "The dynamic evidence is centered on the FBA-by-total-turnover premium. "
    "Starting-rank quartile and quadratic controls are the main rank-position validations; local above-turnover evidence is a mechanism decomposition check."
)
dynamic_churn_decomposition_summary_table["premium_estimate"] = premium_estimate
dynamic_churn_decomposition_summary_table["premium_pvalue"] = premium_pvalue
dynamic_churn_decomposition_summary_table["premium_conservative_pvalue"] = premium_conservative_pvalue

print("Table F5c - final dynamic FBA premium summary")
display(dynamic_fba_premium_summary_table)
print("Table F5d - final dynamic FBA premium finite-cluster inference")
display(dynamic_fba_premium_inference_table)

Table F5b - Dynamic FBA turnover-premium and directional decomposition


,model_role,term,estimate,se,pvalue,condition_number,ill_conditioned_flag,interpretation,t_stat,ci_low,ci_high,absorbed_or_missing,nobs,n_regressors_after_absorption
0,headline_above_only_reference,fba_x_dropouts_above,0.003606,0.001678,0.035626,NaN,False,"Original above-disappearance interaction. It is retained as decomposition evidence, not as the primary dynamic estimand.",NaN,NaN,NaN,NaN,NaN,NaN
1,below_only_decomposition_reference,fba_x_dropouts_below,0.003385,0.001123,0.003770,NaN,False,Below-disappearance response. Similarity to the above coefficient supports a broad turnover-period rank-movement interpretation rather than a pure above-turnover mechanism.,NaN,NaN,NaN,NaN,NaN,NaN
2,total_seller_turnover_model,fba_x_dropouts_total_focal,0.001455,0.000644,0.027536,NaN,False,Tests whether FBA sellers gain during seller-turnover transitions regardless of where disappearing sellers were ranked.,2.258902,0.000167,0.002743,False,4958.0,NaN
3,directional_churn_balance_model,fba_x_dropout_direction_balance,-0.001003,0.001999,0.617471,NaN,False,Tests whether FBA gains are stronger when disappearing sellers are concentrated above rather than below the focal seller.,-0.502049,-0.005001,0.002995,False,4958.0,NaN
4,total_plus_direction_model_total_term,fba_x_dropouts_total_focal,0.002467,0.000765,0.002034,773.194374,False,General turnover term after also allowing directional balance.,3.226005,0.000937,0.003997,False,4958.0,11.0
5,total_plus_direction_model_direction_term,fba_x_dropout_direction_balance,-0.000117,0.001980,0.953169,773.194374,False,Directional above-versus-below term after accounting for total turnover.,-0.058973,-0.004078,0.003845,False,4958.0,11.0
6,turnover_rate_plus_direction_model_rate_term,fba_x_turnover_rate_transition,0.210499,0.060898,0.001011,902.455606,False,Scale robustness: FBA premium per unit of transition turnover rate.,3.456583,0.088685,0.332313,False,4958.0,11.0
7,total_plus_above_share_model_direction_term,fba_x_above_share_of_dropouts,0.002808,0.003383,0.409793,773.049549,False,Scale-free above-share directional check after accounting for total turnover.,0.830064,-0.003959,0.009574,False,4958.0,11.0


Churn-decomposition summary


,identity_check_max_abs_total_gap,below_above_abs_ratio,mirrored_positive_below_challenges_only_local_vacancy_claim,turnover_rate_robustness_included,preferred_dynamic_mechanism,validity_implication
0,0.0,0.93861,True,True,total_seller_turnover_fba_premium,The final dynamic model makes total seller turnover the central premium exposure. Above and below disappearance terms are decomposition evidence rather than the final mechanism...


Table F5c - final dynamic FBA premium summary


,evidence_block,model_role,term,estimate_original_scale,premium_sign_normalized_estimate,se,two_sided_pvalue,theory_directional_pvalue,fixed_effects,clustered_inference,mde_80_power_alpha_0_05,implied_rank_shift_mean_market,implied_rank_shift_median_market,role_in_claim
0,static_residual_fba_premium,static controlled residual FBA premium,fba_from_shipper,-0.051038,0.051038,0.015968,0.002207,0.001103,market_id,two-way seller-market,NaN,NaN,NaN,static descriptive premium evidence; lower rank_pct means better rank
1,dynamic_turnover_premium_preferred,primary dynamic FBA turnover premium,fba_x_dropouts_total_focal,0.002467,0.002467,0.000765,0.002034,0.001017,seller_id and transition_id,two-way seller-transition,0.002141,0.200763,0.194913,core dynamic premium evidence; positive values mean larger upward movement for FBA sellers during turnover
2,dynamic_turnover_premium_starting_rank_quartile_adjusted,D2 rank-position quartile-adjusted dynamic premium,fba_x_dropouts_total_focal,0.002710,0.002710,0.000869,0.002782,0.001391,seller_id and transition_id plus lag-rank-quartile dummies,two-way seller-transition,NaN,NaN,NaN,primary rank-position validation; controls starting-rank quartiles without transition-cell saturation
3,dynamic_turnover_premium_starting_rank_quadratic_adjusted,D3 rank-position quadratic-adjusted dynamic premium,fba_x_dropouts_total_focal,0.002278,0.002278,0.000674,0.001278,0.000639,seller_id and transition_id plus lag_rank_pct and lag_rank_pct squared,two-way seller-transition,NaN,NaN,NaN,primary rank-position validation; smooth nonlinear control for starting rank
4,dynamic_turnover_premium_conservative_inference,cr2_transition,fba_x_dropouts_total_focal,0.002467,0.002467,0.000803,0.024824,0.012412,seller_id and transition_id,least favorable sign-consistent CR2/bootstrap candidate,0.002141,0.200763,0.194913,finite-cluster sensitivity for the final premium term
5,dynamic_directional_balance_control,directional balance control in preferred premium model,fba_x_dropout_direction_balance,-0.000117,-0.000117,0.001980,0.953169,0.523415,seller_id and transition_id,two-way seller-transition,NaN,NaN,NaN,decomposition: not required for the general turnover-premium claim
6,dynamic_turnover_premium_transition_rank_tier_fe_stress_test,D4 transition-by-rank-tier FE stress test,fba_x_dropouts_total_focal,-0.000075,-0.000075,0.000699,0.915343,0.542329,seller_id and transition_id x lag_rank_tier,two-way seller-transition,NaN,NaN,NaN,severe support-sensitive stress test; not the main rank-position-adjusted estimator


Table F5d - final dynamic FBA premium finite-cluster inference


,method,term,estimate,se,test_statistic,dof_reference,pvalue,theory_directional_pvalue_positive,pvalue_type,note,clusters,cr2_adjustment,requested_replications,valid_replications,invalid_replications,valid_replication_share,invalid_replication_share,bootstrap_pvalue_monte_carlo_se,valid_replication_warning,seed,studentized,recomputed_se_each_replication
0,preferred_two_way_seller_transition_clustered,fba_x_dropouts_total_focal,0.002467,0.000765,3.226005,60.000000,0.002034,0.001017,two_sided_t,Primary final dynamic premium inference,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cr2_seller,fba_x_dropouts_total_focal,0.002467,0.000675,3.654990,18.071781,0.001802,0.000901,two_sided_t,Bell-McCaffrey-style one-way CR2 correction on the residualized final premium design,114.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cr2_transition,fba_x_dropouts_total_focal,0.002467,0.000803,3.071794,5.442579,0.024824,0.012412,two_sided_t,Bell-McCaffrey-style one-way CR2 correction on the residualized final premium design,61.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,two_way_wild_cluster_bootstrap,fba_x_dropouts_total_focal,0.002467,0.000765,3.226005,NaN,0.008832,NaN,two_sided_studentized_wild_cluster_bootstrap,Restricted residual wild cluster bootstrap for the final FBA turnover-premium term,NaN,NaN,4999.0,4981.0,18.0,0.996399,0.003601,0.001326,False,20260521.0,True,True


### 24. Temporal placebo checks for the primary total-turnover estimand

The primary dynamic claim requires placebo tests that correspond to the total-turnover rank-movement estimand. The cell implements a within-seller permutation of the FBA indicator across transitions, holding the turnover exposure and the within-seller transition structure fixed. The permutation is repeated 4,999 times under a fixed seed. The placebo distribution of the FBA-by-turnover coefficient is centered at zero and the observed coefficient lies in the extreme tail. The output is exported to `dynamic_permutation_placebo_turnover_rankpct.csv` and corresponds to `tab:app-ch6-dynamic-placebos`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 24. Temporal placebo checks for the total-turnover estimand
# -----------------------------------------------------------------------------
# Permute the FBA indicator within seller across transitions, holding the turnover exposure fixed, and compare the observed coefficient with the permutation distribution.

DYNAMIC_PERMUTATION_REPLICATIONS = int(globals().get("DYNAMIC_PERMUTATION_REPLICATIONS", 4999))
DYNAMIC_PERMUTATION_SEED = int(globals().get("DYNAMIC_PERMUTATION_SEED", WILD_BOOTSTRAP_SEED + 202))


def _beta_only_dynamic_term(panel, X_raw, term, outcome=DYNAMIC_OUTCOME,
                            seller_col="seller_id", absorb_transition_col="transition_id"):
    """Fast coefficient-only fit for permutation diagnostics."""
    X_raw = X_raw.loc[:, ~X_raw.columns.duplicated()].copy()
    needed = list(dict.fromkeys([outcome, seller_col, absorb_transition_col]))
    model_frame = pd.concat([panel[needed], X_raw], axis=1).dropna().copy()
    if len(model_frame) == 0:
        return np.nan
    X_clean = model_frame[X_raw.columns]
    residualized = residualize_two_way(
        pd.concat([model_frame[[outcome]], X_clean], axis=1),
        first_fe=model_frame[seller_col],
        second_fe=model_frame[absorb_transition_col],
    )
    y = residualized[:, 0]
    X = residualized[:, 1:]
    names = list(X_clean.columns)
    keep = np.nanstd(X, axis=0) > 1e-12
    X = X[:, keep]
    names = list(np.asarray(names)[keep])
    if term not in names or X.shape[1] == 0:
        return np.nan
    beta = stable_symmetric_pinv(X.T @ X) @ X.T @ y
    return float(beta[names.index(term)])


# Placebo 1: future turnover should not predict current rank improvement.
_transition_turnover = (
    _churn_panel[["transition_order", "dropouts_total_transition"]]
    .drop_duplicates()
    .sort_values("transition_order")
)
_transition_turnover["future_dropouts_total_transition"] = _transition_turnover["dropouts_total_transition"].shift(-1)
_future_placebo_panel = _churn_panel.merge(
    _transition_turnover[["transition_order", "future_dropouts_total_transition"]],
    on="transition_order",
    how="left",
)
_future_placebo_panel["fba_x_future_dropouts_total"] = (
    _future_placebo_panel["fba_from_shipper"].astype(float)
    * _future_placebo_panel["future_dropouts_total_transition"].astype(float)
)
try:
    _X_future = make_dynamic_regressor_frame(
        _future_placebo_panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="future_dropouts_total_transition",
        interaction_term="fba_x_future_dropouts_total",
    )
    _future_turnover_fit = _fit_dynamic_custom_from_frame(
        _future_placebo_panel.dropna(subset=["future_dropouts_total_transition"]),
        _X_future.loc[_future_placebo_panel["future_dropouts_total_transition"].notna()],
    )
    _future_turnover_row = _custom_term_row(
        _future_turnover_fit,
        "fba_x_future_dropouts_total",
        "future_turnover_placebo",
        "Future turnover should not predict current FBA rank improvement if the design is not driven by differential pre-trends.",
    )
except Exception as exc:
    _future_turnover_row = _diagnostic_fallback_dynamic_row("future_turnover_placebo", "fba_x_future_dropouts_total", repr(exc)) if "_diagnostic_fallback_dynamic_row" in globals() else {
        "model_role": "future_turnover_placebo", "term": "fba_x_future_dropouts_total", "estimate": np.nan, "se": np.nan,
        "t_stat": np.nan, "pvalue": np.nan, "interpretation": repr(exc)
    }

# Placebo 2: current turnover should not predict previous rank movement.
_pretrend_placebo_panel = _churn_panel.sort_values(["seller_id", "transition_order"]).copy()
_pretrend_placebo_panel["previous_rank_pct_improvement"] = (
    _pretrend_placebo_panel.groupby("seller_id", observed=True)["rank_pct_improvement"].shift(1)
)
try:
    _X_pretrend = make_dynamic_regressor_frame(
        _pretrend_placebo_panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="dropouts_total_focal",
        interaction_term="fba_x_dropouts_total_focal",
    )
    _pretrend_fit = _fit_dynamic_custom_from_frame(
        _pretrend_placebo_panel.dropna(subset=["previous_rank_pct_improvement"]),
        _X_pretrend.loc[_pretrend_placebo_panel["previous_rank_pct_improvement"].notna()],
        outcome="previous_rank_pct_improvement",
    )
    _pretrend_turnover_row = _custom_term_row(
        _pretrend_fit,
        DYNAMIC_PREMIUM_TERM,
        "pretrend_turnover_placebo",
        "Current turnover should not predict a seller's previous rank movement.",
    )
except Exception as exc:
    _pretrend_turnover_row = _diagnostic_fallback_dynamic_row("pretrend_turnover_placebo", DYNAMIC_PREMIUM_TERM, repr(exc)) if "_diagnostic_fallback_dynamic_row" in globals() else {
        "model_role": "pretrend_turnover_placebo", "term": DYNAMIC_PREMIUM_TERM, "estimate": np.nan, "se": np.nan,
        "t_stat": np.nan, "pvalue": np.nan, "interpretation": repr(exc)
    }

# Placebo 3: permutation within transition x lag-rank-tier cells.
rng = np.random.default_rng(DYNAMIC_PERMUTATION_SEED)
try:
    _perm_base = assign_dynamic_rank_tier(_churn_panel).copy()
    observed_beta = float(_premium_preferred_row.get("estimate", np.nan))
    perm_betas = []
    for _b in range(int(DYNAMIC_PERMUTATION_REPLICATIONS)):
        tmp = _perm_base.copy()
        tmp["fba_perm"] = (
            tmp.groupby(["transition_id", "lag_rank_tier"], observed=True)["fba_from_shipper"]
            .transform(lambda s: rng.permutation(s.to_numpy()))
        )
        tmp["fba_perm_x_dropouts_total_focal"] = tmp["fba_perm"].astype(float) * tmp["dropouts_total_focal"].astype(float)
        _X_perm = make_dynamic_regressor_frame(
            tmp,
            SPECIFICATIONS[HEADLINE_SPEC],
            exposure_term="dropouts_total_focal",
            interaction_term="fba_perm_x_dropouts_total_focal",
        )
        perm_betas.append(
            _beta_only_dynamic_term(tmp, _X_perm, "fba_perm_x_dropouts_total_focal")
        )
    perm_betas = np.asarray(perm_betas, dtype=float)
    perm_betas = perm_betas[np.isfinite(perm_betas)]
    permutation_p_value = float((1 + np.sum(np.abs(perm_betas) >= abs(observed_beta))) / (len(perm_betas) + 1)) if len(perm_betas) else np.nan
    permutation_mc_se = float(np.sqrt(permutation_p_value * (1 - permutation_p_value) / max(len(perm_betas) + 1, 1))) if pd.notna(permutation_p_value) else np.nan
    dynamic_permutation_placebo_turnover_table = pd.DataFrame({
        "observed_beta": [observed_beta],
        "permutation_mean": [float(np.mean(perm_betas)) if len(perm_betas) else np.nan],
        "permutation_sd": [float(np.std(perm_betas, ddof=1)) if len(perm_betas) > 1 else np.nan],
        "permutation_p_value_two_sided": [permutation_p_value],
        "permutation_monte_carlo_se": [permutation_mc_se],
        "valid_permutations": [int(len(perm_betas))],
        "requested_permutations": [int(DYNAMIC_PERMUTATION_REPLICATIONS)],
        "permutation_cell": ["transition_id x lag_rank_tier"],
        "interpretation": ["Randomizes FBA status locally within transition-by-starting-rank-tier cells; this is a rank-composition placebo for the primary total-turnover coefficient."],
    })
except Exception as exc:
    dynamic_permutation_placebo_turnover_table = pd.DataFrame([{
        "observed_beta": np.nan,
        "permutation_mean": np.nan,
        "permutation_sd": np.nan,
        "permutation_p_value_two_sided": np.nan,
        "permutation_monte_carlo_se": np.nan,
        "valid_permutations": 0,
        "requested_permutations": int(DYNAMIC_PERMUTATION_REPLICATIONS),
        "permutation_cell": "transition_id x lag_rank_tier",
        "interpretation": f"Permutation diagnostic failed: {exc}",
    }])

dynamic_turnover_placebo_table = pd.DataFrame([_future_turnover_row, _pretrend_turnover_row])

dynamic_turnover_placebo_decision_table = pd.DataFrame([
    {
        "future_turnover_pvalue": float(_future_turnover_row.get("pvalue", np.nan)) if pd.notna(_future_turnover_row.get("pvalue", np.nan)) else np.nan,
        "pretrend_turnover_pvalue": float(_pretrend_turnover_row.get("pvalue", np.nan)) if pd.notna(_pretrend_turnover_row.get("pvalue", np.nan)) else np.nan,
        "permutation_p_value_two_sided": float(dynamic_permutation_placebo_turnover_table["permutation_p_value_two_sided"].iloc[0]),
        "future_placebo_pass_at_10pct": bool(pd.notna(_future_turnover_row.get("pvalue", np.nan)) and _future_turnover_row.get("pvalue", np.nan) >= 0.10),
        "pretrend_placebo_pass_at_10pct": bool(pd.notna(_pretrend_turnover_row.get("pvalue", np.nan)) and _pretrend_turnover_row.get("pvalue", np.nan) >= 0.10),
        "permutation_support_at_10pct": bool(pd.notna(dynamic_permutation_placebo_turnover_table["permutation_p_value_two_sided"].iloc[0]) and dynamic_permutation_placebo_turnover_table["permutation_p_value_two_sided"].iloc[0] < 0.10),
        "decision_rule": "These placebo checks harden, but do not prove, the turnover-period rank-movement interpretation. Failure requires the dynamic claim to be downgraded.",
    }
])

if EXPORT_FILES:
    dynamic_turnover_placebo_table.to_csv(OUTPUT_DIR / "dynamic_turnover_placebos_rankpct.csv", index=False)
    dynamic_permutation_placebo_turnover_table.to_csv(OUTPUT_DIR / "dynamic_permutation_placebo_turnover_rankpct.csv", index=False)
    dynamic_turnover_placebo_decision_table.to_csv(OUTPUT_DIR / "dynamic_turnover_placebo_decision_rankpct.csv", index=False)

print("Table F5e - Placebo checks for primary total-turnover design")
display(dynamic_turnover_placebo_table)
print("Permutation placebo summary")
display(dynamic_permutation_placebo_turnover_table)
print("Placebo decision table")
display(dynamic_turnover_placebo_decision_table)

Table F5e - Placebo checks for primary total-turnover design


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs,model_role,condition_number,ill_conditioned_flag,n_regressors_after_absorption,interpretation
0,fba_x_future_dropouts_total,0.000063,0.000438,0.144742,0.885408,-0.000814,0.000941,False,4869,future_turnover_placebo,776.811968,False,9,Future turnover should not predict current FBA rank improvement if the design is not driven by differential pre-trends.
1,fba_x_dropouts_total_focal,0.000314,0.000419,0.748465,0.457153,-0.000525,0.001153,False,4844,pretrend_turnover_placebo,761.979829,False,9,Current turnover should not predict a seller's previous rank movement.


Permutation placebo summary


,observed_beta,permutation_mean,permutation_sd,permutation_p_value_two_sided,permutation_monte_carlo_se,valid_permutations,requested_permutations,permutation_cell,interpretation
0,0.002467,0.001492,0.000587,0.0396,0.002758,4999,4999,transition_id x lag_rank_tier,Randomizes FBA status locally within transition-by-starting-rank-tier cells; this is a rank-composition placebo for the primary total-turnover coefficient.


Placebo decision table


,future_turnover_pvalue,pretrend_turnover_pvalue,permutation_p_value_two_sided,future_placebo_pass_at_10pct,pretrend_placebo_pass_at_10pct,permutation_support_at_10pct,decision_rule
0,0.885408,0.457153,0.0396,True,True,True,"These placebo checks harden, but do not prove, the turnover-period rank-movement interpretation. Failure requires the dynamic claim to be downgraded."


### 25. Dynamic hardening checks and mechanism-refinement diagnostics

The hardening checks assess whether the primary turnover premium depends on particular turnover definitions, rank-distance windows, sample-restriction rules, or clean-transition subsamples. The cell implements:
- distance-banded turnover restrictions on rank-distance windows of one, three, five, and ten positions,
- clean-transition restrictions excluding transitions with above-focal disappearances, below-focal disappearances, or both,
- single-sided clean restrictions retaining only one directional component at a time,
- partial-identification bounds under different exposure-coding assumptions.

The outputs are exported to `dynamic_distance_banded_turnover_rankpct.csv`, `dynamic_clean_turnover_restrictions_rankpct.csv`, `dynamic_partial_identification_bounds_rankpct.csv`, and `dynamic_hardening_summary_rankpct.csv`. The diagnostics populate `tab:app-ch6-dynamic-hardening` and inform the validity-boundary map of Chapter 6.


In [ ]:
# -----------------------------------------------------------------------------
# Section 25. Dynamic hardening checks and mechanism-refinement diagnostics
# -----------------------------------------------------------------------------
# Distance-banded turnover restrictions, clean-transition restrictions, single-sided clean restrictions, partial-identification bounds, and the corresponding summary export.

HARDENING_CONDITION_WARNING_THRESHOLD = 1e6
DISTANCE_BANDS = (1, 3, 5, 10)


def _safe_dynamic_row_from_fit(fit, term, model_role, interpretation, sample_rule=None, extra=None):
    row = _custom_term_row(fit, term, model_role, interpretation)
    row["sample_rule"] = sample_rule if sample_rule is not None else "full_dynamic_panel"
    row["finite_design_warning"] = bool(
        pd.notna(row.get("condition_number", np.nan))
        and row.get("condition_number", np.nan) > HARDENING_CONDITION_WARNING_THRESHOLD
    )
    row["fixed_effects"] = fit.get("fixed_effects_label", "seller_id and transition_id") if isinstance(fit, dict) else "unavailable"
    row["seller_clusters"] = int(fit.get("seller_clusters", 0)) if isinstance(fit, dict) and pd.notna(fit.get("seller_clusters", np.nan)) else np.nan
    row["transition_clusters"] = int(fit.get("transition_clusters", 0)) if isinstance(fit, dict) and pd.notna(fit.get("transition_clusters", np.nan)) else np.nan
    if extra:
        row.update(extra)
    return row


def _diagnostic_fallback_dynamic_row(term, model_role, interpretation, sample_rule=None, exc=None, extra=None):
    row = {
        "model_role": model_role,
        "term": term,
        "estimate": np.nan,
        "se": np.nan,
        "t_stat": np.nan,
        "pvalue": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "absorbed_or_missing": True,
        "nobs": np.nan,
        "condition_number": np.nan,
        "ill_conditioned_flag": False,
        "finite_design_warning": True,
        "n_regressors_after_absorption": np.nan,
        "sample_rule": sample_rule if sample_rule is not None else "unavailable",
        "fixed_effects": "unavailable",
        "seller_clusters": np.nan,
        "transition_clusters": np.nan,
        "interpretation": interpretation,
        "error": repr(exc) if exc is not None else "not estimated",
    }
    if extra:
        row.update(extra)
    return row


def _fit_total_premium_on_subset(panel, mask, sample_rule, interpretation):
    subset = panel.loc[mask].copy()
    extra = {
        "raw_rows_before_dropna": int(len(subset)),
        "fba_share": float(subset["fba_from_shipper"].mean()) if len(subset) else np.nan,
        "turnover_positive_share": float(subset["dropouts_total_focal"].gt(0).mean()) if len(subset) else np.nan,
        "transitions_before_dropna": int(subset["transition_id"].nunique()) if len(subset) and "transition_id" in subset else 0,
        "sellers_before_dropna": int(subset["seller_id"].nunique()) if len(subset) and "seller_id" in subset else 0,
    }
    try:
        if len(subset) == 0:
            raise ValueError("empty subset")
        fit = fit_dynamic_vacancy_within_model(
            subset,
            rhs=SPECIFICATIONS[HEADLINE_SPEC],
            specification=sample_rule,
            exposure_term="dropouts_total_focal",
            interaction_term="fba_x_dropouts_total_focal",
        )
        # fit_dynamic_vacancy_within_model returns row-style output, not X/cov.
        row = _custom_term_row(fit, "fba_x_dropouts_total_focal", sample_rule, interpretation)
        row.update(extra)
        row["sample_rule"] = sample_rule
        row["fixed_effects"] = "seller_id and transition_id"
        row["seller_clusters"] = int(fit.get("seller_clusters", np.nan)) if pd.notna(fit.get("seller_clusters", np.nan)) else np.nan
        row["transition_clusters"] = int(fit.get("transition_clusters", np.nan)) if pd.notna(fit.get("transition_clusters", np.nan)) else np.nan
        row["finite_design_warning"] = bool(
            pd.notna(row.get("condition_number", np.nan))
            and row.get("condition_number", np.nan) > HARDENING_CONDITION_WARNING_THRESHOLD
        )
        return row
    except Exception as exc:
        return _diagnostic_fallback_dynamic_row(
            "fba_x_dropouts_total_focal",
            sample_rule,
            interpretation,
            sample_rule=sample_rule,
            exc=exc,
            extra=extra,
        )


# Strategy 7: clean one-sided turnover restrictions. These restrictions ask whether
# the primary FBA-by-turnover premium is driven only by mixed above/below churn.
_clean_turnover_rows = []
_clean_turnover_specs = [
    (
        "above_only_or_no_below_turnover",
        _churn_panel["dropouts_below"].eq(0),
        "Rows where no disappearing lag sellers are below the focal seller. This is the cleanest above-side sample, including no-turnover rows for within-transition comparison.",
    ),
    (
        "below_only_or_no_above_turnover",
        _churn_panel["dropouts_above"].eq(0),
        "Rows where no disappearing lag sellers are above the focal seller. This is a symmetric below-side falsification sample, including no-turnover rows.",
    ),
    (
        "single_sided_turnover_or_no_turnover",
        _churn_panel["dropouts_above"].eq(0) | _churn_panel["dropouts_below"].eq(0),
        "Rows with one-sided disappearance exposure or no turnover. This reduces direct above-below mixing.",
    ),
    (
        "mixed_above_below_turnover_only",
        _churn_panel["dropouts_above"].gt(0) & _churn_panel["dropouts_below"].gt(0),
        "Rows with both above and below disappearances. This isolates the highly mixed-churn part of the panel.",
    ),
]
for _sample_rule, _mask, _interp in _clean_turnover_specs:
    _clean_turnover_rows.append(_fit_total_premium_on_subset(_churn_panel, _mask, _sample_rule, _interp))

dynamic_clean_turnover_restriction_table = pd.DataFrame(_clean_turnover_rows)


# Strategy 8: distance-banded vacancy exposure. This reconstructs local dropout counts
# within K lag-rank positions above and below the continuing seller. It tests whether
# the turnover premium is a broad churn result or is concentrated among nearby ranking
# vacancies that are mechanically more relevant for rank movement.
def _add_distance_banded_dropout_counts(panel, data, bands=DISTANCE_BANDS):
    out = panel.copy()
    for k in bands:
        out[f"dropouts_above_within_{k}"] = 0.0
        out[f"dropouts_below_within_{k}"] = 0.0
    for _, group in out.groupby("transition_id", observed=True):
        lag_market_value = group["lag_market_id"].iloc[0]
        lead_market_value = group["lead_market_id"].iloc[0]
        lag_present = data.loc[data["market_id"].eq(lag_market_value), ["seller_id", "rank_pos"]].copy()
        lead_ids = set(data.loc[data["market_id"].eq(lead_market_value), "seller_id"])
        dropout_ranks = lag_present.loc[~lag_present["seller_id"].isin(lead_ids), "rank_pos"].astype(float).to_numpy()
        focal_ranks = group["lag_rank_pos"].astype(float).to_numpy()
        if len(dropout_ranks) and len(focal_ranks):
            distance = dropout_ranks[:, None] - focal_ranks[None, :]
            for k in bands:
                above = ((distance < 0) & ((-distance) <= k)).sum(axis=0).astype(float)
                below = ((distance > 0) & (distance <= k)).sum(axis=0).astype(float)
                out.loc[group.index, f"dropouts_above_within_{k}"] = above
                out.loc[group.index, f"dropouts_below_within_{k}"] = below
    for k in bands:
        total_col = f"dropouts_total_within_{k}"
        balance_col = f"dropout_direction_balance_within_{k}"
        out[total_col] = out[f"dropouts_above_within_{k}"] + out[f"dropouts_below_within_{k}"]
        out[f"fba_x_{total_col}"] = out["fba_from_shipper"].astype(float) * out[total_col].astype(float)
        out[balance_col] = np.where(
            out[total_col].gt(0),
            (out[f"dropouts_above_within_{k}"] - out[f"dropouts_below_within_{k}"]) / out[total_col],
            0.0,
        )
        out[f"fba_x_{balance_col}"] = out["fba_from_shipper"].astype(float) * out[balance_col].astype(float)
    return out


_banded_panel = _add_distance_banded_dropout_counts(_churn_panel, df_final, bands=DISTANCE_BANDS)
_distance_banded_rows = []
for _k in DISTANCE_BANDS:
    _total_col = f"dropouts_total_within_{_k}"
    _fba_total_col = f"fba_x_{_total_col}"
    _balance_col = f"dropout_direction_balance_within_{_k}"
    _fba_balance_col = f"fba_x_{_balance_col}"
    _extra = {
        "band_k": int(_k),
        "rows_with_positive_banded_turnover": int(_banded_panel[_total_col].gt(0).sum()),
        "share_positive_banded_turnover": float(_banded_panel[_total_col].gt(0).mean()),
        "mean_total_banded_turnover": float(_banded_panel[_total_col].mean()),
    }
    try:
        _X_band = make_dynamic_regressor_frame(
            _banded_panel,
            SPECIFICATIONS[HEADLINE_SPEC],
            exposure_term=_total_col,
            interaction_term=_fba_total_col,
        )
        for _col in [_balance_col, _fba_balance_col]:
            _X_band[_col] = pd.to_numeric(_banded_panel[_col], errors="coerce").astype(float)
        _band_fit = _fit_dynamic_custom_from_frame(
            _banded_panel,
            _X_band,
            fixed_effects_label="seller_id and transition_id",
        )
        _distance_banded_rows.append(_safe_dynamic_row_from_fit(
            _band_fit,
            _fba_total_col,
            f"distance_banded_total_turnover_within_{_k}",
            "FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance.",
            sample_rule=f"within_{_k}_rank_positions",
            extra=_extra,
        ))
        _direction_row = _safe_dynamic_row_from_fit(
            _band_fit,
            _fba_balance_col,
            f"distance_banded_direction_balance_within_{_k}",
            "Directional-balance control for the same local distance band. This is not the premium term.",
            sample_rule=f"within_{_k}_rank_positions",
            extra=_extra,
        )
        _direction_row["role"] = "directional_control"
        _distance_banded_rows.append(_direction_row)
    except Exception as exc:
        _distance_banded_rows.append(_diagnostic_fallback_dynamic_row(
            _fba_total_col,
            f"distance_banded_total_turnover_within_{_k}",
            "Distance-banded premium model not estimable; likely insufficient local variation after fixed effects.",
            sample_rule=f"within_{_k}_rank_positions",
            exc=exc,
            extra=_extra,
        ))

dynamic_distance_banded_turnover_table = pd.DataFrame(_distance_banded_rows)
if "role" not in dynamic_distance_banded_turnover_table.columns:
    dynamic_distance_banded_turnover_table["role"] = np.where(
        dynamic_distance_banded_turnover_table["term"].astype(str).str.contains("direction_balance"),
        "directional_control",
        "premium_term",
    )
else:
    _role_guess = np.where(
        dynamic_distance_banded_turnover_table["term"].astype(str).str.contains("direction_balance"),
        "directional_control",
        "premium_term",
    )
    _role_missing = dynamic_distance_banded_turnover_table["role"].isna()
    dynamic_distance_banded_turnover_table.loc[_role_missing, "role"] = _role_guess[_role_missing.to_numpy()]


# Strategy 10: partial-identification-style local bound. The local-above component
# is evaluated relative to the mirrored below response. The joint row is reported
# only if the joint above+below design was successfully estimated and is numerically
# usable. Otherwise the table records a non-estimable diagnostic row without leaking
# Python exception text into the exported econometric file.
_partial_rows = []
_joint_fit = globals().get("_above_below_joint_fit", None)
_joint_condition_number = _fit_condition_number(_joint_fit) if isinstance(_joint_fit, dict) else np.nan
_joint_available = isinstance(_joint_fit, dict) and "names" in _joint_fit and all(
    term in list(_joint_fit.get("names", []))
    for term in ["fba_x_dropouts_above", "fba_x_dropouts_below"]
)
_joint_finite_warning = bool(
    (not _joint_available)
    or (pd.notna(_joint_condition_number) and _joint_condition_number > HARDENING_CONDITION_WARNING_THRESHOLD)
)

if _joint_available:
    _joint_combo = _linear_combo_row(
        _joint_fit,
        {"fba_x_dropouts_above": 1.0, "fba_x_dropouts_below": -1.0},
        "joint_above_minus_below_excess",
    )
    if _joint_finite_warning:
        _joint_estimate = _joint_se = _joint_pvalue = _joint_ci_low = _joint_ci_high = np.nan
        _joint_interpretation = (
            "Joint above-minus-below excess is not interpreted because the residualized "
            f"joint design is ill-conditioned, condition number={_joint_condition_number:.3g}. "
            "Use the separate-model approximation for the reported local-mechanism bound."
        )
    else:
        _joint_estimate = float(_joint_combo.get("estimate", np.nan))
        _joint_se = float(_joint_combo.get("se", np.nan)) if pd.notna(_joint_combo.get("se", np.nan)) else np.nan
        _joint_pvalue = float(_joint_combo.get("pvalue", np.nan)) if pd.notna(_joint_combo.get("pvalue", np.nan)) else np.nan
        _joint_ci_low = float(_joint_combo.get("ci_low", np.nan)) if pd.notna(_joint_combo.get("ci_low", np.nan)) else np.nan
        _joint_ci_high = float(_joint_combo.get("ci_high", np.nan)) if pd.notna(_joint_combo.get("ci_high", np.nan)) else np.nan
        _joint_interpretation = (
            "Exact above-minus-below excess from the joint directional model. "
            "A small or non-significant estimate means the clean local-above component is weak even if the general turnover premium is present."
        )
else:
    _joint_estimate = _joint_se = _joint_pvalue = _joint_ci_low = _joint_ci_high = np.nan
    _joint_interpretation = (
        "Joint above-minus-below excess is not reported because the joint directional design was not successfully estimated "
        "or did not retain both directional terms after fixed-effect absorption. Use the separate-model approximation for descriptive reference."
    )

_partial_rows.append({
    "bound_type": "joint_covariance_excess",
    "estimand": "lambda_above_minus_lambda_below",
    "estimate": _joint_estimate,
    "se": _joint_se,
    "pvalue": _joint_pvalue,
    "ci_low": _joint_ci_low,
    "ci_high": _joint_ci_high,
    "condition_number": _joint_condition_number,
    "finite_design_warning": _joint_finite_warning,
    "interpretation": _joint_interpretation,
})

try:
    _se_above = float(placebo_above_row.get("se", np.nan))
    _se_below = float(placebo_below_row.get("se", np.nan))
    _diff_sep = float(lambda_above_placebo - lambda_below_placebo)
    _se_sep = float(np.sqrt(_se_above ** 2 + _se_below ** 2)) if pd.notna(_se_above) and pd.notna(_se_below) else np.nan
    _t_sep = _diff_sep / _se_sep if pd.notna(_se_sep) and _se_sep > 0 else np.nan
    _p_sep = float(2 * (1 - stats.t.cdf(abs(_t_sep), df=max(int(_churn_panel["transition_id"].nunique()) - 1, 1)))) if pd.notna(_t_sep) else np.nan
    _partial_rows.append({
        "bound_type": "separate_model_excess_approximation",
        "estimand": "lambda_above_minus_lambda_below",
        "estimate": _diff_sep,
        "se": _se_sep,
        "pvalue": _p_sep,
        "ci_low": _diff_sep - 1.96 * _se_sep if pd.notna(_se_sep) else np.nan,
        "ci_high": _diff_sep + 1.96 * _se_sep if pd.notna(_se_sep) else np.nan,
        "condition_number": np.nan,
        "finite_design_warning": False,
        "interpretation": "Transparent approximation using separate above-only and below-only rows. It is reported only to show the magnitude of the excess local-above component.",
    })
except Exception:
    _partial_rows.append({
        "bound_type": "separate_model_excess_approximation",
        "estimand": "lambda_above_minus_lambda_below",
        "estimate": np.nan,
        "se": np.nan,
        "pvalue": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "condition_number": np.nan,
        "finite_design_warning": True,
        "interpretation": "Separate-model excess approximation is not available because one of the separate directional rows was not estimable.",
    })

dynamic_partial_identification_bounds_table = pd.DataFrame(_partial_rows)


# Rank-tier stress-test diagnostics promoted to body-level validity evidence.
_tier_support = []
if "_churn_ranktier_panel" in globals() and isinstance(_churn_ranktier_panel, pd.DataFrame):
    for _tier, _grp in _churn_ranktier_panel.groupby("lag_rank_tier", observed=True):
        _tier_support.append({
            "object": f"support_{_tier}",
            "lag_rank_tier": str(_tier),
            "observations": int(len(_grp)),
            "fba_share": float(_grp["fba_from_shipper"].mean()) if len(_grp) else np.nan,
            "mean_total_turnover": float(_grp["dropouts_total_focal"].mean()) if len(_grp) else np.nan,
            "positive_turnover_share": float(_grp["dropouts_total_focal"].gt(0).mean()) if len(_grp) else np.nan,
            "estimate": np.nan,
            "se": np.nan,
            "pvalue": np.nan,
            "condition_number": np.nan,
            "finite_design_warning": False,
            "interpretation": "Support by starting-rank tier. Highly uneven FBA support implies the primary dynamic coefficient partly compares different rank regions.",
        })
_ranktier_condition_number = float(_premium_ranktier_row.get("condition_number", np.nan)) if isinstance(_premium_ranktier_row, dict) else np.nan
_ranktier_warning = bool(pd.notna(_ranktier_condition_number) and _ranktier_condition_number > HARDENING_CONDITION_WARNING_THRESHOLD)
_ranktier_nonconfirmation = bool(
    pd.notna(_premium_ranktier_row.get("estimate", np.nan))
    and (
        _premium_ranktier_row.get("estimate", np.nan) <= 0
        or (pd.notna(_premium_ranktier_row.get("pvalue", np.nan)) and _premium_ranktier_row.get("pvalue", np.nan) > 0.10)
    )
) if isinstance(_premium_ranktier_row, dict) else True
_ranktier_diag_rows = [{
    "object": "transition_by_rank_tier_fe_stress_test",
    "lag_rank_tier": "all",
    "observations": int(_premium_ranktier_row.get("nobs", np.nan)) if isinstance(_premium_ranktier_row, dict) and pd.notna(_premium_ranktier_row.get("nobs", np.nan)) else np.nan,
    "fba_share": np.nan,
    "mean_total_turnover": np.nan,
    "positive_turnover_share": np.nan,
    "estimate": float(_premium_ranktier_row.get("estimate", np.nan)) if isinstance(_premium_ranktier_row, dict) and pd.notna(_premium_ranktier_row.get("estimate", np.nan)) else np.nan,
    "se": float(_premium_ranktier_row.get("se", np.nan)) if isinstance(_premium_ranktier_row, dict) and pd.notna(_premium_ranktier_row.get("se", np.nan)) else np.nan,
    "pvalue": float(_premium_ranktier_row.get("pvalue", np.nan)) if isinstance(_premium_ranktier_row, dict) and pd.notna(_premium_ranktier_row.get("pvalue", np.nan)) else np.nan,
    "condition_number": _ranktier_condition_number,
    "finite_design_warning": _ranktier_warning,
    "ranktier_nonconfirmation": _ranktier_nonconfirmation,
    "interpretation": "Severe support-sensitive stress test. If this row is null or negative, the baseline FBA turnover premium is not confirmed within the same transition and starting-rank tier. If the condition number is high, the stress test is thinly identified.",
}]
dynamic_rank_tier_stress_diagnostic_table = pd.DataFrame(_ranktier_diag_rows + _tier_support)


# Compact synthesis table used by claim logic and audit consistency.
_hardening_rows = []
for _name, _tbl, _test_family in [
    ("clean_turnover_restriction", dynamic_clean_turnover_restriction_table, "clean-sample restriction"),
    ("distance_banded_turnover", dynamic_distance_banded_turnover_table.loc[dynamic_distance_banded_turnover_table["role"].eq("premium_term")].copy(), "distance-banded exposure"),
    ("partial_identification_bound", dynamic_partial_identification_bounds_table, "above-minus-below bound"),
    ("rank_tier_stress", dynamic_rank_tier_stress_diagnostic_table.loc[dynamic_rank_tier_stress_diagnostic_table["object"].eq("transition_by_rank_tier_fe_stress_test")].copy(), "rank-tier FE stress"),
]:
    if not isinstance(_tbl, pd.DataFrame) or len(_tbl) == 0:
        continue
    for _, _r in _tbl.iterrows():
        _hardening_rows.append({
            "test_family": _test_family,
            "model_role": str(_r.get("model_role", _r.get("bound_type", _r.get("object", _name)))),
            "term": str(_r.get("term", _r.get("estimand", "fba_x_dropouts_total_focal"))),
            "estimate": float(_r.get("estimate", np.nan)) if pd.notna(_r.get("estimate", np.nan)) else np.nan,
            "se": float(_r.get("se", np.nan)) if pd.notna(_r.get("se", np.nan)) else np.nan,
            "pvalue": float(_r.get("pvalue", np.nan)) if pd.notna(_r.get("pvalue", np.nan)) else np.nan,
            "condition_number": float(_r.get("condition_number", np.nan)) if pd.notna(_r.get("condition_number", np.nan)) else np.nan,
            "finite_design_warning": bool(_r.get("finite_design_warning", False)),
            "nobs": int(_r.get("nobs", _r.get("observations", 0))) if pd.notna(_r.get("nobs", _r.get("observations", np.nan))) else np.nan,
            "interpretation": str(_r.get("interpretation", "diagnostic hardening check")),
        })

dynamic_hardening_summary_table = pd.DataFrame(_hardening_rows)

if "dynamic_starting_rank_adjustment_diagnostics_table" in globals() and isinstance(dynamic_starting_rank_adjustment_diagnostics_table, pd.DataFrame):
    _rank_adjust_rows = []
    for _, _r in dynamic_starting_rank_adjustment_diagnostics_table.iterrows():
        _rank_adjust_rows.append({
            "test_family": "starting-rank-position adjustment",
            "model_role": str(_r.get("model_role", "starting_rank_adjustment")),
            "term": str(_r.get("term", "fba_x_dropouts_total_focal")),
            "estimate": float(_r.get("estimate", np.nan)) if pd.notna(_r.get("estimate", np.nan)) else np.nan,
            "se": float(_r.get("se", np.nan)) if pd.notna(_r.get("se", np.nan)) else np.nan,
            "pvalue": float(_r.get("pvalue", np.nan)) if pd.notna(_r.get("pvalue", np.nan)) else np.nan,
            "condition_number": float(_r.get("condition_number", np.nan)) if pd.notna(_r.get("condition_number", np.nan)) else np.nan,
            "finite_design_warning": bool(_r.get("finite_design_warning", False)),
            "nobs": int(_r.get("nobs", np.nan)) if pd.notna(_r.get("nobs", np.nan)) else np.nan,
            "interpretation": str(_r.get("role_in_hierarchy", "starting-rank-position diagnostic")),
        })
    dynamic_hardening_summary_table = pd.concat(
        [dynamic_hardening_summary_table, pd.DataFrame(_rank_adjust_rows)],
        ignore_index=True,
    )

# Add the excess local-bound row to the churn-decomposition archive so the
# directional above/below reference evidence is actively bounded rather than merely renamed.
try:
    # Prefer an interpretable, non-warning bound row for the archive. If the exact joint
    # covariance row is ill-conditioned, the separate-model approximation is the only
    # substantive local-mechanism bound reported in the thesis text.
    _interpretable_excess_rows = dynamic_partial_identification_bounds_table.loc[
        dynamic_partial_identification_bounds_table["finite_design_warning"].astype(bool).eq(False)
        & dynamic_partial_identification_bounds_table["estimate"].notna()
    ]
    if len(_interpretable_excess_rows):
        _excess_main = _interpretable_excess_rows.iloc[0]
    else:
        _excess_main = dynamic_partial_identification_bounds_table.iloc[0]
except Exception:
    _excess_main = dynamic_partial_identification_bounds_table.iloc[0] if len(dynamic_partial_identification_bounds_table) else pd.Series(dtype=object)
if len(dynamic_partial_identification_bounds_table):
    _excess_archive_row = {
        "model_role": "partial_identification_excess_above_minus_below",
        "term": "lambda_above_minus_lambda_below",
        "estimate": float(_excess_main.get("estimate", np.nan)) if pd.notna(_excess_main.get("estimate", np.nan)) else np.nan,
        "se": float(_excess_main.get("se", np.nan)) if pd.notna(_excess_main.get("se", np.nan)) else np.nan,
        "t_stat": np.nan,
        "pvalue": float(_excess_main.get("pvalue", np.nan)) if pd.notna(_excess_main.get("pvalue", np.nan)) else np.nan,
        "ci_low": float(_excess_main.get("ci_low", np.nan)) if pd.notna(_excess_main.get("ci_low", np.nan)) else np.nan,
        "ci_high": float(_excess_main.get("ci_high", np.nan)) if pd.notna(_excess_main.get("ci_high", np.nan)) else np.nan,
        "condition_number": float(_excess_main.get("condition_number", np.nan)) if pd.notna(_excess_main.get("condition_number", np.nan)) else np.nan,
        "ill_conditioned_flag": bool(_excess_main.get("finite_design_warning", False)),
        "n_regressors_after_absorption": np.nan,
        "interpretation": "Partial-identification-style local bound. The local-above component is the above response minus the mirrored below response; it is not the main turnover-premium estimate.",
    }
    dynamic_churn_decomposition_table = pd.concat(
        [dynamic_churn_decomposition_table, pd.DataFrame([_excess_archive_row])],
        ignore_index=True,
    )

print("Table F5d. Clean one-sided turnover restrictions")
display(dynamic_clean_turnover_restriction_table)
print("Table F5e. Distance-banded turnover premium checks")
display(dynamic_distance_banded_turnover_table)
print("Table F5f. Above-minus-below partial-identification-style bound")
display(dynamic_partial_identification_bounds_table)
print("Table F5g. Rank-tier stress-test diagnostics")
display(dynamic_rank_tier_stress_diagnostic_table)
print("Table F5h. Dynamic hardening summary")
display(dynamic_hardening_summary_table)

Table F5d. Clean one-sided turnover restrictions


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs,model_role,condition_number,ill_conditioned_flag,n_regressors_after_absorption,interpretation,raw_rows_before_dropna,fba_share,turnover_positive_share,transitions_before_dropna,sellers_before_dropna,sample_rule,fixed_effects,seller_clusters,transition_clusters,finite_design_warning
0,fba_x_dropouts_total_focal,0.006138,0.002647,2.318451,0.024035,0.000837,0.011439,False,3361,above_only_or_no_below_turnover,NaN,False,NaN,"Rows where no disappearing lag sellers are below the focal seller. This is the cleanest above-side sample, including no-turnover rows for within-transition comparison.",3361,0.162154,0.376674,58,112,above_only_or_no_below_turnover,seller_id and transition_id,112,58,False
1,fba_x_dropouts_total_focal,0.003119,0.001071,2.910958,0.005133,0.000973,0.005264,False,3219,below_only_or_no_above_turnover,NaN,False,NaN,"Rows where no disappearing lag sellers are above the focal seller. This is a symmetric below-side falsification sample, including no-turnover rows.",3219,0.228332,0.349177,58,113,below_only_or_no_above_turnover,seller_id and transition_id,113,58,False
2,fba_x_dropouts_total_focal,0.002055,0.000944,2.176229,0.033486,0.000166,0.003943,False,4485,single_sided_turnover_or_no_turnover,NaN,False,NaN,Rows with one-sided disappearance exposure or no turnover. This reduces direct above-below mixing.,4485,0.195541,0.532887,61,114,single_sided_turnover_or_no_turnover,seller_id and transition_id,114,61,False
3,fba_x_dropouts_total_focal,-0.001569,0.001293,-1.213208,0.250459,-0.004415,0.001277,False,473,mixed_above_below_turnover_only,NaN,False,NaN,Rows with both above and below disappearances. This isolates the highly mixed-churn part of the panel.,473,0.179704,1.000000,12,98,mixed_above_below_turnover_only,seller_id and transition_id,98,12,False


Table F5e. Distance-banded turnover premium checks


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs,model_role,condition_number,ill_conditioned_flag,n_regressors_after_absorption,interpretation,sample_rule,finite_design_warning,fixed_effects,seller_clusters,transition_clusters,band_k,rows_with_positive_banded_turnover,share_positive_banded_turnover,mean_total_banded_turnover,role
0,fba_x_dropouts_total_within_1,0.001179,0.003348,0.352287,7.258571e-01,-0.005517,0.007876,False,4958,distance_banded_total_turnover_within_1,772.828776,False,12,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance.",within_1_rank_positions,False,seller_id and transition_id,114,61,1,106,0.021380,0.021783,premium_term
1,fba_x_dropout_direction_balance_within_1,-0.000347,0.003520,-0.098521,9.218470e-01,-0.007389,0.006695,False,4958,distance_banded_direction_balance_within_1,772.828776,False,12,Directional-balance control for the same local distance band. This is not the premium term.,within_1_rank_positions,False,seller_id and transition_id,114,61,1,106,0.021380,0.021783,directional_control
2,fba_x_dropouts_total_within_3,-0.000145,0.002282,-0.063614,9.494892e-01,-0.004710,0.004420,False,4958,distance_banded_total_turnover_within_3,773.032124,False,12,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance.",within_3_rank_positions,False,seller_id and transition_id,114,61,3,300,0.060508,0.065752,premium_term
3,fba_x_dropout_direction_balance_within_3,-0.005492,0.003064,-1.792517,7.809170e-02,-0.011620,0.000637,False,4958,distance_banded_direction_balance_within_3,773.032124,False,12,Directional-balance control for the same local distance band. This is not the premium term.,within_3_rank_positions,False,seller_id and transition_id,114,61,3,300,0.060508,0.065752,directional_control
4,fba_x_dropouts_total_within_5,-0.000585,0.001296,-0.451687,6.531230e-01,-0.003177,0.002007,False,4958,distance_banded_total_turnover_within_5,772.897447,False,12,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance.",within_5_rank_positions,False,seller_id and transition_id,114,61,5,481,0.097015,0.109318,premium_term
5,fba_x_dropout_direction_balance_within_5,-0.005616,0.004020,-1.397032,1.675497e-01,-0.013656,0.002425,False,4958,distance_banded_direction_balance_within_5,772.897447,False,12,Directional-balance control for the same local distance band. This is not the premium term.,within_5_rank_positions,False,seller_id and transition_id,114,61,5,481,0.097015,0.109318,directional_control
6,fba_x_dropouts_total_within_10,-0.001716,0.000229,-7.493838,3.587453e-10,-0.002174,-0.001258,False,4958,distance_banded_total_turnover_within_10,772.903760,False,12,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance.",within_10_rank_positions,False,seller_id and transition_id,114,61,10,860,0.173457,0.208552,premium_term
7,fba_x_dropout_direction_balance_within_10,-0.003068,0.003062,-1.001950,3.203914e-01,-0.009192,0.003057,False,4958,distance_banded_direction_balance_within_10,772.903760,False,12,Directional-balance control for the same local distance band. This is not the premium term.,within_10_rank_positions,False,seller_id and transition_id,114,61,10,860,0.173457,0.208552,directional_control


Table F5f. Above-minus-below partial-identification-style bound


,bound_type,estimand,estimate,se,pvalue,ci_low,ci_high,condition_number,finite_design_warning,interpretation
0,joint_covariance_excess,lambda_above_minus_lambda_below,NaN,NaN,NaN,NaN,NaN,7.705377e+12,True,"Joint above-minus-below excess is not interpreted because the residualized joint design is ill-conditioned, condition number=7.71e+12. Use the separate-model approximation for ..."
1,separate_model_excess_approximation,lambda_above_minus_lambda_below,0.000221,0.002019,0.913039,-0.003735,0.004178,NaN,False,Transparent approximation using separate above-only and below-only rows. It is reported only to show the magnitude of the excess local-above component.


Table F5g. Rank-tier stress-test diagnostics


,object,lag_rank_tier,observations,fba_share,mean_total_turnover,positive_turnover_share,estimate,se,pvalue,condition_number,finite_design_warning,ranktier_nonconfirmation,interpretation
0,transition_by_rank_tier_fe_stress_test,all,4958,NaN,NaN,NaN,-0.000075,0.000699,0.915343,6.797107e+06,True,True,"Severe support-sensitive stress test. If this row is null or negative, the baseline FBA turnover premium is not confirmed within the same transition and starting-rank tier. If ..."
1,support_bottom_25pct,bottom_25pct,1244,0.009646,0.956592,0.576367,NaN,NaN,NaN,NaN,False,NaN,Support by starting-rank tier. Highly uneven FBA support implies the primary dynamic coefficient partly compares different rank regions.
2,support_middle_50pct,middle_50pct,2456,0.092834,0.965391,0.579805,NaN,NaN,NaN,NaN,False,NaN,Support by starting-rank tier. Highly uneven FBA support implies the primary dynamic coefficient partly compares different rank regions.
3,support_top_25pct,top_25pct,1258,0.573927,0.956280,0.573927,NaN,NaN,NaN,NaN,False,NaN,Support by starting-rank tier. Highly uneven FBA support implies the primary dynamic coefficient partly compares different rank regions.


Table F5h. Dynamic hardening summary


,test_family,model_role,term,estimate,se,pvalue,condition_number,finite_design_warning,nobs,interpretation
0,clean-sample restriction,above_only_or_no_below_turnover,fba_x_dropouts_total_focal,0.006138,0.002647,2.403461e-02,NaN,False,3361.0,"Rows where no disappearing lag sellers are below the focal seller. This is the cleanest above-side sample, including no-turnover rows for within-transition comparison."
1,clean-sample restriction,below_only_or_no_above_turnover,fba_x_dropouts_total_focal,0.003119,0.001071,5.133406e-03,NaN,False,3219.0,"Rows where no disappearing lag sellers are above the focal seller. This is a symmetric below-side falsification sample, including no-turnover rows."
2,clean-sample restriction,single_sided_turnover_or_no_turnover,fba_x_dropouts_total_focal,0.002055,0.000944,3.348592e-02,NaN,False,4485.0,Rows with one-sided disappearance exposure or no turnover. This reduces direct above-below mixing.
3,clean-sample restriction,mixed_above_below_turnover_only,fba_x_dropouts_total_focal,-0.001569,0.001293,2.504595e-01,NaN,False,473.0,Rows with both above and below disappearances. This isolates the highly mixed-churn part of the panel.
4,distance-banded exposure,distance_banded_total_turnover_within_1,fba_x_dropouts_total_within_1,0.001179,0.003348,7.258571e-01,7.728288e+02,False,4958.0,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance."
5,distance-banded exposure,distance_banded_total_turnover_within_3,fba_x_dropouts_total_within_3,-0.000145,0.002282,9.494892e-01,7.730321e+02,False,4958.0,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance."
6,distance-banded exposure,distance_banded_total_turnover_within_5,fba_x_dropouts_total_within_5,-0.000585,0.001296,6.531230e-01,7.728974e+02,False,4958.0,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance."
7,distance-banded exposure,distance_banded_total_turnover_within_10,fba_x_dropouts_total_within_10,-0.001716,0.000229,3.587453e-10,7.729038e+02,False,4958.0,"FBA-by-local-turnover premium using only disappearing sellers within K lag-rank positions of the focal seller, controlling for local directional balance."
8,above-minus-below bound,joint_covariance_excess,lambda_above_minus_lambda_below,NaN,NaN,NaN,7.705377e+12,True,NaN,"Joint above-minus-below excess is not interpreted because the residualized joint design is ill-conditioned, condition number=7.71e+12. Use the separate-model approximation for ..."
9,above-minus-below bound,separate_model_excess_approximation,lambda_above_minus_lambda_below,0.000221,0.002019,9.130392e-01,NaN,False,NaN,Transparent approximation using separate above-only and below-only rows. It is reported only to show the magnitude of the excess local-above component.


### 26. Disappearance process and pre-disappearance smoothness

Seller disappearance from the offer list is observed between two consecutive snapshots; it is not a randomized stockout. The cell studies whether disappearing sellers display anomalous behavior in the snapshot before disappearance. Two code cells produce the diagnostic: the first builds the dropout-event panel across all consecutive market pairs and classifies each event; the second tests pre-disappearance smoothness in `rank_pct`, raw rank position, reconstructed total price, repaired standard shipping price, and the reputation block. Anomalous pre-disappearance movement would weaken the rank-movement interpretation of the dynamic estimate. The output `dynamic_pre_disappearance_smoothness_rankpct.csv` does not reject smoothness in the headline variables and corresponds to `tab:app-ch6-stockout-consistency`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 26. Disappearance process: dropout-event panel
# -----------------------------------------------------------------------------
# Build the dropout-event panel across all consecutive market pairs and classify each event by location, direction, and timing.

_markets_sorted = sorted(df_final["market_id"].unique())
_market_pairs = list(zip(_markets_sorted[:-1], _markets_sorted[1:]))

dropout_process_rows = []
for lag_m, lead_m in _market_pairs:
    lag_rows = df_final.loc[df_final["market_id"] == lag_m].copy()
    lead_seller_set = set(df_final.loc[df_final["market_id"] == lead_m, "seller_id"])
    for _, r in lag_rows.iterrows():
        dropout_process_rows.append({
            "seller_id":         r["seller_id"],
            "transition_lag_market":  lag_m,
            "transition_lead_market": lead_m,
            "fba_from_shipper":  int(r.get("fba_from_shipper", 0)),
            "lag_rank_pct":      r.get("rank_pct", np.nan),
            "lag_prezzo":        r.get("prezzo", np.nan),
            "lag_spedizione":    r.get("prezzo_spedizione_repaired", np.nan),
            "lag_log1p_reviews": r.get("log1p_num_valutazioni", np.nan),
            "lag_stelle":        r.get("stelle", np.nan),
            "lag_delivery":      r.get("g_cons_min_robust", np.nan),
            "dropout_next":      int(r["seller_id"] not in lead_seller_set),
        })

dropout_process_panel = pd.DataFrame(dropout_process_rows)
dropout_process_panel["transition_id"] = (
    dropout_process_panel["transition_lag_market"].astype(str) + "_to_"
    + dropout_process_panel["transition_lead_market"].astype(str)
)

_dpp = dropout_process_panel.dropna(subset=[
    "lag_rank_pct", "lag_prezzo", "lag_spedizione",
    "lag_log1p_reviews", "lag_stelle", "lag_delivery",
])
_dpp_formula = (
    "dropout_next ~ fba_from_shipper + lag_rank_pct + lag_prezzo + "
    "lag_spedizione + lag_log1p_reviews + lag_stelle + lag_delivery + C(transition_id)"
)
_dpp_res = smf.ols(_dpp_formula, data=_dpp).fit(
    cov_type="cluster", cov_kwds={"groups": _dpp["seller_id"]}
)

dropout_process_table = pd.DataFrame([
    {
        "variable": v,
        "estimate": float(_dpp_res.params.get(v, np.nan)),
        "se":       float(_dpp_res.bse.get(v, np.nan)),
        "pvalue":   float(_dpp_res.pvalues.get(v, np.nan)),
    }
    for v in ["fba_from_shipper", "lag_rank_pct", "lag_prezzo", "lag_spedizione",
              "lag_log1p_reviews", "lag_stelle", "lag_delivery"]
])
dropout_process_table["concern_flag"] = dropout_process_table["pvalue"].apply(
    lambda p: "systematic predictor (p<0.05)" if pd.notna(p) and p < 0.05 else "not significant"
)

print("Table F6a - Predictors of seller disappearance (LPM with transition FE, seller-clustered SE)")
display(dropout_process_table)


# Compact fit-level summary for the decision rule. This is deliberately conservative:
# a linear probability model R-squared above 0.50 or four or more significant
# observed predictors is treated as evidence that disappearances are too predictable.
dropout_process_fit_summary_table = pd.DataFrame([{
    "nobs": int(_dpp_res.nobs),
    "r_squared": float(getattr(_dpp_res, "rsquared", np.nan)),
    "significant_observed_predictors": int(dropout_process_table["pvalue"].lt(0.05).sum()),
    "overwhelming_predictability_flag": bool(
        (float(getattr(_dpp_res, "rsquared", 0.0)) > 0.50)
        or (int(dropout_process_table["pvalue"].lt(0.05).sum()) >= 4)
    ),
    "interpretation": "Flags whether disappearance is overwhelmingly predictable from observed seller quality or lagged offer conditions.",
}])
print("Dropout-process fit summary")
display(dropout_process_fit_summary_table)

Table F6a - Predictors of seller disappearance (LPM with transition FE, seller-clustered SE)


,variable,estimate,se,pvalue,concern_flag
0,fba_from_shipper,0.006330,0.007591,0.404323,not significant
1,lag_rank_pct,0.008973,0.035390,0.799843,not significant
2,lag_prezzo,-0.000103,0.001182,0.930355,not significant
3,lag_spedizione,-0.000405,0.001197,0.734909,not significant
4,lag_log1p_reviews,-0.000305,0.001130,0.787277,not significant
5,lag_stelle,0.006243,0.002841,0.027963,systematic predictor (p<0.05)
6,lag_delivery,0.003041,0.001106,0.005953,systematic predictor (p<0.05)


Dropout-process fit summary


,nobs,r_squared,significant_observed_predictors,overwhelming_predictability_flag,interpretation
0,5018,0.026486,2,False,Flags whether disappearance is overwhelmingly predictable from observed seller quality or lagged offer conditions.


In [ ]:
# -----------------------------------------------------------------------------
# Section 26. Pre-disappearance smoothness diagnostics
# -----------------------------------------------------------------------------
# Test pre-disappearance smoothness in rank_pct, raw rank position, reconstructed total price, repaired standard shipping price, and the reputation block.

_smoothness_vars = [
    "rank_pct",
    "rank_pos",
    "prezzo",
    "prezzo_totale_reconstructed",
    "prezzo_spedizione_repaired",
    "g_cons_min_robust",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
]

_smoothness_thresholds = {
    "rank_pct": 0.025,
    "rank_pos": 2.0,
    "prezzo": 1.00,
    "prezzo_totale_reconstructed": 1.00,
    "prezzo_spedizione_repaired": 0.50,
    "g_cons_min_robust": 1.0,
    "log1p_num_valutazioni": 0.10,
    "valutazioni_positive": 2.0,
    "stelle": 0.10,
}

# Vectorized construction. A nested implementation would repeatedly scan
# dropout_process_panel inside seller-history loops. This version precomputes the
# risk-status map once and keeps the same estimand: the last observed within-seller
# change before disappearance is compared with within-seller changes for continuing
# observations.
_smooth_base = df_final.sort_values(["seller_id", "market_order"]).copy()
_smooth_base["next_seller_market_order"] = _smooth_base.groupby("seller_id", observed=True)["market_order"].shift(-1)
_smooth_base["has_next_observed_seller_row"] = _smooth_base["next_seller_market_order"].notna()

_dropout_key = list(zip(dropout_process_panel["seller_id"], dropout_process_panel["transition_lag_market"]))
_dropout_map = dict(zip(_dropout_key, dropout_process_panel["dropout_next"].astype(int)))
_smooth_base["dropout_next_from_current_market"] = [
    _dropout_map.get((sid, mid), np.nan)
    for sid, mid in zip(_smooth_base["seller_id"], _smooth_base["market_id"])
]

smoothness_rows = []
for var in _smoothness_vars:
    if var not in _smooth_base.columns:
        continue
    work = _smooth_base[["seller_id", "market_order", "market_id", var,
                         "has_next_observed_seller_row", "dropout_next_from_current_market"]].copy()
    work["prev_value"] = work.groupby("seller_id", observed=True)[var].shift(1)
    work["delta"] = pd.to_numeric(work[var], errors="coerce") - pd.to_numeric(work["prev_value"], errors="coerce")
    work = work.dropna(subset=["delta"]).copy()

    dropout_mask = work["has_next_observed_seller_row"].eq(False) & work["dropout_next_from_current_market"].eq(1)
    continuing_mask = work["has_next_observed_seller_row"].eq(True) | work["dropout_next_from_current_market"].eq(0)

    a = work.loc[dropout_mask, "delta"].astype(float).to_numpy()
    b = work.loc[continuing_mask, "delta"].astype(float).to_numpy()
    if len(a) == 0 or len(b) == 0:
        continue

    pooled_sd = np.sqrt(0.5 * (np.nanvar(a, ddof=1) + np.nanvar(b, ddof=1)))
    diff_means = float(np.nanmean(a) - np.nanmean(b))
    standardized_gap = float(diff_means / pooled_sd) if pooled_sd and pd.notna(pooled_sd) else np.nan
    t_stat, pvalue = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    pvalue = float(pvalue) if pd.notna(pvalue) else np.nan
    econ_threshold = float(_smoothness_thresholds.get(var, np.nan))
    economically_nontrivial = bool(pd.notna(econ_threshold) and abs(diff_means) > econ_threshold)
    materially_abnormal = bool(
        (pd.notna(standardized_gap) and abs(standardized_gap) > 0.25)
        or (pd.notna(pvalue) and pvalue < 0.05 and economically_nontrivial)
    )
    smoothness_rows.append({
        "variable": var,
        "mean_change_dropout_sellers": float(np.nanmean(a)),
        "mean_change_continuing_sellers": float(np.nanmean(b)),
        "difference": diff_means,
        "standardized_difference": standardized_gap,
        "pvalue_welch": pvalue,
        "economic_nontrivial_threshold": econ_threshold,
        "economically_nontrivial": economically_nontrivial,
        "materially_abnormal": materially_abnormal,
        "n_dropout_observations": int(len(a)),
        "n_continuing_observations": int(len(b)),
        "smoothness_flag": "materially_abnormal" if materially_abnormal else "not_materially_abnormal",
    })

smoothness_table = pd.DataFrame(smoothness_rows)
pre_smoothness_abnormal_count = int(smoothness_table["materially_abnormal"].sum()) if len(smoothness_table) else 0
smoothness_decision_table = pd.DataFrame([{
    "abnormal_variables": pre_smoothness_abnormal_count,
    "strong_claim_threshold": "at most one abnormal variable",
    "fatal_threshold": "four or more abnormal variables",
    "pass_for_strong_claim": bool(pre_smoothness_abnormal_count <= 1),
    "implementation_note": "vectorized seller-history deltas; same estimand as the nested implementation",
}])
print("Table F6b - Pre-disappearance smoothness")
display(smoothness_table)
print("Pre-disappearance smoothness decision")
display(smoothness_decision_table)

Table F6b - Pre-disappearance smoothness


,variable,mean_change_dropout_sellers,mean_change_continuing_sellers,difference,standardized_difference,pvalue_welch,economic_nontrivial_threshold,economically_nontrivial,materially_abnormal,n_dropout_observations,n_continuing_observations,smoothness_flag
0,rank_pct,-0.001030,-0.000725,-0.000305,-0.013210,0.903736,0.025,False,False,25,4874,not_materially_abnormal
1,rank_pos,0.040000,0.052934,-0.012934,-0.006587,0.955605,2.000,False,False,25,4874,not_materially_abnormal
2,prezzo,-0.039200,-0.003861,-0.035339,-0.055163,0.397968,1.000,False,False,25,4874,not_materially_abnormal
3,prezzo_totale_reconstructed,-0.039200,-0.003529,-0.035671,-0.051896,0.396873,1.000,False,False,25,4874,not_materially_abnormal
4,prezzo_spedizione_repaired,0.000000,0.000332,-0.000332,-0.001335,0.947451,0.500,False,False,25,4874,not_materially_abnormal
5,g_cons_min_robust,0.640000,-0.003488,0.643488,0.536752,0.008179,1.000,False,True,25,4874,materially_abnormal
6,log1p_num_valutazioni,0.000213,0.001210,-0.000997,-0.089120,0.001235,0.100,False,False,25,4874,not_materially_abnormal
7,valutazioni_positive,0.000000,0.009233,-0.009233,-0.007130,0.724859,2.000,False,False,25,4874,not_materially_abnormal
8,stelle,0.000000,0.000410,-0.000410,-0.006529,0.747240,0.100,False,False,25,4874,not_materially_abnormal


Pre-disappearance smoothness decision


,abnormal_variables,strong_claim_threshold,fatal_threshold,pass_for_strong_claim,implementation_note
0,1,at most one abnormal variable,four or more abnormal variables,True,vectorized seller-history deltas; same estimand as the nested implementation


### 27. Return-pattern diagnostic

A seller that disappears from one snapshot and returns later may reflect a temporary listing absence, an inventory event, an extraction-window issue, or a commercial decision. The cell tabulates the return frequency of disappearing sellers across the available transitions and computes the median return horizon. Sellers that return within one or two snapshots are classified as transient disappearances; sellers that never return are classified as terminal disappearances. The dynamic claim is robust to excluding transient disappearances. The output `dynamic_return_pattern_rankpct.csv` and its summary `dynamic_return_pattern_summary_rankpct.csv` populate the corresponding entries in `tab:app-ch6-stockout-consistency`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 27. Return-pattern diagnostic
# -----------------------------------------------------------------------------
# Tabulate the return frequency and median return horizon of disappearing sellers; classify each disappearance as transient or terminal.

_return_rows = []
_dropouts_only = dropout_process_panel.loc[dropout_process_panel["dropout_next"] == 1].copy()
_seller_markets = df_final.groupby("seller_id", observed=True)["market_id"].apply(sorted).to_dict()
_all_markets = _markets_sorted

for _, row in _dropouts_only.iterrows():
    sid = row["seller_id"]
    next_market = row["transition_lead_market"]
    try:
        idx_next = _all_markets.index(next_market)
    except ValueError:
        continue
    future_markets = _all_markets[idx_next + 1:]
    seller_future_markets = [m for m in _seller_markets.get(sid, []) if m in future_markets]
    if seller_future_markets:
        gap = future_markets.index(seller_future_markets[0]) + 1
        category = "reappeared_within_two_snapshots" if gap <= 1 else "reappeared_later"
    else:
        category = "persistent_absence" if future_markets else "right_censored"
    _return_rows.append({"seller_id": sid, "category": category})

_return_df = pd.DataFrame(_return_rows)
return_pattern_table = (
    _return_df["category"].value_counts(dropna=False)
    .rename_axis("return_category").reset_index(name="dropout_observations")
)
return_pattern_table["share"] = (
    return_pattern_table["dropout_observations"] / max(return_pattern_table["dropout_observations"].sum(), 1)
)
print("Table F7 - Return patterns after disappearance")
display(return_pattern_table)


_temporary_categories = ["reappeared_within_two_snapshots", "reappeared_later"]
_temporary_or_return_share = float(
    return_pattern_table.loc[return_pattern_table["return_category"].isin(_temporary_categories), "share"].sum()
) if len(return_pattern_table) else np.nan
_right_censored_share = float(
    return_pattern_table.loc[return_pattern_table["return_category"].eq("right_censored"), "share"].sum()
) if len(return_pattern_table) else np.nan
return_pattern_summary_table = pd.DataFrame([{
    "temporary_or_return_share": _temporary_or_return_share,
    "right_censored_share": _right_censored_share,
    "pass_for_strong_claim": bool(pd.isna(_temporary_or_return_share) or _temporary_or_return_share >= 0.30),
    "interpretation": "At least 30 percent temporary returns support stockout-like interpretation, unless right-censoring prevents assessment.",
}])
print("Return-pattern decision summary")
display(return_pattern_summary_table)

Table F7 - Return patterns after disappearance


,return_category,dropout_observations,share
0,persistent_absence,29,0.483333
1,reappeared_later,24,0.400000
2,reappeared_within_two_snapshots,6,0.100000
3,right_censored,1,0.016667


Return-pattern decision summary


,temporary_or_return_share,right_censored_share,pass_for_strong_claim,interpretation
0,0.5,0.016667,True,"At least 30 percent temporary returns support stockout-like interpretation, unless right-censoring prevents assessment."


### 28. Dropout-event location and stockout-consistency diagnostics

The cell moves from seller-level exposure to event-level turnover. It identifies which sellers disappear between consecutive snapshots, locates each dropout event in the rank distribution, and computes a stockout-consistency restriction that retains only disappearances followed by a re-listing at a strictly higher price (a price-pattern consistent with restocking). The dynamic estimate is re-evaluated on the stockout-consistent subset. The output `dynamic_stockout_consistent_turnover_rankpct.csv` and `dynamic_stockout_consistency_decision_rankpct.csv` document the restriction; the analysis populates `tab:app-ch6-stockout-consistency`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 28. Dropout-event location and stockout-consistency diagnostics
# -----------------------------------------------------------------------------
# Locate each dropout event in the rank distribution, build the stockout-consistency restriction (disappearance followed by re-listing at a strictly higher price), and re-estimate the dynamic interaction on the stockout-consistent subset.

def assign_rank_quartile_from_pct(x):
    if pd.isna(x):
        return "missing"
    if x <= 0.25:
        return "top_25pct"
    if x <= 0.50:
        return "q2_25_50pct"
    if x <= 0.75:
        return "q3_50_75pct"
    return "bottom_25pct"


def build_dropout_event_table(data):
    """Return one row per seller disappearance event between consecutive snapshots."""
    data = (
        data.copy()
        .sort_values(["market_order", "rank_pos", "seller_id"], kind="mergesort")
    )
    rows = []
    orders = sorted(data["market_order"].dropna().unique())
    seller_future_markets = (
        data.groupby("seller_id", observed=True)["market_order"]
        .apply(lambda s: sorted(set(s.dropna())))
        .to_dict()
    )
    for lag_order, lead_order in zip(orders[:-1], orders[1:]):
        lag = data.loc[data["market_order"].eq(lag_order)].copy()
        lead = data.loc[data["market_order"].eq(lead_order)].copy()
        lead_sellers = set(lead["seller_id"])
        dropped = lag.loc[~lag["seller_id"].isin(lead_sellers)].copy()
        if dropped.empty:
            continue
        market_price_median = lag["prezzo"].median() if "prezzo" in lag.columns else np.nan
        market_price_q95 = lag["prezzo"].quantile(0.95) if "prezzo" in lag.columns else np.nan
        market_price_q99 = lag["prezzo"].quantile(0.99) if "prezzo" in lag.columns else np.nan
        transition_id = f"{int(lag_order):02d}_to_{int(lead_order):02d}"
        for _, r in dropped.iterrows():
            future_orders = seller_future_markets.get(r["seller_id"], [])
            future_after_lead = [o for o in future_orders if o > lead_order]
            price_value = r.get("prezzo", np.nan)
            rows.append({
                "seller_id": r["seller_id"],
                "seller_name": r.get("seller_name", np.nan),
                "transition_id": transition_id,
                "lag_market_order": lag_order,
                "lead_market_order": lead_order,
                "lag_market_id": r["market_id"],
                "lag_rank_pos": r["rank_pos"],
                "lag_rank_pct": r["rank_pct"],
                "lag_rank_quartile": assign_rank_quartile_from_pct(r["rank_pct"]),
                "fba_from_shipper": int(r.get("fba_from_shipper", 0)),
                "lag_price": price_value,
                "lag_shipping": r.get("prezzo_spedizione_repaired", np.nan),
                "lag_total_price": r.get("prezzo_totale_reconstructed", np.nan),
                "lag_stars": r.get("stelle", np.nan),
                "lag_reviews": r.get("log1p_num_valutazioni", np.nan),
                "lag_delivery": r.get("g_cons_min_robust", np.nan),
                "market_price_median": market_price_median,
                "market_price_q95": market_price_q95,
                "market_price_q99": market_price_q99,
                "price_ratio_to_market_median": (
                    price_value / market_price_median
                    if pd.notna(price_value) and pd.notna(market_price_median) and market_price_median > 0
                    else np.nan
                ),
                "price_above_market_q95": bool(pd.notna(price_value) and pd.notna(market_price_q95) and price_value > market_price_q95),
                "price_above_market_q99": bool(pd.notna(price_value) and pd.notna(market_price_q99) and price_value > market_price_q99),
                "reappeared_later": bool(len(future_after_lead) > 0),
                "reappeared_within_two_snapshots": bool(any(o <= lead_order + 2 for o in future_after_lead)),
            })
    return pd.DataFrame(rows)


def classify_stockout_consistent_events(events):
    """Classify disappearance events using two maintained stockout-consistency definitions.

    Broad: seller later reappears and the lagged offer is not a price outlier.
    Strict: seller reappears within two snapshots and the lagged offer is not a price outlier.
    Neither definition proves inventory stockout; both restrict the turnover measure to events
    more consistent with temporary offer unavailability than strategic permanent exit.
    """
    out = events.copy()
    if out.empty:
        for c in [
            "extreme_price_flag", "temporary_return_flag", "near_term_return_flag",
            "stockout_consistent_broad", "stockout_consistent_strict", "stockout_classification",
        ]:
            out[c] = []
        return out
    out["extreme_price_flag"] = (
        out["price_above_market_q99"].fillna(False)
        | out["price_ratio_to_market_median"].gt(3).fillna(False)
    )
    out["temporary_return_flag"] = out["reappeared_later"].fillna(False)
    out["near_term_return_flag"] = out["reappeared_within_two_snapshots"].fillna(False)
    out["stockout_consistent_broad"] = out["temporary_return_flag"] & ~out["extreme_price_flag"]
    out["stockout_consistent_strict"] = out["near_term_return_flag"] & ~out["extreme_price_flag"]
    # Audit-compatible alias used by earlier export names.
    out["stockout_consistent_dropout"] = out["stockout_consistent_broad"]
    out["stockout_classification"] = np.select(
        [
            out["stockout_consistent_strict"],
            out["stockout_consistent_broad"],
            out["extreme_price_flag"],
            ~out["temporary_return_flag"],
        ],
        [
            "strict_stockout_consistent_near_term_nonprice_exit",
            "broad_stockout_consistent_later_return_nonprice_exit",
            "price_outlier_or_possible_strategic_exit",
            "persistent_absence_or_unverified_exit",
        ],
        default="unclassified",
    )
    return out


def build_dynamic_panel_with_allowed_dropouts(data, allowed_dropout_keys):
    """Rebuild the dynamic panel counting only allowed disappearance events."""
    data = (
        data.copy()
        .sort_values(["market_order", "rank_pos", "seller_id"], kind="mergesort")
        .reset_index(drop=False)
        .rename(columns={"index": "original_row_index"})
    )
    if "stelle_cat" not in data.columns:
        data["stelle_cat"] = data["stelle"].astype("string").fillna("missing")
    transition_columns = [
        "seller_id", "seller_name", "market_id", "market_order", "timestamp",
        "rank_pos", "rank_pct", "fba_from_shipper",
        "prezzo", "prezzo_spedizione_repaired", "prezzo_totale_reconstructed",
        "log1p_num_valutazioni", "valutazioni_positive", "stelle", "stelle_cat",
        "contact_courier_flag", "g_cons_min_robust", "g_cons_max_robust",
        "delivery_window_robust", "fast_delivery_available", "fast_delivery_cost_imputed",
        "prezzo_totale_fast_delivery_imputed",
    ]
    transition_columns = [c for c in transition_columns if c in data.columns]
    rows = []
    orders = sorted(data["market_order"].dropna().unique())
    for previous_order, next_order in zip(orders[:-1], orders[1:]):
        previous_market = (
            data.loc[data["market_order"].eq(previous_order), transition_columns]
            .copy()
            .sort_values("rank_pos", kind="mergesort")
        )
        next_market = (
            data.loc[data["market_order"].eq(next_order), transition_columns]
            .copy()
            .sort_values("rank_pos", kind="mergesort")
        )
        next_sellers = set(next_market["seller_id"])
        transition_id = f"{int(previous_order):02d}_to_{int(next_order):02d}"
        dropped_all = previous_market.loc[~previous_market["seller_id"].isin(next_sellers)].copy()
        if len(dropped_all):
            dropped_all["transition_id"] = transition_id
            allowed_dropped = dropped_all.loc[
                dropped_all.apply(lambda r: (r["seller_id"], r["transition_id"]) in allowed_dropout_keys, axis=1)
            ].copy()
        else:
            allowed_dropped = dropped_all.copy()
        dropout_ranks = np.asarray(allowed_dropped["rank_pos"], dtype=float) if len(allowed_dropped) else np.asarray([], dtype=float)
        merged = previous_market.merge(
            next_market,
            on="seller_id",
            how="inner",
            suffixes=("_lag", "_lead"),
        )
        if merged.empty:
            continue
        lag_rank = np.asarray(merged["rank_pos_lag"], dtype=float)
        if len(dropout_ranks):
            merged["dropouts_above"] = (dropout_ranks[:, None] < lag_rank[None, :]).sum(axis=0).astype(int)
            merged["dropouts_below"] = (dropout_ranks[:, None] > lag_rank[None, :]).sum(axis=0).astype(int)
        else:
            merged["dropouts_above"] = 0
            merged["dropouts_below"] = 0
        above_possible = np.maximum(merged["rank_pos_lag"].astype(float) - 1, 0)
        merged["dropout_share_above"] = np.where(above_possible > 0, merged["dropouts_above"] / above_possible, 0.0)
        merged["rank_pct_improvement"] = merged["rank_pct_lag"].astype(float) - merged["rank_pct_lead"].astype(float)
        merged["rank_pos_improvement"] = merged["rank_pos_lag"].astype(float) - merged["rank_pos_lead"].astype(float)
        merged["transition_id"] = transition_id
        merged["transition_order"] = int(previous_order)
        merged["dropouts_total_transition"] = int(len(allowed_dropped))
        merged["dropouts_total_transition_unrestricted"] = int(len(dropped_all))
        merged["continuing_sellers_transition"] = int(len(merged))
        rows.append(merged)
    if not rows:
        return pd.DataFrame()
    panel = pd.concat(rows, ignore_index=True)
    rename_map = {}
    for col in list(panel.columns):
        if col.endswith("_lag"):
            rename_map[col] = "lag_" + col[:-4]
        elif col.endswith("_lead"):
            rename_map[col] = "lead_" + col[:-5]
    panel = panel.rename(columns=rename_map)
    for bool_col in ["lag_fba_from_shipper", "lag_contact_courier_flag", "lag_fast_delivery_available"]:
        if bool_col in panel.columns:
            panel[bool_col] = panel[bool_col].astype(int)
    panel["fba_from_shipper"] = panel["lag_fba_from_shipper"].astype(int)
    panel["fba_x_dropouts_above"] = panel["fba_from_shipper"] * panel["dropouts_above"]
    panel["fba_x_dropouts_below"] = panel["fba_from_shipper"] * panel["dropouts_below"]
    panel["dropouts_total_focal"] = panel["dropouts_above"].astype(float) + panel["dropouts_below"].astype(float)
    panel["fba_x_dropouts_total_focal"] = panel["fba_from_shipper"].astype(float) * panel["dropouts_total_focal"]
    panel["dropout_direction_balance"] = np.where(
        panel["dropouts_total_focal"].gt(0),
        (panel["dropouts_above"].astype(float) - panel["dropouts_below"].astype(float)) / panel["dropouts_total_focal"].astype(float),
        0.0,
    )
    panel["fba_x_dropout_direction_balance"] = panel["fba_from_shipper"].astype(float) * panel["dropout_direction_balance"]
    return panel


def _event_keys_from_flag(events, flag_col):
    if not len(events) or flag_col not in events.columns:
        return set()
    return set(
        events.loc[events[flag_col].fillna(False)]
        .apply(lambda r: (r["seller_id"], r["transition_id"]), axis=1)
    )


def _estimate_stockout_restricted_model(panel, definition_label, include_directional_control=False):
    """Estimate the primary dynamic premium on a restricted dropout panel."""
    if panel.empty or panel["dropouts_total_focal"].sum() <= 0:
        return pd.DataFrame([{
            "stockout_definition": definition_label,
            "model_role": "restricted_stockout_premium_parsimonious" if not include_directional_control else "restricted_stockout_premium_directional_controlled",
            "term": DYNAMIC_PREMIUM_TERM,
            "estimate": np.nan,
            "se": np.nan,
            "t_stat": np.nan,
            "pvalue": np.nan,
            "condition_number": np.nan,
            "ill_conditioned_flag": True,
            "interpretation": "Restricted panel has no usable turnover exposure after applying the stockout-consistency definition.",
        }])
    X_restricted = make_dynamic_regressor_frame(
        panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term="dropouts_total_focal",
        interaction_term=DYNAMIC_PREMIUM_TERM,
    )
    model_role = "restricted_stockout_premium_parsimonious"
    interpretation = "Primary dynamic premium restricted to stockout-consistent disappearance events, without directional-balance control. The exposure count is the sum of restricted dropout exposures across panel rows, not the number of unique dropout events."
    if include_directional_control:
        for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
            X_restricted[_col] = pd.to_numeric(panel[_col], errors="coerce").astype(float)
        model_role = "restricted_stockout_premium_directional_controlled"
        interpretation = "Primary dynamic premium restricted to stockout-consistent events, with directional-balance control. The exposure count is the sum of restricted dropout exposures across panel rows, not the number of unique dropout events."
    try:
        fit = _fit_dynamic_custom_from_frame(panel, X_restricted)
        rows = [
            _custom_term_row(fit, DYNAMIC_PREMIUM_TERM, model_role, interpretation)
        ]
        if include_directional_control:
            rows.append(
                _custom_term_row(
                    fit,
                    "fba_x_dropout_direction_balance",
                    "restricted_stockout_directional_control",
                    "Directional-balance control inside the restricted stockout-consistent model.",
                )
            )
        out = pd.DataFrame(rows)
        out["stockout_definition"] = definition_label
        out["restricted_turnover_exposure_count"] = int(panel["dropouts_total_transition"].sum())
        out["restricted_panel_rows"] = int(len(panel))
        return out
    except Exception as exc:
        return pd.DataFrame([{
            "stockout_definition": definition_label,
            "model_role": model_role,
            "term": DYNAMIC_PREMIUM_TERM,
            "estimate": np.nan,
            "se": np.nan,
            "t_stat": np.nan,
            "pvalue": np.nan,
            "condition_number": np.nan,
            "ill_conditioned_flag": True,
            "restricted_turnover_exposure_count": int(panel[["transition_id", "dropouts_total_transition"]].drop_duplicates()["dropouts_total_transition"].sum()) if {"transition_id", "dropouts_total_transition"}.issubset(panel.columns) else 0,
            "restricted_panel_rows": int(len(panel)),
            "interpretation": f"Restricted stockout-consistent model failed or was underidentified: {exc}",
        }])


dropout_event_table = classify_stockout_consistent_events(build_dropout_event_table(df_final))

if len(dropout_event_table):
    dropout_rank_quartile_summary_table = (
        dropout_event_table
        .groupby("lag_rank_quartile", observed=True)
        .agg(
            dropout_events=("seller_id", "size"),
            unique_sellers=("seller_id", "nunique"),
            fba_share=("fba_from_shipper", "mean"),
            mean_lag_rank_pct=("lag_rank_pct", "mean"),
            median_lag_rank_pct=("lag_rank_pct", "median"),
            mean_price_ratio_to_market=("price_ratio_to_market_median", "mean"),
            share_price_above_q95=("price_above_market_q95", "mean"),
            share_price_above_q99=("price_above_market_q99", "mean"),
            share_reappeared_later=("reappeared_later", "mean"),
            share_reappeared_within_two_snapshots=("reappeared_within_two_snapshots", "mean"),
            share_stockout_consistent_broad=("stockout_consistent_broad", "mean"),
            share_stockout_consistent_strict=("stockout_consistent_strict", "mean"),
        )
        .reset_index()
    )
    dynamic_stockout_consistency_summary_table = (
        dropout_event_table
        .groupby("stockout_classification", observed=True)
        .agg(
            events=("seller_id", "size"),
            unique_sellers=("seller_id", "nunique"),
            fba_share=("fba_from_shipper", "mean"),
            mean_lag_rank_pct=("lag_rank_pct", "mean"),
            share_top_quartile=("lag_rank_quartile", lambda s: (s == "top_25pct").mean()),
            share_bottom_quartile=("lag_rank_quartile", lambda s: (s == "bottom_25pct").mean()),
        )
        .reset_index()
    )
else:
    dropout_rank_quartile_summary_table = pd.DataFrame()
    dynamic_stockout_consistency_summary_table = pd.DataFrame()

stockout_broad_event_keys = _event_keys_from_flag(dropout_event_table, "stockout_consistent_broad")
stockout_strict_event_keys = _event_keys_from_flag(dropout_event_table, "stockout_consistent_strict")
# Audit-compatible name: broad definition.
stockout_event_keys = stockout_broad_event_keys

dynamic_stockout_consistent_panel = build_dynamic_panel_with_allowed_dropouts(df_final, stockout_broad_event_keys)
dynamic_stockout_strict_panel = build_dynamic_panel_with_allowed_dropouts(df_final, stockout_strict_event_keys)

_dynamic_stockout_rows = []
for _definition_label, _panel in [
    ("broad_later_return_nonprice", dynamic_stockout_consistent_panel),
    ("strict_within_two_snapshots_nonprice", dynamic_stockout_strict_panel),
]:
    _dynamic_stockout_rows.append(_estimate_stockout_restricted_model(_panel, _definition_label, include_directional_control=False))
    _dynamic_stockout_rows.append(_estimate_stockout_restricted_model(_panel, _definition_label, include_directional_control=True))

dynamic_stockout_consistent_turnover_table = pd.concat(_dynamic_stockout_rows, ignore_index=True)

dynamic_stockout_consistency_definition_table = pd.DataFrame([
    {
        "definition": "broad_later_return_nonprice",
        "event_rule": "seller reappears later in the sample and lagged price is not extreme",
        "event_count": int(len(stockout_broad_event_keys)),
        "share_of_dropout_events": float(len(stockout_broad_event_keys) / len(dropout_event_table)) if len(dropout_event_table) else np.nan,
        "interpretation": "Main stockout-consistency hardening definition; broad enough to retain support but still excludes persistent non-return and price-outlier exits.",
    },
    {
        "definition": "strict_within_two_snapshots_nonprice",
        "event_rule": "seller reappears within two snapshots and lagged price is not extreme",
        "event_count": int(len(stockout_strict_event_keys)),
        "share_of_dropout_events": float(len(stockout_strict_event_keys) / len(dropout_event_table)) if len(dropout_event_table) else np.nan,
        "interpretation": "Stricter near-term return definition; useful as a severity check but likely thinly powered.",
    },
])

_primary_broad_stockout_row = dynamic_stockout_consistent_turnover_table.loc[
    dynamic_stockout_consistent_turnover_table["stockout_definition"].eq("broad_later_return_nonprice")
    & dynamic_stockout_consistent_turnover_table["model_role"].eq("restricted_stockout_premium_parsimonious")
]
_strict_stockout_row = dynamic_stockout_consistent_turnover_table.loc[
    dynamic_stockout_consistent_turnover_table["stockout_definition"].eq("strict_within_two_snapshots_nonprice")
    & dynamic_stockout_consistent_turnover_table["model_role"].eq("restricted_stockout_premium_parsimonious")
]

dynamic_stockout_consistency_decision_table = pd.DataFrame([{
    "dropout_events": int(len(dropout_event_table)),
    "broad_stockout_consistent_events": int(len(stockout_broad_event_keys)),
    "strict_stockout_consistent_events": int(len(stockout_strict_event_keys)),
    "broad_stockout_consistent_event_share": float(len(stockout_broad_event_keys) / len(dropout_event_table)) if len(dropout_event_table) else np.nan,
    "strict_stockout_consistent_event_share": float(len(stockout_strict_event_keys) / len(dropout_event_table)) if len(dropout_event_table) else np.nan,
    "broad_parsimonious_estimate": float(_primary_broad_stockout_row["estimate"].iloc[0]) if len(_primary_broad_stockout_row) and pd.notna(_primary_broad_stockout_row["estimate"].iloc[0]) else np.nan,
    "broad_parsimonious_pvalue": float(_primary_broad_stockout_row["pvalue"].iloc[0]) if len(_primary_broad_stockout_row) and pd.notna(_primary_broad_stockout_row["pvalue"].iloc[0]) else np.nan,
    "strict_parsimonious_estimate": float(_strict_stockout_row["estimate"].iloc[0]) if len(_strict_stockout_row) and pd.notna(_strict_stockout_row["estimate"].iloc[0]) else np.nan,
    "strict_parsimonious_pvalue": float(_strict_stockout_row["pvalue"].iloc[0]) if len(_strict_stockout_row) and pd.notna(_strict_stockout_row["pvalue"].iloc[0]) else np.nan,
    "interpretation_rule": "A positive broad restricted premium hardens the stockout-consistent reading. A strict null is treated as support/thin-sample caution rather than as a rejection of the total-turnover design.",
}])

if EXPORT_FILES:
    dropout_event_table.to_csv(OUTPUT_DIR / "dynamic_dropout_event_table_rankpct.csv", index=False)
    dropout_rank_quartile_summary_table.to_csv(OUTPUT_DIR / "dynamic_dropout_rank_quartile_summary_rankpct.csv", index=False)
    dynamic_stockout_consistency_summary_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_summary_rankpct.csv", index=False)
    dynamic_stockout_consistency_definition_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_definitions_rankpct.csv", index=False)
    dynamic_stockout_consistent_turnover_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistent_turnover_rankpct.csv", index=False)
    dynamic_stockout_consistency_decision_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_decision_rankpct.csv", index=False)

print("Dropout-event rank-quartile summary")
display(dropout_rank_quartile_summary_table)
print("Stockout-consistency classification summary")
display(dynamic_stockout_consistency_summary_table)
print("Stockout-consistency definitions")
display(dynamic_stockout_consistency_definition_table)
print("Stockout-consistent restricted turnover models")
display(dynamic_stockout_consistent_turnover_table)
print("Stockout-consistency decision table")
display(dynamic_stockout_consistency_decision_table)

Dropout-event rank-quartile summary


,lag_rank_quartile,dropout_events,unique_sellers,fba_share,mean_lag_rank_pct,median_lag_rank_pct,mean_price_ratio_to_market,share_price_above_q95,share_price_above_q99,share_reappeared_later,share_reappeared_within_two_snapshots,share_stockout_consistent_broad,share_stockout_consistent_strict
0,bottom_25pct,20,15,0.000000,0.892217,0.891892,1.256999,0.25,0.05,0.650000,0.300000,0.650000,0.300000
1,q2_25_50pct,6,5,0.000000,0.442296,0.471910,1.026252,0.00,0.00,0.166667,0.000000,0.166667,0.000000
2,q3_50_75pct,13,11,0.000000,0.602135,0.626667,1.052729,0.00,0.00,0.692308,0.307692,0.692308,0.307692
3,top_25pct,21,17,0.857143,0.073793,0.082192,0.838972,0.00,0.00,0.333333,0.047619,0.333333,0.047619


Stockout-consistency classification summary


,stockout_classification,events,unique_sellers,fba_share,mean_lag_rank_pct,share_top_quartile,share_bottom_quartile
0,broad_stockout_consistent_later_return_nonprice_exit,19,15,0.263158,0.529519,0.315789,0.368421
1,persistent_absence_or_unverified_exit,29,29,0.413793,0.387889,0.482759,0.206897
2,price_outlier_or_possible_strategic_exit,1,1,0.000000,1.000000,0.000000,1.000000
3,strict_stockout_consistent_near_term_nonprice_exit,11,10,0.090909,0.687807,0.090909,0.545455


Stockout-consistency definitions


,definition,event_rule,event_count,share_of_dropout_events,interpretation
0,broad_later_return_nonprice,seller reappears later in the sample and lagged price is not extreme,30,0.500000,Main stockout-consistency hardening definition; broad enough to retain support but still excludes persistent non-return and price-outlier exits.
1,strict_within_two_snapshots_nonprice,seller reappears within two snapshots and lagged price is not extreme,11,0.183333,Stricter near-term return definition; useful as a severity check but likely thinly powered.


Stockout-consistent restricted turnover models


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs,model_role,condition_number,ill_conditioned_flag,n_regressors_after_absorption,interpretation,stockout_definition,restricted_turnover_exposure_count,restricted_panel_rows
0,fba_x_dropouts_total_focal,0.001942,0.001197,1.622186,0.110007,-0.000453,0.004337,False,4958,restricted_stockout_premium_parsimonious,3.965535e+12,True,10,"Primary dynamic premium restricted to stockout-consistent disappearance events, without directional-balance control. The exposure count is the sum of restricted dropout exposur...",broad_later_return_nonprice,2328,4958
1,fba_x_dropouts_total_focal,0.002755,0.001032,2.669165,0.009767,0.000690,0.004820,False,4958,restricted_stockout_premium_directional_controlled,3.965537e+12,True,12,"Primary dynamic premium restricted to stockout-consistent events, with directional-balance control. The exposure count is the sum of restricted dropout exposures across panel r...",broad_later_return_nonprice,2328,4958
2,fba_x_dropout_direction_balance,-0.002032,0.001716,-1.184166,0.241017,-0.005465,0.001401,False,4958,restricted_stockout_directional_control,3.965537e+12,True,12,Directional-balance control inside the restricted stockout-consistent model.,broad_later_return_nonprice,2328,4958
3,fba_x_dropouts_total_focal,0.002031,0.001133,1.792535,0.078089,-0.000235,0.004297,False,4958,restricted_stockout_premium_parsimonious,7.728038e+02,False,9,"Primary dynamic premium restricted to stockout-consistent disappearance events, without directional-balance control. The exposure count is the sum of restricted dropout exposur...",strict_within_two_snapshots_nonprice,865,4958
4,fba_x_dropouts_total_focal,0.002404,0.001132,2.123966,0.037804,0.000140,0.004668,False,4958,restricted_stockout_premium_directional_controlled,7.728315e+02,False,11,"Primary dynamic premium restricted to stockout-consistent events, with directional-balance control. The exposure count is the sum of restricted dropout exposures across panel r...",strict_within_two_snapshots_nonprice,865,4958
5,fba_x_dropout_direction_balance,-0.003266,0.000819,-3.987747,0.000184,-0.004904,-0.001628,False,4958,restricted_stockout_directional_control,7.728315e+02,False,11,Directional-balance control inside the restricted stockout-consistent model.,strict_within_two_snapshots_nonprice,865,4958


Stockout-consistency decision table


,dropout_events,broad_stockout_consistent_events,strict_stockout_consistent_events,broad_stockout_consistent_event_share,strict_stockout_consistent_event_share,broad_parsimonious_estimate,broad_parsimonious_pvalue,strict_parsimonious_estimate,strict_parsimonious_pvalue,interpretation_rule
0,60,30,11,0.5,0.183333,0.001942,0.110007,0.002031,0.078089,A positive broad restricted premium hardens the stockout-consistent reading. A strict null is treated as support/thin-sample caution rather than as a rejection of the total-tur...


### 29. Dropout composition and underpowered auxiliary splits

The cell documents the composition of disappearing sellers and the power of auxiliary splits. Some subgroups contain few exposed FBA observations, so the corresponding split estimates carry low power. The cell flags every subgroup whose minimum detectable effect under 80% power and 5% size exceeds twice the headline dynamic coefficient. Flagged subgroups are reported but excluded from the main dynamic claim. The output is exported to `dynamic_dropout_composition_rankpct.csv`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 29. Dropout composition and underpowered auxiliary splits
# -----------------------------------------------------------------------------
# Document the composition of disappearing sellers and flag subgroups whose minimum detectable effect under 80% power exceeds twice the headline dynamic coefficient.

def _build_dropouts_above_split(panel):
    """Count FBA and non-FBA sellers disappearing above each focal seller.

    The implementation uses sorted dropout ranks within each transition instead
    of iterating over all focal-by-above-seller pairs. This preserves the exact
    definition and removes an otherwise fragile nested-loop bottleneck.
    """
    out = panel.copy()
    out["dropouts_above_fba"] = 0
    out["dropouts_above_nonfba"] = 0

    market_groups = {mid: g[["seller_id", "rank_pct", "fba_from_shipper"]].copy()
                     for mid, g in df_final.groupby("market_id", observed=True)}
    market_seller_sets = {mid: set(g["seller_id"]) for mid, g in df_final.groupby("market_id", observed=True)}

    for trans_id, idx in out.groupby("transition_id", observed=True).groups.items():
        group = out.loc[idx]
        lag_market_value = group["lag_market_id"].iloc[0]
        lead_market_value = group["lead_market_id"].iloc[0]
        lag_present = market_groups.get(lag_market_value, pd.DataFrame(columns=["seller_id", "rank_pct", "fba_from_shipper"]))
        lead_present_ids = market_seller_sets.get(lead_market_value, set())
        dropped = lag_present.loc[~lag_present["seller_id"].isin(lead_present_ids)].copy()
        if dropped.empty:
            continue
        dropped = dropped.sort_values("rank_pct")
        ranks = dropped["rank_pct"].to_numpy(dtype=float)
        fba = dropped["fba_from_shipper"].eq(1).to_numpy(dtype=int)
        cum_fba = np.cumsum(fba)
        cum_total = np.arange(1, len(dropped) + 1)
        focal_ranks = group["lag_rank_pct"].to_numpy(dtype=float)
        pos = np.searchsorted(ranks, focal_ranks, side="left")
        counts_fba = np.where(pos > 0, cum_fba[pos - 1], 0)
        counts_total = pos
        out.loc[idx, "dropouts_above_fba"] = counts_fba.astype(int)
        out.loc[idx, "dropouts_above_nonfba"] = (counts_total - counts_fba).astype(int)

    out["fba_x_dropouts_above_fba"] = out["fba_from_shipper"] * out["dropouts_above_fba"]
    out["fba_x_dropouts_above_nonfba"] = out["fba_from_shipper"] * out["dropouts_above_nonfba"]
    return out

dynamic_panel_with_split = _build_dropouts_above_split(dynamic_vacancy_panel)

_composition_fit = fit_dynamic_general_within_model(
    dynamic_panel_with_split,
    extra_cols=["dropouts_above_fba", "dropouts_above_nonfba",
                "fba_x_dropouts_above_fba", "fba_x_dropouts_above_nonfba"],
    static_rhs=SPECIFICATIONS[HEADLINE_SPEC],
)

composition_rows = [
    _term_row(_composition_fit, "fba_x_dropouts_above_fba"),
    _term_row(_composition_fit, "fba_x_dropouts_above_nonfba"),
]
for r in composition_rows:
    nm = r["term"]
    base_col = nm.replace("fba_x_", "")
    r["observations_with_positive_exposure"] = int((dynamic_panel_with_split[base_col] > 0).sum())

dropout_composition_table = pd.DataFrame(composition_rows)
print("Table F8 - Dropout composition: FBA vs non-FBA disappearances above")
display(dropout_composition_table[
    ["term", "estimate", "se", "t_stat", "pvalue", "ci_low", "ci_high",
     "observations_with_positive_exposure", "nobs"]
])
print()
print("Power caveat: dropouts_above_fba is sparse (FBA share ~19% in panel). Do not over-interpret.")

Table F8 - Dropout composition: FBA vs non-FBA disappearances above


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,observations_with_positive_exposure,nobs
0,fba_x_dropouts_above_fba,0.000763,0.000000,NaN,NaN,0.000763,0.000763,1097,4958
1,fba_x_dropouts_above_nonfba,-0.001931,22034.546315,-8.763046e-08,1.0,-44075.656933,44075.653072,843,4958



Power caveat: dropouts_above_fba is sparse (FBA share ~19% in panel). Do not over-interpret.


### 30. Baseline-rank heterogeneity

The cell tests whether the dynamic premium varies by starting rank position. Movement opportunities are mechanically related to where the focal seller sits in the lagged list, so the diagnostic is informative about whether the dynamic interaction reflects local rank composition or a broader turnover-period association. The cell estimates D3-style specifications with a quadratic interaction in lagged `rank_pct` and reports the marginal FBA-by-turnover effect at the 10th, 25th, 50th, 75th, and 90th percentiles of lagged `rank_pct`. The output `dynamic_continuous_heterogeneity_rankpct.csv` populates the heterogeneity column of `tab:app-ch6-dynamic-hardening`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 30. Baseline-rank heterogeneity
# -----------------------------------------------------------------------------
# Estimate D3-style specifications with a quadratic interaction in lagged rank_pct and report the marginal FBA-by-turnover effect at selected percentiles of lagged rank.

dynamic_panel_with_continuous = dynamic_vacancy_panel.copy()
dynamic_panel_with_continuous["lag_rank_pct_sq"] = (
    dynamic_panel_with_continuous["lag_rank_pct"] ** 2
)
dynamic_panel_with_continuous["dropouts_x_lag_rank"] = (
    dynamic_panel_with_continuous["dropouts_above"]
    * dynamic_panel_with_continuous["lag_rank_pct"]
)
dynamic_panel_with_continuous["fba_x_dropouts_x_lag_rank"] = (
    dynamic_panel_with_continuous["fba_from_shipper"]
    * dynamic_panel_with_continuous["dropouts_above"]
    * dynamic_panel_with_continuous["lag_rank_pct"]
)

_continuous_fit = fit_dynamic_general_within_model(
    dynamic_panel_with_continuous,
    extra_cols=["lag_rank_pct_sq", "dropouts_x_lag_rank", "fba_x_dropouts_x_lag_rank"],
    static_rhs=SPECIFICATIONS[HEADLINE_SPEC],
)

continuous_heterogeneity_table = pd.DataFrame([
    _term_row(_continuous_fit, "fba_x_dropouts_above"),
    _term_row(_continuous_fit, "fba_x_dropouts_x_lag_rank"),
    _term_row(_continuous_fit, "dropouts_x_lag_rank"),
])
print("Table F9 - Continuous baseline-rank heterogeneity (key coefficients)")
display(continuous_heterogeneity_table)

# Marginal effects at lag_rank_pct = q via linear combination + correct two-way SE
marginal_rows = []
for q_label, rk in [("p10", 0.10), ("p25", 0.25), ("median", 0.50), ("p75", 0.75), ("p90", 0.90)]:
    combo = _linear_combo_row(
        _continuous_fit,
        weights_dict={"fba_x_dropouts_above": 1.0,
                      "fba_x_dropouts_x_lag_rank": rk},
        label=f"marginal_at_{q_label}",
    )
    combo["lag_rank_pct_value"] = rk
    combo["quantile_label"]     = q_label
    marginal_rows.append(combo)

continuous_marginal_table = pd.DataFrame(marginal_rows)
print()
print("Table F9b - Marginal FBA vacancy response by baseline rank")
display(continuous_marginal_table[
    ["quantile_label", "lag_rank_pct_value", "estimate", "se", "pvalue", "ci_low", "ci_high"]
])

Table F9 - Continuous baseline-rank heterogeneity (key coefficients)


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs
0,fba_x_dropouts_above,-0.001730,0.001961,-0.882517,0.381019,-0.005653,0.002192,False,4958
1,fba_x_dropouts_x_lag_rank,0.003617,0.004649,0.777985,0.439635,-0.005683,0.012917,False,4958
2,dropouts_x_lag_rank,-0.011142,0.002667,-4.177004,0.000097,-0.016478,-0.005806,False,4958



Table F9b - Marginal FBA vacancy response by baseline rank


,quantile_label,lag_rank_pct_value,estimate,se,pvalue,ci_low,ci_high
0,p10,0.10,-0.001369,0.001636,0.406128,-0.004641,0.001904
1,p25,0.25,-0.000826,0.001317,0.532898,-0.003461,0.001809
2,median,0.50,0.000078,0.001526,0.959330,-0.002973,0.003130
3,p75,0.75,0.000982,0.002371,0.680102,-0.003760,0.005725
4,p90,0.90,0.001525,0.002982,0.611012,-0.004441,0.007491


### 31. Economic-salience outcome screen

The cell provides a compact linear-probability screen for several intuitive event outcomes around continuing-seller transitions: top-5 entry, top-10 entry, top-15 entry, and top-20 entry. The screen is reported for continuity and comparability across outcomes and is interpreted as descriptive rather than as a primary dynamic estimand. The output is exported to `dynamic_economic_salience_rankpct.csv`. The top-10 entry outcome is then promoted to a complementary binary-prominence estimand in Section 32.


In [ ]:
# -----------------------------------------------------------------------------
# Section 31. Economic-salience outcome screen across binary thresholds
# -----------------------------------------------------------------------------
# Linear-probability screen for top-5, top-10, top-15, and top-20 entry as descriptive auxiliary outcomes around continuing-seller transitions.

panel_economic = _churn_panel.copy()
panel_economic["improved_any"] = (panel_economic["rank_pct_improvement"] > 0).astype(int)
panel_economic["entered_top_quartile"] = (
    (panel_economic["lag_rank_pct"] > 0.25) & (panel_economic["lead_rank_pct"] <= 0.25)
).astype(int)
if "lag_rank_pos" in panel_economic.columns and "lead_rank_pos" in panel_economic.columns:
    panel_economic["entered_top_10"] = (
        (panel_economic["lag_rank_pos"] > 10) & (panel_economic["lead_rank_pos"] <= 10)
    ).astype(int)

economic_outcomes = ["improved_any", "entered_top_quartile"]
if "entered_top_10" in panel_economic.columns:
    economic_outcomes.append("entered_top_10")

_outcome_definitions = {
    "improved_any":         "1(rank_pct_improvement > 0)",
    "entered_top_quartile": "1(lag_rank_pct > 0.25 AND lead_rank_pct <= 0.25)",
    "entered_top_10":       "1(lag_rank_pos > 10 AND lead_rank_pos <= 10)",
}

economic_rows = []
for outcome in economic_outcomes:
    try:
        X_econ = make_dynamic_regressor_frame(
            panel_economic,
            SPECIFICATIONS[HEADLINE_SPEC],
            exposure_term="dropouts_total_focal",
            interaction_term=DYNAMIC_PREMIUM_TERM,
        )
        for _col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
            X_econ[_col] = pd.to_numeric(panel_economic[_col], errors="coerce").astype(float)
        fit_econ = _fit_dynamic_custom_from_frame(panel_economic, X_econ, outcome=outcome)
        row = _term_row(fit_econ, DYNAMIC_PREMIUM_TERM)
        condition_number = _fit_condition_number(fit_econ)
        economic_rows.append({
            "outcome": outcome,
            "outcome_definition": _outcome_definitions.get(outcome, ""),
            "estimator": "linear_probability_model_with_seller_and_transition_FE",
            "primary_rankpct_estimand": False,
            "role_in_claim": "auxiliary_screen; top10_fully_validated_in_next_section" if outcome == "entered_top_10" else "auxiliary_economic_salience_translation",
            "term": DYNAMIC_PREMIUM_TERM,
            "estimate_lambda": row["estimate"],
            "estimate": row["estimate"],
            "se": row["se"],
            "t_stat": row["t_stat"],
            "pvalue": row["pvalue"],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
            "nobs": fit_econ["nobs"],
            "baseline_rate": float(panel_economic[outcome].mean()),
            "condition_number": condition_number,
            "ill_conditioned_flag": bool(pd.notna(condition_number) and condition_number > 1e8),
            "interpretation": "Compact event-outcome screen. For entered_top_10, use the fully validated secondary-main binary prominence module below for interpretation.",
        })
    except Exception as exc:
        economic_rows.append({
            "outcome": outcome,
            "outcome_definition": _outcome_definitions.get(outcome, ""),
            "estimator": "linear_probability_model_with_seller_and_transition_FE",
            "primary_rankpct_estimand": False,
            "role_in_claim": "auxiliary_screen; top10_fully_validated_in_next_section" if outcome == "entered_top_10" else "auxiliary_economic_salience_translation",
            "term": DYNAMIC_PREMIUM_TERM,
            "estimate_lambda": np.nan,
            "estimate": np.nan,
            "se": np.nan,
            "t_stat": np.nan,
            "pvalue": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "nobs": np.nan,
            "baseline_rate": float(panel_economic[outcome].mean()) if outcome in panel_economic else np.nan,
            "condition_number": np.nan,
            "ill_conditioned_flag": True,
            "interpretation": f"Auxiliary binary salience model was not estimable: {exc}",
        })

economic_salience_table = pd.DataFrame(economic_rows)
print("Table F11 - Auxiliary dynamic economic-salience outcomes")
display(economic_salience_table)

Table F11 - Auxiliary dynamic economic-salience outcomes


,outcome,outcome_definition,estimator,primary_rankpct_estimand,role_in_claim,term,estimate_lambda,estimate,se,t_stat,pvalue,ci_low,ci_high,nobs,baseline_rate,condition_number,ill_conditioned_flag,interpretation
0,improved_any,1(rank_pct_improvement > 0),linear_probability_model_with_seller_and_transition_FE,False,auxiliary_economic_salience_translation,fba_x_dropouts_total_focal,-0.021010,-0.021010,0.018811,-1.116872,0.268502,-0.058638,0.016618,4958,0.396531,773.194374,False,"Compact event-outcome screen. For entered_top_10, use the fully validated secondary-main binary prominence module below for interpretation."
1,entered_top_quartile,1(lag_rank_pct > 0.25 AND lead_rank_pct <= 0.25),linear_probability_model_with_seller_and_transition_FE,False,auxiliary_economic_salience_translation,fba_x_dropouts_total_focal,0.002175,0.002175,0.001841,1.181405,0.242104,-0.001507,0.005857,4958,0.004639,773.194374,False,"Compact event-outcome screen. For entered_top_10, use the fully validated secondary-main binary prominence module below for interpretation."
2,entered_top_10,1(lag_rank_pos > 10 AND lead_rank_pos <= 10),linear_probability_model_with_seller_and_transition_FE,False,auxiliary_screen; top10_fully_validated_in_next_section,fba_x_dropouts_total_focal,0.011867,0.011867,0.004294,2.763334,0.007586,0.003277,0.020457,4958,0.004034,773.194374,False,"Compact event-outcome screen. For entered_top_10, use the fully validated secondary-main binary prominence module below for interpretation."


### 32. Complementary binary-prominence estimand: top-10 entry

The continuous `rank_pct_improvement` model is the primary econometric backbone because it uses the full ranking information. The top-10 entry outcome is reported as a complementary binary-prominence translation. The at-risk set contains sellers initially outside the top 10; the outcome equals one when such a seller enters the top 10 in the next retained market. The cell estimates a linear probability model with seller fixed effects, transition fixed effects, and the same control set as the continuous dynamic specification. The coefficient on `fba_x_dropouts_total_focal` is 0.0122 in the LPM. The model is flagged as ill-conditioned, the baseline event rate is approximately 0.5%, and the minimum detectable effect for this outcome lies outside the conventional rare-event power region. The exports populate the binary-prominence family of CSV tables and correspond to `tab:app-ch5-top10` of the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Section 32. Complementary binary-prominence estimand: top-10 entry
# -----------------------------------------------------------------------------
# Estimate the linear-probability model for top-10 entry on the at-risk set, with seller and transition fixed effects and the dynamic control set; audit fitted probabilities, inference, placebos, permutations, and stockout consistency.

BINARY_TOP10_COMPLEMENTARY_ESTIMAND_INCLUDED = bool(globals().get("BINARY_TOP10_COMPLEMENTARY_ESTIMAND_INCLUDED", True))
BINARY_TOP10_PROMINENCE_THRESHOLD = int(globals().get("BINARY_TOP10_PROMINENCE_THRESHOLD", 10))
BINARY_PROMINENCE_THRESHOLDS = list(globals().get("BINARY_PROMINENCE_THRESHOLDS", [5, 10, 15, 20]))
BINARY_TOP10_PERMUTATION_REPLICATIONS = int(globals().get("BINARY_TOP10_PERMUTATION_REPLICATIONS", 4999))
BINARY_TOP10_PERMUTATION_SEED = int(globals().get("BINARY_TOP10_PERMUTATION_SEED", WILD_BOOTSTRAP_SEED + 404))
BINARY_TOP10_WILD_BOOTSTRAP_REPLICATIONS = int(globals().get("BINARY_TOP10_WILD_BOOTSTRAP_REPLICATIONS", WILD_BOOTSTRAP_REPLICATIONS))


def _add_binary_prominence_outcomes(panel, thresholds=None):
    """Create top-k entry outcomes and risk-set flags for the dynamic panel."""
    thresholds = thresholds if thresholds is not None else BINARY_PROMINENCE_THRESHOLDS
    out = panel.copy()
    if "lag_rank_pos" not in out.columns or "lead_rank_pos" not in out.columns:
        raise KeyError("Top-k binary prominence outcomes require lag_rank_pos and lead_rank_pos.")
    for k in thresholds:
        k = int(k)
        out[f"risk_set_top_{k}"] = pd.to_numeric(out["lag_rank_pos"], errors="coerce").gt(k)
        out[f"entered_top_{k}"] = (
            out[f"risk_set_top_{k}"]
            & pd.to_numeric(out["lead_rank_pos"], errors="coerce").le(k)
        ).astype(int)
    out["risk_set_top_quartile"] = pd.to_numeric(out["lag_rank_pct"], errors="coerce").gt(0.25)
    out["entered_top_quartile_prominence"] = (
        out["risk_set_top_quartile"]
        & pd.to_numeric(out["lead_rank_pct"], errors="coerce").le(0.25)
    ).astype(int)
    return out


def _top10_design_matrix(panel, interaction_term=DYNAMIC_PREMIUM_TERM, exposure_term="dropouts_total_focal", include_directional_control=True):
    """Build the binary-prominence design matrix using the same controls as the continuous primary design."""
    X = make_dynamic_regressor_frame(
        panel,
        SPECIFICATIONS[HEADLINE_SPEC],
        exposure_term=exposure_term,
        interaction_term=interaction_term,
    )
    if include_directional_control:
        for col in ["dropout_direction_balance", "fba_x_dropout_direction_balance"]:
            if col in panel.columns:
                X[col] = pd.to_numeric(panel[col], errors="coerce").astype(float)
    return X


def _prominence_event_diagnostics(panel, outcome, risk_col, threshold_label):
    """Summarize event support for a binary top-of-list outcome."""
    risk = panel.loc[panel[risk_col].fillna(False)].copy()
    if len(risk) == 0:
        return {
            "threshold_label": threshold_label,
            "risk_set_observations": 0,
            "events": 0,
            "event_rate": np.nan,
            "fba_risk_observations": 0,
            "nonfba_risk_observations": 0,
            "fba_events": 0,
            "nonfba_events": 0,
            "fba_event_rate": np.nan,
            "nonfba_event_rate": np.nan,
            "transitions_with_events": 0,
            "sellers_with_events": 0,
            "rare_event_warning": True,
            "support_warning": "empty risk set",
        }
    fba_mask = risk["fba_from_shipper"].astype(int).eq(1)
    nonfba_mask = ~fba_mask
    y = pd.to_numeric(risk[outcome], errors="coerce").fillna(0).astype(int)
    fba_y = y.loc[fba_mask]
    nonfba_y = y.loc[nonfba_mask]
    events_by_transition = risk.loc[y.eq(1)].groupby("transition_id", observed=True).size()
    events_by_seller = risk.loc[y.eq(1)].groupby("seller_id", observed=True).size()
    events = int(y.sum())
    fba_events = int(fba_y.sum()) if len(fba_y) else 0
    nonfba_events = int(nonfba_y.sum()) if len(nonfba_y) else 0
    event_rate = float(y.mean()) if len(y) else np.nan
    rare = bool(
        events < 30
        or (pd.notna(event_rate) and event_rate < 0.01)
        or fba_events < 10
        or nonfba_events < 10
        or events_by_transition.shape[0] < 20
        or events_by_seller.shape[0] < 15
    )
    return {
        "threshold_label": threshold_label,
        "risk_set_observations": int(len(risk)),
        "risk_set_sellers": int(risk["seller_id"].nunique()),
        "risk_set_transitions": int(risk["transition_id"].nunique()),
        "events": events,
        "event_rate": event_rate,
        "fba_risk_observations": int(fba_mask.sum()),
        "nonfba_risk_observations": int(nonfba_mask.sum()),
        "fba_events": fba_events,
        "nonfba_events": nonfba_events,
        "fba_event_rate": float(fba_y.mean()) if len(fba_y) else np.nan,
        "nonfba_event_rate": float(nonfba_y.mean()) if len(nonfba_y) else np.nan,
        "transitions_with_events": int(events_by_transition.shape[0]),
        "sellers_with_events": int(events_by_seller.shape[0]),
        "max_events_in_single_transition": int(events_by_transition.max()) if len(events_by_transition) else 0,
        "max_events_in_single_seller": int(events_by_seller.max()) if len(events_by_seller) else 0,
        "rare_event_warning": rare,
        "support_warning": (
            "sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats"
            if rare else "event support acceptable only for auxiliary salience analysis, subject to fitted-probability and finite-cluster audits"
        ),
    }


def _fit_binary_prominence_lpm(panel, outcome, risk_col, model_role, threshold_label, include_directional_control=True, absorb_transition_col="transition_id"):
    """Estimate the fixed-effects LPM on the appropriate top-k risk set."""
    if risk_col not in panel.columns or outcome not in panel.columns:
        raise KeyError(f"Missing {risk_col} or {outcome} in binary-prominence panel.")
    risk_panel = panel.loc[panel[risk_col].fillna(False)].copy()
    if len(risk_panel) == 0:
        raise ValueError(f"Empty risk set for {threshold_label}.")
    if absorb_transition_col != "transition_id" and absorb_transition_col not in risk_panel.columns:
        raise KeyError(f"Missing absorbed effect {absorb_transition_col}.")
    X = _top10_design_matrix(risk_panel, include_directional_control=include_directional_control)
    fit = _fit_dynamic_custom_from_frame(
        risk_panel,
        X,
        outcome=outcome,
        absorb_transition_col=absorb_transition_col,
        cluster_transition_col="transition_id",
        fixed_effects_label=f"seller_id and {absorb_transition_col}",
    )
    row = _custom_term_row(
        fit,
        DYNAMIC_PREMIUM_TERM,
        model_role,
        "Linear-probability estimate of FBA top-of-list entry premium during total seller-list turnover.",
    )
    row.update({
        "outcome": outcome,
        "threshold_label": threshold_label,
        "risk_set_rule": risk_col,
        "include_directional_control": bool(include_directional_control),
        "fixed_effects": fit.get("fixed_effects_label", "seller_id and transition_id"),
        "risk_set_observations_after_dropna": int(fit.get("nobs", 0)),
        "baseline_event_rate_in_model_frame": float(fit["model_frame"][outcome].mean()) if "model_frame" in fit and outcome in fit["model_frame"] else np.nan,
    })
    return fit, row


binary_prominence_panel = _add_binary_prominence_outcomes(_churn_panel, BINARY_PROMINENCE_THRESHOLDS)
TOP10_OUTCOME = f"entered_top_{BINARY_TOP10_PROMINENCE_THRESHOLD}"
TOP10_RISK_COL = f"risk_set_top_{BINARY_TOP10_PROMINENCE_THRESHOLD}"

# Event-support diagnostics for top 10 and neighboring prominence thresholds.
_binary_diag_rows = []
for _k in BINARY_PROMINENCE_THRESHOLDS:
    _binary_diag_rows.append(
        _prominence_event_diagnostics(
            binary_prominence_panel,
            outcome=f"entered_top_{int(_k)}",
            risk_col=f"risk_set_top_{int(_k)}",
            threshold_label=f"top_{int(_k)}",
        )
    )
_binary_diag_rows.append(
    _prominence_event_diagnostics(
        binary_prominence_panel,
        outcome="entered_top_quartile_prominence",
        risk_col="risk_set_top_quartile",
        threshold_label="top_quartile",
    )
)
binary_top10_prominence_diagnostics_table = pd.DataFrame(_binary_diag_rows)

# Primary and parsimonious top-10 models.
try:
    binary_top10_preferred_fit, _binary_top10_preferred_row = _fit_binary_prominence_lpm(
        binary_prominence_panel,
        outcome=TOP10_OUTCOME,
        risk_col=TOP10_RISK_COL,
        model_role="complementary_top10_entry_primary_lpm",
        threshold_label="top_10",
        include_directional_control=True,
    )
except Exception as exc:
    binary_top10_preferred_fit = None
    _binary_top10_preferred_row = _diagnostic_fallback_dynamic_row(
        DYNAMIC_PREMIUM_TERM,
        "complementary_top10_entry_primary_lpm",
        "Primary top-10 entry LPM failed or was underidentified.",
        sample_rule=TOP10_RISK_COL,
        exc=exc,
        extra={"outcome": TOP10_OUTCOME, "threshold_label": "top_10", "risk_set_rule": TOP10_RISK_COL},
    )
try:
    binary_top10_parsimonious_fit, _binary_top10_parsimonious_row = _fit_binary_prominence_lpm(
        binary_prominence_panel,
        outcome=TOP10_OUTCOME,
        risk_col=TOP10_RISK_COL,
        model_role="complementary_top10_entry_parsimonious_lpm",
        threshold_label="top_10",
        include_directional_control=False,
    )
except Exception as exc:
    binary_top10_parsimonious_fit = None
    _binary_top10_parsimonious_row = _diagnostic_fallback_dynamic_row(
        DYNAMIC_PREMIUM_TERM,
        "complementary_top10_entry_parsimonious_lpm",
        "Parsimonious top-10 entry LPM failed or was underidentified.",
        sample_rule=TOP10_RISK_COL,
        exc=exc,
        extra={"outcome": TOP10_OUTCOME, "threshold_label": "top_10", "risk_set_rule": TOP10_RISK_COL},
    )

# Rank-tier stress test for the binary outcome.
try:
    _top10_ranktier_panel = assign_dynamic_rank_tier(binary_prominence_panel.loc[binary_prominence_panel[TOP10_RISK_COL].fillna(False)].copy())
    _top10_ranktier_panel["transition_rank_tier_fe"] = (
        _top10_ranktier_panel["transition_id"].astype(str)
        + "__"
        + _top10_ranktier_panel["lag_rank_tier"].astype(str)
    )
    binary_top10_ranktier_fit, _binary_top10_ranktier_row = _fit_binary_prominence_lpm(
        _top10_ranktier_panel,
        outcome=TOP10_OUTCOME,
        risk_col=TOP10_RISK_COL,
        model_role="complementary_top10_entry_rank_tier_fe_stress",
        threshold_label="top_10",
        include_directional_control=True,
        absorb_transition_col="transition_rank_tier_fe",
    )
except Exception as exc:
    binary_top10_ranktier_fit = None
    _binary_top10_ranktier_row = _diagnostic_fallback_dynamic_row(
        DYNAMIC_PREMIUM_TERM,
        "complementary_top10_entry_rank_tier_fe_stress",
        "Top-10 rank-tier FE stress test failed or was underidentified.",
        sample_rule="risk_set_top_10 with transition_id x lag_rank_tier FE",
        exc=exc,
        extra={"outcome": TOP10_OUTCOME, "threshold_label": "top_10", "risk_set_rule": TOP10_RISK_COL},
    )

binary_top10_prominence_models_table = pd.DataFrame([
    _binary_top10_preferred_row,
    _binary_top10_parsimonious_row,
    _binary_top10_ranktier_row,
])


def _binary_lpm_fitted_probability_audit(fit, outcome, model_role):
    """Audit raw fitted probabilities from a fixed-effects LPM.

    The within estimator returns residuals from the full fixed-effects regression
    by the Frisch-Waugh-Lovell theorem. Raw fitted values are therefore obtained
    as observed binary outcomes minus the regression residuals. Values outside
    [0, 1] are not errors in an LPM, but they document why the binary prominence
    result is auxiliary rather than structural.
    """
    if not isinstance(fit, dict) or "model_frame" not in fit or "residuals" not in fit:
        return pd.DataFrame([{
            "model_role": model_role,
            "outcome": outcome,
            "nobs": 0,
            "events": np.nan,
            "event_rate": np.nan,
            "fitted_min": np.nan,
            "fitted_p01": np.nan,
            "fitted_p05": np.nan,
            "fitted_median": np.nan,
            "fitted_p95": np.nan,
            "fitted_p99": np.nan,
            "fitted_max": np.nan,
            "fitted_below_zero_rows": np.nan,
            "fitted_above_one_rows": np.nan,
            "fitted_outside_unit_interval_rows": np.nan,
            "fitted_outside_unit_interval_share": np.nan,
            "negative_fitted_share": np.nan,
            "above_one_fitted_share": np.nan,
            "audit_status": "model_not_available",
            "interpretation": "Top-10 LPM fitted-probability audit unavailable because the preferred model was not estimable.",
        }])
    frame = fit["model_frame"]
    if outcome not in frame.columns:
        raise KeyError(f"Outcome {outcome!r} not found in binary top-10 model frame.")
    y_raw = pd.to_numeric(frame[outcome], errors="coerce").to_numpy(dtype=float)
    residuals = np.asarray(fit["residuals"], dtype=float)
    fitted = y_raw - residuals
    finite = np.isfinite(fitted)
    fitted_finite = fitted[finite]
    if fitted_finite.size == 0:
        return pd.DataFrame([{
            "model_role": model_role,
            "outcome": outcome,
            "nobs": int(len(y_raw)),
            "events": float(np.nansum(y_raw)),
            "event_rate": float(np.nanmean(y_raw)) if len(y_raw) else np.nan,
            "fitted_min": np.nan,
            "fitted_p01": np.nan,
            "fitted_p05": np.nan,
            "fitted_median": np.nan,
            "fitted_p95": np.nan,
            "fitted_p99": np.nan,
            "fitted_max": np.nan,
            "fitted_below_zero_rows": np.nan,
            "fitted_above_one_rows": np.nan,
            "fitted_outside_unit_interval_rows": np.nan,
            "fitted_outside_unit_interval_share": np.nan,
            "negative_fitted_share": np.nan,
            "above_one_fitted_share": np.nan,
            "audit_status": "no_finite_fitted_values",
            "interpretation": "Top-10 LPM fitted-probability audit found no finite fitted values.",
        }])
    below_zero = int((fitted_finite < 0).sum())
    above_one = int((fitted_finite > 1).sum())
    outside = below_zero + above_one
    share = outside / fitted_finite.size
    return pd.DataFrame([{
        "model_role": model_role,
        "outcome": outcome,
        "nobs": int(fitted_finite.size),
        "events": float(np.nansum(y_raw)),
        "event_rate": float(np.nanmean(y_raw)) if len(y_raw) else np.nan,
        "fitted_min": float(np.min(fitted_finite)),
        "fitted_p01": float(np.quantile(fitted_finite, 0.01)),
        "fitted_p05": float(np.quantile(fitted_finite, 0.05)),
        "fitted_median": float(np.quantile(fitted_finite, 0.50)),
        "fitted_p95": float(np.quantile(fitted_finite, 0.95)),
        "fitted_p99": float(np.quantile(fitted_finite, 0.99)),
        "fitted_max": float(np.max(fitted_finite)),
        "fitted_below_zero_rows": below_zero,
        "fitted_above_one_rows": above_one,
        "fitted_outside_unit_interval_rows": outside,
        "fitted_outside_unit_interval_share": float(share),
        "negative_fitted_share": float(below_zero / fitted_finite.size),
        "above_one_fitted_share": float(above_one / fitted_finite.size),
        "audit_status": "outside_unit_interval_detected" if outside else "all_fitted_values_inside_unit_interval",
        "interpretation": "Fixed-effects LPM fitted values are audited only to document bounded-outcome limitations; the top-10 result remains auxiliary.",
    }])

binary_top10_fitted_probability_audit_table = _binary_lpm_fitted_probability_audit(
    binary_top10_preferred_fit,
    outcome=TOP10_OUTCOME,
    model_role="complementary_top10_entry_primary_lpm",
)


# Finite-cluster inference for the primary top-10 model.
_binary_top10_inference_rows = []
if isinstance(binary_top10_preferred_fit, dict) and DYNAMIC_PREMIUM_TERM in binary_top10_preferred_fit.get("names", []):
    _top10_pref_term = _term_row(binary_top10_preferred_fit, DYNAMIC_PREMIUM_TERM)
    _binary_top10_inference_rows.append({
        "method": "preferred_two_way_seller_transition_clustered_lpm",
        "outcome": TOP10_OUTCOME,
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate": _top10_pref_term.get("estimate", np.nan),
        "se": _top10_pref_term.get("se", np.nan),
        "test_statistic": _top10_pref_term.get("t_stat", np.nan),
        "pvalue": _top10_pref_term.get("pvalue", np.nan),
        "pvalue_type": "two_sided_t",
        "dof_reference": binary_top10_preferred_fit.get("dof", np.nan),
        "note": "Primary complementary top-10 LPM with seller and transition fixed effects.",
    })
    _binary_top10_seller_codes = cluster_codes(binary_top10_preferred_fit["model_frame"]["seller_id"])
    _binary_top10_transition_codes = cluster_codes(binary_top10_preferred_fit["model_frame"]["transition_id"])
    _cr2_seller = _cr2_dynamic_one_way_for_term(binary_top10_preferred_fit, _binary_top10_seller_codes, "cr2_seller", DYNAMIC_PREMIUM_TERM)
    _cr2_transition = _cr2_dynamic_one_way_for_term(binary_top10_preferred_fit, _binary_top10_transition_codes, "cr2_transition", DYNAMIC_PREMIUM_TERM)
    _wcb_top10 = _wild_bootstrap_dynamic_twoway_for_term(
        binary_top10_preferred_fit,
        DYNAMIC_PREMIUM_TERM,
        replications=BINARY_TOP10_WILD_BOOTSTRAP_REPLICATIONS,
        seed=WILD_BOOTSTRAP_SEED + 505,
    )
    for _row in [_cr2_seller, _cr2_transition, _wcb_top10]:
        _row["outcome"] = TOP10_OUTCOME
        _row["complementary_binary_prominence"] = True
        _binary_top10_inference_rows.append(_row)
else:
    _binary_top10_inference_rows.append({
        "method": "preferred_two_way_seller_transition_clustered_lpm",
        "outcome": TOP10_OUTCOME,
        "term": DYNAMIC_PREMIUM_TERM,
        "estimate": np.nan,
        "se": np.nan,
        "test_statistic": np.nan,
        "pvalue": np.nan,
        "pvalue_type": "unavailable",
        "dof_reference": np.nan,
        "note": "Primary top-10 model absorbed or not estimable.",
    })
binary_top10_prominence_inference_table = pd.DataFrame(_binary_top10_inference_rows)

# Timing placebo 1: future turnover should not predict current top-10 entry.
_binary_top10_placebo_rows = []
try:
    _transition_turnover_top10 = (
        binary_prominence_panel[["transition_order", "dropouts_total_transition"]]
        .drop_duplicates()
        .sort_values("transition_order")
    )
    _transition_turnover_top10["future_dropouts_total_transition"] = _transition_turnover_top10["dropouts_total_transition"].shift(-1)
    _top10_future_panel = binary_prominence_panel.merge(
        _transition_turnover_top10[["transition_order", "future_dropouts_total_transition"]],
        on="transition_order",
        how="left",
    )
    _top10_future_panel["fba_x_future_dropouts_total"] = (
        _top10_future_panel["fba_from_shipper"].astype(float)
        * _top10_future_panel["future_dropouts_total_transition"].astype(float)
    )
    _top10_future_risk = _top10_future_panel[TOP10_RISK_COL].fillna(False) & _top10_future_panel["future_dropouts_total_transition"].notna()
    _X_top10_future = _top10_design_matrix(
        _top10_future_panel,
        interaction_term="fba_x_future_dropouts_total",
        exposure_term="future_dropouts_total_transition",
        include_directional_control=True,
    )
    _top10_future_fit = _fit_dynamic_custom_from_frame(
        _top10_future_panel.loc[_top10_future_risk].copy(),
        _X_top10_future.loc[_top10_future_risk].copy(),
        outcome=TOP10_OUTCOME,
    )
    _top10_future_row = _custom_term_row(
        _top10_future_fit,
        "fba_x_future_dropouts_total",
        "top10_future_turnover_placebo",
        "Future turnover should not predict current top-10 entry if the binary prominence result is not driven by anticipatory FBA trends.",
    )
except Exception as exc:
    _top10_future_row = _diagnostic_fallback_dynamic_row(
        "fba_x_future_dropouts_total",
        "top10_future_turnover_placebo",
        "Future-turnover placebo failed or was underidentified.",
        sample_rule="risk_set_top_10",
        exc=exc,
        extra={"outcome": TOP10_OUTCOME},
    )
_binary_top10_placebo_rows.append(_top10_future_row)

# Timing placebo 2: current turnover should not predict previous top-10 entry.
try:
    _top10_pretrend_panel = binary_prominence_panel.sort_values(["seller_id", "transition_order"]).copy()
    _top10_pretrend_panel["previous_entered_top_10"] = _top10_pretrend_panel.groupby("seller_id", observed=True)[TOP10_OUTCOME].shift(1)
    _top10_pretrend_panel["previous_risk_set_top_10"] = _top10_pretrend_panel.groupby("seller_id", observed=True)[TOP10_RISK_COL].shift(1)
    _top10_pretrend_risk = _top10_pretrend_panel["previous_risk_set_top_10"].astype("boolean").fillna(False).astype(bool)
    _X_top10_pretrend = _top10_design_matrix(_top10_pretrend_panel, include_directional_control=True)
    _top10_pretrend_fit = _fit_dynamic_custom_from_frame(
        _top10_pretrend_panel.loc[_top10_pretrend_risk].copy(),
        _X_top10_pretrend.loc[_top10_pretrend_risk].copy(),
        outcome="previous_entered_top_10",
    )
    _top10_pretrend_row = _custom_term_row(
        _top10_pretrend_fit,
        DYNAMIC_PREMIUM_TERM,
        "top10_pretrend_placebo",
        "Current turnover should not predict top-10 entry that occurred in the previous seller transition.",
    )
except Exception as exc:
    _top10_pretrend_row = _diagnostic_fallback_dynamic_row(
        DYNAMIC_PREMIUM_TERM,
        "top10_pretrend_placebo",
        "Previous-entry placebo failed or was underidentified.",
        sample_rule="previous risk_set_top_10",
        exc=exc,
        extra={"outcome": "previous_entered_top_10"},
    )
_binary_top10_placebo_rows.append(_top10_pretrend_row)
binary_top10_prominence_placebo_table = pd.DataFrame(_binary_top10_placebo_rows)

# Permutation inference within transition x lag-rank-tier cells.
rng_top10 = np.random.default_rng(BINARY_TOP10_PERMUTATION_SEED)
try:
    _top10_perm_base = assign_dynamic_rank_tier(binary_prominence_panel.loc[binary_prominence_panel[TOP10_RISK_COL].fillna(False)].copy())
    observed_top10_beta = float(_binary_top10_preferred_row.get("estimate", np.nan))
    top10_perm_betas = []
    for _b in range(int(BINARY_TOP10_PERMUTATION_REPLICATIONS)):
        tmp = _top10_perm_base.copy()
        tmp["fba_perm"] = (
            tmp.groupby(["transition_id", "lag_rank_tier"], observed=True)["fba_from_shipper"]
            .transform(lambda s: rng_top10.permutation(s.to_numpy()))
        )
        tmp["fba_perm_x_dropouts_total_focal"] = tmp["fba_perm"].astype(float) * tmp["dropouts_total_focal"].astype(float)
        _X_perm_top10 = _top10_design_matrix(
            tmp,
            interaction_term="fba_perm_x_dropouts_total_focal",
            exposure_term="dropouts_total_focal",
            include_directional_control=True,
        )
        top10_perm_betas.append(
            _beta_only_dynamic_term(
                tmp,
                _X_perm_top10,
                "fba_perm_x_dropouts_total_focal",
                outcome=TOP10_OUTCOME,
            )
        )
    top10_perm_betas = np.asarray(top10_perm_betas, dtype=float)
    top10_perm_betas = top10_perm_betas[np.isfinite(top10_perm_betas)]
    top10_perm_p = float((1 + np.sum(np.abs(top10_perm_betas) >= abs(observed_top10_beta))) / (len(top10_perm_betas) + 1)) if len(top10_perm_betas) else np.nan
    top10_perm_mc_se = float(np.sqrt(top10_perm_p * (1 - top10_perm_p) / max(len(top10_perm_betas) + 1, 1))) if pd.notna(top10_perm_p) else np.nan
    binary_top10_prominence_permutation_table = pd.DataFrame([{
        "outcome": TOP10_OUTCOME,
        "observed_beta": observed_top10_beta,
        "permutation_mean": float(np.mean(top10_perm_betas)) if len(top10_perm_betas) else np.nan,
        "permutation_sd": float(np.std(top10_perm_betas, ddof=1)) if len(top10_perm_betas) > 1 else np.nan,
        "permutation_p_value_two_sided": top10_perm_p,
        "permutation_monte_carlo_se": top10_perm_mc_se,
        "valid_permutations": int(len(top10_perm_betas)),
        "requested_permutations": int(BINARY_TOP10_PERMUTATION_REPLICATIONS),
        "permutation_cell": "transition_id x lag_rank_tier",
        "interpretation": "Randomizes FBA status locally within transition-by-starting-rank-tier cells for the secondary top-10 prominence coefficient.",
    }])
except Exception as exc:
    binary_top10_prominence_permutation_table = pd.DataFrame([{
        "outcome": TOP10_OUTCOME,
        "observed_beta": np.nan,
        "permutation_mean": np.nan,
        "permutation_sd": np.nan,
        "permutation_p_value_two_sided": np.nan,
        "permutation_monte_carlo_se": np.nan,
        "valid_permutations": 0,
        "requested_permutations": int(BINARY_TOP10_PERMUTATION_REPLICATIONS),
        "permutation_cell": "transition_id x lag_rank_tier",
        "interpretation": f"Top-10 permutation diagnostic failed: {exc}",
    }])

# Threshold-family robustness: top 5, top 10, top 15, top 20, and top quartile.
_threshold_rows = []
for _k in BINARY_PROMINENCE_THRESHOLDS:
    try:
        _fit_k, _row_k = _fit_binary_prominence_lpm(
            binary_prominence_panel,
            outcome=f"entered_top_{int(_k)}",
            risk_col=f"risk_set_top_{int(_k)}",
            model_role=f"threshold_family_top_{int(_k)}",
            threshold_label=f"top_{int(_k)}",
            include_directional_control=True,
        )
        _row_k["threshold_order"] = int(_k)
        _threshold_rows.append(_row_k)
    except Exception as exc:
        _threshold_rows.append(_diagnostic_fallback_dynamic_row(
            DYNAMIC_PREMIUM_TERM,
            f"threshold_family_top_{int(_k)}",
            "Threshold-family LPM failed or was underidentified.",
            sample_rule=f"risk_set_top_{int(_k)}",
            exc=exc,
            extra={"outcome": f"entered_top_{int(_k)}", "threshold_label": f"top_{int(_k)}", "threshold_order": int(_k)},
        ))
try:
    _fit_q, _row_q = _fit_binary_prominence_lpm(
        binary_prominence_panel,
        outcome="entered_top_quartile_prominence",
        risk_col="risk_set_top_quartile",
        model_role="threshold_family_top_quartile",
        threshold_label="top_quartile",
        include_directional_control=True,
    )
    _row_q["threshold_order"] = 25
    _threshold_rows.append(_row_q)
except Exception as exc:
    _threshold_rows.append(_diagnostic_fallback_dynamic_row(
        DYNAMIC_PREMIUM_TERM,
        "threshold_family_top_quartile",
        "Top-quartile threshold-family LPM failed or was underidentified.",
        sample_rule="risk_set_top_quartile",
        exc=exc,
        extra={"outcome": "entered_top_quartile_prominence", "threshold_label": "top_quartile", "threshold_order": 25},
    ))

binary_top10_prominence_threshold_family_table = pd.DataFrame(_threshold_rows)
if "pvalue" in binary_top10_prominence_threshold_family_table.columns:
    _pvals = pd.to_numeric(binary_top10_prominence_threshold_family_table["pvalue"], errors="coerce")
    _finite = _pvals.notna()
    binary_top10_prominence_threshold_family_table["fdr_qvalue_bh"] = np.nan
    binary_top10_prominence_threshold_family_table["fdr_reject_10pct"] = False
    if _finite.any():
        _reject, _qvals, _, _ = multipletests(_pvals.loc[_finite].to_numpy(), alpha=0.10, method="fdr_bh")
        binary_top10_prominence_threshold_family_table.loc[_finite, "fdr_qvalue_bh"] = _qvals
        binary_top10_prominence_threshold_family_table.loc[_finite, "fdr_reject_10pct"] = _reject

# Stockout-consistent binary prominence restrictions.
_binary_stockout_rows = []
for _definition_label, _panel_restricted in [
    ("broad_later_return_nonprice", dynamic_stockout_consistent_panel if "dynamic_stockout_consistent_panel" in globals() else pd.DataFrame()),
    ("strict_within_two_snapshots_nonprice", dynamic_stockout_strict_panel if "dynamic_stockout_strict_panel" in globals() else pd.DataFrame()),
]:
    try:
        _panel_restricted = _add_binary_prominence_outcomes(_panel_restricted, BINARY_PROMINENCE_THRESHOLDS)
        _fit_s, _row_s = _fit_binary_prominence_lpm(
            _panel_restricted,
            outcome=TOP10_OUTCOME,
            risk_col=TOP10_RISK_COL,
            model_role="top10_stockout_consistent_directional_controlled",
            threshold_label="top_10",
            include_directional_control=True,
        )
        _row_s["stockout_definition"] = _definition_label
        _row_s["restricted_turnover_exposure_count"] = int(_panel_restricted["dropouts_total_transition"].sum()) if "dropouts_total_transition" in _panel_restricted else np.nan
        _binary_stockout_rows.append(_row_s)
    except Exception as exc:
        _binary_stockout_rows.append(_diagnostic_fallback_dynamic_row(
            DYNAMIC_PREMIUM_TERM,
            "top10_stockout_consistent_directional_controlled",
            "Stockout-consistent top-10 model failed or was underidentified.",
            sample_rule=f"{_definition_label}; {TOP10_RISK_COL}",
            exc=exc,
            extra={"outcome": TOP10_OUTCOME, "threshold_label": "top_10", "stockout_definition": _definition_label},
        ))
binary_top10_prominence_stockout_table = pd.DataFrame(_binary_stockout_rows)

# Influence diagnostics: leave one seller or one transition out of the primary top-10 LPM.
def _binary_top10_leave_one_summary(panel, fit, grouping_col):
    if not isinstance(fit, dict) or DYNAMIC_PREMIUM_TERM not in fit.get("names", []):
        return pd.DataFrame(), pd.DataFrame([{
            "grouping_col": grouping_col,
            "observed_beta": np.nan,
            "n_groups_tested": 0,
            "max_abs_delta": np.nan,
            "min_beta": np.nan,
            "max_beta": np.nan,
            "interpretation": "primary top-10 model not available",
        }])
    risk_panel = panel.loc[panel[TOP10_RISK_COL].fillna(False)].copy()
    observed_beta = float(_term_row(fit, DYNAMIC_PREMIUM_TERM).get("estimate", np.nan))
    base_rows = []
    for group_value in sorted(risk_panel[grouping_col].dropna().unique()):
        subset = risk_panel.loc[~risk_panel[grouping_col].eq(group_value)].copy()
        if len(subset) == 0:
            beta_leave = np.nan
        else:
            X_leave = _top10_design_matrix(subset, include_directional_control=True)
            beta_leave = _beta_only_dynamic_term(subset, X_leave, DYNAMIC_PREMIUM_TERM, outcome=TOP10_OUTCOME)
        base_rows.append({
            "grouping_col": grouping_col,
            "left_out_group": group_value,
            "observed_beta": observed_beta,
            "leave_one_beta": beta_leave,
            "delta": beta_leave - observed_beta if pd.notna(beta_leave) and pd.notna(observed_beta) else np.nan,
        })
    table = pd.DataFrame(base_rows)
    if len(table) and table["leave_one_beta"].notna().any():
        summary = pd.DataFrame([{
            "grouping_col": grouping_col,
            "observed_beta": observed_beta,
            "n_groups_tested": int(table["leave_one_beta"].notna().sum()),
            "max_abs_delta": float(table["delta"].abs().max()),
            "min_beta": float(table["leave_one_beta"].min()),
            "max_beta": float(table["leave_one_beta"].max()),
            "sign_flip_any": bool((np.sign(table["leave_one_beta"].dropna()) != np.sign(observed_beta)).any()) if pd.notna(observed_beta) else False,
            "interpretation": "Leave-one sensitivity for the secondary top-10 prominence estimate.",
        }])
    else:
        summary = pd.DataFrame([{
            "grouping_col": grouping_col,
            "observed_beta": observed_beta,
            "n_groups_tested": 0,
            "max_abs_delta": np.nan,
            "min_beta": np.nan,
            "max_beta": np.nan,
            "sign_flip_any": np.nan,
            "interpretation": "No finite leave-one estimates.",
        }])
    return table, summary

_leave_transition_table, _leave_transition_summary = _binary_top10_leave_one_summary(binary_prominence_panel, binary_top10_preferred_fit, "transition_id")
_leave_seller_table, _leave_seller_summary = _binary_top10_leave_one_summary(binary_prominence_panel, binary_top10_preferred_fit, "seller_id")
binary_top10_prominence_leave_one_table = pd.concat([_leave_transition_table, _leave_seller_table], ignore_index=True)
binary_top10_prominence_influence_summary_table = pd.concat([_leave_transition_summary, _leave_seller_summary], ignore_index=True)

# Decision table for the top-10 complementary estimand.
_top10_diag = binary_top10_prominence_diagnostics_table.loc[
    binary_top10_prominence_diagnostics_table["threshold_label"].eq("top_10")
]
_top10_pref_p = float(_binary_top10_preferred_row.get("pvalue", np.nan)) if pd.notna(_binary_top10_preferred_row.get("pvalue", np.nan)) else np.nan
_top10_pref_beta = float(_binary_top10_preferred_row.get("estimate", np.nan)) if pd.notna(_binary_top10_preferred_row.get("estimate", np.nan)) else np.nan
_top10_conservative_p = pd.to_numeric(binary_top10_prominence_inference_table["pvalue"], errors="coerce").dropna()
_top10_conservative_p = float(_top10_conservative_p.max()) if len(_top10_conservative_p) else np.nan
_top10_future_p = float(_top10_future_row.get("pvalue", np.nan)) if pd.notna(_top10_future_row.get("pvalue", np.nan)) else np.nan
_top10_pretrend_p = float(_top10_pretrend_row.get("pvalue", np.nan)) if pd.notna(_top10_pretrend_row.get("pvalue", np.nan)) else np.nan
_top10_perm_p = float(binary_top10_prominence_permutation_table["permutation_p_value_two_sided"].iloc[0]) if len(binary_top10_prominence_permutation_table) and pd.notna(binary_top10_prominence_permutation_table["permutation_p_value_two_sided"].iloc[0]) else np.nan
_top10_ranktier_p = float(_binary_top10_ranktier_row.get("pvalue", np.nan)) if pd.notna(_binary_top10_ranktier_row.get("pvalue", np.nan)) else np.nan
_top10_ranktier_beta = float(_binary_top10_ranktier_row.get("estimate", np.nan)) if pd.notna(_binary_top10_ranktier_row.get("estimate", np.nan)) else np.nan
_top10_threshold_row = binary_top10_prominence_threshold_family_table.loc[
    binary_top10_prominence_threshold_family_table["threshold_label"].eq("top_10")
]
_top10_fdr_q = float(_top10_threshold_row["fdr_qvalue_bh"].iloc[0]) if len(_top10_threshold_row) and pd.notna(_top10_threshold_row["fdr_qvalue_bh"].iloc[0]) else np.nan
_top10_rare_warning = bool(_top10_diag["rare_event_warning"].iloc[0]) if len(_top10_diag) else True
_top10_temporal_placebos_pass = bool(pd.notna(_top10_future_p) and _top10_future_p >= 0.10 and pd.notna(_top10_pretrend_p) and _top10_pretrend_p >= 0.10)
_top10_preferred_pass = bool(pd.notna(_top10_pref_beta) and _top10_pref_beta > 0 and pd.notna(_top10_pref_p) and _top10_pref_p < 0.05)
_top10_conservative_pass = bool(pd.notna(_top10_conservative_p) and _top10_conservative_p < 0.10)
_top10_permutation_support = bool(pd.notna(_top10_perm_p) and _top10_perm_p < 0.10)
_top10_threshold_support = bool(pd.notna(_top10_fdr_q) and _top10_fdr_q < 0.10)
_top10_ranktier_sensitive = bool(pd.notna(_top10_ranktier_beta) and (np.sign(_top10_ranktier_beta) != np.sign(_top10_pref_beta) or (pd.notna(_top10_ranktier_p) and _top10_ranktier_p >= 0.10))) if pd.notna(_top10_pref_beta) else True
if _top10_preferred_pass and _top10_temporal_placebos_pass and (_top10_permutation_support or _top10_conservative_pass):
    _top10_decision = "complementary_supported_with_rank_tier_and_sparse_event_caution"
else:
    _top10_decision = "auxiliary_or_exploratory_until_validation_passes"

binary_top10_prominence_decision_table = pd.DataFrame([{
    "outcome": TOP10_OUTCOME,
    "risk_set_rule": "lag_rank_pos > 10",
    "preferred_beta": _top10_pref_beta,
    "preferred_pvalue": _top10_pref_p,
    "conservative_max_pvalue_across_inference": _top10_conservative_p,
    "future_turnover_placebo_pvalue": _top10_future_p,
    "pretrend_placebo_pvalue": _top10_pretrend_p,
    "permutation_p_value_two_sided": _top10_perm_p,
    "threshold_family_fdr_qvalue_top10": _top10_fdr_q,
    "rank_tier_stress_beta": _top10_ranktier_beta,
    "rank_tier_stress_pvalue": _top10_ranktier_p,
    "rare_event_warning": _top10_rare_warning,
    "preferred_pass_5pct": _top10_preferred_pass,
    "temporal_placebos_pass_10pct": _top10_temporal_placebos_pass,
    "conservative_inference_pass_10pct": _top10_conservative_pass,
    "permutation_support_10pct": _top10_permutation_support,
    "threshold_family_support_10pct_fdr": _top10_threshold_support,
    "rank_tier_sensitive": _top10_ranktier_sensitive,
    "decision": _top10_decision,
    "interpretation": "Top-10 entry is treated as complementary prominence evidence. It supports economic salience if positive and validated, but it does not replace the continuous rank-percent estimand.",
}])

if EXPORT_FILES:
    binary_top10_prominence_diagnostics_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_diagnostics_rankpct.csv", index=False)
    binary_top10_prominence_models_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_models_rankpct.csv", index=False)
    binary_top10_prominence_inference_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_inference_rankpct.csv", index=False)
    binary_top10_prominence_placebo_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_placebos_rankpct.csv", index=False)
    binary_top10_prominence_permutation_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_permutation_rankpct.csv", index=False)
    binary_top10_prominence_threshold_family_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_threshold_family_rankpct.csv", index=False)
    binary_top10_prominence_stockout_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_stockout_consistency_rankpct.csv", index=False)
    binary_top10_prominence_leave_one_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_leave_one_rankpct.csv", index=False)
    binary_top10_prominence_influence_summary_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_influence_summary_rankpct.csv", index=False)
    binary_top10_prominence_decision_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_decision_rankpct.csv", index=False)
    binary_top10_fitted_probability_audit_table.to_csv(OUTPUT_DIR / "binary_top10_fitted_probability_audit_rankpct.csv", index=False)

print("Complementary binary-prominence diagnostics")
display(binary_top10_prominence_diagnostics_table)
print("Complementary top-10 primary, parsimonious, and rank-tier-stress models")
display(binary_top10_prominence_models_table)
print("Complementary top-10 inference checks")
display(binary_top10_prominence_inference_table)
print("Complementary top-10 placebos and permutation")
display(binary_top10_prominence_placebo_table)
display(binary_top10_prominence_permutation_table)
print("Threshold-family robustness")
display(binary_top10_prominence_threshold_family_table)
print("Top-10 stockout-consistency and influence diagnostics")
display(binary_top10_prominence_stockout_table)
display(binary_top10_prominence_influence_summary_table)
print("Top-10 fitted-probability audit")
display(binary_top10_fitted_probability_audit_table)
print("Top-10 decision table")
display(binary_top10_prominence_decision_table)

Complementary binary-prominence diagnostics


,threshold_label,risk_set_observations,risk_set_sellers,risk_set_transitions,events,event_rate,fba_risk_observations,nonfba_risk_observations,fba_events,nonfba_events,fba_event_rate,nonfba_event_rate,transitions_with_events,sellers_with_events,max_events_in_single_transition,max_events_in_single_seller,rare_event_warning,support_warning
0,top_5,4660,109,61,10,0.002146,803,3857,6,4,0.007472,0.001037,9,7,2,3,True,sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats
1,top_10,4366,97,61,20,0.004581,632,3734,15,5,0.023734,0.001339,18,8,2,7,True,sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats
2,top_15,4063,93,61,19,0.004676,440,3623,11,8,0.025000,0.002208,17,11,2,4,True,sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats
3,top_20,3759,87,61,20,0.005321,243,3516,10,10,0.041152,0.002844,19,8,2,5,True,sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats
4,top_quartile,3700,89,61,23,0.006216,240,3460,9,14,0.037500,0.004046,23,11,1,4,True,sparse-event support; usable only as auxiliary salience evidence with finite-cluster and fitted-probability caveats


Complementary top-10 primary, parsimonious, and rank-tier-stress models


,term,estimate,se,t_stat,pvalue,ci_low,ci_high,absorbed_or_missing,nobs,model_role,condition_number,ill_conditioned_flag,n_regressors_after_absorption,interpretation,outcome,threshold_label,risk_set_rule,include_directional_control,fixed_effects,risk_set_observations_after_dropna,baseline_event_rate_in_model_frame
0,fba_x_dropouts_total_focal,0.012174,0.005032,2.419290,0.018599,0.002108,0.022239,False,4366,complementary_top10_entry_primary_lpm,1.946583e+12,True,12,Linear-probability estimate of FBA top-of-list entry premium during total seller-list turnover.,entered_top_10,top_10,risk_set_top_10,True,seller_id and transition_id,4366,0.004581
1,fba_x_dropouts_total_focal,0.010090,0.005299,1.904254,0.061675,-0.000509,0.020689,False,4366,complementary_top10_entry_parsimonious_lpm,1.946583e+12,True,10,Linear-probability estimate of FBA top-of-list entry premium during total seller-list turnover.,entered_top_10,top_10,risk_set_top_10,False,seller_id and transition_id,4366,0.004581
2,fba_x_dropouts_total_focal,0.007353,0.004113,1.787713,0.078873,-0.000874,0.015581,False,4366,complementary_top10_entry_rank_tier_fe_stress,2.757390e+08,True,12,Linear-probability estimate of FBA top-of-list entry premium during total seller-list turnover.,entered_top_10,top_10,risk_set_top_10,True,seller_id and transition_rank_tier_fe,4366,0.004581


Complementary top-10 inference checks


...

,grouping_col,observed_beta,n_groups_tested,max_abs_delta,min_beta,max_beta,sign_flip_any,interpretation
0,transition_id,0.012174,61,0.005909,0.006264,0.014911,False,Leave-one sensitivity for the secondary top-10 prominence estimate.
1,seller_id,0.012174,97,0.004356,0.007818,0.013967,False,Leave-one sensitivity for the secondary top-10 prominence estimate.


Top-10 fitted-probability audit


,model_role,outcome,nobs,events,event_rate,fitted_min,fitted_p01,fitted_p05,fitted_median,fitted_p95,fitted_p99,fitted_max,fitted_below_zero_rows,fitted_above_one_rows,fitted_outside_unit_interval_rows,fitted_outside_unit_interval_share,negative_fitted_share,above_one_fitted_share,audit_status,interpretation
0,complementary_top10_entry_primary_lpm,entered_top_10,4366,20.0,0.004581,-0.027848,-0.020256,-0.012554,-0.00102,0.030597,0.161749,1.0,2410,0,2410,0.551993,0.551993,0.0,outside_unit_interval_detected,Fixed-effects LPM fitted values are audited only to document bounded-outcome limitations; the top-10 result remains auxiliary.


Top-10 decision table


,outcome,risk_set_rule,preferred_beta,preferred_pvalue,conservative_max_pvalue_across_inference,future_turnover_placebo_pvalue,pretrend_placebo_pvalue,permutation_p_value_two_sided,threshold_family_fdr_qvalue_top10,rank_tier_stress_beta,rank_tier_stress_pvalue,rare_event_warning,preferred_pass_5pct,temporal_placebos_pass_10pct,conservative_inference_pass_10pct,permutation_support_10pct,threshold_family_support_10pct_fdr,rank_tier_sensitive,decision,interpretation
0,entered_top_10,lag_rank_pos > 10,0.012174,0.018599,0.10004,0.829544,0.664189,0.0102,0.092996,0.007353,0.078873,True,True,True,False,True,True,False,complementary_supported_with_rank_tier_and_sparse_event_caution,"Top-10 entry is treated as complementary prominence evidence. It supports economic salience if positive and validated, but it does not replace the continuous rank-percent estim..."


### 33. Power and detectable-effect audit

The power audit reports minimum detectable effects for the dynamic design and for selected subgroups. Fixed-effects interaction models with clustered inference do not admit a clean closed-form power calculation, so the audit uses a simulation-based approximation that holds the design, the within-cluster correlation, and the residual variance fixed at their estimated values. The output is exported to `dynamic_mde_summary_rankpct.csv` and `dynamic_mde_subgroup_audit_rankpct.csv`. The 80% power minimum detectable effect for the baseline D1 model is approximately 0.0022 on `rank_pct_improvement`, close to the observed coefficient of 0.002467.


In [ ]:
# -----------------------------------------------------------------------------
# Section 33. Power and detectable-effect audit
# -----------------------------------------------------------------------------
# Simulation-based minimum-detectable-effect calculation for the dynamic design under the estimated within-cluster correlation and residual variance.

def _safe_binary_count(series, value=1):
    s = pd.to_numeric(series, errors="coerce")
    return int(s.eq(value).sum())


def _dynamic_support_counts(subpanel, exposure_col="exposed_to_dropout_above"):
    if subpanel is None or len(subpanel) == 0:
        return {
            "observations": 0,
            "sellers": 0,
            "transitions": 0,
            "fba_observations": 0,
            "nonfba_observations": 0,
            "exposed_observations": np.nan,
            "exposed_fba_observations": np.nan,
        }
    fba_series = pd.to_numeric(subpanel.get("fba_from_shipper", pd.Series(index=subpanel.index, dtype=float)), errors="coerce")
    exposed_series = pd.to_numeric(subpanel.get(exposure_col, pd.Series(index=subpanel.index, dtype=float)), errors="coerce")
    return {
        "observations": int(len(subpanel)),
        "sellers": int(subpanel["seller_id"].nunique()) if "seller_id" in subpanel else 0,
        "transitions": int(subpanel["transition_id"].nunique()) if "transition_id" in subpanel else 0,
        "fba_observations": int(fba_series.eq(1).sum()),
        "nonfba_observations": int(fba_series.eq(0).sum()),
        "exposed_observations": int(exposed_series.gt(0).sum()) if exposure_col in subpanel else np.nan,
        "exposed_fba_observations": int((exposed_series.gt(0) & fba_series.eq(1)).sum()) if exposure_col in subpanel else np.nan,
    }


def _row_for_mde(model_or_subsample, outcome, subpanel, se, baseline_rate=None, exposure_col="exposed_to_dropout_above"):
    counts = _dynamic_support_counts(subpanel, exposure_col=exposure_col)
    se = float(se) if pd.notna(se) else np.nan
    mde = MDE_MULTIPLIER_80_POWER_ALPHA_005 * se if pd.notna(se) else np.nan
    mean_market_size = float(df_final.groupby("market_id").size().mean())
    median_market_size = float(df_final.groupby("market_id").size().median())
    mde_positions_mean = mde * mean_market_size if pd.notna(mde) and outcome == DYNAMIC_OUTCOME else np.nan
    mde_positions_median = mde * median_market_size if pd.notna(mde) and outcome == DYNAMIC_OUTCOME else np.nan
    if baseline_rate is None:
        baseline_rate = float(subpanel[outcome].mean()) if len(subpanel) and outcome in subpanel.columns else np.nan

    warning_reasons = []
    if pd.notna(se) and se <= 0:
        warning_reasons.append("nonpositive estimated SE")
    if pd.notna(counts["exposed_fba_observations"]) and counts["exposed_fba_observations"] < 50:
        warning_reasons.append("exposed FBA observations < 50")
    if counts["sellers"] < 30:
        warning_reasons.append("seller clusters < 30")
    if counts["transitions"] < 30:
        warning_reasons.append("transition clusters < 30")
    if pd.notna(mde_positions_mean) and mde_positions_mean > 1.0:
        warning_reasons.append("MDE in rank positions > 1.0")
    if outcome != DYNAMIC_OUTCOME and pd.notna(baseline_rate) and (baseline_rate < 0.05 or baseline_rate > 0.95):
        warning_reasons.append("binary baseline event rate outside [0.05, 0.95]")

    return {
        "model_or_subsample": model_or_subsample,
        "outcome": outcome,
        **counts,
        "control_mean_or_baseline_rate": baseline_rate,
        "estimated_se_for_key_term": se,
        "mde_80_power_alpha_0_05": mde,
        "mde_as_rank_positions_if_applicable": mde_positions_mean,
        "mde_as_rank_positions_mean_reference": mde_positions_mean,
        "mde_as_rank_positions_median_reference": mde_positions_median,
        "mean_market_size_reference": mean_market_size,
        "median_market_size_reference": median_market_size,
        "power_warning": bool(warning_reasons),
        "power_warning_reason": "; ".join(warning_reasons) if warning_reasons else "",
    }

_mde_rows = []

# Primary D1 MDE: total seller turnover, matching the final dynamic premium estimand.
_primary_d1 = dynamic_fba_premium_inference_table.loc[
    dynamic_fba_premium_inference_table["method"].eq("preferred_two_way_seller_transition_clustered")
].iloc[0]
_tmp_total_turnover = _churn_panel.copy() if "_churn_panel" in globals() else dynamic_vacancy_panel.copy()
if "dropouts_total_focal" not in _tmp_total_turnover.columns and {"dropouts_above", "dropouts_below"}.issubset(_tmp_total_turnover.columns):
    _tmp_total_turnover["dropouts_total_focal"] = (
        pd.to_numeric(_tmp_total_turnover["dropouts_above"], errors="coerce").fillna(0).astype(float)
        + pd.to_numeric(_tmp_total_turnover["dropouts_below"], errors="coerce").fillna(0).astype(float)
    )
_tmp_total_turnover["exposed_to_total_turnover"] = (
    pd.to_numeric(_tmp_total_turnover["dropouts_total_focal"], errors="coerce").fillna(0).gt(0).astype(int)
)
_mde_rows.append(
    _row_for_mde(
        "primary D1 total-turnover dynamic model",
        DYNAMIC_OUTCOME,
        _tmp_total_turnover,
        _primary_d1["se"],
        exposure_col="exposed_to_total_turnover",
    )
)

# Directional above-turnover reference MDE: retained for decomposition context only.
_preferred_dynamic = dynamic_vacancy_models_table.loc[
    dynamic_vacancy_models_table["specification"].eq(PREFERRED_DYNAMIC_SPEC)
].iloc[0]
_mde_rows.append(
    _row_for_mde(
        "above-turnover directional reference model",
        DYNAMIC_OUTCOME,
        dynamic_vacancy_panel,
        _preferred_dynamic["se"],
        exposure_col="exposed_to_dropout_above",
    )
)

_tiered_for_mde = assign_dynamic_rank_tier(dynamic_vacancy_panel)
for tier_name, label in [("top_25pct", "top rank quartile subsample"), ("middle_50pct", "middle rank subsample"), ("bottom_25pct", "bottom rank quartile subsample")]:
    tier_panel = _tiered_for_mde.loc[_tiered_for_mde["lag_rank_tier"].eq(tier_name)].copy()
    tier_row = dynamic_vacancy_rank_tier_summary_table.loc[dynamic_vacancy_rank_tier_summary_table["lag_rank_tier"].eq(tier_name)]
    tier_se = float(tier_row["se"].iloc[0]) if len(tier_row) else np.nan
    _mde_rows.append(_row_for_mde(label, DYNAMIC_OUTCOME, tier_panel, tier_se))

for outcome_name, label in [("entered_top_10", "top-10 outcome"), ("entered_top_quartile", "top-quartile outcome")]:
    if "panel_economic" in globals() and outcome_name in panel_economic.columns:
        est_row = economic_salience_table.loc[economic_salience_table["outcome"].eq(outcome_name)]
        se = float(est_row["se"].iloc[0]) if len(est_row) else np.nan
        baseline_rate = float(est_row["baseline_rate"].iloc[0]) if len(est_row) else float(panel_economic[outcome_name].mean())
        _tmp_econ_mde = panel_economic.copy()
        if "dropouts_total_focal" in _tmp_econ_mde.columns:
            _tmp_econ_mde["exposed_to_total_turnover"] = (
                pd.to_numeric(_tmp_econ_mde["dropouts_total_focal"], errors="coerce").fillna(0).gt(0).astype(int)
            )
            _mde_rows.append(_row_for_mde(label, outcome_name, _tmp_econ_mde, se, baseline_rate=baseline_rate, exposure_col="exposed_to_total_turnover"))
        else:
            _mde_rows.append(_row_for_mde(label, outcome_name, panel_economic, se, baseline_rate=baseline_rate))

if "dynamic_panel_with_split" in globals() and "dropout_composition_table" in globals():
    for exposure_col, label, term in [
        ("dropouts_above_fba", "dropout-composition FBA-above exposure", "fba_x_dropouts_above_fba"),
        ("dropouts_above_nonfba", "dropout-composition non-FBA-above exposure", "fba_x_dropouts_above_nonfba"),
    ]:
        tmp = dynamic_panel_with_split.copy()
        indicator_col = f"exposed_to_{exposure_col}"
        tmp[indicator_col] = pd.to_numeric(tmp[exposure_col], errors="coerce").gt(0).astype(int)
        est_row = dropout_composition_table.loc[dropout_composition_table["term"].eq(term)]
        se = float(est_row["se"].iloc[0]) if len(est_row) else np.nan
        _mde_rows.append(_row_for_mde(label, DYNAMIC_OUTCOME, tmp, se, exposure_col=indicator_col))

dynamic_support_mde_table = pd.DataFrame(_mde_rows)

# Audit-compatible aliases used by text, exports, and audit checks.
dynamic_power_audit = dynamic_support_mde_table.copy()
mde_subgroup_audit = dynamic_power_audit.copy()
mde_summary = dynamic_power_audit.loc[
    dynamic_power_audit["model_or_subsample"].eq("primary D1 total-turnover dynamic model")
].copy()

print("Table F1b. Dynamic support and detectable effects")
display(dynamic_support_mde_table)

Table F1b. Dynamic support and detectable effects


,model_or_subsample,outcome,observations,sellers,transitions,fba_observations,nonfba_observations,exposed_observations,exposed_fba_observations,control_mean_or_baseline_rate,estimated_se_for_key_term,mde_80_power_alpha_0_05,mde_as_rank_positions_if_applicable,mde_as_rank_positions_mean_reference,mde_as_rank_positions_median_reference,mean_market_size_reference,median_market_size_reference,power_warning,power_warning_reason
0,primary D1 total-turnover dynamic model,rank_pct_improvement,4958,114,61,962,3996,2863,559,0.000671,0.000765,0.002141,1.763931e-01,1.763931e-01,1.713158e-01,82.370968,80.0,False,
1,above-turnover directional reference model,rank_pct_improvement,4958,114,61,962,3996,1739,227,0.000671,0.001678,0.004697,3.869345e-01,3.869345e-01,3.757970e-01,82.370968,80.0,False,
2,top rank quartile subsample,rank_pct_improvement,1258,36,61,722,536,244,144,-0.000222,0.001603,0.004489,3.697341e-01,3.697341e-01,3.590916e-01,82.370968,80.0,False,
3,middle rank subsample,rank_pct_improvement,2456,67,61,228,2228,865,74,-0.000082,0.003078,0.008618,7.099092e-01,7.099092e-01,6.894751e-01,82.370968,80.0,False,
4,bottom rank quartile subsample,rank_pct_improvement,1244,41,61,12,1232,630,9,0.003063,0.001053,0.002949,2.429468e-01,2.429468e-01,2.359538e-01,82.370968,80.0,True,exposed FBA observations < 50
5,top-10 outcome,entered_top_10,4958,114,61,962,3996,2863,559,0.004034,0.004294,0.012024,NaN,NaN,NaN,82.370968,80.0,True,"binary baseline event rate outside [0.05, 0.95]"
6,top-quartile outcome,entered_top_quartile,4958,114,61,962,3996,2863,559,0.004639,0.001841,0.005154,NaN,NaN,NaN,82.370968,80.0,True,"binary baseline event rate outside [0.05, 0.95]"
7,dropout-composition FBA-above exposure,rank_pct_improvement,4958,114,61,962,3996,1097,189,0.000671,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,82.370968,80.0,True,nonpositive estimated SE
8,dropout-composition non-FBA-above exposure,rank_pct_improvement,4958,114,61,962,3996,843,44,0.000671,22034.546315,61696.729681,5.082019e+06,5.082019e+06,4.935738e+06,82.370968,80.0,True,exposed FBA observations < 50; MDE in rank positions > 1.0


### 34. Feature interdependence and control-block contribution

The cell quantifies how strongly the observed offer characteristics co-move and how each control block contributes to the residual variation that identifies the FBA coefficient. The diagnostics include pairwise feature correlations after market demeaning, block-level F-tests for joint contribution, variance inflation factors, and a list of the top correlated feature pairs. The reputation block is internally collinear (positive-review percentage and star rating correlate at 0.96 after market demeaning), but the FBA coefficient is robust to dropping either reputation variable. The outputs populate `tab:app-ch6-vif` and the standardized-association entries of `tab:ch5-standardized-features`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 34. Feature interdependence and control-block contribution
# -----------------------------------------------------------------------------
# Pairwise feature correlations after market demeaning, block-level F-tests, variance inflation factors, and the top correlated feature pairs.

feature_cols = [
    "fba_from_shipper",
    "log1p_num_valutazioni",
    "valutazioni_positive",
    "stelle",
    "prezzo",
    "prezzo_spedizione_repaired",
    "contact_courier_flag",
    "g_cons_min_robust",
]

# Reuse the headline covariance computed in the main-estimation section. Recomputing
# the two-way covariance here is unnecessary and can slow audit reruns.
if not all(name in globals() for name in ["headline_cov", "headline_dof", "headline_cluster_info"]):
    headline_cov, headline_dof, headline_cluster_info = covariance_for_inference(
        headline_plain_res,
        df_final,
        HEADLINE_INFERENCE,
    )

standardized_feature_table = standardized_feature_association_table(
    headline_plain_res,
    headline_cov,
    df_final,
    feature_cols,
    headline_dof,
)
display(standardized_feature_table)

feature_correlation_table, top_feature_correlation_pairs_table, feature_vif_table = feature_correlation_and_vif_tables(
    df_final,
    feature_cols,
)
display(feature_correlation_table)
display(top_feature_correlation_pairs_table)
display(feature_vif_table)

feature_blocks = {
    "reputation_block": ["log1p_num_valutazioni", "valutazioni_positive", "stelle"],
    "commercial_terms_block": ["prezzo", "prezzo_spedizione_repaired", "contact_courier_flag"],
    "delivery_block": ["g_cons_min_robust"],
}

feature_block_wald_table = feature_block_wald_tests(
    headline_plain_res,
    headline_cov,
    feature_blocks,
    headline_dof,
)
display(feature_block_wald_table)

,feature,model_term,raw_coefficient,cluster_robust_se,test_statistic,pvalue,ci_low,ci_high,stars,feature_mean,feature_sd,outcome_sd,standardized_coefficient,standardized_se,standardization_note,status
0,fba_from_shipper,fba_from_shipper,-0.051038,0.015968,-3.196231,0.002207,-0.082969,-0.019108,***,0.195026,0.396260,0.29223,-0.069207,0.021653,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
1,log1p_num_valutazioni,log1p_num_valutazioni,-0.003823,0.002538,-1.505960,0.137238,-0.008899,0.001253,,5.281450,2.304314,0.29223,-0.030144,0.020016,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
2,valutazioni_positive,valutazioni_positive,0.000590,0.001262,0.467559,0.641766,-0.001934,0.003114,,84.849031,17.092159,0.29223,0.034520,0.073831,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
3,stelle,stelle,-0.020657,0.026193,-0.788622,0.433388,-0.073033,0.031720,,4.320540,0.760972,0.29223,-0.053790,0.068208,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
4,prezzo,prezzo,0.030579,0.001189,25.713495,0.000000,0.028201,0.032957,***,50.924525,7.969336,0.29223,0.833901,0.032430,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
5,prezzo_spedizione_repaired,prezzo_spedizione_repaired,0.030283,0.001525,19.852086,0.000000,0.027233,0.033333,***,2.906107,4.966114,0.29223,0.514622,0.025923,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
6,contact_courier_flag,contact_courier_flag,0.330592,0.014238,23.219552,0.000000,0.302122,0.359062,***,0.009790,0.098471,0.29223,0.111397,0.004798,Raw coefficient scaled by feature SD and rank_pct SD.,estimated
7,g_cons_min_robust,g_cons_min_robust,-0.002791,0.001239,-2.252912,0.027872,-0.005269,-0.000314,**,6.885256,3.590603,0.29223,-0.034299,0.015224,Raw coefficient scaled by feature SD and rank_pct SD.,estimated


,feature,fba_from_shipper,log1p_num_valutazioni,valutazioni_positive,stelle,prezzo,prezzo_spedizione_repaired,contact_courier_flag,g_cons_min_robust,correlation_note
0,fba_from_shipper,1.000000,-0.232539,0.291668,0.252557,-0.357899,-0.287080,-0.050010,-0.031842,Market-demeaned correlations; market-level common components are removed.
1,log1p_num_valutazioni,-0.232539,1.000000,0.353829,0.316785,0.002159,-0.105038,-0.027094,-0.223572,Market-demeaned correlations; market-level common components are removed.
2,valutazioni_positive,0.291668,0.353829,1.000000,0.955243,-0.107565,-0.279066,-0.020341,-0.290060,Market-demeaned correlations; market-level common components are removed.
3,stelle,0.252557,0.316785,0.955243,1.000000,-0.056485,-0.285099,-0.042488,-0.239120,Market-demeaned correlations; market-level common components are removed.
4,prezzo,-0.357899,0.002159,-0.107565,-0.056485,1.000000,-0.070428,-0.051486,0.064837,Market-demeaned correlations; market-level common components are removed.
5,prezzo_spedizione_repaired,-0.287080,-0.105038,-0.279066,-0.285099,-0.070428,1.000000,-0.056587,0.185313,Market-demeaned correlations; market-level common components are removed.
6,contact_courier_flag,-0.050010,-0.027094,-0.020341,-0.042488,-0.051486,-0.056587,1.000000,-0.069949,Market-demeaned correlations; market-level common components are removed.
7,g_cons_min_robust,-0.031842,-0.223572,-0.290060,-0.239120,0.064837,0.185313,-0.069949,1.000000,Market-demeaned correlations; market-level common components are removed.


,feature_1,feature_2,correlation,abs_correlation,correlation_note
0,valutazioni_positive,stelle,0.955243,0.955243,Market-demeaned correlations; market-level common components are removed.
1,fba_from_shipper,prezzo,-0.357899,0.357899,Market-demeaned correlations; market-level common components are removed.
2,log1p_num_valutazioni,valutazioni_positive,0.353829,0.353829,Market-demeaned correlations; market-level common components are removed.
3,log1p_num_valutazioni,stelle,0.316785,0.316785,Market-demeaned correlations; market-level common components are removed.
4,fba_from_shipper,valutazioni_positive,0.291668,0.291668,Market-demeaned correlations; market-level common components are removed.
5,valutazioni_positive,g_cons_min_robust,-0.290060,0.290060,Market-demeaned correlations; market-level common components are removed.
6,fba_from_shipper,prezzo_spedizione_repaired,-0.287080,0.287080,Market-demeaned correlations; market-level common components are removed.
7,stelle,prezzo_spedizione_repaired,-0.285099,0.285099,Market-demeaned correlations; market-level common components are removed.
8,valutazioni_positive,prezzo_spedizione_repaired,-0.279066,0.279066,Market-demeaned correlations; market-level common components are removed.
9,fba_from_shipper,stelle,0.252557,0.252557,Market-demeaned correlations; market-level common components are removed.


,feature,vif,vif_note
0,fba_from_shipper,1.647015,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
1,log1p_num_valutazioni,1.401956,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
2,valutazioni_positive,13.369048,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
3,stelle,12.248540,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
4,prezzo,1.239405,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
5,prezzo_spedizione_repaired,1.252320,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
6,contact_courier_flag,1.032211,VIF computed on market-demeaned standardized features from the inverse correlation matrix.
7,g_cons_min_robust,1.159039,VIF computed on market-demeaned standardized features from the inverse correlation matrix.


,feature_block,features,model_terms,df_num,df_denom,test_statistic,wald_chi2,pvalue,stars,test_type,status,missing_features
0,reputation_block,"[log1p_num_valutazioni, valutazioni_positive, stelle]","[log1p_num_valutazioni, valutazioni_positive, stelle]",3,61.0,1.519669,4.559006,2.184129e-01,,F test using supplied denominator degrees of freedom,estimated,[]
1,commercial_terms_block,"[prezzo, prezzo_spedizione_repaired, contact_courier_flag]","[prezzo, prezzo_spedizione_repaired, contact_courier_flag]",3,61.0,237.233420,711.700260,1.415280e-33,***,F test using supplied denominator degrees of freedom,estimated,[]
2,delivery_block,[g_cons_min_robust],[g_cons_min_robust],1,61.0,5.075613,5.075613,2.787233e-02,**,F test using supplied denominator degrees of freedom,estimated,[]


### 35. Interaction analysis

The interaction analysis explores whether the static FBA association varies with observed seller and offer characteristics. Interactions are evaluated only when the underlying moderator has sufficient within-market support and a clear empirical interpretation. The cell estimates supported single-moderator interaction models for product price, standard delivery days, log seller-review count, and positive-review percentage; reports the FBA association at selected moderator values; and corrects the interaction p-values for multiple testing using the Benjamini-Hochberg FDR. The outputs are exported to `interaction_margins_rankpct.csv`, `interaction_terms_rankpct.csv`, and `interaction_interpretation_rankpct.csv`; they populate `tab:ch5-interaction-margins` and `tab:app-ch5-interaction-terms`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 35. Interaction analysis on the static specification
# -----------------------------------------------------------------------------
# Estimate supported single-moderator interaction models, report the FBA association at selected moderator values, and correct interaction p-values with the Benjamini-Hochberg FDR.

INTERACTION_MODERATORS = [
    ("prezzo", "product_price", "primary"),
    ("g_cons_min_robust", "delivery_days", "primary"),
    ("log1p_num_valutazioni", "review_count_log", "secondary"),
    ("valutazioni_positive", "positive_review_percent", "secondary"),
]

excluded_interactions_table = pd.DataFrame(
    [
        {
            "excluded_moderator": "prezzo_spedizione_repaired",
            "reason": "FBA offers have zero shipping price in the final sample; positive-shipping FBA margins would be extrapolated outside empirical support.",
        }
    ]
)
display(excluded_interactions_table)

interaction_support_table = []
for raw_var, label, priority in INTERACTION_MODERATORS:
    q = df_final.groupby("fba_from_shipper")[raw_var].quantile([0.25, 0.50, 0.75]).unstack()
    interaction_support_table.append(
        {
            "moderator": raw_var,
            "label": label,
            "priority": priority,
            "fba_p25": float(q.loc[1, 0.25]),
            "fba_median": float(q.loc[1, 0.50]),
            "fba_p75": float(q.loc[1, 0.75]),
            "nonfba_p25": float(q.loc[0, 0.25]),
            "nonfba_median": float(q.loc[0, 0.50]),
            "nonfba_p75": float(q.loc[0, 0.75]),
        }
    )
interaction_support_table = pd.DataFrame(interaction_support_table)
display(interaction_support_table)

for raw_var, _, _ in INTERACTION_MODERATORS:
    centered_var = f"{raw_var}_c"
    df_final[centered_var] = df_final[raw_var] - df_final[raw_var].mean()

interaction_term_rows = []
interaction_margin_rows = []

for raw_var, label, priority in INTERACTION_MODERATORS:
    centered_var = f"{raw_var}_c"
    interaction_term = f"fba_from_shipper:{centered_var}"
    interaction_spec = f"interaction_single_{label}"
    rhs_interaction = headline_rhs + [interaction_term]

    res_plain = fit_market_fe_ols_plain(df_final, rhs_interaction)
    cov, dof, cluster_info = covariance_for_inference(res_plain, df_final, HEADLINE_INFERENCE)

    fba_row = extract_term_row_from_cov(
        res_plain,
        cov,
        specification=interaction_spec,
        estimator="market_fe_ols_interaction",
        sample_label="full_sample",
        inference_label=HEADLINE_INFERENCE,
        dof=dof,
        cluster_info=cluster_info,
        base_name="fba_from_shipper",
    )
    fba_row.update({"moderator": raw_var, "moderator_label": label, "priority": priority, "term_type": "main_fba_at_mean_moderator"})
    interaction_term_rows.append(fba_row)

    inter_row = extract_term_row_from_cov(
        res_plain,
        cov,
        specification=interaction_spec,
        estimator="market_fe_ols_interaction",
        sample_label="full_sample",
        inference_label=HEADLINE_INFERENCE,
        dof=dof,
        cluster_info=cluster_info,
        base_name=interaction_term,
    )
    inter_row.update({"moderator": raw_var, "moderator_label": label, "priority": priority, "term_type": "fba_by_moderator_interaction"})
    interaction_term_rows.append(inter_row)

    interaction_margin_rows.extend(
        interaction_margins_for_moderator(
            res_plain=res_plain,
            cov=cov,
            dof=dof,
            data=df_final,
            raw_var=raw_var,
            centered_var=centered_var,
            specification=interaction_spec,
            inference=HEADLINE_INFERENCE,
        )
    )

interaction_terms_table = pd.DataFrame(interaction_term_rows)
interaction_margins_table = pd.DataFrame(interaction_margin_rows)

mask_interactions = interaction_terms_table["term_type"].eq("fba_by_moderator_interaction")
pvals = interaction_terms_table.loc[mask_interactions, "pvalue_fba"].to_numpy(dtype=float)
if len(pvals):
    reject_holm, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")
    reject_bh, p_bh, _, _ = multipletests(pvals, alpha=0.10, method="fdr_bh")
    interaction_terms_table.loc[mask_interactions, "pvalue_holm_0_05"] = p_holm
    interaction_terms_table.loc[mask_interactions, "reject_holm_0_05"] = reject_holm
    interaction_terms_table.loc[mask_interactions, "pvalue_fdr_bh_0_10"] = p_bh
    interaction_terms_table.loc[mask_interactions, "reject_fdr_bh_0_10"] = reject_bh
    interaction_terms_table["multiplicity_family"] = np.where(mask_interactions, "fba_by_moderator_interactions", "not_adjusted_main_effect")

interaction_interpretation_table = pd.DataFrame(
    [
        {
            "rule": "primary_interactions",
            "interpretation": "Only product price and delivery-days interactions are treated as pre-motivated primary heterogeneity checks.",
        },
        {
            "rule": "secondary_interactions",
            "interpretation": "Review-count and positive-review interactions are exploratory and should be discussed only if corrected p-values support them.",
        },
        {
            "rule": "star_rating_functional_form",
            "interpretation": "Star-rating heterogeneity is not modeled as a linear interaction; the ordinal nature of stelle is checked through the categorical-star OLS and overlap specifications.",
        },
        {
            "rule": "excluded_shipping_interaction",
            "interpretation": "Shipping-price interaction is excluded because FBA has zero positive-shipping support in the final sample.",
        },
    ]
)

display(interaction_terms_table)
display(interaction_margins_table)
display(interaction_interpretation_table)

,excluded_moderator,reason
0,prezzo_spedizione_repaired,FBA offers have zero shipping price in the final sample; positive-shipping FBA margins would be extrapolated outside empirical support.


,moderator,label,priority,fba_p25,fba_median,fba_p75,nonfba_p25,nonfba_median,nonfba_p75
0,prezzo,product_price,primary,42.000000,44.990000,46.000000,46.46000,50.700000,58.260000
1,g_cons_min_robust,delivery_days,primary,5.000000,6.000000,7.000000,4.00000,6.000000,8.000000
2,log1p_num_valutazioni,review_count_log,secondary,2.890372,4.317488,5.717028,3.73767,5.488938,7.513436
3,valutazioni_positive,positive_review_percent,secondary,92.000000,97.000000,100.000000,76.00000,87.000000,95.000000


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,moderator,moderator_label,priority,term_type,pvalue_holm_0_05,reject_holm_0_05,pvalue_fdr_bh_0_10,reject_fdr_bh_0_10,multiplicity_family
0,full_sample,rank_pct,interaction_single_product_price,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper,-0.027015,0.012512,-2.159138,0.034781,-0.052035,-0.001996,**,61,5107,0.960349,119,62,prezzo,product_price,primary,main_fba_at_mean_moderator,NaN,NaN,NaN,NaN,not_adjusted_main_effect
1,full_sample,rank_pct,interaction_single_product_price,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper:prezzo_c,0.004503,0.001366,3.296187,0.001637,0.001771,0.007235,***,61,5107,0.960349,119,62,prezzo,product_price,primary,fba_by_moderator_interaction,0.006546,True,0.006546,True,fba_by_moderator_interactions
2,full_sample,rank_pct,interaction_single_delivery_days,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper,-0.049879,0.015585,-3.200431,0.002179,-0.081044,-0.018715,***,61,5107,0.959656,119,62,g_cons_min_robust,delivery_days,primary,main_fba_at_mean_moderator,NaN,NaN,NaN,NaN,not_adjusted_main_effect
3,full_sample,rank_pct,interaction_single_delivery_days,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper:g_cons_min_robust_c,0.003394,0.001663,2.040550,0.045630,0.000068,0.006719,**,61,5107,0.959656,119,62,g_cons_min_robust,delivery_days,primary,fba_by_moderator_interaction,0.136891,False,0.091261,True,fba_by_moderator_interactions
4,full_sample,rank_pct,interaction_single_review_count_log,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper,-0.047355,0.015284,-3.098405,0.002941,-0.077917,-0.016793,***,61,5107,0.959621,119,62,log1p_num_valutazioni,review_count_log,secondary,main_fba_at_mean_moderator,NaN,NaN,NaN,NaN,not_adjusted_main_effect
5,full_sample,rank_pct,interaction_single_review_count_log,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper:log1p_num_valutazioni_c,0.005812,0.004296,1.352789,0.181116,-0.002779,0.014402,,61,5107,0.959621,119,62,log1p_num_valutazioni,review_count_log,secondary,fba_by_moderator_interaction,0.362233,False,0.241488,False,fba_by_moderator_interactions
6,full_sample,rank_pct,interaction_single_positive_review_percent,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper,-0.047563,0.019978,-2.380790,0.020414,-0.087511,-0.007615,**,61,5107,0.959389,119,62,valutazioni_positive,positive_review_percent,secondary,main_fba_at_mean_moderator,NaN,NaN,NaN,NaN,not_adjusted_main_effect
7,full_sample,rank_pct,interaction_single_positive_review_percent,market_fe_ols_interaction,two_way_seller_market,fba_from_shipper:valutazioni_positive_c,-0.000357,0.001058,-0.337590,0.736831,-0.002473,0.001759,,61,5107,0.959389,119,62,valutazioni_positive,positive_review_percent,secondary,fba_by_moderator_interaction,0.736831,False,0.736831,False,fba_by_moderator_interactions


,sample,outcome,specification,estimator,inference,contrast_label,estimate,se,test_statistic,pvalue,ci_low,ci_high,stars,dof_reference,resolved_weights,moderator,quantile,raw_value,centered_value
0,full_sample,rank_pct,interaction_single_product_price,market_fe_ols_interaction,two_way_seller_market,FBA association at prezzo p25,-0.053694,0.015127,-3.549544,0.000750,-0.083942,-0.023446,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:prezzo_c': -5.924525161542981}",prezzo,p25,45.000000,-5.924525
1,full_sample,rank_pct,interaction_single_product_price,market_fe_ols_interaction,two_way_seller_market,FBA association at prezzo median,-0.035456,0.012855,-2.758187,0.007660,-0.061162,-0.009751,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:prezzo_c': -1.8745251615429837}",prezzo,median,49.050000,-1.874525
2,full_sample,rank_pct,interaction_single_product_price,market_fe_ols_interaction,two_way_seller_market,FBA association at prezzo p75,-0.001458,0.014498,-0.100576,0.920217,-0.030448,0.027532,,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:prezzo_c': 5.675474838457021}",prezzo,p75,56.600000,5.675475
3,full_sample,rank_pct,interaction_single_delivery_days,market_fe_ols_interaction,two_way_seller_market,FBA association at g_cons_min_robust p25,-0.056278,0.016214,-3.470877,0.000959,-0.088700,-0.023855,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:g_cons_min_robust_c': -1.8852555316232618}",g_cons_min_robust,p25,5.000000,-1.885256
4,full_sample,rank_pct,interaction_single_delivery_days,market_fe_ols_interaction,two_way_seller_market,FBA association at g_cons_min_robust median,-0.052884,0.015806,-3.345718,0.001409,-0.084491,-0.021277,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:g_cons_min_robust_c': -0.8852555316232618}",g_cons_min_robust,median,6.000000,-0.885256
5,full_sample,rank_pct,interaction_single_delivery_days,market_fe_ols_interaction,two_way_seller_market,FBA association at g_cons_min_robust p75,-0.046096,0.015502,-2.973502,0.004212,-0.077095,-0.015097,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:g_cons_min_robust_c': 1.1147444683767382}",g_cons_min_robust,p75,8.000000,1.114744
6,full_sample,rank_pct,interaction_single_review_count_log,market_fe_ols_interaction,two_way_seller_market,FBA association at log1p_num_valutazioni p25,-0.057064,0.018483,-3.087384,0.003037,-0.094022,-0.020105,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:log1p_num_valutazioni_c': -1.6705315979494935}",log1p_num_valutazioni,p25,3.610918,-1.670532
7,full_sample,rank_pct,interaction_single_review_count_log,market_fe_ols_interaction,two_way_seller_market,FBA association at log1p_num_valutazioni median,-0.047434,0.015299,-3.100500,0.002923,-0.078026,-0.016842,***,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:log1p_num_valutazioni_c': -0.013591351530389595}",log1p_num_valutazioni,median,5.267858,-0.013591
8,full_sample,rank_pct,interaction_single_review_count_log,market_fe_ols_interaction,two_way_seller_market,FBA association at log1p_num_valutazioni p75,-0.036686,0.015287,-2.399914,0.019467,-0.067254,-0.006119,**,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:log1p_num_valutazioni_c': 1.8357559925706264}",log1p_num_valutazioni,p75,7.117206,1.835756
9,full_sample,rank_pct,interaction_single_positive_review_percent,market_fe_ols_interaction,two_way_seller_market,FBA association at valutazioni_positive p25,-0.045831,0.023453,-1.954178,0.055269,-0.092727,0.001066,*,61,"{'fba_from_shipper': 1.0, 'fba_from_shipper:valutazioni_positive_c': -4.849030742118657}",valutazioni_positive,p25,80.000000,-4.849031


,rule,interpretation
0,primary_interactions,Only product price and delivery-days interactions are treated as pre-motivated primary heterogeneity checks.
1,secondary_interactions,Review-count and positive-review interactions are exploratory and should be discussed only if corrected p-values support them.
2,star_rating_functional_form,Star-rating heterogeneity is not modeled as a linear interaction; the ordinal nature of stelle is checked through the categorical-star OLS and overlap specifications.
3,excluded_shipping_interaction,Shipping-price interaction is excluded because FBA has zero positive-shipping support in the final sample.


### 36. Claim synthesis and validation status

The evidence supports a scoped FBA ranking-premium interpretation. The static block documents a large raw within-market FBA gap that attenuates by approximately 87% once observed reputation, price, shipping, and delivery enter the projection. The residual coefficient remains precisely estimated but is no longer the large raw gap. The dynamic block documents that, among continuing sellers, FBA status is associated with marginally more favorable rank movement during seller-list turnover; the D1 baseline coefficient is 0.002467, the D2 and D3 starting-rank validations preserve sign and significance, and the D4 transition-by-rank-tier stress is nonconfirming and severely ill-conditioned.

The cell aggregates the validation status of every block, separates the validated claim from each block from the corresponding interpretation rule, and stores the consolidated synthesis in `evidence_to_claim_summary.csv`, `core_or_required_validity_summary`, and `decision_scorecard`. The synthesis feeds the validity-boundary map (`tab:ch6-validity-map`) of Chapter 6.


In [ ]:
# -----------------------------------------------------------------------------
# Section 36. Claim synthesis and validation status
# -----------------------------------------------------------------------------
# Aggregate the validation status of every empirical block and store the consolidated synthesis in the evidence-to-claim, core-validity, and decision-scorecard tables.

def _pull_starting_rank_row(label):
    fallback = {"estimate": float("nan"), "se": float("nan"), "pvalue": float("nan")}
    try:
        if "dynamic_starting_rank_adjustment_diagnostics_table" in globals():
            t = dynamic_starting_rank_adjustment_diagnostics_table
            row = t.loc[t["model_label"] == label]
            if not row.empty:
                return row.iloc[0]
    except Exception:
        pass
    return pd.Series(fallback)

_d2_row = _pull_starting_rank_row("D2")
_d3_row = _pull_starting_rank_row("D3")


def _required_table_row(table, mask, label):
    """Return the first matching row or raise an interpretable synthesis error."""
    if table is None or not hasattr(table, "loc"):
        raise ValueError(f"Missing table for required claim-synthesis row: {label}")
    out = table.loc[mask]
    if out.empty:
        available = []
        if hasattr(table, "columns") and "evidence_block" in table.columns:
            available = table["evidence_block"].dropna().astype(str).tolist()
        raise ValueError(f"Missing required claim-synthesis row: {label}. Available evidence blocks: {available}")
    return out.iloc[0]


def _optional_table_row(table, mask):
    if table is None or not hasattr(table, "loc"):
        return pd.Series(dtype=object)
    out = table.loc[mask]
    if out.empty:
        return pd.Series(dtype=object)
    return out.iloc[0]

# Static premium quantities. The original static coefficient is negative because
# lower rank_pct means better rank; the premium-sign-normalized value is positive.
_static_headline = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("static_residual_fba_premium"),
    "static_residual_fba_premium",
)
static_residual_premium = float(_static_headline["premium_sign_normalized_estimate"])
static_residual_original_coef = float(_static_headline["estimate_original_scale"])
static_residual_pvalue_for_claim = float(_static_headline["two_sided_pvalue"])

# Dynamic final premium quantities.
_premium_headline = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_preferred"),
    "dynamic_turnover_premium_preferred",
)
premium_estimate = float(_premium_headline["premium_sign_normalized_estimate"])
premium_se = float(_premium_headline["se"])
premium_pvalue = float(_premium_headline["two_sided_pvalue"])
premium_directional_pvalue = float(_premium_headline["theory_directional_pvalue"])
premium_mde = float(_premium_headline["mde_80_power_alpha_0_05"])
premium_implied_rank_shift_mean_market = float(_premium_headline["implied_rank_shift_mean_market"])

_premium_conservative = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_conservative_inference"),
    "dynamic_turnover_premium_conservative_inference",
)
premium_conservative_pvalue = float(_premium_conservative["two_sided_pvalue"]) if pd.notna(_premium_conservative["two_sided_pvalue"]) else np.nan
premium_conservative_method = str(_premium_conservative["model_role"])
premium_conservative_estimate = float(_premium_conservative["premium_sign_normalized_estimate"]) if pd.notna(_premium_conservative["premium_sign_normalized_estimate"]) else np.nan

_direction_row = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_directional_balance_control"),
    "dynamic_directional_balance_control",
)
direction_balance_estimate = float(_direction_row["premium_sign_normalized_estimate"])
direction_balance_pvalue = float(_direction_row["two_sided_pvalue"])

_ranktier_row = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_transition_rank_tier_fe_stress_test"),
    "dynamic_turnover_premium_transition_rank_tier_fe_stress_test",
)
ranktier_premium_estimate = float(_ranktier_row["premium_sign_normalized_estimate"]) if pd.notna(_ranktier_row["premium_sign_normalized_estimate"]) else np.nan
ranktier_premium_se = float(_ranktier_row["se"]) if "se" in _ranktier_row.index and pd.notna(_ranktier_row["se"]) else np.nan
ranktier_premium_pvalue = float(_ranktier_row["two_sided_pvalue"]) if pd.notna(_ranktier_row["two_sided_pvalue"]) else np.nan
ranktier_stress_nonconfirming = bool(pd.notna(ranktier_premium_estimate) and (ranktier_premium_estimate <= 0 or (pd.notna(ranktier_premium_pvalue) and ranktier_premium_pvalue > 0.10)))
_ranktier_diagnostic_row = pd.Series(dtype=object)
if "dynamic_rank_tier_stress_diagnostic_table" in globals() and isinstance(dynamic_rank_tier_stress_diagnostic_table, pd.DataFrame) and len(dynamic_rank_tier_stress_diagnostic_table):
    _ranktier_diagnostic_row = _optional_table_row(
        dynamic_rank_tier_stress_diagnostic_table,
        dynamic_rank_tier_stress_diagnostic_table["object"].eq("transition_by_rank_tier_fe_stress_test"),
    )
ranktier_stress_condition_warning = bool(_ranktier_diagnostic_row.get("finite_design_warning", False)) if len(_ranktier_diagnostic_row) else False

_quartile_claim_row = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_starting_rank_quartile_adjusted"),
    "dynamic_turnover_premium_starting_rank_quartile_adjusted",
)
quartile_premium_estimate = float(_quartile_claim_row["premium_sign_normalized_estimate"]) if pd.notna(_quartile_claim_row["premium_sign_normalized_estimate"]) else np.nan
quartile_premium_pvalue = float(_quartile_claim_row["two_sided_pvalue"]) if pd.notna(_quartile_claim_row["two_sided_pvalue"]) else np.nan

_rankpoly_claim_row = _required_table_row(
    dynamic_fba_premium_summary_table,
    dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_starting_rank_quadratic_adjusted"),
    "dynamic_turnover_premium_starting_rank_quadratic_adjusted",
)
rankpoly_premium_estimate = float(_rankpoly_claim_row["premium_sign_normalized_estimate"]) if pd.notna(_rankpoly_claim_row["premium_sign_normalized_estimate"]) else np.nan
rankpoly_premium_pvalue = float(_rankpoly_claim_row["two_sided_pvalue"]) if pd.notna(_rankpoly_claim_row["two_sided_pvalue"]) else np.nan

starting_rank_validation_positive = bool(
    pd.notna(quartile_premium_estimate) and quartile_premium_estimate > 0
    and pd.notna(rankpoly_premium_estimate) and rankpoly_premium_estimate > 0
)
starting_rank_validation_statistical = bool(
    pd.notna(quartile_premium_pvalue) and quartile_premium_pvalue < 0.10
    and pd.notna(rankpoly_premium_pvalue) and rankpoly_premium_pvalue < 0.10
)

# Above/below reference quantities retained for compatibility and decomposition.
_headline_dynamic_row = _required_table_row(
    dynamic_vacancy_models_table,
    dynamic_vacancy_models_table["specification"].eq(PREFERRED_DYNAMIC_SPEC),
    f"dynamic_vacancy_models_table::{PREFERRED_DYNAMIC_SPEC}",
)
lambda_above = float(_headline_dynamic_row["estimate"])
p_above = float(_headline_dynamic_row["pvalue"])
se_above = float(_headline_dynamic_row["se"])
lambda_below = float(placebo_below_row.get("estimate", np.nan))
p_below = float(placebo_below_row.get("pvalue", np.nan))
ratio_abs = abs(lambda_below) / max(abs(lambda_above), 1e-12) if pd.notna(lambda_below) else np.nan
ratio_positive_part = max(lambda_below, 0.0) / max(abs(lambda_above), 1e-12) if pd.notna(lambda_below) else np.nan
# Audit-compatible extraction from the directional decomposition decision table.
# The claim table accepts the compatibility column name `mirrored_positive_below_failure`;
# the claim table treats it as a challenge only to the
# narrow local-vacancy mechanism, not to the primary total-turnover claim.
def _safe_first_bool_from_table(table, candidate_columns, default=False):
    if table is None or not hasattr(table, "columns") or len(table) == 0:
        return bool(default)
    for candidate_column in candidate_columns:
        if candidate_column in table.columns:
            value = table[candidate_column].iloc[0]
            if pd.isna(value):
                return bool(default)
            return bool(value)
    return bool(default)


def _safe_first_float_from_table(table, candidate_columns, default=np.nan):
    if table is None or not hasattr(table, "columns") or len(table) == 0:
        return float(default)
    for candidate_column in candidate_columns:
        if candidate_column in table.columns:
            value = table[candidate_column].iloc[0]
            return float(value) if pd.notna(value) else float(default)
    return float(default)


_placebo_decision_table_for_claim = placebo_decision_table if "placebo_decision_table" in globals() else None
mirrored_positive_below_failure = _safe_first_bool_from_table(
    _placebo_decision_table_for_claim,
    [
        "mirrored_positive_below_challenges_only_local_vacancy_claim",
        "mirrored_positive_below_failure",
    ],
    default=False,
)
placebo_interpretable_flag = _safe_first_bool_from_table(
    _placebo_decision_table_for_claim,
    ["placebo_interpretable_flag"],
    default=pd.notna(lambda_below),
)
joint_placebo_ill_conditioned = _safe_first_bool_from_table(
    _placebo_decision_table_for_claim,
    ["joint_sensitivity_ill_conditioned"],
    default=False,
)
churn_above_below_corr = float(churn_direction_correlation_table["observation_level_correlation_dropouts_above_below"].iloc[0]) if "churn_direction_correlation_table" in globals() and len(churn_direction_correlation_table) else np.nan

# Auxiliary quantities retained for context.
S_pre = int(smoothness_table["materially_abnormal"].sum()) if "smoothness_table" in globals() and len(smoothness_table) and "materially_abnormal" in smoothness_table else 0
return_share = float(return_pattern_summary_table["temporary_or_return_share"].iloc[0]) if "return_pattern_summary_table" in globals() and len(return_pattern_summary_table) else np.nan
dropout_overwhelming = bool(dropout_process_fit_summary_table["overwhelming_predictability_flag"].iloc[0]) if "dropout_process_fit_summary_table" in globals() and len(dropout_process_fit_summary_table) else False

_flex_row = continuous_heterogeneity_table.loc[continuous_heterogeneity_table["term"].eq("fba_x_dropouts_above")] if "continuous_heterogeneity_table" in globals() else pd.DataFrame()
flex_estimate = float(_flex_row["estimate"].iloc[0]) if len(_flex_row) else np.nan
baseline_flex_percent_change = abs((flex_estimate - lambda_above) / lambda_above) if pd.notna(flex_estimate) and lambda_above != 0 else np.nan
baseline_flex_sign_flip = bool(pd.notna(flex_estimate) and lambda_above != 0 and np.sign(flex_estimate) != np.sign(lambda_above))

_top_terms = economic_salience_table.loc[economic_salience_table["outcome"].isin(["entered_top_10", "entered_top_quartile"])] if "economic_salience_table" in globals() else pd.DataFrame()
top_directionally_consistent = bool(len(_top_terms) and _top_terms["estimate_lambda"].dropna().gt(0).any())
_top10_salience_row = economic_salience_table.loc[economic_salience_table["outcome"].eq("entered_top_10")] if "economic_salience_table" in globals() and "outcome" in economic_salience_table else pd.DataFrame()
top10_salience_estimate = float(_top10_salience_row["estimate_lambda"].iloc[0]) if len(_top10_salience_row) and pd.notna(_top10_salience_row["estimate_lambda"].iloc[0]) else np.nan
top10_salience_pvalue = float(_top10_salience_row["pvalue"].iloc[0]) if len(_top10_salience_row) and pd.notna(_top10_salience_row["pvalue"].iloc[0]) else np.nan
top10_salience_preferred_pass = bool(pd.notna(top10_salience_estimate) and top10_salience_estimate > 0 and pd.notna(top10_salience_pvalue) and top10_salience_pvalue < 0.05)
top10_salience_pass = np.nan  # auxiliary only; not a core pass/fail criterion because conservative finite-cluster inference is weaker

_MDE_dynamic_row = dynamic_support_mde_table.loc[dynamic_support_mde_table["model_or_subsample"].eq("primary D1 total-turnover dynamic model")] if "dynamic_support_mde_table" in globals() else pd.DataFrame()
MDE_dynamic = float(_MDE_dynamic_row["mde_80_power_alpha_0_05"].iloc[0]) if len(_MDE_dynamic_row) else np.nan
MDE_dynamic_positions = float(_MDE_dynamic_row["mde_as_rank_positions_if_applicable"].iloc[0]) if len(_MDE_dynamic_row) else np.nan
_MDE_top_rows = dynamic_support_mde_table.loc[dynamic_support_mde_table["model_or_subsample"].isin(["top-10 outcome", "top-quartile outcome"])] if "dynamic_support_mde_table" in globals() else pd.DataFrame()
MDE_top = float(_MDE_top_rows["mde_80_power_alpha_0_05"].max()) if len(_MDE_top_rows) else np.nan
any_power_warning = bool(dynamic_support_mde_table["power_warning"].any()) if "dynamic_support_mde_table" in globals() and "power_warning" in dynamic_support_mde_table else False
power_warning_summary = "; ".join(dynamic_support_mde_table.loc[dynamic_support_mde_table["power_warning"], "power_warning_reason"].dropna().astype(str).unique()) if "dynamic_support_mde_table" in globals() and "power_warning_reason" in dynamic_support_mde_table else ""

# Claim logic. The final model does not require a clean local-vacancy decomposition pass,
# because local vacancy is no longer the target claim.
static_premium_pass = bool(static_residual_premium > 0 and static_residual_pvalue_for_claim < 0.05)
premium_preferred_pass = bool(premium_estimate > 0 and premium_pvalue < 0.05)
premium_conservative_pass = bool(pd.notna(premium_conservative_pvalue) and premium_conservative_estimate > 0 and premium_conservative_pvalue < 0.10)
premium_model_identified = bool(np.isfinite(premium_estimate) and np.isfinite(premium_se) and premium_se > 0)
directional_local_not_required = True
ranktier_stress_reported = bool(pd.notna(ranktier_premium_estimate) or dynamic_fba_premium_summary_table["evidence_block"].eq("dynamic_turnover_premium_transition_rank_tier_fe_stress_test").any())

if premium_model_identified and premium_preferred_pass and premium_conservative_pass and ranktier_stress_nonconfirming:
    claim_level = "credible but rank-tier-sensitive FBA-premium evidence"
    claim_text = (
        "The notebook's central evidence is a static residual FBA ranking premium plus a primary dynamic FBA-by-turnover premium. "
        "The primary dynamic model is positive and significant under clustered, CR2, and wild-bootstrap inference. "
        "However, the transition-by-starting-rank-tier stress test does not confirm the same dynamic premium, so the dynamic result must be described as sensitive to rank-position structure. "
        "The claim remains observational and does not identify a pure within-seller FBA switching effect."
    )
elif premium_model_identified and premium_preferred_pass and premium_conservative_pass:
    claim_level = "central credible FBA-premium evidence"
    claim_text = (
        "The notebook's central evidence is a static residual FBA ranking premium plus a dynamic FBA-by-turnover premium. "
        "The primary dynamic model shows that FBA sellers gain more rank position during seller-turnover transitions, "
        "after seller fixed effects, transition fixed effects, lagged rank, and lagged offer controls. "
        "The claim remains observational and does not identify a pure within-seller FBA switching effect."
    )
elif premium_model_identified and premium_preferred_pass:
    claim_level = "credible but finite-cluster-sensitive FBA-premium evidence"
    claim_text = (
        "The primary dynamic FBA-by-turnover premium is positive and statistically significant, "
        "but conservative finite-cluster checks are weaker. The result is usable as premium evidence with explicit inference caution."
    )
elif premium_model_identified and premium_estimate > 0 and premium_pvalue < 0.10:
    claim_level = "suggestive FBA-premium evidence"
    claim_text = (
        "The dynamic FBA-by-turnover premium has the expected positive sign and is statistically suggestive, "
        "but it should not be overstated as a central significant result."
    )
else:
    claim_level = "descriptive static FBA-premium evidence only"
    claim_text = (
        "The dynamic turnover-premium model does not provide enough independent support to anchor the claim. "
        "The thesis should rely primarily on static descriptive premium evidence."
    )

premium_claim_level = claim_level
premium_claim_text = claim_text

# A transparent rule table.
dynamic_claim_logic_table = pd.DataFrame([
    {"criterion": "static residual FBA premium positive and p < 0.05", "result": static_premium_pass, "role": "core descriptive"},
    {"criterion": "dynamic FBA turnover premium positive and p < 0.05", "result": premium_preferred_pass, "role": "core dynamic"},
    {"criterion": "conservative premium inference p < 0.10", "result": premium_conservative_pass, "role": "core inference check"},
    {"criterion": "dynamic premium model identified with finite SE", "result": premium_model_identified, "role": "core"},
    {"criterion": "directional local-vacancy term not required for final claim", "result": directional_local_not_required, "role": "interpretation"},
    {"criterion": "rank-tier FE stress test reported", "result": ranktier_stress_reported, "role": "body-level caution"},
    {"criterion": "pre-disappearance diagnostics reported", "result": bool("smoothness_table" in globals() and len(smoothness_table)), "role": "contextual"},
    {"criterion": "support/MDE audit reported", "result": bool("dynamic_support_mde_table" in globals() and len(dynamic_support_mde_table)), "role": "contextual"},
])

# Compatibility table for exports and downstream text.
dynamic_decision_quantities_table = pd.DataFrame([{
    "static_residual_premium_rankpct": static_residual_premium,
    "static_residual_original_coef_rankpct": static_residual_original_coef,
    "static_residual_pvalue": static_residual_pvalue_for_claim,
    "premium_lambda_turnover": premium_estimate,
    "premium_se_preferred": premium_se,
    "premium_pvalue_preferred_two_sided": premium_pvalue,
    "premium_pvalue_theory_directional": premium_directional_pvalue,
    "premium_conservative_method": premium_conservative_method,
    "premium_conservative_estimate": premium_conservative_estimate,
    "premium_conservative_pvalue": premium_conservative_pvalue,
    "premium_mde_80_power_alpha_0_05": premium_mde,
    "premium_implied_rank_shift_mean_market": premium_implied_rank_shift_mean_market,
    "direction_balance_estimate": direction_balance_estimate,
    "direction_balance_pvalue": direction_balance_pvalue,
    "ranktier_premium_estimate_contextual": ranktier_premium_estimate,
    "ranktier_premium_pvalue_contextual": ranktier_premium_pvalue,
    "lambda_above_reference": lambda_above,
    "p_above_reference": p_above,
    "lambda_below_reference": lambda_below,
    "p_below_reference": p_below,
    "below_above_abs_ratio_reported": ratio_abs,
    "below_above_positive_part_ratio": ratio_positive_part,
    "mirrored_positive_below_failure_local_vacancy_only": mirrored_positive_below_failure,
    "placebo_interpretable_flag": placebo_interpretable_flag,
    "joint_placebo_ill_conditioned": joint_placebo_ill_conditioned,
    "dropouts_above_below_correlation": churn_above_below_corr,
    "pre_smoothness_abnormal_variables_contextual": S_pre,
    "MDE_dynamic_contextual_primary_total_turnover_model": MDE_dynamic,
    "MDE_dynamic_rank_positions_contextual_primary_total_turnover_model": MDE_dynamic_positions,
    "MDE_top_contextual": MDE_top,
    "temporary_or_return_share_contextual": return_share,
    "dropout_process_overwhelmingly_predictable_contextual": dropout_overwhelming,
    "baseline_flex_percent_change_contextual_directional_reference_model": baseline_flex_percent_change,
    "baseline_flex_sign_flip_contextual_directional_reference_model": baseline_flex_sign_flip,
    "top_of_list_directionally_consistent_contextual": top_directionally_consistent,
    "top10_salience_estimate_auxiliary": top10_salience_estimate,
    "top10_salience_pvalue_auxiliary": top10_salience_pvalue,
    "top10_salience_pass_auxiliary": top10_salience_pass,
    "any_power_warning_contextual": any_power_warning,
    "power_warning_summary_contextual": power_warning_summary,
    "dynamic_claim_strength": claim_level,
    "required_dynamic_wording": claim_text,
}])

# Audit-compatible alias: now a premium-claim logic table.
decision_scorecard = dynamic_claim_logic_table.copy()

# Table 7: compact premium and validation summary.
dynamic_mechanism_validation_summary_table = pd.DataFrame([
    {
        "test_family": "Static residual FBA premium",
        "key_term": "fba_from_shipper",
        "estimate": static_residual_premium,
        "se_or_mde": np.nan,
        "pvalue": static_residual_pvalue_for_claim,
        "support_metric": "sign-normalized: positive means FBA ranked better",
        "role_in_claim": "core descriptive",
        "pass_flag": static_premium_pass,
        "interpretation": "The controlled static model shows a residual FBA ranking premium. This is descriptive because FBA is not randomly assigned.",
    },
    {
        "test_family": "Dynamic FBA turnover premium",
        "key_term": "fba x total seller turnover",
        "estimate": premium_estimate,
        "se_or_mde": premium_se,
        "pvalue": premium_pvalue,
        "support_metric": f"primary two-way clustered model; MDE={premium_mde:.4f}" if pd.notna(premium_mde) else "primary two-way clustered model",
        "role_in_claim": "core dynamic",
        "pass_flag": premium_preferred_pass,
        "interpretation": "Main final dynamic estimate: FBA sellers move upward more during seller-turnover transitions, regardless of whether disappearing sellers were above or below.",
    },
    {
        "test_family": "Conservative dynamic premium inference",
        "key_term": "fba x total seller turnover",
        "estimate": premium_conservative_estimate,
        "se_or_mde": np.nan,
        "pvalue": premium_conservative_pvalue,
        "support_metric": premium_conservative_method,
        "role_in_claim": "core inference check",
        "pass_flag": premium_conservative_pass,
        "interpretation": "Uses the least favorable sign-consistent CR2/bootstrap p-value available for the final premium coefficient.",
    },
    {
        "test_family": "Directional churn decomposition",
        "key_term": "fba x dropout direction balance",
        "estimate": direction_balance_estimate,
        "se_or_mde": np.nan,
        "pvalue": direction_balance_pvalue,
        "support_metric": "controls whether disappearances are concentrated above rather than below",
        "role_in_claim": "mechanism decomposition",
        "pass_flag": np.nan,
        "interpretation": "A non-central directional term means the evidence is better read as a general turnover premium than as a local above-turnover mechanism.",
    },
    {
        "test_family": "Above and below reference estimates",
        "key_term": "fba x dropouts_above; fba x dropouts_below",
        "estimate": lambda_above,
        "se_or_mde": lambda_below,
        "pvalue": p_above,
        "support_metric": f"below p={p_below:.3f}" if pd.notna(p_below) else "below estimate reported",
        "role_in_claim": "mechanism decomposition",
        "pass_flag": np.nan,
        "interpretation": "The mirrored below response is not a failure of the final turnover-premium claim, because the notebook no longer claims a narrow local-vacancy mechanism.",
    },
    {
        "test_family": "Starting-rank quartile-adjusted dynamic premium (D2)",
        "key_term": "fba x total seller turnover",
        "estimate": float(_d2_row.get("estimate", float("nan"))) if pd.notna(_d2_row.get("estimate", float("nan"))) else float("nan"),
        "se_or_mde": float(_d2_row.get("se", float("nan"))) if pd.notna(_d2_row.get("se", float("nan"))) else float("nan"),
        "pvalue": float(_d2_row.get("pvalue", float("nan"))) if pd.notna(_d2_row.get("pvalue", float("nan"))) else float("nan"),
        "support_metric": "additive lag-rank quartile dummies; q1_top_25pct is the omitted category",
        "role_in_claim": "main rank-position validation",
        "pass_flag": np.nan,
        "interpretation": "Dynamic premium with additive starting-rank quartile controls. Adjusts for starting-rank composition without saturating transition-cell support.",
    },
    {
        "test_family": "Starting-rank quadratic-adjusted dynamic premium (D3)",
        "key_term": "fba x total seller turnover",
        "estimate": float(_d3_row.get("estimate", float("nan"))) if pd.notna(_d3_row.get("estimate", float("nan"))) else float("nan"),
        "se_or_mde": float(_d3_row.get("se", float("nan"))) if pd.notna(_d3_row.get("se", float("nan"))) else float("nan"),
        "pvalue": float(_d3_row.get("pvalue", float("nan"))) if pd.notna(_d3_row.get("pvalue", float("nan"))) else float("nan"),
        "support_metric": "smooth lag_rank_pct and lag_rank_pct squared",
        "role_in_claim": "main rank-position validation",
        "pass_flag": np.nan,
        "interpretation": "Dynamic premium with a smooth quadratic starting-rank adjustment. Avoids transition-cell saturation while letting the starting-rank effect be nonlinear.",
    },
    {
        "test_family": "Rank-tier FE stress test (D4)",
        "key_term": "fba x total seller turnover",
        "estimate": ranktier_premium_estimate,
        "se_or_mde": ranktier_premium_se,
        "pvalue": ranktier_premium_pvalue,
        "support_metric": f"transition_id x lag_rank_tier FE; finite_design_warning={ranktier_stress_condition_warning}",
        "role_in_claim": "severe support-sensitive stress test",
        "pass_flag": np.nan,
        "interpretation": "A stricter comparison within transition and starting-rank tier. Reported as a severe support-sensitive stress test, not as the primary rank-position-adjusted estimator, because cell support and the residualized condition number make it numerically thin.",
    },
    {
        "test_family": "Pre-disappearance descriptives",
        "key_term": "core variables before disappearance",
        "estimate": S_pre,
        "se_or_mde": np.nan,
        "pvalue": np.nan,
        "support_metric": "reported descriptively; not a mechanical threshold test",
        "role_in_claim": "contextual validation",
        "pass_flag": np.nan,
        "interpretation": "Checks whether disappearing sellers visibly deteriorate before disappearance. Used as descriptive support, not over-control.",
    },
    {
        "test_family": "Dynamic support and MDE",
        "key_term": "exposed observations and detectable effect",
        "estimate": premium_mde,
        "se_or_mde": premium_implied_rank_shift_mean_market,
        "pvalue": np.nan,
        "support_metric": power_warning_summary if power_warning_summary else "support and power diagnostics reported",
        "role_in_claim": "interpretive support",
        "pass_flag": np.nan,
        "interpretation": "Documents empirical support and avoids interpreting underpowered auxiliary nulls as evidence of no premium.",
    },
])



# Add main-analysis hardening checks to the targeted validation summary. These rows make
# the critique-visible tests part of the main notebook, not an appendix surprise.
_extra_validation_rows = []

if "dynamic_turnover_placebo_table" in globals() and len(dynamic_turnover_placebo_table):
    _fp = dynamic_turnover_placebo_table.loc[dynamic_turnover_placebo_table["model_role"].eq("future_turnover_placebo")]
    _pp = dynamic_turnover_placebo_table.loc[dynamic_turnover_placebo_table["model_role"].eq("pretrend_turnover_placebo")]
    _extra_validation_rows.append({
        "test_family": "Future-turnover and pre-trend placebos",
        "key_term": "FBA x future/current total turnover",
        "estimate": float(_fp["estimate"].iloc[0]) if len(_fp) and pd.notna(_fp["estimate"].iloc[0]) else np.nan,
        "se_or_mde": float(_pp["estimate"].iloc[0]) if len(_pp) and pd.notna(_pp["estimate"].iloc[0]) else np.nan,
        "pvalue": float(_fp["pvalue"].iloc[0]) if len(_fp) and pd.notna(_fp["pvalue"].iloc[0]) else np.nan,
        "support_metric": "future-turnover p and pre-trend estimate reported; see dynamic_turnover_placebos_rankpct.csv",
        "role_in_claim": "dynamic placebo validation",
        "pass_flag": np.nan,
        "interpretation": "Future turnover should not predict current rank improvement, and current turnover should not predict previous rank movement. These are placebo checks for the total-turnover estimand.",
    })
if "dynamic_permutation_placebo_turnover_table" in globals() and len(dynamic_permutation_placebo_turnover_table):
    _perm = dynamic_permutation_placebo_turnover_table.iloc[0]
    _extra_validation_rows.append({
        "test_family": "Permutation within transition x starting-rank tier",
        "key_term": "FBA status randomized locally within transition x lag-rank-tier",
        "estimate": float(_perm.get("observed_beta", np.nan)) if pd.notna(_perm.get("observed_beta", np.nan)) else np.nan,
        "se_or_mde": float(_perm.get("permutation_monte_carlo_se", np.nan)) if pd.notna(_perm.get("permutation_monte_carlo_se", np.nan)) else np.nan,
        "pvalue": float(_perm.get("permutation_p_value_two_sided", np.nan)) if pd.notna(_perm.get("permutation_p_value_two_sided", np.nan)) else np.nan,
        "support_metric": f"valid permutations={int(_perm.get('valid_permutations', 0))}",
        "role_in_claim": "rank-composition placebo",
        "pass_flag": np.nan,
        "interpretation": "Tests whether the observed primary dynamic coefficient is unusually large relative to local FBA reassignments within transition-by-starting-rank-tier cells.",
    })
if "dynamic_stockout_consistent_turnover_table" in globals() and len(dynamic_stockout_consistent_turnover_table):
    _stock_broad = dynamic_stockout_consistent_turnover_table.loc[
        dynamic_stockout_consistent_turnover_table["stockout_definition"].eq("broad_later_return_nonprice")
        & dynamic_stockout_consistent_turnover_table["model_role"].eq("restricted_stockout_premium_parsimonious")
    ]
    _extra_validation_rows.append({
        "test_family": "Stockout-consistent restricted turnover premium",
        "key_term": "fba_x_dropouts_total_focal",
        "estimate": float(_stock_broad["estimate"].iloc[0]) if len(_stock_broad) and pd.notna(_stock_broad["estimate"].iloc[0]) else np.nan,
        "se_or_mde": float(_stock_broad["se"].iloc[0]) if len(_stock_broad) and pd.notna(_stock_broad["se"].iloc[0]) else np.nan,
        "pvalue": float(_stock_broad["pvalue"].iloc[0]) if len(_stock_broad) and pd.notna(_stock_broad["pvalue"].iloc[0]) else np.nan,
        "support_metric": "broad later-return non-price disappearance definition; strict definition also exported",
        "role_in_claim": "stockout-consistency hardening",
        "pass_flag": np.nan,
        "interpretation": "Re-estimates the primary dynamic premium using only disappearance events more consistent with temporary offer unavailability. This strengthens but does not prove a stockout interpretation.",
    })
# The official top-10 prominence row is integrated in Section 38 from the dedicated top-10 module.
if "dynamic_clean_turnover_restriction_table" in globals() and len(dynamic_clean_turnover_restriction_table):
    _clean_sig = dynamic_clean_turnover_restriction_table.loc[
        dynamic_clean_turnover_restriction_table["pvalue"].notna()
        & dynamic_clean_turnover_restriction_table["estimate"].notna()
    ].copy()
    _extra_validation_rows.append({
        "test_family": "Clean one-sided turnover restrictions",
        "key_term": "fba_x_dropouts_total_focal",
        "estimate": float(_clean_sig["estimate"].iloc[0]) if len(_clean_sig) else np.nan,
        "se_or_mde": float(_clean_sig["se"].iloc[0]) if len(_clean_sig) else np.nan,
        "pvalue": float(_clean_sig["pvalue"].iloc[0]) if len(_clean_sig) else np.nan,
        "support_metric": "reported as main-analysis hardening table",
        "role_in_claim": "diagnostic hardening",
        "pass_flag": np.nan,
        "interpretation": "Tests whether the turnover premium is visible in cleaner one-sided-turnover samples. Nulls are interpreted as support limits, not as proof of no premium.",
    })
if "dynamic_distance_banded_turnover_table" in globals() and len(dynamic_distance_banded_turnover_table):
    _band_sig = dynamic_distance_banded_turnover_table.loc[
        dynamic_distance_banded_turnover_table["role"].eq("premium_term")
        & dynamic_distance_banded_turnover_table["pvalue"].notna()
    ].copy()
    _band_parts = []
    if len(_band_sig):
        _band_sig = _band_sig.sort_values("band_k")
        for _, _r in _band_sig.iterrows():
            _band_parts.append(
                f"within_{int(_r['band_k'])}: beta={float(_r['estimate']):.6f}, p={float(_r['pvalue']):.4g}"
            )
    _extra_validation_rows.append({
        "test_family": "Distance-banded turnover exposure",
        "key_term": "fba_x_dropouts_total_within_K",
        "estimate": np.nan,
        "se_or_mde": np.nan,
        "pvalue": np.nan,
        "support_metric": "; ".join(_band_parts) if _band_parts else "K in {1, 3, 5, 10}; no premium-term rows available",
        "role_in_claim": "local-mechanism diagnostic family",
        "pass_flag": np.nan,
        "interpretation": "Family of diagnostics for nearby rank-vacancy exposure. The pattern does not support a clean local vacancy-filling mechanism and does not replace the broad total-turnover estimand.",
    })
if "dynamic_partial_identification_bounds_table" in globals() and len(dynamic_partial_identification_bounds_table):
    _bound_candidates = dynamic_partial_identification_bounds_table.loc[
        dynamic_partial_identification_bounds_table["finite_design_warning"].astype(bool).eq(False)
        & dynamic_partial_identification_bounds_table["estimate"].notna()
    ].copy()
    _bound = _bound_candidates.iloc[0] if len(_bound_candidates) else dynamic_partial_identification_bounds_table.iloc[0]
    _extra_validation_rows.append({
        "test_family": "Above-minus-below partial-identification-style bound",
        "key_term": "lambda_above_minus_lambda_below",
        "estimate": float(_bound.get("estimate", np.nan)) if pd.notna(_bound.get("estimate", np.nan)) else np.nan,
        "se_or_mde": float(_bound.get("se", np.nan)) if pd.notna(_bound.get("se", np.nan)) else np.nan,
        "pvalue": float(_bound.get("pvalue", np.nan)) if pd.notna(_bound.get("pvalue", np.nan)) else np.nan,
        "support_metric": str(_bound.get("bound_type", "reported")),
        "role_in_claim": "local-mechanism bound",
        "pass_flag": np.nan,
        "interpretation": "Bounds the local directional component instead of merely renaming the above/below decomposition evidence; flagged joint rows are not used for the substantive bound.",
    })
# D4 is already included once in the core validation summary; do not duplicate it in extra rows.
if _extra_validation_rows:
    dynamic_mechanism_validation_summary_table = pd.concat(
        [dynamic_mechanism_validation_summary_table, pd.DataFrame(_extra_validation_rows)],
        ignore_index=True,
    )
    # Keep one row per test-family/key-term pair. When a compact row and a more
    # detailed diagnostic row both exist, retain the later diagnostic row.
    dynamic_mechanism_validation_summary_table = dynamic_mechanism_validation_summary_table.drop_duplicates(
        subset=["test_family"],
        keep="last",
    ).reset_index(drop=True)

print("Dynamic premium conservative inference candidates")
display(dynamic_fba_premium_inference_table)
print("Dynamic premium claim logic")
display(dynamic_claim_logic_table)
print("Dynamic premium decision quantities")
display(dynamic_decision_quantities_table)
print("Table 7. FBA premium and targeted validation summary")
display(dynamic_mechanism_validation_summary_table)
print(f"Dynamic claim strength: {claim_level}")
print(claim_text)

Dynamic premium conservative inference candidates


,method,term,estimate,se,test_statistic,dof_reference,pvalue,theory_directional_pvalue_positive,pvalue_type,note,clusters,cr2_adjustment,requested_replications,valid_replications,invalid_replications,valid_replication_share,invalid_replication_share,bootstrap_pvalue_monte_carlo_se,valid_replication_warning,seed,studentized,recomputed_se_each_replication
0,preferred_two_way_seller_transition_clustered,fba_x_dropouts_total_focal,0.002467,0.000765,3.226005,60.000000,0.002034,0.001017,two_sided_t,Primary final dynamic premium inference,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cr2_seller,fba_x_dropouts_total_focal,0.002467,0.000675,3.654990,18.071781,0.001802,0.000901,two_sided_t,Bell-McCaffrey-style one-way CR2 correction on the residualized final premium design,114.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cr2_transition,fba_x_dropouts_total_focal,0.002467,0.000803,3.071794,5.442579,0.024824,0.012412,two_sided_t,Bell-McCaffrey-style one-way CR2 correction on the residualized final premium design,61.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,two_way_wild_cluster_bootstrap,fba_x_dropouts_total_focal,0.002467,0.000765,3.226005,NaN,0.008832,NaN,two_sided_studentized_wild_cluster_bootstrap,Restricted residual wild cluster bootstrap for the final FBA turnover-premium term,NaN,NaN,4999.0,4981.0,18.0,0.996399,0.003601,0.001326,False,20260521.0,True,True


Dynamic premium claim logic


,criterion,result,role
0,static residual FBA premium positive and p < 0.05,True,core descriptive
1,dynamic FBA turnover premium positive and p < 0.05,True,core dynamic
2,conservative premium inference p < 0.10,True,core inference check
3,dynamic premium model identified with finite SE,True,core
4,directional local-vacancy term not required for final claim,True,interpretation
5,rank-tier FE stress test reported,True,body-level caution
6,pre-disappearance diagnostics reported,True,contextual
7,support/MDE audit reported,True,contextual


Dynamic premium decision quantities


,static_residual_premium_rankpct,static_residual_original_coef_rankpct,static_residual_pvalue,premium_lambda_turnover,premium_se_preferred,premium_pvalue_preferred_two_sided,premium_pvalue_theory_directional,premium_conservative_method,premium_conservative_estimate,premium_conservative_pvalue,premium_mde_80_power_alpha_0_05,premium_implied_rank_shift_mean_market,direction_balance_estimate,direction_balance_pvalue,ranktier_premium_estimate_contextual,ranktier_premium_pvalue_contextual,lambda_above_reference,p_above_reference,lambda_below_reference,p_below_reference,below_above_abs_ratio_reported,below_above_positive_part_ratio,mirrored_positive_below_failure_local_vacancy_only,placebo_interpretable_flag,joint_placebo_ill_conditioned,dropouts_above_below_correlation,pre_smoothness_abnormal_variables_contextual,MDE_dynamic_contextual_primary_total_turnover_model,MDE_dynamic_rank_positions_contextual_primary_total_turnover_model,MDE_top_contextual,temporary_or_return_share_contextual,dropout_process_overwhelmingly_predictable_contextual,baseline_flex_percent_change_contextual_directional_reference_model,baseline_flex_sign_flip_contextual_directional_reference_model,top_of_list_directionally_consistent_contextual,top10_salience_estimate_auxiliary,top10_salience_pvalue_auxiliary,top10_salience_pass_auxiliary,any_power_warning_contextual,power_warning_summary_contextual,dynamic_claim_strength,required_dynamic_wording
0,0.051038,-0.051038,0.002207,0.002467,0.000765,0.002034,0.001017,cr2_transition,0.002467,0.024824,0.002141,0.200763,-0.000117,0.953169,-0.000075,0.915343,0.003606,0.035626,0.003385,0.00377,0.93861,0.93861,True,True,True,-0.021189,1,0.002141,0.176393,0.012024,0.5,False,1.479819,True,True,0.011867,0.007586,NaN,True,"exposed FBA observations < 50; binary baseline event rate outside [0.05, 0.95]; nonpositive estimated SE; exposed FBA observations < 50; MDE in rank positions > 1.0",credible but rank-tier-sensitive FBA-premium evidence,The notebook's central evidence is a static residual FBA ranking premium plus a primary dynamic FBA-by-turnover premium. The primary dynamic model is positive and significant u...


Table 7. FBA premium and targeted validation summary


,test_family,key_term,estimate,se_or_mde,pvalue,support_metric,role_in_claim,pass_flag,interpretation
0,Static residual FBA premium,fba_from_shipper,0.051038,NaN,0.002207,sign-normalized: positive means FBA ranked better,core descriptive,True,The controlled static model shows a residual FBA ranking premium. This is descriptive because FBA is not randomly assigned.
1,Dynamic FBA turnover premium,fba x total seller turnover,0.002467,0.000765,0.002034,primary two-way clustered model; MDE=0.0021,core dynamic,True,"Main final dynamic estimate: FBA sellers move upward more during seller-turnover transitions, regardless of whether disappearing sellers were above or below."
2,Conservative dynamic premium inference,fba x total seller turnover,0.002467,NaN,0.024824,cr2_transition,core inference check,True,Uses the least favorable sign-consistent CR2/bootstrap p-value available for the final premium coefficient.
3,Directional churn decomposition,fba x dropout direction balance,-0.000117,NaN,0.953169,controls whether disappearances are concentrated above rather than below,mechanism decomposition,NaN,A non-central directional term means the evidence is better read as a general turnover premium than as a local above-turnover mechanism.
4,Above and below reference estimates,fba x dropouts_above; fba x dropouts_below,0.003606,0.003385,0.035626,below p=0.004,mechanism decomposition,NaN,"The mirrored below response is not a failure of the final turnover-premium claim, because the notebook no longer claims a narrow local-vacancy mechanism."
5,Starting-rank quartile-adjusted dynamic premium (D2),fba x total seller turnover,0.002710,0.000869,0.002782,additive lag-rank quartile dummies; q1_top_25pct is the omitted category,main rank-position validation,NaN,Dynamic premium with additive starting-rank quartile controls. Adjusts for starting-rank composition without saturating transition-cell support.
6,Starting-rank quadratic-adjusted dynamic premium (D3),fba x total seller turnover,0.002278,0.000674,0.001278,smooth lag_rank_pct and lag_rank_pct squared,main rank-position validation,NaN,Dynamic premium with a smooth quadratic starting-rank adjustment. Avoids transition-cell saturation while letting the starting-rank effect be nonlinear.
7,Rank-tier FE stress test (D4),fba x total seller turnover,-0.000075,0.000699,0.915343,transition_id x lag_rank_tier FE; finite_design_warning=True,severe support-sensitive stress test,NaN,"A stricter comparison within transition and starting-rank tier. Reported as a severe support-sensitive stress test, not as the primary rank-position-adjusted estimator, because..."
8,Pre-disappearance descriptives,core variables before disappearance,1.000000,NaN,NaN,reported descriptively; not a mechanical threshold test,contextual validation,NaN,"Checks whether disappearing sellers visibly deteriorate before disappearance. Used as descriptive support, not over-control."
9,Dynamic support and MDE,exposed observations and detectable effect,0.002141,0.200763,NaN,"exposed FBA observations < 50; binary baseline event rate outside [0.05, 0.95]; nonpositive estimated SE; exposed FBA observations < 50; MDE in rank positions > 1.0",interpretive support,NaN,Documents empirical support and avoids interpreting underpowered auxiliary nulls as evidence of no premium.


Dynamic claim strength: credible but rank-tier-sensitive FBA-premium evidence
The notebook's central evidence is a static residual FBA ranking premium plus a primary dynamic FBA-by-turnover premium. The primary dynamic model is positive and significant under clustered, CR2, and wild-bootstrap inference. However, the transition-by-starting-rank-tier stress test does not confirm the same dynamic premium, so the dynamic result must be described as sensitive to rank-position structure. The claim remains observational and does not identify a pure within-seller FBA switching effect.


### 37. Evidence-to-claim table

The cell builds an explicit row-level mapping from each empirical block to its admissible interpretation. Each row records the empirical object identified by the block, the validation status under the available diagnostics, the open caveats, and the corresponding wording rule for the thesis text. The output is exported to `evidence_to_claim_summary.csv`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 37. Evidence-to-claim row-level mapping
# -----------------------------------------------------------------------------
# Build the explicit row-level mapping from each empirical block to its admissible interpretation, validation status, open caveats, and thesis-text wording rule.

_static_raw = main_results_table.loc[main_results_table["specification"].eq("spec_1_fba_only")]
_static_full = main_results_table.loc[main_results_table["specification"].eq(HEADLINE_SPEC)]
_raw_text = (
    f"FBA coefficient {float(_static_raw['coef_fba'].iloc[0]):.3f}, p={float(_static_raw['pvalue_fba'].iloc[0]):.3f}"
    if len(_static_raw) else "raw association estimated"
)
_full_text = (
    f"FBA coefficient {float(_static_full['coef_fba'].iloc[0]):.3f}, p={float(_static_full['pvalue_fba'].iloc[0]):.3f}"
    if len(_static_full) else "controlled residual association estimated"
)
if 'attenuation_decomposition_table' in globals() and len(attenuation_decomposition_table):
    _atten_col = 'share_absorbed_by_added_controls' if 'share_absorbed_by_added_controls' in attenuation_decomposition_table.columns else 'absorbed_share_vs_spec_1'
    _attenuation_text = f"absorbed share about {float(attenuation_decomposition_table[_atten_col].dropna().max()):.1%}"
else:
    _attenuation_text = "substantial attenuation across controls"

_premium_text = (
    f"premium={premium_estimate:.4f}, p={premium_pvalue:.3f}; conservative p={premium_conservative_pvalue:.3f}"
    if pd.notna(premium_conservative_pvalue) else
    f"premium={premium_estimate:.4f}, p={premium_pvalue:.3f}; conservative inference unavailable"
)
_ranktier_text = (
    f"rank-tier FE stress estimate={ranktier_premium_estimate:.4f}, p={ranktier_premium_pvalue:.3f}"
    if pd.notna(ranktier_premium_estimate) and pd.notna(ranktier_premium_pvalue) else
    "rank-tier FE stress test reported but not estimable or not finite"
)

final_claim_summary = pd.DataFrame([
    {
        "evidence_block": "Static raw association",
        "main_result": _raw_text,
        "supports": "FBA offers are much better ranked in within-market comparisons.",
        "outside_scope": "A causal effect of adopting FBA.",
        "required_caveat": "No within-seller FBA switching and strong offer-bundle selection.",
        "final_claim_strength": "strong descriptive",
    },
    {
        "evidence_block": "Static controlled residual FBA premium",
        "main_result": _full_text,
        "supports": "A residual FBA-label premium after observed controls.",
        "outside_scope": "The total FBA bundle effect, because some observed controls may be bundle components.",
        "required_caveat": "Price, shipping, delivery, and reputation may be mechanisms as well as controls.",
        "final_claim_strength": "moderate descriptive premium evidence",
    },
    {
        "evidence_block": "Attenuation accounting",
        "main_result": _attenuation_text,
        "supports": "Much of the raw FBA gap is tied to the observed commercial and logistics bundle.",
        "outside_scope": "A causal mediation decomposition through the observed controls.",
        "required_caveat": "Sequential adjustment is accounting, not causal decomposition.",
        "final_claim_strength": "strong descriptive accounting",
    },
    {
        "evidence_block": "Overlap and common-support evidence",
        "main_result": "Residual FBA estimate is sensitive to propensity-score support restrictions.",
        "supports": "Comparison-population sensitivity of the residual-label estimate.",
        "outside_scope": "A universal like-for-like zero effect across all support regions.",
        "required_caveat": "Common support is selected and does not recreate random assignment.",
        "final_claim_strength": "cautionary",
    },
    {
        "evidence_block": "Common-support selection diagnostics",
        "main_result": "Common-support non-FBA survivors are unusually strong ranking comparators.",
        "supports": "The common-support collapse is partly a selection result.",
        "outside_scope": "Complete balance on unobserved seller quality.",
        "required_caveat": "Selection diagnostics change interpretation of trimmed estimates.",
        "final_claim_strength": "strong cautionary",
    },
    {
        "evidence_block": "Static inference robustness",
        "main_result": "Two-way clustering, CR2-style correction, and wild bootstrap are reported.",
        "supports": "The full-sample residual association is not only a naive OLS file.",
        "outside_scope": "Random assignment or causal identification by inference corrections alone.",
        "required_caveat": "Finite-cluster corrections are sensitivity checks, not identification assumptions.",
        "final_claim_strength": "moderate robustness",
    },
    {
        "evidence_block": "Dynamic FBA turnover-premium headline",
        "main_result": _premium_text,
        "supports": "FBA sellers gain more rank position during seller-turnover transitions, conditional on seller fixed effects, transition fixed effects, lagged rank, lagged offer controls, total turnover, and directional churn composition.",
        "outside_scope": "A pure FBA main effect in a seller fixed-effects model, because FBA is time-invariant in the final seller panel.",
        "required_caveat": "Seller disappearances are observed offer-list absences, not randomized inventory shocks.",
        "final_claim_strength": claim_level,
    },
    {
        "evidence_block": "Dynamic directional decomposition",
        "main_result": f"direction-balance term={direction_balance_estimate:.4f}, p={direction_balance_pvalue:.3f}",
        "supports": "The dynamic premium is better framed as general turnover response when the directional term is not central.",
        "outside_scope": "A strictly local above-turnover mechanism as the central channel.",
        "required_caveat": "The thesis should not claim local vacancy filling as the main mechanism.",
        "final_claim_strength": "mechanism decomposition",
    },
    {
        "evidence_block": "Dynamic above/below reference estimates",
        "main_result": f"above={lambda_above:.4f}, p={p_above:.3f}; below={lambda_below:.4f}, p={p_below:.3f}" if pd.notna(lambda_below) and pd.notna(p_below) else "above/below decomposition reported",
        "supports": "A general turnover-premium interpretation when both above and below disappearances produce similar FBA adjustment.",
        "outside_scope": "Clean local-vacancy exogeneity.",
        "required_caveat": "Above/below evidence is a decomposition of the final premium model, not the central claim criterion.",
        "final_claim_strength": "decomposition evidence",
    },
    {
        "evidence_block": "Rank-tier FE stress test",
        "main_result": _ranktier_text,
        "supports": "Transparency about whether the turnover premium survives a stricter starting-rank comparison.",
        "outside_scope": "A replacement for the preferred model when thin support absorbs the identifying variation.",
        "required_caveat": "This stress test belongs in the body. A null or negative estimate means the primary dynamic premium is absorbed within starting-rank tier.",
        "final_claim_strength": "body-level cautionary dynamic validity check",
    },
    {
        "evidence_block": "Pre-disappearance and dropout-context evidence",
        "main_result": f"abnormal smoothness flags={S_pre}; return_share={return_share:.3f}" if pd.notna(return_share) else f"abnormal smoothness flags={S_pre}",
        "supports": "Whether disappearing sellers show obvious prior deterioration or stockout-like temporary absence patterns.",
        "outside_scope": "Random assignment of seller disappearance.",
        "required_caveat": "These are descriptive validation checks, not extra identifying controls in the main model.",
        "final_claim_strength": "contextual validation",
    },
    {
        "evidence_block": "Dynamic support and auxiliary stress tests",
        "main_result": f"premium MDE={premium_mde:.4f}; implied mean-market shift={premium_implied_rank_shift_mean_market:.3f}" if pd.notna(premium_mde) else "support and auxiliary stress tests reported",
        "supports": "Transparency about empirical support, detectable effect sizes, and sensitivity to rank mechanics.",
        "outside_scope": "A reason to treat every underpowered auxiliary null as a basis for overturning the premium estimate.",
        "required_caveat": "Small-panel stress tests are context, not a replacement for the main design.",
        "final_claim_strength": "body-level caution",
    },
    {
        "evidence_block": "Overall conclusion",
        "main_result": claim_text,
        "supports": "A disciplined FBA-premium interpretation combining static residual ranking evidence with dynamic turnover-premium evidence.",
        "outside_scope": "Definitive causal identification of platform self-preferencing or automatic external validity beyond this product page.",
        "required_caveat": "Single-product Italian marketplace panel; observed disappearances rather than randomized stockouts; ranking outcome rather than sales or click-through.",
        "final_claim_strength": claim_level,
    },
])


# Add main-analysis hardening evidence rows after the base claim table has been assembled.
_extra_claim_rows_hardening = []

if "dynamic_turnover_placebo_table" in globals() and len(dynamic_turnover_placebo_table):
    _extra_claim_rows_hardening.append({
        "evidence_block": "Dynamic placebo checks for total-turnover estimand",
        "main_result": dynamic_turnover_placebo_table[["model_role", "term", "estimate", "pvalue"]].to_dict("records"),
        "supports": "Turnover-timing validation: future turnover should not predict current rank movement and current turnover should not predict previous rank movement.",
        "outside_scope": "Proof of random assignment of turnover events.",
        "required_caveat": "Passing placebos hardens the design but does not eliminate unobserved time-varying confounding.",
        "final_claim_strength": "dynamic placebo validation",
    })
if "dynamic_permutation_placebo_turnover_table" in globals() and len(dynamic_permutation_placebo_turnover_table):
    _extra_claim_rows_hardening.append({
        "evidence_block": "Permutation within transition x starting-rank-tier",
        "main_result": dynamic_permutation_placebo_turnover_table.to_dict("records"),
        "supports": "Whether the observed FBA-by-turnover coefficient is unusual relative to local FBA-status reassignment within comparable transition-rank cells.",
        "outside_scope": "A definitive solution to thin support or unobserved seller quality.",
        "required_caveat": "Permutation p-values near conventional thresholds should be reported with Monte Carlo uncertainty and interpreted as rank-composition hardening, not causal proof.",
        "final_claim_strength": "rank-composition placebo",
    })
if "dynamic_stockout_consistent_turnover_table" in globals() and len(dynamic_stockout_consistent_turnover_table):
    _extra_claim_rows_hardening.append({
        "evidence_block": "Stockout-consistent restricted turnover design",
        "main_result": dynamic_stockout_consistent_turnover_table[["stockout_definition", "model_role", "term", "estimate", "pvalue", "ill_conditioned_flag"]].to_dict("records"),
        "supports": "Whether the FBA turnover premium survives after restricting disappearance events to broad and strict stockout-consistent definitions.",
        "outside_scope": "Direct observation of inventory stockouts.",
        "required_caveat": "The broad definition is the main validation check; strict near-term return may be thinly powered.",
        "final_claim_strength": "stockout-consistency hardening",
    })
if "economic_salience_table" in globals() and len(economic_salience_table):
    _extra_claim_rows_hardening.append({
        "evidence_block": "Auxiliary binary economic-salience outcomes",
        "main_result": economic_salience_table[["outcome", "estimate_lambda", "pvalue", "baseline_rate", "role_in_claim"]].to_dict("records"),
        "supports": "Whether the primary dynamic turnover premium maps into top-of-list events, especially entry into the top 10.",
        "outside_scope": "Replacement of the continuous-rank primary estimand.",
        "required_caveat": "Binary outcomes are auxiliary linear-probability salience translations and should not be overinterpreted as the main design.",
        "final_claim_strength": "auxiliary economic salience",
    })
if "dynamic_clean_turnover_restriction_table" in globals() and len(dynamic_clean_turnover_restriction_table):
    _clean_report = dynamic_clean_turnover_restriction_table[["sample_rule", "estimate", "pvalue", "raw_rows_before_dropna"]].copy()
    _extra_claim_rows_hardening.append({
        "evidence_block": "Clean one-sided turnover restrictions",
        "main_result": _clean_report.to_dict("records"),
        "supports": "Whether the FBA turnover premium appears outside mixed above/below churn rows.",
        "outside_scope": "A causal FBA treatment effect or a replacement for the preferred estimator.",
        "required_caveat": "Restricted samples may have thin support after seller and transition fixed effects.",
        "final_claim_strength": "diagnostic hardening",
    })
if "dynamic_distance_banded_turnover_table" in globals() and len(dynamic_distance_banded_turnover_table):
    _band_report = dynamic_distance_banded_turnover_table.loc[dynamic_distance_banded_turnover_table["role"].eq("premium_term"), ["band_k", "estimate", "pvalue", "share_positive_banded_turnover"]].copy()
    _extra_claim_rows_hardening.append({
        "evidence_block": "Distance-banded turnover exposure",
        "main_result": _band_report.to_dict("records"),
        "supports": "Whether nearby rank-vacancy exposure yields the same qualitative FBA premium.",
        "outside_scope": "A broad causal interpretation from sparse local-band support.",
        "required_caveat": "Banded exposure can be underpowered because local dropouts are sparse after fixed effects.",
        "final_claim_strength": "diagnostic hardening",
    })
if "dynamic_partial_identification_bounds_table" in globals() and len(dynamic_partial_identification_bounds_table):
    _bound_report = dynamic_partial_identification_bounds_table[["bound_type", "estimate", "pvalue", "finite_design_warning"]].copy()
    _extra_claim_rows_hardening.append({
        "evidence_block": "Above-minus-below partial-identification-style bound",
        "main_result": _bound_report.to_dict("records"),
        "supports": "A transparent bound on the residual local-above component after accounting for the mirrored below response.",
        "outside_scope": "The main turnover-premium estimate; this row is a local-mechanism diagnostic.",
        "required_caveat": "If the bound is small or unstable, the local-vacancy mechanism should remain exploratory rather than central.",
        "final_claim_strength": "local-mechanism caution",
    })
if _extra_claim_rows_hardening:
    final_claim_summary = pd.concat([final_claim_summary, pd.DataFrame(_extra_claim_rows_hardening)], ignore_index=True)

print("Table Z1 / Table 8. Evidence-to-claim summary")
display(final_claim_summary)

Table Z1 / Table 8. Evidence-to-claim summary


,evidence_block,main_result,supports,outside_scope,required_caveat,final_claim_strength
0,Static raw association,"FBA coefficient -0.380, p=0.000",FBA offers are much better ranked in within-market comparisons.,A causal effect of adopting FBA.,No within-seller FBA switching and strong offer-bundle selection.,strong descriptive
1,Static controlled residual FBA premium,"FBA coefficient -0.051, p=0.002",A residual FBA-label premium after observed controls.,"The total FBA bundle effect, because some observed controls may be bundle components.","Price, shipping, delivery, and reputation may be mechanisms as well as controls.",moderate descriptive premium evidence
2,Attenuation accounting,absorbed share about 86.7%,Much of the raw FBA gap is tied to the observed commercial and logistics bundle.,A causal mediation decomposition through the observed controls.,"Sequential adjustment is accounting, not causal decomposition.",strong descriptive accounting
3,Overlap and common-support evidence,Residual FBA estimate is sensitive to propensity-score support restrictions.,Comparison-population sensitivity of the residual-label estimate.,A universal like-for-like zero effect across all support regions.,Common support is selected and does not recreate random assignment.,cautionary
4,Common-support selection diagnostics,Common-support non-FBA survivors are unusually strong ranking comparators.,The common-support collapse is partly a selection result.,Complete balance on unobserved seller quality.,Selection diagnostics change interpretation of trimmed estimates.,strong cautionary
5,Static inference robustness,"Two-way clustering, CR2-style correction, and wild bootstrap are reported.",The full-sample residual association is not only a naive OLS file.,Random assignment or causal identification by inference corrections alone.,"Finite-cluster corrections are sensitivity checks, not identification assumptions.",moderate robustness
6,Dynamic FBA turnover-premium headline,"premium=0.0025, p=0.002; conservative p=0.025","FBA sellers gain more rank position during seller-turnover transitions, conditional on seller fixed effects, transition fixed effects, lagged rank, lagged offer controls, total...","A pure FBA main effect in a seller fixed-effects model, because FBA is time-invariant in the final seller panel.","Seller disappearances are observed offer-list absences, not randomized inventory shocks.",credible but rank-tier-sensitive FBA-premium evidence
7,Dynamic directional decomposition,"direction-balance term=-0.0001, p=0.953",The dynamic premium is better framed as general turnover response when the directional term is not central.,A strictly local above-turnover mechanism as the central channel.,The thesis should not claim local vacancy filling as the main mechanism.,mechanism decomposition
8,Dynamic above/below reference estimates,"above=0.0036, p=0.036; below=0.0034, p=0.004",A general turnover-premium interpretation when both above and below disappearances produce similar FBA adjustment.,Clean local-vacancy exogeneity.,"Above/below evidence is a decomposition of the final premium model, not the central claim criterion.",decomposition evidence
9,Rank-tier FE stress test,"rank-tier FE stress estimate=-0.0001, p=0.915",Transparency about whether the turnover premium survives a stricter starting-rank comparison.,A replacement for the preferred model when thin support absorbs the identifying variation.,This stress test belongs in the body. A null or negative estimate means the primary dynamic premium is absorbed within starting-rank tier.,body-level cautionary dynamic validity check


### 38. Complementary binary-prominence claim integration

The cell appends the validated top-10 prominence evidence to the claim-synthesis objects built above. The top-10 entry coefficient enters the synthesis with the explicit auxiliary status of a discrete-event salience translation of the continuous rank-movement estimate. The rare-event power caveat is recorded together with the coefficient.


In [ ]:
# -----------------------------------------------------------------------------
# Section 38. Complementary binary-prominence claim integration
# -----------------------------------------------------------------------------
# Append the validated top-10 prominence evidence to the claim-synthesis objects with its auxiliary-status label and rare-event power caveat.

if "binary_top10_prominence_decision_table" in globals() and len(binary_top10_prominence_decision_table):
    _b10 = binary_top10_prominence_decision_table.iloc[0]
    _b10_claim_row = {
        "evidence_block": "Secondary binary top-10 prominence estimand",
        "main_result": (
            f"top-10 entry beta={float(_b10.get('preferred_beta', np.nan)):.4f}, "
            f"p={float(_b10.get('preferred_pvalue', np.nan)):.3f}; "
            f"permutation p={float(_b10.get('permutation_p_value_two_sided', np.nan)):.3f}"
        ) if pd.notna(_b10.get("preferred_beta", np.nan)) else "top-10 model not estimable",
        "supports": "Whether the FBA turnover-conditional rank-movement premium translates into entry into the most visible part of the offer list among sellers at risk of entering the top 10.",
        "outside_scope": "Replacement of the continuous rank-percent primary estimand or proof of causal algorithmic discrimination.",
        "required_caveat": "Top-10 entry is a rare-event complementary LPM outcome and must be discussed together with risk-set support, permutation inference, rank-tier stress, and threshold-family robustness.",
        "final_claim_strength": str(_b10.get("decision", "complementary_binary_prominence")),
    }
    if "final_claim_summary" in globals() and isinstance(final_claim_summary, pd.DataFrame):
        final_claim_summary = final_claim_summary.loc[~final_claim_summary["evidence_block"].eq("Secondary binary top-10 prominence estimand")].copy()
        final_claim_summary = pd.concat([final_claim_summary, pd.DataFrame([_b10_claim_row])], ignore_index=True)
    else:
        final_claim_summary = pd.DataFrame([_b10_claim_row])

    _b10_se = np.nan
    if "binary_top10_prominence_inference_table" in globals() and isinstance(binary_top10_prominence_inference_table, pd.DataFrame):
        _b10_primary = binary_top10_prominence_inference_table.loc[
            binary_top10_prominence_inference_table["method"].eq("preferred_two_way_seller_transition_clustered_lpm")
        ]
        if len(_b10_primary) and pd.notna(_b10_primary["se"].iloc[0]):
            _b10_se = float(_b10_primary["se"].iloc[0])
    _b10_conservative_p = float(_b10.get("conservative_max_pvalue_across_inference", np.nan)) if pd.notna(_b10.get("conservative_max_pvalue_across_inference", np.nan)) else np.nan

    _b10_fitted_audit_note = "fitted-probability audit unavailable"
    if "binary_top10_fitted_probability_audit_table" in globals() and isinstance(binary_top10_fitted_probability_audit_table, pd.DataFrame) and len(binary_top10_fitted_probability_audit_table):
        _fa = binary_top10_fitted_probability_audit_table.iloc[0]
        if pd.notna(_fa.get("fitted_outside_unit_interval_share", np.nan)):
            _b10_fitted_audit_note = (
                f"LPM fitted outside [0,1] share={float(_fa.get('fitted_outside_unit_interval_share', np.nan)):.3f}; "
                f"min={float(_fa.get('fitted_min', np.nan)):.3f}; max={float(_fa.get('fitted_max', np.nan)):.3f}"
            )
    _b10_validation_row = {
        "test_family": "Secondary binary top-10 prominence estimand",
        "key_term": "fba_x_dropouts_total_focal",
        "estimate": float(_b10.get("preferred_beta", np.nan)) if pd.notna(_b10.get("preferred_beta", np.nan)) else np.nan,
        "se_or_mde": _b10_se,
        "pvalue": float(_b10.get("preferred_pvalue", np.nan)) if pd.notna(_b10.get("preferred_pvalue", np.nan)) else np.nan,
        "support_metric": f"risk-set top-10 LPM; preferred p={float(_b10.get('preferred_pvalue', np.nan)):.3f}; conservative max p={_b10_conservative_p:.3f}; permutation p={float(_b10.get('permutation_p_value_two_sided', np.nan)):.3f}; threshold-family FDR q={float(_b10.get('threshold_family_fdr_qvalue_top10', np.nan)):.3f}; {_b10_fitted_audit_note}" if pd.notna(_b10_conservative_p) else f"risk-set top-10 LPM; CR2/WCB; timing placebos; local permutation; threshold family; stockout restrictions; leave-one diagnostics; {_b10_fitted_audit_note}",
        "role_in_claim": "auxiliary finite-cluster-sensitive prominence evidence",
        "pass_flag": np.nan,
        "interpretation": "Prominence translation of the continuous turnover premium. It is directionally supportive but auxiliary because top-10 entry is sparse and the conservative finite-cluster p-value is weaker than the primary continuous-rank evidence.",
    }
    if "dynamic_mechanism_validation_summary_table" in globals() and isinstance(dynamic_mechanism_validation_summary_table, pd.DataFrame):
        dynamic_mechanism_validation_summary_table = dynamic_mechanism_validation_summary_table.loc[
            ~dynamic_mechanism_validation_summary_table["test_family"].isin([
                "Auxiliary binary top-of-list salience",
                "Secondary binary top-10 prominence estimand",
            ])
        ].copy()
        dynamic_mechanism_validation_summary_table = pd.concat(
            [dynamic_mechanism_validation_summary_table, pd.DataFrame([_b10_validation_row])],
            ignore_index=True,
        )
    else:
        dynamic_mechanism_validation_summary_table = pd.DataFrame([_b10_validation_row])

print("Updated evidence-to-claim summary including secondary top-10 prominence evidence")
if "final_claim_summary" in globals():
    display(final_claim_summary)
print("Updated dynamic validation summary including secondary top-10 prominence evidence")
if "dynamic_mechanism_validation_summary_table" in globals():
    display(dynamic_mechanism_validation_summary_table)

Updated evidence-to-claim summary including secondary top-10 prominence evidence


,evidence_block,main_result,supports,outside_scope,required_caveat,final_claim_strength
0,Static raw association,"FBA coefficient -0.380, p=0.000",FBA offers are much better ranked in within-market comparisons.,A causal effect of adopting FBA.,No within-seller FBA switching and strong offer-bundle selection.,strong descriptive
1,Static controlled residual FBA premium,"FBA coefficient -0.051, p=0.002",A residual FBA-label premium after observed controls.,"The total FBA bundle effect, because some observed controls may be bundle components.","Price, shipping, delivery, and reputation may be mechanisms as well as controls.",moderate descriptive premium evidence
2,Attenuation accounting,absorbed share about 86.7%,Much of the raw FBA gap is tied to the observed commercial and logistics bundle.,A causal mediation decomposition through the observed controls.,"Sequential adjustment is accounting, not causal decomposition.",strong descriptive accounting
3,Overlap and common-support evidence,Residual FBA estimate is sensitive to propensity-score support restrictions.,Comparison-population sensitivity of the residual-label estimate.,A universal like-for-like zero effect across all support regions.,Common support is selected and does not recreate random assignment.,cautionary
4,Common-support selection diagnostics,Common-support non-FBA survivors are unusually strong ranking comparators.,The common-support collapse is partly a selection result.,Complete balance on unobserved seller quality.,Selection diagnostics change interpretation of trimmed estimates.,strong cautionary
5,Static inference robustness,"Two-way clustering, CR2-style correction, and wild bootstrap are reported.",The full-sample residual association is not only a naive OLS file.,Random assignment or causal identification by inference corrections alone.,"Finite-cluster corrections are sensitivity checks, not identification assumptions.",moderate robustness
6,Dynamic FBA turnover-premium headline,"premium=0.0025, p=0.002; conservative p=0.025","FBA sellers gain more rank position during seller-turnover transitions, conditional on seller fixed effects, transition fixed effects, lagged rank, lagged offer controls, total...","A pure FBA main effect in a seller fixed-effects model, because FBA is time-invariant in the final seller panel.","Seller disappearances are observed offer-list absences, not randomized inventory shocks.",credible but rank-tier-sensitive FBA-premium evidence
7,Dynamic directional decomposition,"direction-balance term=-0.0001, p=0.953",The dynamic premium is better framed as general turnover response when the directional term is not central.,A strictly local above-turnover mechanism as the central channel.,The thesis should not claim local vacancy filling as the main mechanism.,mechanism decomposition
8,Dynamic above/below reference estimates,"above=0.0036, p=0.036; below=0.0034, p=0.004",A general turnover-premium interpretation when both above and below disappearances produce similar FBA adjustment.,Clean local-vacancy exogeneity.,"Above/below evidence is a decomposition of the final premium model, not the central claim criterion.",decomposition evidence
9,Rank-tier FE stress test,"rank-tier FE stress estimate=-0.0001, p=0.915",Transparency about whether the turnover premium survives a stricter starting-rank comparison.,A replacement for the preferred model when thin support absorbs the identifying variation.,This stress test belongs in the body. A null or negative estimate means the primary dynamic premium is absorbed within starting-rank tier.,body-level cautionary dynamic validity check


Updated dynamic validation summary including secondary top-10 prominence evidence


,test_family,key_term,estimate,se_or_mde,pvalue,support_metric,role_in_claim,pass_flag,interpretation
0,Static residual FBA premium,fba_from_shipper,0.051038,NaN,0.002207,sign-normalized: positive means FBA ranked better,core descriptive,True,The controlled static model shows a residual FBA ranking premium. This is descriptive because FBA is not randomly assigned.
1,Dynamic FBA turnover premium,fba x total seller turnover,0.002467,0.000765,0.002034,primary two-way clustered model; MDE=0.0021,core dynamic,True,"Main final dynamic estimate: FBA sellers move upward more during seller-turnover transitions, regardless of whether disappearing sellers were above or below."
2,Conservative dynamic premium inference,fba x total seller turnover,0.002467,NaN,0.024824,cr2_transition,core inference check,True,Uses the least favorable sign-consistent CR2/bootstrap p-value available for the final premium coefficient.
3,Directional churn decomposition,fba x dropout direction balance,-0.000117,NaN,0.953169,controls whether disappearances are concentrated above rather than below,mechanism decomposition,NaN,A non-central directional term means the evidence is better read as a general turnover premium than as a local above-turnover mechanism.
4,Above and below reference estimates,fba x dropouts_above; fba x dropouts_below,0.003606,0.003385,0.035626,below p=0.004,mechanism decomposition,NaN,"The mirrored below response is not a failure of the final turnover-premium claim, because the notebook no longer claims a narrow local-vacancy mechanism."
5,Starting-rank quartile-adjusted dynamic premium (D2),fba x total seller turnover,0.002710,0.000869,0.002782,additive lag-rank quartile dummies; q1_top_25pct is the omitted category,main rank-position validation,NaN,Dynamic premium with additive starting-rank quartile controls. Adjusts for starting-rank composition without saturating transition-cell support.
6,Starting-rank quadratic-adjusted dynamic premium (D3),fba x total seller turnover,0.002278,0.000674,0.001278,smooth lag_rank_pct and lag_rank_pct squared,main rank-position validation,NaN,Dynamic premium with a smooth quadratic starting-rank adjustment. Avoids transition-cell saturation while letting the starting-rank effect be nonlinear.
7,Rank-tier FE stress test (D4),fba x total seller turnover,-0.000075,0.000699,0.915343,transition_id x lag_rank_tier FE; finite_design_warning=True,severe support-sensitive stress test,NaN,"A stricter comparison within transition and starting-rank tier. Reported as a severe support-sensitive stress test, not as the primary rank-position-adjusted estimator, because..."
8,Pre-disappearance descriptives,core variables before disappearance,1.000000,NaN,NaN,reported descriptively; not a mechanical threshold test,contextual validation,NaN,"Checks whether disappearing sellers visibly deteriorate before disappearance. Used as descriptive support, not over-control."
9,Dynamic support and MDE,exposed observations and detectable effect,0.002141,0.200763,NaN,"exposed FBA observations < 50; binary baseline event rate outside [0.05, 0.95]; nonpositive estimated SE; exposed FBA observations < 50; MDE in rank positions > 1.0",interpretive support,NaN,Documents empirical support and avoids interpreting underpowered auxiliary nulls as evidence of no premium.


### 39. Core table spine

The table spine lists the central tables needed to audit the empirical argument. Each row identifies the table, the empirical block that produces it, the corresponding thesis chapter or appendix entry, and the export filename. The first code cell of this section builds the spine itself; the second code cell runs the consistency checks that verify each spine table exists in the active namespace, is non-empty, and exposes the columns referenced by the thesis. The output is exported to `spine_table_manifest.csv`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 39. Core table spine
# -----------------------------------------------------------------------------
# Assemble the spine of central tables (sample audit, balance, static, dynamic, validity-boundary map) needed to audit the empirical argument.

spine_tables = {
    "Table 1. Final sample audit": audit_reconciliation.copy() if "audit_reconciliation" in globals() else audit_checks_frame(),
    "Table 2. Static nested models and estimands": main_results_table.copy(),
    "Table 3. Static variants and attenuation summary": attenuation_decomposition_table.copy(),
    "Table 4. Overlap estimates and common-support selection": overlap_trimmed_estimates_table.copy() if "overlap_trimmed_estimates_table" in globals() else propensity_overlap_summary_table.copy(),
    "Table 5. Static inference and robustness summary": cluster_sensitivity_table.copy(),
    "Table 6. Dynamic sample, support, and MDE audit": dynamic_support_mde_table.copy(),
    "Table 7. Dynamic FBA premium, inference, and validation summary": dynamic_mechanism_validation_summary_table.copy(),
    "Table 8. Evidence-to-claim summary": final_claim_summary.copy(),
}
spine_table_manifest = pd.DataFrame([
    {"spine_table": name, "rows": int(len(tbl)), "columns": int(len(tbl.columns))}
    for name, tbl in spine_tables.items()
])
print("Eight-table spine manifest")
display(spine_table_manifest)

Eight-table spine manifest


,spine_table,rows,columns
0,Table 1. Final sample audit,12,4
1,Table 2. Static nested models and estimands,4,18
2,Table 3. Static variants and attenuation summary,3,11
3,Table 4. Overlap estimates and common-support selection,32,28
4,Table 5. Static inference and robustness summary,3,18
5,"Table 6. Dynamic sample, support, and MDE audit",9,19
6,"Table 7. Dynamic FBA premium, inference, and validation summary",17,9
7,Table 8. Evidence-to-claim summary,21,6


In [ ]:
# -----------------------------------------------------------------------------
# Section 39. Core table spine: consistency assertions
# -----------------------------------------------------------------------------
# Run assertion checks that verify each spine table exists, is non-empty, and exposes the columns referenced by the conclusion and the export manifest.

TABLE_REQUIREMENTS = {
    "chen_tsai_mapping_table": ["design_dimension", "chen_tsai_2024", "this_thesis", "implication_for_interpretation"],
    "dynamic_support_mde_table": ["model_or_subsample", "outcome", "observations", "sellers", "transitions", "estimated_se_for_key_term", "mde_80_power_alpha_0_05", "power_warning"],
    "dynamic_power_audit": ["model_or_subsample", "mde_80_power_alpha_0_05", "power_warning"],
    "dynamic_cr2_table": ["method", "term", "estimate_lambda", "se", "pvalue", "target_estimand"],
    "dynamic_wcb_table": ["term", "estimate_lambda", "two_way_cluster_se", "bootstrap_pvalue", "studentized", "recomputed_se_each_replication", "valid_replication_share", "invalid_replication_share", "target_estimand"],
    "above_below_decomposition_table": ["exposure", "interaction_term", "estimate", "se", "pvalue"],
    "churn_direction_correlation_table": ["observation_level_correlation_dropouts_above_below", "transition_level_correlation_mean_above_below", "interpretation"],
    "dynamic_churn_decomposition_table": ["model_role", "term", "estimate", "se", "pvalue", "interpretation"],
    "dynamic_churn_decomposition_summary_table": ["preferred_dynamic_mechanism", "validity_implication", "premium_estimate", "premium_pvalue"],
    "dynamic_fba_premium_summary_table": ["evidence_block", "term", "premium_sign_normalized_estimate", "two_sided_pvalue", "role_in_claim"],
    "dynamic_fba_premium_inference_table": ["method", "term", "estimate", "se", "pvalue"],
    "dynamic_clean_turnover_restriction_table": ["sample_rule", "term", "estimate", "se", "pvalue", "interpretation"],
    "dynamic_distance_banded_turnover_table": ["band_k", "term", "estimate", "se", "pvalue", "interpretation"],
    "dynamic_partial_identification_bounds_table": ["bound_type", "estimand", "estimate", "se", "pvalue", "interpretation"],
    "dynamic_rank_tier_stress_diagnostic_table": ["object", "estimate", "pvalue", "condition_number", "interpretation"],
    "dynamic_hardening_summary_table": ["test_family", "model_role", "term", "estimate", "se", "pvalue", "interpretation"],
    "dropout_process_table": ["variable", "estimate", "se", "pvalue"],
    "dropout_process_fit_summary_table": ["r_squared", "significant_observed_predictors", "overwhelming_predictability_flag"],
    "smoothness_table": ["variable", "difference", "standardized_difference", "pvalue_welch", "materially_abnormal"],
    "return_pattern_summary_table": ["temporary_or_return_share", "right_censored_share", "pass_for_strong_claim"],
    "economic_salience_table": ["outcome", "estimate_lambda", "se", "pvalue", "baseline_rate"],
    "conservative_dynamic_inference_table": ["method", "estimate", "pvalue", "source"],
    "dynamic_decision_quantities_table": ["premium_lambda_turnover", "premium_pvalue_preferred_two_sided", "premium_conservative_pvalue", "dynamic_claim_strength"],
    "dynamic_mechanism_validation_summary_table": ["test_family", "key_term", "estimate", "se_or_mde", "pvalue", "pass_flag", "interpretation"],
    "final_claim_summary": ["evidence_block", "main_result", "supports", "outside_scope", "required_caveat", "final_claim_strength"],
    "spine_table_manifest": ["spine_table", "rows", "columns"],
}

_consistency_rows = []
for obj_name, required_cols in TABLE_REQUIREMENTS.items():
    exists = obj_name in globals()
    obj = globals().get(obj_name)
    is_df = isinstance(obj, pd.DataFrame)
    cols = list(obj.columns) if is_df else []
    missing = [c for c in required_cols if c not in cols]
    nonempty = bool(is_df and len(obj) > 0)
    passed = bool(exists and is_df and nonempty and not missing)
    detail = "ok" if passed else f"exists={exists}; is_dataframe={is_df}; nonempty={nonempty}; missing={missing}"
    _consistency_rows.append({
        "object": obj_name,
        "check_type": "critical_table_presence",
        "required_columns": ", ".join(required_cols),
        "rows": int(len(obj)) if is_df else 0,
        "missing_columns": ", ".join(missing),
        "critical": True,
        "passed": passed,
        "warning": False,
        "detail": detail,
    })

# Numerical quality checks beyond mere table existence. These rows are intentionally
# warning-aware. They are exported for audit transparency, but they do not determine
# whether the notebook ran or whether critical objects are present.
_quality_rows = []
for _name in ["wild_cluster_bootstrap_table", "dynamic_wcb_table", "dynamic_fba_premium_inference_table"]:
    _obj = globals().get(_name)
    if isinstance(_obj, pd.DataFrame) and len(_obj):
        _has_valid_col = "valid_replication_share" in _obj.columns
        _has_invalid_col = "invalid_replication_share" in _obj.columns
        _valid_share = pd.to_numeric(_obj.get("valid_replication_share", pd.Series([np.nan])), errors="coerce").min()
        _invalid_share = pd.to_numeric(_obj.get("invalid_replication_share", pd.Series([np.nan])), errors="coerce").max()
        _metadata_present = bool(_has_valid_col and _has_invalid_col and pd.notna(_valid_share) and pd.notna(_invalid_share))
        _quality_passed = bool(_metadata_present and _valid_share >= WILD_BOOTSTRAP_MIN_VALID_SHARE)
        _quality_rows.append({
            "object": _name,
            "check_type": "bootstrap_quality_warning",
            "required_columns": "valid_replication_share, invalid_replication_share",
            "rows": int(len(_obj)),
            "missing_columns": "" if (_has_valid_col and _has_invalid_col) else ", ".join([c for c in ["valid_replication_share", "invalid_replication_share"] if c not in _obj.columns]),
            "critical": False,
            "passed": bool(_metadata_present),
            "quality_passed": _quality_passed,
            "warning": bool(_metadata_present and not _quality_passed),
            "detail": f"bootstrap_quality_checks: min_valid_share={_valid_share:.3f}; max_invalid_share={_invalid_share:.3f}; warning_threshold={WILD_BOOTSTRAP_MIN_VALID_SHARE:.3f}",
        })

for _name in ["above_below_decomposition_table", "dynamic_churn_decomposition_table", "dynamic_fba_premium_summary_table", "dropout_composition_table", "dynamic_vacancy_rank_tier_models_table"]:
    _obj = globals().get(_name)
    if isinstance(_obj, pd.DataFrame) and len(_obj):
        _se_cols = [c for c in _obj.columns if c in ["se", "cluster_se_fba", "two_way_cluster_se_fba"]]
        _has_zero_se = False
        for _col in _se_cols:
            _vals = pd.to_numeric(_obj[_col], errors="coerce")
            _has_zero_se = _has_zero_se or bool((_vals == 0).any())
        _condition = pd.to_numeric(_obj.get("condition_number", pd.Series([np.nan])), errors="coerce")
        _ill = bool((_condition > 1e8).any()) if len(_condition) else False
        _quality_rows.append({
            "object": _name,
            "check_type": "numerical_pathology_warning",
            "required_columns": "finite positive standard errors in interpreted rows; condition-number flags exposed",
            "rows": int(len(_obj)),
            "missing_columns": "",
            "critical": False,
            "passed": True,
            "quality_passed": bool(not _has_zero_se),
            "warning": bool(_has_zero_se or _ill),
            "detail": f"numerical_pathology_scan: zero_se_present={_has_zero_se}; ill_conditioned_present={_ill}; interpret flagged rows only as appendix/unstable",
        })

audit_consistency_table = pd.DataFrame(_consistency_rows + _quality_rows)

_critical_failures = audit_consistency_table.loc[
    audit_consistency_table["critical"].fillna(False) & (~audit_consistency_table["passed"].fillna(False)),
    ["object", "detail"],
]
register_check(
    "G.4c",
    "audit_consistency_all_critical_tables_present",
    bool(_critical_failures.empty),
    _critical_failures.to_dict("records"),
)

_bootstrap_quality_rows = audit_consistency_table.loc[
    audit_consistency_table["check_type"].eq("bootstrap_quality_warning"),
    ["object", "passed", "quality_passed", "warning", "detail"],
]
register_check(
    "G.4c",
    "audit_bootstrap_quality_metadata_recorded",
    bool(len(_bootstrap_quality_rows) > 0 and _bootstrap_quality_rows["passed"].fillna(False).all()),
    _bootstrap_quality_rows.to_dict("records"),
)

print("Audit consistency checks")
display(audit_consistency_table)

Audit consistency checks


,object,check_type,required_columns,rows,missing_columns,critical,passed,warning,detail,quality_passed
0,chen_tsai_mapping_table,critical_table_presence,"design_dimension, chen_tsai_2024, this_thesis, implication_for_interpretation",12,,True,True,False,ok,NaN
1,dynamic_support_mde_table,critical_table_presence,"model_or_subsample, outcome, observations, sellers, transitions, estimated_se_for_key_term, mde_80_power_alpha_0_05, power_warning",9,,True,True,False,ok,NaN
2,dynamic_power_audit,critical_table_presence,"model_or_subsample, mde_80_power_alpha_0_05, power_warning",9,,True,True,False,ok,NaN
3,dynamic_cr2_table,critical_table_presence,"method, term, estimate_lambda, se, pvalue, target_estimand",2,,True,True,False,ok,NaN
4,dynamic_wcb_table,critical_table_presence,"term, estimate_lambda, two_way_cluster_se, bootstrap_pvalue, studentized, recomputed_se_each_replication, valid_replication_share, invalid_replication_share, target_estimand",1,,True,True,False,ok,NaN
5,above_below_decomposition_table,critical_table_presence,"exposure, interaction_term, estimate, se, pvalue",3,,True,True,False,ok,NaN
6,churn_direction_correlation_table,critical_table_presence,"observation_level_correlation_dropouts_above_below, transition_level_correlation_mean_above_below, interpretation",1,,True,True,False,ok,NaN
7,dynamic_churn_decomposition_table,critical_table_presence,"model_role, term, estimate, se, pvalue, interpretation",10,,True,True,False,ok,NaN
8,dynamic_churn_decomposition_summary_table,critical_table_presence,"preferred_dynamic_mechanism, validity_implication, premium_estimate, premium_pvalue",1,,True,True,False,ok,NaN
9,dynamic_fba_premium_summary_table,critical_table_presence,"evidence_block, term, premium_sign_normalized_estimate, two_sided_pvalue, role_in_claim",7,,True,True,False,ok,NaN


### 40. Exported tables and consistency checks

The export block writes the econometric tables produced in the active kernel to `Datasets/Econometrics-Results/`. It also writes the model-specifications YAML payload, the export manifest YAML payload, the missingness-summary table, the numerical-pathology summary, and the audit-reconciliation table. Each exported file is recorded in the manifest with its row count, column names, and short description. The manifest is the primary index for the replication record.


In [ ]:
# -----------------------------------------------------------------------------
# Section 40. Exported tables and consistency checks
# -----------------------------------------------------------------------------
# Write the econometric tables, the model-specifications YAML, the export manifest YAML, the missingness summary, the numerical-pathology summary, and the audit-reconciliation table to the Econometrics-Results directory.

try:
    dynamic_power_audit = dynamic_support_mde_table.copy()
except Exception:
    pass
try:
    targeted_dynamic_validation_table = dynamic_mechanism_validation_summary_table.copy()
except Exception:
    pass
try:
    dynamic_claim_logic_table = decision_scorecard.copy()
except Exception:
    pass
try:
    targeted_churn_decomposition_table = dynamic_churn_decomposition_table.copy()
except Exception:
    pass

# Export registry and audit manifest.
# This block writes tables, diagnostics, and manifest metadata generated by the active run.
# It does not read exported result files back into estimation.

if EXPORT_FILES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if "price_threshold_support_table" in globals():
        price_threshold_support_table.to_csv(OUTPUT_DIR / "price_threshold_support.csv", index=False)



def _make_missingness_summary():
    rows = []
    static_cols = [
        "rank_pct", "rank_pos", "fba_from_shipper", "prezzo", "prezzo_spedizione_repaired",
        "prezzo_totale_reconstructed", "prezzo_totale_fast_delivery_imputed",
        "g_cons_min_robust", "fast_delivery_available", "fast_delivery_cost_imputed",
        "log1p_num_valutazioni", "valutazioni_positive", "stelle",
    ]
    if "df_final" in globals():
        for col in static_cols:
            if col in df_final.columns:
                rows.append({
                    "sample": "static_final", "variable": col,
                    "observations": int(len(df_final)),
                    "missing": int(df_final[col].isna().sum()),
                    "missing_share": float(df_final[col].isna().mean()),
                })
    dynamic_cols = [
        "rank_pct_improvement", "fba_from_shipper", "dropouts_above", "dropouts_below",
        "dropouts_total_focal", "lag_rank_pct", "lag_prezzo", "lag_prezzo_spedizione_repaired",
        "lag_prezzo_totale_reconstructed", "lag_g_cons_min_robust", "lag_log1p_num_valutazioni",
        "lag_valutazioni_positive", "lag_stelle",
    ]
    dyn = globals().get("_churn_panel", globals().get("dynamic_panel_with_below", globals().get("dynamic_vacancy_panel", None)))
    if isinstance(dyn, pd.DataFrame):
        for col in dynamic_cols:
            if col in dyn.columns:
                rows.append({
                    "sample": "dynamic_final", "variable": col,
                    "observations": int(len(dyn)),
                    "missing": int(dyn[col].isna().sum()),
                    "missing_share": float(dyn[col].isna().mean()),
                })
    return pd.DataFrame(rows)

missingness_summary_table = _make_missingness_summary()

_numerical_pathology_rows = []
for _name in [
    "dropouts_below_decomposition_table", "dynamic_churn_decomposition_table", "dropout_composition_table",
    "dynamic_vacancy_rank_tier_models_table", "wild_cluster_bootstrap_table", "dynamic_wcb_table",
    "dynamic_clean_turnover_restriction_table", "dynamic_distance_banded_turnover_table",
    "dynamic_partial_identification_bounds_table", "dynamic_rank_tier_stress_diagnostic_table",
    "dynamic_starting_rank_adjustment_diagnostics_table", "dynamic_starting_rank_quartile_support_table",
    "binary_top10_prominence_stockout_table", "binary_top10_fitted_probability_audit_table",
]:
    _obj = globals().get(_name)
    if not isinstance(_obj, pd.DataFrame) or not len(_obj):
        continue
    _row = {"table": _name, "rows": int(len(_obj))}
    for _col in ["estimate", "se", "pvalue", "condition_number", "valid_replication_share", "invalid_replication_share"]:
        if _col in _obj.columns:
            _vals = pd.to_numeric(_obj[_col], errors="coerce")
            _row[f"{_col}_nonfinite_count"] = int((~np.isfinite(_vals)).sum())
            _row[f"{_col}_min"] = float(np.nanmin(_vals)) if _vals.notna().any() else np.nan
            _row[f"{_col}_max"] = float(np.nanmax(_vals)) if _vals.notna().any() else np.nan
    _row["interpretation_status"] = "review_interpreted_rows_only; degenerate appendix estimates are explicitly flagged and not used for the main claim"
    _numerical_pathology_rows.append(_row)

numerical_pathology_summary_table = pd.DataFrame(_numerical_pathology_rows)


def write_yaml(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(obj, f, allow_unicode=True, sort_keys=False)

audit_checks_csv = OUTPUT_DIR / "econometrics_audit_checks.csv"
audit_reconciliation_csv = OUTPUT_DIR / "econometrics_audit_reconciliation.csv"
identification_csv = OUTPUT_DIR / "identification_diagnostics.csv"
balance_csv = OUTPUT_DIR / "balance_table_by_fba.csv"
fast_delivery_imputation_audit_csv = OUTPUT_DIR / "fast_delivery_imputation_audit.csv"
fast_delivery_premium_benchmark_csv = OUTPUT_DIR / "fast_delivery_premium_benchmark.csv"

balance_tests_csv = OUTPUT_DIR / "balance_hypothesis_tests_by_fba.csv"
delivery_tail_sensitivity_csv = OUTPUT_DIR / "delivery_tail_sensitivity_rankpct.csv"
price_measure_ols_comparison_csv = OUTPUT_DIR / "price_measure_ols_comparison_rankpct.csv"
propensity_overlap_interpretation_csv = OUTPUT_DIR / "propensity_overlap_interpretation_rankpct.csv"
standardized_feature_csv = OUTPUT_DIR / "standardized_feature_association_rankpct.csv"
feature_correlation_csv = OUTPUT_DIR / "feature_correlation_market_demeaned_rankpct.csv"
top_feature_correlation_pairs_csv = OUTPUT_DIR / "top_feature_correlation_pairs_rankpct.csv"
feature_vif_csv = OUTPUT_DIR / "feature_vif_market_demeaned_rankpct.csv"
feature_block_wald_csv = OUTPUT_DIR / "feature_block_wald_tests_rankpct.csv"

rank_summary_csv = OUTPUT_DIR / "rank_summary_by_fba.csv"
market_overlap_csv = OUTPUT_DIR / "market_overlap_summary.csv"
shipping_support_csv = OUTPUT_DIR / "shipping_support_by_fba.csv"
main_results_csv = OUTPUT_DIR / "main_results_market_fe_rankpct.csv"
effect_translation_csv = OUTPUT_DIR / "effect_translation_rankpct.csv"
attenuation_csv = OUTPUT_DIR / "attenuation_decomposition_rankpct.csv"
cluster_sensitivity_csv = OUTPUT_DIR / "cluster_sensitivity_rankpct.csv"
bounded_outcome_csv = OUTPUT_DIR / "bounded_outcome_diagnostics_rankpct.csv"
bounded_method_note_csv = OUTPUT_DIR / "bounded_outcome_method_note_rankpct.csv"
wild_bootstrap_csv = OUTPUT_DIR / "wild_cluster_bootstrap_rankpct.csv"
cr2_correction_csv = OUTPUT_DIR / "cr2_cluster_correction_rankpct.csv"
fractional_logit_csv = OUTPUT_DIR / "fractional_logit_rankpct.csv"
fractional_logit_ape_csv = OUTPUT_DIR / "fractional_logit_ape_rankpct.csv"
functional_form_csv = OUTPUT_DIR / "functional_form_diagnostics_rankpct.csv"
spline_sensitivity_csv = OUTPUT_DIR / "spline_functional_form_sensitivity_rankpct.csv"
extended_controls_csv = OUTPUT_DIR / "extended_control_sensitivity_rankpct.csv"
fast_delivery_premium_sensitivity_csv = OUTPUT_DIR / "fast_delivery_premium_sensitivity_rankpct.csv"
oster_sensitivity_grid_csv = OUTPUT_DIR / "thesis_oster_sensitivity_grid.csv"
functional_form_fragility_csv = OUTPUT_DIR / "functional_form_fragility_summary_rankpct.csv"
market_equal_weighted_csv = OUTPUT_DIR / "market_equal_weighted_rankpct.csv"
observation_influence_summary_csv = OUTPUT_DIR / "observation_influence_summary_rankpct.csv"
top_observation_influence_csv = OUTPUT_DIR / "top_observation_influence_rankpct.csv"
influence_trimmed_csv = OUTPUT_DIR / "influence_trimmed_rankpct.csv"
propensity_overlap_summary_csv = OUTPUT_DIR / "propensity_overlap_summary_rankpct.csv"
propensity_overlap_quantiles_csv = OUTPUT_DIR / "propensity_overlap_quantiles_rankpct.csv"
overlap_trimmed_csv = OUTPUT_DIR / "overlap_trimmed_estimates_rankpct.csv"
overlap_balance_summary_csv = OUTPUT_DIR / "overlap_balance_summary_rankpct.csv"
overlap_balance_tests_csv = OUTPUT_DIR / "overlap_balance_tests_rankpct.csv"
common_support_balance_total_price_csv = OUTPUT_DIR / "common_support_balance_total_price_rankpct.csv"
common_support_nonfba_survivor_csv = OUTPUT_DIR / "common_support_nonfba_survivor_profile_rankpct.csv"
common_support_residual_gap_csv = OUTPUT_DIR / "common_support_residual_gap_rankpct.csv"
dynamic_vacancy_design_summary_csv = OUTPUT_DIR / "dynamic_vacancy_design_summary_rankpct.csv"
dynamic_within_seller_sample_summary_csv = OUTPUT_DIR / "dynamic_within_seller_sample_summary_rankpct.csv"
dynamic_within_seller_variation_csv = OUTPUT_DIR / "dynamic_within_seller_variation_rankpct.csv"
dynamic_consecutive_transition_variation_csv = OUTPUT_DIR / "dynamic_consecutive_transition_variation_rankpct.csv"
dynamic_vacancy_identification_logic_csv = OUTPUT_DIR / "dynamic_vacancy_identification_logic_rankpct.csv"
dynamic_vacancy_transition_summary_csv = OUTPUT_DIR / "dynamic_vacancy_transition_summary_rankpct.csv"
dynamic_vacancy_transition_diagnostics_csv = OUTPUT_DIR / "dynamic_vacancy_transition_diagnostics_rankpct.csv"
dynamic_vacancy_exposure_distribution_csv = OUTPUT_DIR / "dynamic_vacancy_exposure_distribution_rankpct.csv"
dynamic_vacancy_descriptive_csv = OUTPUT_DIR / "dynamic_vacancy_descriptive_rankpct.csv"
dynamic_vacancy_models_csv = OUTPUT_DIR / "dynamic_vacancy_models_rankpct.csv"
dynamic_vacancy_rank_tier_models_csv = OUTPUT_DIR / "dynamic_vacancy_rank_tier_models_rankpct.csv"
dynamic_vacancy_rank_tier_summary_csv = OUTPUT_DIR / "dynamic_vacancy_rank_tier_summary_rankpct.csv"
dynamic_vacancy_effect_translation_csv = OUTPUT_DIR / "dynamic_vacancy_effect_translation_rankpct.csv"
dynamic_vacancy_interpretation_csv = OUTPUT_DIR / "dynamic_vacancy_interpretation_rankpct.csv"
delivery_support_csv = OUTPUT_DIR / "delivery_support_rankpct.csv"
interaction_support_csv = OUTPUT_DIR / "interaction_support_rankpct.csv"
interaction_interpretation_csv = OUTPUT_DIR / "interaction_interpretation_rankpct.csv"
sample_sensitivity_csv = OUTPUT_DIR / "sample_sensitivity_rankpct.csv"
temporal_stability_csv = OUTPUT_DIR / "temporal_stability_rankpct.csv"
mundlak_csv = OUTPUT_DIR / "mundlak_rankpct.csv"
influence_csv = OUTPUT_DIR / "leave_one_cluster_influence_rankpct.csv"
influence_detail_csv = OUTPUT_DIR / "leave_one_cluster_influence_detail_rankpct.csv"
ovb_sensitivity_csv = OUTPUT_DIR / "omitted_variable_sensitivity_rankpct.csv"
excluded_interactions_csv = OUTPUT_DIR / "excluded_interactions_rankpct.csv"
interaction_terms_csv = OUTPUT_DIR / "interaction_terms_rankpct.csv"
interaction_margins_csv = OUTPUT_DIR / "interaction_margins_rankpct.csv"
interpretation_csv = OUTPUT_DIR / "interpretation_table.csv"
sample_preview_csv = OUTPUT_DIR / "final_sample_preview.csv"
missingness_summary_csv = OUTPUT_DIR / "missingness_summary.csv"
attrition_csv = OUTPUT_DIR / "attrition_table.csv"
wild_bootstrap_sensitivity_csv = OUTPUT_DIR / "wild_cluster_bootstrap_sensitivity_rankpct.csv"
dynamic_churn_decomposition_csv = OUTPUT_DIR / "dynamic_churn_decomposition_rankpct.csv"
dynamic_churn_decomposition_summary_csv = OUTPUT_DIR / "dynamic_churn_decomposition_summary_rankpct.csv"
dynamic_fba_premium_summary_csv = OUTPUT_DIR / "dynamic_fba_premium_summary_rankpct.csv"
dynamic_fba_premium_inference_csv = OUTPUT_DIR / "dynamic_fba_premium_inference_rankpct.csv"
dynamic_clean_turnover_restriction_csv = OUTPUT_DIR / "dynamic_clean_turnover_restrictions_rankpct.csv"
dynamic_distance_banded_turnover_csv = OUTPUT_DIR / "dynamic_distance_banded_turnover_rankpct.csv"
dynamic_partial_identification_bounds_csv = OUTPUT_DIR / "dynamic_partial_identification_bounds_rankpct.csv"
dynamic_rank_tier_stress_diagnostic_csv = OUTPUT_DIR / "dynamic_rank_tier_stress_diagnostics_rankpct.csv"
dynamic_starting_rank_adjustment_diagnostics_csv = OUTPUT_DIR / "dynamic_starting_rank_adjustment_diagnostics_rankpct.csv"
dynamic_starting_rank_quartile_support_csv = OUTPUT_DIR / "dynamic_starting_rank_quartile_support_rankpct.csv"
dynamic_hardening_summary_csv = OUTPUT_DIR / "dynamic_hardening_summary_rankpct.csv"
numerical_pathology_summary_csv = OUTPUT_DIR / "numerical_pathology_summary.csv"
specs_yaml = OUTPUT_DIR / "econometric_model_specs.yaml"
manifest_yaml = OUTPUT_DIR / "econometrics_manifest.yaml"
chen_tsai_mapping_csv = OUTPUT_DIR / "chen_tsai_benchmark_mapping.csv"
dynamic_cr2_csv = OUTPUT_DIR / "dynamic_cr2_inference_rankpct.csv"
dynamic_wcb_csv = OUTPUT_DIR / "dynamic_wild_cluster_bootstrap_rankpct.csv"
dynamic_directional_reference_cr2_csv = OUTPUT_DIR / "dynamic_directional_reference_cr2_reference_rankpct.csv"
dynamic_directional_reference_wcb_csv = OUTPUT_DIR / "dynamic_directional_reference_wcb_reference_rankpct.csv"
dropouts_below_decomposition_reference_csv = OUTPUT_DIR / "dynamic_dropouts_below_decomposition_reference_rankpct.csv"
directional_decomposition_decision_reference_csv = OUTPUT_DIR / "dynamic_directional_decomposition_decision_reference_rankpct.csv"
churn_direction_correlation_csv = OUTPUT_DIR / "dynamic_churn_direction_correlation_rankpct.csv"
dropout_process_csv = OUTPUT_DIR / "dynamic_dropout_process_rankpct.csv"
dropout_process_summary_csv = OUTPUT_DIR / "dynamic_dropout_process_summary_rankpct.csv"
smoothness_csv = OUTPUT_DIR / "dynamic_pre_disappearance_smoothness_rankpct.csv"
smoothness_decision_csv = OUTPUT_DIR / "dynamic_pre_disappearance_smoothness_decision_rankpct.csv"
return_pattern_csv = OUTPUT_DIR / "dynamic_return_pattern_rankpct.csv"
return_pattern_summary_csv = OUTPUT_DIR / "dynamic_return_pattern_summary_rankpct.csv"
dropout_composition_csv = OUTPUT_DIR / "dynamic_dropout_composition_rankpct.csv"
continuous_heterogeneity_csv = OUTPUT_DIR / "dynamic_continuous_heterogeneity_rankpct.csv"
continuous_marginal_csv = OUTPUT_DIR / "dynamic_continuous_marginal_effects_rankpct.csv"
economic_salience_csv = OUTPUT_DIR / "dynamic_economic_salience_rankpct.csv"
dynamic_support_mde_csv = OUTPUT_DIR / "dynamic_support_mde_audit_rankpct.csv"
dynamic_decision_quantities_csv = OUTPUT_DIR / "dynamic_decision_quantities_rankpct.csv"
dynamic_decision_scorecard_csv = OUTPUT_DIR / "dynamic_decision_scorecard_rankpct.csv"
dynamic_mechanism_validation_summary_csv = OUTPUT_DIR / "dynamic_mechanism_validation_summary_rankpct.csv"
final_claim_summary_csv = OUTPUT_DIR / "evidence_to_claim_summary.csv"
spine_table_manifest_csv = OUTPUT_DIR / "spine_table_manifest.csv"
dynamic_power_audit_csv = OUTPUT_DIR / "dynamic_power_audit_rankpct.csv"
mde_summary_csv = OUTPUT_DIR / "dynamic_mde_summary_rankpct.csv"
mde_subgroup_audit_csv = OUTPUT_DIR / "dynamic_mde_subgroup_audit_rankpct.csv"
conservative_dynamic_inference_csv = OUTPUT_DIR / "dynamic_conservative_inference_rankpct.csv"
audit_consistency_csv = OUTPUT_DIR / "audit_consistency_checks.csv"

interpretation_table = pd.DataFrame(
    [
        {
            "object": "headline_estimand",
            "interpretation": "conditional within-market association between FBA and normalized competitive position",
        },
        {
            "object": "primary_outcome",
            "interpretation": "rank_pct, where lower values indicate better ranking",
        },
        {
            "object": "headline_inference",
            "interpretation": "two-way seller-market clustered standard errors, with seller-only and market-only clustering reported as sensitivity checks",
        },
        {
            "object": "bounded_outcome_check",
            "interpretation": "bounded-outcome fitted-value diagnostics and a fractional-response methodological note are reported; no second nonlinear estimand is promoted as a replacement for the linear rank-percent estimand",
        },
        {
            "object": "finite_cluster_check",
            "interpretation": "wild cluster bootstrap is reported with a two-way seller-market design that matches the headline inference to check finite-cluster sensitivity around the headline inference",
        },
        {
            "object": "small_sample_cluster_check",
            "interpretation": "CR2-style one-way cluster corrections are reported as conservative small-sample supplements, not as replacements for the two-way headline covariance",
        },
        {
            "object": "functional_form_check",
            "interpretation": "RESET and quadratic-control diagnostics test whether the linear rank-percent result is sensitive to functional-form restrictions; Harvey-Collier is not interpreted because the high-dimensional fixed-effects design makes it ill-conditioned",
        },
        {
            "object": "overlap_check",
            "interpretation": "propensity-score overlap and trimmed-sample estimates are reported for split-price and total-price specifications; they are classified by a common-support adequacy criterion and interpreted as support diagnostics unless observed covariate balance is achieved",
        },
        {
            "object": "overlap_balance_check",
            "interpretation": "FBA and non-FBA balance tests are re-run inside every overlap sample. Standardized mean differences are emphasized over p-values because overlap samples remain large enough for small residual differences to test as statistically significant.",
        },
        {
            "object": "categorical_star_rating_check",
            "interpretation": "parallel OLS and overlap specifications enter stelle as category indicators, using 4.5 stars as the reference, to avoid imposing equal spacing across star-rating levels",
        },
        {
            "object": "positive_review_exclusion_check",
            "interpretation": "parallel OLS and overlap specifications are also estimated without valutazioni_positive to check whether results depend on including a reputation-quality proxy that is highly correlated with stelle",
        },
        {
            "object": "dynamic_vacancy_mechanism",
            "interpretation": "the final dynamic mechanism estimates whether FBA sellers move upward more than non-FBA sellers during seller-turnover transitions, using total turnover as the core premium exposure",
        },
        {
            "object": "dynamic_within_seller_variation_check",
            "interpretation": "within-seller diagnostics show that rank outcomes and turnover exposure vary over time, while FBA status does not; the dynamic design therefore estimates an FBA-by-turnover premium, not within-seller switching into FBA",
        },
        {
            "object": "dynamic_identification_gain",
            "interpretation": "the primary dynamic premium model includes seller fixed effects and transition fixed effects, so it is less exposed to time-invariant seller-selection bias than the static level regressions",
        },
        {
            "object": "dynamic_design_caveat",
            "interpretation": "seller disappearance is observed disappearance from the scraped offer list rather than verified random stock-out, so the dynamic design is a stronger observational mechanism test but outside definitive causal identification",
        },
        {
            "object": "not_identified",
            "interpretation": "a seller fixed-effects FBA effect is not identified because the final seller-level panel has no within-seller FBA switching",
        },
        {
            "object": "control_interpretation",
            "interpretation": "rich controls absorb commercial and logistics channels, so the primary full specification estimates a residual conditional association rather than a causal total effect",
        },
        {
            "object": "interaction_role",
            "interpretation": "interaction models describe heterogeneity in the conditional FBA association across observed offer characteristics, not causal moderation",
        },
    ]
)
display(interpretation_table)

manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "product_label": PRODUCT_LABEL,
    "source_file": str(file_path),
    "source_sha256": file_sha256,
    "final_rows": int(len(df_final)),
    "markets": int(df_final["market_id"].nunique()),
    "sellers": int(df_final["seller_id"].nunique()),
    "outcome": CONTINUOUS_OUTCOME,
    "headline_specification": HEADLINE_SPEC,
    "headline_inference": HEADLINE_INFERENCE,
    "machine_learning_residualization_included": False,
    "equal_market_size_sample_modification_included": False,
    "binary_outcomes_primary_inference_included": bool(BINARY_OUTCOMES_PRIMARY_INFERENCE_INCLUDED),
    "binary_outcomes_auxiliary_salience_included": bool(BINARY_OUTCOMES_AUXILIARY_SALIENCE_INCLUDED),
    "binary_top10_complementary_estimand_included": bool(BINARY_TOP10_COMPLEMENTARY_ESTIMAND_INCLUDED),
    "binary_top10_prominence_threshold": int(BINARY_TOP10_PROMINENCE_THRESHOLD),
    "two_way_wild_bootstrap_included": True,
    "cr2_cluster_correction_included": True,
    "observation_level_influence_included": True,
    "extended_logistics_controls_included": True,
    "fast_delivery_premium_eur": float(FAST_DELIVERY_PREMIUM_EUR),
    "fast_delivery_cost_imputation_included": True,
    "fast_delivery_cost_source": FAST_DELIVERY_PREMIUM_SOURCE,
    "fast_delivery_premium_label": FAST_DELIVERY_PREMIUM_LABEL,
    "fast_delivery_sensitivity_grid_eur": {k: float(v) for k, v in FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR.items()},
    "fast_delivery_premium_sensitivity_estimated": "fast_delivery_premium_sensitivity_table" in globals(),
    "oster_multi_rmax_grid_included": "oster_sensitivity_grid_table" in globals(),
    "functional_form_fragility_summary_included": "functional_form_fragility_table" in globals(),
    "fast_delivery_cost_collinearity_rule": "fast_delivery_cost_imputed is the FBA-only monetary proxy FAST_DELIVERY_PREMIUM_EUR x fast_delivery_available x fba_from_shipper; it is not a directly observed checkout price",
    "interaction_multiple_testing_correction_included": True,
    "price_measure_parallel_checks_included": True,
    "categorical_star_rating_sensitivity_included": True,
    "positive_review_exclusion_sensitivity_included": True,
    "overlap_balance_tests_included": True,
    "dynamic_seller_list_turnover_design_included": True,
    "reference_dynamic_vacancy_tables_retained_for_decomposition": True,
    "primary_cr2_wcb_files_target_total_turnover_premium": True,
    "dynamic_fba_turnover_premium_included": True,
    "dynamic_within_seller_variation_diagnostics_included": True,
    "dynamic_rank_tier_sensitivity_included": True,
    "dynamic_reference_above_exposure_fixed_effects": "seller_id and transition_id",
    "dynamic_fba_premium_fixed_effects": "seller_id and transition_id; D2 adds lag-rank quartile dummies; D3 adds lag_rank_pct squared; D4 absorbs transition_id x lag_rank_tier",
    "dynamic_starting_rank_adjustment_hierarchy": ["D1 baseline", "D2 lag-rank quartile dummies", "D3 lag-rank quadratic", "D4 transition_id x lag_rank_tier stress test"],
    "dynamic_mde_power_audit_included": True,
    "dropouts_below_negative_special_rule_included": False,
    "below_turnover_treated_as_decomposition_not_placebo": True,
    "overlap_validity_gate_included": "overlap_design_decision_table" in globals(),
    "dynamic_future_pretrend_permutation_placebos_included": "dynamic_turnover_placebo_table" in globals(),
    "dynamic_stockout_consistency_event_table_included": "dropout_event_table" in globals(),
    "above_below_churn_correlation_included": True,
    "dynamic_churn_decomposition_included": True,
    "clean_turnover_restrictions_included": True,
    "distance_banded_turnover_included": True,
    "partial_identification_bound_included": True,
    "rank_tier_stress_reported_as_severe_stress_test": True,
    "starting_rank_quartile_adjustment_included": "dynamic_starting_rank_adjustment_diagnostics_table" in globals(),
    "starting_rank_quadratic_adjustment_included": "dynamic_starting_rank_adjustment_diagnostics_table" in globals(),
    "bootstrap_replications_audited_default": int(WILD_BOOTSTRAP_REPLICATIONS),
    "dynamic_permutation_replications_audited_default": int(DYNAMIC_PERMUTATION_REPLICATIONS),
    "binary_top10_permutation_replications_audited_default": int(BINARY_TOP10_PERMUTATION_REPLICATIONS),
    "bootstrap_replication_note": WILD_BOOTSTRAP_REPLICATION_NOTE,
    "conservative_dynamic_inference_rule": "largest sign-consistent p-value across CR2/bootstrap candidates",
    "primary_dynamic_claim": "FBA-by-total-turnover seller-list rank-movement association, validated with parsimonious starting-rank adjustments and qualified by mechanism diagnostics",
    "audit_consistency_checks_included": True,
    "chen_tsai_required_mapping_schema_included": True,
    "evidence_to_claim_summary_included": True,
    "eight_table_spine_manifest_included": True,
    "ratio_preserving_overlap_included": False,
    "exports_written": bool(EXPORT_FILES),
    "output_dir": str(OUTPUT_DIR),
    "documentation_alignment_revision": "final source comments, markdown, limitations, and audit wording aligned to the FBA-only logistics-value-adjusted specification",
    "output_schemas_include_starting_rank_adjustment": True,
    "documentation_alignment_zip_sha256_prefix": None,
    "documentation_alignment_file_count": None,
}

model_specs_payload = {
    "outcome": CONTINUOUS_OUTCOME,
    "lower_values_mean": "better competitive position",
    "specifications": SPECIFICATIONS,
    "headline_specification": HEADLINE_SPEC,
    "headline_inference": HEADLINE_INFERENCE,
    "blocked_design_choices": {
        "machine_learning_residualization": "excluded from this econometric notebook by design",
        "equal_market_size_sample_modification": "excluded because it would change the target sample composition",
        "binary_outcomes": "continuous rank_pct remains primary; entered_top_10 is included as a complementary binary-prominence estimand; other binary outcomes remain auxiliary salience screens",
        "seller_fixed_effects": "not identified for FBA because final within-seller FBA switching equals zero",
        "shipping_interaction": "excluded because FBA offers have zero shipping price support in the final sample",
        "estimator_zoo": "not expanded into Plackett-Luce, GEE, or ordered beta because those estimators would change the estimand rather than directly stress-test the headline linear rank-percent association",
    },
    "fast_delivery_cost_imputation": {
        "premium_eur": float(FAST_DELIVERY_PREMIUM_EUR),
        "premium_label": FAST_DELIVERY_PREMIUM_LABEL,
        "sensitivity_grid_eur": {k: float(v) for k, v in FAST_DELIVERY_PREMIUM_SENSITIVITY_GRID_EUR.items()},
        "variable": "fast_delivery_cost_imputed",
        "definition": "FAST_DELIVERY_PREMIUM_EUR multiplied by fast_delivery_available and fba_from_shipper",
        "source_role": FAST_DELIVERY_PREMIUM_SOURCE,
        "collinearity_rule": "Do not interpret fast_delivery_cost_imputed as a directly observed checkout price; it is an FBA-only logistics-value proxy.",
        "baseline_rule": "EUR 4.99 is retained as the baseline Premium fast-delivery benchmark; EUR 3.99 and EUR 8.99 are sensitivity translations only.",
    },
    "extended_control_specifications": EXTENDED_CONTROL_SPECIFICATIONS,
    "price_measure_specifications": PRICE_MEASURE_SPECIFICATIONS,
    "price_star_specifications": PRICE_STAR_SPECIFICATIONS,
    "price_star_metadata": PRICE_STAR_METADATA,
    "dynamic_vacancy_design": {
        "outcome": "rank_pct_improvement",
        "outcome_definition": "lagged rank_pct minus current rank_pct; positive values indicate upward movement",
        "reference_exposure": "dropouts_above",
        "reference_interaction": "fba_from_shipper times dropouts_above",
        "fixed_effects": ["seller_id", "transition_id"],
        "within_seller_diagnostic": "rank outcomes and turnover exposure vary within seller; fba_from_shipper does not, so the identified dynamic term is the interaction between FBA status and time-varying turnover exposure",
        "inference": "two-way clustering by seller and transition",
        "status": "reference above-exposure table retained for above/below decomposition; not the final primary dynamic claim",
    },
    "dynamic_fba_premium_design": {
        "outcome": "rank_pct_improvement",
        "outcome_definition": "lagged rank_pct minus current rank_pct; positive values indicate upward movement",
        "primary_exposure": "dropouts_total_focal",
        "primary_interaction": "fba_from_shipper times dropouts_total_focal",
        "directional_control": "fba_from_shipper times dropout_direction_balance",
        "fixed_effects": ["seller_id", "transition_id"],
        "rank_position_validation": {
            "D1": "seller_id and transition_id fixed effects with lag_rank_pct",
            "D2": "D1 plus additive lag-rank-quartile indicators; q1_top_25pct omitted",
            "D3": "D1 plus lag_rank_pct squared",
            "D4": "seller_id and transition_id x lag_rank_tier fixed effects as a severe stress test"
        },
        "strict_stress_fixed_effects": ["seller_id", "transition_id x lag_rank_tier"],
        "inference": "two-way clustering by seller and transition, plus CR2 and wild-cluster sensitivity for the premium coefficient",
        "status": "main dynamic premium design, with hardening diagnostics reported in the main text",
    },
    "dynamic_hardening_design": {
        "clean_turnover_restrictions": "one-sided above/no-below, below/no-above, single-sided, and mixed-turnover subsamples",
        "distance_banded_exposure": "dropouts within 1, 3, 5, and 10 lag-rank positions above or below the focal seller",
        "partial_identification_bound": "lambda_above minus lambda_below using joint covariance when available",
        "rank_tier_stress_role": "severe support-sensitive stress test; not the primary rank-position-adjusted estimator",
    },
}

if EXPORT_FILES:
    audit_checks_frame().to_csv(audit_checks_csv, index=False)
    audit_reconciliation.to_csv(audit_reconciliation_csv, index=False)
    if "attrition_table" in globals():
        attrition_table.to_csv(attrition_csv, index=False)
    missingness_summary_table.to_csv(missingness_summary_csv, index=False)
    numerical_pathology_summary_table.to_csv(numerical_pathology_summary_csv, index=False)
    identification_diagnostics.to_csv(identification_csv, index=False)
    balance_table.to_csv(balance_csv, index=False)
    if "fast_delivery_premium_benchmark_table" in globals():
        fast_delivery_premium_benchmark_table.to_csv(fast_delivery_premium_benchmark_csv, index=False)
    if "fast_delivery_imputation_audit_table" in globals():
        fast_delivery_imputation_audit_table.to_csv(fast_delivery_imputation_audit_csv, index=False)
    balance_hypothesis_tests_table.to_csv(balance_tests_csv, index=False)
    rank_summary_by_fba.to_csv(rank_summary_csv)
    market_overlap_summary.to_csv(market_overlap_csv, index=False)
    shipping_support_by_fba.to_csv(shipping_support_csv, index=False)
    main_results_table.to_csv(main_results_csv, index=False)
    price_measure_ols_comparison_table.to_csv(price_measure_ols_comparison_csv, index=False)
    effect_translation_table.to_csv(effect_translation_csv, index=False)
    attenuation_decomposition_table.to_csv(attenuation_csv, index=False)
    cluster_sensitivity_table.to_csv(cluster_sensitivity_csv, index=False)
    bounded_outcome_diagnostics.to_csv(bounded_outcome_csv, index=False)
    bounded_outcome_method_note.to_csv(bounded_method_note_csv, index=False)
    fractional_logit_table.to_csv(fractional_logit_csv, index=False)
    fractional_logit_ape_table.to_csv(fractional_logit_ape_csv, index=False)
    cr2_correction_table.to_csv(cr2_correction_csv, index=False)
    wild_cluster_bootstrap_table.to_csv(wild_bootstrap_csv, index=False)
    if "wild_cluster_bootstrap_sensitivity_table" in globals():
        wild_cluster_bootstrap_sensitivity_table.to_csv(wild_bootstrap_sensitivity_csv, index=False)
    functional_form_diagnostics_table.to_csv(functional_form_csv, index=False)
    spline_functional_form_table.to_csv(spline_sensitivity_csv, index=False)
    extended_control_sensitivity_table.to_csv(extended_controls_csv, index=False)
    if "functional_form_fragility_table" in globals():
        functional_form_fragility_table.to_csv(functional_form_fragility_csv, index=False)
    if "fast_delivery_premium_sensitivity_table" in globals():
        fast_delivery_premium_sensitivity_table.to_csv(fast_delivery_premium_sensitivity_csv, index=False)
    if "oster_sensitivity_grid_table" in globals():
        oster_sensitivity_grid_table.to_csv(oster_sensitivity_grid_csv, index=False)
    market_equal_weighted_table.to_csv(market_equal_weighted_csv, index=False)
    delivery_tail_sensitivity_table.to_csv(delivery_tail_sensitivity_csv, index=False)
    delivery_support_table.to_csv(delivery_support_csv, index=False)
    observation_influence_summary_table.to_csv(observation_influence_summary_csv, index=False)
    top_observation_influence_table.to_csv(top_observation_influence_csv, index=False)
    influence_trimmed_table.to_csv(influence_trimmed_csv, index=False)
    propensity_overlap_summary_table.to_csv(propensity_overlap_summary_csv, index=False)
    propensity_overlap_quantiles_table.to_csv(propensity_overlap_quantiles_csv, index=False)
    propensity_overlap_interpretation_table.to_csv(propensity_overlap_interpretation_csv, index=False)
    overlap_trimmed_estimates_table.to_csv(overlap_trimmed_csv, index=False)
    overlap_balance_summary_table.to_csv(overlap_balance_summary_csv, index=False)
    overlap_balance_tests_table.to_csv(overlap_balance_tests_csv, index=False)
    if "cs_balance_table" in globals():
        cs_balance_table.to_csv(common_support_balance_total_price_csv, index=False)
    if "cs_selection_table" in globals():
        cs_selection_table.to_csv(common_support_nonfba_survivor_csv, index=False)
    if "residual_gap_table" in globals():
        residual_gap_table.to_csv(common_support_residual_gap_csv, index=False)
    dynamic_vacancy_design_summary.to_csv(dynamic_vacancy_design_summary_csv, index=False)
    dynamic_within_seller_sample_summary.to_csv(dynamic_within_seller_sample_summary_csv, index=False)
    dynamic_within_seller_variation_table.to_csv(dynamic_within_seller_variation_csv, index=False)
    dynamic_consecutive_transition_variation_table.to_csv(dynamic_consecutive_transition_variation_csv, index=False)
    dynamic_vacancy_identification_logic_table.to_csv(dynamic_vacancy_identification_logic_csv, index=False)
    dynamic_vacancy_transition_summary_table.to_csv(dynamic_vacancy_transition_summary_csv, index=False)
    dynamic_vacancy_transition_diagnostics_table.to_csv(dynamic_vacancy_transition_diagnostics_csv, index=False)
    dynamic_vacancy_exposure_distribution_table.to_csv(dynamic_vacancy_exposure_distribution_csv, index=False)
    dynamic_vacancy_descriptive_table.to_csv(dynamic_vacancy_descriptive_csv, index=False)
    dynamic_vacancy_models_table.to_csv(dynamic_vacancy_models_csv, index=False)
    dynamic_vacancy_rank_tier_models_table.to_csv(dynamic_vacancy_rank_tier_models_csv, index=False)
    dynamic_vacancy_rank_tier_summary_table.to_csv(dynamic_vacancy_rank_tier_summary_csv, index=False)
    dynamic_vacancy_effect_translation_table.to_csv(dynamic_vacancy_effect_translation_csv, index=False)
    dynamic_vacancy_interpretation_table.to_csv(dynamic_vacancy_interpretation_csv, index=False)
    sample_sensitivity_table.to_csv(sample_sensitivity_csv, index=False)
    temporal_stability_table.to_csv(temporal_stability_csv, index=False)
    mundlak_rank_table.to_csv(mundlak_csv, index=False)
    influence_diagnostics_table.to_csv(influence_csv, index=False)
    if "leave_one_cluster_detail_table" in globals():
        leave_one_cluster_detail_table.to_csv(influence_detail_csv, index=False)
    omitted_variable_sensitivity_table.to_csv(ovb_sensitivity_csv, index=False)
    excluded_interactions_table.to_csv(excluded_interactions_csv, index=False)
    standardized_feature_table.to_csv(standardized_feature_csv, index=False)
    feature_correlation_table.to_csv(feature_correlation_csv, index=False)
    top_feature_correlation_pairs_table.to_csv(top_feature_correlation_pairs_csv, index=False)
    feature_vif_table.to_csv(feature_vif_csv, index=False)
    feature_block_wald_table.to_csv(feature_block_wald_csv, index=False)
    interaction_support_table.to_csv(interaction_support_csv, index=False)
    interaction_terms_table.to_csv(interaction_terms_csv, index=False)
    interaction_margins_table.to_csv(interaction_margins_csv, index=False)
    interaction_interpretation_table.to_csv(interaction_interpretation_csv, index=False)
    interpretation_table.to_csv(interpretation_csv, index=False)
    chen_tsai_mapping_table.to_csv(chen_tsai_mapping_csv, index=False)
    dynamic_cr2_table.to_csv(dynamic_cr2_csv, index=False)
    dynamic_wcb_table.to_csv(dynamic_wcb_csv, index=False)
    if "dynamic_directional_reference_cr2_table" in globals():
        dynamic_directional_reference_cr2_table.to_csv(dynamic_directional_reference_cr2_csv, index=False)
    if "dynamic_directional_reference_wcb_table" in globals():
        dynamic_directional_reference_wcb_table.to_csv(dynamic_directional_reference_wcb_csv, index=False)
    above_below_decomposition_table.to_csv(dropouts_below_decomposition_reference_csv, index=False)
    placebo_decision_table.to_csv(directional_decomposition_decision_reference_csv, index=False)
    churn_direction_correlation_table.to_csv(churn_direction_correlation_csv, index=False)
    dynamic_churn_decomposition_table.to_csv(dynamic_churn_decomposition_csv, index=False)
    dynamic_churn_decomposition_summary_table.to_csv(dynamic_churn_decomposition_summary_csv, index=False)
    dynamic_fba_premium_summary_table.to_csv(dynamic_fba_premium_summary_csv, index=False)
    dynamic_fba_premium_inference_table.to_csv(dynamic_fba_premium_inference_csv, index=False)
    dynamic_clean_turnover_restriction_table.to_csv(dynamic_clean_turnover_restriction_csv, index=False)
    dynamic_distance_banded_turnover_table.to_csv(dynamic_distance_banded_turnover_csv, index=False)
    dynamic_partial_identification_bounds_table.to_csv(dynamic_partial_identification_bounds_csv, index=False)
    dynamic_rank_tier_stress_diagnostic_table.to_csv(dynamic_rank_tier_stress_diagnostic_csv, index=False)
    if "dynamic_starting_rank_adjustment_diagnostics_table" in globals():
        dynamic_starting_rank_adjustment_diagnostics_table.to_csv(dynamic_starting_rank_adjustment_diagnostics_csv, index=False)
    if "dynamic_starting_rank_quartile_support_table" in globals():
        dynamic_starting_rank_quartile_support_table.to_csv(dynamic_starting_rank_quartile_support_csv, index=False)
    dynamic_hardening_summary_table.to_csv(dynamic_hardening_summary_csv, index=False)
    dropout_process_table.to_csv(dropout_process_csv, index=False)
    dropout_process_fit_summary_table.to_csv(dropout_process_summary_csv, index=False)
    smoothness_table.to_csv(smoothness_csv, index=False)
    smoothness_decision_table.to_csv(smoothness_decision_csv, index=False)
    return_pattern_table.to_csv(return_pattern_csv, index=False)
    return_pattern_summary_table.to_csv(return_pattern_summary_csv, index=False)
    dropout_composition_table.to_csv(dropout_composition_csv, index=False)
    continuous_heterogeneity_table.to_csv(continuous_heterogeneity_csv, index=False)
    continuous_marginal_table.to_csv(continuous_marginal_csv, index=False)
    economic_salience_table.to_csv(economic_salience_csv, index=False)
    dynamic_support_mde_table.to_csv(dynamic_support_mde_csv, index=False)
    dynamic_power_audit.to_csv(dynamic_power_audit_csv, index=False)
    mde_summary.to_csv(mde_summary_csv, index=False)
    mde_subgroup_audit.to_csv(mde_subgroup_audit_csv, index=False)
    dynamic_decision_quantities_table.to_csv(dynamic_decision_quantities_csv, index=False)
    conservative_dynamic_inference_table.to_csv(conservative_dynamic_inference_csv, index=False)
    decision_scorecard.to_csv(dynamic_decision_scorecard_csv, index=False)
    dynamic_mechanism_validation_summary_table.to_csv(dynamic_mechanism_validation_summary_csv, index=False)
    final_claim_summary.to_csv(final_claim_summary_csv, index=False)
    spine_table_manifest.to_csv(spine_table_manifest_csv, index=False)
    audit_consistency_table.to_csv(audit_consistency_csv, index=False)
    if "overlap_design_decision_table" in globals():
        overlap_design_decision_table.to_csv(OUTPUT_DIR / "overlap_design_decision_rankpct.csv", index=False)
    if "overlap_design_status_summary_table" in globals():
        overlap_design_status_summary_table.to_csv(OUTPUT_DIR / "overlap_design_status_summary_rankpct.csv", index=False)
    if "dropouts_below_decomposition_table" in globals():
        dropouts_below_decomposition_table.to_csv(OUTPUT_DIR / "dynamic_above_below_decomposition_rankpct.csv", index=False)
    if "directional_decomposition_decision_table" in globals():
        directional_decomposition_decision_table.to_csv(OUTPUT_DIR / "dynamic_directional_decomposition_decision_rankpct.csv", index=False)
    if "dynamic_turnover_placebo_table" in globals():
        dynamic_turnover_placebo_table.to_csv(OUTPUT_DIR / "dynamic_turnover_placebos_rankpct.csv", index=False)
    if "dynamic_permutation_placebo_turnover_table" in globals():
        dynamic_permutation_placebo_turnover_table.to_csv(OUTPUT_DIR / "dynamic_permutation_placebo_turnover_rankpct.csv", index=False)
    if "dynamic_turnover_placebo_decision_table" in globals():
        dynamic_turnover_placebo_decision_table.to_csv(OUTPUT_DIR / "dynamic_turnover_placebo_decision_rankpct.csv", index=False)
    if "dropout_event_table" in globals():
        dropout_event_table.to_csv(OUTPUT_DIR / "dynamic_dropout_event_table_rankpct.csv", index=False)
    if "dropout_rank_quartile_summary_table" in globals():
        dropout_rank_quartile_summary_table.to_csv(OUTPUT_DIR / "dynamic_dropout_rank_quartile_summary_rankpct.csv", index=False)
    if "dynamic_stockout_consistency_summary_table" in globals():
        dynamic_stockout_consistency_summary_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_summary_rankpct.csv", index=False)
    if "dynamic_stockout_consistency_definition_table" in globals():
        dynamic_stockout_consistency_definition_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_definitions_rankpct.csv", index=False)
    if "binary_top10_prominence_decision_table" in globals():
        binary_top10_prominence_decision_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_decision_rankpct.csv", index=False)
    if "binary_top10_prominence_inference_table" in globals():
        binary_top10_prominence_inference_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_inference_rankpct.csv", index=False)
    if "binary_top10_prominence_threshold_family_table" in globals():
        binary_top10_prominence_threshold_family_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_threshold_family_rankpct.csv", index=False)
    if "binary_top10_prominence_permutation_table" in globals():
        binary_top10_prominence_permutation_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_permutation_rankpct.csv", index=False)
    if "binary_top10_prominence_stockout_table" in globals():
        binary_top10_prominence_stockout_table.to_csv(OUTPUT_DIR / "binary_top10_prominence_stockout_consistency_rankpct.csv", index=False)
    if "dynamic_stockout_consistent_turnover_table" in globals():
        dynamic_stockout_consistent_turnover_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistent_turnover_rankpct.csv", index=False)
    if "dynamic_stockout_consistency_decision_table" in globals():
        dynamic_stockout_consistency_decision_table.to_csv(OUTPUT_DIR / "dynamic_stockout_consistency_decision_rankpct.csv", index=False)
    final_sample_preview.to_csv(sample_preview_csv, index=False)
    write_yaml(model_specs_payload, specs_yaml)
    write_yaml(manifest, manifest_yaml)

    print(f"Econometric files written to: {OUTPUT_DIR}")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print("EXPORT_FILES is False; no econometric file files were written.")

# Export logistics-value-adjusted price specification files.
try:
    if "logistics_value_adjusted_results_table" in globals():
        logistics_value_adjusted_results_table.to_csv(
            OUTPUT_DIR / "logistics_value_adjusted_price_results_rankpct.csv", index=False
        )
    if "logistics_value_adjusted_summary_table" in globals():
        logistics_value_adjusted_summary_table.to_csv(
            OUTPUT_DIR / "logistics_value_adjusted_price_summary_rankpct.csv", index=False
        )
    if "logistics_value_adjusted_audit_table" in globals():
        logistics_value_adjusted_audit_table.to_csv(
            OUTPUT_DIR / "logistics_value_adjusted_price_audit_rankpct.csv", index=False
        )
    if "logistics_value_adjusted_design_rank_table" in globals():
        logistics_value_adjusted_design_rank_table.to_csv(
            OUTPUT_DIR / "logistics_value_adjusted_price_design_rank_check_rankpct.csv", index=False
        )
except Exception as _eff_export_err:
    print(f"Logistics-value-adjusted price export warning: {_eff_export_err}")



,object,interpretation
0,headline_estimand,conditional within-market association between FBA and normalized competitive position
1,primary_outcome,"rank_pct, where lower values indicate better ranking"
2,headline_inference,"two-way seller-market clustered standard errors, with seller-only and market-only clustering reported as sensitivity checks"
3,bounded_outcome_check,bounded-outcome fitted-value diagnostics and a fractional-response methodological note are reported; no second nonlinear estimand is promoted as a replacement for the linear ra...
4,finite_cluster_check,wild cluster bootstrap is reported with a two-way seller-market design that matches the headline inference to check finite-cluster sensitivity around the headline inference
5,small_sample_cluster_check,"CR2-style one-way cluster corrections are reported as conservative small-sample supplements, not as replacements for the two-way headline covariance"
6,functional_form_check,RESET and quadratic-control diagnostics test whether the linear rank-percent result is sensitive to functional-form restrictions; Harvey-Collier is not interpreted because the ...
7,overlap_check,propensity-score overlap and trimmed-sample estimates are reported for split-price and total-price specifications; they are classified by a common-support adequacy criterion an...
8,overlap_balance_check,FBA and non-FBA balance tests are re-run inside every overlap sample. Standardized mean differences are emphasized over p-values because overlap samples remain large enough for...
9,categorical_star_rating_check,"parallel OLS and overlap specifications enter stelle as category indicators, using 4.5 stars as the reference, to avoid imposing equal spacing across star-rating levels"


Econometric files written to: /content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results
{
  "notebook_version": "final-empirical-documentation",
  "product_label": "Xiaomi Mi Smart Band 6",
  "source_file": "/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv",
  "source_sha256": "88c85f33e3e640d6261d2ac1bb0b095ebf1c65ac8fc4edab9d850cd3c8abe5b5",
  "final_rows": 5107,
  "markets": 62,
  "sellers": 119,
  "outcome": "rank_pct",
  "headline_specification": "spec_4_fba_reputation_price_shipping_delivery",
  "headline_inference": "two_way_seller_market",
  "machine_learning_residualization_included": false,
  "equal_market_size_sample_modification_included": false,
  "binary_outcomes_primary_inference_included": false,
  "binary_outcomes_auxiliary_salience_included": true,
  "binary_top10_complementary_estimand_included": true,
  "binary_top10_prominence_threshold": 10,
  "two_way_

### 41. Serial-dependence and temporal-persistence audit supplement

The supplement addresses the audit concern that repeated seller observations over time may reduce the effective information content of the panel. The cell estimates a Wooldridge first-difference residual test for the static specification and for the dynamic specification, reports the temporal-persistence ratio by seller, and writes the diagnostic to `static_wooldridge_style_residual_test_rankpct.csv`, `dynamic_wooldridge_style_residual_test_rankpct.csv`, and `temporal_persistence_by_seller_rankpct.csv`. The diagnostic is consistent with the persistence documented in Part I and populates `tab:app-ch6-finite-cluster-inference`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 41. Serial-dependence and temporal-persistence audit supplement
# -----------------------------------------------------------------------------
# Wooldridge first-difference residual test for the static and dynamic specifications, temporal-persistence ratio by seller, and the corresponding export.

from IPython.display import display


def _serial_parse_market_order(data):
    """Return a copy with a numeric market_order usable for within-seller lags."""
    out = data.copy()
    if "market_order" not in out.columns:
        if "timestamp" in out.columns:
            ts = pd.to_datetime(out["timestamp"], errors="coerce")
            order_map = {v: i for i, v in enumerate(sorted(ts.dropna().unique()))}
            out["market_order"] = ts.map(order_map).astype("float")
        elif "market_id" in out.columns:
            order_map = {v: i for i, v in enumerate(sorted(out["market_id"].dropna().astype(str).unique()))}
            out["market_order"] = out["market_id"].astype(str).map(order_map).astype("float")
        else:
            raise KeyError("A timestamp, market_id, or market_order column is required for temporal diagnostics.")
    return out


def _numeric_series(x):
    if pd.api.types.is_bool_dtype(x):
        return x.astype(float)
    return pd.to_numeric(x, errors="coerce")


def _within_seller_persistence_table(data, variables=None, id_col="seller_id", time_col="market_order"):
    """Compute lag-1 persistence by seller for observed columns.

    The table reports all adjacent within-seller pairs and, separately, pairs in
    consecutive market orders. High persistence is not treated as a failure; it
    informs the inference and causal-interpretation caveats.
    """
    d = _serial_parse_market_order(data)
    if variables is None:
        variables = [
            "rank_pct", "rank_pos", "fba_from_shipper", "prezzo", "prezzo_spedizione_repaired",
            "prezzo_totale_reconstructed", "g_cons_min_robust", "g_cons_max_robust",
            "log1p_num_valutazioni", "valutazioni_positive", "stelle",
            "fast_delivery_available", "fast_delivery_cost_imputed",
        ]
    variables = [v for v in variables if v in d.columns]
    rows = []
    base = d[[id_col, time_col] + variables].copy().sort_values([id_col, time_col], kind="mergesort")
    for var in variables:
        tmp = base[[id_col, time_col, var]].copy()
        tmp[var] = _numeric_series(tmp[var])
        tmp = tmp.dropna(subset=[id_col, time_col, var]).sort_values([id_col, time_col], kind="mergesort")
        tmp["lag_value"] = tmp.groupby(id_col, observed=True)[var].shift(1)
        tmp["lag_time"] = tmp.groupby(id_col, observed=True)[time_col].shift(1)
        tmp["time_gap"] = tmp[time_col] - tmp["lag_time"]
        paired = tmp.dropna(subset=["lag_value", "time_gap"]).copy()
        consecutive = paired.loc[np.isclose(paired["time_gap"].astype(float), 1.0)].copy()
        use = consecutive if len(consecutive) >= 10 else paired
        corr = float(use[var].corr(use["lag_value"])) if len(use) >= 3 and use[var].nunique() > 1 and use["lag_value"].nunique() > 1 else np.nan
        abs_change = np.abs(use[var].astype(float) - use["lag_value"].astype(float))
        rows.append({
            "variable": var,
            "observations_with_value": int(tmp.shape[0]),
            "seller_units": int(tmp[id_col].nunique()),
            "adjacent_lag_pairs": int(paired.shape[0]),
            "consecutive_lag_pairs": int(consecutive.shape[0]),
            "lag1_autocorrelation_used": corr,
            "pairs_used_for_corr": "consecutive_market_pairs" if len(consecutive) >= 10 else "all_adjacent_within_seller_pairs",
            "mean_abs_change": float(abs_change.mean()) if len(abs_change) else np.nan,
            "median_abs_change": float(abs_change.median()) if len(abs_change) else np.nan,
            "share_unchanged": float(np.isclose(abs_change, 0.0).mean()) if len(abs_change) else np.nan,
            "interpretation": "High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.",
        })
    return pd.DataFrame(rows)


def _residual_frame_from_statsmodels_result(res, data, id_col="seller_id"):
    """Attach statsmodels residuals to the original row labels used in estimation."""
    try:
        labels = list(res.model.data.row_labels)
        frame = data.loc[labels].copy()
    except Exception:
        frame = data.iloc[: int(len(res.resid))].copy()
    frame = _serial_parse_market_order(frame)
    frame["_model_residual"] = np.asarray(res.resid, dtype=float)
    if id_col not in frame.columns:
        raise KeyError(f"{id_col} not found in model frame.")
    return frame


def _residual_frame_from_dynamic_fit(fit, id_col="seller_id", transition_col="transition_id"):
    """Attach within-estimator residuals from the custom dynamic fit object."""
    frame = fit["model_frame"].copy()
    frame["_model_residual"] = np.asarray(fit["residuals"], dtype=float)
    if "transition_order" not in frame.columns:
        # transition_id is formatted as '00_to_01' in the dynamic panel builder.
        frame["transition_order"] = (
            frame[transition_col].astype(str).str.extract(r"^(\d+)_to_")[0].astype(float)
        )
    return frame


def _clustered_lag_residual_regression(frame, id_col="seller_id", time_col="market_order", residual_col="_model_residual", label="model"):
    """Lag-residual diagnostic: e_it on e_i,t-1 with seller-clustered inference."""
    d = frame[[id_col, time_col, residual_col]].dropna().copy().sort_values([id_col, time_col], kind="mergesort")
    d["lag_residual"] = d.groupby(id_col, observed=True)[residual_col].shift(1)
    d["lag_time"] = d.groupby(id_col, observed=True)[time_col].shift(1)
    d["time_gap"] = d[time_col] - d["lag_time"]
    pairs = d.dropna(subset=["lag_residual", "time_gap"]).copy()
    consecutive = pairs.loc[np.isclose(pairs["time_gap"].astype(float), 1.0)].copy()
    use = consecutive if len(consecutive) >= 10 else pairs
    if len(use) < 10 or use[id_col].nunique() < 2:
        return pd.DataFrame([{
            "model": label, "diagnostic": "lag_residual_regression", "status": "not_enough_pairs",
            "n_pairs": int(len(use)), "clusters": int(use[id_col].nunique()) if len(use) else 0,
            "rho_hat": np.nan, "se_clustered": np.nan, "pvalue_clustered": np.nan,
            "interpretation": "Insufficient within-seller residual pairs for a lag-residual diagnostic.",
        }])
    y = use[residual_col].astype(float).to_numpy()
    X = sm.add_constant(use["lag_residual"].astype(float).to_numpy(), has_constant="add")
    fit = sm.OLS(y, X).fit()
    try:
        V = cov_cluster(fit, use[id_col])
        se = float(np.sqrt(max(V[1, 1], 0.0)))
        dof = max(int(use[id_col].nunique()) - 1, 1)
        tval = float(fit.params[1] / se) if se > 0 else np.nan
        pval = float(2 * stats.t.sf(abs(tval), dof)) if pd.notna(tval) else np.nan
        inference = "seller_clustered"
    except Exception as exc:
        se = float(fit.bse[1])
        tval = float(fit.tvalues[1])
        pval = float(fit.pvalues[1])
        inference = f"ols_fallback_after_cluster_failure: {exc}"
    return pd.DataFrame([{
        "model": label,
        "diagnostic": "lag_residual_regression",
        "status": "estimated",
        "pair_rule": "consecutive_market_pairs" if len(consecutive) >= 10 else "all_adjacent_within_seller_pairs",
        "n_pairs": int(len(use)),
        "clusters": int(use[id_col].nunique()),
        "rho_hat": float(fit.params[1]),
        "se_clustered": se,
        "t_statistic": tval,
        "pvalue_clustered": pval,
        "inference": inference,
        "interpretation": "Significant residual persistence would invalidate iid standard errors; it supports the thesis choice to use clustered/bootstrapped inference.",
    }])


def _wooldridge_style_fd_residual_test(frame, id_col="seller_id", time_col="market_order", residual_col="_model_residual", label="model"):
    """Wooldridge-style residual first-difference test.

    This is reported as a diagnostic rather than as the estimand. It tests whether
    the coefficient in Δe_it on Δe_i,t-1 equals -0.5, the no-serial-correlation
    restriction used in the common Wooldridge/Drukker panel diagnostic.
    """
    d = frame[[id_col, time_col, residual_col]].dropna().copy().sort_values([id_col, time_col], kind="mergesort")
    d["lag_residual"] = d.groupby(id_col, observed=True)[residual_col].shift(1)
    d["lag_time"] = d.groupby(id_col, observed=True)[time_col].shift(1)
    d["time_gap"] = d[time_col] - d["lag_time"]
    d["d_residual"] = d[residual_col] - d["lag_residual"]
    d.loc[~np.isclose(d["time_gap"].fillna(np.nan).astype(float), 1.0), "d_residual"] = np.nan
    d["lag_d_residual"] = d.groupby(id_col, observed=True)["d_residual"].shift(1)
    d["lag_d_time_gap"] = d.groupby(id_col, observed=True)["time_gap"].shift(1)
    use = d.dropna(subset=["d_residual", "lag_d_residual", "lag_d_time_gap"]).copy()
    use = use.loc[np.isclose(use["lag_d_time_gap"].astype(float), 1.0)].copy()
    if len(use) < 10 or use[id_col].nunique() < 2:
        return pd.DataFrame([{
            "model": label, "diagnostic": "wooldridge_style_fd_residual_test", "status": "not_enough_pairs",
            "n_pairs": int(len(use)), "clusters": int(use[id_col].nunique()) if len(use) else 0,
            "coefficient_on_lagged_difference": np.nan, "se_clustered": np.nan,
            "test_null": "coef = -0.5", "pvalue_clustered": np.nan,
            "interpretation": "Insufficient consecutive within-seller residual-difference pairs for the diagnostic.",
        }])
    y = use["d_residual"].astype(float).to_numpy()
    X = use[["lag_d_residual"]].astype(float).to_numpy()  # no intercept by construction
    fit = sm.OLS(y, X).fit()
    try:
        V = cov_cluster(fit, use[id_col])
        se = float(np.sqrt(max(V[0, 0], 0.0)))
        dof = max(int(use[id_col].nunique()) - 1, 1)
        tval = float((fit.params[0] + 0.5) / se) if se > 0 else np.nan
        pval = float(2 * stats.t.sf(abs(tval), dof)) if pd.notna(tval) else np.nan
        inference = "seller_clustered"
    except Exception as exc:
        se = float(fit.bse[0])
        tval = float((fit.params[0] + 0.5) / se) if se > 0 else np.nan
        pval = float(2 * stats.t.sf(abs(tval), max(int(use[id_col].nunique()) - 1, 1))) if pd.notna(tval) else np.nan
        inference = f"ols_fallback_after_cluster_failure: {exc}"
    return pd.DataFrame([{
        "model": label,
        "diagnostic": "wooldridge_style_fd_residual_test",
        "status": "estimated",
        "n_pairs": int(len(use)),
        "clusters": int(use[id_col].nunique()),
        "coefficient_on_lagged_difference": float(fit.params[0]),
        "se_clustered": se,
        "test_null": "coef = -0.5",
        "t_statistic_for_null": tval,
        "pvalue_clustered": pval,
        "inference": inference,
        "interpretation": "Rejection indicates residual serial correlation; the appropriate response is cluster/block/wild-cluster inference plus a cautious observational interpretation.",
    }])


def _serial_decision_table(static_lag, static_wool, dynamic_lag=None, dynamic_wool=None):
    rows = []
    def _row_from(tbl, label):
        if not isinstance(tbl, pd.DataFrame) or tbl.empty:
            return None
        r = tbl.iloc[0]
        p = pd.to_numeric(pd.Series([r.get("pvalue_clustered", np.nan)]), errors="coerce").iloc[0]
        significant = bool(pd.notna(p) and p < 0.05)
        return {
            "evidence_block": label,
            "diagnostic_status": r.get("status", "unknown"),
            "pvalue": p,
            "serial_dependence_flag_5pct": significant,
            "implication_for_estimation": (
                "Do not use iid/HC-only inference; rely on clustered, CR2, and wild-cluster inference."
                if significant else
                "No strong diagnostic rejection at 5%, but repeated observations still justify clustered inference by design."
            ),
        }
    for tbl, label in [
        (static_lag, "static headline residual AR(1)"),
        (static_wool, "static Wooldridge-style FD residual test"),
        (dynamic_lag, "dynamic preferred residual AR(1)"),
        (dynamic_wool, "dynamic Wooldridge-style FD residual test"),
    ]:
        row = _row_from(tbl, label)
        if row is not None:
            rows.append(row)
    rows.append({
        "evidence_block": "overall interpretation rule",
        "diagnostic_status": "interpretive rule",
        "pvalue": np.nan,
        "serial_dependence_flag_5pct": np.nan,
        "implication_for_estimation": (
            "Temporal persistence is not a falsification test for the coefficient. It mainly affects inference and reinforces the non-causal interpretation when persistence reflects unobserved seller quality."
        ),
    })
    return pd.DataFrame(rows)


# A. Persistence in observed columns.
temporal_persistence_by_seller_table = _within_seller_persistence_table(df_final)
print("Temporal persistence in observed seller-level columns")
display(temporal_persistence_by_seller_table)

# B. Static headline residual serial-dependence diagnostics.
_static_residual_frame = _residual_frame_from_statsmodels_result(headline_plain_res, df_final)
static_residual_lag_diagnostic_table = _clustered_lag_residual_regression(
    _static_residual_frame,
    id_col="seller_id",
    time_col="market_order",
    label="static_headline_spec_4",
)
static_wooldridge_style_residual_test_table = _wooldridge_style_fd_residual_test(
    _static_residual_frame,
    id_col="seller_id",
    time_col="market_order",
    label="static_headline_spec_4",
)
print("Static residual serial-dependence diagnostics")
display(static_residual_lag_diagnostic_table)
display(static_wooldridge_style_residual_test_table)

# C. Dynamic residual diagnostics when the preferred dynamic fit is available.
_dynamic_fit_for_serial = globals().get("_churn_total_direction_fit", globals().get("_churn_total_fit", None))
if isinstance(_dynamic_fit_for_serial, dict) and "residuals" in _dynamic_fit_for_serial and "model_frame" in _dynamic_fit_for_serial:
    _dynamic_residual_frame = _residual_frame_from_dynamic_fit(_dynamic_fit_for_serial)
    dynamic_residual_lag_diagnostic_table = _clustered_lag_residual_regression(
        _dynamic_residual_frame,
        id_col="seller_id",
        time_col="transition_order",
        label="dynamic_total_turnover_preferred_available_fit",
    )
    dynamic_wooldridge_style_residual_test_table = _wooldridge_style_fd_residual_test(
        _dynamic_residual_frame,
        id_col="seller_id",
        time_col="transition_order",
        label="dynamic_total_turnover_preferred_available_fit",
    )
else:
    dynamic_residual_lag_diagnostic_table = pd.DataFrame([{
        "model": "dynamic_total_turnover_preferred_available_fit",
        "diagnostic": "lag_residual_regression",
        "status": "dynamic_fit_not_available",
        "interpretation": "Run the dynamic turnover-premium section before this supplement to compute dynamic residual diagnostics.",
    }])
    dynamic_wooldridge_style_residual_test_table = pd.DataFrame([{
        "model": "dynamic_total_turnover_preferred_available_fit",
        "diagnostic": "wooldridge_style_fd_residual_test",
        "status": "dynamic_fit_not_available",
        "interpretation": "Run the dynamic turnover-premium section before this supplement to compute dynamic residual diagnostics.",
    }])
print("Dynamic residual serial-dependence diagnostics")
display(dynamic_residual_lag_diagnostic_table)
display(dynamic_wooldridge_style_residual_test_table)

# D. Thesis-facing implications.
serial_dependence_claim_implications_table = _serial_decision_table(
    static_residual_lag_diagnostic_table,
    static_wooldridge_style_residual_test_table,
    dynamic_residual_lag_diagnostic_table,
    dynamic_wooldridge_style_residual_test_table,
)
serial_dependence_method_note_table = pd.DataFrame([
    {
        "issue": "observed-column persistence",
        "methodological_position": "Expected in repeated seller panels; not a coefficient-falsification test.",
        "thesis_response": "Report temporal persistence and interpret raw row counts as repeated observations rather than independent draws.",
    },
    {
        "issue": "residual serial correlation",
        "methodological_position": "Can invalidate iid or HC-only standard errors and produce over-rejection.",
        "thesis_response": "Use seller/market two-way clustered inference, CR2-style checks, and wild-cluster bootstrap as primary inferential safeguards.",
    },
    {
        "issue": "persistent unobserved seller heterogeneity",
        "methodological_position": "Can bias an observational FBA coefficient if correlated with FBA and ranking outcomes.",
        "thesis_response": "Treat static estimates as conditional associations; use the dynamic seller-FE turnover design as a stronger but still observational mechanism test.",
    },
])
print("Serial-dependence implications for thesis claims")
display(serial_dependence_claim_implications_table)
display(serial_dependence_method_note_table)

# E. Optional exports for appendix / audit replication.
try:
    _serial_output_dir = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path("Econometrics-Results")
    _serial_output_dir.mkdir(parents=True, exist_ok=True)
    temporal_persistence_by_seller_table.to_csv(_serial_output_dir / "temporal_persistence_by_seller_rankpct.csv", index=False)
    static_residual_lag_diagnostic_table.to_csv(_serial_output_dir / "static_residual_lag_diagnostic_rankpct.csv", index=False)
    static_wooldridge_style_residual_test_table.to_csv(_serial_output_dir / "static_wooldridge_style_residual_test_rankpct.csv", index=False)
    dynamic_residual_lag_diagnostic_table.to_csv(_serial_output_dir / "dynamic_residual_lag_diagnostic_rankpct.csv", index=False)
    dynamic_wooldridge_style_residual_test_table.to_csv(_serial_output_dir / "dynamic_wooldridge_style_residual_test_rankpct.csv", index=False)
    serial_dependence_claim_implications_table.to_csv(_serial_output_dir / "serial_dependence_claim_implications_rankpct.csv", index=False)
    serial_dependence_method_note_table.to_csv(_serial_output_dir / "serial_dependence_method_note_rankpct.csv", index=False)
    print(f"Serial-dependence audit tables written to: {_serial_output_dir}")
except Exception as _serial_export_error:
    print(f"Serial-dependence audit export warning: {_serial_export_error}")


Temporal persistence in observed seller-level columns


,variable,observations_with_value,seller_units,adjacent_lag_pairs,consecutive_lag_pairs,lag1_autocorrelation_used,pairs_used_for_corr,mean_abs_change,median_abs_change,share_unchanged,interpretation
0,rank_pct,5107,119,4988,4958,0.994572,consecutive_market_pairs,0.011560,0.005868,0.214804,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
1,rank_pos,5107,119,4988,4958,0.994383,consecutive_market_pairs,1.067971,1.000000,0.419322,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
2,fba_from_shipper,5107,119,4988,4958,1.000000,consecutive_market_pairs,0.000000,0.000000,1.000000,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
3,prezzo,5107,119,4988,4958,0.993645,consecutive_market_pairs,0.148961,0.000000,0.941912,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
4,prezzo_spedizione_repaired,5107,119,4988,4958,0.997127,consecutive_market_pairs,0.015867,0.000000,0.989714,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
5,prezzo_totale_reconstructed,5107,119,4988,4958,0.994267,consecutive_market_pairs,0.164816,0.000000,0.932231,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
6,g_cons_min_robust,5107,119,4988,4958,0.949752,consecutive_market_pairs,0.502622,0.000000,0.685760,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
7,g_cons_max_robust,5107,119,4988,4958,0.977106,consecutive_market_pairs,0.532069,0.000000,0.681121,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
8,log1p_num_valutazioni,5107,119,4988,4958,0.999978,consecutive_market_pairs,0.001322,0.000000,0.818879,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.
9,valutazioni_positive,5107,119,4988,4958,0.994375,consecutive_market_pairs,0.121218,0.000000,0.957241,High persistence affects effective information and motivates clustered/bootstrapped inference; it is not itself evidence of a biased coefficient.


Static residual serial-dependence diagnostics


,model,diagnostic,status,pair_rule,n_pairs,clusters,rho_hat,se_clustered,t_statistic,pvalue_clustered,inference,interpretation
0,static_headline_spec_4,lag_residual_regression,estimated,consecutive_market_pairs,4958,114,0.970036,0.009457,102.570123,2.334230e-113,seller_clustered,Significant residual persistence would invalidate iid standard errors; it supports the thesis choice to use clustered/bootstrapped inference.


,model,diagnostic,status,n_pairs,clusters,coefficient_on_lagged_difference,se_clustered,test_null,t_statistic_for_null,pvalue_clustered,inference,interpretation
0,static_headline_spec_4,wooldridge_style_fd_residual_test,estimated,4815,110,-0.346897,0.075563,coef = -0.5,2.026175,0.045189,seller_clustered,Rejection indicates residual serial correlation; the appropriate response is cluster/block/wild-cluster inference plus a cautious observational interpretation.


...

,model,diagnostic,status,n_pairs,clusters,coefficient_on_lagged_difference,se_clustered,test_null,t_statistic_for_null,pvalue_clustered,inference,interpretation
0,dynamic_total_turnover_preferred_available_fit,wooldridge_style_fd_residual_test,estimated,4680,107,-0.778011,0.085371,coef = -0.5,-3.256489,0.001515,seller_clustered,Rejection indicates residual serial correlation; the appropriate response is cluster/block/wild-cluster inference plus a cautious observational interpretation.


Serial-dependence implications for thesis claims


,evidence_block,diagnostic_status,pvalue,serial_dependence_flag_5pct,implication_for_estimation
0,static headline residual AR(1),estimated,2.334230e-113,True,"Do not use iid/HC-only inference; rely on clustered, CR2, and wild-cluster inference."
1,static Wooldridge-style FD residual test,estimated,4.518891e-02,True,"Do not use iid/HC-only inference; rely on clustered, CR2, and wild-cluster inference."
2,dynamic preferred residual AR(1),estimated,2.309635e-01,False,"No strong diagnostic rejection at 5%, but repeated observations still justify clustered inference by design."
3,dynamic Wooldridge-style FD residual test,estimated,1.515242e-03,True,"Do not use iid/HC-only inference; rely on clustered, CR2, and wild-cluster inference."
4,overall interpretation rule,interpretive rule,NaN,NaN,Temporal persistence is not a falsification test for the coefficient. It mainly affects inference and reinforces the non-causal interpretation when persistence reflects unobser...


,issue,methodological_position,thesis_response
0,observed-column persistence,Expected in repeated seller panels; not a coefficient-falsification test.,Report temporal persistence and interpret raw row counts as repeated observations rather than independent draws.
1,residual serial correlation,Can invalidate iid or HC-only standard errors and produce over-rejection.,"Use seller/market two-way clustered inference, CR2-style checks, and wild-cluster bootstrap as primary inferential safeguards."
2,persistent unobserved seller heterogeneity,Can bias an observational FBA coefficient if correlated with FBA and ranking outcomes.,Treat static estimates as conditional associations; use the dynamic seller-FE turnover design as a stronger but still observational mechanism test.


Serial-dependence audit tables written to: /content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results


### 42. Thesis text insertion: temporal dependence and inference

The block contains the interpretive paragraph that the thesis methods chapter inserts after the serial-dependence audit. The paragraph is recorded inside the notebook so that any future change in the audit numbers can be reflected in the thesis prose without losing the documentation chain.

> Because the same sellers are observed repeatedly across market snapshots, residual dependence operates along the seller dimension as well as along the market dimension. Two-way seller-market clustering addresses this dependence in the static specification; two-way seller-transition clustering addresses the analogous dependence in the dynamic specification. The CR2 finite-cluster correction and the wild cluster bootstrap supplement the asymptotic clustered inference, since the seller dimension has 119 finite clusters. The first-difference residual test of Wooldridge (2010) is consistent with the persistence documented in the data audit and does not invalidate the clustered standard errors used in the headline tables.


### 43. Table-role manifest

The table-role manifest classifies each exported CSV by analytical function. The categories are: main-result tables (S1–S4 headline static, D1–D4 headline dynamic), robustness tables (clustering, CR2, wild bootstrap, sample sensitivity, extended controls), support diagnostics (propensity overlap, balance, missingness), mechanism diagnostics (above/below decomposition, distance-banded turnover, dropout composition, pre-disappearance smoothness, stockout consistency), synthesis tables (evidence-to-claim, core spine, decision scorecard), and audit infrastructure (audit registry, manifest, attrition). The manifest is exported to `econometrics_table_manifest.csv`.


In [ ]:
# -----------------------------------------------------------------------------
# Section 43. Table-role manifest
# -----------------------------------------------------------------------------
# Classify each exported CSV by analytical function and write the role manifest to econometrics_table_manifest.csv.

def _classify_table_role(filename: str) -> tuple:
    """Classify CSV outputs by thesis-facing role.

    The rule is deliberately parsimonious. Main files are limited to core
    estimands, required validity gates, compact inference summaries, and
    claim-facing synthesis. Diagnostic-detail, influence-detail, support-grid,
    stockout-variant, top-k-threshold, and numerical-audit files are appendix or
    audit-only so the output package does not overstate the evidence hierarchy.
    """
    fn = filename.lower()

    audit_exact = {
        "econometrics_manifest.yaml",
        "econometric_model_specs.yaml",
        "econometrics_table_manifest.csv",
        "figure_manifest.csv",
        "synthesis_manifest.csv",
        "audit_consistency_checks.csv",
        "econometrics_audit_checks.csv",
        "numerical_pathology_summary.csv",
        "final_sample_preview.csv",
        "missingness_summary.csv",
        "spine_table_manifest.csv",
    }
    if fn in audit_exact or "table_manifest" in fn or "figure_manifest" in fn or "synthesis_manifest" in fn:
        return ("audit_only", "manifest_or_reproducibility_audit")

    if any(k in fn for k in (
        "serial_dependence", "temporal_persistence_by_seller",
        "residual_lag_diagnostic", "wooldridge_style_residual_test",
    )):
        return ("appendix", "serial_dependence_inference_audit")

    main_exact = {
        # Sample/provenance and claim-facing synthesis.
        "attrition_table.csv",
        "identification_diagnostics.csv",
        "econometrics_audit_reconciliation.csv",
        "interpretation_table.csv",
        "evidence_to_claim_summary.csv",
        # Static core and required static validity gates.
        "main_results_market_fe_rankpct.csv",
        "attenuation_decomposition_rankpct.csv",
        "functional_form_fragility_summary_rankpct.csv",
        "bounded_outcome_diagnostics_rankpct.csv",
        "fractional_logit_ape_rankpct.csv",
        "cluster_sensitivity_rankpct.csv",
        "cr2_cluster_correction_rankpct.csv",
        "wild_cluster_bootstrap_rankpct.csv",
        "overlap_design_status_summary_rankpct.csv",
        "common_support_residual_gap_rankpct.csv",
        # Dynamic core, rank-position validation, and compact inference.
        "dynamic_fba_premium_inference_rankpct.csv",
        "dynamic_fba_premium_summary_rankpct.csv",
        "dynamic_starting_rank_adjustment_diagnostics_rankpct.csv",
        "dynamic_rank_tier_stress_diagnostics_rankpct.csv",
        "dynamic_mde_summary_rankpct.csv",
        "dynamic_turnover_placebo_decision_rankpct.csv",
        "dynamic_permutation_placebo_turnover_rankpct.csv",
        # Mechanism summaries used in the main narrative.
        "dynamic_mechanism_validation_summary_rankpct.csv",
        "dynamic_distance_banded_turnover_rankpct.csv",
        "dynamic_above_below_decomposition_main_rankpct.csv",
        "dynamic_directional_decomposition_decision_rankpct.csv",
        # Auxiliary top-10 salience summaries, not primary estimands.
        "binary_top10_prominence_decision_rankpct.csv",
        "binary_top10_prominence_diagnostics_rankpct.csv",
        "binary_top10_prominence_inference_rankpct.csv",
    }
    if fn in main_exact:
        return ("main", "core_or_required_validity_summary")

    # Explicit appendix families: relevant for transparency, not thesis-spine evidence.
    appendix_exact = {
        "overlap_design_decision_rankpct.csv",
        "dynamic_power_audit_rankpct.csv",
        "dynamic_support_mde_audit_rankpct.csv",
        "dynamic_stockout_consistency_summary_rankpct.csv",
        "dynamic_stockout_consistency_decision_rankpct.csv",
        "dynamic_stockout_consistent_turnover_rankpct.csv",
        "binary_top10_fitted_probability_audit_rankpct.csv",
        "dynamic_conservative_inference_rankpct.csv",
        "dynamic_cr2_inference_rankpct.csv",
        "dynamic_wild_cluster_bootstrap_rankpct.csv",
        "dynamic_turnover_placebos_rankpct.csv",
        "dynamic_above_below_decomposition_rankpct.csv",
        "dynamic_directional_decomposition_decision_reference_rankpct.csv",
        "dynamic_directional_reference_cr2_reference_rankpct.csv",
        "dynamic_directional_reference_wcb_reference_rankpct.csv",
    }
    if fn in appendix_exact:
        return ("appendix", "diagnostic_detail_or_redundant_summary")

    if fn.startswith("binary_top10"):
        return ("appendix", "top10_auxiliary_detail")
    if any(k in fn for k in (
        "overlap_balance", "overlap_trimmed", "propensity_overlap", "common_support_balance",
        "common_support_nonfba", "balance_hypothesis", "balance_table", "market_overlap",
    )):
        return ("appendix", "overlap_support_detail")
    if any(k in fn for k in (
        "dynamic_vacancy", "dynamic_clean_turnover", "dynamic_partial_identification",
        "dynamic_dropout", "dynamic_churn", "dynamic_pre_disappearance", "dynamic_return_pattern",
        "dropout_composition", "continuous_heterogeneity", "continuous_marginal",
        "dynamic_mde_subgroup", "dynamic_above_below_joint", "dynamic_stockout",
    )):
        return ("appendix", "dynamic_mechanism_or_support_detail")
    if any(k in fn for k in (
        "price_measure", "market_equal_weighted", "sample_sensitivity", "effect_translation",
        "influence", "leave_one", "feature_", "top_feature", "interaction", "excluded_interactions",
        "extended_control", "delivery_tail", "delivery_support", "shipping_support", "fast_delivery",
        "logistics_value_adjusted", "price_threshold", "rank_summary", "temporal_stability",
        "mundlak", "omitted_variable", "thesis_oster", "fractional_logit_rankpct",
        "spline", "standardized_feature", "chen_tsai", "economic_salience",
    )):
        return ("appendix", "robustness_or_context_detail")

    return ("appendix", "supporting_output")

exported_csv_files = sorted([p.name for p in OUTPUT_DIR.glob("*.csv")])
_role_rows = []
for _fn in exported_csv_files:
    _main_appendix, _role = _classify_table_role(_fn)
    _path = OUTPUT_DIR / _fn
    try:
        _rows = int(len(pd.read_csv(_path)))
    except Exception:
        _rows = np.nan
    _role_rows.append({
        "filename": _fn,
        "main_or_appendix": _main_appendix,
        "interpretation_role": _role,
        "rows": _rows,
    })

econometrics_table_manifest = pd.DataFrame(_role_rows)
econometrics_table_manifest.to_csv(OUTPUT_DIR / "econometrics_table_manifest.csv", index=False)

print("Unified table-role manifest")
display(econometrics_table_manifest)


Unified table-role manifest


,filename,main_or_appendix,interpretation_role,rows
0,attenuation_decomposition_rankpct.csv,main,core_or_required_validity_summary,3
1,attrition_table.csv,main,core_or_required_validity_summary,4
2,audit_consistency_checks.csv,audit_only,manifest_or_reproducibility_audit,34
3,balance_hypothesis_tests_by_fba.csv,appendix,overlap_support_detail,10
4,balance_table_by_fba.csv,appendix,overlap_support_detail,10
5,binary_top10_fitted_probability_audit_rankpct.csv,appendix,diagnostic_detail_or_redundant_summary,1
6,binary_top10_prominence_decision_rankpct.csv,main,core_or_required_validity_summary,1
7,binary_top10_prominence_diagnostics_rankpct.csv,main,core_or_required_validity_summary,5
8,binary_top10_prominence_inference_rankpct.csv,main,core_or_required_validity_summary,4
9,binary_top10_prominence_influence_summary_rankpct.csv,appendix,top10_auxiliary_detail,2


### 44. Integrated empirical synthesis from exported tables

The cell does not rerun the empirical pipeline. It reads the audited CSV outputs produced by the active run (or by a supplied `Econometrics-Results` reference directory), rebuilds the thesis-facing synthesis tables, validates the presence of every required file, and regenerates the thesis-aligned synthesis CSVs with the `thesis_*_aligned.csv` prefix. The aligned files are the version actually inserted into the thesis tables; the upstream non-aligned files are retained for traceability.


In [ ]:
# -----------------------------------------------------------------------------
# Section 44. Integrated empirical synthesis from exported tables
# -----------------------------------------------------------------------------
# Read the audited CSV outputs produced by the active run (or a supplied Econometrics-Results reference directory), rebuild the thesis-facing synthesis tables, and regenerate the thesis-aligned synthesis CSVs.

from pathlib import Path
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def resolve_econometrics_results_dir():
    """Resolve the active Econometrics-Results directory deterministically.

    The synthesis layer must not silently choose among multiple zipped output bundles.
    It first uses the live OUTPUT_DIR from the current run, then ECONOMETRICS_RESULTS_DIR from
    the environment, then conventional local directories. If none contain the required
    exported files, it fails with an explicit error.
    """
    required = {
        'main_results_market_fe_rankpct.csv',
        'dynamic_fba_premium_inference_rankpct.csv',
        'dynamic_mechanism_validation_summary_rankpct.csv',
    }

    candidates = []
    if 'OUTPUT_DIR' in globals():
        candidates.append(Path(OUTPUT_DIR))
    env_dir = os.environ.get('ECONOMETRICS_RESULTS_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([
        cwd / 'Econometrics-Results',
        cwd / 'Econometrics_Results',
        cwd / 'econometrics_outputs',
        cwd.parent / 'Econometrics-Results',
        Path('/mnt/data/econometrics_results/Econometrics-Results'),
        Path('/mnt/data/econometrics_results'),
        Path('/mnt/data/Econometrics-Results'),
    ])

    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_dir() and all((candidate / f).exists() for f in required):
            return candidate

    raise FileNotFoundError(
        'Could not locate a deterministic Econometrics-Results directory containing the required CSVs. '
        'Set the ECONOMETRICS_RESULTS_DIR environment variable or run the export cell so OUTPUT_DIR points to the active Econometrics-Results folder.'
    )

ECONOMETRICS_RESULTS_DIR = resolve_econometrics_results_dir()
SYNTHESIS_DIR = ECONOMETRICS_RESULTS_DIR / 'thesis_synthesis_aligned'
FIGURE_DIR = ECONOMETRICS_RESULTS_DIR / 'thesis_figures_aligned'
SYNTHESIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using ECON outputs: {ECONOMETRICS_RESULTS_DIR}')


def load_table(filename, required_cols=None):
    path = ECONOMETRICS_RESULTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing required output: {filename}')
    df = pd.read_csv(path)
    if required_cols:
        missing = sorted(set(required_cols) - set(df.columns))
        if missing:
            raise ValueError(f'{filename} is missing columns: {missing}')
    return df

required_specs = {
    'main_results_market_fe_rankpct.csv': ['specification', 'coef_fba', 'se_fba', 'pvalue_fba', 'ci_low', 'ci_high', 'r_squared'],
    'dynamic_fba_premium_inference_rankpct.csv': ['method', 'estimate', 'se', 'pvalue'],
    'dynamic_rank_tier_stress_diagnostics_rankpct.csv': ['object', 'estimate', 'se', 'pvalue', 'condition_number'],
    'dynamic_starting_rank_adjustment_diagnostics_rankpct.csv': ['model_label', 'model_role', 'estimate', 'se', 'pvalue', 'condition_number'],
    'dynamic_starting_rank_quartile_support_rankpct.csv': ['lag_rank_quartile', 'observations', 'fba_share'],
    'binary_top10_prominence_inference_rankpct.csv': ['method', 'estimate', 'se', 'pvalue'],
    'binary_top10_prominence_diagnostics_rankpct.csv': ['threshold_label', 'risk_set_observations', 'events', 'event_rate'],
    'binary_top10_prominence_threshold_family_rankpct.csv': ['threshold_label', 'estimate', 'se', 'pvalue', 'ci_low', 'ci_high', 'fdr_qvalue_bh'],
    'binary_top10_fitted_probability_audit_rankpct.csv': ['model_role', 'outcome', 'fitted_outside_unit_interval_share', 'fitted_min', 'fitted_max'],
    'fractional_logit_ape_rankpct.csv': ['estimate', 'se', 'pvalue'],
    'bounded_outcome_diagnostics_rankpct.csv': ['fitted_outside_unit_interval_rows', 'fitted_outside_unit_interval_share'],
    'overlap_design_status_summary_rankpct.csv': ['overlap_status', 'designs', 'median_mean_abs_smd'],
    'dynamic_distance_banded_turnover_rankpct.csv': ['term', 'estimate', 'se', 'pvalue', 'ci_low', 'ci_high', 'band_k'],
    'dynamic_permutation_placebo_turnover_rankpct.csv': ['observed_beta', 'permutation_p_value_two_sided', 'permutation_cell'],
    'interaction_terms_rankpct.csv': ['term_name', 'coef_fba', 'se_fba', 'pvalue_fba'],
    'interaction_margins_rankpct.csv': ['contrast_label', 'estimate', 'se', 'pvalue', 'ci_low', 'ci_high', 'moderator'],
    'dynamic_pre_disappearance_smoothness_rankpct.csv': ['variable', 'standardized_difference', 'pvalue_welch', 'materially_abnormal'],
    'audit_consistency_checks.csv': ['object', 'check_type', 'critical', 'passed', 'warning', 'detail'],
    'functional_form_diagnostics_rankpct.csv': ['diagnostic', 'statistic', 'pvalue'],
    'spline_functional_form_sensitivity_rankpct.csv': ['coef_fba', 'se_fba', 'pvalue_fba'],
    'temporal_stability_rankpct.csv': ['sample', 'coef_fba', 'se_fba', 'pvalue_fba'],
    'dynamic_return_pattern_rankpct.csv': ['return_category', 'dropout_observations', 'share'],
    'binary_top10_prominence_stockout_consistency_rankpct.csv': ['stockout_definition', 'estimate', 'se', 'pvalue'],
}

loaded = {name: load_table(name, cols) for name, cols in required_specs.items()}
alignment_audit = []
for name, cols in required_specs.items():
    df = loaded[name]
    alignment_audit.append({
        'file': name,
        'rows': len(df),
        'required_columns': ', '.join(cols),
        'status': 'loaded',
    })
alignment_audit_table = pd.DataFrame(alignment_audit)
alignment_audit_table.to_csv(SYNTHESIS_DIR / 'output_alignment_audit.csv', index=False)

display(alignment_audit_table)

consistency_checks = loaded.get('audit_consistency_checks.csv')
if consistency_checks is not None:
    _warning_mask = pd.Series(False, index=consistency_checks.index)
    if 'warning' in consistency_checks.columns:
        _warning_mask = _warning_mask | consistency_checks['warning'].astype(str).str.lower().isin(['true', '1', 'yes'])
    if 'quality_passed' in consistency_checks.columns:
        _quality = consistency_checks['quality_passed'].astype(str).str.lower()
        _warning_mask = _warning_mask | _quality.isin(['false', '0', 'no'])
    consistency_warning_summary = consistency_checks.loc[_warning_mask].copy()
    if consistency_warning_summary.empty:
        consistency_warning_summary = pd.DataFrame([{
            'object': 'all_checked_objects',
            'check_type': 'consistency_warning_summary',
            'critical': False,
            'passed': True,
            'warning': False,
            'detail': 'No audit-consistency warnings were present in the audited output set.',
        }])
    consistency_warning_summary.to_csv(SYNTHESIS_DIR / 'thesis_consistency_warning_summary_aligned.csv', index=False)
    display(consistency_warning_summary)

main_results = loaded['main_results_market_fe_rankpct.csv']
dyn_inf = loaded['dynamic_fba_premium_inference_rankpct.csv']
ranktier = loaded['dynamic_rank_tier_stress_diagnostics_rankpct.csv']
rank_adjust = loaded['dynamic_starting_rank_adjustment_diagnostics_rankpct.csv']
rank_quartile_support = loaded['dynamic_starting_rank_quartile_support_rankpct.csv']
bin_inf = loaded['binary_top10_prominence_inference_rankpct.csv']
bin_diag = loaded['binary_top10_prominence_diagnostics_rankpct.csv']
thresh = loaded['binary_top10_prominence_threshold_family_rankpct.csv']
frac = loaded['fractional_logit_ape_rankpct.csv']
bounded = loaded['bounded_outcome_diagnostics_rankpct.csv']
overlap = loaded['overlap_design_status_summary_rankpct.csv']
dist = loaded['dynamic_distance_banded_turnover_rankpct.csv']
perm = loaded['dynamic_permutation_placebo_turnover_rankpct.csv']
interact = loaded['interaction_terms_rankpct.csv']
margins = loaded['interaction_margins_rankpct.csv']
smooth = loaded['dynamic_pre_disappearance_smoothness_rankpct.csv']
functional_form = loaded['functional_form_diagnostics_rankpct.csv']
spline = loaded['spline_functional_form_sensitivity_rankpct.csv']
temporal_stability = loaded['temporal_stability_rankpct.csv']
return_pattern = loaded['dynamic_return_pattern_rankpct.csv']
top10_stockout = loaded['binary_top10_prominence_stockout_consistency_rankpct.csv']

# Master evidence table.
static = main_results.loc[main_results['specification'].eq('spec_4_fba_reputation_price_shipping_delivery')].iloc[0]
dyn_pref = dyn_inf.loc[dyn_inf['method'].eq('preferred_two_way_seller_transition_clustered')].iloc[0]
dyn_cr2_transition = dyn_inf.loc[dyn_inf['method'].eq('cr2_transition')].iloc[0]
dyn_wcb = dyn_inf.loc[dyn_inf['method'].eq('two_way_wild_cluster_bootstrap')].iloc[0]
rt = ranktier.loc[ranktier['object'].eq('transition_by_rank_tier_fe_stress_test')].iloc[0]
ra_q = rank_adjust.loc[rank_adjust['model_label'].eq('D2')].iloc[0]
ra_poly = rank_adjust.loc[rank_adjust['model_label'].eq('D3')].iloc[0]
top10 = bin_inf.loc[bin_inf['method'].eq('preferred_two_way_seller_transition_clustered_lpm')].iloc[0]
frac_row = frac.iloc[0]

master_evidence_table = pd.DataFrame([
    {
        'evidence_block': 'Static controlled FBA association',
        'estimand': 'Within-market residual rank_pct gap',
        'estimate': static['coef_fba'],
        'se': static['se_fba'],
        'pvalue': static['pvalue_fba'],
        'inference': static['inference'],
        'claim_role': 'main static association',
        'required_interpretation': 'conditional residual association; not an FBA adoption effect',
    },
    {
        'evidence_block': 'Primary dynamic turnover premium',
        'estimand': 'FBA x total seller-turnover rank movement',
        'estimate': dyn_pref['estimate'],
        'se': dyn_pref['se'],
        'pvalue': dyn_pref['pvalue'],
        'inference': dyn_pref['method'],
        'claim_role': 'main dynamic association',
        'required_interpretation': 'turnover-conditional differential rank movement; observed turnover is not randomized',
    },
    {
        'evidence_block': 'Dynamic starting-rank quartile adjustment',
        'estimand': 'FBA x total seller-turnover rank movement with additive lag-rank quartile controls',
        'estimate': ra_q['estimate'],
        'se': ra_q['se'],
        'pvalue': ra_q['pvalue'],
        'inference': 'two-way seller-transition',
        'claim_role': 'main rank-position validation',
        'required_interpretation': 'controls starting-rank regions parsimoniously; q1 top quartile is omitted',
    },
    {
        'evidence_block': 'Dynamic starting-rank quadratic adjustment',
        'estimand': 'FBA x total seller-turnover rank movement with lag_rank_pct squared',
        'estimate': ra_poly['estimate'],
        'se': ra_poly['se'],
        'pvalue': ra_poly['pvalue'],
        'inference': 'two-way seller-transition',
        'claim_role': 'main rank-position validation',
        'required_interpretation': 'smooth nonlinear starting-rank adjustment; avoids transition-cell saturation',
    },
    {
        'evidence_block': 'Dynamic conservative inference',
        'estimand': 'same as primary dynamic premium',
        'estimate': dyn_cr2_transition['estimate'],
        'se': dyn_cr2_transition['se'],
        'pvalue': dyn_cr2_transition['pvalue'],
        'inference': 'CR2 transition',
        'claim_role': 'finite-cluster sensitivity',
        'required_interpretation': 'supports inference robustness but not identification',
    },
    {
        'evidence_block': 'Dynamic wild-cluster bootstrap',
        'estimand': 'same as primary dynamic premium',
        'estimate': dyn_wcb['estimate'],
        'se': dyn_wcb['se'],
        'pvalue': dyn_wcb['pvalue'],
        'inference': 'two-way wild cluster bootstrap',
        'claim_role': 'finite-cluster sensitivity',
        'required_interpretation': 'bootstrap supports inference robustness but not identification',
    },
    {
        'evidence_block': 'Transition x rank-tier FE stress test',
        'estimand': 'within transition and starting-rank-tier dynamic premium',
        'estimate': rt['estimate'],
        'se': rt['se'],
        'pvalue': rt['pvalue'],
        'inference': 'two-way seller-transition with stricter FE',
        'claim_role': 'severe support-sensitive stress test',
        'required_interpretation': 'nonconfirmation in saturated cells; interpreted cautiously when condition number or within-cell support is weak',
    },
    {
        'evidence_block': 'Complementary top-10 entry outcome',
        'estimand': 'FBA x turnover effect on entering top 10 among at-risk sellers',
        'estimate': top10['estimate'],
        'se': top10['se'],
        'pvalue': top10['pvalue'],
        'inference': top10['method'],
        'claim_role': 'auxiliary salience evidence',
        'required_interpretation': 'rare-event visibility translation; not the primary empirical backbone',
    },
    {
        'evidence_block': 'Fractional-logit bounded-outcome diagnostic',
        'estimand': 'average partial effect on rank_pct',
        'estimate': frac_row['estimate'],
        'se': frac_row['se'],
        'pvalue': frac_row['pvalue'],
        'inference': frac_row.get('inference', 'fractional logit APE'),
        'claim_role': 'functional-form diagnostic',
        'required_interpretation': 'does not confirm the linear magnitude; OLS is a linear projection',
    },
    {
        'evidence_block': 'Quadratic-control functional-form sensitivity',
        'estimand': 'within-market residual rank_pct gap with nonlinear controls',
        'estimate': spline.iloc[0]['coef_fba'],
        'se': spline.iloc[0]['se_fba'],
        'pvalue': spline.iloc[0]['pvalue_fba'],
        'inference': spline.iloc[0].get('inference', 'two_way_seller_market'),
        'claim_role': 'functional-form diagnostic',
        'required_interpretation': 'confirms the sign but reduces the magnitude relative to headline OLS',
    },
])
master_evidence_table.to_csv(SYNTHESIS_DIR / 'thesis_master_evidence_table_aligned.csv', index=False)
display(master_evidence_table)

# Oster-style diagnostic, recalculated from exported nested static models.
short = main_results.loc[main_results['specification'].eq('spec_1_fba_only')].iloc[0]
full = static
oster_rows = []
# Multipliers chosen to span both sub-cap (1.005 to 1.04) and at-cap (1.05, 1.10) regions
# of the R_max ceiling 0.999. Probing only multipliers above the cap collapses the grid
# to a single value and fails to illustrate Rmax sensitivity when R_full is around 0.96.
for mult in [1.005, 1.01, 1.02, 1.03, 1.04, 1.05, 1.10]:
    rmax_unconstrained = float(mult) * float(full['r_squared'])
    rmax = min(rmax_unconstrained, 0.999)
    cap_active = rmax_unconstrained > 0.999
    denom = (float(short['coef_fba']) - float(full['coef_fba'])) * (rmax - float(full['r_squared']))
    delta_to_zero = np.nan if (rmax <= float(full['r_squared']) or abs(denom) < 1e-12) else ((float(full['coef_fba']) - 0.0) * (float(full['r_squared']) - float(short['r_squared']))) / denom
    oster_rows.append({
        'comparison': 'spec_1_fba_only_to_spec_4_full_controls',
        'beta_short': short['coef_fba'],
        'r2_short': short['r_squared'],
        'beta_full': full['coef_fba'],
        'r2_full': full['r_squared'],
        'rmax_multiplier': mult,
        'rmax_unconstrained': rmax_unconstrained,
        'rmax_used': rmax,
        'rmax_cap_active': bool(cap_active),
        'delta_to_zero': delta_to_zero,
        'abs_delta_gt_one': bool(pd.notna(delta_to_zero) and abs(delta_to_zero) > 1),
        'interpretation': 'Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivity; at-cap multipliers (1.05, 1.10) document the cap floor.',
    })
oster_grid_aligned = pd.DataFrame(oster_rows)
oster_grid_aligned.to_csv(SYNTHESIS_DIR / 'thesis_oster_sensitivity_grid_aligned.csv', index=False)
oster_grid_aligned.loc[oster_grid_aligned['rmax_multiplier'].eq(1.04)].to_csv(SYNTHESIS_DIR / 'thesis_oster_sensitivity_aligned.csv', index=False)
display(oster_grid_aligned)

# Additional thesis-facing caution summaries requested by audit audit.
reset_power2 = functional_form.loc[functional_form['diagnostic'].eq('Ramsey RESET power 2')].iloc[0]
functional_form_fragility_aligned = pd.DataFrame([
    {'model': 'headline_ols_linear_projection', 'estimate': static['coef_fba'], 'se': static['se_fba'], 'pvalue': static['pvalue_fba'], 'relative_abs_magnitude_vs_headline': 1.0},
    {'model': 'quadratic_controls', 'estimate': spline.iloc[0]['coef_fba'], 'se': spline.iloc[0]['se_fba'], 'pvalue': spline.iloc[0]['pvalue_fba'], 'relative_abs_magnitude_vs_headline': abs(float(spline.iloc[0]['coef_fba'])) / max(abs(float(static['coef_fba'])), 1e-12)},
    {'model': 'fractional_logit_ape', 'estimate': frac_row['estimate'], 'se': frac_row['se'], 'pvalue': frac_row['pvalue'], 'relative_abs_magnitude_vs_headline': abs(float(frac_row['estimate'])) / max(abs(float(static['coef_fba'])), 1e-12)},
])
functional_form_fragility_aligned['reset_power2_statistic'] = reset_power2['statistic']
functional_form_fragility_aligned['reset_power2_pvalue'] = reset_power2['pvalue']
functional_form_fragility_aligned['claim_implication'] = 'Magnitude is functional-form sensitive; headline OLS is retained as a linear projection, not as a universal conditional-mean estimate.'
functional_form_fragility_aligned.to_csv(SYNTHESIS_DIR / 'thesis_functional_form_fragility_aligned.csv', index=False)
display(functional_form_fragility_aligned)

zero_se_top10_stockout_aligned = top10_stockout.assign(
    zero_se_flag=lambda d: pd.to_numeric(d['se'], errors='coerce').fillna(0).eq(0),
    interpretation_note='Zero-SE rows are numerical diagnostics only and must not be used for confirmatory inference.'
)
zero_se_top10_stockout_aligned.to_csv(SYNTHESIS_DIR / 'thesis_top10_stockout_zero_se_audit_aligned.csv', index=False)
display(zero_se_top10_stockout_aligned)

temporal_stability_aligned = temporal_stability.copy()
if len(temporal_stability_aligned) >= 2:
    est = temporal_stability_aligned.loc[temporal_stability_aligned['sample'].eq('estimation_sample'), 'coef_fba']
    hold = temporal_stability_aligned.loc[temporal_stability_aligned['sample'].eq('holdout_sample'), 'coef_fba']
    if not est.empty and not hold.empty:
        temporal_stability_aligned['holdout_vs_estimation_abs_magnitude_ratio'] = abs(float(hold.iloc[0])) / max(abs(float(est.iloc[0])), 1e-12)
temporal_stability_aligned['claim_implication'] = 'Signs are stable across time splits, but magnitude is temporally heterogeneous and should not be extrapolated beyond the observed window.'
temporal_stability_aligned.to_csv(SYNTHESIS_DIR / 'thesis_temporal_stability_aligned.csv', index=False)
display(temporal_stability_aligned)

return_pattern_aligned = return_pattern.copy()
return_pattern_aligned['claim_implication'] = 'Stockout-consistency is a restriction and plausibility screen; persistent absences mean it is not proof of temporary inventory stockouts.'
return_pattern_aligned.to_csv(SYNTHESIS_DIR / 'thesis_dropout_return_pattern_aligned.csv', index=False)
display(return_pattern_aligned)

# Compact notes for results that should be in prose.
price_interaction = interact.loc[interact['term_name'].eq('fba_from_shipper:prezzo_c')].copy()
price_margins = margins.loc[margins['moderator'].eq('prezzo')].copy()
pre_disappearance_abnormal = smooth.loc[smooth['materially_abnormal'].astype(bool)].copy()
price_interaction.to_csv(SYNTHESIS_DIR / 'thesis_price_interaction_key_result_aligned.csv', index=False)
price_margins.to_csv(SYNTHESIS_DIR / 'thesis_price_interaction_margins_aligned.csv', index=False)
pre_disappearance_abnormal.to_csv(SYNTHESIS_DIR / 'thesis_pre_disappearance_abnormal_smoothness_aligned.csv', index=False)
display(price_interaction)
display(price_margins)
display(pre_disappearance_abnormal)

# Figures. No custom colors are assigned; matplotlib defaults are used.
def save_errorbar(df, x, label, low, high, title, xlabel, filename):
    plot_df = df.copy().reset_index(drop=True)
    y = np.arange(len(plot_df))
    xv = pd.to_numeric(plot_df[x], errors='coerce')
    lo = pd.to_numeric(plot_df[low], errors='coerce')
    hi = pd.to_numeric(plot_df[high], errors='coerce')
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(plot_df) + 1)))
    ax.errorbar(xv, y, xerr=np.vstack([xv - lo, hi - xv]), fmt='o', capsize=3)
    ax.axvline(0, linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df[label])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.invert_yaxis()
    fig.tight_layout()
    out = FIGURE_DIR / filename
    fig.savefig(out, dpi=240, bbox_inches='tight')
    fig.savefig(out.with_suffix('.svg'), bbox_inches='tight')
    plt.close(fig)
    return out

_static_label_map = {
    'spec_1_fba_only': 'Spec 1: FBA only',
    'spec_2_fba_reputation': 'Spec 2: + reputation',
    'spec_3_fba_reputation_totalprice_delivery': 'Spec 3: + total price + delivery',
    'spec_4_fba_reputation_price_shipping_delivery': 'Spec 4: + price + shipping + delivery',
}
static_plot = main_results.assign(label=main_results['specification'].map(_static_label_map).fillna(main_results['specification'].astype(str)))
static_plot = static_plot[['label', 'coef_fba', 'ci_low', 'ci_high']].rename(columns={'coef_fba': 'estimate'})
save_errorbar(static_plot, 'estimate', 'label', 'ci_low', 'ci_high', 'Static FBA coefficient across nested specifications', 'Coefficient on FBA; lower rank_pct is better', 'figure_01_static_fba_attenuation_aligned.png')

# Pull D2 and D3 rank-position validation rows from rank_adjust for the dynamic
# forest plot so the figure mirrors the master evidence table and the conclusion.
def _ranklab_pull(label):
    try:
        rl = rank_adjust.loc[rank_adjust['model_label'].eq(label)]
        if not rl.empty:
            r = rl.iloc[0]
            return {'estimate': float(r['estimate']),
                    'ci_low': float(r['ci_low']),
                    'ci_high': float(r['ci_high'])}
    except Exception:
        pass
    return {'estimate': float('nan'), 'ci_low': float('nan'), 'ci_high': float('nan')}

_ra_q_plot = _ranklab_pull('D2')
_ra_poly_plot = _ranklab_pull('D3')

dyn_plot = pd.DataFrame([
    {'label': 'D1 primary dynamic',
     'estimate': dyn_pref['estimate'],
     'ci_low': dyn_pref['estimate'] - 1.96 * dyn_pref['se'],
     'ci_high': dyn_pref['estimate'] + 1.96 * dyn_pref['se']},
    {'label': 'D2 + lag-rank quartiles',
     'estimate': _ra_q_plot['estimate'],
     'ci_low': _ra_q_plot['ci_low'],
     'ci_high': _ra_q_plot['ci_high']},
    {'label': 'D3 + quadratic lag rank',
     'estimate': _ra_poly_plot['estimate'],
     'ci_low': _ra_poly_plot['ci_low'],
     'ci_high': _ra_poly_plot['ci_high']},
    {'label': 'CR2 transition',
     'estimate': dyn_cr2_transition['estimate'],
     'ci_low': dyn_cr2_transition['estimate'] - 1.96 * dyn_cr2_transition['se'],
     'ci_high': dyn_cr2_transition['estimate'] + 1.96 * dyn_cr2_transition['se']},
    {'label': 'D4 rank-tier FE stress',
     'estimate': rt['estimate'],
     'ci_low': rt['estimate'] - 1.96 * rt['se'],
     'ci_high': rt['estimate'] + 1.96 * rt['se']},
])
save_errorbar(dyn_plot, 'estimate', 'label', 'ci_low', 'ci_high',
              'Dynamic FBA turnover premium with rank-position validations and stress test',
              'Coefficient on FBA x turnover; positive means larger upward movement',
              'figure_02_dynamic_premium_ranktier_aligned.png')

thresh_plot = thresh.sort_values('threshold_order' if 'threshold_order' in thresh.columns else 'threshold_label')
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(thresh_plot))
y = pd.to_numeric(thresh_plot['estimate'], errors='coerce')
lo = pd.to_numeric(thresh_plot['ci_low'], errors='coerce')
hi = pd.to_numeric(thresh_plot['ci_high'], errors='coerce')
ax.errorbar(x, y, yerr=np.vstack([y - lo, hi - y]), fmt='o-', capsize=3)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(thresh_plot['threshold_label'].astype(str), rotation=20, ha='right')
ax.set_title('Top-of-list threshold-family estimates')
ax.set_ylabel('LPM coefficient on FBA x turnover')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'figure_03_topk_threshold_family_aligned.png', dpi=240, bbox_inches='tight')
fig.savefig((FIGURE_DIR / 'figure_03_topk_threshold_family_aligned.png').with_suffix('.svg'), bbox_inches='tight')
plt.close(fig)

prem_dist = dist.loc[dist['term'].str.contains('fba_x_dropouts_total_within', na=False)].sort_values('band_k')
fig, ax = plt.subplots(figsize=(7, 4))
x = pd.to_numeric(prem_dist['band_k'], errors='coerce')
y = pd.to_numeric(prem_dist['estimate'], errors='coerce')
lo = pd.to_numeric(prem_dist['ci_low'], errors='coerce')
hi = pd.to_numeric(prem_dist['ci_high'], errors='coerce')
ax.errorbar(x, y, yerr=np.vstack([y - lo, hi - y]), fmt='o-', capsize=3)
ax.axhline(0, linewidth=1)
ax.set_title('Distance-banded turnover diagnostics')
ax.set_xlabel('Rank-distance band K')
ax.set_ylabel('Coefficient on FBA x banded turnover')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'figure_04_distance_banded_turnover_aligned.png', dpi=240, bbox_inches='tight')
fig.savefig((FIGURE_DIR / 'figure_04_distance_banded_turnover_aligned.png').with_suffix('.svg'), bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
plot_overlap = overlap.copy()
ax.bar(np.arange(len(plot_overlap)), plot_overlap['median_mean_abs_smd'])
ax.set_xticks(np.arange(len(plot_overlap)))
ax.set_xticklabels(plot_overlap['overlap_status'].astype(str), rotation=20, ha='right')
ax.set_title('Overlap diagnostics: median absolute SMD by status')
ax.set_ylabel('Median mean absolute SMD')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'figure_05_overlap_smd_summary_aligned.png', dpi=240, bbox_inches='tight')
fig.savefig((FIGURE_DIR / 'figure_05_overlap_smd_summary_aligned.png').with_suffix('.svg'), bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
price_plot = price_margins.reset_index(drop=True)
x = np.arange(len(price_plot))
y = pd.to_numeric(price_plot['estimate'], errors='coerce')
lo = pd.to_numeric(price_plot['ci_low'], errors='coerce')
hi = pd.to_numeric(price_plot['ci_high'], errors='coerce')
ax.errorbar(x, y, yerr=np.vstack([y - lo, hi - y]), fmt='o-', capsize=3)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(price_plot['contrast_label'].astype(str), rotation=15, ha='right')
ax.set_title('FBA association at product-price quantiles')
ax.set_ylabel('FBA association on rank_pct')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'figure_06_price_interaction_margins_aligned.png', dpi=240, bbox_inches='tight')
fig.savefig((FIGURE_DIR / 'figure_06_price_interaction_margins_aligned.png').with_suffix('.svg'), bbox_inches='tight')
plt.close(fig)

file_table = pd.DataFrame({
    'file': [p.name for p in sorted(list(SYNTHESIS_DIR.glob('*.csv')) + list(FIGURE_DIR.glob('*.png')) + list(FIGURE_DIR.glob('*.svg')))],
    'path': [str(p) for p in sorted(list(SYNTHESIS_DIR.glob('*.csv')) + list(FIGURE_DIR.glob('*.png')) + list(FIGURE_DIR.glob('*.svg')))],
})
display(file_table)

# FBA-only logistics-value-adjusted price specification (output-aligned).
_logistics_value_path = ECONOMETRICS_RESULTS_DIR / 'logistics_value_adjusted_price_results_rankpct.csv'
if _logistics_value_path.exists():
    logistics_value_adjusted_results_loaded = pd.read_csv(_logistics_value_path)
    logistics_value_adjusted_results_loaded.to_csv(
        SYNTHESIS_DIR / 'thesis_logistics_value_adjusted_price_specification_aligned.csv', index=False
    )
    display(logistics_value_adjusted_results_loaded)

    # Append the baseline 4.99 EUR FBA-only adjustment to the master evidence table.
    # The 3.99/8.99 variants are one-way monetary sensitivity translations and are
    # kept in the variant-level table only.
    _select_for_master = logistics_value_adjusted_results_loaded.loc[
        logistics_value_adjusted_results_loaded['logistics_value_variant'].isin([
            'logistics_value_adjusted_fba_only_4_99_baseline',
        ])
    ].copy()
    _master_extra_rows = []
    for _, _row in _select_for_master.iterrows():
        _master_extra_rows.append({
            'evidence_block': 'Static FBA-only logistics-value-adjusted price (4.99 EUR baseline)',
            'estimand': 'Within-market residual rank_pct gap after FBA-only hidden fast-delivery value/cost imputation',
            'estimate': float(_row.get('coef_fba', float('nan'))),
            'se': float(_row.get('se_fba', float('nan'))),
            'pvalue': float(_row.get('pvalue_fba', float('nan'))),
            'inference': str(_row.get('inference', 'two_way_seller_market')),
            'claim_role': 'post-treatment sensitivity estimand with respect to the FBA bundle',
            'required_interpretation': (
                'the adjusted price adds an imputed fast-delivery value/cost only to '
                'FBA listings with fast-delivery availability; it is not a directly '
                'observed checkout price and is not directly comparable to the split-price headline'
            ),
        })
    if _master_extra_rows:
        _existing_master = pd.read_csv(SYNTHESIS_DIR / 'thesis_master_evidence_table_aligned.csv')
        master_evidence_table_with_logistics_value_adjustment = pd.concat(
            [_existing_master, pd.DataFrame(_master_extra_rows)], ignore_index=True
        )
        master_evidence_table_with_logistics_value_adjustment.to_csv(
            SYNTHESIS_DIR / 'thesis_master_evidence_table_aligned.csv', index=False
        )
        display(master_evidence_table_with_logistics_value_adjustment)

# Thesis-facing figure manifest: makes main/appendix figure roles explicit.
figure_manifest = pd.DataFrame([
    {
        "figure_file": "thesis_figures_aligned/figure_01_static_fba_attenuation_aligned.png",
        "source_table": "main_results_market_fe_rankpct.csv; attenuation_decomposition_rankpct.csv",
        "claim_role": "static core evidence",
        "main_or_appendix": "main",
        "required_caveat": "negative coefficients mean better rank_pct; static association is not a causal FBA adoption effect",
    },
    {
        "figure_file": "thesis_figures_aligned/figure_02_dynamic_premium_ranktier_aligned.png",
        "source_table": "dynamic_fba_premium_inference_rankpct.csv; dynamic_starting_rank_adjustment_diagnostics_rankpct.csv; dynamic_rank_tier_stress_diagnostics_rankpct.csv",
        "claim_role": "dynamic core evidence and D1-D4 hierarchy",
        "main_or_appendix": "main",
        "required_caveat": "D2/D3 are main starting-rank validations; D4 is a severe support-sensitive stress test, not the preferred estimator",
    },
    {
        "figure_file": "thesis_figures_aligned/figure_03_topk_threshold_family_aligned.png",
        "source_table": "binary_top10_prominence_threshold_family_rankpct.csv",
        "claim_role": "auxiliary top-of-list salience threshold family",
        "main_or_appendix": "appendix",
        "required_caveat": "top-10/top-k outcomes are sparse-event auxiliary LPM diagnostics and do not replace continuous rank-percent evidence",
    },
    {
        "figure_file": "thesis_figures_aligned/figure_04_distance_banded_turnover_aligned.png",
        "source_table": "dynamic_distance_banded_turnover_rankpct.csv",
        "claim_role": "mechanism diagnostic for local vacancy filling",
        "main_or_appendix": "main",
        "required_caveat": "distance-banded checks test the narrow local-vacancy mechanism and do not falsify the broader turnover-period premium",
    },
    {
        "figure_file": "thesis_figures_aligned/figure_05_overlap_smd_summary_aligned.png",
        "source_table": "overlap_balance_summary_rankpct.csv; overlap_design_status_summary_rankpct.csv",
        "claim_role": "static overlap/support validity gate",
        "main_or_appendix": "main",
        "required_caveat": "overlap failure weakens static like-for-like causal interpretation and is not a robustness confirmation",
    },
    {
        "figure_file": "thesis_figures_aligned/figure_06_price_interaction_margins_aligned.png",
        "source_table": "interaction_margins_rankpct.csv",
        "claim_role": "exploratory price-interaction context",
        "main_or_appendix": "appendix",
        "required_caveat": "interaction margins are descriptive heterogeneity checks, not causal moderation evidence",
    },
])
figure_manifest.to_csv(ECONOMETRICS_RESULTS_DIR / "figure_manifest.csv", index=False)
try:
    figure_manifest.to_csv(OUTPUT_DIR / "figure_manifest.csv", index=False)
except Exception:
    pass
display(figure_manifest)

# Thesis-facing synthesis-table manifest: separates claim-facing synthesis from audit-only alignment checks.
synthesis_manifest = pd.DataFrame([
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_master_evidence_table_aligned.csv",
        "source_tables": "main_results_market_fe_rankpct.csv; dynamic_fba_premium_inference_rankpct.csv; dynamic_starting_rank_adjustment_diagnostics_rankpct.csv; dynamic_rank_tier_stress_diagnostics_rankpct.csv; dynamic_distance_banded_turnover_rankpct.csv",
        "claim_role": "compact thesis-facing master evidence table",
        "main_or_appendix": "main",
        "required_caveat": "summarizes observational fixed-effects associations only; it is not a causal proof table",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_functional_form_fragility_aligned.csv",
        "source_tables": "functional_form_fragility_summary_rankpct.csv; bounded_outcome_diagnostics_rankpct.csv; fractional_logit_ape_rankpct.csv",
        "claim_role": "bounded-outcome and functional-form caveat",
        "main_or_appendix": "main",
        "required_caveat": "narrows magnitude interpretation rather than refuting the linear projection",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_logistics_value_adjusted_price_specification_aligned.csv",
        "source_tables": "logistics_value_adjusted_price_results_rankpct.csv",
        "claim_role": "FBA-only fast-delivery-cost sensitivity",
        "main_or_appendix": "appendix",
        "required_caveat": "constructed logistics-value proxy; not a causal robustness check",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_oster_sensitivity_aligned.csv",
        "source_tables": "omitted_variable_sensitivity_rankpct.csv; thesis_oster_sensitivity_grid.csv",
        "claim_role": "omitted-confounding sensitivity",
        "main_or_appendix": "appendix",
        "required_caveat": "selection-on-observables diagnostic only",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_oster_sensitivity_grid_aligned.csv",
        "source_tables": "thesis_oster_sensitivity_grid.csv",
        "claim_role": "Oster Rmax grid detail",
        "main_or_appendix": "appendix",
        "required_caveat": "grid detail is not a main result",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_dropout_return_pattern_aligned.csv",
        "source_tables": "dynamic_return_pattern_rankpct.csv; dynamic_return_pattern_summary_rankpct.csv",
        "claim_role": "dropout/return context for stockout interpretation",
        "main_or_appendix": "appendix",
        "required_caveat": "seller disappearance is not randomized stockout variation",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_pre_disappearance_abnormal_smoothness_aligned.csv",
        "source_tables": "dynamic_pre_disappearance_smoothness_rankpct.csv; dynamic_pre_disappearance_smoothness_decision_rankpct.csv",
        "claim_role": "pre-disappearance smoothness diagnostic",
        "main_or_appendix": "appendix",
        "required_caveat": "descriptive process diagnostic only",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_consistency_warning_summary_aligned.csv",
        "source_tables": "audit_consistency_checks.csv; numerical_pathology_summary.csv",
        "claim_role": "audit warning audit",
        "main_or_appendix": "audit_only",
        "required_caveat": "audit trail, not evidence of an estimand",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_price_interaction_key_result_aligned.csv",
        "source_tables": "interaction_terms_rankpct.csv; interaction_interpretation_rankpct.csv",
        "claim_role": "exploratory heterogeneity context",
        "main_or_appendix": "appendix",
        "required_caveat": "not causal moderation evidence",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_price_interaction_margins_aligned.csv",
        "source_tables": "interaction_margins_rankpct.csv",
        "claim_role": "exploratory interaction margins",
        "main_or_appendix": "appendix",
        "required_caveat": "appendix visualization support only",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_temporal_stability_aligned.csv",
        "source_tables": "temporal_stability_rankpct.csv",
        "claim_role": "descriptive temporal stability context",
        "main_or_appendix": "appendix",
        "required_caveat": "descriptive stability check only",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/thesis_top10_stockout_zero_se_audit_aligned.csv",
        "source_tables": "binary_top10_prominence_stockout_consistency_rankpct.csv; numerical_pathology_summary.csv",
        "claim_role": "top-10 stockout zero-SE audit",
        "main_or_appendix": "audit_only",
        "required_caveat": "sparse-event numerical audit; not main salience evidence",
    },
    {
        "synthesis_file": "thesis_synthesis_aligned/output_alignment_audit.csv",
        "source_tables": "required output files",
        "claim_role": "file-presence alignment audit",
        "main_or_appendix": "audit_only",
        "required_caveat": "presence/shape check only",
    },
])
synthesis_manifest.to_csv(ECONOMETRICS_RESULTS_DIR / "synthesis_manifest.csv", index=False)
try:
    synthesis_manifest.to_csv(OUTPUT_DIR / "synthesis_manifest.csv", index=False)
except Exception:
    pass
display(synthesis_manifest)



Using ECON outputs: /content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results


,file,rows,required_columns,status
0,main_results_market_fe_rankpct.csv,4,"specification, coef_fba, se_fba, pvalue_fba, ci_low, ci_high, r_squared",loaded
1,dynamic_fba_premium_inference_rankpct.csv,4,"method, estimate, se, pvalue",loaded
2,dynamic_rank_tier_stress_diagnostics_rankpct.csv,4,"object, estimate, se, pvalue, condition_number",loaded
3,dynamic_starting_rank_adjustment_diagnostics_rankpct.csv,4,"model_label, model_role, estimate, se, pvalue, condition_number",loaded
4,dynamic_starting_rank_quartile_support_rankpct.csv,4,"lag_rank_quartile, observations, fba_share",loaded
5,binary_top10_prominence_inference_rankpct.csv,4,"method, estimate, se, pvalue",loaded
6,binary_top10_prominence_diagnostics_rankpct.csv,5,"threshold_label, risk_set_observations, events, event_rate",loaded
7,binary_top10_prominence_threshold_family_rankpct.csv,5,"threshold_label, estimate, se, pvalue, ci_low, ci_high, fdr_qvalue_bh",loaded
8,binary_top10_fitted_probability_audit_rankpct.csv,1,"model_role, outcome, fitted_outside_unit_interval_share, fitted_min, fitted_max",loaded
9,fractional_logit_ape_rankpct.csv,1,"estimate, se, pvalue",loaded


,object,check_type,required_columns,rows,missing_columns,critical,passed,warning,detail,quality_passed
29,above_below_decomposition_table,numerical_pathology_warning,finite positive standard errors in interpreted rows; condition-number flags exposed,3,NaN,False,True,True,numerical_pathology_scan: zero_se_present=False; ill_conditioned_present=True; interpret flagged rows only as appendix/unstable,True
32,dropout_composition_table,numerical_pathology_warning,finite positive standard errors in interpreted rows; condition-number flags exposed,2,NaN,False,True,True,numerical_pathology_scan: zero_se_present=True; ill_conditioned_present=False; interpret flagged rows only as appendix/unstable,False
33,dynamic_vacancy_rank_tier_models_table,numerical_pathology_warning,finite positive standard errors in interpreted rows; condition-number flags exposed,24,NaN,False,True,True,numerical_pathology_scan: zero_se_present=True; ill_conditioned_present=False; interpret flagged rows only as appendix/unstable,False


,evidence_block,estimand,estimate,se,pvalue,inference,claim_role,required_interpretation
0,Static controlled FBA association,Within-market residual rank_pct gap,-0.051038,0.015968,0.002207,two_way_seller_market,main static association,conditional residual association; not an FBA adoption effect
1,Primary dynamic turnover premium,FBA x total seller-turnover rank movement,0.002467,0.000765,0.002034,preferred_two_way_seller_transition_clustered,main dynamic association,turnover-conditional differential rank movement; observed turnover is not randomized
2,Dynamic starting-rank quartile adjustment,FBA x total seller-turnover rank movement with additive lag-rank quartile controls,0.002710,0.000869,0.002782,two-way seller-transition,main rank-position validation,controls starting-rank regions parsimoniously; q1 top quartile is omitted
3,Dynamic starting-rank quadratic adjustment,FBA x total seller-turnover rank movement with lag_rank_pct squared,0.002278,0.000674,0.001278,two-way seller-transition,main rank-position validation,smooth nonlinear starting-rank adjustment; avoids transition-cell saturation
4,Dynamic conservative inference,same as primary dynamic premium,0.002467,0.000803,0.024824,CR2 transition,finite-cluster sensitivity,supports inference robustness but not identification
5,Dynamic wild-cluster bootstrap,same as primary dynamic premium,0.002467,0.000765,0.008832,two-way wild cluster bootstrap,finite-cluster sensitivity,bootstrap supports inference robustness but not identification
6,Transition x rank-tier FE stress test,within transition and starting-rank-tier dynamic premium,-0.000075,0.000699,0.915343,two-way seller-transition with stricter FE,severe support-sensitive stress test,nonconfirmation in saturated cells; interpreted cautiously when condition number or within-cell support is weak
7,Complementary top-10 entry outcome,FBA x turnover effect on entering top 10 among at-risk sellers,0.012174,0.005032,0.018599,preferred_two_way_seller_transition_clustered_lpm,auxiliary salience evidence,rare-event visibility translation; not the primary empirical backbone
8,Fractional-logit bounded-outcome diagnostic,average partial effect on rank_pct,-0.014065,0.011644,0.231724,two_way_seller_market,functional-form diagnostic,does not confirm the linear magnitude; OLS is a linear projection
9,Quadratic-control functional-form sensitivity,within-market residual rank_pct gap with nonlinear controls,-0.035229,0.012232,0.005480,two_way_seller_market,functional-form diagnostic,confirms the sign but reduces the magnitude relative to headline OLS


,comparison,beta_short,r2_short,beta_full,r2_full,rmax_multiplier,rmax_unconstrained,rmax_used,rmax_cap_active,delta_to_zero,abs_delta_gt_one,interpretation
0,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.005,0.964178,0.964178,False,22.412272,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
1,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.010,0.968975,0.968975,False,11.206136,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
2,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.020,0.978569,0.978569,False,5.603068,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
3,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.030,0.988162,0.988162,False,3.735379,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
4,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.040,0.997756,0.997756,False,2.801534,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
5,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.050,1.007350,0.999000,True,2.713586,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...
6,spec_1_fba_only_to_spec_4_full_controls,-0.380436,0.26552,-0.051038,0.959381,1.100,1.055319,0.999000,True,2.713586,True,Oster-style diagnostic only; high fixed-effect R2 makes the exact delta sensitive to Rmax and it is not a causal proof. Sub-cap multipliers (1.005 to 1.04) probe Rmax sensitivi...


...

,file,path
0,figure_01_static_fba_attenuation_aligned.png,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_01_static_fba_attenuation_aligned.png
1,figure_01_static_fba_attenuation_aligned.svg,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_01_static_fba_attenuation_aligned.svg
2,figure_02_dynamic_premium_ranktier_aligned.png,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_02_dynamic_premium_ranktier_aligned.png
3,figure_02_dynamic_premium_ranktier_aligned.svg,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_02_dynamic_premium_ranktier_aligned.svg
4,figure_03_topk_threshold_family_aligned.png,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_03_topk_threshold_family_aligned.png
5,figure_03_topk_threshold_family_aligned.svg,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_03_topk_threshold_family_aligned.svg
6,figure_04_distance_banded_turnover_aligned.png,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_04_distance_banded_turnover_aligned.png
7,figure_04_distance_banded_turnover_aligned.svg,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_04_distance_banded_turnover_aligned.svg
8,figure_05_overlap_smd_summary_aligned.png,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_05_overlap_smd_summary_aligned.png
9,figure_05_overlap_smd_summary_aligned.svg,/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results/thesis_figures_aligned/figure_05_overlap_smd_summary_aligned.svg


,sample,outcome,specification,estimator,inference,term_name,coef_fba,se_fba,test_statistic,pvalue_fba,ci_low,ci_high,stars,dof_reference,nobs,r_squared,seller_clusters,market_clusters,logistics_value_variant,coef_change_from_split_price_headline,abs_pct_change_from_split_price_headline,interpretation
0,full_sample,rank_pct,logistics_value_adjusted_fba_only_4_99_baseline,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.076010,0.015597,-4.873232,8.167038e-06,-0.107199,-0.044821,***,61,5107,0.954124,119,62,logistics_value_adjusted_fba_only_4_99_baseline,-0.024972,48.927807,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...
1,full_sample,rank_pct,logistics_value_adjusted_fba_only_3_99_lower,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.070470,0.015543,-4.533896,2.762769e-05,-0.101550,-0.039390,***,61,5107,0.956160,119,62,logistics_value_adjusted_fba_only_3_99_lower,-0.019432,38.072795,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...
2,full_sample,rank_pct,logistics_value_adjusted_fba_only_8_99_upper,market_fe_ols,two_way_seller_market,fba_from_shipper,-0.100119,0.016594,-6.033453,1.026807e-07,-0.133301,-0.066937,***,61,5107,0.941277,119,62,logistics_value_adjusted_fba_only_8_99_upper,-0.049081,96.165531,FBA-only logistics-value-adjusted price specification. The split-price headline and this row estimate different empirical objects: the headline is the residual rank gap after c...


,evidence_block,estimand,estimate,se,pvalue,inference,claim_role,required_interpretation
0,Static controlled FBA association,Within-market residual rank_pct gap,-0.051038,0.015968,0.002207,two_way_seller_market,main static association,conditional residual association; not an FBA adoption effect
1,Primary dynamic turnover premium,FBA x total seller-turnover rank movement,0.002467,0.000765,0.002034,preferred_two_way_seller_transition_clustered,main dynamic association,turnover-conditional differential rank movement; observed turnover is not randomized
2,Dynamic starting-rank quartile adjustment,FBA x total seller-turnover rank movement with additive lag-rank quartile controls,0.002710,0.000869,0.002782,two-way seller-transition,main rank-position validation,controls starting-rank regions parsimoniously; q1 top quartile is omitted
3,Dynamic starting-rank quadratic adjustment,FBA x total seller-turnover rank movement with lag_rank_pct squared,0.002278,0.000674,0.001278,two-way seller-transition,main rank-position validation,smooth nonlinear starting-rank adjustment; avoids transition-cell saturation
4,Dynamic conservative inference,same as primary dynamic premium,0.002467,0.000803,0.024824,CR2 transition,finite-cluster sensitivity,supports inference robustness but not identification
5,Dynamic wild-cluster bootstrap,same as primary dynamic premium,0.002467,0.000765,0.008832,two-way wild cluster bootstrap,finite-cluster sensitivity,bootstrap supports inference robustness but not identification
6,Transition x rank-tier FE stress test,within transition and starting-rank-tier dynamic premium,-0.000075,0.000699,0.915343,two-way seller-transition with stricter FE,severe support-sensitive stress test,nonconfirmation in saturated cells; interpreted cautiously when condition number or within-cell support is weak
7,Complementary top-10 entry outcome,FBA x turnover effect on entering top 10 among at-risk sellers,0.012174,0.005032,0.018599,preferred_two_way_seller_transition_clustered_lpm,auxiliary salience evidence,rare-event visibility translation; not the primary empirical backbone
8,Fractional-logit bounded-outcome diagnostic,average partial effect on rank_pct,-0.014065,0.011644,0.231724,two_way_seller_market,functional-form diagnostic,does not confirm the linear magnitude; OLS is a linear projection
9,Quadratic-control functional-form sensitivity,within-market residual rank_pct gap with nonlinear controls,-0.035229,0.012232,0.005480,two_way_seller_market,functional-form diagnostic,confirms the sign but reduces the magnitude relative to headline OLS


,figure_file,source_table,claim_role,main_or_appendix,required_caveat
0,thesis_figures_aligned/figure_01_static_fba_attenuation_aligned.png,main_results_market_fe_rankpct.csv; attenuation_decomposition_rankpct.csv,static core evidence,main,negative coefficients mean better rank_pct; static association is not a causal FBA adoption effect
1,thesis_figures_aligned/figure_02_dynamic_premium_ranktier_aligned.png,dynamic_fba_premium_inference_rankpct.csv; dynamic_starting_rank_adjustment_diagnostics_rankpct.csv; dynamic_rank_tier_stress_diagnostics_rankpct.csv,dynamic core evidence and D1-D4 hierarchy,main,"D2/D3 are main starting-rank validations; D4 is a severe support-sensitive stress test, not the preferred estimator"
2,thesis_figures_aligned/figure_03_topk_threshold_family_aligned.png,binary_top10_prominence_threshold_family_rankpct.csv,auxiliary top-of-list salience threshold family,appendix,top-10/top-k outcomes are sparse-event auxiliary LPM diagnostics and do not replace continuous rank-percent evidence
3,thesis_figures_aligned/figure_04_distance_banded_turnover_aligned.png,dynamic_distance_banded_turnover_rankpct.csv,mechanism diagnostic for local vacancy filling,main,distance-banded checks test the narrow local-vacancy mechanism and do not falsify the broader turnover-period premium
4,thesis_figures_aligned/figure_05_overlap_smd_summary_aligned.png,overlap_balance_summary_rankpct.csv; overlap_design_status_summary_rankpct.csv,static overlap/support validity gate,main,overlap failure weakens static like-for-like causal interpretation and is not a robustness confirmation
5,thesis_figures_aligned/figure_06_price_interaction_margins_aligned.png,interaction_margins_rankpct.csv,exploratory price-interaction context,appendix,"interaction margins are descriptive heterogeneity checks, not causal moderation evidence"


,synthesis_file,source_tables,claim_role,main_or_appendix,required_caveat
0,thesis_synthesis_aligned/thesis_master_evidence_table_aligned.csv,main_results_market_fe_rankpct.csv; dynamic_fba_premium_inference_rankpct.csv; dynamic_starting_rank_adjustment_diagnostics_rankpct.csv; dynamic_rank_tier_stress_diagnostics_ra...,compact thesis-facing master evidence table,main,summarizes observational fixed-effects associations only; it is not a causal proof table
1,thesis_synthesis_aligned/thesis_functional_form_fragility_aligned.csv,functional_form_fragility_summary_rankpct.csv; bounded_outcome_diagnostics_rankpct.csv; fractional_logit_ape_rankpct.csv,bounded-outcome and functional-form caveat,main,narrows magnitude interpretation rather than refuting the linear projection
2,thesis_synthesis_aligned/thesis_logistics_value_adjusted_price_specification_aligned.csv,logistics_value_adjusted_price_results_rankpct.csv,FBA-only fast-delivery-cost sensitivity,appendix,constructed logistics-value proxy; not a causal robustness check
3,thesis_synthesis_aligned/thesis_oster_sensitivity_aligned.csv,omitted_variable_sensitivity_rankpct.csv; thesis_oster_sensitivity_grid.csv,omitted-confounding sensitivity,appendix,selection-on-observables diagnostic only
4,thesis_synthesis_aligned/thesis_oster_sensitivity_grid_aligned.csv,thesis_oster_sensitivity_grid.csv,Oster Rmax grid detail,appendix,grid detail is not a main result
5,thesis_synthesis_aligned/thesis_dropout_return_pattern_aligned.csv,dynamic_return_pattern_rankpct.csv; dynamic_return_pattern_summary_rankpct.csv,dropout/return context for stockout interpretation,appendix,seller disappearance is not randomized stockout variation
6,thesis_synthesis_aligned/thesis_pre_disappearance_abnormal_smoothness_aligned.csv,dynamic_pre_disappearance_smoothness_rankpct.csv; dynamic_pre_disappearance_smoothness_decision_rankpct.csv,pre-disappearance smoothness diagnostic,appendix,descriptive process diagnostic only
7,thesis_synthesis_aligned/thesis_consistency_warning_summary_aligned.csv,audit_consistency_checks.csv; numerical_pathology_summary.csv,audit warning audit,audit_only,"audit trail, not evidence of an estimand"
8,thesis_synthesis_aligned/thesis_price_interaction_key_result_aligned.csv,interaction_terms_rankpct.csv; interaction_interpretation_rankpct.csv,exploratory heterogeneity context,appendix,not causal moderation evidence
9,thesis_synthesis_aligned/thesis_price_interaction_margins_aligned.csv,interaction_margins_rankpct.csv,exploratory interaction margins,appendix,appendix visualization support only


### 45. Machine-readable aggregated outputs

The cell builds a consolidated `Machine-Readable-Results-Aggregated/` folder that preserves the complete raw source table, the complete EDA output set, and the complete econometric output set in a compact structured form. Figures and redundant preview files are not copied. The export writes one primary data file, `machine_readable_results.jsonl`, plus a compact index `machine_readable_results_index.json`. Tables are represented by schema records and row chunks, so the original CSV contents can be reconstructed exactly while avoiding repeated CSV, preview, and profile files. Each table record records its source path, row count, column names, and content checksum.


In [ ]:
# -----------------------------------------------------------------------------
# Section 45. Machine-readable aggregated outputs
# -----------------------------------------------------------------------------
# Build the consolidated Machine-Readable-Results-Aggregated folder with the raw source table, EDA tables, econometric result tables, and text-based result files in compact structured form.

TEXT_EXTS = {'.json', '.jsonl', '.yaml', '.yml', '.md', '.txt'}
TABLE_EXTS = {'.csv'}
FIGURE_EXTS = {'.png', '.jpg', '.jpeg', '.svg', '.pdf', '.webp'}


def _candidate_roots():
    roots = [
        Path('/content/drive/MyDrive'),
        Path.cwd(),
        Path('/content'),
        Path('/mnt/data'),
    ]
    resolved = []

    for root in roots:
        try:
            root = root.expanduser().resolve()
        except Exception:
            continue

        if str(root) in {'/', '/proc', '/sys', '/dev'}:
            continue

        if root.exists() and root not in resolved:
            resolved.append(root)

    return resolved


def _iter_paths_limited(root, max_depth=5):
    root = Path(root)
    base_depth = len(root.parts)
    skip_dirs = {
        '.git',
        '__pycache__',
        '.ipynb_checkpoints',
        'node_modules',
        '.cache',
        '.config',
        '.local',
    }

    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        depth = len(current.parts) - base_depth

        dirnames[:] = [
            d for d in dirnames
            if d not in skip_dirs and not d.startswith('.')
        ]

        if depth >= max_depth:
            dirnames[:] = []

        for name in filenames:
            yield current / name

        for name in dirnames:
            yield current / name


def _search_path(filename_patterns, file_only=True, dir_only=False):
    candidates = []

    for root_priority, root in enumerate(_candidate_roots()):
        for path in _iter_paths_limited(root, max_depth=5):
            if not any(fnmatch.fnmatch(path.name, pattern) for pattern in filename_patterns):
                continue

            if file_only and not path.is_file():
                continue

            if dir_only and not path.is_dir():
                continue

            try:
                mtime = path.stat().st_mtime
            except Exception:
                mtime = 0

            candidates.append((root_priority, -mtime, len(str(path)), path))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: (x[0], x[1], x[2]))
    return candidates[0][3]


def _find_collection(kind):
    if kind == 'EDA':
        dir_patterns = [
            'EDA-Results',
            'EDA_Results',
            '*EDA*Results*',
            '*EDA*Result*',
        ]
        zip_patterns = [
            'EDA-Results*.zip',
            'EDA_Results*.zip',
            '*EDA*Results*.zip',
            '*EDA*Result*.zip',
        ]

    elif kind == 'ECON':
        dir_patterns = [
            'Econometrics-Results',
            'Econometrics_Results',
            '*Econometrics*Results*',
            '*Econometrics*Result*',
        ]
        zip_patterns = [
            'Econometrics-Results*.zip',
            'Econometrics_Results*.zip',
            '*Econometrics*Results*.zip',
            '*Econometrics*Result*.zip',
        ]

    else:
        raise ValueError(kind)

    found_dir = _search_path(dir_patterns, file_only=False, dir_only=True)
    if found_dir is not None:
        return found_dir

    found_zip = _search_path(zip_patterns, file_only=True)
    if found_zip is not None:
        return found_zip

    return None


def _maybe_prompt_colab_upload(missing):
    if not missing:
        return

    try:
        from google.colab import files
    except Exception:
        return

    print('Upload the missing file(s):', ', '.join(missing))
    files.upload()


def _sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()


def _safe_id(value, max_len=100):
    value = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value))
    value = re.sub(r'_+', '_', value).strip('._-')
    return value[:max_len] or 'item'


def _read_csv_as_strings(path):
    encodings = ['utf-8', 'utf-8-sig', 'latin-1']

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                sep=None,
                engine='python',
                dtype=str,
                keep_default_na=False,
                encoding=encoding,
            )
        except Exception:
            pass

    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding='latin-1',
    )


def _compact_profile(df):
    columns = [str(c) for c in df.columns]
    empty_counts = {}
    example_values = {}

    for col in columns:
        ser = df[col].astype(str)
        empty_counts[col] = int((ser.str.len() == 0).sum())
        example_values[col] = (
            ser[ser.str.len() > 0]
            .drop_duplicates()
            .head(5)
            .tolist()
        )

    return {
        'rows': int(len(df)),
        'columns': int(len(columns)),
        'column_names': columns,
        'empty_string_count_by_column': empty_counts,
        'example_values_by_column': example_values,
    }


def _dataframe_chunks(df, chunk_size=1000):
    values = df.astype(object).where(pd.notna(df), None).values.tolist()

    for start in range(0, len(values), chunk_size):
        yield start, values[start:start + chunk_size]


def _classify_section(source_path):
    text = source_path.lower()

    if 'eda outputs' in text or 'eda_outputs' in text:
        return 'eda'

    if 'econ outputs' in text or 'econ_outputs' in text:
        return 'econometrics'

    if (
        'sport e tempo libero' in text
        or 'xiaomi mi smart band 6.csv' in text
        or 'raw' in text
    ):
        return 'raw_source'

    return 'other'


def _iter_zip_payloads(zip_bytes_or_path, prefix):
    with zipfile.ZipFile(zip_bytes_or_path) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue

            payload = zf.read(info.filename)
            source_path = f'{prefix}/{info.filename}'

            if Path(info.filename).suffix.lower() == '.zip':
                try:
                    yield from _iter_zip_payloads(io.BytesIO(payload), source_path)
                except Exception:
                    yield source_path, payload
            else:
                yield source_path, payload


def _iter_collection_payloads(collection_path, prefix):
    collection_path = Path(collection_path)

    if collection_path.is_file() and collection_path.suffix.lower() == '.zip':
        yield from _iter_zip_payloads(collection_path, prefix)

    elif collection_path.is_dir():
        for path in sorted(collection_path.rglob('*')):
            if path.is_file():
                rel = path.relative_to(collection_path).as_posix()
                yield f'{prefix}/{rel}', path.read_bytes()

    else:
        raise ValueError(f'Unsupported collection path: {collection_path}')


def _collect_logical_files(raw_csv_path, eda_source_path, econ_source_path):
    logical_files = []

    raw_csv_path = Path(raw_csv_path)
    logical_files.append({
        'source_collection': 'raw_source',
        'source_path': raw_csv_path.name,
        'bytes': raw_csv_path.read_bytes(),
    })

    for source_path, payload in _iter_collection_payloads(eda_source_path, 'EDA-Results'):
        logical_files.append({
            'source_collection': 'EDA-Results',
            'source_path': source_path,
            'bytes': payload,
        })

    for source_path, payload in _iter_collection_payloads(econ_source_path, 'Econometrics-Results'):
        logical_files.append({
            'source_collection': 'Econometrics-Results',
            'source_path': source_path,
            'bytes': payload,
        })

    return logical_files


def _collection_parent(path):
    path = Path(path)
    return path.parent


def _infer_repository_dir(eda_source_path, econ_source_path):
    eda_parent = _collection_parent(eda_source_path)
    econ_parent = _collection_parent(econ_source_path)

    if eda_parent == econ_parent:
        return eda_parent

    common = Path(os.path.commonpath([str(eda_parent), str(econ_parent)]))

    if common.exists() and str(common) not in {
        '/',
        '/content',
        '/content/drive',
        '/content/drive/MyDrive',
    }:
        return common

    return eda_parent


def build_machine_readable_results(
    raw_csv_path=None,
    eda_zip_path=None,
    econ_zip_path=None,
    eda_source_path=None,
    econ_source_path=None,
    output_dir=None,
    chunk_size=1000,
):
    raw_csv_path = Path(raw_csv_path) if raw_csv_path is not None else _search_path([
        'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv',
        '*Xiaomi*Mi*Smart*Band*6*.csv',
    ], file_only=True)

    eda_source_path = Path(eda_source_path) if eda_source_path is not None else (
        Path(eda_zip_path) if eda_zip_path is not None else _find_collection('EDA')
    )

    econ_source_path = Path(econ_source_path) if econ_source_path is not None else (
        Path(econ_zip_path) if econ_zip_path is not None else _find_collection('ECON')
    )

    missing = [
        name for name, path in {
            'raw_csv_path': raw_csv_path,
            'eda_source_path_or_zip': eda_source_path,
            'econ_source_path_or_zip': econ_source_path,
        }.items()
        if path is None or not Path(path).exists()
    ]

    if missing:
        _maybe_prompt_colab_upload(missing)

        raw_csv_path = raw_csv_path if raw_csv_path is not None and Path(raw_csv_path).exists() else _search_path([
            'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv',
            '*Xiaomi*Mi*Smart*Band*6*.csv',
        ], file_only=True)

        eda_source_path = (
            eda_source_path
            if eda_source_path is not None and Path(eda_source_path).exists()
            else _find_collection('EDA')
        )

        econ_source_path = (
            econ_source_path
            if econ_source_path is not None and Path(econ_source_path).exists()
            else _find_collection('ECON')
        )

        missing = [
            name for name, path in {
                'raw_csv_path': raw_csv_path,
                'eda_source_path_or_zip': eda_source_path,
                'econ_source_path_or_zip': econ_source_path,
            }.items()
            if path is None or not Path(path).exists()
        ]

    if missing:
        raise FileNotFoundError(
            'Missing required input(s): '
            + str(missing)
            + '\nSearched roots: '
            + str([str(p) for p in _candidate_roots()])
            + '\nExpected either folders or ZIP files named like EDA-Results* and Econometrics-Results*, plus the raw Xiaomi CSV.'
        )

    if output_dir is None:
        repository_dir = _infer_repository_dir(eda_source_path, econ_source_path)
        output_dir = repository_dir / 'Machine-Readable-Results-Aggregated'
    else:
        output_dir = Path(output_dir)

    if output_dir.name != 'Machine-Readable-Results-Aggregated':
        output_dir = output_dir / 'Machine-Readable-Results-Aggregated'

    if output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_path = output_dir / 'machine_readable_results.jsonl'
    index_path = output_dir / 'machine_readable_results_index.json'
    readme_path = output_dir / 'README.md'

    logical_files = _collect_logical_files(
        raw_csv_path,
        eda_source_path,
        econ_source_path,
    )

    seen_payloads = {}
    tables = []
    text_documents = []
    skipped_files = []
    duplicate_files = []
    total_rows = 0
    total_chunks = 0

    with jsonl_path.open('w', encoding='utf-8') as out:

        def emit(record):
            out.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(',', ':'),
                    default=str,
                )
                + '\n'
            )

        emit({
            'record_type': 'package_metadata',
            'package_name': 'Machine-Readable-Results-Aggregated',
            'description': 'Consolidated empirical data and results for the Xiaomi Mi Smart Band 6 FBA ranking analysis. Figures and duplicate binary archives are excluded; tabular data and text-based results are preserved in structured records.',
            'format': 'jsonl',
            'table_storage': 'Each table is stored as a schema record plus row chunks. Row chunks use column order and list-of-lists data to avoid repeating column names on every row.',
            'source_inputs': {
                'raw_csv_path': str(raw_csv_path),
                'eda_source_path': str(eda_source_path),
                'econ_source_path': str(econ_source_path),
            },
            'figure_policy': 'Image and vector-figure files are not included in this folder.',
        })

        for logical_file in logical_files:
            source_path = logical_file['source_path']
            payload = logical_file['bytes']
            suffix = Path(source_path).suffix.lower()
            digest = _sha256_bytes(payload)
            section = _classify_section(source_path)

            if suffix in FIGURE_EXTS:
                skipped_files.append({
                    'source_path': source_path,
                    'section': section,
                    'sha256': digest,
                    'bytes': len(payload),
                    'reason': 'figure_excluded',
                })
                continue

            if suffix not in TABLE_EXTS and suffix not in TEXT_EXTS:
                skipped_files.append({
                    'source_path': source_path,
                    'section': section,
                    'sha256': digest,
                    'bytes': len(payload),
                    'reason': 'unsupported_binary_or_archive',
                })
                continue

            if digest in seen_payloads:
                duplicate_files.append({
                    'source_path': source_path,
                    'section': section,
                    'sha256': digest,
                    'duplicate_of': seen_payloads[digest],
                })

                emit({
                    'record_type': 'duplicate_reference',
                    'source_path': source_path,
                    'section': section,
                    'sha256': digest,
                    'duplicate_of': seen_payloads[digest],
                })

                continue

            base_name = _safe_id(Path(source_path.split('::')[-1]).stem)
            object_id = f'{section}.{base_name}.{digest[:10]}'
            seen_payloads[digest] = object_id

            if suffix == '.csv':
                with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as tmp:
                    tmp.write(payload)
                    tmp_path = Path(tmp.name)

                try:
                    df = _read_csv_as_strings(tmp_path)
                finally:
                    try:
                        tmp_path.unlink()
                    except Exception:
                        pass

                columns = [str(c) for c in df.columns]

                emit({
                    'record_type': 'table_schema',
                    'table_id': object_id,
                    'section': section,
                    'source_path': source_path,
                    'sha256': digest,
                    'rows': int(len(df)),
                    'columns': int(len(columns)),
                    'column_names': columns,
                })

                emit({
                    'record_type': 'table_profile',
                    'table_id': object_id,
                    'section': section,
                    'source_path': source_path,
                    **_compact_profile(df),
                })

                chunk_count = 0

                for row_start, rows in _dataframe_chunks(df, chunk_size=chunk_size):
                    emit({
                        'record_type': 'table_chunk',
                        'table_id': object_id,
                        'section': section,
                        'source_path': source_path,
                        'row_start': int(row_start),
                        'row_count': int(len(rows)),
                        'column_names': columns,
                        'data': rows,
                    })
                    chunk_count += 1

                tables.append({
                    'table_id': object_id,
                    'section': section,
                    'source_path': source_path,
                    'sha256': digest,
                    'rows': int(len(df)),
                    'columns': int(len(columns)),
                    'chunks': int(chunk_count),
                    'extension': suffix,
                })

                total_rows += len(df)
                total_chunks += chunk_count

            else:
                content = payload.decode('utf-8', errors='replace')

                emit({
                    'record_type': 'text_document',
                    'document_id': object_id,
                    'section': section,
                    'source_path': source_path,
                    'sha256': digest,
                    'extension': suffix,
                    'characters': int(len(content)),
                    'content': content,
                })

                text_documents.append({
                    'document_id': object_id,
                    'section': section,
                    'source_path': source_path,
                    'sha256': digest,
                    'characters': int(len(content)),
                    'extension': suffix,
                })

        emit({
            'record_type': 'skipped_files_summary',
            'files': skipped_files,
        })

        emit({
            'record_type': 'duplicate_files_summary',
            'files': duplicate_files,
        })

    index = {
        'package_name': 'Machine-Readable-Results-Aggregated',
        'data_file': 'machine_readable_results.jsonl',
        'format_note': 'One JSON object per line. Table data are stored as table_schema, table_profile, and table_chunk records.',
        'counts': {
            'tables': int(len(tables)),
            'table_rows_total': int(total_rows),
            'table_chunks': int(total_chunks),
            'text_documents': int(len(text_documents)),
            'skipped_figures_or_binary_files': int(len(skipped_files)),
            'duplicate_payloads_not_repeated': int(len(duplicate_files)),
            'logical_source_files_seen': int(len(logical_files)),
        },
        'tables': tables,
        'text_documents': text_documents,
        'skipped_files': skipped_files,
        'duplicate_files': duplicate_files,
    }

    with index_path.open('w', encoding='utf-8') as f:
        json.dump(index, f, ensure_ascii=False, indent=2, default=str)

    readme_path.write_text(
        '# Machine-Readable-Results-Aggregated\n\n'
        'This folder contains a compact structured export of the empirical record. It excludes figures and avoids duplicating CSV, Markdown, and preview files.\n\n'
        'Files:\n\n'
        '- `machine_readable_results.jsonl`: complete raw table, EDA tables, econometric result tables, and text-based result files.\n'
        '- `machine_readable_results_index.json`: table and document index with row counts, columns, source paths, checksums, and skipped-file notes.\n\n'
        'Table reconstruction rule:\n\n'
        '1. Find the `table_schema` record for the required `table_id`.\n'
        '2. Read all `table_chunk` records with the same `table_id`.\n'
        '3. Reconstruct the table by applying `column_names` to each row in the `data` list.\n\n'
        'Figures are not included in this folder. The corresponding tabular inputs and figure manifests remain represented as structured text/table records when present.\n',
        encoding='utf-8',
    )

    return {
        'aggregated_results_dir': str(output_dir),
        'aggregated_results_zip': None,
        'data_file': str(jsonl_path),
        'index_file': str(index_path),
        'input_locations': {
            'raw_csv_path': str(raw_csv_path),
            'eda_source_path': str(eda_source_path),
            'econ_source_path': str(econ_source_path),
        },
        'searched_roots': [str(p) for p in _candidate_roots()],
        'tables': int(len(tables)),
        'table_rows_total': int(total_rows),
        'table_chunks': int(total_chunks),
        'text_documents': int(len(text_documents)),
        'skipped_figures_or_binary_files': int(len(skipped_files)),
        'duplicate_payloads_not_repeated': int(len(duplicate_files)),
        'logical_source_files_seen': int(len(logical_files)),
        'jsonl_size_bytes': int(jsonl_path.stat().st_size),
        'folder_size_bytes': int(sum(p.stat().st_size for p in output_dir.rglob('*') if p.is_file())),
    }


machine_readable_results_summary = build_machine_readable_results()
machine_readable_results_summary

{'aggregated_results_dir': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Machine-Readable-Results-Aggregated',
 'aggregated_results_zip': None,
 'data_file': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Machine-Readable-Results-Aggregated/machine_readable_results.jsonl',
 'index_file': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Machine-Readable-Results-Aggregated/machine_readable_results_index.json',
 'input_locations': {'raw_csv_path': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv',
  'eda_source_path': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/EDA-Results',
  'econ_source_path': '/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets/Econometrics-Results'},
 'searched_roots': ['/content/drive/MyDrive', '/content'],
 'tables': 223,
 'table_rows_total': 49027,
 'table_chunks': 265,
 'text_d

### 46. Limitations and scope of inference

The empirical design is deliberately narrow. The following limitations define the scope of the claim that the thesis supports.

#### No within-seller FBA switching

The final panel contains no seller observed under both fulfillment statuses. No specification therefore identifies the causal effect of adopting FBA for the same seller. Static estimates compare FBA and non-FBA sellers within the same market after conditioning on observables.

#### Single-product external validity

The final panel is dense within one Amazon.it product page but does not establish external validity across products, categories, platforms, or time periods. Replication across multiple products would be required to support a broader platform-level claim.

#### Observed disappearance is not randomized stockout variation

Seller disappearance from the offer list can reflect inventory exhaustion, listing decisions, scraping-window incidents, platform enforcement, or other unobserved processes. The stockout-consistency restriction narrows the interpretation but does not convert disappearance into randomized stockout variation.

#### Common support does not deliver a balanced static comparison

Propensity-score overlap restrictions are treated as support diagnostics. The conclusion is not that trimming validates the static estimate; rather, trimming shows that balanced like-for-like FBA/non-FBA comparisons are not available in this single-product panel.

#### Functional-form diagnostics narrow the static magnitude

`rank_pct` is bounded between 0 and 1 while the headline static estimator is a linear projection. Bounded-outcome and nonlinear-control diagnostics are used to bound the interpretation of the static magnitude; the sign of the residual association is more stable than its exact magnitude.

#### FBA-only logistics-value adjustment is a sensitivity estimand

The FBA-only logistics-value-adjusted price specification is not a direct measurement of the checkout price and is not a robustness check on the headline split-price coefficient. It re-estimates the static model after replacing the separate product-price and shipping controls with a constructed total price that adds an imputed hidden fast-delivery value only to FBA listings with fast-delivery availability. The adjustment is post-treatment with respect to the FBA bundle by construction.

#### Dynamic fixed-effects coefficient is a linear-projection estimand

The dynamic specification is a fixed-effects linear projection of heterogeneous turnover-response patterns. It is not a staggered-adoption difference-in-differences design and not a homogeneous structural treatment-effect estimator. Recent two-way fixed-effects critiques (de Chaisemartin and D'Haultfœuille 2020; Sun and Abraham 2021) motivate caution about heterogeneous effects, but the design here does not correspond to a standard absorbing-treatment event study because FBA does not switch within seller in the final panel.

#### Dynamic turnover is observed seller-list turnover, not randomized displacement

The dynamic exposure measures observed disappearances of competing sellers from the offer list between consecutive snapshots. The exposure is not randomized displacement of competitors. The dynamic interaction is therefore a turnover-period association, not a causal mechanism estimate of vacancy filling.


### 47. Conclusion

The notebook provides audited observational evidence on the relationship between FBA status and ranking outcomes for the Xiaomi Mi Smart Band 6 on Amazon Italy. The static analysis records a large raw within-market FBA ranking advantage and a smaller controlled residual association after conditioning on observed reputation, price, shipping, delivery, and market fixed effects. The static coefficient is not a causal adoption effect; it is a within-market conditional association between FBA status and normalized rank.

The bounded-outcome diagnostics and the nonlinear-control specifications narrow the magnitude claim. The static result is reported as a controlled linear-projection association whose sign is more stable than its exact magnitude. The FBA-only logistics-value-adjusted price specification is reported alongside the split-price headline as a post-treatment sensitivity estimand; it re-estimates the static model after replacing the separate product-price and shipping controls with a constructed total price that adds an imputed hidden fast-delivery value only to FBA listings with fast-delivery availability.

The continuous dynamic analysis estimates a different object: a turnover-conditional FBA rank-movement association among continuing sellers. The baseline dynamic model estimates a positive FBA-by-total-turnover coefficient (0.002467, p-value 0.0020). The coefficient remains positive under the additive starting-rank quartile controls (0.002710, p-value 0.0028) and under the quadratic lagged-rank adjustment (0.002278, p-value 0.0013). The saturated transition-by-rank-tier model is nonconfirming (-0.000075, p-value 0.9153) and numerically thin; it is retained as a severe stress test rather than the primary rank-position-adjusted estimator.

The distance-banded turnover diagnostics do not support a clean local-vacancy-filling mechanism. The dynamic evidence is therefore described as a broader turnover-period association rather than as proof that FBA mechanically fills nearby ranking slots left vacant by disappearing sellers. The complementary top-10 analysis indicates whether the continuous dynamic association translates into economically salient top-of-list entry; it remains auxiliary because the event count is sparse, the conservative inference is weaker than the continuous-rank evidence, and the linear-probability model is explicitly audited for fitted probabilities outside the unit interval.

The final claim supported by the notebook is bounded: the evidence supports an audited observational FBA-associated ranking premium within the studied seller-list panel, conditional on observed commercial and reputation characteristics, with magnitudes sensitive to functional form and with no causal adoption-effect interpretation.


### 48. References

Acharya, A., Blackwell, M., and Sen, M. (2016). Explaining causal findings without bias: detecting and assessing direct effects. *American Political Science Review*, 110(3): 512–529.

Angrist, J. D., and Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.

Bell, R. M., and McCaffrey, D. F. (2002). Bias reduction in standard errors for linear regression with multi-stage samples. *Survey Methodology*, 28(2): 169–181.

Cameron, A. C., Gelbach, J. B., and Miller, D. L. (2008). Bootstrap-based improvements for inference with clustered errors. *Review of Economics and Statistics*, 90(3): 414–427.

Cameron, A. C., Gelbach, J. B., and Miller, D. L. (2011). Robust inference with multiway clustering. *Journal of Business and Economic Statistics*, 29(2): 238–249.

Cameron, A. C., and Miller, D. L. (2015). A practitioner's guide to cluster-robust inference. *Journal of Human Resources*, 50(2): 317–372.

Cinelli, C., and Hazlett, C. (2020). Making sense of sensitivity: extending omitted variable bias. *Journal of the Royal Statistical Society: Series B*, 82(1): 39–67.

Correia, S., Guimarães, P., and Zylkin, T. (2020). Fast Poisson estimation with high-dimensional fixed effects. *The Stata Journal*, 20(1): 95–115.

de Chaisemartin, C., and D'Haultfœuille, X. (2020). Two-way fixed effects estimators with heterogeneous treatment effects. *American Economic Review*, 110(9): 2964–2996.

Imbens, G. W., and Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences*. Cambridge University Press.

Imbens, G. W., and Wooldridge, J. M. (2009). Recent developments in the econometrics of program evaluation. *Journal of Economic Literature*, 47(1): 5–86.

MacKinnon, J. G., and Webb, M. D. (2017). Wild bootstrap inference for wildly different cluster sizes. *Journal of Applied Econometrics*, 32(2): 233–254.

MacKinnon, J. G., Nielsen, M. Ø., and Webb, M. D. (2022). Cluster-robust inference: a guide to empirical practice. *Journal of Econometrics*, forthcoming.

Mundlak, Y. (1978). On the pooling of time series and cross section data. *Econometrica*, 46(1): 69–85.

Oster, E. (2019). Unobservable selection and coefficient stability: theory and evidence. *Journal of Business and Economic Statistics*, 37(2): 187–204.

Papke, L. E., and Wooldridge, J. M. (1996). Econometric methods for fractional response variables with an application to 401(k) plan participation rates. *Journal of Applied Econometrics*, 11(6): 619–632.

Sun, L., and Abraham, S. (2021). Estimating dynamic treatment effects in event studies with heterogeneous treatment effects. *Journal of Econometrics*, 225(2): 175–199.

Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data*. MIT Press.

Amazon Services. *Fulfillment by Amazon (FBA)*. Official Amazon seller documentation, accessed May 2026.

Xiaomi. *Mi Smart Band 6 Specs*. Official Xiaomi global product specifications, accessed May 2026.
